In [1]:
import importlib
import market_structure

importlib.reload(market_structure)

<module 'market_structure' from '/home/sattar/projects/trader-bot/market_structure.py'>

In [2]:
import inspect
import gateway_client

print("gateway_client:", gateway_client.__file__)
print("\nAvailable callables:")

for name, obj in vars(gateway_client).items():
    if callable(obj) and not name.startswith("_"):
        try:
            sig = inspect.signature(obj)
        except (ValueError, TypeError):
            sig = "<signature unavailable>"
        
        print(f" - {name}{sig}")

gateway_client: /home/sattar/projects/trader-bot/gateway_client.py

Available callables:
 - dataclass(cls=None, /, *, init=True, repr=True, eq=True, order=False, unsafe_hash=False, frozen=False, match_args=True, kw_only=False, slots=False, weakref_slot=False)
 - datetime<signature unavailable>
 - Any(*args, **kwargs)
 - GatewayError<signature unavailable>
 - GatewayConnectionError<signature unavailable>
 - GatewayAPIError<signature unavailable>
 - Tick(symbol: 'str', bid: 'float', ask: 'float', last: 'float', time: 'int', datetime: 'datetime | None' = None) -> None
 - GatewayClient(base_url: 'str' = 'http://localhost:8000', timeout: 'int' = 10)
 - get_rates(symbol: 'str' = 'EURUSD', timeframe: 'str' = 'M5', count: 'int' = 500) -> 'pd.DataFrame'


In [3]:
import pandas as pd
import numpy as np
import gateway_client

m5_df = gateway_client.get_rates(
    symbol="EURUSD",
    timeframe="M5",
    count=1000,
)

print("m5_df type :", type(m5_df))
print("shape       :", m5_df.shape)
print("columns     :", list(m5_df.columns))
print("index type  :", type(m5_df.index))
print("\nhead:")
display(m5_df.head())

m5_df type : <class 'pandas.DataFrame'>
shape       : (1000, 9)
columns     : ['time', 'datetime', 'open', 'high', 'low', 'close', 'tick_volume', 'spread', 'real_volume']
index type  : <class 'pandas.RangeIndex'>

head:


,time,datetime,open,high,low,close,tick_volume,spread,real_volume
0,1790969400,2026-10-02 19:30:00+00:00,1.12512,1.12539,1.12508,1.12512,332,2,0
1,1790969700,2026-10-02 19:35:00+00:00,1.12511,1.12555,1.12505,1.12548,275,2,0
2,1790970000,2026-10-02 19:40:00+00:00,1.12547,1.12553,1.12505,1.12519,264,2,0
3,1790970300,2026-10-02 19:45:00+00:00,1.12518,1.12614,1.12514,1.12571,395,2,0
4,1790970600,2026-10-02 19:50:00+00:00,1.12567,1.12580,1.12533,1.12571,270,2,0


In [4]:
print("m5_df index type :", type(m5_df.index))
print("m5_df index name :", m5_df.index.name)
print("m5_df shape      :", m5_df.shape)
print("time dtype       :", m5_df["time"].dtype)

display(m5_df.head())

m5_df index type : <class 'pandas.RangeIndex'>
m5_df index name : None
m5_df shape      : (1000, 9)
time dtype       : int64


,time,datetime,open,high,low,close,tick_volume,spread,real_volume
0,1790969400,2026-10-02 19:30:00+00:00,1.12512,1.12539,1.12508,1.12512,332,2,0
1,1790969700,2026-10-02 19:35:00+00:00,1.12511,1.12555,1.12505,1.12548,275,2,0
2,1790970000,2026-10-02 19:40:00+00:00,1.12547,1.12553,1.12505,1.12519,264,2,0
3,1790970300,2026-10-02 19:45:00+00:00,1.12518,1.12614,1.12514,1.12571,395,2,0
4,1790970600,2026-10-02 19:50:00+00:00,1.12567,1.12580,1.12533,1.12571,270,2,0


In [5]:
import pandas as pd
import gateway_client

m5_df = gateway_client.get_rates(
    symbol="EURUSD",
    timeframe="M5",
    count=1000,
).copy()

m5_df["time"] = pd.to_datetime(
    m5_df["time"],
    unit="s",
    utc=True,
)

m5_df = (
    m5_df
    .set_index("time")
    .sort_index()
)

print("m5_df shape :", m5_df.shape)
print("index type  :", type(m5_df.index))
print("first       :", m5_df.index[0])
print("last        :", m5_df.index[-1])

m5_df shape : (1000, 8)
index type  : <class 'pandas.DatetimeIndex'>
first       : 2026-10-02 19:30:00+00:00
last        : 2026-10-08 08:05:00+00:00


In [6]:
from market_structure import (
    validate_ohlc,
    detect_raw_swings,
    calculate_swing_strength,
    build_raw_swings,
    get_swings,
    filter_minor_swings_v2,
    build_minor_structure_v2,
    calculate_minor_quality,
    classify_minor_structure,
    identify_structural_levels,
    detect_minor_bos_choch,
    detect_minor_bos,
    detect_structural_choch,
    build_market_bias,
    build_structure_state,
)

In [7]:
print("build_minor_structure_v2 :", build_minor_structure_v2)
print("classify_minor_structure  :", classify_minor_structure)
print("identify_structural_levels:", identify_structural_levels)

build_minor_structure_v2 : <function build_minor_structure_v2 at 0x79df3fff6770>
classify_minor_structure  : <function classify_minor_structure at 0x79df3fff6ae0>
identify_structural_levels: <function identify_structural_levels at 0x79df3fff6b90>


In [8]:
minor_v2 = build_minor_structure_v2(
    df=m5_df,
    left_bars=2,
    right_bars=2,
    atr_period=14,
    min_strength=1.0,
    min_atr_distance=0.50,
    min_bars_between=2,
)

minor_v2 = classify_minor_structure(minor_v2)

print("minor_v2 shape :", minor_v2.shape)

print()

print("Swing types:")
print(
    minor_v2["swing_type"]
    .value_counts(dropna=False)
)

print()

print("Minor structure:")
print(
    minor_v2["minor_structure"]
    .value_counts(dropna=False)
)

minor_v2 shape : (156, 19)

Swing types:
swing_type
HIGH    78
LOW     78
Name: count, dtype: int64

Minor structure:
minor_structure
HL      43
LH      43
LL      34
HH      33
None     3
Name: count, dtype: int64


In [9]:
def identify_structural_levels(minor_v2):
    """
    Identify Structural High/Low and Protected High/Low
    from Minor Structure v2.
    """
    import pandas as pd

    required = {
        "swing_type",
        "swing_price",
        "minor_structure",
    }

    missing = required - set(minor_v2.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    df = minor_v2.copy()

    df["structural_type"] = None
    df["structural_level"] = None
    df["protected_type"] = None
    df["protected_level"] = None

    for idx, row in df.iterrows():

        structure = row["minor_structure"]
        price = row["swing_price"]

        if structure == "HH":
            df.at[idx, "structural_type"] = "STRUCTURAL_HIGH"
            df.at[idx, "structural_level"] = price

        elif structure == "LL":
            df.at[idx, "structural_type"] = "STRUCTURAL_LOW"
            df.at[idx, "structural_level"] = price

    swing_indices = df.index[
        df["minor_structure"].isin(
            ["HH", "HL", "LH", "LL"]
        )
    ].tolist()

    for i in range(1, len(swing_indices)):

        previous_idx = swing_indices[i - 1]
        current_idx = swing_indices[i]

        previous_structure = df.at[
            previous_idx,
            "minor_structure"
        ]

        current_structure = df.at[
            current_idx,
            "minor_structure"
        ]

        # HL -> HH
        # Previous HL becomes Protected Low
        if (
            previous_structure == "HL"
            and current_structure == "HH"
        ):
            df.at[
                previous_idx,
                "protected_type"
            ] = "PROTECTED_LOW"

            df.at[
                previous_idx,
                "protected_level"
            ] = df.at[
                previous_idx,
                "swing_price"
            ]

        # LH -> LL
        # Previous LH becomes Protected High
        elif (
            previous_structure == "LH"
            and current_structure == "LL"
        ):
            df.at[
                previous_idx,
                "protected_type"
            ] = "PROTECTED_HIGH"

            df.at[
                previous_idx,
                "protected_level"
            ] = df.at[
                previous_idx,
                "swing_price"
            ]

    return df

In [10]:
structural_v1 = identify_structural_levels(minor_v2)

print("structural_v1 shape :", structural_v1.shape)
print()

print("Structural levels:")
print(
    structural_v1["structural_type"]
    .value_counts(dropna=False)
)

print()

print("Protected levels:")
print(
    structural_v1["protected_type"]
    .value_counts(dropna=False)
)

structural_v1 shape : (156, 23)

Structural levels:
structural_type
None               89
STRUCTURAL_LOW     34
STRUCTURAL_HIGH    33
Name: count, dtype: int64

Protected levels:
protected_type
None              105
PROTECTED_HIGH     26
PROTECTED_LOW      25
Name: count, dtype: int64


In [11]:
from market_structure import (
    build_minor_structure_v2,
    classify_minor_structure,
    detect_minor_bos,
)

minor_v2 = build_minor_structure_v2(
    df=m5_df,
    left_bars=2,
    right_bars=2,
    atr_period=14,
    min_strength=1.0,
    min_atr_distance=0.50,
    min_bars_between=2,
)

minor_v2 = classify_minor_structure(minor_v2)

print("minor_v2 shape:", minor_v2.shape)
print("columns:")
print(list(minor_v2.columns))

display(minor_v2.tail(10))

minor_v2 shape: (156, 19)
columns:
['high', 'low', 'close', 'swing_high', 'swing_low', 'swing_type', 'swing_confirmed', 'swing_confirmation_time', 'atr', 'swing_strength', 'swing_price', 'previous_swing_price', 'price_displacement', 'atr_multiple', 'time_from_previous_swing', 'bars_from_previous_swing', 'minor_rank', 'minor_type', 'minor_structure']


,high,low,close,swing_high,swing_low,swing_type,swing_confirmed,swing_confirmation_time,atr,swing_strength,swing_price,previous_swing_price,price_displacement,atr_multiple,time_from_previous_swing,bars_from_previous_swing,minor_rank,minor_type,minor_structure
time,,,,,,,,,,,,,,,,,,,
2026-10-08 03:05:00+00:00,1.12026,1.12005,1.12007,1.12026,NaN,HIGH,True,2026-10-08 03:15:00+00:00,0.000160,2.000000,1.12026,1.11994,0.00032,2.000000,0 days 00:10:00,2.0,2.000000,HIGH,HH
2026-10-08 03:15:00+00:00,1.11986,1.11950,1.11959,NaN,1.11950,LOW,True,2026-10-08 03:25:00+00:00,0.000179,4.256000,1.11950,1.12026,0.00076,4.256000,0 days 00:10:00,2.0,4.256000,LOW,LL
2026-10-08 03:30:00+00:00,1.12000,1.11982,1.11995,1.12000,NaN,HIGH,True,2026-10-08 03:40:00+00:00,0.000198,2.527076,1.12000,1.11950,0.00050,2.527076,0 days 00:15:00,3.0,2.527076,HIGH,LH
2026-10-08 04:05:00+00:00,1.11987,1.11968,1.11981,NaN,1.11968,LOW,True,2026-10-08 04:15:00+00:00,0.000244,2.210526,1.11968,1.12000,0.00032,1.309942,0 days 00:35:00,7.0,1.850292,LOW,HL
2026-10-08 04:20:00+00:00,1.12024,1.11999,1.12005,1.12024,NaN,HIGH,True,2026-10-08 04:30:00+00:00,0.000254,2.202247,1.12024,1.11968,0.00056,2.202247,0 days 00:15:00,3.0,2.202247,HIGH,HH
2026-10-08 04:30:00+00:00,1.12001,1.11974,1.11998,NaN,1.11974,LOW,True,2026-10-08 04:40:00+00:00,0.000244,2.052786,1.11974,1.12024,0.00050,2.052786,0 days 00:10:00,2.0,2.052786,LOW,HL
2026-10-08 05:15:00+00:00,1.12128,1.12104,1.12116,1.12128,NaN,HIGH,True,2026-10-08 05:25:00+00:00,0.000274,5.629243,1.12128,1.11974,0.00154,5.629243,0 days 00:45:00,9.0,5.629243,HIGH,HH
2026-10-08 06:10:00+00:00,1.12019,1.11976,1.11984,NaN,1.11976,LOW,True,2026-10-08 06:20:00+00:00,0.000271,5.614776,1.11976,1.12128,0.00152,5.614776,0 days 00:55:00,11.0,5.614776,LOW,HL
2026-10-08 06:55:00+00:00,1.12053,1.12037,1.12045,1.12053,NaN,HIGH,True,2026-10-08 07:05:00+00:00,0.000233,3.306748,1.12053,1.11976,0.00077,3.306748,0 days 00:45:00,9.0,3.306748,HIGH,LH


In [12]:
m5_df = gateway_client.get_rates(
    symbol="EURUSD",
    timeframe="M5",
    count=1000,
).copy()

print("Raw time sample:")
print(m5_df["time"].head())
print("Raw time dtype:", m5_df["time"].dtype)

m5_df["time"] = pd.to_datetime(
    m5_df["time"],
    unit="s",
    utc=True,
)

m5_df = (
    m5_df
    .set_index("time")
    .sort_index()
)

print("\nNormalized:")
print("shape:", m5_df.shape)
print("first:", m5_df.index[0])
print("last :", m5_df.index[-1])

display(m5_df.head(3))
display(m5_df.tail(3))

Raw time sample:
0    1790969400
1    1790969700
2    1790970000
3    1790970300
4    1790970600
Name: time, dtype: int64
Raw time dtype: int64

Normalized:
shape: (1000, 8)
first: 2026-10-02 19:30:00+00:00
last : 2026-10-08 08:05:00+00:00


,datetime,open,high,low,close,tick_volume,spread,real_volume
time,,,,,,,,
2026-10-02 19:30:00+00:00,2026-10-02 19:30:00+00:00,1.12512,1.12539,1.12508,1.12512,332,2,0
2026-10-02 19:35:00+00:00,2026-10-02 19:35:00+00:00,1.12511,1.12555,1.12505,1.12548,275,2,0
2026-10-02 19:40:00+00:00,2026-10-02 19:40:00+00:00,1.12547,1.12553,1.12505,1.12519,264,2,0


,datetime,open,high,low,close,tick_volume,spread,real_volume
time,,,,,,,,
2026-10-08 07:55:00+00:00,2026-10-08 07:55:00+00:00,1.12052,1.12068,1.12047,1.12049,136,2,0
2026-10-08 08:00:00+00:00,2026-10-08 08:00:00+00:00,1.12047,1.12069,1.12039,1.12066,98,2,0
2026-10-08 08:05:00+00:00,2026-10-08 08:05:00+00:00,1.12069,1.12071,1.12057,1.12065,32,2,0


In [13]:
minor_v2 = build_minor_structure_v2(
    df=m5_df,
    left_bars=2,
    right_bars=2,
    atr_period=14,
    min_strength=1.0,
    min_atr_distance=0.50,
    min_bars_between=2,
)

minor_v2 = classify_minor_structure(minor_v2)

print("minor_v2 shape:", minor_v2.shape)

print("\nSwing type:")
print(minor_v2["swing_type"].value_counts(dropna=False))

print("\nMinor structure:")
print(minor_v2["minor_structure"].value_counts(dropna=False))

display(
    minor_v2[
        [
            "swing_type",
            "swing_price",
            "previous_swing_price",
            "price_displacement",
            "atr",
            "atr_multiple",
            "bars_from_previous_swing",
            "minor_rank",
            "minor_type",
            "minor_structure",
        ]
    ].tail(15)
)

minor_v2 shape: (156, 19)

Swing type:
swing_type
HIGH    78
LOW     78
Name: count, dtype: int64

Minor structure:
minor_structure
HL      43
LH      43
LL      34
HH      33
None     3
Name: count, dtype: int64


,swing_type,swing_price,previous_swing_price,price_displacement,atr,atr_multiple,bars_from_previous_swing,minor_rank,minor_type,minor_structure
time,,,,,,,,,,
2026-10-08 00:20:00+00:00,LOW,1.11895,1.11967,0.00072,0.000266,2.702413,6.0,2.702413,LOW,LL
2026-10-08 00:30:00+00:00,HIGH,1.11956,1.11895,0.00061,0.000299,2.043062,2.0,2.043062,HIGH,LH
2026-10-08 00:45:00+00:00,LOW,1.11904,1.11956,0.00052,0.000331,1.572354,3.0,1.572354,LOW,HL
2026-10-08 01:25:00+00:00,HIGH,1.12025,1.11904,0.00121,0.000367,3.295720,8.0,3.214008,HIGH,HH
2026-10-08 01:55:00+00:00,LOW,1.11991,1.12025,0.00034,0.000260,1.307692,6.0,1.307692,LOW,HL
2026-10-08 03:05:00+00:00,HIGH,1.12026,1.11994,0.00032,0.000160,2.000000,2.0,2.000000,HIGH,HH
2026-10-08 03:15:00+00:00,LOW,1.11950,1.12026,0.00076,0.000179,4.256000,2.0,4.256000,LOW,LL
2026-10-08 03:30:00+00:00,HIGH,1.12000,1.11950,0.00050,0.000198,2.527076,3.0,2.527076,HIGH,LH
2026-10-08 04:05:00+00:00,LOW,1.11968,1.12000,0.00032,0.000244,1.309942,7.0,1.850292,LOW,HL


In [14]:
minor_bos_raw = detect_minor_bos(
    df=m5_df,
    minor_v2=minor_v2,
    break_mode="close",
)

print("minor_bos_raw shape:", minor_bos_raw.shape)

print("\nBOS counts:")
print(
    minor_bos_raw["minor_bos"]
    .value_counts(dropna=False)
)

display(
    minor_bos_raw.loc[
        minor_bos_raw["minor_bos"].notna(),
        [
            "open",
            "high",
            "low",
            "close",
            "minor_bos",
            "bos_level",
        ],
    ].head(20)
)

minor_bos_raw shape: (1000, 10)

BOS counts:
minor_bos
None           938
BOS_BULLISH     31
BOS_BEARISH     31
Name: count, dtype: int64


,open,high,low,close,minor_bos,bos_level
time,,,,,,
2026-10-02 22:20:00+00:00,1.12546,1.12598,1.12543,1.12589,BOS_BULLISH,1.12577
2026-10-02 23:05:00+00:00,1.12564,1.12567,1.12535,1.12541,BOS_BEARISH,1.12548
2026-10-02 23:40:00+00:00,1.12508,1.12522,1.12495,1.12498,BOS_BEARISH,1.12501
2026-10-05 00:15:00+00:00,1.12569,1.12578,1.12558,1.12578,BOS_BULLISH,1.12553
2026-10-05 02:05:00+00:00,1.12569,1.12578,1.12554,1.12562,BOS_BEARISH,1.12563
2026-10-05 02:40:00+00:00,1.12534,1.12535,1.12518,1.12528,BOS_BEARISH,1.1253
2026-10-05 03:10:00+00:00,1.12535,1.12535,1.12477,1.12483,BOS_BEARISH,1.12509
2026-10-05 08:20:00+00:00,1.11848,1.11886,1.11831,1.11885,BOS_BULLISH,1.11869
2026-10-05 10:30:00+00:00,1.11906,1.11984,1.11891,1.11976,BOS_BULLISH,1.11946


In [15]:
from market_structure import (
    identify_structural_levels,
)

print("identify_structural_levels loaded:")
print(identify_structural_levels)

identify_structural_levels loaded:
<function identify_structural_levels at 0x79df3fff6b90>


In [16]:
structural_v1 = identify_structural_levels(
    minor_v2
)

print("structural_v1 shape:", structural_v1.shape)

print("\nStructural type counts:")
print(
    structural_v1["structural_type"]
    .value_counts(dropna=False)
)

print("\nProtected levels:")
display(
    structural_v1.loc[
        structural_v1["structural_type"].isin(
            ["PROTECTED_HIGH", "PROTECTED_LOW"]
        ),
        [
            "swing_type",
            "swing_price",
            "minor_structure",
            "structural_type",
        ],
    ].tail(20)
)

structural_v1 shape: (156, 23)

Structural type counts:
structural_type
None               89
STRUCTURAL_LOW     34
STRUCTURAL_HIGH    33
Name: count, dtype: int64

Protected levels:


,swing_type,swing_price,minor_structure,structural_type
time,,,,


In [17]:
print("minor_structure counts:")
print(
    minor_v2["minor_structure"]
    .value_counts(dropna=False)
)

print("\nstructural_type by minor_structure:")
print(
    pd.crosstab(
        minor_v2["minor_structure"],
        structural_v1["structural_type"],
        dropna=False,
    )
)

minor_structure counts:
minor_structure
HL      43
LH      43
LL      34
HH      33
None     3
Name: count, dtype: int64

structural_type by minor_structure:
structural_type  STRUCTURAL_HIGH  STRUCTURAL_LOW  NaN
minor_structure                                      
HH                            33               0    0
HL                             0               0   43
LH                             0               0   43
LL                             0              34    0
NaN                            0               0    3


In [18]:
display(
    minor_v2.loc[
        minor_v2["minor_structure"].isin(["HL", "LH"]),
        [
            "swing_type",
            "swing_price",
            "previous_swing_price",
            "minor_structure",
            "minor_rank",
            "minor_type",
        ],
    ].head(30)
)

,swing_type,swing_price,previous_swing_price,minor_structure,minor_rank,minor_type
time,,,,,,
2026-10-02 22:50:00+00:00,LOW,1.12548,1.12605,HL,1.895487,LOW
2026-10-02 23:00:00+00:00,HIGH,1.12594,1.12548,LH,1.487298,HIGH
2026-10-02 23:30:00+00:00,HIGH,1.12553,1.12501,LH,1.733333,HIGH
2026-10-05 01:30:00+00:00,LOW,1.12563,1.12598,HL,1.096197,LOW
2026-10-05 02:30:00+00:00,HIGH,1.12564,1.12530,LH,1.506329,HIGH
2026-10-05 03:00:00+00:00,HIGH,1.12561,1.12509,LH,2.333333,HIGH
2026-10-05 06:25:00+00:00,HIGH,1.11869,1.11607,LH,2.836814,HIGH
2026-10-05 07:05:00+00:00,LOW,1.11608,1.11869,HL,3.585868,LOW
2026-10-05 09:35:00+00:00,LOW,1.11689,1.11987,HL,5.241206,LOW


In [19]:
import inspect

print(inspect.getsource(identify_structural_levels))

def identify_structural_levels(minor_v2):
    """
    Identify Structural High/Low and Protected High/Low
    from Minor Structure v2.

    Required columns:
        - swing_type
        - swing_price
        - minor_structure

    Returns
    -------
    pd.DataFrame
        Same DataFrame with:
        - structural_type
        - structural_level
        - protected_type
        - protected_level
    """

    import pandas as pd

    required = {
        "swing_type",
        "swing_price",
        "minor_structure",
    }

    missing = required - set(minor_v2.columns)

    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}"
        )

    df = minor_v2.copy()

    # ---------------------------------------------------------
    # Output columns
    # ---------------------------------------------------------

    df["structural_type"] = None
    df["structural_level"] = None

    df["protected_type"] = None
    df["protected_level"] = None

In [20]:
structure_sequence = (
    minor_v2[
        minor_v2["minor_structure"].isin(
            ["HH", "HL", "LH", "LL"]
        )
    ][["swing_type", "swing_price", "minor_structure"]]
    .copy()
)

structure_sequence["next_structure"] = (
    structure_sequence["minor_structure"].shift(-1)
)

transition_counts = (
    structure_sequence
    .groupby(
        ["minor_structure", "next_structure"],
        dropna=False
    )
    .size()
    .sort_values(ascending=False)
)

print("STRUCTURE TRANSITIONS")
print("=" * 70)
print(transition_counts)

STRUCTURE TRANSITIONS
minor_structure  next_structure
LL               LH                27
HH               HL                26
LH               LL                26
HL               HH                25
LH               HL                17
HL               LH                16
HH               LL                 7
LL               HH                 7
HL               NaN                1
                 LL                 1
dtype: int64


In [21]:
protected_candidates = structure_sequence.loc[
    (
        (
            (structure_sequence["minor_structure"] == "HL") &
            (structure_sequence["next_structure"] == "HH")
        )
        |
        (
            (structure_sequence["minor_structure"] == "LH") &
            (structure_sequence["next_structure"] == "LL")
        )
    )
]

print("Protected candidates:", len(protected_candidates))

display(protected_candidates)

Protected candidates: 51


,swing_type,swing_price,minor_structure,next_structure
time,,,,
2026-10-02 23:00:00+00:00,HIGH,1.12594,LH,LL
2026-10-02 23:30:00+00:00,HIGH,1.12553,LH,LL
2026-10-05 01:30:00+00:00,LOW,1.12563,HL,HH
2026-10-05 02:30:00+00:00,HIGH,1.12564,LH,LL
2026-10-05 03:00:00+00:00,HIGH,1.12561,LH,LL
2026-10-05 07:05:00+00:00,LOW,1.11608,HL,HH
2026-10-05 10:20:00+00:00,LOW,1.11843,HL,HH
2026-10-05 11:05:00+00:00,LOW,1.11950,HL,HH
2026-10-05 13:05:00+00:00,HIGH,1.12095,LH,LL


In [22]:
display(
    minor_v2[
        [
            "swing_type",
            "swing_price",
            "minor_structure",
        ]
    ].head(40)
)

,swing_type,swing_price,minor_structure
time,,,
2026-10-02 21:05:00+00:00,HIGH,1.12577,None
2026-10-02 21:55:00+00:00,LOW,1.12480,None
2026-10-02 22:30:00+00:00,HIGH,1.12605,HH
2026-10-02 22:50:00+00:00,LOW,1.12548,HL
2026-10-02 23:00:00+00:00,HIGH,1.12594,LH
2026-10-02 23:20:00+00:00,LOW,1.12501,LL
2026-10-02 23:30:00+00:00,HIGH,1.12553,LH
2026-10-02 23:45:00+00:00,LOW,1.12487,LL
2026-10-05 01:10:00+00:00,HIGH,1.12598,HH


In [23]:
print("PROTECTED TYPE COUNTS")
print("=" * 60)

print(
    structural_v1["protected_type"]
    .value_counts(dropna=False)
)

print("\nProtected levels:")
display(
    structural_v1.loc[
        structural_v1["protected_type"].notna(),
        [
            "swing_type",
            "swing_price",
            "minor_structure",
            "protected_type",
            "protected_level",
        ],
    ]
)

PROTECTED TYPE COUNTS
protected_type
None              105
PROTECTED_HIGH     26
PROTECTED_LOW      25
Name: count, dtype: int64

Protected levels:


,swing_type,swing_price,minor_structure,protected_type,protected_level
time,,,,,
2026-10-02 23:00:00+00:00,HIGH,1.12594,LH,PROTECTED_HIGH,1.12594
2026-10-02 23:30:00+00:00,HIGH,1.12553,LH,PROTECTED_HIGH,1.12553
2026-10-05 01:30:00+00:00,LOW,1.12563,HL,PROTECTED_LOW,1.12563
2026-10-05 02:30:00+00:00,HIGH,1.12564,LH,PROTECTED_HIGH,1.12564
2026-10-05 03:00:00+00:00,HIGH,1.12561,LH,PROTECTED_HIGH,1.12561
2026-10-05 07:05:00+00:00,LOW,1.11608,HL,PROTECTED_LOW,1.11608
2026-10-05 10:20:00+00:00,LOW,1.11843,HL,PROTECTED_LOW,1.11843
2026-10-05 11:05:00+00:00,LOW,1.11950,HL,PROTECTED_LOW,1.1195
2026-10-05 13:05:00+00:00,HIGH,1.12095,LH,PROTECTED_HIGH,1.12095


In [24]:
expected_protected_low = (
    structure_sequence["minor_structure"].eq("HL")
    & structure_sequence["next_structure"].eq("HH")
).sum()

expected_protected_high = (
    structure_sequence["minor_structure"].eq("LH")
    & structure_sequence["next_structure"].eq("LL")
).sum()

actual_protected_low = (
    structural_v1["protected_type"] == "PROTECTED_LOW"
).sum()

actual_protected_high = (
    structural_v1["protected_type"] == "PROTECTED_HIGH"
).sum()

print("Expected Protected Low :", expected_protected_low)
print("Actual Protected Low   :", actual_protected_low)

print("Expected Protected High:", expected_protected_high)
print("Actual Protected High  :", actual_protected_high)

Expected Protected Low : 25
Actual Protected Low   : 25
Expected Protected High: 26
Actual Protected High  : 26


In [25]:
print("=" * 100)
print("PROTECTED LEVELS — STRUCTURAL SOURCE AUDIT")
print("=" * 100)

protected_source = structural_v1.loc[
    structural_v1["protected_type"].notna(),
    [
        "swing_type",
        "swing_price",
        "minor_structure",
        "protected_type",
        "protected_level",
    ],
].copy()

print("Total Protected Levels :", len(protected_source))
print()

print("Protected Type Counts:")
print(
    protected_source["protected_type"]
    .value_counts()
)

print()
print("Protected Level Null Check:")
print(
    protected_source[
        ["protected_type", "protected_level"]
    ].isna().sum()
)

print()
print("Duplicate Protected Timestamps:")
print(
    protected_source.index.duplicated().sum()
)

print()
display(protected_source)

PROTECTED LEVELS — STRUCTURAL SOURCE AUDIT
Total Protected Levels : 51

Protected Type Counts:
protected_type
PROTECTED_HIGH    26
PROTECTED_LOW     25
Name: count, dtype: int64

Protected Level Null Check:
protected_type     0
protected_level    0
dtype: int64

Duplicate Protected Timestamps:
0



,swing_type,swing_price,minor_structure,protected_type,protected_level
time,,,,,
2026-10-02 23:00:00+00:00,HIGH,1.12594,LH,PROTECTED_HIGH,1.12594
2026-10-02 23:30:00+00:00,HIGH,1.12553,LH,PROTECTED_HIGH,1.12553
2026-10-05 01:30:00+00:00,LOW,1.12563,HL,PROTECTED_LOW,1.12563
2026-10-05 02:30:00+00:00,HIGH,1.12564,LH,PROTECTED_HIGH,1.12564
2026-10-05 03:00:00+00:00,HIGH,1.12561,LH,PROTECTED_HIGH,1.12561
2026-10-05 07:05:00+00:00,LOW,1.11608,HL,PROTECTED_LOW,1.11608
2026-10-05 10:20:00+00:00,LOW,1.11843,HL,PROTECTED_LOW,1.11843
2026-10-05 11:05:00+00:00,LOW,1.11950,HL,PROTECTED_LOW,1.1195
2026-10-05 13:05:00+00:00,HIGH,1.12095,LH,PROTECTED_HIGH,1.12095


In [26]:
def build_protected_registry_v1(
    structural_df,
    m5_df,
    break_mode="close",
):
    """
    Protected Registry v1

    Identity:
        PH_### -> PROTECTED_HIGH
        PL_### -> PROTECTED_LOW

    Lifecycle:
        ACTIVE
        BROKEN

    Break contract:
        PROTECTED_HIGH -> close/high ABOVE level
        PROTECTED_LOW  -> close/low BELOW level

    Only candles strictly after created_at are eligible.
    """

    required_structural = {
        "protected_type",
        "protected_level",
    }

    missing = required_structural - set(structural_df.columns)

    if missing:
        raise ValueError(
            f"Missing structural columns: {sorted(missing)}"
        )

    required_m5 = {
        "open",
        "high",
        "low",
        "close",
    }

    missing_m5 = required_m5 - set(m5_df.columns)

    if missing_m5:
        raise ValueError(
            f"Missing M5 columns: {sorted(missing_m5)}"
        )

    if break_mode not in {"close", "wick"}:
        raise ValueError(
            "break_mode must be 'close' or 'wick'"
        )

    protected = structural_df.loc[
        structural_df["protected_type"].isin(
            ["PROTECTED_HIGH", "PROTECTED_LOW"]
        )
        & structural_df["protected_level"].notna()
    ].copy()

    protected = protected.sort_index()

    records = []

    high_counter = 0
    low_counter = 0

    for created_at, row in protected.iterrows():

        protected_type = row["protected_type"]
        level = float(row["protected_level"])

        if protected_type == "PROTECTED_HIGH":
            high_counter += 1
            level_id = f"PH_{high_counter:03d}"
            break_direction = "BOS_BULLISH"

        else:
            low_counter += 1
            level_id = f"PL_{low_counter:03d}"
            break_direction = "BOS_BEARISH"

        # STRICT NO LOOK-AHEAD:
        # creation candle itself is excluded.
        future = m5_df.loc[
            m5_df.index > created_at
        ]

        break_time = pd.NaT
        break_price = float("nan")
        state = "ACTIVE"

        for candle_time, candle in future.iterrows():

            if protected_type == "PROTECTED_HIGH":

                if break_mode == "close":
                    broken = candle["close"] > level
                    event_price = candle["close"]
                else:
                    broken = candle["high"] > level
                    event_price = candle["high"]

            else:

                if break_mode == "close":
                    broken = candle["close"] < level
                    event_price = candle["close"]
                else:
                    broken = candle["low"] < level
                    event_price = candle["low"]

            if broken:
                break_time = candle_time
                break_price = float(event_price)
                state = "BROKEN"
                break

        records.append(
            {
                "level_id": level_id,
                "type": protected_type,
                "level": level,
                "created_at": created_at,
                "state": state,
                "break_time": break_time,
                "break_price": break_price,
                "break_direction": (
                    break_direction
                    if state == "BROKEN"
                    else None
                ),
            }
        )

    registry = pd.DataFrame(records)

    if not registry.empty:

        registry["created_at"] = pd.to_datetime(
            registry["created_at"],
            utc=True,
        )

        registry["break_time"] = pd.to_datetime(
            registry["break_time"],
            utc=True,
        )

        registry = (
            registry
            .sort_values("created_at")
            .reset_index(drop=True)
        )

    return registry

In [27]:
protected_registry_v1 = build_protected_registry_v1(
    structural_df=structural_v1,
    m5_df=m5_df,
    break_mode="close",
)

print("=" * 100)
print("PROTECTED REGISTRY V1 — CORRECTED")
print("=" * 100)

print("Total Registry Rows :", len(protected_registry_v1))

print("\nType:")
print(
    protected_registry_v1["type"]
    .value_counts()
)

print("\nState:")
print(
    protected_registry_v1["state"]
    .value_counts()
)

print("\nBreak Direction:")
print(
    protected_registry_v1["break_direction"]
    .value_counts(dropna=False)
)

print("\nID Prefix:")
print(
    protected_registry_v1["level_id"]
    .str[:2]
    .value_counts()
)

display(protected_registry_v1)

PROTECTED REGISTRY V1 — CORRECTED
Total Registry Rows : 51

Type:
type
PROTECTED_HIGH    26
PROTECTED_LOW     25
Name: count, dtype: int64

State:
state
BROKEN    39
ACTIVE    12
Name: count, dtype: int64

Break Direction:
break_direction
BOS_BULLISH    21
BOS_BEARISH    18
NaN            12
Name: count, dtype: int64

ID Prefix:
level_id
PH    26
PL    25
Name: count, dtype: int64


,level_id,type,level,created_at,state,break_time,break_price,break_direction
0,PH_001,PROTECTED_HIGH,1.12594,2026-10-02 23:00:00+00:00,BROKEN,2026-10-05 01:45:00+00:00,1.12595,BOS_BULLISH
1,PH_002,PROTECTED_HIGH,1.12553,2026-10-02 23:30:00+00:00,BROKEN,2026-10-05 00:15:00+00:00,1.12578,BOS_BULLISH
2,PL_001,PROTECTED_LOW,1.12563,2026-10-05 01:30:00+00:00,BROKEN,2026-10-05 02:05:00+00:00,1.12562,BOS_BEARISH
3,PH_003,PROTECTED_HIGH,1.12564,2026-10-05 02:30:00+00:00,BROKEN,2026-10-06 14:20:00+00:00,1.12590,BOS_BULLISH
4,PH_004,PROTECTED_HIGH,1.12561,2026-10-05 03:00:00+00:00,BROKEN,2026-10-06 14:20:00+00:00,1.12590,BOS_BULLISH
5,PL_002,PROTECTED_LOW,1.11608,2026-10-05 07:05:00+00:00,ACTIVE,NaT,NaN,NaN
6,PL_003,PROTECTED_LOW,1.11843,2026-10-05 10:20:00+00:00,BROKEN,2026-10-07 14:35:00+00:00,1.11801,BOS_BEARISH
7,PL_004,PROTECTED_LOW,1.11950,2026-10-05 11:05:00+00:00,BROKEN,2026-10-05 16:35:00+00:00,1.11936,BOS_BEARISH
8,PH_005,PROTECTED_HIGH,1.12095,2026-10-05 13:05:00+00:00,BROKEN,2026-10-05 14:25:00+00:00,1.12139,BOS_BULLISH
9,PH_006,PROTECTED_HIGH,1.12086,2026-10-05 13:25:00+00:00,BROKEN,2026-10-05 14:25:00+00:00,1.12139,BOS_BULLISH


In [28]:
print("=" * 100)
print("PROTECTED REGISTRY V1 — BREAK CONTRACT AUDIT")
print("=" * 100)

ph = protected_registry_v1[
    protected_registry_v1["type"] == "PROTECTED_HIGH"
]

pl = protected_registry_v1[
    protected_registry_v1["type"] == "PROTECTED_LOW"
]

ph_direction_ok = (
    ph.loc[
        ph["state"] == "BROKEN",
        "break_direction"
    ].eq("BOS_BULLISH").all()
)

pl_direction_ok = (
    pl.loc[
        pl["state"] == "BROKEN",
        "break_direction"
    ].eq("BOS_BEARISH").all()
)

print(
    "PROTECTED_HIGH → BOS_BULLISH :",
    "PASSED" if ph_direction_ok else "FAILED"
)

print(
    "PROTECTED_LOW  → BOS_BEARISH :",
    "PASSED" if pl_direction_ok else "FAILED"
)

print(
    "\nPH count:",
    len(ph),
    "| PL count:",
    len(pl),
)

PROTECTED REGISTRY V1 — BREAK CONTRACT AUDIT
PROTECTED_HIGH → BOS_BULLISH : PASSED
PROTECTED_LOW  → BOS_BEARISH : PASSED

PH count: 26 | PL count: 25


In [29]:
print("=" * 100)
print("PROTECTED REGISTRY V1 — BREAK DIRECTION CONTRACT")
print("=" * 100)

broken = protected_registry_v1[
    protected_registry_v1["state"] == "BROKEN"
].copy()

broken_ph = broken[
    broken["type"] == "PROTECTED_HIGH"
]

broken_pl = broken[
    broken["type"] == "PROTECTED_LOW"
]

ph_direction_ok = (
    broken_ph["break_direction"]
    .eq("BOS_BULLISH")
    .all()
)

pl_direction_ok = (
    broken_pl["break_direction"]
    .eq("BOS_BEARISH")
    .all()
)

print(
    "BROKEN PH → BOS_BULLISH :",
    "PASSED" if ph_direction_ok else "FAILED"
)

print(
    "BROKEN PL → BOS_BEARISH :",
    "PASSED" if pl_direction_ok else "FAILED"
)

print()
print("Broken PH :", len(broken_ph))
print("Broken PL :", len(broken_pl))

direction_ok = (
    ph_direction_ok
    and pl_direction_ok
)

print()
print(
    "BREAK DIRECTION CONTRACT :",
    "PASSED" if direction_ok else "FAILED"
)

PROTECTED REGISTRY V1 — BREAK DIRECTION CONTRACT
BROKEN PH → BOS_BULLISH : PASSED
BROKEN PL → BOS_BEARISH : PASSED

Broken PH : 21
Broken PL : 18

BREAK DIRECTION CONTRACT : PASSED


In [30]:
print("=" * 100)
print("PROTECTED REGISTRY V1 — FINAL REGRESSION GATE")
print("=" * 100)

r = protected_registry_v1

checks = {}

# ==================================================
# 1. Required columns
# ==================================================

required_columns = {
    "level_id",
    "type",
    "level",
    "created_at",
    "state",
    "break_time",
    "break_price",
    "break_direction",
}

checks["REQUIRED COLUMNS"] = (
    required_columns.issubset(r.columns)
)


# ==================================================
# 2. Row count
# ==================================================

checks["ROW COUNT"] = len(r) == 44


# ==================================================
# 3. ID uniqueness
# ==================================================

checks["LEVEL ID UNIQUENESS"] = r["level_id"].is_unique


# ==================================================
# 4. ID identity
# ==================================================

checks["PH IDENTITY"] = (
    r.loc[
        r["type"] == "PROTECTED_HIGH",
        "level_id",
    ]
    .str.startswith("PH_")
    .all()
)

checks["PL IDENTITY"] = (
    r.loc[
        r["type"] == "PROTECTED_LOW",
        "level_id",
    ]
    .str.startswith("PL_")
    .all()
)


# ==================================================
# 5. Expected type counts
# ==================================================

checks["PH COUNT"] = (
    (r["type"] == "PROTECTED_HIGH").sum() == 27
)

checks["PL COUNT"] = (
    (r["type"] == "PROTECTED_LOW").sum() == 17
)


# ==================================================
# 6. Level identity preservation
# ==================================================

source_ph = sorted(
    structural_v1.loc[
        structural_v1["protected_type"]
        == "PROTECTED_HIGH",
        "protected_level",
    ]
    .astype(float)
    .tolist()
)

source_pl = sorted(
    structural_v1.loc[
        structural_v1["protected_type"]
        == "PROTECTED_LOW",
        "protected_level",
    ]
    .astype(float)
    .tolist()
)

registry_ph = sorted(
    r.loc[
        r["type"] == "PROTECTED_HIGH",
        "level",
    ]
    .astype(float)
    .tolist()
)

registry_pl = sorted(
    r.loc[
        r["type"] == "PROTECTED_LOW",
        "level",
    ]
    .astype(float)
    .tolist()
)

checks["LEVEL IDENTITY PRESERVATION"] = (
    source_ph == registry_ph
    and
    source_pl == registry_pl
)


# ==================================================
# 7. State contract
# ==================================================

checks["LIFECYCLE STATE CONTRACT"] = (
    set(r["state"].dropna().unique())
    <= {"ACTIVE", "BROKEN"}
)


# ==================================================
# 8. ACTIVE contract
# ==================================================

active = r[r["state"] == "ACTIVE"]

checks["ACTIVE CONTRACT"] = (
    active["break_time"].isna().all()
    and
    active["break_price"].isna().all()
    and
    active["break_direction"].isna().all()
)


# ==================================================
# 9. BROKEN contract
# ==================================================

broken = r[r["state"] == "BROKEN"]

checks["BROKEN CONTRACT"] = (
    broken["break_time"].notna().all()
    and
    broken["break_price"].notna().all()
    and
    broken["break_direction"].notna().all()
)


# ==================================================
# 10. UTC-aware timestamps
# ==================================================

checks["CREATED_AT UTC-AWARE"] = isinstance(
    r["created_at"].dtype,
    pd.DatetimeTZDtype,
)

checks["BREAK_TIME UTC-AWARE"] = isinstance(
    r["break_time"].dtype,
    pd.DatetimeTZDtype,
)


# ==================================================
# 11. Break occurs after creation
# ==================================================

checks["BREAK TIME > CREATED AT"] = (
    broken["break_time"]
    > broken["created_at"]
).all()


# ==================================================
# 12. Break event exists in M5
# ==================================================

m5_index = set(m5_df.index)

checks["BREAK TIME EXISTS IN M5"] = all(
    ts in m5_index
    for ts in broken["break_time"]
)


# ==================================================
# 13. No look-ahead
# ==================================================

checks["NO LOOK-AHEAD"] = (
    r["break_time"].isna()
    |
    (r["break_time"] > r["created_at"])
).all()


# ==================================================
# 14. Break price contract
# ==================================================

break_price_ok = True

for _, row in broken.iterrows():

    candle = m5_df.loc[row["break_time"]]

    if row["type"] == "PROTECTED_HIGH":

        if not (
            row["break_price"] == candle["close"]
            and candle["close"] > row["level"]
        ):
            break_price_ok = False
            break

    elif row["type"] == "PROTECTED_LOW":

        if not (
            row["break_price"] == candle["close"]
            and candle["close"] < row["level"]
        ):
            break_price_ok = False
            break

checks["BREAK PRICE CONTRACT"] = break_price_ok


# ==================================================
# 15. Break direction contract
# ==================================================

checks["BREAK DIRECTION CONTRACT"] = (
    broken.loc[
        broken["type"] == "PROTECTED_HIGH",
        "break_direction",
    ]
    .eq("BOS_BULLISH")
    .all()
    and
    broken.loc[
        broken["type"] == "PROTECTED_LOW",
        "break_direction",
    ]
    .eq("BOS_BEARISH")
    .all()
)


# ==================================================
# 16. Created timestamps unique
# ==================================================

checks["CREATED_AT UNIQUENESS"] = (
    r["created_at"].is_unique
)


# ==================================================
# 17. ID ↔ type immutability
# ==================================================

checks["ID TYPE IMMUTABILITY"] = (
    r.groupby("level_id")["type"]
    .nunique()
    .max()
    == 1
)


# ==================================================
# 18. ID ↔ level immutability
# ==================================================

checks["ID LEVEL IMMUTABILITY"] = (
    r.groupby("level_id")["level"]
    .nunique()
    .max()
    == 1
)


# ==================================================
# PRINT RESULTS
# ==================================================

for name, passed in checks.items():

    print(
        f"{name:<35}:",
        "PASSED" if passed else "FAILED"
    )


# ==================================================
# FINAL
# ==================================================

print()
print("=" * 100)

if all(checks.values()):
    print(
        "FINAL RESULT : "
        "✅ PROTECTED REGISTRY V1 PASSED"
    )
else:
    print(
        "FINAL RESULT : "
        "❌ PROTECTED REGISTRY V1 FAILED"
    )

print("=" * 100)

PROTECTED REGISTRY V1 — FINAL REGRESSION GATE
REQUIRED COLUMNS                   : PASSED
ROW COUNT                          : FAILED
LEVEL ID UNIQUENESS                : PASSED
PH IDENTITY                        : PASSED
PL IDENTITY                        : PASSED
PH COUNT                           : FAILED
PL COUNT                           : FAILED
LEVEL IDENTITY PRESERVATION        : PASSED
LIFECYCLE STATE CONTRACT           : PASSED
ACTIVE CONTRACT                    : PASSED
BROKEN CONTRACT                    : PASSED
CREATED_AT UTC-AWARE               : PASSED
BREAK_TIME UTC-AWARE               : PASSED
BREAK TIME > CREATED AT            : PASSED
BREAK TIME EXISTS IN M5            : PASSED
NO LOOK-AHEAD                      : PASSED
BREAK PRICE CONTRACT               : PASSED
BREAK DIRECTION CONTRACT           : PASSED
CREATED_AT UNIQUENESS              : PASSED
ID TYPE IMMUTABILITY               : PASSED
ID LEVEL IMMUTABILITY              : PASSED

FINAL RESULT : ❌ PROTECTED RE

In [31]:
# df_bos = detect_minor_bos(
#     df,
#     minor_v2,
#     break_mode="close"
# )

In [32]:
# print("df exists:", "df" in globals())
# print("minor_v2 exists:", "minor_v2" in globals())
# print("detect_minor_bos exists:", "detect_minor_bos" in globals())

In [33]:
# print("Index name:", minor_v2.index.name)
# print("Columns:")
# print(minor_v2.columns.tolist())

In [34]:
# df_bos = detect_minor_bos(
#     df,
#     minor_v2,
#     break_mode="close"
# )

In [35]:
# # ==========================================
# # Run Minor BOS
# # ==========================================

# df_bos = detect_minor_bos(
#     df,
#     minor_v2,
#     break_mode="close"
# )

# # فقط کندل‌هایی که BOS دارند
# bos_events = df_bos[
#     df_bos["minor_bos"].notna()
# ]

# print("=" * 60)
# print("Minor BOS")
# print("=" * 60)

# print("Total BOS:", len(bos_events))

# print("\nLast 30 BOS Events:")

# print(
#     bos_events[
#         [
#             "high",
#             "low",
#             "close",
#             "minor_bos",
#             "bos_level"
#         ]
#     ].tail(30)
# )

In [36]:
print("bos_events exists :", "bos_events" in globals())

if "bos_events" in globals():
    print("shape :", bos_events.shape)
    print("columns :")
    print(bos_events.columns.tolist())

bos_events exists : False


In [37]:
def detect_structural_bos_v2(
    m5_df,
    protected_registry,
    break_mode="close",
):
    """
    Structural BOS v2

    Canonical source:
        Protected Registry v1

    Break contract:
        PROTECTED_HIGH -> BOS_BULLISH
        PROTECTED_LOW  -> BOS_BEARISH

    Only BROKEN registry levels generate BOS.
    No look-ahead.
    One BOS event per protected level.
    """

    import pandas as pd

    required_registry = {
        "level_id",
        "type",
        "level",
        "created_at",
        "state",
    }

    missing = required_registry - set(protected_registry.columns)

    if missing:
        raise ValueError(
            f"Missing registry columns: {sorted(missing)}"
        )

    required_m5 = {
        "open",
        "high",
        "low",
        "close",
    }

    missing_m5 = required_m5 - set(m5_df.columns)

    if missing_m5:
        raise ValueError(
            f"Missing M5 columns: {sorted(missing_m5)}"
        )

    if break_mode not in {"close", "wick"}:
        raise ValueError(
            "break_mode must be 'close' or 'wick'"
        )

    records = []

    for _, row in protected_registry.iterrows():

        level_id = row["level_id"]
        level_type = row["type"]
        level = float(row["level"])
        created_at = row["created_at"]

        # Only BROKEN levels can produce BOS
        if row["state"] != "BROKEN":
            continue

        # No look-ahead
        future = m5_df.loc[
            m5_df.index > created_at
        ]

        bos_time = pd.NaT
        bos_price = float("nan")
        bos_direction = None

        for candle_time, candle in future.iterrows():

            if level_type == "PROTECTED_HIGH":

                if break_mode == "close":
                    broken = candle["close"] > level
                    event_price = candle["close"]
                else:
                    broken = candle["high"] > level
                    event_price = candle["high"]

                direction = "BOS_BULLISH"

            elif level_type == "PROTECTED_LOW":

                if break_mode == "close":
                    broken = candle["close"] < level
                    event_price = candle["close"]
                else:
                    broken = candle["low"] < level
                    event_price = candle["low"]

                direction = "BOS_BEARISH"

            else:
                continue

            if broken:
                bos_time = candle_time
                bos_price = float(event_price)
                bos_direction = direction
                break

        if pd.isna(bos_time):
            raise RuntimeError(
                f"Registry inconsistency: "
                f"{level_id} is BROKEN but no BOS event was found."
            )

        records.append(
            {
                "bos_id": f"BOS_{level_id}",
                "level_id": level_id,
                "bos_direction": bos_direction,
                "bos_level": level,
                "bos_time": bos_time,
                "bos_price": bos_price,
                "created_at": created_at,
                "source_type": level_type,
            }
        )

    result = pd.DataFrame(records)

    if result.empty:
        return result

    result["bos_time"] = pd.to_datetime(
        result["bos_time"],
        utc=True,
    )

    result["created_at"] = pd.to_datetime(
        result["created_at"],
        utc=True,
    )

    result = (
        result
        .sort_values("bos_time")
        .reset_index(drop=True)
    )

    return result

In [38]:
structural_bos_v2 = detect_structural_bos_v2(
    m5_df=m5_df,
    protected_registry=protected_registry_v1,
    break_mode="close",
)

print("structural_bos_v2 shape :", structural_bos_v2.shape)
print()
print(
    structural_bos_v2["bos_direction"]
    .value_counts()
)

structural_bos_v2 shape : (39, 8)

bos_direction
BOS_BULLISH    21
BOS_BEARISH    18
Name: count, dtype: int64


In [39]:
print("\n" + "=" * 80)
print("STRUCTURAL BOS V2 — LEVEL FREQUENCY")
print("=" * 80)

bos_frequency = (
    structural_bos_v2.loc[
        structural_bos_v2["bos_id"].notna(),
        [
            "bos_id",
            "level_id",
            "bos_level",
            "bos_direction",
        ],
    ]
    .groupby(
        [
            "bos_id",
            "level_id",
            "bos_level",
            "bos_direction",
        ],
        dropna=False,
    )
    .size()
    .sort_values(ascending=False)
    .head(20)
)

print(bos_frequency)


STRUCTURAL BOS V2 — LEVEL FREQUENCY
bos_id      level_id  bos_level  bos_direction
BOS_PH_001  PH_001    1.12594    BOS_BULLISH      1
BOS_PH_002  PH_002    1.12553    BOS_BULLISH      1
BOS_PH_003  PH_003    1.12564    BOS_BULLISH      1
BOS_PH_004  PH_004    1.12561    BOS_BULLISH      1
BOS_PH_005  PH_005    1.12095    BOS_BULLISH      1
BOS_PH_006  PH_006    1.12086    BOS_BULLISH      1
BOS_PH_007  PH_007    1.12109    BOS_BULLISH      1
BOS_PH_008  PH_008    1.12138    BOS_BULLISH      1
BOS_PH_009  PH_009    1.11981    BOS_BULLISH      1
BOS_PH_010  PH_010    1.12127    BOS_BULLISH      1
BOS_PH_011  PH_011    1.12059    BOS_BULLISH      1
BOS_PH_012  PH_012    1.12173    BOS_BULLISH      1
BOS_PH_013  PH_013    1.12216    BOS_BULLISH      1
BOS_PH_014  PH_014    1.12200    BOS_BULLISH      1
BOS_PH_015  PH_015    1.12664    BOS_BULLISH      1
BOS_PH_016  PH_016    1.12571    BOS_BULLISH      1
BOS_PH_017  PH_017    1.12602    BOS_BULLISH      1
BOS_PH_020  PH_020    1.12589   

In [40]:
print("\n" + "=" * 80)
print("STRUCTURAL BOS V2 — IDENTITY CHECK")
print("=" * 80)

level_frequency = (
    structural_bos_v2
    .groupby("level_id")
    .size()
)

print("Total BOS events :", len(structural_bos_v2))
print("Unique levels    :", level_frequency.index.nunique())
print("Max events/level :", level_frequency.max())
print("Min events/level :", level_frequency.min())

print()

if (
    len(structural_bos_v2) == 36
    and level_frequency.index.nunique() == 36
    and level_frequency.max() == 1
    and level_frequency.min() == 1
):
    print("ONE EVENT PER LEVEL : PASSED")
else:
    print("ONE EVENT PER LEVEL : FAILED")


STRUCTURAL BOS V2 — IDENTITY CHECK
Total BOS events : 39
Unique levels    : 39
Max events/level : 1
Min events/level : 1

ONE EVENT PER LEVEL : FAILED


In [41]:
# def identify_structural_levels(minor_v2):
#     """
#     Identify Structural High/Low and Protected High/Low
#     from Minor Structure v2.
#     """

#     required = {
#         "swing_type",
#         "swing_price",
#         "minor_structure",
#     }

#     missing = required - set(minor_v2.columns)

#     if missing:
#         raise ValueError(
#             f"Missing required columns: {sorted(missing)}"
#         )

#     df = minor_v2.copy()

#     # ---------------------------------------------------------
#     # Output columns
#     # ---------------------------------------------------------

#     df["structural_type"] = None
#     df["structural_level"] = None

#     df["protected_type"] = None
#     df["protected_level"] = None

#     # ---------------------------------------------------------
#     # Structural High / Low
#     # ---------------------------------------------------------

#     for idx, row in df.iterrows():

#         structure = row["minor_structure"]
#         price = row["swing_price"]

#         # HH -> Structural High
#         if structure == "HH":

#             df.at[idx, "structural_type"] = "STRUCTURAL_HIGH"
#             df.at[idx, "structural_level"] = price

#         # LL -> Structural Low
#         elif structure == "LL":

#             df.at[idx, "structural_type"] = "STRUCTURAL_LOW"
#             df.at[idx, "structural_level"] = price

#     # ---------------------------------------------------------
#     # Protected High / Low
#     #
#     # HL -> HH  => Protected Low
#     # LH -> LL  => Protected High
#     # ---------------------------------------------------------

#     swing_indices = df.index[
#         df["minor_structure"].isin(["HH", "HL", "LH", "LL"])
#     ].tolist()

#     for i in range(1, len(swing_indices)):

#         previous_idx = swing_indices[i - 1]
#         current_idx = swing_indices[i]

#         previous_structure = df.at[
#             previous_idx,
#             "minor_structure"
#         ]

#         current_structure = df.at[
#             current_idx,
#             "minor_structure"
#         ]

#         # HL -> HH
#         if (
#             previous_structure == "HL"
#             and current_structure == "HH"
#         ):

#             df.at[
#                 previous_idx,
#                 "protected_type"
#             ] = "PROTECTED_LOW"

#             df.at[
#                 previous_idx,
#                 "protected_level"
#             ] = df.at[
#                 previous_idx,
#                 "swing_price"
#             ]

#         # LH -> LL
#         elif (
#             previous_structure == "LH"
#             and current_structure == "LL"
#         ):

#             df.at[
#                 previous_idx,
#                 "protected_type"
#             ] = "PROTECTED_HIGH"

#             df.at[
#                 previous_idx,
#                 "protected_level"
#             ] = df.at[
#                 previous_idx,
#                 "swing_price"
#             ]

#     return df

In [42]:
# def detect_structural_bos_v2(
#     structural_v2,
#     protected_registry_v4,
#     break_mode="close"
# ):
#     """
#     Structural BOS v2

#     BOS is detected directly from OHLC after a Protected
#     Level has been created.

#     Bullish BOS:
#         close > Protected High

#     Bearish BOS:
#         close < Protected Low

#     Each Protected Level can generate BOS only once.
#     """

#     required_df = {
#         "high",
#         "low",
#         "close",
#     }

#     missing_df = required_df - set(structural_v2.columns)

#     if missing_df:
#         raise ValueError(
#             f"Missing required columns in structural_v2: "
#             f"{sorted(missing_df)}"
#         )

#     required_registry = {
#         "level_id",
#         "type",
#         "level",
#         "created_at",
#     }

#     missing_registry = (
#         required_registry
#         - set(protected_registry_v4.columns)
#     )

#     if missing_registry:
#         raise ValueError(
#             f"Missing required columns in protected_registry_v4: "
#             f"{sorted(missing_registry)}"
#         )

#     if break_mode not in {"close", "wick"}:
#         raise ValueError(
#             "break_mode must be 'close' or 'wick'"
#         )

#     df = structural_v2.copy()

#     df["structural_bos"] = None
#     df["bos_level"] = None
#     df["bos_direction"] = None
#     df["broken_level_id"] = None
#     df["broken_level_type"] = None

#     # --------------------------------------------------
#     # Build level list
#     # --------------------------------------------------

#     levels = []

#     for _, row in protected_registry_v4.iterrows():

#         levels.append({
#             "level_id": row["level_id"],
#             "type": row["type"],
#             "level": row["level"],
#             "created_at": row["created_at"],
#             "broken": False,
#         })

#     # --------------------------------------------------
#     # Scan candles chronologically
#     # --------------------------------------------------

#     for idx, row in df.iterrows():

#         high = row["high"]
#         low = row["low"]
#         close = row["close"]

#         bullish_candidates = []
#         bearish_candidates = []

#         for level in levels:

#             # Level does not exist yet
#             if idx < level["created_at"]:
#                 continue

#             # Already broken
#             if level["broken"]:
#                 continue

#             # ------------------------------------------
#             # Protected High → Bullish BOS
#             # ------------------------------------------

#             if level["type"] == "PROTECTED_HIGH":

#                 if break_mode == "close":
#                     broken = close > level["level"]
#                 else:
#                     broken = high > level["level"]

#                 if broken:
#                     bullish_candidates.append(level)

#             # ------------------------------------------
#             # Protected Low → Bearish BOS
#             # ------------------------------------------

#             elif level["type"] == "PROTECTED_LOW":

#                 if break_mode == "close":
#                     broken = close < level["level"]
#                 else:
#                     broken = low < level["level"]

#                 if broken:
#                     bearish_candidates.append(level)

#         # ------------------------------------------------
#         # Bullish BOS
#         # ------------------------------------------------

#         if bullish_candidates:

#             broken = max(
#                 bullish_candidates,
#                 key=lambda x: x["created_at"]
#             )

#             df.at[idx, "structural_bos"] = "BOS_BULLISH"
#             df.at[idx, "bos_level"] = broken["level"]
#             df.at[idx, "bos_direction"] = "BULLISH"
#             df.at[idx, "broken_level_id"] = broken["level_id"]
#             df.at[idx, "broken_level_type"] = broken["type"]

#             broken["broken"] = True

#         # ------------------------------------------------
#         # Bearish BOS
#         # ------------------------------------------------

#         if bearish_candidates:

#             broken = max(
#                 bearish_candidates,
#                 key=lambda x: x["created_at"]
#             )

#             df.at[idx, "structural_bos"] = "BOS_BEARISH"
#             df.at[idx, "bos_level"] = broken["level"]
#             df.at[idx, "bos_direction"] = "BEARISH"
#             df.at[idx, "broken_level_id"] = broken["level_id"]
#             df.at[idx, "broken_level_type"] = broken["type"]

#             broken["broken"] = True

#     return df

In [43]:
# def attach_protected_level_ids(structural_v1, protected_registry_v4):
#     """
#     Attach Protected Level ID to structural_v1.
#     """

#     df = structural_v1.copy()

#     df["protected_level_id"] = None

#     for _, level in protected_registry_v4.iterrows():

#         created_at = level["created_at"]

#         if created_at in df.index:
#             df.at[
#                 created_at,
#                 "protected_level_id"
#             ] = level["level_id"]

#     return df

In [44]:
# structural_v1 = identify_structural_levels(
#     minor_v2
# )

# print("=" * 90)
# print("Structural v1")
# print("=" * 90)

# print(
#     structural_v1[
#         structural_v1["structural_type"].notna()
#         |
#         structural_v1["protected_type"].notna()
#     ][
#         [
#             "swing_type",
#             "swing_price",
#             "minor_structure",
#             "structural_type",
#             "structural_level",
#             "protected_type",
#             "protected_level",
#         ]
#     ].to_string()
# )

In [45]:
def build_protected_level_registry_v4(structural_v1):
    """
    Protected Level Registry v4

    Tracks the complete lifecycle of each Protected High/Low.

    States:
        ACTIVE
        BROKEN
    """

    required = {
        "high",
        "low",
        "close",
        "protected_type",
        "protected_level",
    }

    missing = required - set(structural_v1.columns)

    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}"
        )

    df = structural_v1.copy()

    registry = []

    active_high = None
    active_low = None

    high_id = 0
    low_id = 0

    for idx, row in df.iterrows():

        close = row["close"]

        # =====================================================
        # 1. Check Protected High
        # =====================================================

        if active_high is not None:

            if close > active_high["level"]:

                active_high["state"] = "BROKEN"
                active_high["break_time"] = idx
                active_high["break_price"] = close
                active_high["break_direction"] = "BULLISH"

                registry.append(active_high)

                active_high = None

        # =====================================================
        # 2. Check Protected Low
        # =====================================================

        if active_low is not None:

            if close < active_low["level"]:

                active_low["state"] = "BROKEN"
                active_low["break_time"] = idx
                active_low["break_price"] = close
                active_low["break_direction"] = "BEARISH"

                registry.append(active_low)

                active_low = None

        # =====================================================
        # 3. Register Protected High
        # =====================================================

        if row["protected_type"] == "PROTECTED_HIGH":

            high_id += 1

            active_high = {
                "level_id": f"PH_{high_id:03d}",
                "type": "PROTECTED_HIGH",
                "level": row["protected_level"],
                "created_at": idx,
                "state": "ACTIVE",
                "break_time": None,
                "break_price": None,
                "break_direction": None,
            }

        # =====================================================
        # 4. Register Protected Low
        # =====================================================

        elif row["protected_type"] == "PROTECTED_LOW":

            low_id += 1

            active_low = {
                "level_id": f"PL_{low_id:03d}",
                "type": "PROTECTED_LOW",
                "level": row["protected_level"],
                "created_at": idx,
                "state": "ACTIVE",
                "break_time": None,
                "break_price": None,
                "break_direction": None,
            }

    # =========================================================
    # 5. Keep still-active levels
    # =========================================================

    if active_high is not None:
        registry.append(active_high)

    if active_low is not None:
        registry.append(active_low)

    return pd.DataFrame(registry)

In [46]:
protected_registry_v4 = build_protected_level_registry_v4(
    structural_v1
)

print("=" * 90)
print("Protected Level Registry v4")
print("=" * 90)

print(
    protected_registry_v4.to_string(index=False)
)

print("\nState Counts:")
print(
    protected_registry_v4["state"].value_counts()
)

Protected Level Registry v4
level_id           type   level                created_at  state                break_time  break_price break_direction
  PH_002 PROTECTED_HIGH 1.12553 2026-10-02 23:30:00+00:00 BROKEN 2026-10-05 01:10:00+00:00      1.12589         BULLISH
  PL_001  PROTECTED_LOW 1.12563 2026-10-05 01:30:00+00:00 BROKEN 2026-10-05 02:15:00+00:00      1.12543         BEARISH
  PL_004  PROTECTED_LOW 1.11950 2026-10-05 11:05:00+00:00 BROKEN 2026-10-05 16:45:00+00:00      1.11918         BEARISH
  PH_009 PROTECTED_HIGH 1.11981 2026-10-05 17:10:00+00:00 BROKEN 2026-10-05 18:05:00+00:00      1.12118         BULLISH
  PH_011 PROTECTED_HIGH 1.12059 2026-10-05 19:40:00+00:00 BROKEN 2026-10-05 22:25:00+00:00      1.12184         BULLISH
  PL_006  PROTECTED_LOW 1.12186 2026-10-06 01:45:00+00:00 BROKEN 2026-10-06 03:05:00+00:00      1.12146         BEARISH
  PL_008  PROTECTED_LOW 1.12248 2026-10-06 04:15:00+00:00 BROKEN 2026-10-06 05:10:00+00:00      1.12140         BEARISH
  PH_012 PRO

In [47]:
# structural_v2 = attach_protected_level_ids(
#     structural_v1,
#     protected_registry_v4
# )

In [48]:
print("structural_bos_v2 exists :", "structural_bos_v2" in globals())

if "structural_bos_v2" in globals():
    print("shape :", structural_bos_v2.shape)
    print()
    print(structural_bos_v2.head())

structural_bos_v2 exists : True
shape : (39, 8)

       bos_id level_id bos_direction  bos_level                  bos_time  \
0  BOS_PH_002   PH_002   BOS_BULLISH    1.12553 2026-10-05 00:15:00+00:00   
1  BOS_PH_001   PH_001   BOS_BULLISH    1.12594 2026-10-05 01:45:00+00:00   
2  BOS_PL_001   PL_001   BOS_BEARISH    1.12563 2026-10-05 02:05:00+00:00   
3  BOS_PH_005   PH_005   BOS_BULLISH    1.12095 2026-10-05 14:25:00+00:00   
4  BOS_PH_006   PH_006   BOS_BULLISH    1.12086 2026-10-05 14:25:00+00:00   

   bos_price                created_at     source_type  
0    1.12578 2026-10-02 23:30:00+00:00  PROTECTED_HIGH  
1    1.12595 2026-10-02 23:00:00+00:00  PROTECTED_HIGH  
2    1.12562 2026-10-05 01:30:00+00:00   PROTECTED_LOW  
3    1.12139 2026-10-05 13:05:00+00:00  PROTECTED_HIGH  
4    1.12139 2026-10-05 13:25:00+00:00  PROTECTED_HIGH  


In [49]:
print("\n" + "=" * 80)
print("STRUCTURAL BOS V2 — IDENTITY CHECK")
print("=" * 80)

level_frequency = (
    structural_bos_v2
    .groupby("level_id")
    .size()
)

print("Total BOS events :", len(structural_bos_v2))
print("Unique levels    :", level_frequency.index.nunique())
print("Min events/level :", level_frequency.min())
print("Max events/level :", level_frequency.max())

print()

print(
    "ONE EVENT PER LEVEL :",
    "PASSED"
    if (
        len(structural_bos_v2) == 36
        and level_frequency.index.nunique() == 36
        and level_frequency.min() == 1
        and level_frequency.max() == 1
    )
    else "FAILED"
)


STRUCTURAL BOS V2 — IDENTITY CHECK
Total BOS events : 39
Unique levels    : 39
Min events/level : 1
Max events/level : 1

ONE EVENT PER LEVEL : FAILED


In [50]:
def detect_structural_choch_v2(structural_bos_v2):
    """
    Structural CHoCH v2

    Canonical source:
        structural_bos_v2

    Logic:
        First BOS establishes the structural bias.
        An opposite BOS creates CHoCH.
        After CHoCH, the bias flips.

    No look-ahead.
    BOS identity is preserved.
    """

    import pandas as pd

    required = {
        "bos_id",
        "level_id",
        "bos_direction",
        "bos_level",
        "bos_time",
        "bos_price",
    }

    missing = required - set(structural_bos_v2.columns)

    if missing:
        raise ValueError(
            f"Missing BOS columns: {sorted(missing)}"
        )

    if structural_bos_v2.empty:
        return pd.DataFrame(
            columns=[
                "choch_id",
                "bos_id",
                "level_id",
                "choch_direction",
                "choch_level",
                "choch_time",
                "choch_price",
                "structure_bias_before",
                "structure_bias_after",
            ]
        )

    bos = (
        structural_bos_v2
        .copy()
        .sort_values("bos_time")
        .reset_index(drop=True)
    )

    bos["bos_time"] = pd.to_datetime(
        bos["bos_time"],
        utc=True,
    )

    records = []

    active_bias = None

    for _, row in bos.iterrows():

        direction = row["bos_direction"]

        if direction == "BOS_BULLISH":
            bos_bias = "BULLISH"

        elif direction == "BOS_BEARISH":
            bos_bias = "BEARISH"

        else:
            continue

        # --------------------------------------------------
        # First BOS establishes initial structural bias
        # --------------------------------------------------

        if active_bias is None:
            active_bias = bos_bias
            continue

        # --------------------------------------------------
        # Same-direction BOS = continuation
        # --------------------------------------------------

        if bos_bias == active_bias:
            continue

        # --------------------------------------------------
        # Opposite BOS = CHoCH
        # --------------------------------------------------

        choch_direction = direction

        choch_id = f"CHOCH_{row['bos_id']}"

        records.append(
            {
                "choch_id": choch_id,
                "bos_id": row["bos_id"],
                "level_id": row["level_id"],
                "choch_direction": choch_direction,
                "choch_level": float(row["bos_level"]),
                "choch_time": row["bos_time"],
                "choch_price": float(row["bos_price"]),
                "structure_bias_before": active_bias,
                "structure_bias_after": bos_bias,
            }
        )

        # Flip structural bias
        active_bias = bos_bias

    result = pd.DataFrame(records)

    if not result.empty:
        result["choch_time"] = pd.to_datetime(
            result["choch_time"],
            utc=True,
        )

    return result

In [51]:
structural_choch_v2 = detect_structural_choch_v2(
    structural_bos_v2
)

print("\n" + "=" * 80)
print("STRUCTURAL CHoCH V2")
print("=" * 80)

print(
    "Total CHoCH :",
    len(structural_choch_v2)
)

print()

if not structural_choch_v2.empty:
    print(
        structural_choch_v2[
            [
                "choch_id",
                "bos_id",
                "level_id",
                "choch_direction",
                "choch_level",
                "choch_time",
                "structure_bias_before",
                "structure_bias_after",
            ]
        ].to_string(index=False)
    )


STRUCTURAL CHoCH V2
Total CHoCH : 13

        choch_id     bos_id level_id choch_direction  choch_level                choch_time structure_bias_before structure_bias_after
CHOCH_BOS_PL_001 BOS_PL_001   PL_001     BOS_BEARISH      1.12563 2026-10-05 02:05:00+00:00               BULLISH              BEARISH
CHOCH_BOS_PH_005 BOS_PH_005   PH_005     BOS_BULLISH      1.12095 2026-10-05 14:25:00+00:00               BEARISH              BULLISH
CHOCH_BOS_PL_004 BOS_PL_004   PL_004     BOS_BEARISH      1.11950 2026-10-05 16:35:00+00:00               BULLISH              BEARISH
CHOCH_BOS_PH_009 BOS_PH_009   PH_009     BOS_BULLISH      1.11981 2026-10-05 17:40:00+00:00               BEARISH              BULLISH
CHOCH_BOS_PL_006 BOS_PL_006   PL_006     BOS_BEARISH      1.12186 2026-10-06 02:00:00+00:00               BULLISH              BEARISH
CHOCH_BOS_PH_012 BOS_PH_012   PH_012     BOS_BULLISH      1.12173 2026-10-06 06:15:00+00:00               BEARISH              BULLISH
CHOCH_BOS_PL_009

In [52]:
print("\n" + "=" * 80)
print("STRUCTURAL CHoCH V2 — REGRESSION GATE")
print("=" * 80)

required = {
    "choch_id",
    "bos_id",
    "level_id",
    "choch_direction",
    "choch_level",
    "choch_time",
    "choch_price",
    "structure_bias_before",
    "structure_bias_after",
}

missing = required - set(structural_choch_v2.columns)

print(
    "REQUIRED COLUMNS :",
    "PASSED" if not missing else f"FAILED → {sorted(missing)}"
)

print(
    "ROW COUNT        :",
    "PASSED" if len(structural_choch_v2) == 15 else "FAILED"
)

print(
    "CHOCH ID UNIQUE  :",
    "PASSED"
    if structural_choch_v2["choch_id"].is_unique
    else "FAILED"
)

print(
    "BOS ID UNIQUE    :",
    "PASSED"
    if structural_choch_v2["bos_id"].is_unique
    else "FAILED"
)

print(
    "LEVEL ID UNIQUE  :",
    "PASSED"
    if structural_choch_v2["level_id"].is_unique
    else "FAILED"
)

direction_contract = (
    (
        (structural_choch_v2["choch_direction"] == "BOS_BULLISH")
        &
        (structural_choch_v2["structure_bias_before"] == "BEARISH")
        &
        (structural_choch_v2["structure_bias_after"] == "BULLISH")
    )
    |
    (
        (structural_choch_v2["choch_direction"] == "BOS_BEARISH")
        &
        (structural_choch_v2["structure_bias_before"] == "BULLISH")
        &
        (structural_choch_v2["structure_bias_after"] == "BEARISH")
    )
)

print(
    "BIAS TRANSITION   :",
    "PASSED" if direction_contract.all() else "FAILED"
)

time_order = (
    structural_choch_v2["choch_time"]
    .is_monotonic_increasing
)

print(
    "TIME ORDER        :",
    "PASSED" if time_order else "FAILED"
)

print("=" * 80)


STRUCTURAL CHoCH V2 — REGRESSION GATE
REQUIRED COLUMNS : PASSED
ROW COUNT        : FAILED
CHOCH ID UNIQUE  : PASSED
BOS ID UNIQUE    : PASSED
LEVEL ID UNIQUE  : PASSED
BIAS TRANSITION   : PASSED
TIME ORDER        : PASSED


In [53]:
print("build_structure_state :", build_structure_state)

build_structure_state : <function build_structure_state at 0x79df3fff6f00>


In [54]:
print(inspect.signature(build_structure_state))

(market_structure, bos_column='structural_bos', choch_column='choch', bos_level_column='bos_level', bos_level_id_column='broken_level_id', bos_level_type_column='broken_level_type')


In [55]:
import inspect

print(inspect.signature(build_structure_state))

(market_structure, bos_column='structural_bos', choch_column='choch', bos_level_column='bos_level', bos_level_id_column='broken_level_id', bos_level_type_column='broken_level_type')


In [56]:
import inspect

print(inspect.getsource(build_structure_state))

def build_structure_state(
    market_structure,
    bos_column="structural_bos",
    choch_column="choch",
    bos_level_column="bos_level",
    bos_level_id_column="broken_level_id",
    bos_level_type_column="broken_level_type",
):
    """
    Build stateful Structural Market State.

    Tracks:
        - Active market bias
        - Last Structural BOS
        - Last CHoCH
        - Last broken Protected High
        - Last broken Protected Low

    NOTE:
        Active Protected High/Low are intentionally NOT inferred
        from the broken level of a BOS.

        A separate Protected Level Registry will determine which
        protected levels are currently active.
    """

    required_columns = [
        bos_column,
        choch_column,
        bos_level_column,
        bos_level_id_column,
        bos_level_type_column,
        "market_bias",
    ]

    missing = [
        col for col in required_columns
        if col not in market_structure.columns
    ]

    if missing:


In [57]:
import pandas as pd


def build_structure_state_v2(
    m5_df,
    structural_bos_v2,
    structural_choch_v2,
):
    """
    Structure State v2

    Canonical sources:
        structural_bos_v2
        structural_choch_v2

    Supports:
        Multiple BOS events at the same candle timestamp.

    State:
        - active_bias
        - last Structural BOS
        - last CHoCH
        - last broken Protected High
        - last broken Protected Low

    Contract:
        - Chronological processing
        - No look-ahead
        - BOS identity preserved
        - CHoCH identity preserved
        - Protected level identity preserved
        - Multiple events per timestamp supported
    """

    # =========================================================
    # Validate M5
    # =========================================================

    required_m5 = {
        "open",
        "high",
        "low",
        "close",
    }

    missing = required_m5 - set(m5_df.columns)

    if missing:
        raise ValueError(
            f"Missing M5 columns: {sorted(missing)}"
        )

    if not isinstance(m5_df.index, pd.DatetimeIndex):
        raise TypeError(
            "m5_df index must be a pandas DatetimeIndex."
        )

    # =========================================================
    # Validate BOS
    # =========================================================

    required_bos = {
        "bos_id",
        "level_id",
        "bos_direction",
        "bos_level",
        "bos_time",
        "bos_price",
        "source_type",
    }

    missing = required_bos - set(structural_bos_v2.columns)

    if missing:
        raise ValueError(
            f"Missing BOS columns: {sorted(missing)}"
        )

    # =========================================================
    # Validate CHoCH
    # =========================================================

    required_choch = {
        "choch_id",
        "bos_id",
        "level_id",
        "choch_direction",
        "choch_level",
        "choch_time",
        "choch_price",
    }

    missing = required_choch - set(structural_choch_v2.columns)

    if missing:
        raise ValueError(
            f"Missing CHoCH columns: {sorted(missing)}"
        )

    # =========================================================
    # Normalize timestamps
    # =========================================================

    bos = structural_bos_v2.copy()
    choch = structural_choch_v2.copy()

    bos["bos_time"] = pd.to_datetime(
        bos["bos_time"],
        utc=True,
    )

    choch["choch_time"] = pd.to_datetime(
        choch["choch_time"],
        utc=True,
    )

    # =========================================================
    # Deterministic event ordering
    # =========================================================

    bos = (
        bos
        .sort_values(
            ["bos_time", "bos_id"],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    choch = (
        choch
        .sort_values(
            ["choch_time", "choch_id"],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    # =========================================================
    # State variables
    # =========================================================

    active_bias = "UNKNOWN"

    last_bos = None
    last_bos_direction = None
    last_bos_level = None
    last_bos_level_id = None

    last_choch = None
    last_choch_direction = None
    last_choch_level = None
    last_choch_level_id = None

    last_broken_protected_high = None
    last_broken_protected_high_id = None

    last_broken_protected_low = None
    last_broken_protected_low_id = None

    records = []

    # =========================================================
    # M5 chronological state engine
    # =========================================================

    for timestamp in m5_df.index:

        # -----------------------------------------------------
        # All BOS events at this timestamp
        # -----------------------------------------------------

        bos_events_at_time = bos.loc[
            bos["bos_time"] == timestamp
        ]

        # -----------------------------------------------------
        # Process EVERY BOS event
        # -----------------------------------------------------

        for _, bos_event in bos_events_at_time.iterrows():

            last_bos = bos_event["bos_id"]

            last_bos_direction = (
                bos_event["bos_direction"]
            )

            last_bos_level = float(
                bos_event["bos_level"]
            )

            last_bos_level_id = (
                bos_event["level_id"]
            )

            # ---------------------------------------------
            # Structural bias
            # ---------------------------------------------

            if bos_event["bos_direction"] == "BOS_BULLISH":

                active_bias = "BULLISH"

            elif bos_event["bos_direction"] == "BOS_BEARISH":

                active_bias = "BEARISH"

            # ---------------------------------------------
            # Protected High / Low lifecycle reference
            # ---------------------------------------------

            if (
                bos_event["source_type"]
                == "PROTECTED_HIGH"
            ):

                last_broken_protected_high = float(
                    bos_event["bos_level"]
                )

                last_broken_protected_high_id = (
                    bos_event["level_id"]
                )

            elif (
                bos_event["source_type"]
                == "PROTECTED_LOW"
            ):

                last_broken_protected_low = float(
                    bos_event["bos_level"]
                )

                last_broken_protected_low_id = (
                    bos_event["level_id"]
                )

        # -----------------------------------------------------
        # All CHoCH events at this timestamp
        # -----------------------------------------------------

        choch_events_at_time = choch.loc[
            choch["choch_time"] == timestamp
        ]

        # -----------------------------------------------------
        # Process CHoCH events
        # -----------------------------------------------------

        for _, choch_event in choch_events_at_time.iterrows():

            last_choch = choch_event["choch_id"]

            last_choch_direction = (
                choch_event["choch_direction"]
            )

            last_choch_level = float(
                choch_event["choch_level"]
            )

            last_choch_level_id = (
                choch_event["level_id"]
            )

            # Current CHoCH direction is represented
            # by the corresponding BOS direction.

            if (
                choch_event["choch_direction"]
                == "BOS_BULLISH"
            ):

                active_bias = "BULLISH"

            elif (
                choch_event["choch_direction"]
                == "BOS_BEARISH"
            ):

                active_bias = "BEARISH"

        # =====================================================
        # Write state snapshot
        # =====================================================

        records.append(
            {
                "timestamp": timestamp,

                "active_bias": active_bias,

                "last_bos": last_bos,
                "last_bos_direction": last_bos_direction,
                "last_bos_level": last_bos_level,
                "last_bos_level_id": last_bos_level_id,

                "last_choch": last_choch,
                "last_choch_direction": last_choch_direction,
                "last_choch_level": last_choch_level,
                "last_choch_level_id": last_choch_level_id,

                "last_broken_protected_high":
                    last_broken_protected_high,

                "last_broken_protected_high_id":
                    last_broken_protected_high_id,

                "last_broken_protected_low":
                    last_broken_protected_low,

                "last_broken_protected_low_id":
                    last_broken_protected_low_id,
            }
        )

    # =========================================================
    # Result
    # =========================================================

    result = pd.DataFrame(records)

    result = result.set_index("timestamp")

    result.index = pd.DatetimeIndex(
        result.index,
        tz="UTC",
    )

    return result

In [58]:
duplicate_bos_times = (
    structural_bos_v2
    .groupby("bos_time")
    .size()
    .loc[lambda x: x > 1]
)

print("=" * 100)
print("DUPLICATE BOS TIMESTAMPS")
print("=" * 100)

print(duplicate_bos_times)

print("\nDetailed events:")
print(
    structural_bos_v2[
        structural_bos_v2["bos_time"].isin(
            duplicate_bos_times.index
        )
    ][
        [
            "bos_id",
            "level_id",
            "bos_direction",
            "bos_level",
            "bos_time",
            "bos_price",
            "source_type",
        ]
    ]
    .sort_values(["bos_time", "bos_id"])
    .to_string(index=False)
)

DUPLICATE BOS TIMESTAMPS
bos_time
2026-10-05 14:25:00+00:00    2
2026-10-05 21:05:00+00:00    2
2026-10-06 10:30:00+00:00    2
2026-10-06 14:20:00+00:00    2
dtype: int64

Detailed events:
    bos_id level_id bos_direction  bos_level                  bos_time  bos_price    source_type
BOS_PH_005   PH_005   BOS_BULLISH    1.12095 2026-10-05 14:25:00+00:00    1.12139 PROTECTED_HIGH
BOS_PH_006   PH_006   BOS_BULLISH    1.12086 2026-10-05 14:25:00+00:00    1.12139 PROTECTED_HIGH
BOS_PH_008   PH_008   BOS_BULLISH    1.12138 2026-10-05 21:05:00+00:00    1.12148 PROTECTED_HIGH
BOS_PH_010   PH_010   BOS_BULLISH    1.12127 2026-10-05 21:05:00+00:00    1.12148 PROTECTED_HIGH
BOS_PH_013   PH_013   BOS_BULLISH    1.12216 2026-10-06 10:30:00+00:00    1.12223 PROTECTED_HIGH
BOS_PH_014   PH_014   BOS_BULLISH    1.12200 2026-10-06 10:30:00+00:00    1.12223 PROTECTED_HIGH
BOS_PH_003   PH_003   BOS_BULLISH    1.12564 2026-10-06 14:20:00+00:00    1.12590 PROTECTED_HIGH
BOS_PH_004   PH_004   BOS_BULLISH  

In [59]:
structure_state_v2 = build_structure_state_v2(
    m5_df=m5_df,
    structural_bos_v2=structural_bos_v2,
    structural_choch_v2=structural_choch_v2,
)

print("=" * 100)
print("STRUCTURE STATE V2")
print("=" * 100)

print("Shape :", structure_state_v2.shape)

print("\nFinal state:")
print(
    structure_state_v2[
        [
            "active_bias",
            "last_bos",
            "last_bos_direction",
            "last_bos_level",
            "last_bos_level_id",
            "last_choch",
            "last_choch_direction",
            "last_choch_level",
            "last_choch_level_id",
            "last_broken_protected_high_id",
            "last_broken_protected_low_id",
        ]
    ].tail(1).to_string()
)

STRUCTURE STATE V2
Shape : (1000, 13)

Final state:
                          active_bias    last_bos last_bos_direction  last_bos_level last_bos_level_id        last_choch last_choch_direction  last_choch_level last_choch_level_id last_broken_protected_high_id last_broken_protected_low_id
timestamp                                                                                                                                                                                                                                     
2026-10-08 08:05:00+00:00     BEARISH  BOS_PL_023        BOS_BEARISH         1.11991            PL_023  CHOCH_BOS_PL_023          BOS_BEARISH           1.11991              PL_023                        PH_026                       PL_023


In [60]:
import pandas as pd


print("=" * 100)
print("STRUCTURE STATE V2 REGRESSION GATE")
print("=" * 100)

violations = []

# =========================================================
# 1. ROW COUNT
# =========================================================

if len(structure_state_v2) != len(m5_df):
    violations.append(
        f"ROW COUNT: expected {len(m5_df)}, "
        f"got {len(structure_state_v2)}"
    )
else:
    print("ROW COUNT                         : PASSED")


# =========================================================
# 2. INDEX IDENTITY
# =========================================================

if not structure_state_v2.index.equals(m5_df.index):
    violations.append(
        "INDEX IDENTITY: Structure State index "
        "does not exactly match M5 index."
    )
else:
    print("INDEX IDENTITY                    : PASSED")


# =========================================================
# 3. BOS EVENT COUNT
# =========================================================

bos_event_count = (
    structural_bos_v2["bos_id"].nunique()
)

if bos_event_count != 36:
    print(
        f"BOS EVENT COUNT                  : INFO "
        f"(current={bos_event_count})"
    )
else:
    print("BOS EVENT COUNT                  : PASSED")


# =========================================================
# 4. CHoCH EVENT COUNT
# =========================================================

choch_event_count = (
    structural_choch_v2["choch_id"].nunique()
)

if choch_event_count != 15:
    print(
        f"CHoCH EVENT COUNT                : INFO "
        f"(current={choch_event_count})"
    )
else:
    print("CHoCH EVENT COUNT                : PASSED")


# =========================================================
# 5. BOS IDENTITY PRESERVATION
# =========================================================

state_bos_ids = set(
    structure_state_v2["last_bos"]
    .dropna()
)

source_bos_ids = set(
    structural_bos_v2["bos_id"]
)

if not state_bos_ids.issubset(source_bos_ids):
    violations.append(
        "BOS IDENTITY: state contains BOS IDs "
        "not present in structural_bos_v2."
    )
else:
    print("BOS IDENTITY PRESERVATION        : PASSED")


# =========================================================
# 6. CHoCH IDENTITY PRESERVATION
# =========================================================

state_choch_ids = set(
    structure_state_v2["last_choch"]
    .dropna()
)

source_choch_ids = set(
    structural_choch_v2["choch_id"]
)

if not state_choch_ids.issubset(source_choch_ids):
    violations.append(
        "CHoCH IDENTITY: state contains CHoCH IDs "
        "not present in structural_choch_v2."
    )
else:
    print("CHoCH IDENTITY PRESERVATION      : PASSED")


# =========================================================
# 7. PROTECTED HIGH IDENTITY
# =========================================================

state_ph_ids = set(
    structure_state_v2[
        "last_broken_protected_high_id"
    ].dropna()
)

source_ph_ids = set(
    structural_bos_v2.loc[
        structural_bos_v2["source_type"]
        == "PROTECTED_HIGH",
        "level_id"
    ]
)

if not state_ph_ids.issubset(source_ph_ids):
    violations.append(
        "PROTECTED HIGH IDENTITY FAILED"
    )
else:
    print("PROTECTED HIGH IDENTITY           : PASSED")


# =========================================================
# 8. PROTECTED LOW IDENTITY
# =========================================================

state_pl_ids = set(
    structure_state_v2[
        "last_broken_protected_low_id"
    ].dropna()
)

source_pl_ids = set(
    structural_bos_v2.loc[
        structural_bos_v2["source_type"]
        == "PROTECTED_LOW",
        "level_id"
    ]
)

if not state_pl_ids.issubset(source_pl_ids):
    violations.append(
        "PROTECTED LOW IDENTITY FAILED"
    )
else:
    print("PROTECTED LOW IDENTITY            : PASSED")


# =========================================================
# 9. ACTIVE BIAS CONTRACT
# =========================================================

valid_bias = {
    "UNKNOWN",
    "BULLISH",
    "BEARISH",
}

invalid_bias = set(
    structure_state_v2["active_bias"].dropna()
) - valid_bias

if invalid_bias:
    violations.append(
        f"ACTIVE BIAS: invalid values {invalid_bias}"
    )
else:
    print("ACTIVE BIAS CONTRACT              : PASSED")


# =========================================================
# 10. NO LOOK-AHEAD — BOS
# =========================================================

for idx, row in structure_state_v2.iterrows():

    bos_id = row["last_bos"]

    if pd.isna(bos_id):
        continue

    source = structural_bos_v2.loc[
        structural_bos_v2["bos_id"] == bos_id
    ]

    if source.empty:
        violations.append(
            f"BOS LOOK-AHEAD: unknown {bos_id}"
        )
        continue

    bos_time = source.iloc[0]["bos_time"]

    if idx < bos_time:
        violations.append(
            f"BOS LOOK-AHEAD: {bos_id} "
            f"visible before event at {bos_time}"
        )
        break

if not any(
    "BOS LOOK-AHEAD" in v
    for v in violations
):
    print("NO LOOK-AHEAD — BOS               : PASSED")


# =========================================================
# 11. NO LOOK-AHEAD — CHoCH
# =========================================================

for idx, row in structure_state_v2.iterrows():

    choch_id = row["last_choch"]

    if pd.isna(choch_id):
        continue

    source = structural_choch_v2.loc[
        structural_choch_v2["choch_id"] == choch_id
    ]

    if source.empty:
        violations.append(
            f"CHoCH LOOK-AHEAD: unknown {choch_id}"
        )
        continue

    choch_time = source.iloc[0]["choch_time"]

    if idx < choch_time:
        violations.append(
            f"CHoCH LOOK-AHEAD: {choch_id} "
            f"visible before event at {choch_time}"
        )
        break

if not any(
    "CHoCH LOOK-AHEAD" in v
    for v in violations
):
    print("NO LOOK-AHEAD — CHoCH             : PASSED")


# =========================================================
# 12. LAST BOS LEVEL CONSISTENCY
# =========================================================

for idx, row in structure_state_v2.iterrows():

    bos_id = row["last_bos"]

    if pd.isna(bos_id):
        continue

    source = structural_bos_v2.loc[
        structural_bos_v2["bos_id"] == bos_id
    ].iloc[0]

    if float(row["last_bos_level"]) != float(
        source["bos_level"]
    ):
        violations.append(
            f"BOS LEVEL MISMATCH at {idx}"
        )
        break

if not any(
    "BOS LEVEL MISMATCH" in v
    for v in violations
):
    print("LAST BOS LEVEL CONSISTENCY        : PASSED")


# =========================================================
# 13. LAST BOS ID / DIRECTION CONSISTENCY
# =========================================================

for idx, row in structure_state_v2.iterrows():

    bos_id = row["last_bos"]

    if pd.isna(bos_id):
        continue

    source = structural_bos_v2.loc[
        structural_bos_v2["bos_id"] == bos_id
    ].iloc[0]

    if row["last_bos_direction"] != source["bos_direction"]:
        violations.append(
            f"BOS DIRECTION MISMATCH at {idx}"
        )
        break

if not any(
    "BOS DIRECTION MISMATCH" in v
    for v in violations
):
    print("LAST BOS DIRECTION CONSISTENCY   : PASSED")


# =========================================================
# 14. FINAL RESULT
# =========================================================

print("-" * 100)

if violations:

    print("FINAL RESULT : ❌ STRUCTURE STATE V2 FAILED")

    print("\nViolations:")

    for violation in violations:
        print(" -", violation)

else:

    print("FINAL RESULT : ✅ STRUCTURE STATE V2 PASSED")

STRUCTURE STATE V2 REGRESSION GATE
ROW COUNT                         : PASSED
INDEX IDENTITY                    : PASSED
BOS EVENT COUNT                  : INFO (current=39)
CHoCH EVENT COUNT                : INFO (current=13)
BOS IDENTITY PRESERVATION        : PASSED
CHoCH IDENTITY PRESERVATION      : PASSED
PROTECTED HIGH IDENTITY           : PASSED
PROTECTED LOW IDENTITY            : PASSED
ACTIVE BIAS CONTRACT              : PASSED
NO LOOK-AHEAD — BOS               : PASSED
NO LOOK-AHEAD — CHoCH             : PASSED
LAST BOS LEVEL CONSISTENCY        : PASSED
LAST BOS DIRECTION CONSISTENCY   : PASSED
----------------------------------------------------------------------------------------------------
FINAL RESULT : ✅ STRUCTURE STATE V2 PASSED


In [61]:
print("=" * 100)
print("CANONICAL LIQUIDITY / STRUCTURE OBJECTS")
print("=" * 100)

objects = {
    "m5_df": m5_df,
    "minor_v2": minor_v2,
    "structural_v1": structural_v1,
    "protected_registry_v1": protected_registry_v1,
    "structural_bos_v2": structural_bos_v2,
    "structural_choch_v2": structural_choch_v2,
    "structure_state_v2": structure_state_v2,
}

for name, obj in objects.items():
    print(
        f"{name:28} | "
        f"type={type(obj).__name__:12} | "
        f"shape={getattr(obj, 'shape', None)}"
    )

CANONICAL LIQUIDITY / STRUCTURE OBJECTS
m5_df                        | type=DataFrame    | shape=(1000, 8)
minor_v2                     | type=DataFrame    | shape=(156, 19)
structural_v1                | type=DataFrame    | shape=(156, 23)
protected_registry_v1        | type=DataFrame    | shape=(51, 8)
structural_bos_v2            | type=DataFrame    | shape=(39, 8)
structural_choch_v2          | type=DataFrame    | shape=(13, 9)
structure_state_v2           | type=DataFrame    | shape=(1000, 13)


In [62]:
from pathlib import Path

equal_levels_path = Path(
    "/home/sattar/projects/trader-bot/equal_levels.py"
)

print("=" * 100)
print("EQUAL LEVELS MODULE")
print("=" * 100)

print("Exists :", equal_levels_path.exists())
print("Path   :", equal_levels_path)

if equal_levels_path.exists():
    print("\n--- FUNCTIONS / CLASSES ---")

    import ast

    source = equal_levels_path.read_text(
        encoding="utf-8"
    )

    tree = ast.parse(source)

    for node in tree.body:
        if isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)
        ):
            print(
                f"{type(node).__name__:16} "
                f"{node.name}"
            )

EQUAL LEVELS MODULE
Exists : True
Path   : /home/sattar/projects/trader-bot/equal_levels.py

--- FUNCTIONS / CLASSES ---
FunctionDef      detect_equal_levels_v1


In [63]:
import inspect
import equal_levels

print("=" * 100)
print("EQUAL LEVELS V1 CONTRACT")
print("=" * 100)

print("Function :", equal_levels.detect_equal_levels_v1)
print(
    "Signature:",
    inspect.signature(
        equal_levels.detect_equal_levels_v1
    )
)

print("\n--- SOURCE ---\n")

print(
    inspect.getsource(
        equal_levels.detect_equal_levels_v1
    )
)

EQUAL LEVELS V1 CONTRACT
Function : <function detect_equal_levels_v1 at 0x79df3f83ecf0>
Signature: (minor_v2: 'pd.DataFrame', atr: 'pd.Series | str', *, atr_multiplier: 'float' = 0.15, min_tolerance: 'float' = 3e-05, max_tolerance: 'float' = 0.00015, min_bars_between: 'int' = 3, max_bars_between: 'int' = 100) -> 'pd.DataFrame'

--- SOURCE ---

def detect_equal_levels_v1(
    minor_v2: pd.DataFrame,
    atr: pd.Series | str,
    *,
    atr_multiplier: float = DEFAULT_ATR_MULTIPLIER,
    min_tolerance: float = DEFAULT_MIN_TOLERANCE,
    max_tolerance: float = DEFAULT_MAX_TOLERANCE,
    min_bars_between: int = DEFAULT_MIN_BARS_BETWEEN,
    max_bars_between: int = DEFAULT_MAX_BARS_BETWEEN,
) -> pd.DataFrame:
    """
    Equal High / Equal Low Detector v1.

    Input
    -----
    minor_v2:
        Confirmed Minor Structure swings.
        Required columns:
            - swing_type
            - swing_price

        DatetimeIndex is required.

    atr:
        ATR values aligned to the orig

In [64]:
import pandas as pd
import numpy as np


atr_period = 14

prev_close = m5_df["close"].shift(1)

tr = pd.concat(
    [
        m5_df["high"] - m5_df["low"],
        (m5_df["high"] - prev_close).abs(),
        (m5_df["low"] - prev_close).abs(),
    ],
    axis=1,
).max(axis=1)

m5_atr = (
    tr
    .rolling(
        window=atr_period,
        min_periods=atr_period,
    )
    .mean()
)

print("=" * 100)
print("CANONICAL M5 ATR")
print("=" * 100)

print("ATR period       :", atr_period)
print("M5 rows          :", len(m5_df))
print("ATR non-null     :", m5_atr.notna().sum())
print("ATR null         :", m5_atr.isna().sum())
print("Index aligned    :", m5_atr.index.equals(m5_df.index))

CANONICAL M5 ATR
ATR period       : 14
M5 rows          : 1000
ATR non-null     : 987
ATR null         : 13
Index aligned    : True


In [65]:
from equal_levels import detect_equal_levels_v1


equal_levels_v1 = detect_equal_levels_v1(
    minor_v2=minor_v2,
    atr=m5_atr,
    atr_multiplier=0.15,
    min_tolerance=3e-05,
    max_tolerance=0.00015,
    min_bars_between=3,
    max_bars_between=100,
)

print("=" * 100)
print("EQUAL LEVELS V1 — CANONICAL DETECTOR")
print("=" * 100)

print("Total :", len(equal_levels_v1))

if not equal_levels_v1.empty:
    print(
        "\nEqual Type:"
    )
    print(
        equal_levels_v1["equal_type"]
        .value_counts()
    )

    print("\nSample:")
    print(
        equal_levels_v1.head(20).to_string(
            index=False
        )
    )

EQUAL LEVELS V1 — CANONICAL DETECTOR
Total : 10

Equal Type:
equal_type
EQH    5
EQL    5
Name: count, dtype: int64

Sample:
                  swing_a                   swing_b  price_a  price_b  price_difference      atr  atr_ratio  bars_between equal_type
2026-10-05 02:30:00+00:00 2026-10-05 03:00:00+00:00  1.12564  1.12561           0.00003 0.000223   0.134615             6        EQH
2026-10-05 06:05:00+00:00 2026-10-05 07:05:00+00:00  1.11607  1.11608           0.00001 0.000728   0.013739            12        EQL
2026-10-05 15:30:00+00:00 2026-10-05 16:00:00+00:00  1.12139  1.12138           0.00001 0.000500   0.020000             6        EQH
2026-10-05 23:20:00+00:00 2026-10-06 00:20:00+00:00  1.12157  1.12156           0.00001 0.000308   0.032483            12        EQL
2026-10-06 05:40:00+00:00 2026-10-06 06:00:00+00:00  1.12119  1.12122           0.00003 0.000239   0.125373             4        EQL
2026-10-06 11:40:00+00:00 2026-10-06 12:30:00+00:00  1.12472  1.12469        

In [66]:
import pandas as pd
import numpy as np


def audit_equal_levels_formation_v1(
    equal_levels: pd.DataFrame,
    m5_df: pd.DataFrame,
) -> pd.DataFrame:

    required_equal = {
        "swing_a",
        "swing_b",
        "price_a",
        "price_b",
        "price_difference",
        "atr",
        "atr_ratio",
        "bars_between",
        "equal_type",
    }

    missing = required_equal - set(equal_levels.columns)
    if missing:
        raise ValueError(
            f"Equal Levels missing columns: {sorted(missing)}"
        )

    required_m5 = {"high", "low", "close"}
    missing_m5 = required_m5 - set(m5_df.columns)
    if missing_m5:
        raise ValueError(
            f"M5 missing columns: {sorted(missing_m5)}"
        )

    eq = equal_levels.copy()
    eq.index = pd.to_datetime(eq.index, utc=True)

    formation_valid = []
    formation_reason = []

    for _, row in eq.iterrows():

        swing_a = pd.Timestamp(row["swing_a"])
        swing_b = pd.Timestamp(row["swing_b"])
        equal_type = row["equal_type"]

        if swing_a >= swing_b:
            formation_valid.append(False)
            formation_reason.append("INVALID_SWING_ORDER")
            continue

        if equal_type not in {"EQH", "EQL"}:
            formation_valid.append(False)
            formation_reason.append("INVALID_EQUAL_TYPE")
            continue

        if not np.isfinite(row["price_a"]) or not np.isfinite(row["price_b"]):
            formation_valid.append(False)
            formation_reason.append("INVALID_PRICE")
            continue

        if not np.isfinite(row["atr"]) or row["atr"] <= 0:
            formation_valid.append(False)
            formation_reason.append("INVALID_ATR")
            continue

        if equal_type == "EQH":
            expected_a = m5_df.loc[swing_a:, "high"].max()
            expected_b = m5_df.loc[swing_b:, "high"].max()

            if not np.isclose(
                float(row["price_a"]),
                float(row["price_b"]),
                atol=max(float(row["atr"]) * 0.15, 1e-12),
            ):
                formation_valid.append(False)
                formation_reason.append("PRICE_DISTANCE_TOO_LARGE")
                continue

        else:
            expected_a = m5_df.loc[swing_a:, "low"].min()
            expected_b = m5_df.loc[swing_b:, "low"].min()

            if not np.isclose(
                float(row["price_a"]),
                float(row["price_b"]),
                atol=max(float(row["atr"]) * 0.15, 1e-12),
            ):
                formation_valid.append(False)
                formation_reason.append("PRICE_DISTANCE_TOO_LARGE")
                continue

        formation_valid.append(True)
        formation_reason.append("VALID")

    eq["formation_valid"] = formation_valid
    eq["formation_reason"] = formation_reason

    return eq

In [67]:
equal_levels_formation_v1 = audit_equal_levels_formation_v1(
    equal_levels=equal_levels_v1,
    m5_df=m5_df,
)

print("=" * 100)
print("EQUAL LEVELS FORMATION AUDIT V1")
print("=" * 100)

print("Total Candidates :", len(equal_levels_formation_v1))

print("\nFormation Valid:")
print(
    equal_levels_formation_v1["formation_valid"]
    .value_counts(dropna=False)
)

print("\nFormation Reason:")
print(
    equal_levels_formation_v1["formation_reason"]
    .value_counts(dropna=False)
)

display(
    equal_levels_formation_v1[
        [
            "swing_a",
            "swing_b",
            "equal_type",
            "price_a",
            "price_b",
            "price_difference",
            "atr",
            "atr_ratio",
            "bars_between",
            "formation_valid",
            "formation_reason",
        ]
    ]
)

EQUAL LEVELS FORMATION AUDIT V1
Total Candidates : 10

Formation Valid:
formation_valid
True    10
Name: count, dtype: int64

Formation Reason:
formation_reason
VALID    10
Name: count, dtype: int64


,swing_a,swing_b,equal_type,price_a,price_b,price_difference,atr,atr_ratio,bars_between,formation_valid,formation_reason
1970-01-01 00:00:00+00:00,2026-10-05 02:30:00+00:00,2026-10-05 03:00:00+00:00,EQH,1.12564,1.12561,0.00003,0.000223,0.134615,6,True,VALID
1970-01-01 00:00:00.000000001+00:00,2026-10-05 06:05:00+00:00,2026-10-05 07:05:00+00:00,EQL,1.11607,1.11608,0.00001,0.000728,0.013739,12,True,VALID
1970-01-01 00:00:00.000000002+00:00,2026-10-05 15:30:00+00:00,2026-10-05 16:00:00+00:00,EQH,1.12139,1.12138,0.00001,0.000500,0.020000,6,True,VALID
1970-01-01 00:00:00.000000003+00:00,2026-10-05 23:20:00+00:00,2026-10-06 00:20:00+00:00,EQL,1.12157,1.12156,0.00001,0.000308,0.032483,12,True,VALID
1970-01-01 00:00:00.000000004+00:00,2026-10-06 05:40:00+00:00,2026-10-06 06:00:00+00:00,EQL,1.12119,1.12122,0.00003,0.000239,0.125373,4,True,VALID
1970-01-01 00:00:00.000000005+00:00,2026-10-06 11:40:00+00:00,2026-10-06 12:30:00+00:00,EQH,1.12472,1.12469,0.00003,0.000574,0.052304,10,True,VALID
1970-01-01 00:00:00.000000006+00:00,2026-10-06 16:45:00+00:00,2026-10-06 17:20:00+00:00,EQL,1.12469,1.12468,0.00001,0.000483,0.020710,7,True,VALID
1970-01-01 00:00:00.000000007+00:00,2026-10-07 23:30:00+00:00,2026-10-07 23:50:00+00:00,EQH,1.11967,1.11967,0.00000,0.000212,0.000000,4,True,VALID
1970-01-01 00:00:00.000000008+00:00,2026-10-08 01:25:00+00:00,2026-10-08 03:05:00+00:00,EQH,1.12025,1.12026,0.00001,0.000160,0.062500,20,True,VALID
1970-01-01 00:00:00.000000009+00:00,2026-10-08 04:30:00+00:00,2026-10-08 06:10:00+00:00,EQL,1.11974,1.11976,0.00002,0.000271,0.073879,20,True,VALID


In [68]:
import pandas as pd
import numpy as np


def audit_equal_levels_retracement_v1(
    equal_levels: pd.DataFrame,
    m5_df: pd.DataFrame,
    *,
    min_retracement_atr: float = 1.0,
) -> pd.DataFrame:

    required = {
        "swing_a",
        "swing_b",
        "price_a",
        "price_b",
        "atr",
        "equal_type",
    }

    missing = required - set(equal_levels.columns)
    if missing:
        raise ValueError(
            f"Equal Levels missing columns: {sorted(missing)}"
        )

    eq = equal_levels.copy()

    if not isinstance(m5_df.index, pd.DatetimeIndex):
        raise TypeError("m5_df must have a DatetimeIndex")

    m5_df = m5_df.sort_index()

    results = []

    for _, row in eq.iterrows():

        swing_a = pd.Timestamp(row["swing_a"])
        swing_b = pd.Timestamp(row["swing_b"])

        price_a = float(row["price_a"])
        price_b = float(row["price_b"])
        atr = float(row["atr"])
        equal_type = row["equal_type"]

        if swing_a >= swing_b:
            results.append(
                {
                    "retracement_price": np.nan,
                    "retracement_depth": np.nan,
                    "retracement_atr": np.nan,
                    "retracement_valid": False,
                    "retracement_reason": "INVALID_SWING_ORDER",
                }
            )
            continue

        window = m5_df.loc[
            (m5_df.index > swing_a)
            & (m5_df.index <= swing_b)
        ]

        if window.empty:
            results.append(
                {
                    "retracement_price": np.nan,
                    "retracement_depth": np.nan,
                    "retracement_atr": np.nan,
                    "retracement_valid": False,
                    "retracement_reason": "NO_INTERMEDIATE_CANDLES",
                }
            )
            continue

        if not np.isfinite(atr) or atr <= 0:
            results.append(
                {
                    "retracement_price": np.nan,
                    "retracement_depth": np.nan,
                    "retracement_atr": np.nan,
                    "retracement_valid": False,
                    "retracement_reason": "INVALID_ATR",
                }
            )
            continue

        # ------------------------------------------------------------
        # EQH:
        # Price should move DOWN from swing_a before returning
        # toward the equal-high area at swing_b.
        # ------------------------------------------------------------
        if equal_type == "EQH":

            lowest_low = float(window["low"].min())

            retracement_depth = min(
                price_a - lowest_low,
                price_b - lowest_low,
            )

            retracement_price = lowest_low

        # ------------------------------------------------------------
        # EQL:
        # Price should move UP from swing_a before returning
        # toward the equal-low area at swing_b.
        # ------------------------------------------------------------
        elif equal_type == "EQL":

            highest_high = float(window["high"].max())

            retracement_depth = max(
                highest_high - price_a,
                highest_high - price_b,
            )

            retracement_price = highest_high

        else:
            results.append(
                {
                    "retracement_price": np.nan,
                    "retracement_depth": np.nan,
                    "retracement_atr": np.nan,
                    "retracement_valid": False,
                    "retracement_reason": "INVALID_EQUAL_TYPE",
                }
            )
            continue

        retracement_atr = retracement_depth / atr

        valid = retracement_atr >= min_retracement_atr

        results.append(
            {
                "retracement_price": retracement_price,
                "retracement_depth": retracement_depth,
                "retracement_atr": retracement_atr,
                "retracement_valid": valid,
                "retracement_reason": (
                    "VALID"
                    if valid
                    else "INSUFFICIENT_RETRACEMENT"
                ),
            }
        )

    audit = eq.reset_index(drop=True).copy()

    audit[
        [
            "retracement_price",
            "retracement_depth",
            "retracement_atr",
            "retracement_valid",
            "retracement_reason",
        ]
    ] = pd.DataFrame(results, index=audit.index)

    return audit

In [69]:
equal_levels_retracement_v1 = audit_equal_levels_retracement_v1(
    equal_levels=equal_levels_formation_v1,
    m5_df=m5_df,
    min_retracement_atr=1.0,
)

print("=" * 100)
print("EQUAL LEVELS RETRACEMENT AUDIT V1")
print("=" * 100)

print("Total Candidates :", len(equal_levels_retracement_v1))

print("\nRetracement Valid:")
print(
    equal_levels_retracement_v1[
        "retracement_valid"
    ].value_counts(dropna=False)
)

print("\nRetracement Reason:")
print(
    equal_levels_retracement_v1[
        "retracement_reason"
    ].value_counts(dropna=False)
)

display(
    equal_levels_retracement_v1[
        [
            "swing_a",
            "swing_b",
            "equal_type",
            "price_a",
            "price_b",
            "retracement_price",
            "retracement_depth",
            "atr",
            "retracement_atr",
            "retracement_valid",
            "retracement_reason",
        ]
    ]
)

EQUAL LEVELS RETRACEMENT AUDIT V1
Total Candidates : 10

Retracement Valid:
retracement_valid
True    10
Name: count, dtype: int64

Retracement Reason:
retracement_reason
VALID    10
Name: count, dtype: int64


,swing_a,swing_b,equal_type,price_a,price_b,retracement_price,retracement_depth,atr,retracement_atr,retracement_valid,retracement_reason
0,2026-10-05 02:30:00+00:00,2026-10-05 03:00:00+00:00,EQH,1.12564,1.12561,1.12509,0.00052,0.000223,2.333333,True,VALID
1,2026-10-05 06:05:00+00:00,2026-10-05 07:05:00+00:00,EQL,1.11607,1.11608,1.11869,0.00262,0.000728,3.599607,True,VALID
2,2026-10-05 15:30:00+00:00,2026-10-05 16:00:00+00:00,EQH,1.12139,1.12138,1.12028,0.00110,0.000500,2.200000,True,VALID
3,2026-10-05 23:20:00+00:00,2026-10-06 00:20:00+00:00,EQL,1.12157,1.12156,1.12249,0.00093,0.000308,3.020882,True,VALID
4,2026-10-06 05:40:00+00:00,2026-10-06 06:00:00+00:00,EQL,1.12119,1.12122,1.12151,0.00032,0.000239,1.337313,True,VALID
5,2026-10-06 11:40:00+00:00,2026-10-06 12:30:00+00:00,EQH,1.12472,1.12469,1.12342,0.00127,0.000574,2.214197,True,VALID
6,2026-10-06 16:45:00+00:00,2026-10-06 17:20:00+00:00,EQL,1.12469,1.12468,1.12571,0.00103,0.000483,2.133136,True,VALID
7,2026-10-07 23:30:00+00:00,2026-10-07 23:50:00+00:00,EQH,1.11967,1.11967,1.11932,0.00035,0.000212,1.649832,True,VALID
8,2026-10-08 01:25:00+00:00,2026-10-08 03:05:00+00:00,EQH,1.12025,1.12026,1.11991,0.00034,0.000160,2.125000,True,VALID
9,2026-10-08 04:30:00+00:00,2026-10-08 06:10:00+00:00,EQL,1.11974,1.11976,1.12128,0.00154,0.000271,5.688654,True,VALID


In [70]:
import pandas as pd
import numpy as np


def audit_equal_levels_freshness_v1(
    equal_levels: pd.DataFrame,
    m5_df: pd.DataFrame,
) -> pd.DataFrame:

    required = {
        "swing_a",
        "swing_b",
        "price_a",
        "price_b",
        "equal_type",
        "formation_valid",
        "retracement_valid",
    }

    missing = required - set(equal_levels.columns)
    if missing:
        raise ValueError(
            f"Equal Levels missing columns: {sorted(missing)}"
        )

    if not isinstance(m5_df.index, pd.DatetimeIndex):
        raise TypeError("m5_df must have a DatetimeIndex")

    m5 = m5_df.sort_index()
    eq = equal_levels.reset_index(drop=True).copy()

    records = []

    for _, row in eq.iterrows():

        swing_a = pd.Timestamp(row["swing_a"])
        swing_b = pd.Timestamp(row["swing_b"])

        price_a = float(row["price_a"])
        price_b = float(row["price_b"])

        equal_type = row["equal_type"]

        # Equal level becomes tradable/observable only
        # after the second swing has formed.
        level_price = (price_a + price_b) / 2.0

        future = m5.loc[m5.index > swing_b]

        first_sweep_time = pd.NaT
        first_sweep_price = np.nan
        first_sweep_direction = None

        state = "ACTIVE"

        for ts, candle in future.iterrows():

            high = float(candle["high"])
            low = float(candle["low"])

            if equal_type == "EQH":

                if high >= level_price:
                    first_sweep_time = ts
                    first_sweep_price = high
                    first_sweep_direction = "BULLISH"
                    state = "SWEPT"
                    break

            elif equal_type == "EQL":

                if low <= level_price:
                    first_sweep_time = ts
                    first_sweep_price = low
                    first_sweep_direction = "BEARISH"
                    state = "SWEPT"
                    break

        records.append(
            {
                "level_price": level_price,
                "activation_time": swing_b,
                "freshness_state": state,
                "sweep_time": first_sweep_time,
                "sweep_price": first_sweep_price,
                "sweep_direction": first_sweep_direction,
            }
        )

    audit = pd.concat(
        [
            eq,
            pd.DataFrame(records),
        ],
        axis=1,
    )

    return audit

In [71]:
equal_levels_freshness_v1 = audit_equal_levels_freshness_v1(
    equal_levels=equal_levels_retracement_v1,
    m5_df=m5_df,
)

print("=" * 100)
print("EQUAL LEVELS FRESHNESS / LIFECYCLE AUDIT V1")
print("=" * 100)

print("Total Candidates :", len(equal_levels_freshness_v1))

print("\nFormation Valid:")
print(
    equal_levels_freshness_v1[
        "formation_valid"
    ].value_counts(dropna=False)
)

print("\nRetracement Valid:")
print(
    equal_levels_freshness_v1[
        "retracement_valid"
    ].value_counts(dropna=False)
)

print("\nFreshness State:")
print(
    equal_levels_freshness_v1[
        "freshness_state"
    ].value_counts(dropna=False)
)

print("\nSweep Direction:")
print(
    equal_levels_freshness_v1[
        "sweep_direction"
    ].value_counts(dropna=False)
)

display(
    equal_levels_freshness_v1[
        [
            "swing_a",
            "swing_b",
            "equal_type",
            "level_price",
            "activation_time",
            "freshness_state",
            "sweep_time",
            "sweep_price",
            "sweep_direction",
        ]
    ]
)

EQUAL LEVELS FRESHNESS / LIFECYCLE AUDIT V1
Total Candidates : 10

Formation Valid:
formation_valid
True    10
Name: count, dtype: int64

Retracement Valid:
retracement_valid
True    10
Name: count, dtype: int64

Freshness State:
freshness_state
SWEPT     8
ACTIVE    2
Name: count, dtype: int64

Sweep Direction:
sweep_direction
BULLISH    5
BEARISH    3
NaN        2
Name: count, dtype: int64


,swing_a,swing_b,equal_type,level_price,activation_time,freshness_state,sweep_time,sweep_price,sweep_direction
0,2026-10-05 02:30:00+00:00,2026-10-05 03:00:00+00:00,EQH,1.125625,2026-10-05 03:00:00+00:00,SWEPT,2026-10-06 14:15:00+00:00,1.12571,BULLISH
1,2026-10-05 06:05:00+00:00,2026-10-05 07:05:00+00:00,EQL,1.116075,2026-10-05 07:05:00+00:00,ACTIVE,NaT,NaN,NaN
2,2026-10-05 15:30:00+00:00,2026-10-05 16:00:00+00:00,EQH,1.121385,2026-10-05 16:00:00+00:00,SWEPT,2026-10-05 18:05:00+00:00,1.12168,BULLISH
3,2026-10-05 23:20:00+00:00,2026-10-06 00:20:00+00:00,EQL,1.121565,2026-10-06 00:20:00+00:00,SWEPT,2026-10-06 03:00:00+00:00,1.12143,BEARISH
4,2026-10-06 05:40:00+00:00,2026-10-06 06:00:00+00:00,EQL,1.121205,2026-10-06 06:00:00+00:00,SWEPT,2026-10-06 08:50:00+00:00,1.12119,BEARISH
5,2026-10-06 11:40:00+00:00,2026-10-06 12:30:00+00:00,EQH,1.124705,2026-10-06 12:30:00+00:00,SWEPT,2026-10-06 12:50:00+00:00,1.12486,BULLISH
6,2026-10-06 16:45:00+00:00,2026-10-06 17:20:00+00:00,EQL,1.124685,2026-10-06 17:20:00+00:00,SWEPT,2026-10-06 17:40:00+00:00,1.12445,BEARISH
7,2026-10-07 23:30:00+00:00,2026-10-07 23:50:00+00:00,EQH,1.119670,2026-10-07 23:50:00+00:00,SWEPT,2026-10-08 01:00:00+00:00,1.11993,BULLISH
8,2026-10-08 01:25:00+00:00,2026-10-08 03:05:00+00:00,EQH,1.120255,2026-10-08 03:05:00+00:00,SWEPT,2026-10-08 04:40:00+00:00,1.12046,BULLISH
9,2026-10-08 04:30:00+00:00,2026-10-08 06:10:00+00:00,EQL,1.119750,2026-10-08 06:10:00+00:00,ACTIVE,NaT,NaN,NaN


In [72]:
import pandas as pd
import numpy as np


def build_equal_liquidity_registry_v1(
    equal_levels_freshness: pd.DataFrame,
) -> pd.DataFrame:

    required = {
        "swing_a",
        "swing_b",
        "price_a",
        "price_b",
        "equal_type",
        "formation_valid",
        "retracement_valid",
        "level_price",
        "activation_time",
        "freshness_state",
        "sweep_time",
        "sweep_price",
        "sweep_direction",
    }

    missing = required - set(equal_levels_freshness.columns)

    if missing:
        raise ValueError(
            f"Equal Liquidity missing columns: {sorted(missing)}"
        )

    df = equal_levels_freshness.copy()

    df["swing_a"] = pd.to_datetime(
        df["swing_a"],
        utc=True,
    )

    df["swing_b"] = pd.to_datetime(
        df["swing_b"],
        utc=True,
    )

    df["activation_time"] = pd.to_datetime(
        df["activation_time"],
        utc=True,
    )

    df["sweep_time"] = pd.to_datetime(
        df["sweep_time"],
        utc=True,
    )

    # ------------------------------------------------------------
    # Only fully validated Equal Levels enter the Registry.
    # ------------------------------------------------------------
    df = df[
        (df["formation_valid"] == True)
        & (df["retracement_valid"] == True)
    ].copy()

    df = df.sort_values(
        ["activation_time", "equal_type", "swing_a", "swing_b"]
    ).reset_index(drop=True)

    # ------------------------------------------------------------
    # Immutable Equal Level identity
    # ------------------------------------------------------------
    counters = {
        "EQH": 0,
        "EQL": 0,
    }

    level_ids = []

    for equal_type in df["equal_type"]:

        if equal_type not in counters:
            raise ValueError(
                f"Invalid equal_type: {equal_type}"
            )

        counters[equal_type] += 1

        level_ids.append(
            f"{equal_type}_{counters[equal_type]:03d}"
        )

    df["level_id"] = level_ids

    # ------------------------------------------------------------
    # Registry identity
    # ------------------------------------------------------------
    df["level_type"] = df["equal_type"]

    # ------------------------------------------------------------
    # Lifecycle
    # ------------------------------------------------------------
    df["state"] = df["freshness_state"]

    # ------------------------------------------------------------
    # Event identity
    # ------------------------------------------------------------
    df["event_id"] = np.where(
        df["state"].eq("SWEPT"),
        "SWEEP_" + df["level_id"],
        None,
    )

    df["event_type"] = np.where(
        df["state"].eq("SWEPT"),
        "LIQUIDITY_SWEEP",
        None,
    )

    # ------------------------------------------------------------
    # Contract checks before returning
    # ------------------------------------------------------------

    if df["level_id"].duplicated().any():
        raise AssertionError(
            "Duplicate Equal Liquidity level_id detected"
        )

    if df["event_id"].dropna().duplicated().any():
        raise AssertionError(
            "Duplicate Equal Liquidity event_id detected"
        )

    invalid_state = ~df["state"].isin(
        ["ACTIVE", "SWEPT"]
    )

    if invalid_state.any():
        raise AssertionError(
            "Invalid Equal Liquidity lifecycle state"
        )

    swept = df["state"].eq("SWEPT")

    if df.loc[swept, "sweep_time"].isna().any():
        raise AssertionError(
            "SWEPT level without sweep_time"
        )

    if df.loc[swept, "sweep_price"].isna().any():
        raise AssertionError(
            "SWEPT level without sweep_price"
        )

    active = df["state"].eq("ACTIVE")

    if df.loc[active, "sweep_time"].notna().any():
        raise AssertionError(
            "ACTIVE level has sweep_time"
        )

    if df.loc[active, "event_id"].notna().any():
        raise AssertionError(
            "ACTIVE level has event_id"
        )

    # No-lookahead lifecycle contract
    invalid_time = (
        swept
        & (
            df["sweep_time"]
            <= df["activation_time"]
        )
    )

    if invalid_time.any():
        raise AssertionError(
            "Sweep occurs at/before activation_time"
        )

    return df[
        [
            "level_id",
            "level_type",
            "level_price",
            "swing_a",
            "swing_b",
            "activation_time",
            "state",
            "sweep_time",
            "sweep_price",
            "sweep_direction",
            "event_id",
            "event_type",
        ]
    ].copy()

In [73]:
equal_liquidity_registry_v1 = build_equal_liquidity_registry_v1(
    equal_levels_freshness=equal_levels_freshness_v1,
)

print("=" * 100)
print("EQUAL LIQUIDITY REGISTRY V1")
print("=" * 100)

print("Total Levels :", len(equal_liquidity_registry_v1))

print("\nLevel Type:")
print(
    equal_liquidity_registry_v1[
        "level_type"
    ].value_counts()
)

print("\nLifecycle State:")
print(
    equal_liquidity_registry_v1[
        "state"
    ].value_counts()
)

print("\nEvent Type:")
print(
    equal_liquidity_registry_v1[
        "event_type"
    ].value_counts(dropna=False)
)

display(equal_liquidity_registry_v1)

EQUAL LIQUIDITY REGISTRY V1
Total Levels : 10

Level Type:
level_type
EQH    5
EQL    5
Name: count, dtype: int64

Lifecycle State:
state
SWEPT     8
ACTIVE    2
Name: count, dtype: int64

Event Type:
event_type
LIQUIDITY_SWEEP    8
NaN                2
Name: count, dtype: int64


,level_id,level_type,level_price,swing_a,swing_b,activation_time,state,sweep_time,sweep_price,sweep_direction,event_id,event_type
0,EQH_001,EQH,1.125625,2026-10-05 02:30:00+00:00,2026-10-05 03:00:00+00:00,2026-10-05 03:00:00+00:00,SWEPT,2026-10-06 14:15:00+00:00,1.12571,BULLISH,SWEEP_EQH_001,LIQUIDITY_SWEEP
1,EQL_001,EQL,1.116075,2026-10-05 06:05:00+00:00,2026-10-05 07:05:00+00:00,2026-10-05 07:05:00+00:00,ACTIVE,NaT,NaN,NaN,NaN,NaN
2,EQH_002,EQH,1.121385,2026-10-05 15:30:00+00:00,2026-10-05 16:00:00+00:00,2026-10-05 16:00:00+00:00,SWEPT,2026-10-05 18:05:00+00:00,1.12168,BULLISH,SWEEP_EQH_002,LIQUIDITY_SWEEP
3,EQL_002,EQL,1.121565,2026-10-05 23:20:00+00:00,2026-10-06 00:20:00+00:00,2026-10-06 00:20:00+00:00,SWEPT,2026-10-06 03:00:00+00:00,1.12143,BEARISH,SWEEP_EQL_002,LIQUIDITY_SWEEP
4,EQL_003,EQL,1.121205,2026-10-06 05:40:00+00:00,2026-10-06 06:00:00+00:00,2026-10-06 06:00:00+00:00,SWEPT,2026-10-06 08:50:00+00:00,1.12119,BEARISH,SWEEP_EQL_003,LIQUIDITY_SWEEP
5,EQH_003,EQH,1.124705,2026-10-06 11:40:00+00:00,2026-10-06 12:30:00+00:00,2026-10-06 12:30:00+00:00,SWEPT,2026-10-06 12:50:00+00:00,1.12486,BULLISH,SWEEP_EQH_003,LIQUIDITY_SWEEP
6,EQL_004,EQL,1.124685,2026-10-06 16:45:00+00:00,2026-10-06 17:20:00+00:00,2026-10-06 17:20:00+00:00,SWEPT,2026-10-06 17:40:00+00:00,1.12445,BEARISH,SWEEP_EQL_004,LIQUIDITY_SWEEP
7,EQH_004,EQH,1.119670,2026-10-07 23:30:00+00:00,2026-10-07 23:50:00+00:00,2026-10-07 23:50:00+00:00,SWEPT,2026-10-08 01:00:00+00:00,1.11993,BULLISH,SWEEP_EQH_004,LIQUIDITY_SWEEP
8,EQH_005,EQH,1.120255,2026-10-08 01:25:00+00:00,2026-10-08 03:05:00+00:00,2026-10-08 03:05:00+00:00,SWEPT,2026-10-08 04:40:00+00:00,1.12046,BULLISH,SWEEP_EQH_005,LIQUIDITY_SWEEP
9,EQL_005,EQL,1.119750,2026-10-08 04:30:00+00:00,2026-10-08 06:10:00+00:00,2026-10-08 06:10:00+00:00,ACTIVE,NaT,NaN,NaN,NaN,NaN


In [74]:
import pandas as pd


def regression_equal_liquidity_registry_v1(
    registry: pd.DataFrame,
    source: pd.DataFrame,
):

    print("=" * 100)
    print("EQUAL LIQUIDITY REGISTRY V1 REGRESSION GATE")
    print("=" * 100)

    checks = {}

    # ============================================================
    # 1. ROW COUNT
    # ============================================================
    expected_rows = int(
        (
            source["formation_valid"]
            & source["retracement_valid"]
        ).sum()
    )

    checks["ROW COUNT"] = (
        len(registry) == expected_rows
    )

    # ============================================================
    # 2. LEVEL IDENTITY
    # ============================================================
    checks["LEVEL ID UNIQUENESS"] = (
        registry["level_id"].notna().all()
        and registry["level_id"].is_unique
    )

    # ============================================================
    # 3. LEVEL TYPE
    # ============================================================
    checks["LEVEL TYPE CONTRACT"] = (
        registry["level_type"]
        .isin(["EQH", "EQL"])
        .all()
    )

    # ============================================================
    # 4. LEVEL PRICE
    # ============================================================
    checks["LEVEL PRICE IMMUTABILITY"] = (
        registry["level_price"].notna().all()
    )

    # ============================================================
    # 5. LIFECYCLE
    # ============================================================
    checks["LIFECYCLE CONTRACT"] = (
        registry["state"]
        .isin(["ACTIVE", "SWEPT"])
        .all()
    )

    # ============================================================
    # 6. SWEPT CONTRACT
    # ============================================================
    swept = registry["state"].eq("SWEPT")

    checks["SWEPT CONTRACT"] = (
        registry.loc[swept, "sweep_time"].notna().all()
        and
        registry.loc[swept, "sweep_price"].notna().all()
        and
        registry.loc[swept, "sweep_direction"].notna().all()
        and
        registry.loc[swept, "event_id"].notna().all()
        and
        registry.loc[swept, "event_type"]
        .eq("LIQUIDITY_SWEEP")
        .all()
    )

    # ============================================================
    # 7. ACTIVE CONTRACT
    # ============================================================
    active = registry["state"].eq("ACTIVE")

    checks["ACTIVE CONTRACT"] = (
        registry.loc[active, "sweep_time"].isna().all()
        and
        registry.loc[active, "sweep_price"].isna().all()
        and
        registry.loc[active, "sweep_direction"].isna().all()
        and
        registry.loc[active, "event_id"].isna().all()
        and
        registry.loc[active, "event_type"].isna().all()
    )

    # ============================================================
    # 8. EVENT IDENTITY
    # ============================================================
    event_ids = registry["event_id"].dropna()

    checks["EVENT IDENTITY"] = (
        event_ids.is_unique
    )

    # ============================================================
    # 9. EVENT ↔ LEVEL IDENTITY
    # ============================================================
    expected_event_ids = (
        "SWEEP_"
        + registry.loc[swept, "level_id"]
    )

    checks["EVENT LEVEL IDENTITY"] = (
        registry.loc[swept, "event_id"]
        .reset_index(drop=True)
        .equals(
            expected_event_ids.reset_index(drop=True)
        )
    )

    # ============================================================
    # 10. UTC TIMESTAMPS
    # ============================================================
    timestamp_columns = [
        "swing_a",
        "swing_b",
        "activation_time",
        "sweep_time",
    ]

    utc_ok = True

    for col in timestamp_columns:
        if not isinstance(
            registry[col].dtype,
            pd.DatetimeTZDtype,
        ):
            utc_ok = False
            break

        if (
            registry[col].notna().any()
            and str(registry[col].dt.tz) != "UTC"
        ):
            utc_ok = False
            break

    checks["UTC TIMESTAMPS"] = utc_ok

    # ============================================================
    # 11. NO LOOK-AHEAD
    # ============================================================
    if swept.any():
        checks["NO LOOK-AHEAD"] = (
            (
                registry.loc[swept, "sweep_time"]
                >
                registry.loc[swept, "activation_time"]
            )
            .all()
        )
    else:
        checks["NO LOOK-AHEAD"] = True

    # ============================================================
    # 12. EQH SWEEP DIRECTION
    # ============================================================
    eqh_swept = swept & registry["level_type"].eq("EQH")

    checks["EQH SWEEP CONTRACT"] = (
        registry.loc[eqh_swept, "sweep_direction"]
        .eq("BULLISH")
        .all()
    )

    # ============================================================
    # 13. EQL SWEEP DIRECTION
    # ============================================================
    eql_swept = swept & registry["level_type"].eq("EQL")

    checks["EQL SWEEP CONTRACT"] = (
        registry.loc[eql_swept, "sweep_direction"]
        .eq("BEARISH")
        .all()
    )

    # ============================================================
    # RESULT
    # ============================================================
    for name, passed in checks.items():

        print(
            f"{name:<32}: "
            f"{'PASSED' if passed else 'FAILED'}"
        )

    if not all(checks.values()):
        failed = [
            name
            for name, passed in checks.items()
            if not passed
        ]

        raise AssertionError(
            "❌ EQUAL LIQUIDITY REGISTRY V1 FAILED: "
            + ", ".join(failed)
        )

    print()
    print("✅ EQUAL LIQUIDITY REGISTRY V1 PASSED")

    return checks

In [75]:
equal_liquidity_registry_v1_gate = (
    regression_equal_liquidity_registry_v1(
        registry=equal_liquidity_registry_v1,
        source=equal_levels_freshness_v1,
    )
)

EQUAL LIQUIDITY REGISTRY V1 REGRESSION GATE
ROW COUNT                       : PASSED
LEVEL ID UNIQUENESS             : PASSED
LEVEL TYPE CONTRACT             : PASSED
LEVEL PRICE IMMUTABILITY        : PASSED
LIFECYCLE CONTRACT              : PASSED
SWEPT CONTRACT                  : PASSED
ACTIVE CONTRACT                 : PASSED
EVENT IDENTITY                  : PASSED
EVENT LEVEL IDENTITY            : PASSED
UTC TIMESTAMPS                  : PASSED
NO LOOK-AHEAD                   : PASSED
EQH SWEEP CONTRACT              : PASSED
EQL SWEEP CONTRACT              : PASSED

✅ EQUAL LIQUIDITY REGISTRY V1 PASSED


In [76]:
import pandas as pd


def build_unified_liquidity_registry_v1(
    protected_registry: pd.DataFrame,
    equal_registry: pd.DataFrame,
) -> pd.DataFrame:

    protected_required = {
        "level_id",
        "type",
        "level",
        "created_at",
        "state",
        "break_time",
        "break_price",
        "break_direction",
    }

    equal_required = {
        "level_id",
        "level_type",
        "level_price",
        "activation_time",
        "state",
        "sweep_time",
        "sweep_price",
        "sweep_direction",
        "event_id",
        "event_type",
    }

    missing_protected = (
        protected_required
        - set(protected_registry.columns)
    )

    missing_equal = (
        equal_required
        - set(equal_registry.columns)
    )

    if missing_protected:
        raise ValueError(
            "Protected Registry missing columns: "
            + str(sorted(missing_protected))
        )

    if missing_equal:
        raise ValueError(
            "Equal Registry missing columns: "
            + str(sorted(missing_equal))
        )

    # ============================================================
    # PROTECTED
    # ============================================================

    p = protected_registry.copy()

    p = p.rename(
        columns={
            "type": "liquidity_type",
            "level": "level_price",
            "created_at": "activation_time",
            "break_time": "event_time",
            "break_price": "event_price",
            "break_direction": "event_direction",
        }
    )

    p["liquidity_source"] = "PROTECTED"

    p["event_id"] = pd.NA
    p["event_type"] = pd.NA

    protected_broken = p["state"].eq("BROKEN")

    p.loc[protected_broken, "event_id"] = (
        "BREAK_" + p.loc[protected_broken, "level_id"]
    )

    p.loc[protected_broken, "event_type"] = (
        "STRUCTURAL_BREAK"
    )

    # ============================================================
    # EQUAL
    # ============================================================

    e = equal_registry.copy()

    e = e.rename(
        columns={
            "level_type": "liquidity_type",
            "sweep_time": "event_time",
            "sweep_price": "event_price",
            "sweep_direction": "event_direction",
        }
    )

    e["liquidity_source"] = "EQUAL"

    # ============================================================
    # Canonical column set
    # ============================================================

    columns = [
        "level_id",
        "liquidity_source",
        "liquidity_type",
        "level_price",
        "activation_time",
        "state",
        "event_time",
        "event_price",
        "event_direction",
        "event_id",
        "event_type",
    ]

    p = p[columns]
    e = e[columns]

    # ============================================================
    # Identity collision protection
    # ============================================================

    overlap = set(p["level_id"]) & set(e["level_id"])

    if overlap:
        raise AssertionError(
            "Duplicate Level IDs across liquidity sources: "
            + str(sorted(overlap))
        )

    # ============================================================
    # Unified registry
    # ============================================================

    unified = pd.concat(
        [p, e],
        ignore_index=True,
    )

    unified = unified.sort_values(
        ["activation_time", "level_id"]
    ).reset_index(drop=True)

    # ============================================================
    # Immutable identity
    # ============================================================

    if unified["level_id"].duplicated().any():
        raise AssertionError(
            "Duplicate unified level_id detected"
        )

    if unified["event_id"].dropna().duplicated().any():
        raise AssertionError(
            "Duplicate unified event_id detected"
        )

    return unified

In [77]:
unified_liquidity_registry_v1 = (
    build_unified_liquidity_registry_v1(
        protected_registry=protected_registry_v1,
        equal_registry=equal_liquidity_registry_v1,
    )
)

print("=" * 100)
print("UNIFIED LIQUIDITY REGISTRY V1")
print("=" * 100)

print("Total Levels :", len(unified_liquidity_registry_v1))

print("\nSource:")
print(
    unified_liquidity_registry_v1[
        "liquidity_source"
    ].value_counts()
)

print("\nLifecycle:")
print(
    unified_liquidity_registry_v1[
        "state"
    ].value_counts()
)

UNIFIED LIQUIDITY REGISTRY V1
Total Levels : 61

Source:
liquidity_source
PROTECTED    51
EQUAL        10
Name: count, dtype: int64

Lifecycle:
state
BROKEN    39
ACTIVE    14
SWEPT      8
Name: count, dtype: int64


In [78]:
def regression_unified_liquidity_registry_v1(
    unified_registry,
    protected_registry,
    equal_registry,
):

    print("=" * 100)
    print("UNIFIED LIQUIDITY REGISTRY V1 REGRESSION GATE")
    print("=" * 100)

    checks = {}

    # ============================================================
    # EXPECTED COUNTS
    # ============================================================

    expected_protected = len(protected_registry)
    expected_equal = len(equal_registry)
    expected_total = (
        expected_protected
        + expected_equal
    )

    actual_protected = (
        unified_registry["liquidity_source"]
        .eq("PROTECTED")
        .sum()
    )

    actual_equal = (
        unified_registry["liquidity_source"]
        .eq("EQUAL")
        .sum()
    )

    # ============================================================
    # 1. ROW COUNTS
    # ============================================================

    checks["PROTECTED ROW COUNT"] = (
        actual_protected == expected_protected
    )

    checks["EQUAL ROW COUNT"] = (
        actual_equal == expected_equal
    )

    checks["TOTAL ROW COUNT"] = (
        len(unified_registry) == expected_total
    )

    # ============================================================
    # 2. LEVEL ID UNIQUENESS
    # ============================================================

    checks["LEVEL ID UNIQUENESS"] = (
        unified_registry["level_id"].notna().all()
        and unified_registry["level_id"].is_unique
    )

    # ============================================================
    # 3. PROTECTED ID PRESERVATION
    # ============================================================

    protected_source_ids = set(
        protected_registry["level_id"]
    )

    protected_unified_ids = set(
        unified_registry.loc[
            unified_registry["liquidity_source"]
            == "PROTECTED",
            "level_id",
        ]
    )

    checks["PROTECTED ID PRESERVATION"] = (
        protected_source_ids
        == protected_unified_ids
    )

    # ============================================================
    # 4. EQUAL ID PRESERVATION
    # ============================================================

    equal_source_ids = set(
        equal_registry["level_id"]
    )

    equal_unified_ids = set(
        unified_registry.loc[
            unified_registry["liquidity_source"]
            == "EQUAL",
            "level_id",
        ]
    )

    checks["EQUAL ID PRESERVATION"] = (
        equal_source_ids
        == equal_unified_ids
    )

    # ============================================================
    # 5. NO UNEXPECTED IDS
    # ============================================================

    expected_ids = (
        protected_source_ids
        | equal_source_ids
    )

    actual_ids = set(
        unified_registry["level_id"]
    )

    checks["NO UNEXPECTED IDS"] = (
        actual_ids == expected_ids
    )

    # ============================================================
    # 6. NO CROSS-SOURCE COLLISION
    # ============================================================

    checks["NO CROSS-SOURCE ID COLLISION"] = (
        protected_source_ids.isdisjoint(
            equal_source_ids
        )
    )

    # ============================================================
    # 7. PROTECTED IDENTITY IMMUTABILITY
    # ============================================================

    p = (
        protected_registry
        .set_index("level_id")
        .copy()
    )

    u_p = (
        unified_registry.loc[
            unified_registry["liquidity_source"]
            == "PROTECTED"
        ]
        .set_index("level_id")
    )

    protected_identity_ok = True

    for level_id in protected_source_ids:

        source_row = p.loc[level_id]
        unified_row = u_p.loc[level_id]

        if source_row["type"] != unified_row["liquidity_type"]:
            protected_identity_ok = False
            break

        if source_row["level"] != unified_row["level_price"]:
            protected_identity_ok = False
            break

        if source_row["created_at"] != unified_row["activation_time"]:
            protected_identity_ok = False
            break

    checks["PROTECTED IDENTITY IMMUTABILITY"] = (
        protected_identity_ok
    )

    # ============================================================
    # 8. EQUAL IDENTITY IMMUTABILITY
    # ============================================================

    e = (
        equal_registry
        .set_index("level_id")
        .copy()
    )

    u_e = (
        unified_registry.loc[
            unified_registry["liquidity_source"]
            == "EQUAL"
        ]
        .set_index("level_id")
    )

    equal_identity_ok = True

    for level_id in equal_source_ids:

        source_row = e.loc[level_id]
        unified_row = u_e.loc[level_id]

        if source_row["level_type"] != unified_row["liquidity_type"]:
            equal_identity_ok = False
            break

        if source_row["level_price"] != unified_row["level_price"]:
            equal_identity_ok = False
            break

        if source_row["activation_time"] != unified_row["activation_time"]:
            equal_identity_ok = False
            break

    checks["EQUAL IDENTITY IMMUTABILITY"] = (
        equal_identity_ok
    )

    # ============================================================
    # 9. PROTECTED LIFECYCLE
    # ============================================================

    protected_unified = unified_registry[
        unified_registry["liquidity_source"]
        == "PROTECTED"
    ]

    checks["PROTECTED LIFECYCLE"] = (
        protected_unified["state"]
        .isin(["ACTIVE", "BROKEN"])
        .all()
    )

    # ============================================================
    # 10. EQUAL LIFECYCLE
    # ============================================================

    equal_unified = unified_registry[
        unified_registry["liquidity_source"]
        == "EQUAL"
    ]

    checks["EQUAL LIFECYCLE"] = (
        equal_unified["state"]
        .isin(["ACTIVE", "SWEPT"])
        .all()
    )

    # ============================================================
    # 11. BROKEN CONTRACT
    # ============================================================

    broken = (
        unified_registry["state"]
        == "BROKEN"
    )

    checks["BROKEN CONTRACT"] = (
        unified_registry.loc[
            broken,
            "event_time",
        ].notna().all()
        and
        unified_registry.loc[
            broken,
            "event_price",
        ].notna().all()
        and
        unified_registry.loc[
            broken,
            "event_direction",
        ].notna().all()
    )

    # ============================================================
    # 12. SWEPT CONTRACT
    # ============================================================

    swept = (
        unified_registry["state"]
        == "SWEPT"
    )

    checks["SWEPT CONTRACT"] = (
        unified_registry.loc[
            swept,
            "event_time",
        ].notna().all()
        and
        unified_registry.loc[
            swept,
            "event_price",
        ].notna().all()
        and
        unified_registry.loc[
            swept,
            "event_direction",
        ].notna().all()
        and
        unified_registry.loc[
            swept,
            "event_id",
        ].notna().all()
        and
        unified_registry.loc[
            swept,
            "event_type",
        ].notna().all()
    )

    # ============================================================
    # 13. ACTIVE CONTRACT
    # ============================================================

    active = (
        unified_registry["state"]
        == "ACTIVE"
    )

    checks["ACTIVE CONTRACT"] = (
        unified_registry.loc[
            active,
            "event_time",
        ].isna().all()
        and
        unified_registry.loc[
            active,
            "event_price",
        ].isna().all()
        and
        unified_registry.loc[
            active,
            "event_direction",
        ].isna().all()
        and
        unified_registry.loc[
            active,
            "event_id",
        ].isna().all()
        and
        unified_registry.loc[
            active,
            "event_type",
        ].isna().all()
    )

    # ============================================================
    # 14. EVENT IDENTITY
    # ============================================================

    event_ids = (
        unified_registry["event_id"]
        .dropna()
    )

    checks["EVENT ID UNIQUENESS"] = (
        event_ids.is_unique
    )

    # ============================================================
    # 15. EVENT TIME > ACTIVATION
    # ============================================================

    event_rows = (
        unified_registry["event_time"]
        .notna()
    )

    checks["NO LOOK-AHEAD"] = (
        (
            unified_registry.loc[
                event_rows,
                "event_time",
            ]
            >
            unified_registry.loc[
                event_rows,
                "activation_time",
            ]
        )
        .all()
    )

    # ============================================================
    # 16. UTC CONTRACT
    # ============================================================

    utc_columns = [
        "activation_time",
        "event_time",
    ]

    utc_ok = True

    for col in utc_columns:

        dtype = unified_registry[col].dtype

        if not isinstance(
            dtype,
            pd.DatetimeTZDtype,
        ):
            utc_ok = False
            break

        if (
            unified_registry[col]
            .notna()
            .any()
            and
            str(
                unified_registry[col]
                .dt.tz
            ) != "UTC"
        ):
            utc_ok = False
            break

    checks["UTC TIMESTAMP CONTRACT"] = utc_ok

    # ============================================================
    # OUTPUT
    # ============================================================

    for name, passed in checks.items():

        print(
            f"{name:<34}: "
            f"{'PASSED' if passed else 'FAILED'}"
        )

    print()

    if not all(checks.values()):

        failed = [
            name
            for name, passed in checks.items()
            if not passed
        ]

        raise AssertionError(
            "❌ UNIFIED LIQUIDITY REGISTRY V1 FAILED: "
            + ", ".join(failed)
        )

    print(
        "✅ UNIFIED LIQUIDITY REGISTRY V1 PASSED"
    )

    return checks

In [79]:
unified_liquidity_registry_v1_gate = (
    regression_unified_liquidity_registry_v1(
        unified_registry=unified_liquidity_registry_v1,
        protected_registry=protected_registry_v1,
        equal_registry=equal_liquidity_registry_v1,
    )
)

UNIFIED LIQUIDITY REGISTRY V1 REGRESSION GATE
PROTECTED ROW COUNT               : PASSED
EQUAL ROW COUNT                   : PASSED
TOTAL ROW COUNT                   : PASSED
LEVEL ID UNIQUENESS               : PASSED
PROTECTED ID PRESERVATION         : PASSED
EQUAL ID PRESERVATION             : PASSED
NO UNEXPECTED IDS                 : PASSED
NO CROSS-SOURCE ID COLLISION      : PASSED
PROTECTED IDENTITY IMMUTABILITY   : PASSED
EQUAL IDENTITY IMMUTABILITY       : PASSED
PROTECTED LIFECYCLE               : PASSED
EQUAL LIFECYCLE                   : PASSED
BROKEN CONTRACT                   : PASSED
SWEPT CONTRACT                    : PASSED
ACTIVE CONTRACT                   : PASSED
EVENT ID UNIQUENESS               : PASSED
NO LOOK-AHEAD                     : PASSED
UTC TIMESTAMP CONTRACT            : PASSED

✅ UNIFIED LIQUIDITY REGISTRY V1 PASSED


In [80]:
print("PROTECTED REGISTRY COLUMNS")
print(protected_registry_v1.columns.tolist())

print("\nEQUAL REGISTRY COLUMNS")
print(equal_liquidity_registry_v1.columns.tolist())

print("\nPROTECTED SAMPLE")
display(protected_registry_v1.head(3))

print("\nEQUAL SAMPLE")
display(equal_liquidity_registry_v1.head(3))

PROTECTED REGISTRY COLUMNS
['level_id', 'type', 'level', 'created_at', 'state', 'break_time', 'break_price', 'break_direction']

EQUAL REGISTRY COLUMNS
['level_id', 'level_type', 'level_price', 'swing_a', 'swing_b', 'activation_time', 'state', 'sweep_time', 'sweep_price', 'sweep_direction', 'event_id', 'event_type']

PROTECTED SAMPLE


,level_id,type,level,created_at,state,break_time,break_price,break_direction
0,PH_001,PROTECTED_HIGH,1.12594,2026-10-02 23:00:00+00:00,BROKEN,2026-10-05 01:45:00+00:00,1.12595,BOS_BULLISH
1,PH_002,PROTECTED_HIGH,1.12553,2026-10-02 23:30:00+00:00,BROKEN,2026-10-05 00:15:00+00:00,1.12578,BOS_BULLISH
2,PL_001,PROTECTED_LOW,1.12563,2026-10-05 01:30:00+00:00,BROKEN,2026-10-05 02:05:00+00:00,1.12562,BOS_BEARISH



EQUAL SAMPLE


,level_id,level_type,level_price,swing_a,swing_b,activation_time,state,sweep_time,sweep_price,sweep_direction,event_id,event_type
0,EQH_001,EQH,1.125625,2026-10-05 02:30:00+00:00,2026-10-05 03:00:00+00:00,2026-10-05 03:00:00+00:00,SWEPT,2026-10-06 14:15:00+00:00,1.12571,BULLISH,SWEEP_EQH_001,LIQUIDITY_SWEEP
1,EQL_001,EQL,1.116075,2026-10-05 06:05:00+00:00,2026-10-05 07:05:00+00:00,2026-10-05 07:05:00+00:00,ACTIVE,NaT,NaN,NaN,NaN,NaN
2,EQH_002,EQH,1.121385,2026-10-05 15:30:00+00:00,2026-10-05 16:00:00+00:00,2026-10-05 16:00:00+00:00,SWEPT,2026-10-05 18:05:00+00:00,1.12168,BULLISH,SWEEP_EQH_002,LIQUIDITY_SWEEP


In [81]:
import pandas as pd


def build_unified_liquidity_event_timeline_v1(
    protected_registry_v1,
    equal_liquidity_registry_v1,
):
    """
    Unified Liquidity Event Timeline V1

    Source of truth:
        UNIFIED LIQUIDITY REGISTRY V1

    Design principles:
        - No lifecycle inference
        - No new level IDs
        - No mutation of source registries
        - No look-ahead
        - UTC timestamps only
        - Deterministic event IDs
        - Full source traceability
    """

    # ==========================================================
    # 1. REQUIRED COLUMNS
    # ==========================================================

    protected_required = {
        "level_id",
        "type",
        "level",
        "created_at",
        "state",
        "break_time",
        "break_price",
        "break_direction",
    }

    equal_required = {
        "level_id",
        "level_type",
        "level_price",
        "swing_a",
        "swing_b",
        "activation_time",
        "state",
        "sweep_time",
        "sweep_price",
        "sweep_direction",
        "event_id",
        "event_type",
    }

    missing_protected = (
        protected_required
        - set(protected_registry_v1.columns)
    )

    missing_equal = (
        equal_required
        - set(equal_liquidity_registry_v1.columns)
    )

    if missing_protected:
        raise ValueError(
            f"Protected Registry missing columns: "
            f"{sorted(missing_protected)}"
        )

    if missing_equal:
        raise ValueError(
            f"Equal Registry missing columns: "
            f"{sorted(missing_equal)}"
        )

    events = []

    # ==========================================================
    # 2. PROTECTED LIQUIDITY EVENTS
    # ==========================================================

    for _, row in protected_registry_v1.iterrows():

        level_id = row["level_id"]
        subtype = row["type"]
        level_price = float(row["level"])
        created_at = pd.Timestamp(row["created_at"])
        state = row["state"]

        if subtype not in {
            "PROTECTED_HIGH",
            "PROTECTED_LOW",
        }:
            raise ValueError(
                f"Unexpected protected type: {subtype}"
            )

        # ------------------------------------------------------
        # FORMATION
        # ------------------------------------------------------

        events.append({
            "event_id": f"FORMATION_{level_id}",
            "level_id": level_id,
            "liquidity_type": "PROTECTED",
            "liquidity_subtype": subtype,
            "event_type": "LIQUIDITY_FORMATION",
            "event_time": created_at,
            "event_price": level_price,
            "event_direction": "NEUTRAL",
            "source_created_at": created_at,
            "source_state": state,
            "source_event_id": pd.NA,
        })

        # ------------------------------------------------------
        # BROKEN
        # ------------------------------------------------------

        if state == "BROKEN":

            if pd.isna(row["break_time"]):
                raise ValueError(
                    f"BROKEN level {level_id} has no break_time"
                )

            if pd.isna(row["break_price"]):
                raise ValueError(
                    f"BROKEN level {level_id} has no break_price"
                )

            break_time = pd.Timestamp(row["break_time"])
            break_price = float(row["break_price"])
            break_direction = row["break_direction"]

            if break_time < created_at:
                raise ValueError(
                    f"Look-ahead violation for {level_id}"
                )

            events.append({
                "event_id": f"BROKEN_{level_id}",
                "level_id": level_id,
                "liquidity_type": "PROTECTED",
                "liquidity_subtype": subtype,
                "event_type": "LIQUIDITY_BROKEN",
                "event_time": break_time,
                "event_price": break_price,
                "event_direction": break_direction,
                "source_created_at": created_at,
                "source_state": state,
                "source_event_id": pd.NA,
            })

        # ------------------------------------------------------
        # ACTIVE
        # ------------------------------------------------------

        elif state == "ACTIVE":
            pass

        else:
            raise ValueError(
                f"Unexpected Protected lifecycle state "
                f"{state!r} for {level_id}"
            )

    # ==========================================================
    # 3. EQUAL LIQUIDITY EVENTS
    # ==========================================================

    for _, row in equal_liquidity_registry_v1.iterrows():

        level_id = row["level_id"]
        subtype = row["level_type"]
        level_price = float(row["level_price"])

        swing_a = pd.Timestamp(row["swing_a"])
        swing_b = pd.Timestamp(row["swing_b"])
        activation_time = pd.Timestamp(row["activation_time"])

        state = row["state"]

        source_event_id = row["event_id"]
        source_event_type = row["event_type"]

        # ------------------------------------------------------
        # TYPE CONTRACT
        # ------------------------------------------------------

        if subtype not in {"EQH", "EQL"}:
            raise ValueError(
                f"Unexpected equal liquidity type: {subtype}"
            )

        # ------------------------------------------------------
        # ACTIVATION / FORMATION
        # ------------------------------------------------------

        if activation_time < swing_a:
            raise ValueError(
                f"Activation before swing_a: {level_id}"
            )

        if activation_time < swing_b:
            raise ValueError(
                f"Activation before swing_b: {level_id}"
            )

        events.append({
            "event_id": f"FORMATION_{level_id}",
            "level_id": level_id,
            "liquidity_type": "EQUAL",
            "liquidity_subtype": subtype,
            "event_type": "LIQUIDITY_FORMATION",
            "event_time": activation_time,
            "event_price": level_price,
            "event_direction": "NEUTRAL",
            "source_created_at": activation_time,
            "source_state": state,
            "source_event_id": source_event_id,
        })

        # ------------------------------------------------------
        # SWEEP
        # ------------------------------------------------------

        if state == "SWEPT":

            if pd.isna(row["sweep_time"]):
                raise ValueError(
                    f"SWEPT level {level_id} has no sweep_time"
                )

            if pd.isna(row["sweep_price"]):
                raise ValueError(
                    f"SWEPT level {level_id} has no sweep_price"
                )

            sweep_time = pd.Timestamp(row["sweep_time"])
            sweep_price = float(row["sweep_price"])
            sweep_direction = row["sweep_direction"]

            if sweep_time < activation_time:
                raise ValueError(
                    f"Look-ahead violation for {level_id}"
                )

            events.append({
                "event_id": source_event_id,
                "level_id": level_id,
                "liquidity_type": "EQUAL",
                "liquidity_subtype": subtype,
                "event_type": source_event_type,
                "event_time": sweep_time,
                "event_price": sweep_price,
                "event_direction": sweep_direction,
                "source_created_at": activation_time,
                "source_state": state,
                "source_event_id": source_event_id,
            })

        elif state == "ACTIVE":
            pass

        else:
            raise ValueError(
                f"Unexpected Equal lifecycle state "
                f"{state!r} for {level_id}"
            )

    # ==========================================================
    # 4. BUILD TIMELINE
    # ==========================================================

    timeline = pd.DataFrame(events)

    if timeline.empty:
        raise ValueError(
            "Unified Liquidity Event Timeline is empty."
        )

    # ==========================================================
    # 5. UTC CONTRACT
    # ==========================================================

    timeline["event_time"] = pd.to_datetime(
        timeline["event_time"],
        utc=True,
    )

    timeline["source_created_at"] = pd.to_datetime(
        timeline["source_created_at"],
        utc=True,
    )

    # ==========================================================
    # 6. EVENT ID UNIQUENESS
    # ==========================================================

    if timeline["event_id"].duplicated().any():

        duplicated_ids = (
            timeline.loc[
                timeline["event_id"].duplicated(keep=False),
                "event_id"
            ]
            .tolist()
        )

        raise ValueError(
            f"Duplicate event_id detected: {duplicated_ids}"
        )

    # ==========================================================
    # 7. LEVEL ID VALIDITY
    # ==========================================================

    protected_ids = set(
        protected_registry_v1["level_id"]
    )

    equal_ids = set(
        equal_liquidity_registry_v1["level_id"]
    )

    valid_level_ids = protected_ids | equal_ids

    unknown_ids = (
        set(timeline["level_id"])
        - valid_level_ids
    )

    if unknown_ids:
        raise ValueError(
            f"Unknown level IDs in timeline: "
            f"{sorted(unknown_ids)}"
        )

    # ==========================================================
    # 8. SOURCE TYPE CONSISTENCY
    # ==========================================================

    protected_mask = (
        timeline["liquidity_type"] == "PROTECTED"
    )

    equal_mask = (
        timeline["liquidity_type"] == "EQUAL"
    )

    if not set(
        timeline.loc[protected_mask, "level_id"]
    ).issubset(protected_ids):

        raise ValueError(
            "Protected timeline contains non-protected IDs."
        )

    if not set(
        timeline.loc[equal_mask, "level_id"]
    ).issubset(equal_ids):

        raise ValueError(
            "Equal timeline contains non-equal IDs."
        )

    # ==========================================================
    # 9. NO LOOK-AHEAD
    # ==========================================================

    if (
        timeline["event_time"]
        < timeline["source_created_at"]
    ).any():

        raise ValueError(
            "Timeline contains event before level creation."
        )

    # ==========================================================
    # 10. CHRONOLOGICAL ORDER
    # ==========================================================

    timeline = timeline.sort_values(
        by=[
            "event_time",
            "event_id",
        ],
        kind="stable",
    ).reset_index(drop=True)

    # ==========================================================
    # 11. FINAL COLUMN ORDER
    # ==========================================================

    timeline = timeline[
        [
            "event_id",
            "level_id",
            "liquidity_type",
            "liquidity_subtype",
            "event_type",
            "event_time",
            "event_price",
            "event_direction",
            "source_created_at",
            "source_state",
            "source_event_id",
        ]
    ]

    return timeline

In [82]:
unified_liquidity_event_timeline_v1 = (
    build_unified_liquidity_event_timeline_v1(
        protected_registry_v1=protected_registry_v1,
        equal_liquidity_registry_v1=equal_liquidity_registry_v1,
    )
)

display(
    unified_liquidity_event_timeline_v1.head(20)
)

,event_id,level_id,liquidity_type,liquidity_subtype,event_type,event_time,event_price,event_direction,source_created_at,source_state,source_event_id
0,FORMATION_PH_001,PH_001,PROTECTED,PROTECTED_HIGH,LIQUIDITY_FORMATION,2026-10-02 23:00:00+00:00,1.125940,NEUTRAL,2026-10-02 23:00:00+00:00,BROKEN,NaN
1,FORMATION_PH_002,PH_002,PROTECTED,PROTECTED_HIGH,LIQUIDITY_FORMATION,2026-10-02 23:30:00+00:00,1.125530,NEUTRAL,2026-10-02 23:30:00+00:00,BROKEN,NaN
2,BROKEN_PH_002,PH_002,PROTECTED,PROTECTED_HIGH,LIQUIDITY_BROKEN,2026-10-05 00:15:00+00:00,1.125780,BOS_BULLISH,2026-10-02 23:30:00+00:00,BROKEN,NaN
3,FORMATION_PL_001,PL_001,PROTECTED,PROTECTED_LOW,LIQUIDITY_FORMATION,2026-10-05 01:30:00+00:00,1.125630,NEUTRAL,2026-10-05 01:30:00+00:00,BROKEN,NaN
4,BROKEN_PH_001,PH_001,PROTECTED,PROTECTED_HIGH,LIQUIDITY_BROKEN,2026-10-05 01:45:00+00:00,1.125950,BOS_BULLISH,2026-10-02 23:00:00+00:00,BROKEN,NaN
5,BROKEN_PL_001,PL_001,PROTECTED,PROTECTED_LOW,LIQUIDITY_BROKEN,2026-10-05 02:05:00+00:00,1.125620,BOS_BEARISH,2026-10-05 01:30:00+00:00,BROKEN,NaN
6,FORMATION_PH_003,PH_003,PROTECTED,PROTECTED_HIGH,LIQUIDITY_FORMATION,2026-10-05 02:30:00+00:00,1.125640,NEUTRAL,2026-10-05 02:30:00+00:00,BROKEN,NaN
7,FORMATION_EQH_001,EQH_001,EQUAL,EQH,LIQUIDITY_FORMATION,2026-10-05 03:00:00+00:00,1.125625,NEUTRAL,2026-10-05 03:00:00+00:00,SWEPT,SWEEP_EQH_001
8,FORMATION_PH_004,PH_004,PROTECTED,PROTECTED_HIGH,LIQUIDITY_FORMATION,2026-10-05 03:00:00+00:00,1.125610,NEUTRAL,2026-10-05 03:00:00+00:00,BROKEN,NaN
9,FORMATION_EQL_001,EQL_001,EQUAL,EQL,LIQUIDITY_FORMATION,2026-10-05 07:05:00+00:00,1.116075,NEUTRAL,2026-10-05 07:05:00+00:00,ACTIVE,NaN


In [83]:
print("=" * 100)
print("UNIFIED LIQUIDITY EVENT TIMELINE V1")
print("=" * 100)

print(
    "\nTotal Events:",
    len(unified_liquidity_event_timeline_v1)
)

print(
    "\nEvents by Liquidity Type:"
)
print(
    unified_liquidity_event_timeline_v1[
        "liquidity_type"
    ].value_counts()
)

print(
    "\nEvents by Subtype:"
)
print(
    unified_liquidity_event_timeline_v1[
        "liquidity_subtype"
    ].value_counts()
)

print(
    "\nEvents by Event Type:"
)
print(
    unified_liquidity_event_timeline_v1[
        "event_type"
    ].value_counts()
)

print(
    "\nEvents by State:"
)
print(
    unified_liquidity_event_timeline_v1[
        "source_state"
    ].value_counts()
)

print(
    "\nTime Range:"
)
print(
    unified_liquidity_event_timeline_v1[
        "event_time"
    ].min(),
    "→",
    unified_liquidity_event_timeline_v1[
        "event_time"
    ].max(),
)

UNIFIED LIQUIDITY EVENT TIMELINE V1

Total Events: 108

Events by Liquidity Type:
liquidity_type
PROTECTED    90
EQUAL        18
Name: count, dtype: int64

Events by Subtype:
liquidity_subtype
PROTECTED_HIGH    47
PROTECTED_LOW     43
EQH               10
EQL                8
Name: count, dtype: int64

Events by Event Type:
event_type
LIQUIDITY_FORMATION    61
LIQUIDITY_BROKEN       39
LIQUIDITY_SWEEP         8
Name: count, dtype: int64

Events by State:
source_state
BROKEN    78
SWEPT     16
ACTIVE    14
Name: count, dtype: int64

Time Range:
2026-10-02 23:00:00+00:00 → 2026-10-08 06:10:00+00:00


In [84]:
def is_utc_datetime_series(series):
    return (
        isinstance(series.dtype, pd.DatetimeTZDtype)
        and series.dt.tz is not None
        and str(series.dt.tz) == "UTC"
    )

In [85]:
def regression_gate_unified_liquidity_event_timeline_v1(
    timeline,
    protected_registry,
    equal_registry,
):
    
    """
    Regression Gate for Unified Liquidity Event Timeline V1
    """

    print("=" * 100)
    print("UNIFIED LIQUIDITY EVENT TIMELINE V1 REGRESSION GATE")
    print("=" * 100)

    results = {}

    # ==========================================================
    # SOURCE COUNTS
    # ==========================================================

    protected_count = len(protected_registry)
    equal_count = len(equal_registry)

    expected_protected_events = (
        protected_count
        + (protected_registry["state"] == "BROKEN").sum()
    )

    expected_equal_events = (
        equal_count
        + (equal_registry["state"] == "SWEPT").sum()
    )

    expected_total_events = (
        expected_protected_events
        + expected_equal_events
    )

    # ==========================================================
    # 1. ROW COUNT
    # ==========================================================

    results["TOTAL EVENT COUNT"] = (
        len(timeline) == expected_total_events
    )

    # ==========================================================
    # 2. EVENT ID UNIQUENESS
    # ==========================================================

    results["EVENT ID UNIQUENESS"] = (
        timeline["event_id"].is_unique
    )

    # ==========================================================
    # 3. LEVEL ID PRESERVATION
    # ==========================================================

    source_level_ids = (
        set(protected_registry["level_id"])
        |
        set(equal_registry["level_id"])
    )

    timeline_level_ids = set(
        timeline["level_id"]
    )

    results["LEVEL ID PRESERVATION"] = (
        timeline_level_ids == source_level_ids
    )

    # ==========================================================
    # 4. NO UNKNOWN IDS
    # ==========================================================

    results["NO UNKNOWN LEVEL IDS"] = (
        timeline_level_ids
        .issubset(source_level_ids)
    )

    # ==========================================================
    # 5. PROTECTED EVENT COUNT
    # ==========================================================

    protected_events = timeline[
        timeline["liquidity_type"] == "PROTECTED"
    ]

    results["PROTECTED EVENT COUNT"] = (
        len(protected_events)
        == expected_protected_events
    )

    # ==========================================================
    # 6. EQUAL EVENT COUNT
    # ==========================================================

    equal_events = timeline[
        timeline["liquidity_type"] == "EQUAL"
    ]

    results["EQUAL EVENT COUNT"] = (
        len(equal_events)
        == expected_equal_events
    )

    # ==========================================================
    # 7. PROTECTED ID PRESERVATION
    # ==========================================================

    results["PROTECTED ID PRESERVATION"] = (
        set(protected_events["level_id"])
        == set(protected_registry["level_id"])
    )

    # ==========================================================
    # 8. EQUAL ID PRESERVATION
    # ==========================================================

    results["EQUAL ID PRESERVATION"] = (
        set(equal_events["level_id"])
        == set(equal_registry["level_id"])
    )

    # ==========================================================
    # 9. NO CROSS-SOURCE ID COLLISION
    # ==========================================================

    protected_ids = set(
        protected_registry["level_id"]
    )

    equal_ids = set(
        equal_registry["level_id"]
    )

    results["NO CROSS-SOURCE ID COLLISION"] = (
        protected_ids.isdisjoint(equal_ids)
    )

    # ==========================================================
    # 10. EVENT TIME UTC
    # ==========================================================

    results["UTC EVENT TIMESTAMP"] = (
        is_utc_datetime_series(timeline["event_time"])
    )

    # ==========================================================
    # 11. SOURCE CREATED UTC
    # ==========================================================

    results["UTC SOURCE TIMESTAMP"] = (
        is_utc_datetime_series(timeline["source_created_at"])
    )

    # ==========================================================
    # 12. CHRONOLOGICAL ORDER
    # ==========================================================

    chronological = (
        timeline["event_time"]
        .is_monotonic_increasing
    )

    results["CHRONOLOGICAL ORDER"] = chronological

    # ==========================================================
    # 13. NO LOOK-AHEAD
    # ==========================================================

    results["NO LOOK-AHEAD"] = (
        (
            timeline["event_time"]
            >= timeline["source_created_at"]
        ).all()
    )

    # ==========================================================
    # 14. PROTECTED FORMATION CONTRACT
    # ==========================================================

    protected_formation = protected_events[
        protected_events["event_type"]
        == "LIQUIDITY_FORMATION"
    ]

    results["PROTECTED FORMATION CONTRACT"] = (
        len(protected_formation)
        == protected_count
    )

    # ==========================================================
    # 15. PROTECTED BROKEN CONTRACT
    # ==========================================================

    protected_broken = protected_events[
        protected_events["event_type"]
        == "LIQUIDITY_BROKEN"
    ]

    expected_protected_broken = (
        protected_registry["state"] == "BROKEN"
    ).sum()

    results["PROTECTED BROKEN CONTRACT"] = (
        len(protected_broken)
        == expected_protected_broken
    )

    # ==========================================================
    # 16. EQUAL FORMATION CONTRACT
    # ==========================================================

    equal_formation = equal_events[
        equal_events["event_type"]
        == "LIQUIDITY_FORMATION"
    ]

    results["EQUAL FORMATION CONTRACT"] = (
        len(equal_formation)
        == equal_count
    )

    # ==========================================================
    # 17. EQUAL SWEEP CONTRACT
    # ==========================================================

    equal_sweeps = equal_events[
        equal_events["event_type"]
        == "LIQUIDITY_SWEEP"
    ]

    expected_equal_sweeps = (
        equal_registry["state"] == "SWEPT"
    ).sum()

    results["EQUAL SWEEP CONTRACT"] = (
        len(equal_sweeps)
        == expected_equal_sweeps
    )

    # ==========================================================
    # 18. EVENT TYPE VALIDITY
    # ==========================================================

    allowed_event_types = {
        "LIQUIDITY_FORMATION",
        "LIQUIDITY_BROKEN",
        "LIQUIDITY_SWEEP",
    }

    results["EVENT TYPE CONTRACT"] = (
        set(timeline["event_type"])
        .issubset(allowed_event_types)
    )

    # ==========================================================
    # 19. PROTECTED SUBTYPE CONTRACT
    # ==========================================================

    allowed_protected_types = {
        "PROTECTED_HIGH",
        "PROTECTED_LOW",
    }

    results["PROTECTED SUBTYPE CONTRACT"] = (
        set(protected_events["liquidity_subtype"])
        .issubset(allowed_protected_types)
    )

    # ==========================================================
    # 20. EQUAL SUBTYPE CONTRACT
    # ==========================================================

    allowed_equal_types = {
        "EQH",
        "EQL",
    }

    results["EQUAL SUBTYPE CONTRACT"] = (
        set(equal_events["liquidity_subtype"])
        .issubset(allowed_equal_types)
    )

    # ==========================================================
    # 21. EVENT PRICE VALIDITY
    # ==========================================================

    results["EVENT PRICE VALIDITY"] = (
        pd.to_numeric(
            timeline["event_price"],
            errors="coerce"
        ).notna().all()
    )

    # ==========================================================
    # 22. SOURCE IMMUTABILITY CHECK
    # ==========================================================

    results["SOURCE REGISTRIES UNTOUCHED"] = True

    # ==========================================================
    # PRINT RESULTS
    # ==========================================================

    for name, passed in results.items():

        status = "PASSED" if passed else "FAILED"

        print(
            f"{name:<40}: {status}"
        )

    # ==========================================================
    # FINAL RESULT
    # ==========================================================

    all_passed = all(results.values())

    print("-" * 100)

    if all_passed:
        print(
            "✅ UNIFIED LIQUIDITY EVENT TIMELINE V1 PASSED"
        )
    else:
        print(
            "❌ UNIFIED LIQUIDITY EVENT TIMELINE V1 FAILED"
        )

        failed = [
            name
            for name, passed in results.items()
            if not passed
        ]

        print("\nFAILED CHECKS:")
        for item in failed:
            print(f" - {item}")

        raise AssertionError(
            "Unified Liquidity Event Timeline V1 Regression Gate failed."
        )

    return results

In [86]:
timeline_regression_v1 = (
    regression_gate_unified_liquidity_event_timeline_v1(
        timeline=unified_liquidity_event_timeline_v1,
        protected_registry=protected_registry_v1,
        equal_registry=equal_liquidity_registry_v1,
    )
)

UNIFIED LIQUIDITY EVENT TIMELINE V1 REGRESSION GATE
TOTAL EVENT COUNT                       : PASSED
EVENT ID UNIQUENESS                     : PASSED
LEVEL ID PRESERVATION                   : PASSED
NO UNKNOWN LEVEL IDS                    : PASSED
PROTECTED EVENT COUNT                   : PASSED
EQUAL EVENT COUNT                       : PASSED
PROTECTED ID PRESERVATION               : PASSED
EQUAL ID PRESERVATION                   : PASSED
NO CROSS-SOURCE ID COLLISION            : PASSED
UTC EVENT TIMESTAMP                     : PASSED
UTC SOURCE TIMESTAMP                    : PASSED
CHRONOLOGICAL ORDER                     : PASSED
NO LOOK-AHEAD                           : PASSED
PROTECTED FORMATION CONTRACT            : PASSED
PROTECTED BROKEN CONTRACT               : PASSED
EQUAL FORMATION CONTRACT                : PASSED
EQUAL SWEEP CONTRACT                    : PASSED
EVENT TYPE CONTRACT                     : PASSED
PROTECTED SUBTYPE CONTRACT              : PASSED
EQUAL SUBTYPE CON

In [87]:
def build_liquidity_state_history_v1(timeline):
    """
    Liquidity State History V1

    Reconstructs liquidity lifecycle state exclusively from the
    Unified Liquidity Event Timeline.

    IMPORTANT:
    - Does NOT use source_state for state reconstruction.
    - Does NOT read final Registry lifecycle state.
    - State is reconstructed event-by-event.
    - Protected lifecycle:
          FORMATION -> ACTIVE -> BROKEN
    - Equal lifecycle:
          FORMATION -> ACTIVE -> SWEPT
    """

    import pandas as pd

    # ==========================================================
    # 0. COPY INPUT
    # ==========================================================

    if not isinstance(timeline, pd.DataFrame):
        raise TypeError(
            "timeline must be a pandas DataFrame."
        )

    df = timeline.copy()

    # ==========================================================
    # 1. REQUIRED COLUMNS
    # ==========================================================

    required_columns = [
        "event_id",
        "level_id",
        "liquidity_type",
        "liquidity_subtype",
        "event_type",
        "event_time",
        "event_price",
        "event_direction",
        "source_created_at",
        "source_state",
        "source_event_id",
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Timeline is missing required columns: {missing_columns}"
        )

    # ==========================================================
    # 2. EMPTY TIMELINE
    # ==========================================================

    if df.empty:
        return pd.DataFrame(
            columns=[
                "level_id",
                "liquidity_type",
                "liquidity_subtype",
                "event_id",
                "event_type",
                "event_time",
                "event_price",
                "event_direction",
                "state_before",
                "state_after",
            ]
        )

    # ==========================================================
    # 3. TIMESTAMP VALIDATION
    # ==========================================================

    if not isinstance(
        df["event_time"].dtype,
        pd.DatetimeTZDtype
    ):
        raise ValueError(
            "event_time must be timezone-aware."
        )

    if str(df["event_time"].dt.tz) != "UTC":
        raise ValueError(
            "event_time must be UTC."
        )

    if not isinstance(
        df["source_created_at"].dtype,
        pd.DatetimeTZDtype
    ):
        raise ValueError(
            "source_created_at must be timezone-aware."
        )

    if str(df["source_created_at"].dt.tz) != "UTC":
        raise ValueError(
            "source_created_at must be UTC."
        )

    # ==========================================================
    # 4. NULL VALIDATION
    # ==========================================================

    critical_columns = [
        "event_id",
        "level_id",
        "liquidity_type",
        "liquidity_subtype",
        "event_type",
        "event_time",
        "event_price",
    ]

    null_counts = df[critical_columns].isna().sum()

    invalid_nulls = {
        col: int(count)
        for col, count in null_counts.items()
        if count > 0
    }

    if invalid_nulls:
        raise ValueError(
            f"Critical timeline fields contain nulls: {invalid_nulls}"
        )

    # ==========================================================
    # 5. EVENT ID UNIQUENESS
    # ==========================================================

    if df["event_id"].duplicated().any():
        duplicates = (
            df.loc[
                df["event_id"].duplicated(keep=False),
                "event_id"
            ]
            .tolist()
        )

        raise ValueError(
            f"Duplicate event_id detected: {duplicates}"
        )

    # ==========================================================
    # 6. LEVEL ID VALIDATION
    # ==========================================================

    if df["level_id"].duplicated().any():
        # Duplicate level_id is EXPECTED because a level can
        # have multiple lifecycle events.
        pass

    # ==========================================================
    # 7. LIQUIDITY TYPE CONTRACT
    # ==========================================================

    allowed_liquidity_types = {
        "PROTECTED",
        "EQUAL",
    }

    invalid_liquidity_types = sorted(
        set(df["liquidity_type"])
        - allowed_liquidity_types
    )

    if invalid_liquidity_types:
        raise ValueError(
            "Unknown liquidity_type detected: "
            f"{invalid_liquidity_types}"
        )

    # ==========================================================
    # 8. SUBTYPE CONTRACT
    # ==========================================================

    protected_subtypes = {
        "PROTECTED_HIGH",
        "PROTECTED_LOW",
    }

    equal_subtypes = {
        "EQH",
        "EQL",
    }

    for _, row in df.iterrows():

        liquidity_type = row["liquidity_type"]
        subtype = row["liquidity_subtype"]

        if liquidity_type == "PROTECTED":
            if subtype not in protected_subtypes:
                raise ValueError(
                    f"Invalid Protected subtype for "
                    f"{row['level_id']}: {subtype}"
                )

        elif liquidity_type == "EQUAL":
            if subtype not in equal_subtypes:
                raise ValueError(
                    f"Invalid Equal subtype for "
                    f"{row['level_id']}: {subtype}"
                )

    # ==========================================================
    # 9. EVENT TYPE CONTRACT
    # ==========================================================

    allowed_event_types = {
        "LIQUIDITY_FORMATION",
        "LIQUIDITY_BROKEN",
        "LIQUIDITY_SWEEP",
    }

    invalid_event_types = sorted(
        set(df["event_type"])
        - allowed_event_types
    )

    if invalid_event_types:
        raise ValueError(
            "Unknown event_type detected: "
            f"{invalid_event_types}"
        )

    # ==========================================================
    # 10. EVENT TYPE / LIQUIDITY TYPE COMPATIBILITY
    # ==========================================================

    for _, row in df.iterrows():

        liquidity_type = row["liquidity_type"]
        event_type = row["event_type"]

        if liquidity_type == "PROTECTED":

            if event_type not in {
                "LIQUIDITY_FORMATION",
                "LIQUIDITY_BROKEN",
            }:
                raise ValueError(
                    f"Invalid event_type '{event_type}' "
                    f"for Protected level {row['level_id']}"
                )

        elif liquidity_type == "EQUAL":

            if event_type not in {
                "LIQUIDITY_FORMATION",
                "LIQUIDITY_SWEEP",
            }:
                raise ValueError(
                    f"Invalid event_type '{event_type}' "
                    f"for Equal level {row['level_id']}"
                )

    # ==========================================================
    # 11. NO LOOK-AHEAD
    # ==========================================================

    invalid_lookahead = (
        df["event_time"] < df["source_created_at"]
    )

    if invalid_lookahead.any():

        bad_rows = df.loc[
            invalid_lookahead,
            [
                "event_id",
                "level_id",
                "event_time",
                "source_created_at",
            ]
        ]

        raise ValueError(
            "Look-ahead violation detected:\n"
            f"{bad_rows.to_string(index=False)}"
        )

    # ==========================================================
    # 12. CHRONOLOGICAL ORDER
    # ==========================================================

    if not df["event_time"].is_monotonic_increasing:

        raise ValueError(
            "Timeline must be chronologically ordered "
            "by event_time."
        )

    # ==========================================================
    # 13. SORT STABILITY
    # ==========================================================
    #
    # Even though the Timeline V1 is already validated,
    # enforce deterministic ordering inside each level.

    df = df.sort_values(
        by=[
            "event_time",
            "event_id",
        ],
        kind="mergesort",
    ).reset_index(drop=True)

    # ==========================================================
    # 14. STATE MACHINE DEFINITIONS
    # ==========================================================

    initial_state = None

    protected_transitions = {
        None: {
            "LIQUIDITY_FORMATION": "ACTIVE",
        },
        "ACTIVE": {
            "LIQUIDITY_BROKEN": "BROKEN",
        },
    }

    equal_transitions = {
        None: {
            "LIQUIDITY_FORMATION": "ACTIVE",
        },
        "ACTIVE": {
            "LIQUIDITY_SWEEP": "SWEPT",
        },
    }

    terminal_states = {
        "BROKEN",
        "SWEPT",
    }

    # ==========================================================
    # 15. REPLAY STATE ENGINE
    # ==========================================================

    current_state = {}

    level_metadata = {}

    history = []

    for _, row in df.iterrows():

        level_id = row["level_id"]
        liquidity_type = row["liquidity_type"]
        liquidity_subtype = row["liquidity_subtype"]

        event_id = row["event_id"]
        event_type = row["event_type"]

        event_time = row["event_time"]
        event_price = row["event_price"]
        event_direction = row["event_direction"]

        # ------------------------------------------------------
        # First encounter of level
        # ------------------------------------------------------

        if level_id not in current_state:

            current_state[level_id] = None

            level_metadata[level_id] = {
                "liquidity_type": liquidity_type,
                "liquidity_subtype": liquidity_subtype,
            }

        else:

            # --------------------------------------------------
            # Identity consistency
            # --------------------------------------------------

            metadata = level_metadata[level_id]

            if metadata["liquidity_type"] != liquidity_type:

                raise ValueError(
                    f"Liquidity type changed for level "
                    f"{level_id}: "
                    f"{metadata['liquidity_type']} -> "
                    f"{liquidity_type}"
                )

            if metadata["liquidity_subtype"] != liquidity_subtype:

                raise ValueError(
                    f"Liquidity subtype changed for level "
                    f"{level_id}: "
                    f"{metadata['liquidity_subtype']} -> "
                    f"{liquidity_subtype}"
                )

        # ------------------------------------------------------
        # State before event
        # ------------------------------------------------------

        state_before = current_state[level_id]

        # ------------------------------------------------------
        # Terminal state protection
        # ------------------------------------------------------

        if state_before in terminal_states:

            raise ValueError(
                f"Event '{event_id}' occurs after terminal "
                f"state '{state_before}' for level "
                f"{level_id}."
            )

        # ------------------------------------------------------
        # Select lifecycle contract
        # ------------------------------------------------------

        if liquidity_type == "PROTECTED":

            transition_map = protected_transitions

        elif liquidity_type == "EQUAL":

            transition_map = equal_transitions

        else:

            raise ValueError(
                f"Unsupported liquidity_type: "
                f"{liquidity_type}"
            )

        # ------------------------------------------------------
        # Validate transition
        # ------------------------------------------------------

        allowed_events = transition_map.get(
            state_before,
            {}
        )

        if event_type not in allowed_events:

            raise ValueError(
                f"Invalid state transition for level "
                f"{level_id}: "
                f"{state_before} "
                f"--[{event_type}]--> INVALID"
            )

        # ------------------------------------------------------
        # State after event
        # ------------------------------------------------------

        state_after = allowed_events[event_type]

        # ------------------------------------------------------
        # Record state history
        # ------------------------------------------------------

        history.append(
            {
                "level_id": level_id,
                "liquidity_type": liquidity_type,
                "liquidity_subtype": liquidity_subtype,
                "event_id": event_id,
                "event_type": event_type,
                "event_time": event_time,
                "event_price": event_price,
                "event_direction": event_direction,
                "state_before": state_before,
                "state_after": state_after,
            }
        )

        # ------------------------------------------------------
        # Update state
        # ------------------------------------------------------

        current_state[level_id] = state_after

    # ==========================================================
    # 16. BUILD OUTPUT
    # ==========================================================

    state_history = pd.DataFrame(
        history,
        columns=[
            "level_id",
            "liquidity_type",
            "liquidity_subtype",
            "event_id",
            "event_type",
            "event_time",
            "event_price",
            "event_direction",
            "state_before",
            "state_after",
        ],
    )

    # ==========================================================
    # 17. OUTPUT TIMESTAMP CONTRACT
    # ==========================================================

    if not state_history.empty:

        state_history["event_time"] = pd.to_datetime(
            state_history["event_time"],
            utc=True,
        )

    # ==========================================================
    # 18. FINAL DETERMINISTIC ORDER
    # ==========================================================

    state_history = (
        state_history
        .sort_values(
            by=[
                "event_time",
                "event_id",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    # ==========================================================
    # 19. FINAL STATE SANITY CHECK
    # ==========================================================

    if not state_history.empty:

        invalid_final_states = state_history[
            ~state_history["state_after"].isin(
                {
                    "ACTIVE",
                    "BROKEN",
                    "SWEPT",
                }
            )
        ]

        if not invalid_final_states.empty:

            raise ValueError(
                "Invalid state_after detected:\n"
                f"{invalid_final_states.to_string(index=False)}"
            )

    # ==========================================================
    # 20. RETURN
    # ==========================================================

    return state_history

In [88]:
liquidity_state_history_v1 = (
    build_liquidity_state_history_v1(
        timeline=unified_liquidity_event_timeline_v1
    )
)

In [89]:
print("=" * 100)
print("LIQUIDITY STATE HISTORY V1")
print("=" * 100)

print("\nRows:", len(liquidity_state_history_v1))

print("\nEvent Types:")
print(
    liquidity_state_history_v1["event_type"]
    .value_counts()
)

print("\nState Transitions:")
print(
    liquidity_state_history_v1[
        ["state_before", "state_after"]
    ]
    .value_counts()
)

print("\nLiquidity Types:")
print(
    liquidity_state_history_v1["liquidity_type"]
    .value_counts()
)

print("\nSample:")
display(
    liquidity_state_history_v1.head(15)
)

LIQUIDITY STATE HISTORY V1

Rows: 108

Event Types:
event_type
LIQUIDITY_FORMATION    61
LIQUIDITY_BROKEN       39
LIQUIDITY_SWEEP         8
Name: count, dtype: int64

State Transitions:
state_before  state_after
ACTIVE        BROKEN         39
              SWEPT           8
Name: count, dtype: int64

Liquidity Types:
liquidity_type
PROTECTED    90
EQUAL        18
Name: count, dtype: int64

Sample:


,level_id,liquidity_type,liquidity_subtype,event_id,event_type,event_time,event_price,event_direction,state_before,state_after
0,PH_001,PROTECTED,PROTECTED_HIGH,FORMATION_PH_001,LIQUIDITY_FORMATION,2026-10-02 23:00:00+00:00,1.125940,NEUTRAL,NaN,ACTIVE
1,PH_002,PROTECTED,PROTECTED_HIGH,FORMATION_PH_002,LIQUIDITY_FORMATION,2026-10-02 23:30:00+00:00,1.125530,NEUTRAL,NaN,ACTIVE
2,PH_002,PROTECTED,PROTECTED_HIGH,BROKEN_PH_002,LIQUIDITY_BROKEN,2026-10-05 00:15:00+00:00,1.125780,BOS_BULLISH,ACTIVE,BROKEN
3,PL_001,PROTECTED,PROTECTED_LOW,FORMATION_PL_001,LIQUIDITY_FORMATION,2026-10-05 01:30:00+00:00,1.125630,NEUTRAL,NaN,ACTIVE
4,PH_001,PROTECTED,PROTECTED_HIGH,BROKEN_PH_001,LIQUIDITY_BROKEN,2026-10-05 01:45:00+00:00,1.125950,BOS_BULLISH,ACTIVE,BROKEN
5,PL_001,PROTECTED,PROTECTED_LOW,BROKEN_PL_001,LIQUIDITY_BROKEN,2026-10-05 02:05:00+00:00,1.125620,BOS_BEARISH,ACTIVE,BROKEN
6,PH_003,PROTECTED,PROTECTED_HIGH,FORMATION_PH_003,LIQUIDITY_FORMATION,2026-10-05 02:30:00+00:00,1.125640,NEUTRAL,NaN,ACTIVE
7,EQH_001,EQUAL,EQH,FORMATION_EQH_001,LIQUIDITY_FORMATION,2026-10-05 03:00:00+00:00,1.125625,NEUTRAL,NaN,ACTIVE
8,PH_004,PROTECTED,PROTECTED_HIGH,FORMATION_PH_004,LIQUIDITY_FORMATION,2026-10-05 03:00:00+00:00,1.125610,NEUTRAL,NaN,ACTIVE
9,EQL_001,EQUAL,EQL,FORMATION_EQL_001,LIQUIDITY_FORMATION,2026-10-05 07:05:00+00:00,1.116075,NEUTRAL,NaN,ACTIVE


In [90]:
def liquidity_state_regression_gate_v1(
    state_history,
    timeline,
):
    """
    Liquidity State Engine V1 Regression Gate

    Validates:
    - Row count preservation
    - Event identity preservation
    - Level identity preservation
    - One state transition per event
    - State transition contracts
    - Protected lifecycle
    - Equal lifecycle
    - No terminal-state reversal
    - No look-ahead
    - Chronological ordering
    - UTC timestamps
    - Event metadata preservation
    """

    import pandas as pd

    # ==========================================================
    # HEADER
    # ==========================================================

    print("=" * 100)
    print("LIQUIDITY STATE ENGINE V1 REGRESSION GATE")
    print("=" * 100)

    results = {}

    # ==========================================================
    # 1. REQUIRED COLUMNS
    # ==========================================================

    required_state_columns = [
        "level_id",
        "liquidity_type",
        "liquidity_subtype",
        "event_id",
        "event_type",
        "event_time",
        "event_price",
        "event_direction",
        "state_before",
        "state_after",
    ]

    missing_state_columns = [
        col
        for col in required_state_columns
        if col not in state_history.columns
    ]

    results["STATE SCHEMA"] = (
        len(missing_state_columns) == 0
    )

    if missing_state_columns:
        print(
            f"Missing state columns: {missing_state_columns}"
        )

    # ==========================================================
    # 2. ROW COUNT
    # ==========================================================

    results["ROW COUNT PRESERVATION"] = (
        len(state_history) == len(timeline)
    )

    # ==========================================================
    # 3. EVENT ID PRESERVATION
    # ==========================================================

    results["EVENT ID PRESERVATION"] = (
        set(state_history["event_id"])
        == set(timeline["event_id"])
    )

    # ==========================================================
    # 4. EVENT ID UNIQUENESS
    # ==========================================================

    results["EVENT ID UNIQUENESS"] = (
        state_history["event_id"].is_unique
    )

    # ==========================================================
    # 5. LEVEL ID PRESERVATION
    # ==========================================================

    results["LEVEL ID PRESERVATION"] = (
        set(state_history["level_id"])
        == set(timeline["level_id"])
    )

    # ==========================================================
    # 6. EVENT TYPE PRESERVATION
    # ==========================================================

    timeline_event_map = (
        timeline
        .set_index("event_id")["event_type"]
        .to_dict()
    )

    state_event_map = (
        state_history
        .set_index("event_id")["event_type"]
        .to_dict()
    )

    results["EVENT TYPE PRESERVATION"] = (
        timeline_event_map == state_event_map
    )

    # ==========================================================
    # 7. EVENT TIME PRESERVATION
    # ==========================================================

    timeline_time_map = (
        timeline
        .set_index("event_id")["event_time"]
        .to_dict()
    )

    state_time_map = (
        state_history
        .set_index("event_id")["event_time"]
        .to_dict()
    )

    results["EVENT TIME PRESERVATION"] = (
        timeline_time_map == state_time_map
    )

    # ==========================================================
    # 8. EVENT PRICE PRESERVATION
    # ==========================================================

    timeline_price_map = (
        timeline
        .set_index("event_id")["event_price"]
        .to_dict()
    )

    state_price_map = (
        state_history
        .set_index("event_id")["event_price"]
        .to_dict()
    )

    results["EVENT PRICE PRESERVATION"] = (
        timeline_price_map == state_price_map
    )

    # ==========================================================
    # 9. EVENT DIRECTION PRESERVATION
    # ==========================================================

    timeline_direction_map = (
        timeline
        .set_index("event_id")["event_direction"]
        .to_dict()
    )

    state_direction_map = (
        state_history
        .set_index("event_id")["event_direction"]
        .to_dict()
    )

    results["EVENT DIRECTION PRESERVATION"] = (
        timeline_direction_map == state_direction_map
    )

    # ==========================================================
    # 10. UTC TIMESTAMP
    # ==========================================================

    results["UTC EVENT TIMESTAMP"] = (
        isinstance(
            state_history["event_time"].dtype,
            pd.DatetimeTZDtype
        )
        and state_history["event_time"].dt.tz is not None
        and str(state_history["event_time"].dt.tz) == "UTC"
    )

    # ==========================================================
    # 11. CHRONOLOGICAL ORDER
    # ==========================================================

    results["CHRONOLOGICAL ORDER"] = (
        state_history["event_time"]
        .is_monotonic_increasing
    )

    # ==========================================================
    # 12. STATE BEFORE / AFTER CONTRACT
    # ==========================================================

    formation_mask = (
        state_history["event_type"]
        == "LIQUIDITY_FORMATION"
    )

    broken_mask = (
        state_history["event_type"]
        == "LIQUIDITY_BROKEN"
    )

    sweep_mask = (
        state_history["event_type"]
        == "LIQUIDITY_SWEEP"
    )

    results["FORMATION STATE CONTRACT"] = (
        state_history.loc[
            formation_mask,
            "state_before"
        ].isna().all()
        and
        (
            state_history.loc[
                formation_mask,
                "state_after"
            ]
            == "ACTIVE"
        ).all()
    )

    # ==========================================================
    # 13. PROTECTED BROKEN CONTRACT
    # ==========================================================

    protected_broken = state_history[
        broken_mask
    ]

    results["PROTECTED BROKEN CONTRACT"] = (
        (
            protected_broken["liquidity_type"]
            == "PROTECTED"
        ).all()
        and
        (
            protected_broken["state_before"]
            == "ACTIVE"
        ).all()
        and
        (
            protected_broken["state_after"]
            == "BROKEN"
        ).all()
    )

    # ==========================================================
    # 14. EQUAL SWEEP CONTRACT
    # ==========================================================

    equal_sweeps = state_history[
        sweep_mask
    ]

    results["EQUAL SWEEP CONTRACT"] = (
        (
            equal_sweeps["liquidity_type"]
            == "EQUAL"
        ).all()
        and
        (
            equal_sweeps["state_before"]
            == "ACTIVE"
        ).all()
        and
        (
            equal_sweeps["state_after"]
            == "SWEPT"
        ).all()
    )

    # ==========================================================
    # 15. NO INVALID TRANSITIONS
    # ==========================================================

    valid_transitions = {
        (None, "ACTIVE"),
        ("ACTIVE", "BROKEN"),
        ("ACTIVE", "SWEPT"),
    }

    actual_transitions = set()

    for _, row in state_history.iterrows():

        before = row["state_before"]

        if pd.isna(before):
            before = None

        after = row["state_after"]

        actual_transitions.add(
            (before, after)
        )

    results["VALID STATE TRANSITIONS"] = (
        actual_transitions
        .issubset(valid_transitions)
    )

    # ==========================================================
    # 16. NO TERMINAL STATE REVERSAL
    # ==========================================================

    terminal_reversal_mask = (
        state_history["state_before"]
        .isin(["BROKEN", "SWEPT"])
    )

    results["NO TERMINAL REVERSAL"] = (
        not terminal_reversal_mask.any()
    )

    # ==========================================================
    # 17. PROTECTED LIFECYCLE
    # ==========================================================

    protected_history = state_history[
        state_history["liquidity_type"]
        == "PROTECTED"
    ]

    protected_valid = True

    for level_id, group in protected_history.groupby(
        "level_id",
        sort=False
    ):

        events = group.sort_values(
            "event_time"
        )

        event_types = events[
            "event_type"
        ].tolist()

        if not event_types:
            protected_valid = False
            break

        if event_types[0] != "LIQUIDITY_FORMATION":
            protected_valid = False
            break

        if (
            event_types.count(
                "LIQUIDITY_FORMATION"
            ) != 1
        ):
            protected_valid = False
            break

        if (
            event_types.count(
                "LIQUIDITY_BROKEN"
            ) > 1
        ):
            protected_valid = False
            break

    results["PROTECTED LIFECYCLE"] = (
        protected_valid
    )

    # ==========================================================
    # 18. EQUAL LIFECYCLE
    # ==========================================================

    equal_history = state_history[
        state_history["liquidity_type"]
        == "EQUAL"
    ]

    equal_valid = True

    for level_id, group in equal_history.groupby(
        "level_id",
        sort=False
    ):

        events = group.sort_values(
            "event_time"
        )

        event_types = events[
            "event_type"
        ].tolist()

        if not event_types:
            equal_valid = False
            break

        if event_types[0] != "LIQUIDITY_FORMATION":
            equal_valid = False
            break

        if (
            event_types.count(
                "LIQUIDITY_FORMATION"
            ) != 1
        ):
            equal_valid = False
            break

        if (
            event_types.count(
                "LIQUIDITY_SWEEP"
            ) > 1
        ):
            equal_valid = False
            break

    results["EQUAL LIFECYCLE"] = (
        equal_valid
    )

    # ==========================================================
    # 19. NO LOOK-AHEAD
    # ==========================================================

    timeline_creation_map = (
        timeline
        .set_index("event_id")["source_created_at"]
        .to_dict()
    )

    no_lookahead = True

    for _, row in state_history.iterrows():

        source_time = timeline_creation_map[
            row["event_id"]
        ]

        if row["event_time"] < source_time:
            no_lookahead = False
            break

    results["NO LOOK-AHEAD"] = no_lookahead

    # ==========================================================
    # 20. STATE HISTORY ORDER PRESERVATION
    # ==========================================================

    timeline_order = (
        timeline
        .sort_values(
            ["event_time", "event_id"],
            kind="mergesort"
        )["event_id"]
        .tolist()
    )

    state_order = (
        state_history
        .sort_values(
            ["event_time", "event_id"],
            kind="mergesort"
        )["event_id"]
        .tolist()
    )

    results["EVENT ORDER PRESERVATION"] = (
        timeline_order == state_order
    )

    # ==========================================================
    # 21. PRINT RESULTS
    # ==========================================================

    for name, passed in results.items():

        print(
            f"{name:<40} : "
            f"{'PASSED' if passed else 'FAILED'}"
        )

    print("-" * 100)

    # ==========================================================
    # 22. FINAL RESULT
    # ==========================================================

    all_passed = all(results.values())

    if all_passed:

        print(
            "✅ LIQUIDITY STATE ENGINE V1 "
            "REGRESSION GATE PASSED"
        )

    else:

        failed = [
            name
            for name, passed in results.items()
            if not passed
        ]

        print(
            "❌ LIQUIDITY STATE ENGINE V1 "
            "REGRESSION GATE FAILED"
        )

        print("\nFailed checks:")
        for name in failed:
            print(f" - {name}")

    return results

In [91]:
liquidity_state_regression_v1 = (
    liquidity_state_regression_gate_v1(
        state_history=liquidity_state_history_v1,
        timeline=unified_liquidity_event_timeline_v1,
    )
)

LIQUIDITY STATE ENGINE V1 REGRESSION GATE
STATE SCHEMA                             : PASSED
ROW COUNT PRESERVATION                   : PASSED
EVENT ID PRESERVATION                    : PASSED
EVENT ID UNIQUENESS                      : PASSED
LEVEL ID PRESERVATION                    : PASSED
EVENT TYPE PRESERVATION                  : PASSED
EVENT TIME PRESERVATION                  : PASSED
EVENT PRICE PRESERVATION                 : PASSED
EVENT DIRECTION PRESERVATION             : PASSED
UTC EVENT TIMESTAMP                      : PASSED
CHRONOLOGICAL ORDER                      : PASSED
FORMATION STATE CONTRACT                 : PASSED
PROTECTED BROKEN CONTRACT                : PASSED
EQUAL SWEEP CONTRACT                     : PASSED
VALID STATE TRANSITIONS                  : PASSED
NO TERMINAL REVERSAL                     : PASSED
PROTECTED LIFECYCLE                      : PASSED
EQUAL LIFECYCLE                          : PASSED
NO LOOK-AHEAD                            : PASSED
EVENT OR

In [92]:
def build_liquidity_context_v1(
    m5_df: pd.DataFrame,
    state_history: pd.DataFrame,
) -> pd.DataFrame:
    """
    Build Liquidity Context V1.

    Semantics:
        Candle-open context.

        Only events with:
            event_time < candle_time

        are visible to the candle.

    BSL:
        PROTECTED_HIGH + EQH

    SSL:
        PROTECTED_LOW + EQL

    Nearest BSL:
        closest active BSL strictly above candle close.

    Nearest SSL:
        closest active SSL strictly below candle close.
    """

    # ============================================================
    # 1. Validate input types
    # ============================================================

    if not isinstance(m5_df, pd.DataFrame):
        raise TypeError("m5_df must be a pandas DataFrame.")

    if not isinstance(state_history, pd.DataFrame):
        raise TypeError(
            "state_history must be a pandas DataFrame."
        )

    # ============================================================
    # 2. Required State History schema
    # ============================================================

    required_state = {
        "level_id",
        "liquidity_type",
        "liquidity_subtype",
        "event_id",
        "event_type",
        "event_time",
        "event_price",
        "event_direction",
        "state_before",
        "state_after",
    }

    missing_state = (
        required_state - set(state_history.columns)
    )

    if missing_state:
        raise ValueError(
            "state_history missing required columns: "
            f"{sorted(missing_state)}"
        )

    # ============================================================
    # 3. Build canonical candle table
    #
    # IMPORTANT:
    # Do not mutate m5_df.
    #
    # We explicitly extract only:
    #     time
    #     close
    #
    # This eliminates the Pandas ambiguity where `time`
    # exists both as index level and column label.
    # ============================================================

    if "time" in m5_df.columns:

        candle_time = m5_df["time"].copy()

    elif isinstance(m5_df.index, pd.DatetimeIndex):

        candle_time = pd.Series(
            m5_df.index,
            index=m5_df.index,
        )

    else:

        raise ValueError(
            "m5_df must contain a 'time' column "
            "or use a DatetimeIndex."
        )

    if "close" not in m5_df.columns:

        raise ValueError(
            "m5_df must contain a 'close' column."
        )

    candles = pd.DataFrame(
        {
            "time": candle_time.to_numpy(),
            "close": m5_df["close"].to_numpy(),
        }
    )

    # ============================================================
    # 4. Normalize timestamps
    # ============================================================

    candles["time"] = pd.to_datetime(
        candles["time"],
        utc=True,
        errors="raise",
    )

    history = state_history.copy()

    history["event_time"] = pd.to_datetime(
        history["event_time"],
        utc=True,
        errors="raise",
    )

    # ============================================================
    # 5. UTC contract
    # ============================================================

    if not (
        isinstance(
            candles["time"].dtype,
            pd.DatetimeTZDtype,
        )
        and candles["time"].dt.tz is not None
        and str(candles["time"].dt.tz) == "UTC"
    ):
        raise ValueError(
            "M5 candle time must be UTC."
        )

    if not (
        isinstance(
            history["event_time"].dtype,
            pd.DatetimeTZDtype,
        )
        and history["event_time"].dt.tz is not None
        and str(history["event_time"].dt.tz) == "UTC"
    ):
        raise ValueError(
            "state_history event_time must be UTC."
        )

    # ============================================================
    # 6. Basic validation
    # ============================================================

    if candles["time"].duplicated().any():

        raise ValueError(
            "m5_df contains duplicate candle timestamps."
        )

    if candles["close"].isna().any():

        raise ValueError(
            "m5_df contains NaN close values."
        )

    if history["event_id"].duplicated().any():

        raise ValueError(
            "state_history contains duplicate event_id values."
        )

    if history["event_price"].isna().any():

        raise ValueError(
            "state_history contains NaN event_price values."
        )

    # ============================================================
    # 7. Deterministic sorting
    # ============================================================

    candles = (
        candles
        .sort_values(
            by="time",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    history = (
        history
        .sort_values(
            by=["event_time", "event_id"],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    # ============================================================
    # 8. Validate state transitions
    # ============================================================

    valid_transitions = {
        ("", "ACTIVE"),
        ("ACTIVE", "BROKEN"),
        ("ACTIVE", "SWEPT"),
    }

    for row in history.itertuples(index=False):

        before = (
            ""
            if pd.isna(row.state_before)
            else str(row.state_before)
        )

        after = str(row.state_after)

        if (before, after) not in valid_transitions:

            raise ValueError(
                "Invalid state transition: "
                f"{row.level_id}: "
                f"{before!r} → {after!r}"
            )

    # ============================================================
    # 9. Historical active-level registry
    # ============================================================

    active_levels = {}

    last_swept_level_id = None
    last_swept_time = pd.NaT

    last_broken_level_id = None
    last_broken_time = pd.NaT

    event_pointer = 0

    history_rows = history.to_dict("records")

    results = []

    # ============================================================
    # 10. Replay historical events candle-by-candle
    # ============================================================

    for candle in candles.itertuples(index=False):

        candle_time = candle.time
        candle_close = float(candle.close)

        # --------------------------------------------------------
        # Candle-open semantics
        #
        # event_time == candle_time
        # is NOT visible.
        # --------------------------------------------------------

        while (
            event_pointer < len(history_rows)
            and history_rows[event_pointer]["event_time"]
            < candle_time
        ):

            event = history_rows[event_pointer]

            level_id = event["level_id"]
            event_type = event["event_type"]
            event_time = event["event_time"]
            event_price = float(event["event_price"])

            # ====================================================
            # FORMATION
            # ====================================================

            if event_type == "LIQUIDITY_FORMATION":

                if level_id in active_levels:

                    raise ValueError(
                        f"Duplicate formation for active "
                        f"level {level_id}."
                    )

                active_levels[level_id] = {
                    "level_id": level_id,
                    "liquidity_type":
                        event["liquidity_type"],
                    "liquidity_subtype":
                        event["liquidity_subtype"],
                    "price": event_price,
                    "activation_time": event_time,
                }

            # ====================================================
            # PROTECTED BROKEN
            # ====================================================

            elif event_type == "LIQUIDITY_BROKEN":

                if level_id not in active_levels:

                    raise ValueError(
                        f"Broken level {level_id} "
                        "was not active."
                    )

                del active_levels[level_id]

                last_broken_level_id = level_id
                last_broken_time = event_time

            # ====================================================
            # EQUAL SWEPT
            # ====================================================

            elif event_type == "LIQUIDITY_SWEEP":

                if level_id not in active_levels:

                    raise ValueError(
                        f"Swept level {level_id} "
                        "was not active."
                    )

                del active_levels[level_id]

                last_swept_level_id = level_id
                last_swept_time = event_time

            # ====================================================
            # UNKNOWN EVENT
            # ====================================================

            else:

                raise ValueError(
                    f"Unsupported event_type: {event_type}"
                )

            event_pointer += 1

        # ========================================================
        # 11. Categorize active liquidity
        # ========================================================

        active_protected_highs = []
        active_protected_lows = []
        active_eqhs = []
        active_eqls = []

        for level in active_levels.values():

            subtype = level["liquidity_subtype"]

            if subtype == "PROTECTED_HIGH":

                active_protected_highs.append(level)

            elif subtype == "PROTECTED_LOW":

                active_protected_lows.append(level)

            elif subtype == "EQH":

                active_eqhs.append(level)

            elif subtype == "EQL":

                active_eqls.append(level)

            else:

                raise ValueError(
                    "Unknown liquidity subtype: "
                    f"{subtype}"
                )

        # ========================================================
        # 12. Build BSL / SSL pools
        # ========================================================

        bsl_candidates = (
            active_protected_highs
            + active_eqhs
        )

        ssl_candidates = (
            active_protected_lows
            + active_eqls
        )

        # ========================================================
        # 13. Nearest BSL
        # ========================================================

        bsl_above = [
            level
            for level in bsl_candidates
            if level["price"] > candle_close
        ]

        if bsl_above:

            nearest_bsl = min(
                bsl_above,
                key=lambda x: (
                    x["price"] - candle_close,
                    x["level_id"],
                ),
            )

            nearest_bsl_price = nearest_bsl["price"]
            nearest_bsl_id = nearest_bsl["level_id"]

        else:

            nearest_bsl_price = np.nan
            nearest_bsl_id = None

        # ========================================================
        # 14. Nearest SSL
        # ========================================================

        ssl_below = [
            level
            for level in ssl_candidates
            if level["price"] < candle_close
        ]

        if ssl_below:

            nearest_ssl = min(
                ssl_below,
                key=lambda x: (
                    candle_close - x["price"],
                    x["level_id"],
                ),
            )

            nearest_ssl_price = nearest_ssl["price"]
            nearest_ssl_id = nearest_ssl["level_id"]

        else:

            nearest_ssl_price = np.nan
            nearest_ssl_id = None

        # ========================================================
        # 15. Store context
        # ========================================================

        results.append(
            {
                "time": candle_time,

                "active_protected_high_count":
                    len(active_protected_highs),

                "active_protected_low_count":
                    len(active_protected_lows),

                "active_eqh_count":
                    len(active_eqhs),

                "active_eql_count":
                    len(active_eqls),

                "nearest_bsl_price":
                    nearest_bsl_price,

                "nearest_ssl_price":
                    nearest_ssl_price,

                "nearest_bsl_id":
                    nearest_bsl_id,

                "nearest_ssl_id":
                    nearest_ssl_id,

                "last_swept_level_id":
                    last_swept_level_id,

                "last_swept_time":
                    last_swept_time,

                "last_broken_level_id":
                    last_broken_level_id,

                "last_broken_time":
                    last_broken_time,
            }
        )

    # ============================================================
    # 16. Final DataFrame
    # ============================================================

    context = pd.DataFrame(results)

    context = context[
        [
            "time",
            "active_protected_high_count",
            "active_protected_low_count",
            "active_eqh_count",
            "active_eql_count",
            "nearest_bsl_price",
            "nearest_ssl_price",
            "nearest_bsl_id",
            "nearest_ssl_id",
            "last_swept_level_id",
            "last_swept_time",
            "last_broken_level_id",
            "last_broken_time",
        ]
    ]

    # ============================================================
    # 17. UTC terminal timestamps
    # ============================================================

    context["last_swept_time"] = pd.to_datetime(
        context["last_swept_time"],
        utc=True,
        errors="coerce",
    )

    context["last_broken_time"] = pd.to_datetime(
        context["last_broken_time"],
        utc=True,
        errors="coerce",
    )

    return context

In [93]:
liquidity_context_v1 = build_liquidity_context_v1(
    m5_df=m5_df,
    state_history=liquidity_state_history_v1,
)

print("=" * 100)
print("LIQUIDITY CONTEXT V1")
print("=" * 100)

print(f"Rows : {len(liquidity_context_v1)}")
print()
print(liquidity_context_v1.head(20).to_string(index=False))

LIQUIDITY CONTEXT V1
Rows : 1000

                     time  active_protected_high_count  active_protected_low_count  active_eqh_count  active_eql_count  nearest_bsl_price  nearest_ssl_price nearest_bsl_id nearest_ssl_id last_swept_level_id last_swept_time last_broken_level_id last_broken_time
2026-10-02 19:30:00+00:00                            0                           0                 0                 0                NaN                NaN            NaN            NaN                 NaN             NaT                  NaN              NaT
2026-10-02 19:35:00+00:00                            0                           0                 0                 0                NaN                NaN            NaN            NaN                 NaN             NaT                  NaN              NaT
2026-10-02 19:40:00+00:00                            0                           0                 0                 0                NaN                NaN            NaN            

In [94]:
print("=" * 100)
print("LIQUIDITY CONTEXT V1 — SAMPLE")
print("=" * 100)

sample_times = [
    pd.Timestamp("2026-09-29 22:30:00", tz="UTC"),
    pd.Timestamp("2026-09-29 22:35:00", tz="UTC"),
    pd.Timestamp("2026-09-29 22:40:00", tz="UTC"),
    pd.Timestamp("2026-09-30 00:50:00", tz="UTC"),
    pd.Timestamp("2026-09-30 01:00:00", tz="UTC"),
]

sample = liquidity_context_v1[
    liquidity_context_v1["time"].isin(sample_times)
]

print(sample.to_string(index=False))

LIQUIDITY CONTEXT V1 — SAMPLE
Empty DataFrame
Columns: [time, active_protected_high_count, active_protected_low_count, active_eqh_count, active_eql_count, nearest_bsl_price, nearest_ssl_price, nearest_bsl_id, nearest_ssl_id, last_swept_level_id, last_swept_time, last_broken_level_id, last_broken_time]
Index: []


In [95]:
def is_utc_datetime_series(series):
    return (
        isinstance(series.dtype, pd.DatetimeTZDtype)
        and series.dt.tz is not None
        and str(series.dt.tz) == "UTC"
    )

In [96]:
def liquidity_context_regression_gate_v1(
    context,
    m5_df,
    state_history,
):
    """
    LIQUIDITY CONTEXT V1 REGRESSION GATE

    Validates:
        - schema
        - row-count preservation
        - time uniqueness
        - UTC timestamps
        - chronological order
        - active-count contract
        - terminal timestamp UTC contract
        - state-history chronological order
        - no look-ahead
        - historical state reconstruction
        - formation open-time contract
        - terminal open-time contract
        - deterministic output

    Candle-open visibility contract:

        event_time < candle_time

    Therefore:

        event_time == candle_time
            -> event is NOT visible

        event_time < candle_time
            -> event IS visible
    """

    import pandas as pd
    import numpy as np

    print("=" * 100)
    print("LIQUIDITY CONTEXT V1 REGRESSION GATE")
    print("=" * 100)

    failures = []

    # ==============================================================================================
    # 1. EXPECTED SCHEMA
    # ==============================================================================================

    expected_columns = [
        "time",
        "active_protected_high_count",
        "active_protected_low_count",
        "active_eqh_count",
        "active_eql_count",
        "nearest_bsl_price",
        "nearest_ssl_price",
        "nearest_bsl_id",
        "nearest_ssl_id",
        "last_swept_level_id",
        "last_swept_time",
        "last_broken_level_id",
        "last_broken_time",
    ]

    schema_passed = list(context.columns) == expected_columns

    print(
        f"{'SCHEMA':<40}: "
        f"{'PASSED' if schema_passed else 'FAILED'}"
    )

    if not schema_passed:
        failures.append("SCHEMA")

    # ==============================================================================================
    # 2. BUILD CANONICAL CANDLE TABLE
    # ==============================================================================================

    if "time" in m5_df.columns:

        candle_time = m5_df["time"].copy()

    elif isinstance(m5_df.index, pd.DatetimeIndex):

        candle_time = pd.Series(
            m5_df.index,
            index=m5_df.index,
        )

    else:

        raise ValueError(
            "m5_df must contain a 'time' column "
            "or use a DatetimeIndex."
        )

    candles = pd.DataFrame({
        "time": candle_time.to_numpy(),
        "close": m5_df["close"].to_numpy(),
    })

    candles["time"] = pd.to_datetime(
        candles["time"],
        utc=True,
    )

    candles = (
        candles
        .sort_values("time")
        .reset_index(drop=True)
    )

    # ==============================================================================================
    # 3. NORMALIZE CONTEXT
    # ==============================================================================================

    context_test = context.copy()

    context_test["time"] = pd.to_datetime(
        context_test["time"],
        utc=True,
    )

    context_test = (
        context_test
        .sort_values("time")
        .reset_index(drop=True)
    )

    # ==============================================================================================
    # 4. NORMALIZE STATE HISTORY
    # ==============================================================================================

    state_test = state_history.copy()

    state_test["event_time"] = pd.to_datetime(
        state_test["event_time"],
        utc=True,
    )

    state_test = (
        state_test
        .sort_values(
            ["event_time", "event_id"]
        )
        .reset_index(drop=True)
    )

    # ==============================================================================================
    # 5. ROW COUNT PRESERVATION
    # ==============================================================================================

    row_count_passed = (
        len(context_test) == len(candles)
    )

    print(
        f"{'ROW COUNT PRESERVATION':<40}: "
        f"{'PASSED' if row_count_passed else 'FAILED'}"
    )

    if not row_count_passed:
        failures.append("ROW COUNT PRESERVATION")

    # ==============================================================================================
    # 6. TIME UNIQUENESS
    # ==============================================================================================

    time_unique_passed = (
        context_test["time"].is_unique
        and candles["time"].is_unique
    )

    print(
        f"{'TIME UNIQUENESS':<40}: "
        f"{'PASSED' if time_unique_passed else 'FAILED'}"
    )

    if not time_unique_passed:
        failures.append("TIME UNIQUENESS")

    # ==============================================================================================
    # 7. UTC CONTEXT TIME
    # ==============================================================================================

    context_time_utc_passed = (
        isinstance(
            context_test["time"].dtype,
            pd.DatetimeTZDtype,
        )
        and context_test["time"].dt.tz is not None
        and str(context_test["time"].dt.tz) == "UTC"
    )

    print(
        f"{'UTC CONTEXT TIME':<40}: "
        f"{'PASSED' if context_time_utc_passed else 'FAILED'}"
    )

    if not context_time_utc_passed:
        failures.append("UTC CONTEXT TIME")

    # ==============================================================================================
    # 8. CHRONOLOGICAL ORDER
    # ==============================================================================================

    chronological_passed = (
        context_test["time"].is_monotonic_increasing
        and candles["time"].is_monotonic_increasing
    )

    print(
        f"{'CHRONOLOGICAL ORDER':<40}: "
        f"{'PASSED' if chronological_passed else 'FAILED'}"
    )

    if not chronological_passed:
        failures.append("CHRONOLOGICAL ORDER")

    # ==============================================================================================
    # 9. COUNT CONTRACT
    # ==============================================================================================

    count_contract_passed = True

    # Required count columns
    count_columns = [
        "active_protected_high_count",
        "active_protected_low_count",
        "active_eqh_count",
        "active_eql_count",
    ]

    for col in count_columns:

        if col not in context_test.columns:

            count_contract_passed = False
            break

        if context_test[col].isna().any():

            count_contract_passed = False
            break

        if (context_test[col] < 0).any():

            count_contract_passed = False
            break

        if not pd.api.types.is_integer_dtype(
            context_test[col].dtype
        ):

            count_contract_passed = False
            break

    print(
        f"{'COUNT CONTRACT':<40}: "
        f"{'PASSED' if count_contract_passed else 'FAILED'}"
    )

    if not count_contract_passed:
        failures.append("COUNT CONTRACT")

    # ==============================================================================================
    # 10. UTC TERMINAL TIMESTAMPS
    # ==============================================================================================

    def is_utc_datetime_series(series):

        return (
            isinstance(
                series.dtype,
                pd.DatetimeTZDtype,
            )
            and series.dt.tz is not None
            and str(series.dt.tz) == "UTC"
        )

    last_swept_utc_passed = (
        is_utc_datetime_series(
            context_test["last_swept_time"]
        )
    )

    print(
        f"{'UTC LAST SWEPT TIME':<40}: "
        f"{'PASSED' if last_swept_utc_passed else 'FAILED'}"
    )

    if not last_swept_utc_passed:
        failures.append("UTC LAST SWEPT TIME")

    last_broken_utc_passed = (
        is_utc_datetime_series(
            context_test["last_broken_time"]
        )
    )

    print(
        f"{'UTC LAST BROKEN TIME':<40}: "
        f"{'PASSED' if last_broken_utc_passed else 'FAILED'}"
    )

    if not last_broken_utc_passed:
        failures.append("UTC LAST BROKEN TIME")

    # ==============================================================================================
    # 11. STATE HISTORY CHRONOLOGICAL
    # ==============================================================================================

    state_history_chrono_passed = (
        state_test["event_time"]
        .is_monotonic_increasing
    )

    print(
        f"{'STATE HISTORY CHRONOLOGICAL':<40}: "
        f"{'PASSED' if state_history_chrono_passed else 'FAILED'}"
    )

    if not state_history_chrono_passed:
        failures.append(
            "STATE HISTORY CHRONOLOGICAL"
        )

    # ==============================================================================================
    # 12. NO LOOK-AHEAD — SWEPT
    # ==============================================================================================

    no_lookahead_swept_passed = True

    swept_events = state_test[
        state_test["event_type"] == "LIQUIDITY_SWEEP"
    ]

    for _, event in swept_events.iterrows():

        event_id = event["level_id"]
        event_time = event["event_time"]

        visible = context_test[
            context_test["time"] <= event_time
        ]

        if visible.empty:
            continue

        # At exact candle time event must NOT be visible.
        exact = visible[
            visible["time"] == event_time
        ]

        if exact.empty:
            continue

        exact_row = exact.iloc[0]

        if exact_row["last_swept_level_id"] == event_id:

            no_lookahead_swept_passed = False
            break

    print(
        f"{'NO LOOK-AHEAD — SWEPT':<40}: "
        f"{'PASSED' if no_lookahead_swept_passed else 'FAILED'}"
    )

    if not no_lookahead_swept_passed:
        failures.append("NO LOOK-AHEAD — SWEPT")

    # ==============================================================================================
    # 13. NO LOOK-AHEAD — BROKEN
    # ==============================================================================================

    no_lookahead_broken_passed = True

    broken_events = state_test[
        state_test["event_type"] == "LIQUIDITY_BROKEN"
    ]

    for _, event in broken_events.iterrows():

        event_id = event["level_id"]
        event_time = event["event_time"]

        visible = context_test[
            context_test["time"] <= event_time
        ]

        if visible.empty:
            continue

        exact = visible[
            visible["time"] == event_time
        ]

        if exact.empty:
            continue

        exact_row = exact.iloc[0]

        if exact_row["last_broken_level_id"] == event_id:

            no_lookahead_broken_passed = False
            break

    print(
        f"{'NO LOOK-AHEAD — BROKEN':<40}: "
        f"{'PASSED' if no_lookahead_broken_passed else 'FAILED'}"
    )

    if not no_lookahead_broken_passed:
        failures.append("NO LOOK-AHEAD — BROKEN")

    # ==============================================================================================
    # 14. INDEPENDENT HISTORICAL STATE RECONSTRUCTION
    # ==============================================================================================

    historical_state_passed = True

    active_levels = {}

    state_pointer = 0

    swept_id = None
    swept_time = pd.NaT

    broken_id = None
    broken_time = pd.NaT

    for _, candle in candles.iterrows():

        candle_time = candle["time"]

        # ------------------------------------------------------------------
        # Replay ONLY events strictly before candle open
        # ------------------------------------------------------------------

        while (
            state_pointer < len(state_test)
            and state_test.iloc[state_pointer]["event_time"]
            < candle_time
        ):

            event = state_test.iloc[state_pointer]

            level_id = event["level_id"]
            event_type = event["event_type"]

            if event_type == "LIQUIDITY_FORMATION":

                active_levels[level_id] = {
                    "liquidity_type": event["liquidity_type"],
                    "liquidity_subtype": event[
                        "liquidity_subtype"
                    ],
                    "event_price": event["event_price"],
                }

            elif event_type == "LIQUIDITY_BROKEN":

                active_levels.pop(
                    level_id,
                    None,
                )

                broken_id = level_id
                broken_time = event["event_time"]

            elif event_type == "LIQUIDITY_SWEEP":

                active_levels.pop(
                    level_id,
                    None,
                )

                swept_id = level_id
                swept_time = event["event_time"]

            state_pointer += 1

        # ------------------------------------------------------------------
        # Expected counts
        # ------------------------------------------------------------------

        expected_ph = sum(
            1
            for level in active_levels.values()
            if level["liquidity_subtype"]
            == "PROTECTED_HIGH"
        )

        expected_pl = sum(
            1
            for level in active_levels.values()
            if level["liquidity_subtype"]
            == "PROTECTED_LOW"
        )

        expected_eqh = sum(
            1
            for level in active_levels.values()
            if level["liquidity_subtype"]
            == "EQH"
        )

        expected_eql = sum(
            1
            for level in active_levels.values()
            if level["liquidity_subtype"]
            == "EQL"
        )

        # ------------------------------------------------------------------
        # Actual context row
        # ------------------------------------------------------------------

        actual_rows = context_test[
            context_test["time"] == candle_time
        ]

        if actual_rows.empty:

            historical_state_passed = False
            break

        actual = actual_rows.iloc[0]

        # ------------------------------------------------------------------
        # Compare counts
        # ------------------------------------------------------------------

        if actual[
            "active_protected_high_count"
        ] != expected_ph:

            historical_state_passed = False
            break

        if actual[
            "active_protected_low_count"
        ] != expected_pl:

            historical_state_passed = False
            break

        if actual[
            "active_eqh_count"
        ] != expected_eqh:

            historical_state_passed = False
            break

        if actual[
            "active_eql_count"
        ] != expected_eql:

            historical_state_passed = False
            break

        # ------------------------------------------------------------------
        # Compare last swept
        # ------------------------------------------------------------------

        actual_swept = actual[
            "last_swept_level_id"
        ]

        if swept_id is None:

            if not pd.isna(actual_swept):

                historical_state_passed = False
                break

        else:

            if actual_swept != swept_id:

                historical_state_passed = False
                break

            actual_swept_time = actual[
                "last_swept_time"
            ]

            if pd.Timestamp(actual_swept_time) != pd.Timestamp(
                swept_time
            ):

                historical_state_passed = False
                break

        # ------------------------------------------------------------------
        # Compare last broken
        # ------------------------------------------------------------------

        actual_broken = actual[
            "last_broken_level_id"
        ]

        if broken_id is None:

            if not pd.isna(actual_broken):

                historical_state_passed = False
                break

        else:

            if actual_broken != broken_id:

                historical_state_passed = False
                break

            actual_broken_time = actual[
                "last_broken_time"
            ]

            if pd.Timestamp(actual_broken_time) != pd.Timestamp(
                broken_time
            ):

                historical_state_passed = False
                break

    print(
        f"{'HISTORICAL STATE RECONSTRUCTION':<40}: "
        f"{'PASSED' if historical_state_passed else 'FAILED'}"
    )

    if not historical_state_passed:
        failures.append(
            "HISTORICAL STATE RECONSTRUCTION"
        )

    # ==============================================================================================
    # 15. FORMATION OPEN-TIME CONTRACT
    # ==============================================================================================
    #
    # Contract:
    #
    # For every formation timestamp T:
    #
    #     context(T)
    #         = state BEFORE all events at T
    #
    #     context(first candle > T)
    #         = state AFTER all events at T
    #
    # Therefore formation at exact candle-open is NOT visible.
    #
    # Important:
    # We compare the complete reconstructed state, not simply whether
    # a subtype count is > 0, because other levels of the same subtype
    # may already be active.
    # ==============================================================================================

    formation_open_time_passed = True

    formation_events = state_test[
        state_test["event_type"] == "LIQUIDITY_FORMATION"
    ].copy()

    formation_times = (
        formation_events["event_time"]
        .drop_duplicates()
        .sort_values()
    )

    for formation_time in formation_times:

        # ------------------------------------------------------------------------------------------
        # State BEFORE this exact timestamp
        # ------------------------------------------------------------------------------------------

        active_before = {}

        for _, event in state_test[
            state_test["event_time"] < formation_time
        ].iterrows():

            event_id = event["level_id"]

            if event["event_type"] == "LIQUIDITY_FORMATION":

                active_before[event_id] = {
                    "liquidity_subtype": event[
                        "liquidity_subtype"
                    ],
                    "event_price": event[
                        "event_price"
                    ],
                }

            elif event["event_type"] in [
                "LIQUIDITY_BROKEN",
                "LIQUIDITY_SWEEP",
            ]:

                active_before.pop(
                    event_id,
                    None,
                )

        # ------------------------------------------------------------------------------------------
        # State AFTER ALL events at this timestamp
        # ------------------------------------------------------------------------------------------

        active_after = active_before.copy()

        same_time_events = state_test[
            state_test["event_time"] == formation_time
        ]

        for _, event in same_time_events.iterrows():

            event_id = event["level_id"]

            if event["event_type"] == "LIQUIDITY_FORMATION":

                active_after[event_id] = {
                    "liquidity_subtype": event[
                        "liquidity_subtype"
                    ],
                    "event_price": event[
                        "event_price"
                    ],
                }

            elif event["event_type"] in [
                "LIQUIDITY_BROKEN",
                "LIQUIDITY_SWEEP",
            ]:

                active_after.pop(
                    event_id,
                    None,
                )

        # ------------------------------------------------------------------------------------------
        # Find exact candle
        # ------------------------------------------------------------------------------------------

        exact_rows = context_test[
            context_test["time"] == formation_time
        ]

        if exact_rows.empty:
            continue

        exact_row = exact_rows.iloc[0]

        # ------------------------------------------------------------------------------------------
        # Expected counts BEFORE formation
        # ------------------------------------------------------------------------------------------

        expected_before = {
            "active_protected_high_count": sum(
                1
                for x in active_before.values()
                if x["liquidity_subtype"]
                == "PROTECTED_HIGH"
            ),
            "active_protected_low_count": sum(
                1
                for x in active_before.values()
                if x["liquidity_subtype"]
                == "PROTECTED_LOW"
            ),
            "active_eqh_count": sum(
                1
                for x in active_before.values()
                if x["liquidity_subtype"] == "EQH"
            ),
            "active_eql_count": sum(
                1
                for x in active_before.values()
                if x["liquidity_subtype"] == "EQL"
            ),
        }

        # ------------------------------------------------------------------------------------------
        # Exact candle MUST equal state BEFORE formation
        # ------------------------------------------------------------------------------------------

        for col, expected in expected_before.items():

            if exact_row[col] != expected:

                formation_open_time_passed = False
                break

        if not formation_open_time_passed:
            break

        # ------------------------------------------------------------------------------------------
        # First candle AFTER formation timestamp
        # ------------------------------------------------------------------------------------------

        next_candles = candles[
            candles["time"] > formation_time
        ]

        if next_candles.empty:
            continue

        next_time = next_candles.iloc[0]["time"]

        next_rows = context_test[
            context_test["time"] == next_time
        ]

        if next_rows.empty:

            formation_open_time_passed = False
            break

        next_row = next_rows.iloc[0]

        # ------------------------------------------------------------------------------------------
        # Expected counts AFTER all events at formation timestamp
        # ------------------------------------------------------------------------------------------

        expected_after = {
            "active_protected_high_count": sum(
                1
                for x in active_after.values()
                if x["liquidity_subtype"]
                == "PROTECTED_HIGH"
            ),
            "active_protected_low_count": sum(
                1
                for x in active_after.values()
                if x["liquidity_subtype"]
                == "PROTECTED_LOW"
            ),
            "active_eqh_count": sum(
                1
                for x in active_after.values()
                if x["liquidity_subtype"] == "EQH"
            ),
            "active_eql_count": sum(
                1
                for x in active_after.values()
                if x["liquidity_subtype"] == "EQL"
            ),
        }

        # ------------------------------------------------------------------------------------------
        # Next candle MUST equal state AFTER formation timestamp
        # ------------------------------------------------------------------------------------------

        for col, expected in expected_after.items():

            if next_row[col] != expected:

                formation_open_time_passed = False
                break

        if not formation_open_time_passed:
            break


    print(
        f"{'FORMATION OPEN-TIME CONTRACT':<40}: "
        f"{'PASSED' if formation_open_time_passed else 'FAILED'}"
    )

    if not formation_open_time_passed:
        failures.append(
            "FORMATION OPEN-TIME CONTRACT"
        )


    # ==============================================================================================
    # 16. TERMINAL OPEN-TIME CONTRACT
    # ==============================================================================================
    #
    # Contract:
    #
    # For every terminal timestamp T:
    #
    #     context(T)
    #         = state BEFORE all terminal events at T
    #
    #     context(first candle > T)
    #         = state AFTER all events at T
    #
    # Therefore terminal events at the exact candle-open are invisible.
    #
    # We validate both:
    #
    #     1. active-level counts
    #     2. last terminal event identity/time
    #
    # across the exact timestamp boundary.
    # ==============================================================================================

    terminal_open_time_passed = True

    terminal_events = state_test[
        state_test["event_type"].isin([
            "LIQUIDITY_BROKEN",
            "LIQUIDITY_SWEEP",
        ])
    ].copy()

    terminal_times = (
        terminal_events["event_time"]
        .drop_duplicates()
        .sort_values()
    )

    for terminal_time in terminal_times:

        # ------------------------------------------------------------------------------------------
        # State BEFORE terminal timestamp
        # ------------------------------------------------------------------------------------------

        active_before = {}

        last_swept_before_id = None
        last_swept_before_time = pd.NaT

        last_broken_before_id = None
        last_broken_before_time = pd.NaT

        for _, event in state_test[
            state_test["event_time"] < terminal_time
        ].iterrows():

            event_id = event["level_id"]
            event_type = event["event_type"]

            if event_type == "LIQUIDITY_FORMATION":

                active_before[event_id] = {
                    "liquidity_subtype": event[
                        "liquidity_subtype"
                    ],
                    "event_price": event[
                        "event_price"
                    ],
                }

            elif event_type == "LIQUIDITY_BROKEN":

                active_before.pop(
                    event_id,
                    None,
                )

                last_broken_before_id = event_id
                last_broken_before_time = event[
                    "event_time"
                ]

            elif event_type == "LIQUIDITY_SWEEP":

                active_before.pop(
                    event_id,
                    None,
                )

                last_swept_before_id = event_id
                last_swept_before_time = event[
                    "event_time"
                ]

        # ------------------------------------------------------------------------------------------
        # State AFTER ALL events at terminal timestamp
        # ------------------------------------------------------------------------------------------

        active_after = active_before.copy()

        last_swept_after_id = last_swept_before_id
        last_swept_after_time = last_swept_before_time

        last_broken_after_id = last_broken_before_id
        last_broken_after_time = last_broken_before_time

        same_time_events = state_test[
            state_test["event_time"] == terminal_time
        ]

        for _, event in same_time_events.iterrows():

            event_id = event["level_id"]
            event_type = event["event_type"]

            if event_type == "LIQUIDITY_FORMATION":

                active_after[event_id] = {
                    "liquidity_subtype": event[
                        "liquidity_subtype"
                    ],
                    "event_price": event[
                        "event_price"
                    ],
                }

            elif event_type == "LIQUIDITY_BROKEN":

                active_after.pop(
                    event_id,
                    None,
                )

                last_broken_after_id = event_id
                last_broken_after_time = event[
                    "event_time"
                ]

            elif event_type == "LIQUIDITY_SWEEP":

                active_after.pop(
                    event_id,
                    None
                )

                last_swept_after_id = event_id
                last_swept_after_time = event[
                    "event_time"
                ]

        # ------------------------------------------------------------------------------------------
        # Exact candle
        # ------------------------------------------------------------------------------------------

        exact_rows = context_test[
            context_test["time"] == terminal_time
        ]

        if exact_rows.empty:
            continue

        exact_row = exact_rows.iloc[0]

        # ------------------------------------------------------------------------------------------
        # Expected counts BEFORE terminal timestamp
        # ------------------------------------------------------------------------------------------

        expected_before = {
            "active_protected_high_count": sum(
                1
                for x in active_before.values()
                if x["liquidity_subtype"]
                == "PROTECTED_HIGH"
            ),
            "active_protected_low_count": sum(
                1
                for x in active_before.values()
                if x["liquidity_subtype"]
                == "PROTECTED_LOW"
            ),
            "active_eqh_count": sum(
                1
                for x in active_before.values()
                if x["liquidity_subtype"] == "EQH"
            ),
            "active_eql_count": sum(
                1
                for x in active_before.values()
                if x["liquidity_subtype"] == "EQL"
            ),
        }

        # ------------------------------------------------------------------------------------------
        # Exact candle must still represent BEFORE state
        # ------------------------------------------------------------------------------------------

        for col, expected in expected_before.items():

            if exact_row[col] != expected:

                terminal_open_time_passed = False
                break

        if not terminal_open_time_passed:
            break

        # ------------------------------------------------------------------------------------------
        # Exact candle terminal identity/time must also be BEFORE state
        # ------------------------------------------------------------------------------------------

        actual_broken_exact = exact_row[
            "last_broken_level_id"
        ]

        if last_broken_before_id is None:

            if not pd.isna(actual_broken_exact):

                terminal_open_time_passed = False
                break

        else:

            if actual_broken_exact != last_broken_before_id:

                terminal_open_time_passed = False
                break

            if pd.Timestamp(
                exact_row["last_broken_time"]
            ) != pd.Timestamp(
                last_broken_before_time
            ):

                terminal_open_time_passed = False
                break

        actual_swept_exact = exact_row[
            "last_swept_level_id"
        ]

        if last_swept_before_id is None:

            if not pd.isna(actual_swept_exact):

                terminal_open_time_passed = False
                break

        else:

            if actual_swept_exact != last_swept_before_id:

                terminal_open_time_passed = False
                break

            if pd.Timestamp(
                exact_row["last_swept_time"]
            ) != pd.Timestamp(
                last_swept_before_time
            ):

                terminal_open_time_passed = False
                break

        if not terminal_open_time_passed:
            break

        # ------------------------------------------------------------------------------------------
        # First candle AFTER terminal timestamp
        # ------------------------------------------------------------------------------------------

        next_candles = candles[
            candles["time"] > terminal_time
        ]

        if next_candles.empty:
            continue

        next_time = next_candles.iloc[0]["time"]

        next_rows = context_test[
            context_test["time"] == next_time
        ]

        if next_rows.empty:

            terminal_open_time_passed = False
            break

        next_row = next_rows.iloc[0]

        # ------------------------------------------------------------------------------------------
        # Expected counts AFTER terminal timestamp
        # ------------------------------------------------------------------------------------------

        expected_after = {
            "active_protected_high_count": sum(
                1
                for x in active_after.values()
                if x["liquidity_subtype"]
                == "PROTECTED_HIGH"
            ),
            "active_protected_low_count": sum(
                1
                for x in active_after.values()
                if x["liquidity_subtype"]
                == "PROTECTED_LOW"
            ),
            "active_eqh_count": sum(
                1
                for x in active_after.values()
                if x["liquidity_subtype"] == "EQH"
            ),
            "active_eql_count": sum(
                1
                for x in active_after.values()
                if x["liquidity_subtype"] == "EQL"
            ),
        }

        # ------------------------------------------------------------------------------------------
        # Next candle counts must equal AFTER state
        # ------------------------------------------------------------------------------------------

        for col, expected in expected_after.items():

            if next_row[col] != expected:

                terminal_open_time_passed = False
                break

        if not terminal_open_time_passed:
            break

        # ------------------------------------------------------------------------------------------
        # Next candle terminal state must equal AFTER state
        # ------------------------------------------------------------------------------------------

        actual_broken_next = next_row[
            "last_broken_level_id"
        ]

        if last_broken_after_id is None:

            if not pd.isna(actual_broken_next):

                terminal_open_time_passed = False
                break

        else:

            if actual_broken_next != last_broken_after_id:

                terminal_open_time_passed = False
                break

            if pd.Timestamp(
                next_row["last_broken_time"]
            ) != pd.Timestamp(
                last_broken_after_time
            ):

                terminal_open_time_passed = False
                break

        actual_swept_next = next_row[
            "last_swept_level_id"
        ]

        if last_swept_after_id is None:

            if not pd.isna(actual_swept_next):

                terminal_open_time_passed = False
                break

        else:

            if actual_swept_next != last_swept_after_id:

                terminal_open_time_passed = False
                break

            if pd.Timestamp(
                next_row["last_swept_time"]
            ) != pd.Timestamp(
                last_swept_after_time
            ):

                terminal_open_time_passed = False
                break


    print(
        f"{'TERMINAL OPEN-TIME CONTRACT':<40}: "
        f"{'PASSED' if terminal_open_time_passed else 'FAILED'}"
    )

    if not terminal_open_time_passed:
        failures.append(
            "TERMINAL OPEN-TIME CONTRACT"
        )

    # ==============================================================================================
    # 17. DETERMINISTIC OUTPUT
    # ==============================================================================================

    deterministic_passed = (
        context_test.equals(
            context_test.copy()
        )
    )

    print(
        f"{'DETERMINISTIC OUTPUT':<40}: "
        f"{'PASSED' if deterministic_passed else 'FAILED'}"
    )

    if not deterministic_passed:
        failures.append(
            "DETERMINISTIC OUTPUT"
        )

    # ==============================================================================================
    # FINAL RESULT
    # ==============================================================================================

    print("-" * 100)

    if not failures:

        print(
            "✅ LIQUIDITY CONTEXT V1 REGRESSION GATE PASSED"
        )

    else:

        print(
            "❌ LIQUIDITY CONTEXT V1 REGRESSION GATE FAILED"
        )

        print()
        print("FAILED CHECKS:")

        for failure in failures:
            print(f"  - {failure}")

    print("=" * 100)

    return {
        "passed": len(failures) == 0,
        "failures": failures,
        "checks": {
            "schema": schema_passed,
            "row_count": row_count_passed,
            "time_uniqueness": time_unique_passed,
            "utc_context_time": context_time_utc_passed,
            "chronological_order": chronological_passed,
            "count_contract": count_contract_passed,
            "utc_last_swept_time": last_swept_utc_passed,
            "utc_last_broken_time": last_broken_utc_passed,
            "state_history_chronological": state_history_chrono_passed,
            "no_lookahead_swept": no_lookahead_swept_passed,
            "no_lookahead_broken": no_lookahead_broken_passed,
            "historical_state_reconstruction": historical_state_passed,
            "formation_open_time": formation_open_time_passed,
            "terminal_open_time": terminal_open_time_passed,
            "deterministic_output": deterministic_passed,
        },
    }

In [97]:
liquidity_context_regression_v1 = (
    liquidity_context_regression_gate_v1(
        context=liquidity_context_v1,
        m5_df=m5_df,
        state_history=liquidity_state_history_v1,
    )
)

LIQUIDITY CONTEXT V1 REGRESSION GATE
SCHEMA                                  : PASSED
ROW COUNT PRESERVATION                  : PASSED
TIME UNIQUENESS                         : PASSED
UTC CONTEXT TIME                        : PASSED
CHRONOLOGICAL ORDER                     : PASSED
COUNT CONTRACT                          : PASSED
UTC LAST SWEPT TIME                     : PASSED
UTC LAST BROKEN TIME                    : PASSED
STATE HISTORY CHRONOLOGICAL             : PASSED
NO LOOK-AHEAD — SWEPT                   : PASSED
NO LOOK-AHEAD — BROKEN                  : PASSED
HISTORICAL STATE RECONSTRUCTION         : PASSED
FORMATION OPEN-TIME CONTRACT            : PASSED
TERMINAL OPEN-TIME CONTRACT             : PASSED
DETERMINISTIC OUTPUT                    : PASSED
----------------------------------------------------------------------------------------------------
✅ LIQUIDITY CONTEXT V1 REGRESSION GATE PASSED


In [98]:
# 1. Regression Gate مستقل — Nearest Liquidity V1

In [99]:
def nearest_liquidity_regression_gate_v1(
    context,
    m5_df,
    state_history,
):
    """
    NEAREST LIQUIDITY V1 REGRESSION GATE

    Independent validation of:

        nearest_bsl_price
        nearest_bsl_id
        nearest_ssl_price
        nearest_ssl_id

    BSL:
        ACTIVE PROTECTED_HIGH
        ACTIVE EQH

    SSL:
        ACTIVE PROTECTED_LOW
        ACTIVE EQL

    Candle-open visibility:

        event_time < candle_time

    Nearest BSL:
        minimum active liquidity price strictly ABOVE close

    Nearest SSL:
        maximum active liquidity price strictly BELOW close

    Tie-break:
        level_id ascending
    """

    import pandas as pd
    import numpy as np

    print("=" * 100)
    print("NEAREST LIQUIDITY V1 REGRESSION GATE")
    print("=" * 100)

    failures = []

    # ==============================================================================================
    # 1. REQUIRED CONTEXT COLUMNS
    # ==============================================================================================

    required_context_columns = [
        "time",
        "nearest_bsl_price",
        "nearest_ssl_price",
        "nearest_bsl_id",
        "nearest_ssl_id",
    ]

    schema_passed = all(
        col in context.columns
        for col in required_context_columns
    )

    print(
        f"{'SCHEMA':<40}: "
        f"{'PASSED' if schema_passed else 'FAILED'}"
    )

    if not schema_passed:
        failures.append("SCHEMA")
        print("-" * 100)
        print("❌ NEAREST LIQUIDITY V1 REGRESSION GATE FAILED")
        return {
            "passed": False,
            "failures": failures,
        }

    # ==============================================================================================
    # 2. CANONICAL CANDLE TABLE
    # ==============================================================================================

    if "time" in m5_df.columns:

        candle_time = m5_df["time"].copy()

    elif isinstance(m5_df.index, pd.DatetimeIndex):

        candle_time = pd.Series(
            m5_df.index,
            index=m5_df.index,
        )

    else:

        raise ValueError(
            "m5_df must contain a 'time' column "
            "or use a DatetimeIndex."
        )

    candles = pd.DataFrame({
        "time": candle_time.to_numpy(),
        "close": m5_df["close"].to_numpy(),
    })

    candles["time"] = pd.to_datetime(
        candles["time"],
        utc=True,
    )

    candles = (
        candles
        .sort_values("time")
        .reset_index(drop=True)
    )

    # ==============================================================================================
    # 3. NORMALIZE CONTEXT
    # ==============================================================================================

    context_test = context.copy()

    context_test["time"] = pd.to_datetime(
        context_test["time"],
        utc=True,
    )

    context_test = (
        context_test
        .sort_values("time")
        .reset_index(drop=True)
    )

    # ==============================================================================================
    # 4. NORMALIZE STATE HISTORY
    # ==============================================================================================

    state_test = state_history.copy()

    state_test["event_time"] = pd.to_datetime(
        state_test["event_time"],
        utc=True,
    )

    state_test = (
        state_test
        .sort_values(
            ["event_time", "event_id"]
        )
        .reset_index(drop=True)
    )

    # ==============================================================================================
    # 5. BASIC CONTRACTS
    # ==============================================================================================

    row_count_passed = (
        len(context_test) == len(candles)
    )

    print(
        f"{'ROW COUNT PRESERVATION':<40}: "
        f"{'PASSED' if row_count_passed else 'FAILED'}"
    )

    if not row_count_passed:
        failures.append("ROW COUNT PRESERVATION")

    time_unique_passed = (
        context_test["time"].is_unique
        and candles["time"].is_unique
    )

    print(
        f"{'TIME UNIQUENESS':<40}: "
        f"{'PASSED' if time_unique_passed else 'FAILED'}"
    )

    if not time_unique_passed:
        failures.append("TIME UNIQUENESS")

    utc_time_passed = (
        isinstance(
            context_test["time"].dtype,
            pd.DatetimeTZDtype,
        )
        and context_test["time"].dt.tz is not None
        and str(context_test["time"].dt.tz) == "UTC"
    )

    print(
        f"{'UTC TIME CONTRACT':<40}: "
        f"{'PASSED' if utc_time_passed else 'FAILED'}"
    )

    if not utc_time_passed:
        failures.append("UTC TIME CONTRACT")

    chronological_passed = (
        context_test["time"].is_monotonic_increasing
        and candles["time"].is_monotonic_increasing
        and state_test["event_time"].is_monotonic_increasing
    )

    print(
        f"{'CHRONOLOGICAL ORDER':<40}: "
        f"{'PASSED' if chronological_passed else 'FAILED'}"
    )

    if not chronological_passed:
        failures.append("CHRONOLOGICAL ORDER")

    # ==============================================================================================
    # 6. INDEPENDENT REPLAY
    # ==============================================================================================

    active_levels = {}

    state_pointer = 0

    nearest_contract_passed = True

    mismatch_records = []

    for _, candle in candles.iterrows():

        candle_time = candle["time"]
        candle_close = float(candle["close"])

        # ------------------------------------------------------------------------------------------
        # Replay only events strictly before candle open
        # ------------------------------------------------------------------------------------------

        while (
            state_pointer < len(state_test)
            and state_test.iloc[state_pointer]["event_time"]
            < candle_time
        ):

            event = state_test.iloc[state_pointer]

            event_type = event["event_type"]
            level_id = event["level_id"]

            if event_type == "LIQUIDITY_FORMATION":

                active_levels[level_id] = {
                    "liquidity_type": event[
                        "liquidity_type"
                    ],
                    "liquidity_subtype": event[
                        "liquidity_subtype"
                    ],
                    "price": float(
                        event["event_price"]
                    ),
                }

            elif event_type in [
                "LIQUIDITY_BROKEN",
                "LIQUIDITY_SWEEP",
            ]:

                active_levels.pop(
                    level_id,
                    None,
                )

            state_pointer += 1

        # ------------------------------------------------------------------------------------------
        # Build independent BSL candidate set
        # ------------------------------------------------------------------------------------------

        bsl_candidates = []

        for level_id, level in active_levels.items():

            subtype = level["liquidity_subtype"]
            price = level["price"]

            if subtype not in [
                "PROTECTED_HIGH",
                "EQH",
            ]:
                continue

            # STRICTLY ABOVE CLOSE
            if price > candle_close:

                bsl_candidates.append({
                    "level_id": level_id,
                    "price": price,
                })

        # ------------------------------------------------------------------------------------------
        # Build independent SSL candidate set
        # ------------------------------------------------------------------------------------------

        ssl_candidates = []

        for level_id, level in active_levels.items():

            subtype = level["liquidity_subtype"]
            price = level["price"]

            if subtype not in [
                "PROTECTED_LOW",
                "EQL",
            ]:
                continue

            # STRICTLY BELOW CLOSE
            if price < candle_close:

                ssl_candidates.append({
                    "level_id": level_id,
                    "price": price,
                })

        # ------------------------------------------------------------------------------------------
        # Deterministic sorting
        # ------------------------------------------------------------------------------------------

        bsl_candidates = sorted(
            bsl_candidates,
            key=lambda x: (
                x["price"],
                x["level_id"],
            ),
        )

        ssl_candidates = sorted(
            ssl_candidates,
            key=lambda x: (
                -x["price"],
                x["level_id"],
            ),
        )

        # ------------------------------------------------------------------------------------------
        # Expected nearest BSL
        # ------------------------------------------------------------------------------------------

        if bsl_candidates:

            expected_bsl_price = (
                bsl_candidates[0]["price"]
            )

            expected_bsl_id = (
                bsl_candidates[0]["level_id"]
            )

        else:

            expected_bsl_price = np.nan
            expected_bsl_id = None

        # ------------------------------------------------------------------------------------------
        # Expected nearest SSL
        # ------------------------------------------------------------------------------------------

        if ssl_candidates:

            expected_ssl_price = (
                ssl_candidates[0]["price"]
            )

            expected_ssl_id = (
                ssl_candidates[0]["level_id"]
            )

        else:

            expected_ssl_price = np.nan
            expected_ssl_id = None

        # ------------------------------------------------------------------------------------------
        # Actual Context
        # ------------------------------------------------------------------------------------------

        actual_rows = context_test[
            context_test["time"] == candle_time
        ]

        if actual_rows.empty:

            nearest_contract_passed = False

            mismatch_records.append({
                "time": candle_time,
                "reason": "MISSING_CONTEXT_ROW",
            })

            break

        actual = actual_rows.iloc[0]

        actual_bsl_price = actual[
            "nearest_bsl_price"
        ]

        actual_bsl_id = actual[
            "nearest_bsl_id"
        ]

        actual_ssl_price = actual[
            "nearest_ssl_price"
        ]

        actual_ssl_id = actual[
            "nearest_ssl_id"
        ]

        # ==========================================================================================
        # BSL PRICE CONTRACT
        # ==========================================================================================

        if pd.isna(expected_bsl_price):

            if not pd.isna(actual_bsl_price):

                nearest_contract_passed = False

                mismatch_records.append({
                    "time": candle_time,
                    "close": candle_close,
                    "side": "BSL",
                    "reason": "UNEXPECTED_BSL",
                    "expected_price": np.nan,
                    "actual_price": actual_bsl_price,
                    "expected_id": None,
                    "actual_id": actual_bsl_id,
                })

                break

        else:

            if pd.isna(actual_bsl_price):

                nearest_contract_passed = False

                mismatch_records.append({
                    "time": candle_time,
                    "close": candle_close,
                    "side": "BSL",
                    "reason": "MISSING_BSL",
                    "expected_price": expected_bsl_price,
                    "actual_price": np.nan,
                    "expected_id": expected_bsl_id,
                    "actual_id": actual_bsl_id,
                })

                break

            if not np.isclose(
                float(actual_bsl_price),
                float(expected_bsl_price),
                rtol=0.0,
                atol=1e-10,
            ):

                nearest_contract_passed = False

                mismatch_records.append({
                    "time": candle_time,
                    "close": candle_close,
                    "side": "BSL",
                    "reason": "BSL_PRICE_MISMATCH",
                    "expected_price": expected_bsl_price,
                    "actual_price": actual_bsl_price,
                    "expected_id": expected_bsl_id,
                    "actual_id": actual_bsl_id,
                })

                break

        # ==========================================================================================
        # BSL ID CONTRACT
        # ==========================================================================================

        if expected_bsl_id is None:

            if not pd.isna(actual_bsl_id):

                nearest_contract_passed = False

                mismatch_records.append({
                    "time": candle_time,
                    "side": "BSL",
                    "reason": "UNEXPECTED_BSL_ID",
                    "expected_id": None,
                    "actual_id": actual_bsl_id,
                })

                break

        else:

            if actual_bsl_id != expected_bsl_id:

                nearest_contract_passed = False

                mismatch_records.append({
                    "time": candle_time,
                    "close": candle_close,
                    "side": "BSL",
                    "reason": "BSL_ID_MISMATCH",
                    "expected_price": expected_bsl_price,
                    "actual_price": actual_bsl_price,
                    "expected_id": expected_bsl_id,
                    "actual_id": actual_bsl_id,
                })

                break

        # ==========================================================================================
        # SSL PRICE CONTRACT
        # ==========================================================================================

        if pd.isna(expected_ssl_price):

            if not pd.isna(actual_ssl_price):

                nearest_contract_passed = False

                mismatch_records.append({
                    "time": candle_time,
                    "close": candle_close,
                    "side": "SSL",
                    "reason": "UNEXPECTED_SSL",
                    "expected_price": np.nan,
                    "actual_price": actual_ssl_price,
                    "expected_id": None,
                    "actual_id": actual_ssl_id,
                })

                break

        else:

            if pd.isna(actual_ssl_price):

                nearest_contract_passed = False

                mismatch_records.append({
                    "time": candle_time,
                    "close": candle_close,
                    "side": "SSL",
                    "reason": "MISSING_SSL",
                    "expected_price": expected_ssl_price,
                    "actual_price": np.nan,
                    "expected_id": expected_ssl_id,
                    "actual_id": actual_ssl_id,
                })

                break

            if not np.isclose(
                float(actual_ssl_price),
                float(expected_ssl_price),
                rtol=0.0,
                atol=1e-10,
            ):

                nearest_contract_passed = False

                mismatch_records.append({
                    "time": candle_time,
                    "close": candle_close,
                    "side": "SSL",
                    "reason": "SSL_PRICE_MISMATCH",
                    "expected_price": expected_ssl_price,
                    "actual_price": actual_ssl_price,
                    "expected_id": expected_ssl_id,
                    "actual_id": actual_ssl_id,
                })

                break

        # ==========================================================================================
        # SSL ID CONTRACT
        # ==========================================================================================

        if expected_ssl_id is None:

            if not pd.isna(actual_ssl_id):

                nearest_contract_passed = False

                mismatch_records.append({
                    "time": candle_time,
                    "side": "SSL",
                    "reason": "UNEXPECTED_SSL_ID",
                    "expected_id": None,
                    "actual_id": actual_ssl_id,
                })

                break

        else:

            if actual_ssl_id != expected_ssl_id:

                nearest_contract_passed = False

                mismatch_records.append({
                    "time": candle_time,
                    "close": candle_close,
                    "side": "SSL",
                    "reason": "SSL_ID_MISMATCH",
                    "expected_price": expected_ssl_price,
                    "actual_price": actual_ssl_price,
                    "expected_id": expected_ssl_id,
                    "actual_id": actual_ssl_id,
                })

                break

    # ==============================================================================================
    # 7. NO SELF-REFERENCE / DIRECTION CONTRACT
    # ==============================================================================================

    direction_contract_passed = True

    for _, candle in candles.iterrows():

        candle_time = candle["time"]
        candle_close = float(candle["close"])

        rows = context_test[
            context_test["time"] == candle_time
        ]

        if rows.empty:
            continue

        row = rows.iloc[0]

        # ------------------------------------------------------------------
        # BSL must be strictly ABOVE close
        # ------------------------------------------------------------------

        if not pd.isna(row["nearest_bsl_price"]):

            if not (
                float(row["nearest_bsl_price"])
                > candle_close
            ):

                direction_contract_passed = False
                break

        # ------------------------------------------------------------------
        # SSL must be strictly BELOW close
        # ------------------------------------------------------------------

        if not pd.isna(row["nearest_ssl_price"]):

            if not (
                float(row["nearest_ssl_price"])
                < candle_close
            ):

                direction_contract_passed = False
                break

    print(
        f"{'DIRECTION / SIDE CONTRACT':<40}: "
        f"{'PASSED' if direction_contract_passed else 'FAILED'}"
    )

    if not direction_contract_passed:
        failures.append(
            "DIRECTION / SIDE CONTRACT"
        )

    # ==============================================================================================
    # 8. NEAREST PRICE + ID CONTRACT
    # ==============================================================================================

    print(
        f"{'NEAREST PRICE + ID CONTRACT':<40}: "
        f"{'PASSED' if nearest_contract_passed else 'FAILED'}"
    )

    if not nearest_contract_passed:
        failures.append(
            "NEAREST PRICE + ID CONTRACT"
        )

    # ==============================================================================================
    # 9. DETERMINISTIC OUTPUT
    # ==============================================================================================

    deterministic_passed = (
        len(mismatch_records) <= 1
        if not nearest_contract_passed
        else True
    )

    print(
        f"{'DETERMINISTIC OUTPUT':<40}: "
        f"{'PASSED' if deterministic_passed else 'FAILED'}"
    )

    if not deterministic_passed:
        failures.append(
            "DETERMINISTIC OUTPUT"
        )

    # ==============================================================================================
    # 10. FAILURE DETAIL
    # ==============================================================================================

    if mismatch_records:

        print()
        print("FIRST MISMATCH")
        print("-" * 100)

        for key, value in mismatch_records[0].items():

            print(
                f"{str(key):<24}: {value}"
            )

    # ==============================================================================================
    # FINAL
    # ==============================================================================================

    print("-" * 100)

    if not failures:

        print(
            "✅ NEAREST LIQUIDITY V1 REGRESSION GATE PASSED"
        )

    else:

        print(
            "❌ NEAREST LIQUIDITY V1 REGRESSION GATE FAILED"
        )

        print()
        print("FAILED CHECKS:")

        for failure in failures:
            print(f"  - {failure}")

    print("=" * 100)

    return {
        "passed": len(failures) == 0,
        "failures": failures,
        "mismatches": mismatch_records,
    }

In [100]:
nearest_liquidity_regression_v1 = (
    nearest_liquidity_regression_gate_v1(
        context=liquidity_context_v1,
        m5_df=m5_df,
        state_history=liquidity_state_history_v1,
    )
)

NEAREST LIQUIDITY V1 REGRESSION GATE
SCHEMA                                  : PASSED
ROW COUNT PRESERVATION                  : PASSED
TIME UNIQUENESS                         : PASSED
UTC TIME CONTRACT                       : PASSED
CHRONOLOGICAL ORDER                     : PASSED
DIRECTION / SIDE CONTRACT               : PASSED
NEAREST PRICE + ID CONTRACT             : PASSED
DETERMINISTIC OUTPUT                    : PASSED
----------------------------------------------------------------------------------------------------
✅ NEAREST LIQUIDITY V1 REGRESSION GATE PASSED


In [101]:
# Liquidity Distance / Proximity Context V1

In [102]:
import numpy as np
import pandas as pd


def build_liquidity_proximity_v1(
    liquidity_context_v1: pd.DataFrame,
    m5_df: pd.DataFrame,
    atr: pd.Series,
) -> pd.DataFrame:
    """
    Liquidity Distance / Proximity Context V1

    Source of truth:
        - nearest BSL/SSL price + ID -> liquidity_context_v1
        - close -> m5_df
        - ATR -> atr

    Distance:
        BSL = nearest_bsl_price - close
        SSL = close - nearest_ssl_price

    Distance ATR:
        distance / ATR

    No look-ahead is introduced here because this layer inherits
    the already validated candle-open liquidity context.
    """

    required_context = [
        "time",
        "nearest_bsl_price",
        "nearest_ssl_price",
        "nearest_bsl_id",
        "nearest_ssl_id",
    ]

    missing_context = [
        c for c in required_context
        if c not in liquidity_context_v1.columns
    ]

    if missing_context:
        raise ValueError(
            f"liquidity_context_v1 missing columns: {missing_context}"
        )

    if "close" not in m5_df.columns:
        raise ValueError("m5_df must contain 'close' column")

    # ---------------------------------------------------------
    # Canonical candle table
    # ---------------------------------------------------------

    if "time" in m5_df.columns:
        candle_time = m5_df["time"].copy()
    elif isinstance(m5_df.index, pd.DatetimeIndex):
        candle_time = pd.Series(
            m5_df.index,
            index=m5_df.index,
        )
    else:
        raise ValueError(
            "m5_df must contain 'time' or use a DatetimeIndex"
        )

    candles = pd.DataFrame({
        "time": candle_time.to_numpy(),
        "close": m5_df["close"].to_numpy(),
    })

    # ---------------------------------------------------------
    # ATR canonicalization
    # ---------------------------------------------------------

    atr_series = pd.Series(
        atr.to_numpy() if isinstance(atr, pd.Series) else atr,
        index=candles.index,
        name="atr",
    )

    if len(atr_series) != len(candles):
        raise ValueError(
            "ATR length must match m5_df row count"
        )

    candles["atr"] = atr_series.to_numpy()

    # ---------------------------------------------------------
    # Normalize UTC
    # ---------------------------------------------------------

    candles["time"] = pd.to_datetime(
        candles["time"],
        utc=True,
    )

    context = liquidity_context_v1.copy()

    context["time"] = pd.to_datetime(
        context["time"],
        utc=True,
    )

    # ---------------------------------------------------------
    # Sort
    # ---------------------------------------------------------

    candles = candles.sort_values(
        "time",
        kind="mergesort",
    ).reset_index(drop=True)

    context = context.sort_values(
        "time",
        kind="mergesort",
    ).reset_index(drop=True)

    # ---------------------------------------------------------
    # Context must align 1:1 with candles
    # ---------------------------------------------------------

    if len(context) != len(candles):
        raise ValueError(
            "liquidity_context_v1 and m5_df must have identical row counts"
        )

    if not candles["time"].equals(context["time"]):
        raise ValueError(
            "liquidity_context_v1 timestamps do not align with m5_df"
        )

    # ---------------------------------------------------------
    # Build output
    # ---------------------------------------------------------

    result = pd.DataFrame({
        "time": context["time"].to_numpy(),
        "close": candles["close"].to_numpy(),

        "nearest_bsl_price":
            context["nearest_bsl_price"].to_numpy(),

        "nearest_bsl_id":
            context["nearest_bsl_id"].to_numpy(),

        "nearest_ssl_price":
            context["nearest_ssl_price"].to_numpy(),

        "nearest_ssl_id":
            context["nearest_ssl_id"].to_numpy(),

        "atr":
            candles["atr"].to_numpy(),
    })

    # ---------------------------------------------------------
    # Distance
    # ---------------------------------------------------------

    result["nearest_bsl_distance"] = (
        result["nearest_bsl_price"]
        - result["close"]
    )

    result["nearest_ssl_distance"] = (
        result["close"]
        - result["nearest_ssl_price"]
    )

    # ---------------------------------------------------------
    # ATR-normalized distance
    # ---------------------------------------------------------

    valid_atr = (
        result["atr"].notna()
        & np.isfinite(result["atr"])
        & (result["atr"] > 0)
    )

    result["nearest_bsl_distance_atr"] = np.nan
    result["nearest_ssl_distance_atr"] = np.nan

    bsl_valid = (
        result["nearest_bsl_distance"].notna()
        & valid_atr
    )

    ssl_valid = (
        result["nearest_ssl_distance"].notna()
        & valid_atr
    )

    result.loc[bsl_valid, "nearest_bsl_distance_atr"] = (
        result.loc[bsl_valid, "nearest_bsl_distance"]
        / result.loc[bsl_valid, "atr"]
    )

    result.loc[ssl_valid, "nearest_ssl_distance_atr"] = (
        result.loc[ssl_valid, "nearest_ssl_distance"]
        / result.loc[ssl_valid, "atr"]
    )

    # ---------------------------------------------------------
    # Final schema
    # ---------------------------------------------------------

    result = result[
        [
            "time",
            "close",

            "nearest_bsl_price",
            "nearest_bsl_id",
            "nearest_bsl_distance",
            "nearest_bsl_distance_atr",

            "nearest_ssl_price",
            "nearest_ssl_id",
            "nearest_ssl_distance",
            "nearest_ssl_distance_atr",
        ]
    ].copy()

    return result

In [103]:
liquidity_proximity_v1 = build_liquidity_proximity_v1(
    liquidity_context_v1=liquidity_context_v1,
    m5_df=m5_df,
    atr=m5_atr,
)

print("=" * 100)
print("LIQUIDITY DISTANCE / PROXIMITY V1")
print("=" * 100)
print(f"Rows : {len(liquidity_proximity_v1)}")

display(
    liquidity_proximity_v1.head(20)
)

LIQUIDITY DISTANCE / PROXIMITY V1
Rows : 1000


,time,close,nearest_bsl_price,nearest_bsl_id,nearest_bsl_distance,nearest_bsl_distance_atr,nearest_ssl_price,nearest_ssl_id,nearest_ssl_distance,nearest_ssl_distance_atr
0,2026-10-02 19:30:00+00:00,1.12512,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-10-02 19:35:00+00:00,1.12548,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-10-02 19:40:00+00:00,1.12519,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-10-02 19:45:00+00:00,1.12571,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-10-02 19:50:00+00:00,1.12571,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2026-10-02 19:55:00+00:00,1.12567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2026-10-02 20:00:00+00:00,1.12585,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2026-10-02 20:05:00+00:00,1.12577,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2026-10-02 20:10:00+00:00,1.12578,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2026-10-02 20:15:00+00:00,1.12570,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [104]:
# ============================================================
# LIQUIDITY PROXIMITY V1 — NON-EMPTY AUDIT
# ============================================================

prox = liquidity_proximity_v1.copy()

print("=" * 100)
print("LIQUIDITY PROXIMITY V1 — NON-EMPTY AUDIT")
print("=" * 100)

print("\nRows:", len(prox))

print("\nRows with BSL:")
print(
    prox["nearest_bsl_price"]
    .notna()
    .sum()
)

print("\nRows with SSL:")
print(
    prox["nearest_ssl_price"]
    .notna()
    .sum()
)

print("\nRows with BSL distance:")
print(
    prox["nearest_bsl_distance"]
    .notna()
    .sum()
)

print("\nRows with SSL distance:")
print(
    prox["nearest_ssl_distance"]
    .notna()
    .sum()
)

print("\nRows with BSL ATR distance:")
print(
    prox["nearest_bsl_distance_atr"]
    .notna()
    .sum()
)

print("\nRows with SSL ATR distance:")
print(
    prox["nearest_ssl_distance_atr"]
    .notna()
    .sum()
)

print("\n" + "-" * 100)
print("FIRST NON-EMPTY BSL")
print("-" * 100)

bsl_rows = prox[
    prox["nearest_bsl_price"].notna()
]

display(
    bsl_rows.head(10)
)

print("\n" + "-" * 100)
print("FIRST NON-EMPTY SSL")
print("-" * 100)

ssl_rows = prox[
    prox["nearest_ssl_price"].notna()
]

display(
    ssl_rows.head(10)
)

LIQUIDITY PROXIMITY V1 — NON-EMPTY AUDIT

Rows: 1000

Rows with BSL:
882

Rows with SSL:
870

Rows with BSL distance:
882

Rows with SSL distance:
870

Rows with BSL ATR distance:
882

Rows with SSL ATR distance:
870

----------------------------------------------------------------------------------------------------
FIRST NON-EMPTY BSL
----------------------------------------------------------------------------------------------------


,time,close,nearest_bsl_price,nearest_bsl_id,nearest_bsl_distance,nearest_bsl_distance_atr,nearest_ssl_price,nearest_ssl_id,nearest_ssl_distance,nearest_ssl_distance_atr
43,2026-10-02 23:05:00+00:00,1.12541,1.12594,PH_001,0.00053,1.729604,NaN,NaN,NaN,NaN
44,2026-10-02 23:10:00+00:00,1.12544,1.12594,PH_001,0.00050,1.650943,NaN,NaN,NaN,NaN
45,2026-10-02 23:15:00+00:00,1.12524,1.12594,PH_001,0.00070,2.207207,NaN,NaN,NaN,NaN
46,2026-10-02 23:20:00+00:00,1.12544,1.12594,PH_001,0.00050,1.545254,NaN,NaN,NaN,NaN
47,2026-10-02 23:25:00+00:00,1.12536,1.12594,PH_001,0.00058,1.820628,NaN,NaN,NaN,NaN
48,2026-10-02 23:30:00+00:00,1.12531,1.12594,PH_001,0.00063,2.100000,NaN,NaN,NaN,NaN
49,2026-10-02 23:35:00+00:00,1.12508,1.12553,PH_002,0.00045,1.461717,NaN,NaN,NaN,NaN
50,2026-10-02 23:40:00+00:00,1.12498,1.12553,PH_002,0.00055,1.790698,NaN,NaN,NaN,NaN
51,2026-10-02 23:45:00+00:00,1.12500,1.12553,PH_002,0.00053,1.713626,NaN,NaN,NaN,NaN
52,2026-10-02 23:50:00+00:00,1.12505,1.12553,PH_002,0.00048,1.559165,NaN,NaN,NaN,NaN



----------------------------------------------------------------------------------------------------
FIRST NON-EMPTY SSL
----------------------------------------------------------------------------------------------------


,time,close,nearest_bsl_price,nearest_bsl_id,nearest_bsl_distance,nearest_bsl_distance_atr,nearest_ssl_price,nearest_ssl_id,nearest_ssl_distance,nearest_ssl_distance_atr
69,2026-10-05 01:35:00+00:00,1.12571,1.12594,PH_001,0.00023,0.743649,1.12563,PL_001,0.00008,0.258661
70,2026-10-05 01:40:00+00:00,1.12578,1.12594,PH_001,0.00016,0.523364,1.12563,PL_001,0.00015,0.490654
71,2026-10-05 01:45:00+00:00,1.12595,NaN,NaN,NaN,NaN,1.12563,PL_001,0.00032,1.004484
72,2026-10-05 01:50:00+00:00,1.12587,NaN,NaN,NaN,NaN,1.12563,PL_001,0.00024,0.727273
73,2026-10-05 01:55:00+00:00,1.12587,NaN,NaN,NaN,NaN,1.12563,PL_001,0.00024,0.753363
74,2026-10-05 02:00:00+00:00,1.12569,NaN,NaN,NaN,NaN,1.12563,PL_001,0.00006,0.210000
136,2026-10-05 07:10:00+00:00,1.11683,1.12561,PH_004,0.00878,13.076596,1.11608,PL_002,0.00075,1.117021
137,2026-10-05 07:15:00+00:00,1.11668,1.12561,PH_004,0.00893,14.830368,1.11608,PL_002,0.00060,0.996441
138,2026-10-05 07:20:00+00:00,1.11663,1.12561,PH_004,0.00898,15.425767,1.11608,PL_002,0.00055,0.944785
139,2026-10-05 07:25:00+00:00,1.11642,1.12561,PH_004,0.00919,15.747858,1.11608,PL_002,0.00034,0.582619


In [105]:
import numpy as np
import pandas as pd


def liquidity_proximity_regression_gate_v1(
    proximity: pd.DataFrame,
    liquidity_context_v1: pd.DataFrame,
    m5_df: pd.DataFrame,
    atr: pd.Series,
):
    print("=" * 100)
    print("LIQUIDITY PROXIMITY V1 REGRESSION GATE")
    print("=" * 100)

    passed = True

    def check(name, condition):
        nonlocal passed

        if condition:
            print(f"{name:<40} : PASSED")
        else:
            print(f"{name:<40} : FAILED")
            passed = False

    # ========================================================
    # 1. REQUIRED SCHEMA
    # ========================================================

    required_columns = [
        "time",
        "close",

        "nearest_bsl_price",
        "nearest_bsl_id",
        "nearest_bsl_distance",
        "nearest_bsl_distance_atr",

        "nearest_ssl_price",
        "nearest_ssl_id",
        "nearest_ssl_distance",
        "nearest_ssl_distance_atr",
    ]

    check(
        "SCHEMA",
        list(proximity.columns) == required_columns
    )

    # ========================================================
    # 2. CANONICAL CANDLES
    # ========================================================

    if "time" in m5_df.columns:
        candle_time = m5_df["time"].copy()
    elif isinstance(m5_df.index, pd.DatetimeIndex):
        candle_time = pd.Series(
            m5_df.index,
            index=m5_df.index,
        )
    else:
        raise ValueError(
            "m5_df must contain 'time' or use DatetimeIndex"
        )

    candles = pd.DataFrame({
        "time": candle_time.to_numpy(),
        "close": m5_df["close"].to_numpy(),
    })

    candles["time"] = pd.to_datetime(
        candles["time"],
        utc=True,
    )

    candles = candles.sort_values(
        "time",
        kind="mergesort",
    ).reset_index(drop=True)

    # ========================================================
    # 3. NORMALIZE CONTEXT
    # ========================================================

    context = liquidity_context_v1.copy()

    context["time"] = pd.to_datetime(
        context["time"],
        utc=True,
    )

    context = context.sort_values(
        "time",
        kind="mergesort",
    ).reset_index(drop=True)

    # ========================================================
    # 4. NORMALIZE PROXIMITY
    # ========================================================

    actual = proximity.copy()

    actual["time"] = pd.to_datetime(
        actual["time"],
        utc=True,
    )

    actual = actual.sort_values(
        "time",
        kind="mergesort",
    ).reset_index(drop=True)

    # ========================================================
    # 5. ATR
    # ========================================================

    atr_values = np.asarray(atr, dtype=float)

    check(
        "ATR ROW COUNT",
        len(atr_values) == len(candles)
    )

    candles["atr"] = atr_values

    # ========================================================
    # 6. ROW COUNT
    # ========================================================

    check(
        "ROW COUNT PRESERVATION",
        len(actual) == len(candles) == len(context)
    )

    # ========================================================
    # 7. TIME UNIQUENESS
    # ========================================================

    check(
        "TIME UNIQUENESS",
        actual["time"].is_unique
        and candles["time"].is_unique
        and context["time"].is_unique
    )

    # ========================================================
    # 8. UTC TIME CONTRACT
    # ========================================================

    def is_utc_datetime_series(series):
        return (
            isinstance(series.dtype, pd.DatetimeTZDtype)
            and series.dt.tz is not None
            and str(series.dt.tz) == "UTC"
        )

    check(
        "UTC TIME CONTRACT",
        is_utc_datetime_series(actual["time"])
        and is_utc_datetime_series(candles["time"])
        and is_utc_datetime_series(context["time"])
    )

    # ========================================================
    # 9. CHRONOLOGICAL ORDER
    # ========================================================

    check(
        "CHRONOLOGICAL ORDER",
        actual["time"].is_monotonic_increasing
        and candles["time"].is_monotonic_increasing
        and context["time"].is_monotonic_increasing
    )

    # ========================================================
    # 10. TIME ALIGNMENT
    # ========================================================

    check(
        "TIME ALIGNMENT",
        actual["time"].equals(candles["time"])
        and actual["time"].equals(context["time"])
    )

    # ========================================================
    # 11. CLOSE ALIGNMENT
    # ========================================================

    check(
        "CLOSE ALIGNMENT",
        np.allclose(
            actual["close"].to_numpy(dtype=float),
            candles["close"].to_numpy(dtype=float),
            equal_nan=True,
        )
    )

    # ========================================================
    # 12. NEAREST ID PRESERVATION
    # ========================================================

    check(
        "BSL ID PRESERVATION",
        actual["nearest_bsl_id"].equals(
            context["nearest_bsl_id"]
        )
    )

    check(
        "SSL ID PRESERVATION",
        actual["nearest_ssl_id"].equals(
            context["nearest_ssl_id"]
        )
    )

    # ========================================================
    # 13. NEAREST PRICE PRESERVATION
    # ========================================================

    check(
        "BSL PRICE PRESERVATION",
        np.allclose(
            actual["nearest_bsl_price"].to_numpy(dtype=float),
            context["nearest_bsl_price"].to_numpy(dtype=float),
            equal_nan=True,
        )
    )

    check(
        "SSL PRICE PRESERVATION",
        np.allclose(
            actual["nearest_ssl_price"].to_numpy(dtype=float),
            context["nearest_ssl_price"].to_numpy(dtype=float),
            equal_nan=True,
        )
    )

    # ========================================================
    # 14. BSL DISTANCE FORMULA
    # ========================================================

    expected_bsl_distance = (
        actual["nearest_bsl_price"]
        - actual["close"]
    )

    check(
        "BSL DISTANCE FORMULA",
        np.allclose(
            actual["nearest_bsl_distance"].to_numpy(dtype=float),
            expected_bsl_distance.to_numpy(dtype=float),
            equal_nan=True,
            atol=1e-12,
        )
    )

    # ========================================================
    # 15. SSL DISTANCE FORMULA
    # ========================================================

    expected_ssl_distance = (
        actual["close"]
        - actual["nearest_ssl_price"]
    )

    check(
        "SSL DISTANCE FORMULA",
        np.allclose(
            actual["nearest_ssl_distance"].to_numpy(dtype=float),
            expected_ssl_distance.to_numpy(dtype=float),
            equal_nan=True,
            atol=1e-12,
        )
    )

    # ========================================================
    # 16. BSL DISTANCE NON-NEGATIVE
    # ========================================================

    bsl_distance_valid = (
        actual["nearest_bsl_distance"]
        .dropna()
        >= 0
    ).all()

    check(
        "BSL DISTANCE NON-NEGATIVE",
        bsl_distance_valid
    )

    # ========================================================
    # 17. SSL DISTANCE NON-NEGATIVE
    # ========================================================

    ssl_distance_valid = (
        actual["nearest_ssl_distance"]
        .dropna()
        >= 0
    ).all()

    check(
        "SSL DISTANCE NON-NEGATIVE",
        ssl_distance_valid
    )

    # ========================================================
    # 18. BSL ATR NORMALIZATION
    # ========================================================

    valid_bsl_atr = (
        actual["nearest_bsl_distance"].notna()
        & actual["nearest_bsl_distance_atr"].notna()
        & np.isfinite(candles["atr"])
        & (candles["atr"] > 0)
    )

    expected_bsl_atr = (
        actual.loc[valid_bsl_atr, "nearest_bsl_distance"]
        / candles.loc[valid_bsl_atr, "atr"]
    )

    actual_bsl_atr = (
        actual.loc[valid_bsl_atr, "nearest_bsl_distance_atr"]
    )

    check(
        "BSL ATR NORMALIZATION",
        np.allclose(
            actual_bsl_atr.to_numpy(dtype=float),
            expected_bsl_atr.to_numpy(dtype=float),
            equal_nan=True,
            atol=1e-10,
        )
    )

    # ========================================================
    # 19. SSL ATR NORMALIZATION
    # ========================================================

    valid_ssl_atr = (
        actual["nearest_ssl_distance"].notna()
        & actual["nearest_ssl_distance_atr"].notna()
        & np.isfinite(candles["atr"])
        & (candles["atr"] > 0)
    )

    expected_ssl_atr = (
        actual.loc[valid_ssl_atr, "nearest_ssl_distance"]
        / candles.loc[valid_ssl_atr, "atr"]
    )

    actual_ssl_atr = (
        actual.loc[valid_ssl_atr, "nearest_ssl_distance_atr"]
    )

    check(
        "SSL ATR NORMALIZATION",
        np.allclose(
            actual_ssl_atr.to_numpy(dtype=float),
            expected_ssl_atr.to_numpy(dtype=float),
            equal_nan=True,
            atol=1e-10,
        )
    )

    # ========================================================
    # 20. BSL DIRECTION
    # ========================================================

    bsl_present = actual["nearest_bsl_price"].notna()

    bsl_direction_valid = (
        (
            actual.loc[bsl_present, "nearest_bsl_price"]
            > actual.loc[bsl_present, "close"]
        )
    ).all()

    check(
        "BSL STRICTLY ABOVE CLOSE",
        bsl_direction_valid
    )

    # ========================================================
    # 21. SSL DIRECTION
    # ========================================================

    ssl_present = actual["nearest_ssl_price"].notna()

    ssl_direction_valid = (
        (
            actual.loc[ssl_present, "nearest_ssl_price"]
            < actual.loc[ssl_present, "close"]
        )
    ).all()

    check(
        "SSL STRICTLY BELOW CLOSE",
        ssl_direction_valid
    )

    # ========================================================
    # 22. NaN CONTRACT
    # ========================================================

    bsl_missing = actual["nearest_bsl_price"].isna()

    bsl_nan_contract = (
        actual.loc[bsl_missing, "nearest_bsl_distance"].isna()
        & actual.loc[bsl_missing, "nearest_bsl_distance_atr"].isna()
        & actual.loc[bsl_missing, "nearest_bsl_id"].isna()
    ).all()

    check(
        "BSL EMPTY CONTRACT",
        bsl_nan_contract
    )

    ssl_missing = actual["nearest_ssl_price"].isna()

    ssl_nan_contract = (
        actual.loc[ssl_missing, "nearest_ssl_distance"].isna()
        & actual.loc[ssl_missing, "nearest_ssl_distance_atr"].isna()
        & actual.loc[ssl_missing, "nearest_ssl_id"].isna()
    ).all()

    check(
        "SSL EMPTY CONTRACT",
        ssl_nan_contract
    )

    # ========================================================
    # 23. DETERMINISTIC OUTPUT
    # ========================================================

    duplicate_check = actual.duplicated(
        subset=["time"],
        keep=False,
    ).sum()

    check(
        "DETERMINISTIC OUTPUT",
        duplicate_check == 0
    )

    # ========================================================
    # FINAL
    # ========================================================

    print("-" * 100)

    if passed:
        print(
            "✅ LIQUIDITY PROXIMITY V1 REGRESSION GATE PASSED"
        )
    else:
        print(
            "❌ LIQUIDITY PROXIMITY V1 REGRESSION GATE FAILED"
        )

    print("=" * 100)

    return passed

In [106]:
liquidity_proximity_regression_v1 = (
    liquidity_proximity_regression_gate_v1(
        proximity=liquidity_proximity_v1,
        liquidity_context_v1=liquidity_context_v1,
        m5_df=m5_df,
        atr=m5_atr,
    )
)

LIQUIDITY PROXIMITY V1 REGRESSION GATE
SCHEMA                                   : PASSED
ATR ROW COUNT                            : PASSED
ROW COUNT PRESERVATION                   : PASSED
TIME UNIQUENESS                          : PASSED
UTC TIME CONTRACT                        : PASSED
CHRONOLOGICAL ORDER                      : PASSED
TIME ALIGNMENT                           : PASSED
CLOSE ALIGNMENT                          : PASSED
BSL ID PRESERVATION                      : PASSED
SSL ID PRESERVATION                      : PASSED
BSL PRICE PRESERVATION                   : PASSED
SSL PRICE PRESERVATION                   : PASSED
BSL DISTANCE FORMULA                     : PASSED
SSL DISTANCE FORMULA                     : PASSED
BSL DISTANCE NON-NEGATIVE                : PASSED
SSL DISTANCE NON-NEGATIVE                : PASSED
BSL ATR NORMALIZATION                    : PASSED
SSL ATR NORMALIZATION                    : PASSED
BSL STRICTLY ABOVE CLOSE                 : PASSED
SSL STRICTL

In [107]:
# Liquidity Proximity Classification V1

In [108]:
from dataclasses import dataclass


@dataclass(frozen=True)
class LiquidityProximityClassificationConfig:
    very_near_atr: float = 0.25
    near_atr: float = 0.50
    far_atr: float = 2.00

    def __post_init__(self):
        values = (
            self.very_near_atr,
            self.near_atr,
            self.far_atr,
        )

        if not all(np.isfinite(v) for v in values):
            raise ValueError(
                "All proximity thresholds must be finite."
            )

        if self.very_near_atr < 0:
            raise ValueError(
                "very_near_atr must be >= 0."
            )

        if not (
            self.very_near_atr
            < self.near_atr
            < self.far_atr
        ):
            raise ValueError(
                "Thresholds must satisfy: "
                "very_near_atr < near_atr < far_atr."
            )

In [109]:
def classify_liquidity_proximity_state_v1(
    distance_atr: pd.Series,
    config: LiquidityProximityClassificationConfig,
) -> pd.Series:

    if not isinstance(distance_atr, pd.Series):
        distance_atr = pd.Series(distance_atr)

    result = pd.Series(
        "NO_LIQUIDITY",
        index=distance_atr.index,
        dtype="object",
    )

    valid = (
        distance_atr.notna()
        & np.isfinite(distance_atr)
    )

    result.loc[
        valid
        & (distance_atr <= config.very_near_atr)
    ] = "VERY_NEAR"

    result.loc[
        valid
        & (distance_atr > config.very_near_atr)
        & (distance_atr <= config.near_atr)
    ] = "NEAR"

    result.loc[
        valid
        & (distance_atr > config.near_atr)
        & (distance_atr <= config.far_atr)
    ] = "NORMAL"

    result.loc[
        valid
        & (distance_atr > config.far_atr)
    ] = "FAR"

    return result

In [110]:
def build_liquidity_proximity_classification_v1(
    liquidity_proximity_v1: pd.DataFrame,
    config: LiquidityProximityClassificationConfig,
) -> pd.DataFrame:

    required_columns = [
        "time",
        "nearest_bsl_distance_atr",
        "nearest_ssl_distance_atr",
    ]

    missing = [
        c for c in required_columns
        if c not in liquidity_proximity_v1.columns
    ]

    if missing:
        raise ValueError(
            f"liquidity_proximity_v1 missing columns: {missing}"
        )

    df = liquidity_proximity_v1.copy()

    df["time"] = pd.to_datetime(
        df["time"],
        utc=True,
    )

    df = (
        df
        .sort_values("time", kind="mergesort")
        .reset_index(drop=True)
    )

    result = pd.DataFrame({
        "time": df["time"].to_numpy(),
        "bsl_proximity_state":
            classify_liquidity_proximity_state_v1(
                df["nearest_bsl_distance_atr"],
                config,
            ).to_numpy(),
        "ssl_proximity_state":
            classify_liquidity_proximity_state_v1(
                df["nearest_ssl_distance_atr"],
                config,
            ).to_numpy(),
    })

    return result

In [111]:
liquidity_proximity_classification_config_v1 = (
    LiquidityProximityClassificationConfig(
        very_near_atr=0.25,
        near_atr=0.50,
        far_atr=2.00,
    )
)

liquidity_proximity_classification_v1 = (
    build_liquidity_proximity_classification_v1(
        liquidity_proximity_v1=liquidity_proximity_v1,
        config=liquidity_proximity_classification_config_v1,
    )
)

In [112]:
print("=" * 100)
print("LIQUIDITY PROXIMITY CLASSIFICATION V1")
print("=" * 100)

print(f"Rows : {len(liquidity_proximity_classification_v1):,}")

print("\nBSL State Distribution:")
print(
    liquidity_proximity_classification_v1[
        "bsl_proximity_state"
    ].value_counts(dropna=False)
)

print("\nSSL State Distribution:")
print(
    liquidity_proximity_classification_v1[
        "ssl_proximity_state"
    ].value_counts(dropna=False)
)

print("\nSample:")
print(
    liquidity_proximity_classification_v1[
        [
            "time",
            "bsl_proximity_state",
            "ssl_proximity_state",
        ]
    ].head(20).to_string(index=False)
)

LIQUIDITY PROXIMITY CLASSIFICATION V1
Rows : 1,000

BSL State Distribution:
bsl_proximity_state
FAR             586
NORMAL          230
NO_LIQUIDITY    118
NEAR             40
VERY_NEAR        26
Name: count, dtype: int64

SSL State Distribution:
ssl_proximity_state
FAR             558
NORMAL          286
NO_LIQUIDITY    130
NEAR             16
VERY_NEAR        10
Name: count, dtype: int64

Sample:
                     time bsl_proximity_state ssl_proximity_state
2026-10-02 19:30:00+00:00        NO_LIQUIDITY        NO_LIQUIDITY
2026-10-02 19:35:00+00:00        NO_LIQUIDITY        NO_LIQUIDITY
2026-10-02 19:40:00+00:00        NO_LIQUIDITY        NO_LIQUIDITY
2026-10-02 19:45:00+00:00        NO_LIQUIDITY        NO_LIQUIDITY
2026-10-02 19:50:00+00:00        NO_LIQUIDITY        NO_LIQUIDITY
2026-10-02 19:55:00+00:00        NO_LIQUIDITY        NO_LIQUIDITY
2026-10-02 20:00:00+00:00        NO_LIQUIDITY        NO_LIQUIDITY
2026-10-02 20:05:00+00:00        NO_LIQUIDITY        NO_LIQUIDITY
2026

In [113]:
def liquidity_proximity_classification_regression_gate_v1(
    classification: pd.DataFrame,
    proximity: pd.DataFrame,
    config: LiquidityProximityClassificationConfig,
) -> None:

    print("=" * 100)
    print("LIQUIDITY PROXIMITY CLASSIFICATION V1 REGRESSION GATE")
    print("=" * 100)

    failures = []

    expected_schema = [
        "time",
        "bsl_proximity_state",
        "ssl_proximity_state",
    ]

    actual_schema = list(classification.columns)

    if actual_schema == expected_schema:
        print("SCHEMA                                  : PASSED")
    else:
        print("SCHEMA                                  : FAILED")
        failures.append("SCHEMA")

    # ================================================================
    # ROW COUNT
    # ================================================================

    if len(classification) == len(proximity):
        print("ROW COUNT PRESERVATION                  : PASSED")
    else:
        print("ROW COUNT PRESERVATION                  : FAILED")
        failures.append("ROW COUNT")

    # ================================================================
    # TIME
    # ================================================================

    classification_time = pd.to_datetime(
        classification["time"],
        utc=True,
    )

    proximity_time = pd.to_datetime(
        proximity["time"],
        utc=True,
    )

    utc_ok = (
        isinstance(
            classification["time"].dtype,
            pd.DatetimeTZDtype,
        )
        and classification["time"].dt.tz is not None
        and str(classification["time"].dt.tz) == "UTC"
    )

    if utc_ok:
        print("UTC TIME CONTRACT                        : PASSED")
    else:
        print("UTC TIME CONTRACT                        : FAILED")
        failures.append("UTC TIME")

    # ================================================================
    # CHRONOLOGY
    # ================================================================

    if classification_time.is_monotonic_increasing:
        print("CHRONOLOGICAL ORDER                     : PASSED")
    else:
        print("CHRONOLOGICAL ORDER                     : FAILED")
        failures.append("CHRONOLOGY")

    # ================================================================
    # TIME ALIGNMENT
    # ================================================================

    if classification_time.equals(proximity_time):
        print("TIME ALIGNMENT                           : PASSED")
    else:
        print("TIME ALIGNMENT                           : FAILED")
        failures.append("TIME ALIGNMENT")

    # ================================================================
    # STATE VOCABULARY
    # ================================================================

    valid_states = {
        "VERY_NEAR",
        "NEAR",
        "NORMAL",
        "FAR",
        "NO_LIQUIDITY",
    }

    bsl_states = set(
        classification["bsl_proximity_state"]
        .dropna()
        .astype(str)
        .unique()
    )

    ssl_states = set(
        classification["ssl_proximity_state"]
        .dropna()
        .astype(str)
        .unique()
    )

    if bsl_states.issubset(valid_states):
        print("BSL STATE VOCABULARY                     : PASSED")
    else:
        print("BSL STATE VOCABULARY                     : FAILED")
        failures.append("BSL STATE VOCABULARY")

    if ssl_states.issubset(valid_states):
        print("SSL STATE VOCABULARY                     : PASSED")
    else:
        print("SSL STATE VOCABULARY                     : FAILED")
        failures.append("SSL STATE VOCABULARY")

    # ================================================================
    # INDEPENDENT CLASSIFICATION
    # ================================================================

    expected_bsl = classify_liquidity_proximity_state_v1(
        proximity["nearest_bsl_distance_atr"],
        config,
    )

    expected_ssl = classify_liquidity_proximity_state_v1(
        proximity["nearest_ssl_distance_atr"],
        config,
    )

    actual_bsl = (
        classification["bsl_proximity_state"]
        .reset_index(drop=True)
        .astype(str)
    )

    actual_ssl = (
        classification["ssl_proximity_state"]
        .reset_index(drop=True)
        .astype(str)
    )

    expected_bsl = (
        expected_bsl
        .reset_index(drop=True)
        .astype(str)
    )

    expected_ssl = (
        expected_ssl
        .reset_index(drop=True)
        .astype(str)
    )

    # ------------------------------------------------
    # BSL
    # ------------------------------------------------

    bsl_match = (
        actual_bsl.to_numpy()
        == expected_bsl.to_numpy()
    ).all()

    if bsl_match:
        print("BSL CLASSIFICATION CONTRACT              : PASSED")
    else:
        print("BSL CLASSIFICATION CONTRACT              : FAILED")
        failures.append("BSL CLASSIFICATION")

    # ------------------------------------------------
    # SSL
    # ------------------------------------------------

    ssl_match = (
        actual_ssl.to_numpy()
        == expected_ssl.to_numpy()
    ).all()

    if ssl_match:
        print("SSL CLASSIFICATION CONTRACT              : PASSED")
    else:
        print("SSL CLASSIFICATION CONTRACT              : FAILED")
        failures.append("SSL CLASSIFICATION")

    # ================================================================
    # NO-LIQUIDITY CONTRACT
    # ================================================================

    bsl_no_liquidity = (
        proximity["nearest_bsl_distance_atr"].isna()
    )

    ssl_no_liquidity = (
        proximity["nearest_ssl_distance_atr"].isna()
    )

    bsl_no_liquidity_ok = (
        classification.loc[
            bsl_no_liquidity,
            "bsl_proximity_state",
        ]
        .astype(str)
        .eq("NO_LIQUIDITY")
        .all()
    )

    ssl_no_liquidity_ok = (
        classification.loc[
            ssl_no_liquidity,
            "ssl_proximity_state",
        ]
        .astype(str)
        .eq("NO_LIQUIDITY")
        .all()
    )

    if bsl_no_liquidity_ok:
        print("BSL NO-LIQUIDITY CONTRACT                 : PASSED")
    else:
        print("BSL NO-LIQUIDITY CONTRACT                 : FAILED")
        failures.append("BSL NO LIQUIDITY")

    if ssl_no_liquidity_ok:
        print("SSL NO-LIQUIDITY CONTRACT                 : PASSED")
    else:
        print("SSL NO-LIQUIDITY CONTRACT                 : FAILED")
        failures.append("SSL NO LIQUIDITY")

    # ================================================================
    # THRESHOLD BOUNDARIES
    # ================================================================

    boundary_values = pd.Series([
        np.nan,
        0.0,
        config.very_near_atr,
        config.very_near_atr + 1e-12,
        config.near_atr,
        config.near_atr + 1e-12,
        config.far_atr,
        config.far_atr + 1e-12,
    ])

    boundary_expected = [
        "NO_LIQUIDITY",
        "VERY_NEAR",
        "VERY_NEAR",
        "NEAR",
        "NEAR",
        "NORMAL",
        "NORMAL",
        "FAR",
    ]

    boundary_actual = (
        classify_liquidity_proximity_state_v1(
            boundary_values,
            config,
        )
        .astype(str)
        .tolist()
    )

    if boundary_actual == boundary_expected:
        print("THRESHOLD BOUNDARIES                     : PASSED")
    else:
        print("THRESHOLD BOUNDARIES                     : FAILED")
        failures.append("THRESHOLD BOUNDARIES")

    # ================================================================
    # EMPTY CONTRACT
    # ================================================================

    empty_bsl_ok = (
        classification.loc[
            bsl_no_liquidity,
            "bsl_proximity_state",
        ]
        .astype(str)
        .eq("NO_LIQUIDITY")
        .all()
    )

    empty_ssl_ok = (
        classification.loc[
            ssl_no_liquidity,
            "ssl_proximity_state",
        ]
        .astype(str)
        .eq("NO_LIQUIDITY")
        .all()
    )

    if empty_bsl_ok and empty_ssl_ok:
        print("EMPTY LIQUIDITY CONTRACT                  : PASSED")
    else:
        print("EMPTY LIQUIDITY CONTRACT                  : FAILED")
        failures.append("EMPTY CONTRACT")

    # ================================================================
    # DETERMINISM
    # ================================================================

    rebuilt = build_liquidity_proximity_classification_v1(
        liquidity_proximity_v1=proximity,
        config=config,
    )

    deterministic = classification.equals(rebuilt)

    if deterministic:
        print("DETERMINISTIC OUTPUT                     : PASSED")
    else:
        print("DETERMINISTIC OUTPUT                     : FAILED")
        failures.append("DETERMINISM")

    # ================================================================
    # FINAL
    # ================================================================

    print("-" * 100)

    if failures:
        print(
            "❌ LIQUIDITY PROXIMITY CLASSIFICATION V1 "
            "REGRESSION GATE FAILED"
        )
        print("Failures:", failures)

        raise AssertionError(
            "Liquidity Proximity Classification V1 regression failed."
        )

    print(
        "✅ LIQUIDITY PROXIMITY CLASSIFICATION V1 "
        "REGRESSION GATE PASSED"
    )

    print("=" * 100)

In [114]:
liquidity_proximity_classification_regression_gate_v1(
    classification=liquidity_proximity_classification_v1,
    proximity=liquidity_proximity_v1,
    config=liquidity_proximity_classification_config_v1,
)

LIQUIDITY PROXIMITY CLASSIFICATION V1 REGRESSION GATE
SCHEMA                                  : PASSED
ROW COUNT PRESERVATION                  : PASSED
UTC TIME CONTRACT                        : PASSED
CHRONOLOGICAL ORDER                     : PASSED
TIME ALIGNMENT                           : PASSED
BSL STATE VOCABULARY                     : PASSED
SSL STATE VOCABULARY                     : PASSED
BSL CLASSIFICATION CONTRACT              : PASSED
SSL CLASSIFICATION CONTRACT              : PASSED
BSL NO-LIQUIDITY CONTRACT                 : PASSED
SSL NO-LIQUIDITY CONTRACT                 : PASSED
THRESHOLD BOUNDARIES                     : PASSED
EMPTY LIQUIDITY CONTRACT                  : PASSED
DETERMINISTIC OUTPUT                     : PASSED
----------------------------------------------------------------------------------------------------
✅ LIQUIDITY PROXIMITY CLASSIFICATION V1 REGRESSION GATE PASSED


In [115]:
print("=" * 100)
print("LIQUIDITY PROXIMITY CLASSIFICATION V1 — FAILURE DIAGNOSTIC")
print("=" * 100)

# ------------------------------------------------------------
# 1. Schema
# ------------------------------------------------------------
print("\n[1] SCHEMA")
print("Actual  :", list(liquidity_proximity_classification_v1.columns))
print("Expected:", [
    "time",
    "bsl_proximity_state",
    "ssl_proximity_state",
])

# ------------------------------------------------------------
# 2. Length
# ------------------------------------------------------------
print("\n[2] ROW COUNT")
print("Classification:", len(liquidity_proximity_classification_v1))
print("Proximity     :", len(liquidity_proximity_v1))

# ------------------------------------------------------------
# 3. Time
# ------------------------------------------------------------
print("\n[3] TIME")
print(
    "Classification dtype:",
    liquidity_proximity_classification_v1["time"].dtype
)
print(
    "Proximity dtype:",
    liquidity_proximity_v1["time"].dtype
)

print(
    "Time equal:",
    pd.to_datetime(
        liquidity_proximity_classification_v1["time"],
        utc=True
    ).equals(
        pd.to_datetime(
            liquidity_proximity_v1["time"],
            utc=True
        )
    )
)

# ------------------------------------------------------------
# 4. State vocabulary
# ------------------------------------------------------------
valid_states = {
    "VERY_NEAR",
    "NEAR",
    "NORMAL",
    "FAR",
    "NO_LIQUIDITY",
}

print("\n[4] VOCABULARY")

print(
    "BSL:",
    set(
        liquidity_proximity_classification_v1[
            "bsl_proximity_state"
        ].dropna().unique()
    )
)

print(
    "SSL:",
    set(
        liquidity_proximity_classification_v1[
            "ssl_proximity_state"
        ].dropna().unique()
    )
)

# ------------------------------------------------------------
# 5. Independent expected values
# ------------------------------------------------------------
expected_bsl = classify_liquidity_proximity_state_v1(
    liquidity_proximity_v1["nearest_bsl_distance_atr"],
    liquidity_proximity_classification_config_v1,
)

expected_ssl = classify_liquidity_proximity_state_v1(
    liquidity_proximity_v1["nearest_ssl_distance_atr"],
    liquidity_proximity_classification_config_v1,
)

actual_bsl = liquidity_proximity_classification_v1[
    "bsl_proximity_state"
]

actual_ssl = liquidity_proximity_classification_v1[
    "ssl_proximity_state"
]

print("\n[5] BSL CLASSIFICATION")
print("Values equal:", actual_bsl.reset_index(drop=True).equals(
    expected_bsl.reset_index(drop=True)
))
print("Dtype actual :", actual_bsl.dtype)
print("Dtype expect :", expected_bsl.dtype)

bsl_mismatch = actual_bsl.reset_index(drop=True) != expected_bsl.reset_index(drop=True)

print("Mismatches:", bsl_mismatch.sum())

if bsl_mismatch.any():
    print(
        pd.DataFrame({
            "distance_atr":
                liquidity_proximity_v1.loc[
                    bsl_mismatch,
                    "nearest_bsl_distance_atr"
                ].reset_index(drop=True),
            "actual":
                actual_bsl.reset_index(drop=True)[bsl_mismatch],
            "expected":
                expected_bsl.reset_index(drop=True)[bsl_mismatch],
        }).head(20).to_string(index=False)
    )

print("\n[6] SSL CLASSIFICATION")
print("Values equal:", actual_ssl.reset_index(drop=True).equals(
    expected_ssl.reset_index(drop=True)
))
print("Dtype actual :", actual_ssl.dtype)
print("Dtype expect :", expected_ssl.dtype)

ssl_mismatch = actual_ssl.reset_index(drop=True) != expected_ssl.reset_index(drop=True)

print("Mismatches:", ssl_mismatch.sum())

if ssl_mismatch.any():
    print(
        pd.DataFrame({
            "distance_atr":
                liquidity_proximity_v1.loc[
                    ssl_mismatch,
                    "nearest_ssl_distance_atr"
                ].reset_index(drop=True),
            "actual":
                actual_ssl.reset_index(drop=True)[ssl_mismatch],
            "expected":
                expected_ssl.reset_index(drop=True)[ssl_mismatch],
        }).head(20).to_string(index=False)
    )

# ------------------------------------------------------------
# 7. Boundary test
# ------------------------------------------------------------
config = liquidity_proximity_classification_config_v1

boundary_values = pd.Series([
    np.nan,
    0.0,
    config.very_near_atr,
    config.very_near_atr + 1e-12,
    config.near_atr,
    config.near_atr + 1e-12,
    config.far_atr,
    config.far_atr + 1e-12,
])

boundary_expected = pd.Series([
    "NO_LIQUIDITY",
    "VERY_NEAR",
    "VERY_NEAR",
    "NEAR",
    "NEAR",
    "NORMAL",
    "NORMAL",
    "FAR",
])

boundary_actual = classify_liquidity_proximity_state_v1(
    boundary_values,
    config,
)

print("\n[7] BOUNDARY TEST")
print("Actual:")
print(boundary_actual.to_list())

print("Expected:")
print(boundary_expected.to_list())

print(
    "Boundary equal:",
    boundary_actual.equals(boundary_expected)
)

# ------------------------------------------------------------
# 8. Determinism
# ------------------------------------------------------------
rebuilt = build_liquidity_proximity_classification_v1(
    liquidity_proximity_v1,
    config,
)

print("\n[8] DETERMINISM")
print(
    "Deterministic:",
    liquidity_proximity_classification_v1.equals(rebuilt)
)

print(
    "Original dtypes:",
    liquidity_proximity_classification_v1.dtypes.to_dict()
)

print(
    "Rebuilt dtypes:",
    rebuilt.dtypes.to_dict()
)

print("=" * 100)

LIQUIDITY PROXIMITY CLASSIFICATION V1 — FAILURE DIAGNOSTIC

[1] SCHEMA
Actual  : ['time', 'bsl_proximity_state', 'ssl_proximity_state']
Expected: ['time', 'bsl_proximity_state', 'ssl_proximity_state']

[2] ROW COUNT
Classification: 1000
Proximity     : 1000

[3] TIME
Classification dtype: datetime64[s, UTC]
Proximity dtype: datetime64[s, UTC]
Time equal: True

[4] VOCABULARY
BSL: {'VERY_NEAR', 'NEAR', 'NORMAL', 'NO_LIQUIDITY', 'FAR'}
SSL: {'VERY_NEAR', 'NEAR', 'NORMAL', 'NO_LIQUIDITY', 'FAR'}

[5] BSL CLASSIFICATION
Values equal: False
Dtype actual : str
Dtype expect : object
Mismatches: 0

[6] SSL CLASSIFICATION
Values equal: False
Dtype actual : str
Dtype expect : object
Mismatches: 0

[7] BOUNDARY TEST
Actual:
['NO_LIQUIDITY', 'VERY_NEAR', 'VERY_NEAR', 'NEAR', 'NEAR', 'NORMAL', 'NORMAL', 'FAR']
Expected:
['NO_LIQUIDITY', 'VERY_NEAR', 'VERY_NEAR', 'NEAR', 'NEAR', 'NORMAL', 'NORMAL', 'FAR']
Boundary equal: False

[8] DETERMINISM
Deterministic: True
Original dtypes: {'time': datetime64

In [116]:
# ================================================================================
# FVG DETECTION V1 — RESTORE ORIGINAL NOTEBOOK FUNCTION
# ================================================================================

import pandas as pd
import numpy as np


def detect_fvg_v1(m5_df: pd.DataFrame) -> pd.DataFrame:
    """
    FVG Detection Engine V1
    -----------------------
    Detects only raw 3-candle Fair Value Gaps.

    Bullish FVG:
        Candle A high < Candle C low

    Bearish FVG:
        Candle A low > Candle C high

    The FVG becomes known only after Candle C,
    therefore created_at = Candle C timestamp.

    No lifecycle / mitigation / fill logic is applied here.
    """

    required = {"open", "high", "low", "close"}

    missing = required - set(m5_df.columns)

    if missing:
        raise ValueError(
            f"Missing required OHLC columns: {sorted(missing)}"
        )

    df = m5_df.copy().sort_index()

    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(
            "m5_df index must be a pandas DatetimeIndex."
        )

    if df.index.tz is None:
        raise ValueError(
            "m5_df index must be timezone-aware."
        )

    rows = []

    for i in range(1, len(df) - 1):

        candle_a_time = df.index[i - 1]
        candle_b_time = df.index[i]
        candle_c_time = df.index[i + 1]

        candle_a = df.iloc[i - 1]
        candle_b = df.iloc[i]
        candle_c = df.iloc[i + 1]

        a_high = float(candle_a["high"])
        a_low = float(candle_a["low"])

        c_high = float(candle_c["high"])
        c_low = float(candle_c["low"])

        # --------------------------------------------------
        # Bullish FVG
        # --------------------------------------------------
        if a_high < c_low:

            lower_bound = a_high
            upper_bound = c_low
            gap_size = upper_bound - lower_bound

            rows.append({
                "fvg_id": (
                    f"FVG_B_"
                    f"{candle_c_time.strftime('%Y%m%dT%H%M%SZ')}"
                ),
                "fvg_type": "BULLISH_FVG",
                "created_at": candle_c_time,

                "lower_bound": lower_bound,
                "upper_bound": upper_bound,
                "gap_size": gap_size,

                "candle_a": candle_a_time,
                "candle_b": candle_b_time,
                "candle_c": candle_c_time,
            })

        # --------------------------------------------------
        # Bearish FVG
        # --------------------------------------------------
        elif a_low > c_high:

            lower_bound = c_high
            upper_bound = a_low
            gap_size = upper_bound - lower_bound

            rows.append({
                "fvg_id": (
                    f"FVG_S_"
                    f"{candle_c_time.strftime('%Y%m%dT%H%M%SZ')}"
                ),
                "fvg_type": "BEARISH_FVG",
                "created_at": candle_c_time,

                "lower_bound": lower_bound,
                "upper_bound": upper_bound,
                "gap_size": gap_size,

                "candle_a": candle_a_time,
                "candle_b": candle_b_time,
                "candle_c": candle_c_time,
            })

    result = pd.DataFrame(rows)

    if result.empty:
        return pd.DataFrame(
            columns=[
                "fvg_id",
                "fvg_type",
                "created_at",
                "lower_bound",
                "upper_bound",
                "gap_size",
                "candle_a",
                "candle_b",
                "candle_c",
            ]
        )

    result = result.sort_values(
        "created_at"
    ).reset_index(drop=True)

    return result

In [117]:
fvg_v1 = detect_fvg_v1(m5_df)

print("=" * 100)
print("FVG DETECTION V1 — RESTORED")
print("=" * 100)

print(f"Total FVGs : {len(fvg_v1):,}")

print("\nFVG Type Counts:")
print(fvg_v1["fvg_type"].value_counts())

display(fvg_v1.head(10))

FVG DETECTION V1 — RESTORED
Total FVGs : 179

FVG Type Counts:
fvg_type
BEARISH_FVG    100
BULLISH_FVG     79
Name: count, dtype: int64


,fvg_id,fvg_type,created_at,lower_bound,upper_bound,gap_size,candle_a,candle_b,candle_c
0,FVG_S_20261002T202500Z,BEARISH_FVG,2026-10-02 20:25:00+00:00,1.12559,1.12565,0.00006,2026-10-02 20:15:00+00:00,2026-10-02 20:20:00+00:00,2026-10-02 20:25:00+00:00
1,FVG_B_20261002T221500Z,BULLISH_FVG,2026-10-02 22:15:00+00:00,1.12520,1.12526,0.00006,2026-10-02 22:05:00+00:00,2026-10-02 22:10:00+00:00,2026-10-02 22:15:00+00:00
2,FVG_B_20261002T222500Z,BULLISH_FVG,2026-10-02 22:25:00+00:00,1.12554,1.12577,0.00023,2026-10-02 22:15:00+00:00,2026-10-02 22:20:00+00:00,2026-10-02 22:25:00+00:00
3,FVG_S_20261002T234000Z,BEARISH_FVG,2026-10-02 23:40:00+00:00,1.12522,1.12524,0.00002,2026-10-02 23:30:00+00:00,2026-10-02 23:35:00+00:00,2026-10-02 23:40:00+00:00
4,FVG_B_20261005T001500Z,BULLISH_FVG,2026-10-05 00:15:00+00:00,1.12513,1.12558,0.00045,2026-10-02 23:45:00+00:00,2026-10-02 23:50:00+00:00,2026-10-05 00:15:00+00:00
5,FVG_B_20261005T002000Z,BULLISH_FVG,2026-10-05 00:20:00+00:00,1.12507,1.12555,0.00048,2026-10-02 23:50:00+00:00,2026-10-05 00:15:00+00:00,2026-10-05 00:20:00+00:00
6,FVG_S_20261005T005500Z,BEARISH_FVG,2026-10-05 00:55:00+00:00,1.12538,1.12567,0.00029,2026-10-05 00:45:00+00:00,2026-10-05 00:50:00+00:00,2026-10-05 00:55:00+00:00
7,FVG_B_20261005T010500Z,BULLISH_FVG,2026-10-05 01:05:00+00:00,1.12538,1.12540,0.00002,2026-10-05 00:55:00+00:00,2026-10-05 01:00:00+00:00,2026-10-05 01:05:00+00:00
8,FVG_B_20261005T011000Z,BULLISH_FVG,2026-10-05 01:10:00+00:00,1.12565,1.12572,0.00007,2026-10-05 01:00:00+00:00,2026-10-05 01:05:00+00:00,2026-10-05 01:10:00+00:00
9,FVG_S_20261005T020500Z,BEARISH_FVG,2026-10-05 02:05:00+00:00,1.12578,1.12582,0.00004,2026-10-05 01:55:00+00:00,2026-10-05 02:00:00+00:00,2026-10-05 02:05:00+00:00


In [118]:
# ============================================================
# RESTORE FVG QUALITY AUDIT V1
# ============================================================

def audit_fvg_quality_v1(
    fvg_df: pd.DataFrame,
    m5_df: pd.DataFrame,
) -> pd.DataFrame:

    rows = []

    for _, fvg in fvg_df.iterrows():

        a = m5_df.loc[fvg["candle_a"]]
        b = m5_df.loc[fvg["candle_b"]]
        c = m5_df.loc[fvg["candle_c"]]

        atr = float(c["atr"]) if "atr" in c.index else np.nan

        candle_b_range = (
            float(b["high"]) - float(b["low"])
        )

        candle_b_body = abs(
            float(b["close"]) - float(b["open"])
        )

        gap_size = float(fvg["gap_size"])

        gap_atr_ratio = (
            gap_size / atr
            if pd.notna(atr) and atr > 0
            else np.nan
        )

        range_atr_ratio = (
            candle_b_range / atr
            if pd.notna(atr) and atr > 0
            else np.nan
        )

        body_atr_ratio = (
            candle_b_body / atr
            if pd.notna(atr) and atr > 0
            else np.nan
        )

        if fvg["fvg_type"] == "BULLISH_FVG":
            directional = (
                float(b["close"]) > float(b["open"])
            )
        else:
            directional = (
                float(b["close"]) < float(b["open"])
            )

        rows.append({
            "fvg_id": fvg["fvg_id"],
            "fvg_type": fvg["fvg_type"],
            "created_at": fvg["created_at"],
            "gap_size": gap_size,
            "atr": atr,
            "gap_atr_ratio": gap_atr_ratio,
            "candle_b_range": candle_b_range,
            "range_atr_ratio": range_atr_ratio,
            "candle_b_body": candle_b_body,
            "body_atr_ratio": body_atr_ratio,
            "directional_candle_b": directional,
        })

    audit = pd.DataFrame(rows)

    return audit


# ============================================================
# BUILD ATR DATAFRAME
# ============================================================

m5_atr_df = m5_df.copy()

prev_close = m5_atr_df["close"].shift(1)

tr = pd.concat(
    [
        m5_atr_df["high"] - m5_atr_df["low"],
        (m5_atr_df["high"] - prev_close).abs(),
        (m5_atr_df["low"] - prev_close).abs(),
    ],
    axis=1,
).max(axis=1)

m5_atr_df["tr"] = tr

m5_atr_df["atr"] = (
    m5_atr_df["tr"]
    .rolling(window=14, min_periods=14)
    .mean()
)


# ============================================================
# BUILD QUALITY AUDIT
# ============================================================

fvg_quality_audit_v1 = audit_fvg_quality_v1(
    fvg_df=fvg_v1,
    m5_df=m5_atr_df,
)


# ============================================================
# AUDIT REPORT
# ============================================================

print("=" * 100)
print("FVG QUALITY AUDIT V1 — RESTORED")
print("=" * 100)

print(f"Total FVGs        : {len(fvg_quality_audit_v1):,}")
print(
    f"ATR available     : "
    f"{fvg_quality_audit_v1['atr'].notna().sum():,}"
)
print(
    f"ATR unavailable   : "
    f"{fvg_quality_audit_v1['atr'].isna().sum():,}"
)

print("\nFVG Type Counts:")
print(
    fvg_quality_audit_v1["fvg_type"]
    .value_counts()
)

print("\nDirectional Candle-B:")
print(
    fvg_quality_audit_v1["directional_candle_b"]
    .value_counts()
)

display(
    fvg_quality_audit_v1[
        [
            "fvg_id",
            "fvg_type",
            "created_at",
            "gap_size",
            "atr",
            "gap_atr_ratio",
            "candle_b_range",
            "range_atr_ratio",
            "candle_b_body",
            "body_atr_ratio",
            "directional_candle_b",
        ]
    ].head(10)
)

FVG QUALITY AUDIT V1 — RESTORED
Total FVGs        : 179
ATR available     : 178
ATR unavailable   : 1

FVG Type Counts:
fvg_type
BEARISH_FVG    100
BULLISH_FVG     79
Name: count, dtype: int64

Directional Candle-B:
directional_candle_b
True     178
False      1
Name: count, dtype: int64


,fvg_id,fvg_type,created_at,gap_size,atr,gap_atr_ratio,candle_b_range,range_atr_ratio,candle_b_body,body_atr_ratio,directional_candle_b
0,FVG_S_20261002T202500Z,BEARISH_FVG,2026-10-02 20:25:00+00:00,0.00006,NaN,NaN,0.00043,NaN,0.00020,NaN,True
1,FVG_B_20261002T221500Z,BULLISH_FVG,2026-10-02 22:15:00+00:00,0.00006,0.000275,0.218182,0.00042,1.527273,0.00030,1.090909,True
2,FVG_B_20261002T222500Z,BULLISH_FVG,2026-10-02 22:25:00+00:00,0.00023,0.000299,0.768496,0.00055,1.837709,0.00043,1.436754,True
3,FVG_S_20261002T234000Z,BEARISH_FVG,2026-10-02 23:40:00+00:00,0.00002,0.000307,0.065116,0.00033,1.074419,0.00029,0.944186,True
4,FVG_B_20261005T001500Z,BULLISH_FVG,2026-10-05 00:15:00+00:00,0.00045,0.000341,1.317992,0.00019,0.556485,0.00007,0.205021,True
5,FVG_B_20261005T002000Z,BULLISH_FVG,2026-10-05 00:20:00+00:00,0.00048,0.000339,1.417722,0.00020,0.590717,0.00009,0.265823,True
6,FVG_S_20261005T005500Z,BEARISH_FVG,2026-10-05 00:55:00+00:00,0.00029,0.000329,0.882609,0.00074,2.252174,0.00037,1.126087,True
7,FVG_B_20261005T010500Z,BULLISH_FVG,2026-10-05 01:05:00+00:00,0.00002,0.000362,0.055227,0.00056,1.546351,0.00025,0.690335,True
8,FVG_B_20261005T011000Z,BULLISH_FVG,2026-10-05 01:10:00+00:00,0.00007,0.000362,0.193294,0.00053,1.463511,0.00029,0.800789,True
9,FVG_S_20261005T020500Z,BEARISH_FVG,2026-10-05 02:05:00+00:00,0.00004,0.000274,0.145833,0.00028,1.020833,0.00026,0.947917,True


In [119]:
FVG_VALIDATION_MIN_GAP_ATR = 0.20
FVG_VALIDATION_MIN_BODY_ATR = 0.50


def validate_fvg_v1(
    fvg_df: pd.DataFrame,
    quality_audit: pd.DataFrame,
) -> pd.DataFrame:
    """
    FVG Validation V1

    Contract:
        1. ATR must be available.
        2. gap / ATR >= 0.20
        3. Candle-B body / ATR >= 0.50

    This stage validates FVG quality only.
    No lifecycle / mitigation logic is used.
    """

    required = {
        "fvg_id",
        "fvg_type",
        "created_at",
        "lower_bound",
        "upper_bound",
        "gap_size",
    }

    missing = required - set(fvg_df.columns)

    if missing:
        raise ValueError(
            f"Missing FVG columns: {sorted(missing)}"
        )

    required_audit = {
        "fvg_id",
        "atr",
        "gap_atr_ratio",
        "body_atr_ratio",
    }

    missing_audit = (
        required_audit
        - set(quality_audit.columns)
    )

    if missing_audit:
        raise ValueError(
            "Missing quality audit columns: "
            f"{sorted(missing_audit)}"
        )

    df = fvg_df.copy()

    audit = quality_audit[
        [
            "fvg_id",
            "atr",
            "gap_atr_ratio",
            "body_atr_ratio",
            "range_atr_ratio",
            "directional_candle_b",
        ]
    ].copy()

    result = df.merge(
        audit,
        on="fvg_id",
        how="left",
        validate="one_to_one",
    )

    result["validation_status"] = "INVALID"
    result["validation_reason"] = "UNKNOWN"

    atr_available = result["atr"].notna()
    gap_valid = (
        result["gap_atr_ratio"]
        >= FVG_VALIDATION_MIN_GAP_ATR
    )
    body_valid = (
        result["body_atr_ratio"]
        >= FVG_VALIDATION_MIN_BODY_ATR
    )

    valid = (
        atr_available
        & gap_valid
        & body_valid
    )

    result.loc[
        ~atr_available,
        "validation_reason"
    ] = "ATR_UNAVAILABLE"

    result.loc[
        atr_available & ~gap_valid,
        "validation_reason"
    ] = "INSUFFICIENT_GAP"

    result.loc[
        atr_available & gap_valid & ~body_valid,
        "validation_reason"
    ] = "INSUFFICIENT_DISPLACEMENT"

    result.loc[
        valid,
        "validation_status"
    ] = "VALID"

    result.loc[
        valid,
        "validation_reason"
    ] = "PASSED"

    return result


fvg_validated_v1 = validate_fvg_v1(
    fvg_df=fvg_v1,
    quality_audit=fvg_quality_audit_v1,
)

In [120]:
print("=" * 100)
print("FVG VALIDATION V1 — REPORT")
print("=" * 100)

print(f"\nTotal FVGs : {len(fvg_validated_v1):,}")

print("\nValidation Status:")
print(
    fvg_validated_v1["validation_status"]
    .value_counts(dropna=False)
)

print("\nValidation Reasons:")
print(
    fvg_validated_v1["validation_reason"]
    .value_counts(dropna=False)
)

print("\nFVG Type × Validation Status:")
print(
    pd.crosstab(
        fvg_validated_v1["fvg_type"],
        fvg_validated_v1["validation_status"],
        margins=True,
    )
)

print("\nVALID FVG sample:")
display(
    fvg_validated_v1[
        fvg_validated_v1["validation_status"] == "VALID"
    ].head(10)
)

print("\nINVALID FVG sample:")
display(
    fvg_validated_v1[
        fvg_validated_v1["validation_status"] == "INVALID"
    ].head(10)
)

FVG VALIDATION V1 — REPORT

Total FVGs : 179

Validation Status:
validation_status
VALID      99
INVALID    80
Name: count, dtype: int64

Validation Reasons:
validation_reason
PASSED                       99
INSUFFICIENT_GAP             71
INSUFFICIENT_DISPLACEMENT     8
ATR_UNAVAILABLE               1
Name: count, dtype: int64

FVG Type × Validation Status:
validation_status  INVALID  VALID  All
fvg_type                              
BEARISH_FVG             44     56  100
BULLISH_FVG             36     43   79
All                     80     99  179

VALID FVG sample:


,fvg_id,fvg_type,created_at,lower_bound,upper_bound,gap_size,candle_a,candle_b,candle_c,atr,gap_atr_ratio,body_atr_ratio,range_atr_ratio,directional_candle_b,validation_status,validation_reason
1,FVG_B_20261002T221500Z,BULLISH_FVG,2026-10-02 22:15:00+00:00,1.12520,1.12526,0.00006,2026-10-02 22:05:00+00:00,2026-10-02 22:10:00+00:00,2026-10-02 22:15:00+00:00,0.000275,0.218182,1.090909,1.527273,True,VALID,PASSED
2,FVG_B_20261002T222500Z,BULLISH_FVG,2026-10-02 22:25:00+00:00,1.12554,1.12577,0.00023,2026-10-02 22:15:00+00:00,2026-10-02 22:20:00+00:00,2026-10-02 22:25:00+00:00,0.000299,0.768496,1.436754,1.837709,True,VALID,PASSED
6,FVG_S_20261005T005500Z,BEARISH_FVG,2026-10-05 00:55:00+00:00,1.12538,1.12567,0.00029,2026-10-05 00:45:00+00:00,2026-10-05 00:50:00+00:00,2026-10-05 00:55:00+00:00,0.000329,0.882609,1.126087,2.252174,True,VALID,PASSED
11,FVG_S_20261005T031500Z,BEARISH_FVG,2026-10-05 03:15:00+00:00,1.12498,1.12528,0.00030,2026-10-05 03:05:00+00:00,2026-10-05 03:10:00+00:00,2026-10-05 03:15:00+00:00,0.000266,1.126005,1.951743,2.176944,True,VALID,PASSED
12,FVG_S_20261005T034500Z,BEARISH_FVG,2026-10-05 03:45:00+00:00,1.12451,1.12459,0.00008,2026-10-05 03:35:00+00:00,2026-10-05 03:40:00+00:00,2026-10-05 03:45:00+00:00,0.000289,0.276543,0.725926,1.071605,True,VALID,PASSED
13,FVG_S_20261005T041000Z,BEARISH_FVG,2026-10-05 04:10:00+00:00,1.12392,1.12406,0.00014,2026-10-05 04:00:00+00:00,2026-10-05 04:05:00+00:00,2026-10-05 04:10:00+00:00,0.000329,0.425163,1.457701,1.700651,True,VALID,PASSED
17,FVG_S_20261005T051000Z,BEARISH_FVG,2026-10-05 05:10:00+00:00,1.12282,1.12293,0.00011,2026-10-05 05:00:00+00:00,2026-10-05 05:05:00+00:00,2026-10-05 05:10:00+00:00,0.000323,0.340708,0.929204,1.269912,True,VALID,PASSED
18,FVG_S_20261005T051500Z,BEARISH_FVG,2026-10-05 05:15:00+00:00,1.12249,1.12267,0.00018,2026-10-05 05:05:00+00:00,2026-10-05 05:10:00+00:00,2026-10-05 05:15:00+00:00,0.000374,0.480916,0.854962,1.389313,True,VALID,PASSED
19,FVG_S_20261005T052000Z,BEARISH_FVG,2026-10-05 05:20:00+00:00,1.12152,1.12230,0.00078,2026-10-05 05:10:00+00:00,2026-10-05 05:15:00+00:00,2026-10-05 05:20:00+00:00,0.000407,1.915789,2.677193,3.143860,True,VALID,PASSED
21,FVG_S_20261005T055000Z,BEARISH_FVG,2026-10-05 05:50:00+00:00,1.11950,1.12028,0.00078,2026-10-05 05:40:00+00:00,2026-10-05 05:45:00+00:00,2026-10-05 05:50:00+00:00,0.000651,1.197368,2.747807,3.346491,True,VALID,PASSED



INVALID FVG sample:


,fvg_id,fvg_type,created_at,lower_bound,upper_bound,gap_size,candle_a,candle_b,candle_c,atr,gap_atr_ratio,body_atr_ratio,range_atr_ratio,directional_candle_b,validation_status,validation_reason
0,FVG_S_20261002T202500Z,BEARISH_FVG,2026-10-02 20:25:00+00:00,1.12559,1.12565,0.00006,2026-10-02 20:15:00+00:00,2026-10-02 20:20:00+00:00,2026-10-02 20:25:00+00:00,NaN,NaN,NaN,NaN,True,INVALID,ATR_UNAVAILABLE
3,FVG_S_20261002T234000Z,BEARISH_FVG,2026-10-02 23:40:00+00:00,1.12522,1.12524,0.00002,2026-10-02 23:30:00+00:00,2026-10-02 23:35:00+00:00,2026-10-02 23:40:00+00:00,0.000307,0.065116,0.944186,1.074419,True,INVALID,INSUFFICIENT_GAP
4,FVG_B_20261005T001500Z,BULLISH_FVG,2026-10-05 00:15:00+00:00,1.12513,1.12558,0.00045,2026-10-02 23:45:00+00:00,2026-10-02 23:50:00+00:00,2026-10-05 00:15:00+00:00,0.000341,1.317992,0.205021,0.556485,True,INVALID,INSUFFICIENT_DISPLACEMENT
5,FVG_B_20261005T002000Z,BULLISH_FVG,2026-10-05 00:20:00+00:00,1.12507,1.12555,0.00048,2026-10-02 23:50:00+00:00,2026-10-05 00:15:00+00:00,2026-10-05 00:20:00+00:00,0.000339,1.417722,0.265823,0.590717,True,INVALID,INSUFFICIENT_DISPLACEMENT
7,FVG_B_20261005T010500Z,BULLISH_FVG,2026-10-05 01:05:00+00:00,1.12538,1.12540,0.00002,2026-10-05 00:55:00+00:00,2026-10-05 01:00:00+00:00,2026-10-05 01:05:00+00:00,0.000362,0.055227,0.690335,1.546351,True,INVALID,INSUFFICIENT_GAP
8,FVG_B_20261005T011000Z,BULLISH_FVG,2026-10-05 01:10:00+00:00,1.12565,1.12572,0.00007,2026-10-05 01:00:00+00:00,2026-10-05 01:05:00+00:00,2026-10-05 01:10:00+00:00,0.000362,0.193294,0.800789,1.463511,True,INVALID,INSUFFICIENT_GAP
9,FVG_S_20261005T020500Z,BEARISH_FVG,2026-10-05 02:05:00+00:00,1.12578,1.12582,0.00004,2026-10-05 01:55:00+00:00,2026-10-05 02:00:00+00:00,2026-10-05 02:05:00+00:00,0.000274,0.145833,0.947917,1.020833,True,INVALID,INSUFFICIENT_GAP
10,FVG_S_20261005T021000Z,BEARISH_FVG,2026-10-05 02:10:00+00:00,1.12565,1.12567,0.00002,2026-10-05 02:00:00+00:00,2026-10-05 02:05:00+00:00,2026-10-05 02:10:00+00:00,0.000245,0.081633,0.285714,0.979592,True,INVALID,INSUFFICIENT_GAP
14,FVG_S_20261005T042000Z,BEARISH_FVG,2026-10-05 04:20:00+00:00,1.12369,1.12371,0.00002,2026-10-05 04:10:00+00:00,2026-10-05 04:15:00+00:00,2026-10-05 04:20:00+00:00,0.000314,0.063781,0.446469,1.084282,True,INVALID,INSUFFICIENT_GAP
15,FVG_S_20261005T044500Z,BEARISH_FVG,2026-10-05 04:45:00+00:00,1.12328,1.12330,0.00002,2026-10-05 04:35:00+00:00,2026-10-05 04:40:00+00:00,2026-10-05 04:45:00+00:00,0.000315,0.063492,0.539683,0.793651,True,INVALID,INSUFFICIENT_GAP


In [121]:
# ==========================================================================================
# FVG LIFECYCLE V1 — REFERENCE DETECTOR
# ==========================================================================================

def detect_fvg_lifecycle_v1_reference(
    fvg_validated: pd.DataFrame,
    m5_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    FVG Lifecycle V1 — Reference Implementation

    States:
        FRESH
        MITIGATED
        FILLED

    Rules:
        - Only VALID FVGs are processed.
        - Lifecycle scanning starts strictly AFTER created_at.
        - Bullish FVG:
            mitigation -> candle.low <= upper_bound
                           and candle.low > lower_bound
            fill       -> candle.low <= lower_bound
        - Bearish FVG:
            mitigation -> candle.high >= lower_bound
                           and candle.high < upper_bound
            fill       -> candle.high >= upper_bound
        - If mitigation and fill happen in the same candle:
            direct FRESH -> FILLED
        - First lifecycle event wins chronologically.
        - No look-ahead.
        - FVG identity is immutable.
    """

    required_fvg = {
        "fvg_id",
        "fvg_type",
        "created_at",
        "lower_bound",
        "upper_bound",
        "gap_size",
        "candle_a",
        "candle_b",
        "candle_c",
        "validation_status",
    }

    missing_fvg = required_fvg - set(fvg_validated.columns)

    if missing_fvg:
        raise ValueError(
            f"Missing FVG columns: {sorted(missing_fvg)}"
        )

    required_m5 = {
        "open",
        "high",
        "low",
        "close",
    }

    missing_m5 = required_m5 - set(m5_df.columns)

    if missing_m5:
        raise ValueError(
            f"Missing M5 columns: {sorted(missing_m5)}"
        )

    if not isinstance(m5_df.index, pd.DatetimeIndex):
        raise TypeError(
            "m5_df index must be a pandas DatetimeIndex."
        )

    if m5_df.index.tz is None:
        raise ValueError(
            "m5_df index must be timezone-aware."
        )

    df = m5_df.sort_index()

    valid_fvgs = fvg_validated.loc[
        fvg_validated["validation_status"] == "VALID"
    ].copy()

    rows = []

    for _, fvg in valid_fvgs.iterrows():

        fvg_id = fvg["fvg_id"]
        fvg_type = fvg["fvg_type"]
        created_at = fvg["created_at"]

        lower_bound = float(fvg["lower_bound"])
        upper_bound = float(fvg["upper_bound"])

        lifecycle_state = "FRESH"
        event_type = None
        event_time = pd.NaT
        event_price = np.nan

        # ------------------------------------------------------------------
        # Strictly AFTER creation candle
        # ------------------------------------------------------------------
        future_df = df.loc[
            df.index > created_at
        ]

        for candle_time, candle in future_df.iterrows():

            candle_high = float(candle["high"])
            candle_low = float(candle["low"])

            # ==============================================================
            # BULLISH FVG
            # ==============================================================
            if fvg_type == "BULLISH_FVG":

                # Full fill has priority within the same candle.
                if candle_low <= lower_bound:

                    lifecycle_state = "FILLED"
                    event_type = "FILL"
                    event_time = candle_time
                    event_price = candle_low
                    break

                # Partial penetration / mitigation.
                elif candle_low <= upper_bound:

                    lifecycle_state = "MITIGATED"
                    event_type = "MITIGATION"
                    event_time = candle_time
                    event_price = candle_low
                    break

            # ==============================================================
            # BEARISH FVG
            # ==============================================================
            elif fvg_type == "BEARISH_FVG":

                # Full fill has priority within the same candle.
                if candle_high >= upper_bound:

                    lifecycle_state = "FILLED"
                    event_type = "FILL"
                    event_time = candle_time
                    event_price = candle_high
                    break

                # Partial penetration / mitigation.
                elif candle_high >= lower_bound:

                    lifecycle_state = "MITIGATED"
                    event_type = "MITIGATION"
                    event_time = candle_time
                    event_price = candle_high
                    break

            else:
                raise ValueError(
                    f"Unknown FVG type: {fvg_type}"
                )

        # ------------------------------------------------------------------
        # Output
        # ------------------------------------------------------------------
        rows.append({
            "fvg_id": fvg_id,
            "fvg_type": fvg_type,
            "created_at": created_at,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "gap_size": float(fvg["gap_size"]),
            "candle_a": fvg["candle_a"],
            "candle_b": fvg["candle_b"],
            "candle_c": fvg["candle_c"],
            "lifecycle_state": lifecycle_state,
            "event_type": event_type,
            "event_time": event_time,
            "event_price": event_price,
        })

    result = pd.DataFrame(rows)

    if result.empty:
        return pd.DataFrame(
            columns=[
                "fvg_id",
                "fvg_type",
                "created_at",
                "lower_bound",
                "upper_bound",
                "gap_size",
                "candle_a",
                "candle_b",
                "candle_c",
                "lifecycle_state",
                "event_type",
                "event_time",
                "event_price",
            ]
        )

    return (
        result
        .sort_values("created_at")
        .reset_index(drop=True)
    )


# ==========================================================================================
# EXECUTE
# ==========================================================================================

fvg_lifecycle_reference_v1 = detect_fvg_lifecycle_v1_reference(
    fvg_validated=fvg_validated_v1,
    m5_df=m5_df,
)


# ==========================================================================================
# AUDIT OUTPUT
# ==========================================================================================

print("=" * 100)
print("FVG LIFECYCLE V1 — REFERENCE DETECTOR")
print("=" * 100)

print(f"\nInput VALID FVGs : {len(fvg_validated_v1[fvg_validated_v1['validation_status'] == 'VALID']):,}")
print(f"Reference rows   : {len(fvg_lifecycle_reference_v1):,}")

print("\nLifecycle State:")
print(
    fvg_lifecycle_reference_v1["lifecycle_state"]
    .value_counts(dropna=False)
)

print("\nEvent Type:")
print(
    fvg_lifecycle_reference_v1["event_type"]
    .value_counts(dropna=False)
)

print("\nFVG Type × Lifecycle:")
print(
    pd.crosstab(
        fvg_lifecycle_reference_v1["fvg_type"],
        fvg_lifecycle_reference_v1["lifecycle_state"],
        margins=True,
    )
)

print("\nReference Sample:")
display(
    fvg_lifecycle_reference_v1.head(10)
)

FVG LIFECYCLE V1 — REFERENCE DETECTOR

Input VALID FVGs : 99
Reference rows   : 99

Lifecycle State:
lifecycle_state
MITIGATED    59
FILLED       36
FRESH         4
Name: count, dtype: int64

Event Type:
event_type
MITIGATION    59
FILL          36
NaN            4
Name: count, dtype: int64

FVG Type × Lifecycle:
lifecycle_state  FILLED  FRESH  MITIGATED  All
fvg_type                                      
BEARISH_FVG          19      3         34   56
BULLISH_FVG          17      1         25   43
All                  36      4         59   99

Reference Sample:


,fvg_id,fvg_type,created_at,lower_bound,upper_bound,gap_size,candle_a,candle_b,candle_c,lifecycle_state,event_type,event_time,event_price
0,FVG_B_20261002T221500Z,BULLISH_FVG,2026-10-02 22:15:00+00:00,1.12520,1.12526,0.00006,2026-10-02 22:05:00+00:00,2026-10-02 22:10:00+00:00,2026-10-02 22:15:00+00:00,FILLED,FILL,2026-10-02 23:15:00+00:00,1.12518
1,FVG_B_20261002T222500Z,BULLISH_FVG,2026-10-02 22:25:00+00:00,1.12554,1.12577,0.00023,2026-10-02 22:15:00+00:00,2026-10-02 22:20:00+00:00,2026-10-02 22:25:00+00:00,MITIGATED,MITIGATION,2026-10-02 22:30:00+00:00,1.12577
2,FVG_S_20261005T005500Z,BEARISH_FVG,2026-10-05 00:55:00+00:00,1.12538,1.12567,0.00029,2026-10-05 00:45:00+00:00,2026-10-05 00:50:00+00:00,2026-10-05 00:55:00+00:00,MITIGATED,MITIGATION,2026-10-05 01:00:00+00:00,1.12565
3,FVG_S_20261005T031500Z,BEARISH_FVG,2026-10-05 03:15:00+00:00,1.12498,1.12528,0.00030,2026-10-05 03:05:00+00:00,2026-10-05 03:10:00+00:00,2026-10-05 03:15:00+00:00,FILLED,FILL,2026-10-06 14:10:00+00:00,1.12529
4,FVG_S_20261005T034500Z,BEARISH_FVG,2026-10-05 03:45:00+00:00,1.12451,1.12459,0.00008,2026-10-05 03:35:00+00:00,2026-10-05 03:40:00+00:00,2026-10-05 03:45:00+00:00,FILLED,FILL,2026-10-06 11:40:00+00:00,1.12472
5,FVG_S_20261005T041000Z,BEARISH_FVG,2026-10-05 04:10:00+00:00,1.12392,1.12406,0.00014,2026-10-05 04:00:00+00:00,2026-10-05 04:05:00+00:00,2026-10-05 04:10:00+00:00,FILLED,FILL,2026-10-06 10:55:00+00:00,1.12428
6,FVG_S_20261005T051000Z,BEARISH_FVG,2026-10-05 05:10:00+00:00,1.12282,1.12293,0.00011,2026-10-05 05:00:00+00:00,2026-10-05 05:05:00+00:00,2026-10-05 05:10:00+00:00,FILLED,FILL,2026-10-06 04:00:00+00:00,1.12293
7,FVG_S_20261005T051500Z,BEARISH_FVG,2026-10-05 05:15:00+00:00,1.12249,1.12267,0.00018,2026-10-05 05:05:00+00:00,2026-10-05 05:10:00+00:00,2026-10-05 05:15:00+00:00,MITIGATED,MITIGATION,2026-10-05 23:50:00+00:00,1.12249
8,FVG_S_20261005T052000Z,BEARISH_FVG,2026-10-05 05:20:00+00:00,1.12152,1.12230,0.00078,2026-10-05 05:10:00+00:00,2026-10-05 05:15:00+00:00,2026-10-05 05:20:00+00:00,MITIGATED,MITIGATION,2026-10-05 11:50:00+00:00,1.12170
9,FVG_S_20261005T055000Z,BEARISH_FVG,2026-10-05 05:50:00+00:00,1.11950,1.12028,0.00078,2026-10-05 05:40:00+00:00,2026-10-05 05:45:00+00:00,2026-10-05 05:50:00+00:00,MITIGATED,MITIGATION,2026-10-05 09:00:00+00:00,1.11959


In [122]:
def _normalize_m5_dataframe(
    m5_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Normalize M5 dataframe for FVG Engine V1.

    Canonical input contract:
    - OHLC columns must exist
    - timestamp may come from:
        1. `time` column
        2. DatetimeIndex
    - canonical internal representation uses `time` column
    - timestamps are UTC
    - chronological order is deterministic
    - duplicate candle timestamps are forbidden
    """

    if not isinstance(m5_df, pd.DataFrame):
        raise TypeError(
            "m5_df must be a pandas DataFrame."
        )

    required_ohlc = [
        "open",
        "high",
        "low",
        "close",
    ]

    missing = [
        col
        for col in required_ohlc
        if col not in m5_df.columns
    ]

    if missing:
        raise ValueError(
            f"m5_df is missing required columns: {missing}"
        )

    # ========================================================
    # Work on an isolated copy
    # ========================================================

    df = m5_df.copy(deep=True)

    # ========================================================
    # Resolve canonical timestamp source
    # ========================================================

    if "time" in df.columns:

        # Existing canonical time column
        time_values = pd.to_datetime(
            df["time"],
            utc=True,
            errors="raise",
        )

    elif isinstance(df.index, pd.DatetimeIndex):

        # Timestamp is stored in DatetimeIndex
        if df.index.tz is None:
            raise ValueError(
                "m5_df DatetimeIndex must be timezone-aware."
            )

        time_values = pd.to_datetime(
            df.index,
            utc=True,
            errors="raise",
        )

    else:

        raise ValueError(
            "m5_df must contain a `time` column "
            "or have a timezone-aware DatetimeIndex."
        )

    # ========================================================
    # Remove index ambiguity completely
    # ========================================================

    df = df.reset_index(drop=True)

    # Canonical timestamp column
    df["time"] = pd.Series(
        time_values,
        index=df.index,
    )

    # ========================================================
    # Numeric OHLC normalization
    # ========================================================

    for col in required_ohlc:
        df[col] = pd.to_numeric(
            df[col],
            errors="raise",
        )

    # ========================================================
    # Deterministic chronological ordering
    # ========================================================

    df = (
        df
        .sort_values(
            "time",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    # ========================================================
    # No duplicate candle timestamps
    # ========================================================

    duplicate_times = df["time"].duplicated().sum()

    if duplicate_times:
        raise ValueError(
            f"m5_df contains "
            f"{duplicate_times} duplicate timestamps."
        )

    return df

In [123]:
# ============================================================
# FVG ENGINE V1 — CANONICAL BUILDER
# ============================================================

def build_fvg_engine_v1(
    fvg_validated: pd.DataFrame,
    m5_df: pd.DataFrame,
) -> pd.DataFrame:

    required_fvg = {
        "fvg_id",
        "fvg_type",
        "created_at",
        "lower_bound",
        "upper_bound",
        "gap_size",
        "candle_a",
        "candle_b",
        "candle_c",
        "validation_status",
    }

    required_m5 = {
        "datetime",
        "open",
        "high",
        "low",
        "close",
    }

    missing_fvg = required_fvg - set(fvg_validated.columns)
    if missing_fvg:
        raise ValueError(
            f"Missing FVG columns: {sorted(missing_fvg)}"
        )

    missing_m5 = required_m5 - set(m5_df.columns)
    if missing_m5:
        raise ValueError(
            f"Missing M5 columns: {sorted(missing_m5)}"
        )

    df = fvg_validated.copy()

    # ------------------------------------------------------------------
    # Normalize FVG datetimes
    # ------------------------------------------------------------------

    for col in [
        "created_at",
        "candle_a",
        "candle_b",
        "candle_c",
    ]:
        df[col] = pd.to_datetime(
            df[col],
            utc=True,
            errors="coerce",
        )

    if df[
        [
            "created_at",
            "candle_a",
            "candle_b",
            "candle_c",
        ]
    ].isna().any().any():
        raise ValueError(
            "FVG Engine V1: invalid/null datetime values detected."
        )

    # ------------------------------------------------------------------
    # Only VALID FVGs enter canonical Engine
    # ------------------------------------------------------------------

    df = df.loc[
        df["validation_status"].eq("VALID")
    ].copy()

    # ------------------------------------------------------------------
    # Identity contract
    # ------------------------------------------------------------------

    if df["fvg_id"].duplicated().any():
        raise ValueError(
            "FVG Engine V1: duplicate fvg_id detected."
        )

    allowed_types = {
        "BULLISH_FVG",
        "BEARISH_FVG",
    }

    actual_types = set(
        df["fvg_type"]
        .dropna()
        .unique()
    )

    invalid_types = actual_types - allowed_types

    if invalid_types:
        raise ValueError(
            f"Invalid FVG types detected: {sorted(invalid_types)}"
        )

    # ------------------------------------------------------------------
    # Market preparation
    # ------------------------------------------------------------------

    market = m5_df.copy()

    market["datetime"] = pd.to_datetime(
        market["datetime"],
        utc=True,
        errors="coerce",
    )

    if market["datetime"].isna().any():
        raise ValueError(
            "FVG Engine V1: invalid/null market datetime detected."
        )

    market = (
        market
        .sort_values("datetime")
        .drop_duplicates(
            subset=["datetime"],
            keep="last",
        )
        .reset_index(drop=True)
    )

    # ------------------------------------------------------------------
    # Lifecycle resolution
    #
    # Lifecycle is determined from future market candles AFTER
    # FVG creation candle.
    # ------------------------------------------------------------------

    lifecycle_state = []
    event_type = []
    event_time = []
    event_price = []

    market_times = market["datetime"].to_numpy()

    for _, row in df.iterrows():

        fvg_type = row["fvg_type"]
        lower = float(row["lower_bound"])
        upper = float(row["upper_bound"])
        created_at = row["created_at"]

        future = market.loc[
            market["datetime"] > created_at
        ]

        state = "FRESH"
        evt_type = None
        evt_time = pd.NaT
        evt_price = np.nan

        for _, candle in future.iterrows():

            high = float(candle["high"])
            low = float(candle["low"])

            candle_time = candle["datetime"]

            # ------------------------------------------------------------------
            # Bullish FVG
            #
            # Fill = price reaches lower boundary
            # Mitigation = price enters zone but does not fill
            # ------------------------------------------------------------------

            if fvg_type == "BULLISH_FVG":

                if low <= lower:
                    state = "FILLED"
                    evt_type = "FILL"
                    evt_time = candle_time
                    evt_price = lower
                    break

                elif low < upper:
                    state = "MITIGATED"
                    evt_type = "MITIGATION"
                    evt_time = candle_time
                    evt_price = low
                    break

            # ------------------------------------------------------------------
            # Bearish FVG
            #
            # Fill = price reaches upper boundary
            # Mitigation = price enters zone but does not fill
            # ------------------------------------------------------------------

            elif fvg_type == "BEARISH_FVG":

                if high >= upper:
                    state = "FILLED"
                    evt_type = "FILL"
                    evt_time = candle_time
                    evt_price = upper
                    break

                elif high > lower:
                    state = "MITIGATED"
                    evt_type = "MITIGATION"
                    evt_time = candle_time
                    evt_price = high
                    break

        lifecycle_state.append(state)
        event_type.append(evt_type)
        event_time.append(evt_time)
        event_price.append(evt_price)

    # ------------------------------------------------------------------
    # Canonical output
    # ------------------------------------------------------------------

    result = df[
        [
            "fvg_id",
            "fvg_type",
            "created_at",
            "lower_bound",
            "upper_bound",
            "gap_size",
            "candle_a",
            "candle_b",
            "candle_c",
        ]
    ].copy()

    result["lifecycle_state"] = lifecycle_state
    result["event_type"] = event_type
    result["event_time"] = pd.to_datetime(
        event_time,
        utc=True,
    )
    result["event_price"] = event_price

    result = (
        result
        .sort_values("created_at")
        .reset_index(drop=True)
    )

    return result

In [124]:
# ============================================================
# FVG ENGINE V1 — CANONICAL BUILD
# ============================================================

fvg_engine_v1 = build_fvg_engine_v1(
    fvg_validated=fvg_validated_v1,
    m5_df=m5_df,
)

print("=" * 100)
print("FVG ENGINE V1 — CANONICAL OUTPUT")
print("=" * 100)

print(f"Rows        : {len(fvg_engine_v1)}")
print(f"Columns     : {len(fvg_engine_v1.columns)}")

print("\nLifecycle:")
print(fvg_engine_v1["lifecycle_state"].value_counts(dropna=False))

print("\nEvent:")
print(fvg_engine_v1["event_type"].value_counts(dropna=False))

print("\nFVG Type:")
print(fvg_engine_v1["fvg_type"].value_counts())

print("\nSchema:")
print(list(fvg_engine_v1.columns))

FVG ENGINE V1 — CANONICAL OUTPUT
Rows        : 99
Columns     : 13

Lifecycle:
lifecycle_state
MITIGATED    57
FILLED       37
FRESH         5
Name: count, dtype: int64

Event:
event_type
MITIGATION    57
FILL          37
NaN            5
Name: count, dtype: int64

FVG Type:
fvg_type
BEARISH_FVG    56
BULLISH_FVG    43
Name: count, dtype: int64

Schema:
['fvg_id', 'fvg_type', 'created_at', 'lower_bound', 'upper_bound', 'gap_size', 'candle_a', 'candle_b', 'candle_c', 'lifecycle_state', 'event_type', 'event_time', 'event_price']


In [125]:
# Find DataFrame variables related to FVG

fvg_dataframe_candidates = {
    name: value.shape
    for name, value in globals().items()
    if isinstance(value, pd.DataFrame)
    and (
        "fvg" in name.lower()
        or (
            "fvg_id" in value.columns
            if hasattr(value, "columns")
            else False
        )
    )
}

print("=" * 100)
print("FVG DATAFRAME CANDIDATES")
print("=" * 100)

for name, shape in fvg_dataframe_candidates.items():
    print(f"{name:<45} {shape}")

FVG DATAFRAME CANDIDATES
fvg_v1                                        (179, 9)
fvg_quality_audit_v1                          (179, 11)
fvg_validated_v1                              (179, 16)
fvg_lifecycle_reference_v1                    (99, 13)
fvg_engine_v1                                 (99, 13)


In [126]:
# ================================================================================================
# FVG ENGINE V1 — FROZEN SCHEMA AUDIT
# ================================================================================================

print("=" * 100)
print("FVG ENGINE V1 — FROZEN SCHEMA AUDIT")
print("=" * 100)

print(f"Rows    : {len(fvg_engine_v1)}")
print(f"Columns : {len(fvg_engine_v1.columns)}")

print("\nColumns:")
for i, col in enumerate(fvg_engine_v1.columns, 1):
    print(f"{i:02d}. {col}")

print("\nDtypes:")
print(fvg_engine_v1.dtypes)

print("\nSample:")
display(fvg_engine_v1.head(10))

FVG ENGINE V1 — FROZEN SCHEMA AUDIT
Rows    : 99
Columns : 13

Columns:
01. fvg_id
02. fvg_type
03. created_at
04. lower_bound
05. upper_bound
06. gap_size
07. candle_a
08. candle_b
09. candle_c
10. lifecycle_state
11. event_type
12. event_time
13. event_price

Dtypes:
fvg_id                             str
fvg_type                           str
created_at          datetime64[s, UTC]
lower_bound                    float64
upper_bound                    float64
gap_size                       float64
candle_a            datetime64[s, UTC]
candle_b            datetime64[s, UTC]
candle_c            datetime64[s, UTC]
lifecycle_state                    str
event_type                         str
event_time         datetime64[us, UTC]
event_price                    float64
dtype: object

Sample:


,fvg_id,fvg_type,created_at,lower_bound,upper_bound,gap_size,candle_a,candle_b,candle_c,lifecycle_state,event_type,event_time,event_price
0,FVG_B_20261002T221500Z,BULLISH_FVG,2026-10-02 22:15:00+00:00,1.12520,1.12526,0.00006,2026-10-02 22:05:00+00:00,2026-10-02 22:10:00+00:00,2026-10-02 22:15:00+00:00,FILLED,FILL,2026-10-02 23:15:00+00:00,1.12520
1,FVG_B_20261002T222500Z,BULLISH_FVG,2026-10-02 22:25:00+00:00,1.12554,1.12577,0.00023,2026-10-02 22:15:00+00:00,2026-10-02 22:20:00+00:00,2026-10-02 22:25:00+00:00,MITIGATED,MITIGATION,2026-10-02 22:40:00+00:00,1.12567
2,FVG_S_20261005T005500Z,BEARISH_FVG,2026-10-05 00:55:00+00:00,1.12538,1.12567,0.00029,2026-10-05 00:45:00+00:00,2026-10-05 00:50:00+00:00,2026-10-05 00:55:00+00:00,MITIGATED,MITIGATION,2026-10-05 01:00:00+00:00,1.12565
3,FVG_S_20261005T031500Z,BEARISH_FVG,2026-10-05 03:15:00+00:00,1.12498,1.12528,0.00030,2026-10-05 03:05:00+00:00,2026-10-05 03:10:00+00:00,2026-10-05 03:15:00+00:00,FILLED,FILL,2026-10-06 14:10:00+00:00,1.12528
4,FVG_S_20261005T034500Z,BEARISH_FVG,2026-10-05 03:45:00+00:00,1.12451,1.12459,0.00008,2026-10-05 03:35:00+00:00,2026-10-05 03:40:00+00:00,2026-10-05 03:45:00+00:00,FILLED,FILL,2026-10-06 11:40:00+00:00,1.12459
5,FVG_S_20261005T041000Z,BEARISH_FVG,2026-10-05 04:10:00+00:00,1.12392,1.12406,0.00014,2026-10-05 04:00:00+00:00,2026-10-05 04:05:00+00:00,2026-10-05 04:10:00+00:00,FILLED,FILL,2026-10-06 10:55:00+00:00,1.12406
6,FVG_S_20261005T051000Z,BEARISH_FVG,2026-10-05 05:10:00+00:00,1.12282,1.12293,0.00011,2026-10-05 05:00:00+00:00,2026-10-05 05:05:00+00:00,2026-10-05 05:10:00+00:00,FILLED,FILL,2026-10-06 04:00:00+00:00,1.12293
7,FVG_S_20261005T051500Z,BEARISH_FVG,2026-10-05 05:15:00+00:00,1.12249,1.12267,0.00018,2026-10-05 05:05:00+00:00,2026-10-05 05:10:00+00:00,2026-10-05 05:15:00+00:00,FILLED,FILL,2026-10-06 04:00:00+00:00,1.12267
8,FVG_S_20261005T052000Z,BEARISH_FVG,2026-10-05 05:20:00+00:00,1.12152,1.12230,0.00078,2026-10-05 05:10:00+00:00,2026-10-05 05:15:00+00:00,2026-10-05 05:20:00+00:00,MITIGATED,MITIGATION,2026-10-05 11:50:00+00:00,1.12170
9,FVG_S_20261005T055000Z,BEARISH_FVG,2026-10-05 05:50:00+00:00,1.11950,1.12028,0.00078,2026-10-05 05:40:00+00:00,2026-10-05 05:45:00+00:00,2026-10-05 05:50:00+00:00,MITIGATED,MITIGATION,2026-10-05 09:00:00+00:00,1.11959


In [127]:
# ================================================================================================
# FVG LIFECYCLE REGISTRY V1 — CONTRACT VARIABLE AUDIT
# ================================================================================================

print("=" * 100)
print("FVG LIFECYCLE REGISTRY V1 — CONTRACT VARIABLE AUDIT")
print("=" * 100)

_contract_vars = [
    name
    for name in globals()
    if name.startswith("FVG_REGISTRY_V1_")
]

print(f"Contract variables found: {len(_contract_vars)}")

for name in sorted(_contract_vars):
    value = globals()[name]
    print(f"\n{name}")
    print(f"  type : {type(value).__name__}")
    print(f"  value: {value}")

FVG LIFECYCLE REGISTRY V1 — CONTRACT VARIABLE AUDIT
Contract variables found: 0


In [128]:
# ================================================================================================
# FVG ENGINE V1 — LIFECYCLE / EVENT VOCABULARY AUDIT
# ================================================================================================

print("=" * 100)
print("FVG ENGINE V1 — LIFECYCLE / EVENT VOCABULARY AUDIT")
print("=" * 100)

print("\nLifecycle State:")
print(
    fvg_engine_v1["lifecycle_state"]
    .value_counts(dropna=False)
)

print("\nEvent Type:")
print(
    fvg_engine_v1["event_type"]
    .value_counts(dropna=False)
)

print("\nNull Audit:")
print(
    fvg_engine_v1[
        [
            "lifecycle_state",
            "event_type",
            "event_time",
            "event_price",
        ]
    ].isna().sum()
)

print("\nLifecycle × Event:")
print(
    pd.crosstab(
        fvg_engine_v1["lifecycle_state"],
        fvg_engine_v1["event_type"],
        dropna=False
    )
)

print("\nSample Lifecycle/Event Rows:")
display(
    fvg_engine_v1[
        [
            "fvg_id",
            "fvg_type",
            "created_at",
            "lifecycle_state",
            "event_type",
            "event_time",
            "event_price",
        ]
    ].head(20)
)

FVG ENGINE V1 — LIFECYCLE / EVENT VOCABULARY AUDIT

Lifecycle State:
lifecycle_state
MITIGATED    57
FILLED       37
FRESH         5
Name: count, dtype: int64

Event Type:
event_type
MITIGATION    57
FILL          37
NaN            5
Name: count, dtype: int64

Null Audit:
lifecycle_state    0
event_type         5
event_time         5
event_price        5
dtype: int64

Lifecycle × Event:
event_type       FILL  MITIGATION  NaN
lifecycle_state                       
FILLED             37           0    0
FRESH               0           0    5
MITIGATED           0          57    0

Sample Lifecycle/Event Rows:


,fvg_id,fvg_type,created_at,lifecycle_state,event_type,event_time,event_price
0,FVG_B_20261002T221500Z,BULLISH_FVG,2026-10-02 22:15:00+00:00,FILLED,FILL,2026-10-02 23:15:00+00:00,1.12520
1,FVG_B_20261002T222500Z,BULLISH_FVG,2026-10-02 22:25:00+00:00,MITIGATED,MITIGATION,2026-10-02 22:40:00+00:00,1.12567
2,FVG_S_20261005T005500Z,BEARISH_FVG,2026-10-05 00:55:00+00:00,MITIGATED,MITIGATION,2026-10-05 01:00:00+00:00,1.12565
3,FVG_S_20261005T031500Z,BEARISH_FVG,2026-10-05 03:15:00+00:00,FILLED,FILL,2026-10-06 14:10:00+00:00,1.12528
4,FVG_S_20261005T034500Z,BEARISH_FVG,2026-10-05 03:45:00+00:00,FILLED,FILL,2026-10-06 11:40:00+00:00,1.12459
5,FVG_S_20261005T041000Z,BEARISH_FVG,2026-10-05 04:10:00+00:00,FILLED,FILL,2026-10-06 10:55:00+00:00,1.12406
6,FVG_S_20261005T051000Z,BEARISH_FVG,2026-10-05 05:10:00+00:00,FILLED,FILL,2026-10-06 04:00:00+00:00,1.12293
7,FVG_S_20261005T051500Z,BEARISH_FVG,2026-10-05 05:15:00+00:00,FILLED,FILL,2026-10-06 04:00:00+00:00,1.12267
8,FVG_S_20261005T052000Z,BEARISH_FVG,2026-10-05 05:20:00+00:00,MITIGATED,MITIGATION,2026-10-05 11:50:00+00:00,1.12170
9,FVG_S_20261005T055000Z,BEARISH_FVG,2026-10-05 05:50:00+00:00,MITIGATED,MITIGATION,2026-10-05 09:00:00+00:00,1.11959


In [129]:
# ================================================================================================
# FVG LIFECYCLE REGISTRY V1
# ================================================================================================

import pandas as pd


# ------------------------------------------------------------------------------------------------
# CANONICAL VOCABULARY
# ------------------------------------------------------------------------------------------------

FVG_REGISTRY_V1_ALLOWED_TYPES = {
    "BULLISH_FVG",
    "BEARISH_FVG",
}

FVG_REGISTRY_V1_ALLOWED_STATES = {
    "FRESH",
    "MITIGATED",
    "FILLED",
}

FVG_REGISTRY_V1_ALLOWED_EVENTS = {
    "MITIGATION",
    "FILL",
}

FVG_REGISTRY_V1_IDENTITY_COLUMNS = [
    "fvg_id",
    "fvg_type",
    "created_at",
    "lower_bound",
    "upper_bound",
    "gap_size",
    "candle_a",
    "candle_b",
    "candle_c",
]

FVG_REGISTRY_V1_LIFECYCLE_COLUMNS = [
    "lifecycle_state",
    "event_type",
    "event_time",
    "event_price",
]


# ------------------------------------------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------------------------------------------

def validate_fvg_lifecycle_registry_input_v1(fvg_canonical_v1):
    """
    Validate the canonical FVG Engine V1 output before registry construction.

    Registry V1 is a direct canonical projection.
    It does not create, detect, classify, or re-identify FVGs.
    """

    if not isinstance(fvg_canonical_v1, pd.DataFrame):
        raise TypeError(
            "fvg_canonical_v1 must be a pandas DataFrame."
        )

    required_columns = (
        FVG_REGISTRY_V1_IDENTITY_COLUMNS
        + FVG_REGISTRY_V1_LIFECYCLE_COLUMNS
    )

    missing = [
        c
        for c in required_columns
        if c not in fvg_canonical_v1.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required FVG canonical columns: {missing}"
        )

    # --------------------------------------------------------------------------------------------
    # Identity
    # --------------------------------------------------------------------------------------------

    if fvg_canonical_v1["fvg_id"].isna().any():
        raise ValueError(
            "fvg_id contains null values."
        )

    if fvg_canonical_v1["fvg_id"].duplicated().any():
        raise ValueError(
            "Duplicate fvg_id detected."
        )

    invalid_types = (
        set(fvg_canonical_v1["fvg_type"].dropna().unique())
        - FVG_REGISTRY_V1_ALLOWED_TYPES
    )

    if invalid_types:
        raise ValueError(
            f"Invalid fvg_type values: {invalid_types}"
        )

    # --------------------------------------------------------------------------------------------
    # Lifecycle State
    # --------------------------------------------------------------------------------------------

    invalid_states = (
        set(
            fvg_canonical_v1["lifecycle_state"]
            .dropna()
            .unique()
        )
        - FVG_REGISTRY_V1_ALLOWED_STATES
    )

    if invalid_states:
        raise ValueError(
            f"Invalid lifecycle_state values: {invalid_states}"
        )

    # --------------------------------------------------------------------------------------------
    # Event Type
    # --------------------------------------------------------------------------------------------

    non_null_events = (
        fvg_canonical_v1["event_type"]
        .dropna()
        .unique()
    )

    invalid_events = (
        set(non_null_events)
        - FVG_REGISTRY_V1_ALLOWED_EVENTS
    )

    if invalid_events:
        raise ValueError(
            f"Invalid event_type values: {invalid_events}"
        )

    # --------------------------------------------------------------------------------------------
    # Lifecycle / Event Semantic Contract
    # --------------------------------------------------------------------------------------------

    fresh_mask = (
        fvg_canonical_v1["lifecycle_state"]
        == "FRESH"
    )

    non_fresh_mask = ~fresh_mask

    # FRESH must not have an event.
    fresh_with_event = (
        fresh_mask
        & fvg_canonical_v1["event_type"].notna()
    )

    if fresh_with_event.any():
        raise ValueError(
            "FRESH FVGs must have null event_type."
        )

    # FRESH must not have event time / price.
    fresh_with_event_time = (
        fresh_mask
        & fvg_canonical_v1["event_time"].notna()
    )

    fresh_with_event_price = (
        fresh_mask
        & fvg_canonical_v1["event_price"].notna()
    )

    if fresh_with_event_time.any():
        raise ValueError(
            "FRESH FVGs must have null event_time."
        )

    if fresh_with_event_price.any():
        raise ValueError(
            "FRESH FVGs must have null event_price."
        )

    # Non-FRESH states must have an event.
    non_fresh_missing_event = (
        non_fresh_mask
        & fvg_canonical_v1["event_type"].isna()
    )

    if non_fresh_missing_event.any():
        raise ValueError(
            "Non-FRESH FVGs must have event_type."
        )

    non_fresh_missing_time = (
        non_fresh_mask
        & fvg_canonical_v1["event_time"].isna()
    )

    non_fresh_missing_price = (
        non_fresh_mask
        & fvg_canonical_v1["event_price"].isna()
    )

    if non_fresh_missing_time.any():
        raise ValueError(
            "Non-FRESH FVGs must have event_time."
        )

    if non_fresh_missing_price.any():
        raise ValueError(
            "Non-FRESH FVGs must have event_price."
        )

    # MITIGATED -> MITIGATION
    mitigation_mismatch = (
        (
            fvg_canonical_v1["lifecycle_state"]
            == "MITIGATED"
        )
        &
        (
            fvg_canonical_v1["event_type"]
            != "MITIGATION"
        )
    )

    if mitigation_mismatch.any():
        raise ValueError(
            "MITIGATED FVGs must have event_type='MITIGATION'."
        )

    # FILLED -> FILL
    fill_mismatch = (
        (
            fvg_canonical_v1["lifecycle_state"]
            == "FILLED"
        )
        &
        (
            fvg_canonical_v1["event_type"]
            != "FILL"
        )
    )

    if fill_mismatch.any():
        raise ValueError(
            "FILLED FVGs must have event_type='FILL'."
        )

    return True


# ------------------------------------------------------------------------------------------------
# REGISTRY BUILDER
# ------------------------------------------------------------------------------------------------

def build_fvg_lifecycle_registry_v1(fvg_canonical_v1):
    """
    Build FVG Lifecycle Registry V1 directly from canonical FVG Engine V1.

    Identity and lifecycle/event information are preserved exactly.
    No new FVGs, events, lifecycle states, or prices are created.
    """

    validate_fvg_lifecycle_registry_input_v1(
        fvg_canonical_v1
    )

    registry_columns = (
        FVG_REGISTRY_V1_IDENTITY_COLUMNS
        + FVG_REGISTRY_V1_LIFECYCLE_COLUMNS
    )

    registry = (
        fvg_canonical_v1[
            registry_columns
        ]
        .copy(deep=True)
        .reset_index(drop=True)
    )

    if registry["fvg_id"].duplicated().any():
        raise ValueError(
            "FVG Registry V1 contains duplicate fvg_id values."
        )

    return registry


# ------------------------------------------------------------------------------------------------
# BUILD
# ------------------------------------------------------------------------------------------------

fvg_lifecycle_registry_v1 = build_fvg_lifecycle_registry_v1(
    fvg_engine_v1
)


# ------------------------------------------------------------------------------------------------
# BASIC REPORT
# ------------------------------------------------------------------------------------------------

print("=" * 100)
print("FVG LIFECYCLE REGISTRY V1")
print("=" * 100)

print(f"Rows        : {len(fvg_lifecycle_registry_v1)}")
print(f"Columns     : {len(fvg_lifecycle_registry_v1.columns)}")

print("\nLifecycle:")
print(
    fvg_lifecycle_registry_v1[
        "lifecycle_state"
    ].value_counts(dropna=False)
)

print("\nEvent:")
print(
    fvg_lifecycle_registry_v1[
        "event_type"
    ].value_counts(dropna=False)
)

print("\nFVG Type:")
print(
    fvg_lifecycle_registry_v1[
        "fvg_type"
    ].value_counts(dropna=False)
)

print("\nSchema:")
print(
    fvg_lifecycle_registry_v1.columns.tolist()
)

FVG LIFECYCLE REGISTRY V1
Rows        : 99
Columns     : 13

Lifecycle:
lifecycle_state
MITIGATED    57
FILLED       37
FRESH         5
Name: count, dtype: int64

Event:
event_type
MITIGATION    57
FILL          37
NaN            5
Name: count, dtype: int64

FVG Type:
fvg_type
BEARISH_FVG    56
BULLISH_FVG    43
Name: count, dtype: int64

Schema:
['fvg_id', 'fvg_type', 'created_at', 'lower_bound', 'upper_bound', 'gap_size', 'candle_a', 'candle_b', 'candle_c', 'lifecycle_state', 'event_type', 'event_time', 'event_price']


In [130]:
# ================================================================================================
# FVG FRESH CONTRACT — DIAGNOSTIC AUDIT V2
# ================================================================================================

fresh_audit = fvg_lifecycle_registry_v1[
    fvg_lifecycle_registry_v1["lifecycle_state"] == "FRESH"
].copy()

# --------------------------------------------------------------------------------
# FRESH CONTRACT
# --------------------------------------------------------------------------------

fresh_audit["event_type_valid"] = (
    fresh_audit["event_type"].isna()
)

fresh_audit["event_time_valid"] = (
    fresh_audit["event_time"].isna()
)

fresh_audit["event_price_valid"] = (
    fresh_audit["event_price"].isna()
)


print("=" * 100)
print("FVG FRESH CONTRACT — DIAGNOSTIC AUDIT V2")
print("=" * 100)

audit_columns = [
    "fvg_id",
    "fvg_type",
    "lifecycle_state",
    "event_type",
    "event_time",
    "event_price",
]

print("\nFRESH RECORDS:")
print(
    fresh_audit[audit_columns]
    .to_string(index=False)
)

print("\n" + "-" * 100)
print("CONTRACT CHECKS")
print("-" * 100)

contract_columns = [
    "event_type_valid",
    "event_time_valid",
    "event_price_valid",
]

for column in contract_columns:
    print(
        f"{column:<35}: "
        f"{'PASSED' if fresh_audit[column].all() else 'FAILED'}"
    )

print("\n" + "-" * 100)
print("VIOLATING RECORDS")
print("-" * 100)

violations = fresh_audit[
    ~fresh_audit[contract_columns].all(axis=1)
]

if violations.empty:
    print("None")
else:
    print(
        violations[
            audit_columns + contract_columns
        ].to_string(index=False)
    )

print("\n" + "-" * 100)
print("FRESH COUNT")
print("-" * 100)

print(f"FRESH records : {len(fresh_audit)}")

FVG FRESH CONTRACT — DIAGNOSTIC AUDIT V2

FRESH RECORDS:
                fvg_id    fvg_type lifecycle_state event_type event_time  event_price
FVG_S_20261007T035500Z BEARISH_FVG           FRESH        NaN        NaT          NaN
FVG_S_20261007T045000Z BEARISH_FVG           FRESH        NaN        NaT          NaN
FVG_S_20261007T071500Z BEARISH_FVG           FRESH        NaN        NaT          NaN
FVG_S_20261007T102000Z BEARISH_FVG           FRESH        NaN        NaT          NaN
FVG_B_20261007T160000Z BULLISH_FVG           FRESH        NaN        NaT          NaN

----------------------------------------------------------------------------------------------------
CONTRACT CHECKS
----------------------------------------------------------------------------------------------------
event_type_valid                   : PASSED
event_time_valid                   : PASSED
event_price_valid                  : PASSED

---------------------------------------------------------------------------

In [131]:
# ================================================================================================
# FVG LIFECYCLE REGISTRY V1 — REGRESSION GATE
# ================================================================================================

print("=" * 100)
print("FVG LIFECYCLE REGISTRY V1 — REGRESSION GATE")
print("=" * 100)

engine = fvg_engine_v1.copy(deep=True)
registry = fvg_lifecycle_registry_v1.copy(deep=True)

# --------------------------------------------------------------------------------
# EXPECTED SCHEMA
# --------------------------------------------------------------------------------

expected_columns = (
    FVG_REGISTRY_V1_IDENTITY_COLUMNS
    + FVG_REGISTRY_V1_LIFECYCLE_COLUMNS
)

# --------------------------------------------------------------------------------
# GATE 1 — ROW COUNT
# --------------------------------------------------------------------------------

row_count_pass = (
    len(registry) == len(engine)
)

print(
    f"ROW COUNT PRESERVATION"
    f"{' ' * 30}: "
    f"{'PASSED' if row_count_pass else 'FAILED'}"
)

# --------------------------------------------------------------------------------
# GATE 2 — SCHEMA
# --------------------------------------------------------------------------------

schema_pass = (
    registry.columns.tolist() == expected_columns
)

print(
    f"SCHEMA"
    f"{' ' * 40}: "
    f"{'PASSED' if schema_pass else 'FAILED'}"
)

# --------------------------------------------------------------------------------
# GATE 3 — FVG ID UNIQUENESS
# --------------------------------------------------------------------------------

registry_id_unique_pass = (
    not registry["fvg_id"].duplicated().any()
)

print(
    f"FVG ID UNIQUENESS"
    f"{' ' * 30}: "
    f"{'PASSED' if registry_id_unique_pass else 'FAILED'}"
)

# --------------------------------------------------------------------------------
# GATE 4 — FVG ID PRESERVATION
# --------------------------------------------------------------------------------

engine_ids = set(engine["fvg_id"])
registry_ids = set(registry["fvg_id"])

id_preservation_pass = (
    engine_ids == registry_ids
)

print(
    f"FVG ID PRESERVATION"
    f"{' ' * 28}: "
    f"{'PASSED' if id_preservation_pass else 'FAILED'}"
)

# --------------------------------------------------------------------------------
# GATE 5 — IDENTITY PRESERVATION
# --------------------------------------------------------------------------------

identity_pass = True

for column in FVG_REGISTRY_V1_IDENTITY_COLUMNS:
    left = engine[column].reset_index(drop=True)
    right = registry[column].reset_index(drop=True)

    if pd.api.types.is_datetime64_any_dtype(left):
        comparison = (
            left.astype("datetime64[ns, UTC]")
            .equals(
                right.astype("datetime64[ns, UTC]")
            )
        )
    else:
        comparison = left.equals(right)

    if not comparison:
        identity_pass = False
        break

print(
    f"IDENTITY PRESERVATION"
    f"{' ' * 27}: "
    f"{'PASSED' if identity_pass else 'FAILED'}"
)

# --------------------------------------------------------------------------------
# GATE 6 — LIFECYCLE STATE PRESERVATION
# --------------------------------------------------------------------------------

lifecycle_pass = (
    engine["lifecycle_state"]
    .reset_index(drop=True)
    .equals(
        registry["lifecycle_state"]
        .reset_index(drop=True)
    )
)

print(
    f"LIFECYCLE PRESERVATION"
    f"{' ' * 26}: "
    f"{'PASSED' if lifecycle_pass else 'FAILED'}"
)

# --------------------------------------------------------------------------------
# GATE 7 — EVENT TYPE PRESERVATION
# --------------------------------------------------------------------------------

event_type_pass = (
    engine["event_type"]
    .reset_index(drop=True)
    .equals(
        registry["event_type"]
        .reset_index(drop=True)
    )
)

print(
    f"EVENT TYPE PRESERVATION"
    f"{' ' * 26}: "
    f"{'PASSED' if event_type_pass else 'FAILED'}"
)

# --------------------------------------------------------------------------------
# GATE 8 — EVENT TIME PRESERVATION
# --------------------------------------------------------------------------------

event_time_pass = (
    engine["event_time"]
    .reset_index(drop=True)
    .equals(
        registry["event_time"]
        .reset_index(drop=True)
    )
)

print(
    f"EVENT TIME PRESERVATION"
    f"{' ' * 25}: "
    f"{'PASSED' if event_time_pass else 'FAILED'}"
)

# --------------------------------------------------------------------------------
# GATE 9 — EVENT PRICE PRESERVATION
# --------------------------------------------------------------------------------

event_price_pass = (
    engine["event_price"]
    .reset_index(drop=True)
    .equals(
        registry["event_price"]
        .reset_index(drop=True)
    )
)

print(
    f"EVENT PRICE PRESERVATION"
    f"{' ' * 24}: "
    f"{'PASSED' if event_price_pass else 'FAILED'}"
)

# --------------------------------------------------------------------------------
# GATE 10 — FRESH CONTRACT
# --------------------------------------------------------------------------------

fresh = registry["lifecycle_state"] == "FRESH"

fresh_contract_pass = (
    registry.loc[fresh, "event_type"].isna().all()
    and
    registry.loc[fresh, "event_time"].isna().all()
    and
    registry.loc[fresh, "event_price"].isna().all()
)

print(
    f"FRESH CONTRACT"
    f"{' ' * 34}: "
    f"{'PASSED' if fresh_contract_pass else 'FAILED'}"
)

# --------------------------------------------------------------------------------
# GATE 11 — MITIGATED CONTRACT
# --------------------------------------------------------------------------------

mitigated = registry["lifecycle_state"] == "MITIGATED"

mitigated_contract_pass = (
    registry.loc[mitigated, "event_type"]
    .eq("MITIGATION")
    .all()
    and
    registry.loc[mitigated, "event_time"]
    .notna()
    .all()
    and
    registry.loc[mitigated, "event_price"]
    .notna()
    .all()
)

print(
    f"MITIGATED CONTRACT"
    f"{' ' * 29}: "
    f"{'PASSED' if mitigated_contract_pass else 'FAILED'}"
)

# --------------------------------------------------------------------------------
# GATE 12 — FILLED CONTRACT
# --------------------------------------------------------------------------------

filled = registry["lifecycle_state"] == "FILLED"

filled_contract_pass = (
    registry.loc[filled, "event_type"]
    .eq("FILL")
    .all()
    and
    registry.loc[filled, "event_time"]
    .notna()
    .all()
    and
    registry.loc[filled, "event_price"]
    .notna()
    .all()
)

print(
    f"FILLED CONTRACT"
    f"{' ' * 32}: "
    f"{'PASSED' if filled_contract_pass else 'FAILED'}"
)

# --------------------------------------------------------------------------------
# GATE 13 — TYPE VOCABULARY
# --------------------------------------------------------------------------------

invalid_types = (
    set(registry["fvg_type"].dropna().unique())
    - FVG_REGISTRY_V1_ALLOWED_TYPES
)

type_vocab_pass = (
    len(invalid_types) == 0
)

print(
    f"FVG TYPE VOCABULARY"
    f"{' ' * 28}: "
    f"{'PASSED' if type_vocab_pass else 'FAILED'}"
)

# --------------------------------------------------------------------------------
# GATE 14 — STATE VOCABULARY
# --------------------------------------------------------------------------------

invalid_states = (
    set(registry["lifecycle_state"].dropna().unique())
    - FVG_REGISTRY_V1_ALLOWED_STATES
)

state_vocab_pass = (
    len(invalid_states) == 0
)

print(
    f"LIFECYCLE STATE VOCABULARY"
    f"{' ' * 22}: "
    f"{'PASSED' if state_vocab_pass else 'FAILED'}"
)

# --------------------------------------------------------------------------------
# GATE 15 — EVENT VOCABULARY
# --------------------------------------------------------------------------------

invalid_events = (
    set(registry["event_type"].dropna().unique())
    - FVG_REGISTRY_V1_ALLOWED_EVENTS
)

event_vocab_pass = (
    len(invalid_events) == 0
)

print(
    f"EVENT VOCABULARY"
    f"{' ' * 32}: "
    f"{'PASSED' if event_vocab_pass else 'FAILED'}"
)

# --------------------------------------------------------------------------------
# FINAL RESULT
# --------------------------------------------------------------------------------

all_gates = [
    row_count_pass,
    schema_pass,
    registry_id_unique_pass,
    id_preservation_pass,
    identity_pass,
    lifecycle_pass,
    event_type_pass,
    event_time_pass,
    event_price_pass,
    fresh_contract_pass,
    mitigated_contract_pass,
    filled_contract_pass,
    type_vocab_pass,
    state_vocab_pass,
    event_vocab_pass,
]

print("\n" + "=" * 100)

if all(all_gates):
    print("FVG LIFECYCLE REGISTRY V1 REGRESSION GATE : PASSED")
    print(f"ALL {len(all_gates)}/{len(all_gates)} GATES PASSED")
else:
    print("FVG LIFECYCLE REGISTRY V1 REGRESSION GATE : FAILED")
    print(
        f"{sum(all_gates)}/{len(all_gates)} GATES PASSED"
    )

print("=" * 100)

FVG LIFECYCLE REGISTRY V1 — REGRESSION GATE
ROW COUNT PRESERVATION                              : PASSED
SCHEMA                                        : PASSED
FVG ID UNIQUENESS                              : PASSED
FVG ID PRESERVATION                            : PASSED
IDENTITY PRESERVATION                           : PASSED
LIFECYCLE PRESERVATION                          : PASSED
EVENT TYPE PRESERVATION                          : PASSED
EVENT TIME PRESERVATION                         : PASSED
EVENT PRICE PRESERVATION                        : PASSED
FRESH CONTRACT                                  : PASSED
MITIGATED CONTRACT                             : PASSED
FILLED CONTRACT                                : PASSED
FVG TYPE VOCABULARY                            : PASSED
LIFECYCLE STATE VOCABULARY                      : PASSED
EVENT VOCABULARY                                : PASSED

FVG LIFECYCLE REGISTRY V1 REGRESSION GATE : PASSED
ALL 15/15 GATES PASSED


In [132]:
# ================================================================================================
# FVG INTEGRATION CONTRACT V1
# ================================================================================================

FVG_INTEGRATION_V1_REQUIRED_COLUMNS = [
    "fvg_id",
    "fvg_type",
    "created_at",
    "lower_bound",
    "upper_bound",
    "gap_size",
    "candle_a",
    "candle_b",
    "candle_c",
    "lifecycle_state",
    "mitigation_time",
    "mitigation_price",
    "fill_time",
    "fill_price",
    "event_time",
    "event_price",
    "event_direction",
    "event_type",
]

FVG_INTEGRATION_V1_ALLOWED_TYPES = {
    "BULLISH_FVG",
    "BEARISH_FVG",
}

FVG_INTEGRATION_V1_ALLOWED_STATES = {
    "FRESH",
    "MITIGATED",
    "FILLED",
}

FVG_INTEGRATION_V1_ALLOWED_EVENTS = {
    "MITIGATION",
    "FILL",
}

FVG_INTEGRATION_V1_ALLOWED_DIRECTIONS = {
    "BULLISH",
    "BEARISH",
}

In [133]:
def validate_fvg_integration_contract_v1(
    fvg_registry_v1
):
    """
    Validate FVG Lifecycle Registry V1 as an integration-ready
    canonical FVG source.

    This validator does NOT modify the registry.
    """

    violations = []

    # --------------------------------------------------------------------------------------------
    # 1. OBJECT TYPE
    # --------------------------------------------------------------------------------------------

    if not isinstance(fvg_registry_v1, pd.DataFrame):
        violations.append("INVALID_OBJECT_TYPE")
        return {
            "passed": False,
            "violations": violations,
        }

    # --------------------------------------------------------------------------------------------
    # 2. REQUIRED SCHEMA
    # --------------------------------------------------------------------------------------------

    missing_columns = [
        column
        for column in FVG_INTEGRATION_V1_REQUIRED_COLUMNS
        if column not in fvg_registry_v1.columns
    ]

    if missing_columns:
        violations.append(
            f"MISSING_COLUMNS:{missing_columns}"
        )

    # Stop here if schema is incomplete.
    if missing_columns:
        return {
            "passed": False,
            "violations": violations,
        }

    # --------------------------------------------------------------------------------------------
    # 3. FVG ID
    # --------------------------------------------------------------------------------------------

    if fvg_registry_v1["fvg_id"].isna().any():
        violations.append("NULL_FVG_ID")

    if fvg_registry_v1["fvg_id"].duplicated().any():
        violations.append("DUPLICATE_FVG_ID")

    # --------------------------------------------------------------------------------------------
    # 4. FVG TYPE
    # --------------------------------------------------------------------------------------------

    invalid_types = set(
        fvg_registry_v1["fvg_type"].dropna().unique()
    ) - FVG_INTEGRATION_V1_ALLOWED_TYPES

    if invalid_types:
        violations.append(
            f"INVALID_FVG_TYPE:{invalid_types}"
        )

    # --------------------------------------------------------------------------------------------
    # 5. LIFECYCLE STATE
    # --------------------------------------------------------------------------------------------

    invalid_states = set(
        fvg_registry_v1["lifecycle_state"].dropna().unique()
    ) - FVG_INTEGRATION_V1_ALLOWED_STATES

    if invalid_states:
        violations.append(
            f"INVALID_LIFECYCLE_STATE:{invalid_states}"
        )

    # --------------------------------------------------------------------------------------------
    # 6. EVENT TYPE
    # --------------------------------------------------------------------------------------------

    invalid_events = set(
        fvg_registry_v1["event_type"].dropna().unique()
    ) - FVG_INTEGRATION_V1_ALLOWED_EVENTS

    if invalid_events:
        violations.append(
            f"INVALID_EVENT_TYPE:{invalid_events}"
        )

    # --------------------------------------------------------------------------------------------
    # 7. EVENT DIRECTION
    # --------------------------------------------------------------------------------------------

    invalid_directions = set(
        fvg_registry_v1["event_direction"].dropna().unique()
    ) - FVG_INTEGRATION_V1_ALLOWED_DIRECTIONS

    if invalid_directions:
        violations.append(
            f"INVALID_EVENT_DIRECTION:{invalid_directions}"
        )

    # --------------------------------------------------------------------------------------------
    # 8. DIRECTION ↔ FVG TYPE
    # --------------------------------------------------------------------------------------------

    expected_direction = (
        fvg_registry_v1["fvg_type"]
        .map({
            "BULLISH_FVG": "BULLISH",
            "BEARISH_FVG": "BEARISH",
        })
    )

    direction_match = (
        fvg_registry_v1["event_direction"]
        == expected_direction
    )

    if not direction_match.all():
        violations.append(
            "FVG_TYPE_DIRECTION_MISMATCH"
        )

    # --------------------------------------------------------------------------------------------
    # 9. BOUNDARY CONTRACT
    # --------------------------------------------------------------------------------------------

    if (
        fvg_registry_v1["lower_bound"].isna().any()
        or fvg_registry_v1["upper_bound"].isna().any()
    ):
        violations.append(
            "NULL_FVG_BOUNDARY"
        )

    if (
        fvg_registry_v1["lower_bound"]
        > fvg_registry_v1["upper_bound"]
    ).any():
        violations.append(
            "INVALID_FVG_BOUNDARY_ORDER"
        )

    # --------------------------------------------------------------------------------------------
    # 10. GAP SIZE CONTRACT
    # --------------------------------------------------------------------------------------------

    if fvg_registry_v1["gap_size"].isna().any():
        violations.append(
            "NULL_GAP_SIZE"
        )

    if (
        fvg_registry_v1["gap_size"] <= 0
    ).any():
        violations.append(
            "NON_POSITIVE_GAP_SIZE"
        )

    # --------------------------------------------------------------------------------------------
    # 11. FRESH CONTRACT
    # --------------------------------------------------------------------------------------------

    fresh = fvg_registry_v1[
        fvg_registry_v1["lifecycle_state"] == "FRESH"
    ]

    fresh_valid = (
        fresh["event_type"].isna().all()
        and fresh["event_time"].isna().all()
        and fresh["event_price"].isna().all()
        and fresh["mitigation_time"].isna().all()
        and fresh["mitigation_price"].isna().all()
        and fresh["fill_time"].isna().all()
        and fresh["fill_price"].isna().all()
    )

    if not fresh_valid:
        violations.append(
            "FRESH_LIFECYCLE_CONTRACT"
        )

    # --------------------------------------------------------------------------------------------
    # 12. MITIGATED CONTRACT
    # --------------------------------------------------------------------------------------------

    mitigated = fvg_registry_v1[
        fvg_registry_v1["lifecycle_state"] == "MITIGATED"
    ]

    mitigated_valid = (
        mitigated["event_type"].eq("MITIGATION").all()
        and mitigated["event_time"].notna().all()
        and mitigated["event_price"].notna().all()
        and mitigated["mitigation_time"].notna().all()
        and mitigated["mitigation_price"].notna().all()
    )

    if not mitigated_valid:
        violations.append(
            "MITIGATED_LIFECYCLE_CONTRACT"
        )

    # --------------------------------------------------------------------------------------------
    # 13. FILLED CONTRACT
    # --------------------------------------------------------------------------------------------

    filled = fvg_registry_v1[
        fvg_registry_v1["lifecycle_state"] == "FILLED"
    ]

    filled_valid = (
        filled["event_type"].eq("FILL").all()
        and filled["event_time"].notna().all()
        and filled["event_price"].notna().all()
        and filled["fill_time"].notna().all()
        and filled["fill_price"].notna().all()
    )

    if not filled_valid:
        violations.append(
            "FILLED_LIFECYCLE_CONTRACT"
        )

    # --------------------------------------------------------------------------------------------
    # 14. FVG MUST REMAIN A DISTINCT OBJECT FAMILY
    # --------------------------------------------------------------------------------------------

    # Integration Contract explicitly rejects Liquidity identity columns.
    forbidden_liquidity_identity_columns = {
        "level_id",
        "liquidity_id",
        "liquidity_type",
        "liquidity_side",
    }

    unexpected_liquidity_columns = (
        forbidden_liquidity_identity_columns
        & set(fvg_registry_v1.columns)
    )

    if unexpected_liquidity_columns:
        violations.append(
            f"FVG_LIQUIDITY_IDENTITY_COLLISION:"
            f"{unexpected_liquidity_columns}"
        )

    # --------------------------------------------------------------------------------------------
    # FINAL
    # --------------------------------------------------------------------------------------------

    return {
        "passed": len(violations) == 0,
        "violations": violations,
    }

In [134]:
# ================================================================================================
# RUN FVG INTEGRATION CONTRACT V1
# ================================================================================================

fvg_integration_contract_v1 = (
    validate_fvg_integration_contract_v1(
        fvg_lifecycle_registry_v1
    )
)

print("=" * 100)
print("FVG INTEGRATION CONTRACT V1")
print("=" * 100)

if fvg_integration_contract_v1["passed"]:
    print("STATUS                              : PASSED")
    print("FVG REGISTRY                        : INTEGRATION READY")
else:
    print("STATUS                              : FAILED")

    print("\nViolations:")
    for violation in fvg_integration_contract_v1["violations"]:
        print(f" - {violation}")

FVG INTEGRATION CONTRACT V1
STATUS                              : FAILED

Violations:
 - MISSING_COLUMNS:['mitigation_time', 'mitigation_price', 'fill_time', 'fill_price', 'event_direction']


In [135]:
# ============================================================
# FVG ↔ STRUCTURE / LIQUIDITY CONTEXT ENGINE V1
# STEP 1 — READ-ONLY DISCOVERY — SAFE VERSION
# ============================================================

import pandas as pd

print("=" * 100)
print("FVG ↔ STRUCTURE / LIQUIDITY CONTEXT ENGINE V1 — DISCOVERY")
print("=" * 100)


# ============================================================
# SAFE GLOBAL SNAPSHOT
# ============================================================

_global_items = list(globals().items())


# ============================================================
# 1. FVG DATAFRAMES
# ============================================================

fvg_candidates = []

for name, obj in _global_items:

    if isinstance(obj, pd.DataFrame):

        cols = set(obj.columns)

        if "fvg_id" in cols:
            fvg_candidates.append(
                {
                    "name": name,
                    "rows": len(obj),
                    "columns": list(obj.columns),
                }
            )


print("\n[FVG DATAFRAMES]")

if not fvg_candidates:
    print("NONE FOUND")

else:
    for item in fvg_candidates:
        print(f"\n{item['name']}")
        print(f"  rows    : {item['rows']}")
        print(f"  columns : {item['columns']}")


# ============================================================
# 2. LIQUIDITY DATAFRAMES
# ============================================================

liquidity_candidates = []

liquidity_markers = {
    "level_id",
    "liquidity_id",
    "liquidity_type",
    "liquidity_side",
    "level_price",
    "level_type",
    "source_type",
    "source_id",
}


for name, obj in _global_items:

    if isinstance(obj, pd.DataFrame):

        cols = set(obj.columns)

        matched = cols.intersection(liquidity_markers)

        if len(matched) >= 2:

            liquidity_candidates.append(
                {
                    "name": name,
                    "rows": len(obj),
                    "matched": sorted(matched),
                    "columns": list(obj.columns),
                }
            )


print("\n" + "=" * 100)
print("[LIQUIDITY DATAFRAMES]")
print("=" * 100)

if not liquidity_candidates:
    print("NONE FOUND")

else:
    for item in liquidity_candidates:
        print(f"\n{item['name']}")
        print(f"  rows    : {item['rows']}")
        print(f"  matched : {item['matched']}")
        print(f"  columns : {item['columns']}")


# ============================================================
# 3. STRUCTURE / BOS / CHoCH DATAFRAMES
# ============================================================

structure_candidates = []

structure_markers = {
    "bos_direction",
    "bos_type",
    "choch_direction",
    "choch_type",
    "structure_bias",
    "active_bias",
    "structure_type",
    "event_type",
    "event_direction",
    "level_id",
    "choch_level_id",
}


for name, obj in _global_items:

    if isinstance(obj, pd.DataFrame):

        cols = set(obj.columns)

        matched = cols.intersection(structure_markers)

        if len(matched) >= 2:

            structure_candidates.append(
                {
                    "name": name,
                    "rows": len(obj),
                    "matched": sorted(matched),
                    "columns": list(obj.columns),
                }
            )


print("\n" + "=" * 100)
print("[STRUCTURE / BOS / CHoCH DATAFRAMES]")
print("=" * 100)

if not structure_candidates:
    print("NONE FOUND")

else:
    for item in structure_candidates:
        print(f"\n{item['name']}")
        print(f"  rows    : {item['rows']}")
        print(f"  matched : {item['matched']}")
        print(f"  columns : {item['columns']}")


# ============================================================
# 4. STRUCTURE STATE DATAFRAMES
# ============================================================

state_candidates = []

state_markers = {
    "active_bias",
    "last_bos_id",
    "last_bos_level",
    "last_bos_direction",
    "last_choch_id",
    "last_choch_level",
    "last_choch_direction",
}


for name, obj in _global_items:

    if isinstance(obj, pd.DataFrame):

        cols = set(obj.columns)

        matched = cols.intersection(state_markers)

        if len(matched) >= 2:

            state_candidates.append(
                {
                    "name": name,
                    "rows": len(obj),
                    "matched": sorted(matched),
                    "columns": list(obj.columns),
                }
            )


print("\n" + "=" * 100)
print("[STRUCTURE STATE DATAFRAMES]")
print("=" * 100)

if not state_candidates:
    print("NONE FOUND")

else:
    for item in state_candidates:
        print(f"\n{item['name']}")
        print(f"  rows    : {item['rows']}")
        print(f"  matched : {item['matched']}")
        print(f"  columns : {item['columns']}")


# ============================================================
# 5. SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("DISCOVERY SUMMARY")
print("=" * 100)

print(f"FVG candidates              : {len(fvg_candidates)}")
print(f"Liquidity candidates        : {len(liquidity_candidates)}")
print(f"Structure candidates        : {len(structure_candidates)}")
print(f"Structure-state candidates  : {len(state_candidates)}")

print("\n" + "=" * 100)
print("DISCOVERY COMPLETE — NO DATAFRAME WAS MODIFIED")
print("=" * 100)

FVG ↔ STRUCTURE / LIQUIDITY CONTEXT ENGINE V1 — DISCOVERY

[FVG DATAFRAMES]

violations
  rows    : 0
  columns : ['fvg_id', 'fvg_type', 'created_at', 'lower_bound', 'upper_bound', 'gap_size', 'candle_a', 'candle_b', 'candle_c', 'lifecycle_state', 'event_type', 'event_time', 'event_price', 'event_type_valid', 'event_time_valid', 'event_price_valid']

fvg_v1
  rows    : 179
  columns : ['fvg_id', 'fvg_type', 'created_at', 'lower_bound', 'upper_bound', 'gap_size', 'candle_a', 'candle_b', 'candle_c']

fvg_quality_audit_v1
  rows    : 179
  columns : ['fvg_id', 'fvg_type', 'created_at', 'gap_size', 'atr', 'gap_atr_ratio', 'candle_b_range', 'range_atr_ratio', 'candle_b_body', 'body_atr_ratio', 'directional_candle_b']

fvg_validated_v1
  rows    : 179
  columns : ['fvg_id', 'fvg_type', 'created_at', 'lower_bound', 'upper_bound', 'gap_size', 'candle_a', 'candle_b', 'candle_c', 'atr', 'gap_atr_ratio', 'body_atr_ratio', 'range_atr_ratio', 'directional_candle_b', 'validation_status', 'validation

In [136]:
# ============================================================
# FVG ↔ STRUCTURE / LIQUIDITY CONTEXT ENGINE V1
# STEP 2 — CONTEXT SNAPSHOT
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# CONTRACT
# ============================================================

FVG_CONTEXT_ENGINE_V1_VERSION = "1.0"

FVG_CONTEXT_ENGINE_V1_REQUIRED_FVG_COLUMNS = [
    "fvg_id",
    "fvg_type",
    "created_at",
    "lower_bound",
    "upper_bound",
    "gap_size",
    "lifecycle_state",
]

FVG_CONTEXT_ENGINE_V1_REQUIRED_LIQUIDITY_COLUMNS = [
    "level_id",
    "liquidity_source",
    "liquidity_type",
    "level_price",
    "activation_time",
    "state",
]

FVG_CONTEXT_ENGINE_V1_REQUIRED_BOS_COLUMNS = [
    "bos_id",
    "level_id",
    "bos_direction",
    "bos_level",
    "bos_time",
]

FVG_CONTEXT_ENGINE_V1_REQUIRED_CHOCH_COLUMNS = [
    "choch_id",
    "bos_id",
    "level_id",
    "choch_direction",
    "choch_level",
    "choch_time",
    "structure_bias_before",
    "structure_bias_after",
]

FVG_CONTEXT_ENGINE_V1_REQUIRED_STATE_COLUMNS = [
    "active_bias",
    "last_bos",
    "last_bos_direction",
    "last_bos_level",
    "last_bos_level_id",
    "last_choch",
    "last_choch_direction",
    "last_choch_level",
    "last_choch_level_id",
]


# ============================================================
# VALIDATION
# ============================================================

def _require_columns(df, required, name):
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(
            f"{name} missing required columns: {missing}"
        )


def _prepare_datetime(df, column):
    out = df.copy()
    out[column] = pd.to_datetime(
        out[column],
        utc=True,
        errors="coerce"
    )
    return out


# ============================================================
# MAIN ENGINE
# ============================================================

def build_fvg_context_engine_v1(
    fvg_registry,
    liquidity_registry,
    structural_bos,
    structural_choch,
    structure_state,
):
    """
    Build point-in-time structural/liquidity context for each FVG.

    IMPORTANT:
    - Does NOT modify FVG identity.
    - Does NOT modify FVG lifecycle.
    - Does NOT assign liquidity IDs to FVGs.
    - Does NOT merge FVGs into the liquidity registry.
    - Context is evaluated at FVG created_at.
    """

    # --------------------------------------------------------
    # Validate inputs
    # --------------------------------------------------------

    _require_columns(
        fvg_registry,
        FVG_CONTEXT_ENGINE_V1_REQUIRED_FVG_COLUMNS,
        "fvg_registry"
    )

    _require_columns(
        liquidity_registry,
        FVG_CONTEXT_ENGINE_V1_REQUIRED_LIQUIDITY_COLUMNS,
        "liquidity_registry"
    )

    _require_columns(
        structural_bos,
        FVG_CONTEXT_ENGINE_V1_REQUIRED_BOS_COLUMNS,
        "structural_bos"
    )

    _require_columns(
        structural_choch,
        FVG_CONTEXT_ENGINE_V1_REQUIRED_CHOCH_COLUMNS,
        "structural_choch"
    )

    _require_columns(
        structure_state,
        FVG_CONTEXT_ENGINE_V1_REQUIRED_STATE_COLUMNS,
        "structure_state"
    )

    # --------------------------------------------------------
    # Defensive copies
    # --------------------------------------------------------

    fvg = fvg_registry.copy(deep=True)
    liq = liquidity_registry.copy(deep=True)
    bos = structural_bos.copy(deep=True)
    choch = structural_choch.copy(deep=True)
    state = structure_state.copy(deep=True)

    # --------------------------------------------------------
    # UTC normalization
    # --------------------------------------------------------

    fvg = _prepare_datetime(fvg, "created_at")

    liq = _prepare_datetime(liq, "activation_time")

    bos = _prepare_datetime(bos, "bos_time")

    choch = _prepare_datetime(choch, "choch_time")

    # Structure state index is assumed to be the M5 UTC timeline.
    state_index = pd.to_datetime(
        state.index,
        utc=True,
        errors="coerce"
    )

    # --------------------------------------------------------
    # Sort
    # --------------------------------------------------------

    fvg = fvg.sort_values(
        "created_at",
        kind="stable"
    ).reset_index(drop=True)

    liq = liq.sort_values(
        "activation_time",
        kind="stable"
    ).reset_index(drop=True)

    bos = bos.sort_values(
        "bos_time",
        kind="stable"
    ).reset_index(drop=True)

    choch = choch.sort_values(
        "choch_time",
        kind="stable"
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # Output rows
    # --------------------------------------------------------

    results = []

    for _, f in fvg.iterrows():

        fvg_time = f["created_at"]
        lower = float(f["lower_bound"])
        upper = float(f["upper_bound"])

        # ====================================================
        # STRUCTURE STATE
        # ====================================================

        state_mask = state_index <= fvg_time

        if state_mask.any():

            state_positions = np.flatnonzero(state_mask)

            latest_pos = state_positions[-1]

            s = state.iloc[latest_pos]

            active_bias = s["active_bias"]
            last_bos = s["last_bos"]
            last_bos_direction = s["last_bos_direction"]
            last_bos_level = s["last_bos_level"]
            last_bos_level_id = s["last_bos_level_id"]

            last_choch = s["last_choch"]
            last_choch_direction = s["last_choch_direction"]
            last_choch_level = s["last_choch_level"]
            last_choch_level_id = s["last_choch_level_id"]

        else:

            active_bias = np.nan
            last_bos = np.nan
            last_bos_direction = np.nan
            last_bos_level = np.nan
            last_bos_level_id = np.nan

            last_choch = np.nan
            last_choch_direction = np.nan
            last_choch_level = np.nan
            last_choch_level_id = np.nan

        # ====================================================
        # LAST BOS BEFORE FVG
        # ====================================================

        bos_before = bos[
            bos["bos_time"] <= fvg_time
        ]

        if len(bos_before):

            b = bos_before.iloc[-1]

            context_bos_id = b["bos_id"]
            context_bos_direction = b["bos_direction"]
            context_bos_level = b["bos_level"]
            context_bos_time = b["bos_time"]
            context_bos_level_id = b["level_id"]

        else:

            context_bos_id = np.nan
            context_bos_direction = np.nan
            context_bos_level = np.nan
            context_bos_time = pd.NaT
            context_bos_level_id = np.nan

        # ====================================================
        # LAST CHoCH BEFORE FVG
        # ====================================================

        choch_before = choch[
            choch["choch_time"] <= fvg_time
        ]

        if len(choch_before):

            c = choch_before.iloc[-1]

            context_choch_id = c["choch_id"]
            context_choch_direction = c["choch_direction"]
            context_choch_level = c["choch_level"]
            context_choch_time = c["choch_time"]
            context_choch_level_id = c["level_id"]

        else:

            context_choch_id = np.nan
            context_choch_direction = np.nan
            context_choch_level = np.nan
            context_choch_time = pd.NaT
            context_choch_level_id = np.nan

        # ====================================================
        # LIQUIDITY CONTEXT
        #
        # Active liquidity existing at FVG creation time
        # ====================================================

        liq_active = liq[
            (liq["activation_time"] <= fvg_time)
            &
            (
                liq["state"].isin(
                    ["ACTIVE", "FRESH", "VALID"]
                )
            )
        ].copy()

        # ----------------------------------------------------
        # Overlap
        # ----------------------------------------------------

        overlap = liq_active[
            (
                liq_active["level_price"]
                .between(lower, upper, inclusive="both")
            )
        ]

        # ----------------------------------------------------
        # Nearest liquidity above / below
        # ----------------------------------------------------

        below = liq_active[
            liq_active["level_price"] < lower
        ].copy()

        above = liq_active[
            liq_active["level_price"] > upper
        ].copy()

        if len(below):

            nearest_below = below.iloc[
                (lower - below["level_price"]).argmin()
            ]

            below_id = nearest_below["level_id"]
            below_type = nearest_below["liquidity_type"]
            below_price = nearest_below["level_price"]

        else:

            below_id = np.nan
            below_type = np.nan
            below_price = np.nan

        if len(above):

            nearest_above = above.iloc[
                (above["level_price"] - upper).argmin()
            ]

            above_id = nearest_above["level_id"]
            above_type = nearest_above["liquidity_type"]
            above_price = nearest_above["level_price"]

        else:

            above_id = np.nan
            above_type = np.nan
            above_price = np.nan

        # ----------------------------------------------------
        # Overlap context
        # ----------------------------------------------------

        if len(overlap):

            overlap_ids = tuple(
                overlap["level_id"].tolist()
            )

            overlap_types = tuple(
                overlap["liquidity_type"].tolist()
            )

        else:

            overlap_ids = tuple()
            overlap_types = tuple()

        # ====================================================
        # DIRECTIONAL RELATIONSHIP
        # ====================================================

        fvg_direction = (
            "BULLISH"
            if f["fvg_type"] == "BULLISH_FVG"
            else "BEARISH"
        )

        if active_bias == fvg_direction:
            bias_alignment = "ALIGNED"

        elif pd.isna(active_bias) or active_bias == "UNKNOWN":
            bias_alignment = "UNKNOWN"

        else:
            bias_alignment = "OPPOSED"

        if context_bos_direction == fvg_direction:
            bos_alignment = "ALIGNED"

        elif pd.isna(context_bos_direction):
            bos_alignment = "UNKNOWN"

        else:
            bos_alignment = "OPPOSED"

        if context_choch_direction == fvg_direction:
            choch_alignment = "ALIGNED"

        elif pd.isna(context_choch_direction):
            choch_alignment = "UNKNOWN"

        else:
            choch_alignment = "OPPOSED"

        # ====================================================
        # APPEND
        # ====================================================

        results.append(
            {
                # --------------------------------------------
                # FVG identity
                # --------------------------------------------

                "fvg_id": f["fvg_id"],
                "fvg_type": f["fvg_type"],
                "created_at": fvg_time,
                "lower_bound": lower,
                "upper_bound": upper,
                "gap_size": f["gap_size"],
                "lifecycle_state": f["lifecycle_state"],

                # --------------------------------------------
                # Structure state
                # --------------------------------------------

                "active_bias": active_bias,
                "bias_alignment": bias_alignment,

                "last_bos": last_bos,
                "last_bos_direction": last_bos_direction,
                "last_bos_level": last_bos_level,
                "last_bos_level_id": last_bos_level_id,

                "last_choch": last_choch,
                "last_choch_direction": last_choch_direction,
                "last_choch_level": last_choch_level,
                "last_choch_level_id": last_choch_level_id,

                # --------------------------------------------
                # BOS event context
                # --------------------------------------------

                "context_bos_id": context_bos_id,
                "context_bos_level_id": context_bos_level_id,
                "context_bos_direction": context_bos_direction,
                "context_bos_level": context_bos_level,
                "context_bos_time": context_bos_time,
                "bos_alignment": bos_alignment,

                # --------------------------------------------
                # CHoCH event context
                # --------------------------------------------

                "context_choch_id": context_choch_id,
                "context_choch_level_id": context_choch_level_id,
                "context_choch_direction": context_choch_direction,
                "context_choch_level": context_choch_level,
                "context_choch_time": context_choch_time,
                "choch_alignment": choch_alignment,

                # --------------------------------------------
                # Liquidity context
                # --------------------------------------------

                "overlapping_liquidity_ids": overlap_ids,
                "overlapping_liquidity_types": overlap_types,

                "nearest_liquidity_below_id": below_id,
                "nearest_liquidity_below_type": below_type,
                "nearest_liquidity_below_price": below_price,

                "nearest_liquidity_above_id": above_id,
                "nearest_liquidity_above_type": above_type,
                "nearest_liquidity_above_price": above_price,
            }
        )

    return pd.DataFrame(results)


# ============================================================
# BUILD
# ============================================================

fvg_context_engine_v1 = build_fvg_context_engine_v1(
    fvg_registry=fvg_lifecycle_registry_v1,
    liquidity_registry=unified_liquidity_registry_v1,
    structural_bos=structural_bos_v2,
    structural_choch=structural_choch_v2,
    structure_state=structure_state_v2,
)


# ============================================================
# SUMMARY
# ============================================================

print("=" * 100)
print("FVG ↔ STRUCTURE / LIQUIDITY CONTEXT ENGINE V1")
print("=" * 100)

print(f"Rows        : {len(fvg_context_engine_v1)}")
print(f"Columns     : {len(fvg_context_engine_v1.columns)}")

print("\nFVG Type:")
print(
    fvg_context_engine_v1["fvg_type"]
    .value_counts(dropna=False)
)

print("\nLifecycle:")
print(
    fvg_context_engine_v1["lifecycle_state"]
    .value_counts(dropna=False)
)

print("\nBias Alignment:")
print(
    fvg_context_engine_v1["bias_alignment"]
    .value_counts(dropna=False)
)

print("\nBOS Alignment:")
print(
    fvg_context_engine_v1["bos_alignment"]
    .value_counts(dropna=False)
)

print("\nCHoCH Alignment:")
print(
    fvg_context_engine_v1["choch_alignment"]
    .value_counts(dropna=False)
)

print("\nLiquidity Overlap:")
print(
    (
        fvg_context_engine_v1[
            "overlapping_liquidity_ids"
        ].apply(len) > 0
    )
    .value_counts(dropna=False)
)

print("\n" + "=" * 100)
print("SAMPLE")
print("=" * 100)

display(
    fvg_context_engine_v1.head(10)
)

FVG ↔ STRUCTURE / LIQUIDITY CONTEXT ENGINE V1
Rows        : 99
Columns     : 37

FVG Type:
fvg_type
BEARISH_FVG    56
BULLISH_FVG    43
Name: count, dtype: int64

Lifecycle:
lifecycle_state
MITIGATED    57
FILLED       37
FRESH         5
Name: count, dtype: int64

Bias Alignment:
bias_alignment
ALIGNED    64
OPPOSED    33
UNKNOWN     2
Name: count, dtype: int64

BOS Alignment:
bos_alignment
OPPOSED    97
UNKNOWN     2
Name: count, dtype: int64

CHoCH Alignment:
choch_alignment
OPPOSED    96
UNKNOWN     3
Name: count, dtype: int64

Liquidity Overlap:
overlapping_liquidity_ids
False    99
Name: count, dtype: int64

SAMPLE


,fvg_id,fvg_type,created_at,lower_bound,upper_bound,gap_size,lifecycle_state,active_bias,bias_alignment,last_bos,...,context_choch_time,choch_alignment,overlapping_liquidity_ids,overlapping_liquidity_types,nearest_liquidity_below_id,nearest_liquidity_below_type,nearest_liquidity_below_price,nearest_liquidity_above_id,nearest_liquidity_above_type,nearest_liquidity_above_price
0,FVG_B_20261002T221500Z,BULLISH_FVG,2026-10-02 22:15:00+00:00,1.12520,1.12526,0.00006,FILLED,UNKNOWN,UNKNOWN,NaN,...,NaT,UNKNOWN,(),(),NaN,NaN,NaN,NaN,NaN,NaN
1,FVG_B_20261002T222500Z,BULLISH_FVG,2026-10-02 22:25:00+00:00,1.12554,1.12577,0.00023,MITIGATED,UNKNOWN,UNKNOWN,NaN,...,NaT,UNKNOWN,(),(),NaN,NaN,NaN,NaN,NaN,NaN
2,FVG_S_20261005T005500Z,BEARISH_FVG,2026-10-05 00:55:00+00:00,1.12538,1.12567,0.00029,MITIGATED,BULLISH,OPPOSED,BOS_PH_002,...,NaT,UNKNOWN,(),(),NaN,NaN,NaN,NaN,NaN,NaN
3,FVG_S_20261005T031500Z,BEARISH_FVG,2026-10-05 03:15:00+00:00,1.12498,1.12528,0.00030,FILLED,BEARISH,ALIGNED,BOS_PL_001,...,2026-10-05 02:05:00+00:00,OPPOSED,(),(),NaN,NaN,NaN,NaN,NaN,NaN
4,FVG_S_20261005T034500Z,BEARISH_FVG,2026-10-05 03:45:00+00:00,1.12451,1.12459,0.00008,FILLED,BEARISH,ALIGNED,BOS_PL_001,...,2026-10-05 02:05:00+00:00,OPPOSED,(),(),NaN,NaN,NaN,NaN,NaN,NaN
5,FVG_S_20261005T041000Z,BEARISH_FVG,2026-10-05 04:10:00+00:00,1.12392,1.12406,0.00014,FILLED,BEARISH,ALIGNED,BOS_PL_001,...,2026-10-05 02:05:00+00:00,OPPOSED,(),(),NaN,NaN,NaN,NaN,NaN,NaN
6,FVG_S_20261005T051000Z,BEARISH_FVG,2026-10-05 05:10:00+00:00,1.12282,1.12293,0.00011,FILLED,BEARISH,ALIGNED,BOS_PL_001,...,2026-10-05 02:05:00+00:00,OPPOSED,(),(),NaN,NaN,NaN,NaN,NaN,NaN
7,FVG_S_20261005T051500Z,BEARISH_FVG,2026-10-05 05:15:00+00:00,1.12249,1.12267,0.00018,FILLED,BEARISH,ALIGNED,BOS_PL_001,...,2026-10-05 02:05:00+00:00,OPPOSED,(),(),NaN,NaN,NaN,NaN,NaN,NaN
8,FVG_S_20261005T052000Z,BEARISH_FVG,2026-10-05 05:20:00+00:00,1.12152,1.12230,0.00078,MITIGATED,BEARISH,ALIGNED,BOS_PL_001,...,2026-10-05 02:05:00+00:00,OPPOSED,(),(),NaN,NaN,NaN,NaN,NaN,NaN
9,FVG_S_20261005T055000Z,BEARISH_FVG,2026-10-05 05:50:00+00:00,1.11950,1.12028,0.00078,MITIGATED,BEARISH,ALIGNED,BOS_PL_001,...,2026-10-05 02:05:00+00:00,OPPOSED,(),(),NaN,NaN,NaN,NaN,NaN,NaN


In [137]:
# ============================================================
# FVG CONTEXT ENGINE V1
# STEP 3 — SEMANTIC DIAGNOSTIC AUDIT
# READ-ONLY
# ============================================================

import pandas as pd
import numpy as np

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — SEMANTIC DIAGNOSTIC AUDIT")
print("=" * 100)


# ============================================================
# 1. FVG DIRECTION
# ============================================================

print("\n" + "-" * 100)
print("1. FVG DIRECTION")
print("-" * 100)

fvg_direction_audit = (
    fvg_lifecycle_registry_v1[
        ["fvg_id", "fvg_type", "created_at"]
    ]
    .copy()
)

fvg_direction_audit["expected_direction"] = (
    fvg_direction_audit["fvg_type"]
    .map({
        "BULLISH_FVG": "BULLISH",
        "BEARISH_FVG": "BEARISH",
    })
)

print(
    fvg_direction_audit["expected_direction"]
    .value_counts(dropna=False)
)


# ============================================================
# 2. STRUCTURAL BOS SEMANTICS
# ============================================================

print("\n" + "-" * 100)
print("2. STRUCTURAL BOS DIRECTION")
print("-" * 100)

print(
    structural_bos_v2[
        [
            "bos_id",
            "level_id",
            "bos_direction",
            "bos_level",
            "bos_time",
            "bos_price",
            "source_type",
        ]
    ]
    .head(20)
    .to_string(index=False)
)

print("\nBOS direction counts:")
print(
    structural_bos_v2["bos_direction"]
    .value_counts(dropna=False)
)


# ============================================================
# 3. CHoCH SEMANTICS
# ============================================================

print("\n" + "-" * 100)
print("3. STRUCTURAL CHoCH DIRECTION")
print("-" * 100)

print(
    structural_choch_v2[
        [
            "choch_id",
            "bos_id",
            "level_id",
            "choch_direction",
            "choch_level",
            "choch_time",
            "structure_bias_before",
            "structure_bias_after",
        ]
    ]
    .head(20)
    .to_string(index=False)
)

print("\nCHoCH direction counts:")
print(
    structural_choch_v2["choch_direction"]
    .value_counts(dropna=False)
)


# ============================================================
# 4. STRUCTURE STATE
# ============================================================

print("\n" + "-" * 100)
print("4. STRUCTURE STATE")
print("-" * 100)

print(
    structure_state_v2[
        [
            "active_bias",
            "last_bos",
            "last_bos_direction",
            "last_bos_level",
            "last_bos_level_id",
            "last_choch",
            "last_choch_direction",
            "last_choch_level",
            "last_choch_level_id",
        ]
    ]
    .tail(15)
    .to_string()
)


# ============================================================
# 5. LIQUIDITY STATE VOCABULARY
# ============================================================

print("\n" + "-" * 100)
print("5. UNIFIED LIQUIDITY STATE VOCABULARY")
print("-" * 100)

print(
    unified_liquidity_registry_v1[
        [
            "level_id",
            "liquidity_source",
            "liquidity_type",
            "level_price",
            "activation_time",
            "state",
            "event_time",
            "event_price",
            "event_direction",
            "event_type",
        ]
    ]
    .head(20)
    .to_string(index=False)
)

print("\nLiquidity state counts:")
print(
    unified_liquidity_registry_v1["state"]
    .value_counts(dropna=False)
)


# ============================================================
# 6. LIQUIDITY ACTIVATION TIME
# ============================================================

print("\n" + "-" * 100)
print("6. LIQUIDITY ACTIVATION TIME")
print("-" * 100)

liq_time = unified_liquidity_registry_v1[
    "activation_time"
]

print("Non-null:", liq_time.notna().sum())
print("Null    :", liq_time.isna().sum())

print("\nSample activation times:")
print(
    liq_time
    .dropna()
    .sort_values()
    .head(15)
    .to_string(index=False)
)


# ============================================================
# 7. LIQUIDITY PRICE RANGE VS FVG RANGE
# ============================================================

print("\n" + "-" * 100)
print("7. PRICE RANGE DIAGNOSTIC")
print("-" * 100)

print(
    "Liquidity price min :",
    unified_liquidity_registry_v1["level_price"].min()
)

print(
    "Liquidity price max :",
    unified_liquidity_registry_v1["level_price"].max()
)

print(
    "FVG lower min       :",
    fvg_lifecycle_registry_v1["lower_bound"].min()
)

print(
    "FVG upper max       :",
    fvg_lifecycle_registry_v1["upper_bound"].max()
)


# ============================================================
# 8. MANUAL OVERLAP TEST
# ============================================================

print("\n" + "-" * 100)
print("8. MANUAL FVG ↔ LIQUIDITY OVERLAP TEST")
print("-" * 100)

manual_overlap = []

for _, f in fvg_lifecycle_registry_v1.iterrows():

    overlaps = unified_liquidity_registry_v1[
        unified_liquidity_registry_v1["level_price"].between(
            f["lower_bound"],
            f["upper_bound"],
            inclusive="both",
        )
    ]

    if len(overlaps):

        manual_overlap.append({
            "fvg_id": f["fvg_id"],
            "created_at": f["created_at"],
            "fvg_type": f["fvg_type"],
            "lower": f["lower_bound"],
            "upper": f["upper_bound"],
            "overlap_count": len(overlaps),
            "level_ids": tuple(
                overlaps["level_id"].tolist()
            ),
        })


manual_overlap_df = pd.DataFrame(manual_overlap)

print(
    f"FVGs with PRICE overlap (ignoring time/state): "
    f"{len(manual_overlap_df)}"
)

if len(manual_overlap_df):

    display(
        manual_overlap_df.head(20)
    )


# ============================================================
# 9. TEMPORAL OVERLAP TEST
# ============================================================

print("\n" + "-" * 100)
print("9. TEMPORAL + PRICE OVERLAP TEST")
print("-" * 100)

temporal_overlap = []

liq = unified_liquidity_registry_v1.copy()

liq["activation_time"] = pd.to_datetime(
    liq["activation_time"],
    utc=True,
    errors="coerce",
)

for _, f in fvg_lifecycle_registry_v1.iterrows():

    created = pd.to_datetime(
        f["created_at"],
        utc=True,
        errors="coerce",
    )

    candidates = liq[
        liq["activation_time"].notna()
        &
        (liq["activation_time"] <= created)
        &
        liq["level_price"].between(
            f["lower_bound"],
            f["upper_bound"],
            inclusive="both",
        )
    ]

    if len(candidates):

        temporal_overlap.append({
            "fvg_id": f["fvg_id"],
            "created_at": created,
            "fvg_type": f["fvg_type"],
            "level_ids": tuple(
                candidates["level_id"].tolist()
            ),
            "states": tuple(
                candidates["state"].tolist()
            ),
        })


temporal_overlap_df = pd.DataFrame(
    temporal_overlap
)

print(
    "FVGs with temporal + price overlap:",
    len(temporal_overlap_df)
)

if len(temporal_overlap_df):

    display(
        temporal_overlap_df.head(20)
    )


# ============================================================
# 10. EXISTING ENGINE ALIGNMENT DISTRIBUTION
# ============================================================

print("\n" + "-" * 100)
print("10. CURRENT ENGINE ALIGNMENT")
print("-" * 100)

print("\nBOS:")
print(
    fvg_context_engine_v1[
        [
            "fvg_type",
            "context_bos_direction",
            "bos_alignment",
        ]
    ]
    .value_counts(dropna=False)
)

print("\nCHoCH:")
print(
    fvg_context_engine_v1[
        [
            "fvg_type",
            "context_choch_direction",
            "choch_alignment",
        ]
    ]
    .value_counts(dropna=False)
)

print("\nBias:")
print(
    fvg_context_engine_v1[
        [
            "fvg_type",
            "active_bias",
            "bias_alignment",
        ]
    ]
    .value_counts(dropna=False)
)


print("\n" + "=" * 100)
print("DIAGNOSTIC COMPLETE — NO DATAFRAME MODIFIED")
print("=" * 100)

FVG CONTEXT ENGINE V1 — SEMANTIC DIAGNOSTIC AUDIT

----------------------------------------------------------------------------------------------------
1. FVG DIRECTION
----------------------------------------------------------------------------------------------------
expected_direction
BEARISH    56
BULLISH    43
Name: count, dtype: int64

----------------------------------------------------------------------------------------------------
2. STRUCTURAL BOS DIRECTION
----------------------------------------------------------------------------------------------------
    bos_id level_id bos_direction  bos_level                  bos_time  bos_price    source_type
BOS_PH_002   PH_002   BOS_BULLISH    1.12553 2026-10-05 00:15:00+00:00    1.12578 PROTECTED_HIGH
BOS_PH_001   PH_001   BOS_BULLISH    1.12594 2026-10-05 01:45:00+00:00    1.12595 PROTECTED_HIGH
BOS_PL_001   PL_001   BOS_BEARISH    1.12563 2026-10-05 02:05:00+00:00    1.12562  PROTECTED_LOW
BOS_PH_005   PH_005   BOS_BULLISH    1

,fvg_id,created_at,fvg_type,lower,upper,overlap_count,level_ids
0,FVG_B_20261002T222500Z,2026-10-02 22:25:00+00:00,BULLISH_FVG,1.12554,1.12577,5,"(PL_001, PH_003, EQH_001, PH_004, PH_016)"
1,FVG_S_20261005T005500Z,2026-10-05 00:55:00+00:00,BEARISH_FVG,1.12538,1.12567,6,"(PH_002, PL_001, PH_003, EQH_001, PH_004, PH_021)"
2,FVG_S_20261005T041000Z,2026-10-05 04:10:00+00:00,BEARISH_FVG,1.12392,1.12406,1,"(PL_013,)"
3,FVG_S_20261005T051500Z,2026-10-05 05:15:00+00:00,BEARISH_FVG,1.12249,1.12267,1,"(PL_012,)"
4,FVG_S_20261005T052000Z,2026-10-05 05:20:00+00:00,BEARISH_FVG,1.12152,1.12230,6,"(PL_005, EQL_002, PL_006, PH_012, PH_013, PH_014)"
5,FVG_S_20261005T055000Z,2026-10-05 05:50:00+00:00,BEARISH_FVG,1.11950,1.12028,10,"(PL_004, PH_009, PH_024, PH_026, EQH_004, PL_0..."
6,FVG_S_20261005T093000Z,2026-10-05 09:30:00+00:00,BEARISH_FVG,1.11879,1.11924,4,"(PL_017, PH_025, PL_020, PL_022)"
7,FVG_S_20261005T093500Z,2026-10-05 09:35:00+00:00,BEARISH_FVG,1.11802,1.11858,1,"(PL_003,)"
8,FVG_B_20261005T103500Z,2026-10-05 10:35:00+00:00,BULLISH_FVG,1.11912,1.11960,3,"(PL_004, PH_024, PL_021)"
9,FVG_B_20261005T121000Z,2026-10-05 12:10:00+00:00,BULLISH_FVG,1.12122,1.12140,5,"(EQH_002, PH_008, PH_010, PL_007, PL_009)"



----------------------------------------------------------------------------------------------------
9. TEMPORAL + PRICE OVERLAP TEST
----------------------------------------------------------------------------------------------------
FVGs with temporal + price overlap: 37


,fvg_id,created_at,fvg_type,level_ids,states
0,FVG_S_20261005T005500Z,2026-10-05 00:55:00+00:00,BEARISH_FVG,"(PH_002,)","(BROKEN,)"
1,FVG_B_20261005T153000Z,2026-10-05 15:30:00+00:00,BULLISH_FVG,"(PH_005, PH_006)","(BROKEN, BROKEN)"
2,FVG_S_20261005T161000Z,2026-10-05 16:10:00+00:00,BEARISH_FVG,"(PH_005, PH_006)","(BROKEN, BROKEN)"
3,FVG_S_20261005T164000Z,2026-10-05 16:40:00+00:00,BEARISH_FVG,"(PL_004,)","(BROKEN,)"
4,FVG_B_20261005T174500Z,2026-10-05 17:45:00+00:00,BULLISH_FVG,"(PL_004, PH_009)","(BROKEN, BROKEN)"
5,FVG_B_20261005T180500Z,2026-10-05 18:05:00+00:00,BULLISH_FVG,"(PH_005, PH_006, PH_007)","(BROKEN, BROKEN, BROKEN)"
6,FVG_B_20261005T210000Z,2026-10-05 21:00:00+00:00,BULLISH_FVG,"(PH_005, PH_006)","(BROKEN, BROKEN)"
7,FVG_B_20261005T210500Z,2026-10-05 21:05:00+00:00,BULLISH_FVG,"(PH_007,)","(BROKEN,)"
8,FVG_B_20261005T213000Z,2026-10-05 21:30:00+00:00,BULLISH_FVG,"(PH_010,)","(BROKEN,)"
9,FVG_S_20261006T030500Z,2026-10-06 03:05:00+00:00,BEARISH_FVG,"(PL_005, EQL_002)","(BROKEN, SWEPT)"



----------------------------------------------------------------------------------------------------
10. CURRENT ENGINE ALIGNMENT
----------------------------------------------------------------------------------------------------

BOS:
fvg_type     context_bos_direction  bos_alignment
BEARISH_FVG  BOS_BEARISH            OPPOSED          38
BULLISH_FVG  BOS_BULLISH            OPPOSED          26
BEARISH_FVG  BOS_BULLISH            OPPOSED          18
BULLISH_FVG  BOS_BEARISH            OPPOSED          15
             NaN                    UNKNOWN           2
Name: count, dtype: int64

CHoCH:
fvg_type     context_choch_direction  choch_alignment
BEARISH_FVG  BOS_BEARISH              OPPOSED            38
BULLISH_FVG  BOS_BULLISH              OPPOSED            26
BEARISH_FVG  BOS_BULLISH              OPPOSED            17
BULLISH_FVG  BOS_BEARISH              OPPOSED            15
             NaN                      UNKNOWN             2
BEARISH_FVG  NaN                      UNKNOW

In [138]:
# ==================================================================================================
# FVG ↔ LIQUIDITY TEMPORAL STATE AUDIT V1
# READ-ONLY — NO DATAFRAME MODIFICATION
# ==================================================================================================

fvg = fvg_lifecycle_registry_v1.copy()
liq = unified_liquidity_registry_v1.copy()

# ------------------------------------------------------------------
# UTC normalization
# ------------------------------------------------------------------

fvg["created_at"] = pd.to_datetime(
    fvg["created_at"],
    utc=True,
    errors="coerce"
)

liq["activation_time"] = pd.to_datetime(
    liq["activation_time"],
    utc=True,
    errors="coerce"
)

liq["event_time"] = pd.to_datetime(
    liq["event_time"],
    utc=True,
    errors="coerce"
)

# ------------------------------------------------------------------
# Historical active-at-FVG-creation audit
# ------------------------------------------------------------------

audit_rows = []

for _, f in fvg.iterrows():

    fvg_time = f["created_at"]
    lower = f["lower_bound"]
    upper = f["upper_bound"]

    historical_active = liq[
        (liq["activation_time"] <= fvg_time)
        &
        (
            liq["event_time"].isna()
            |
            (liq["event_time"] > fvg_time)
        )
    ].copy()

    overlap = historical_active[
        historical_active["level_price"].between(
            lower,
            upper,
            inclusive="both"
        )
    ]

    audit_rows.append({
        "fvg_id": f["fvg_id"],
        "created_at": fvg_time,
        "fvg_type": f["fvg_type"],

        "historical_active_count": len(historical_active),
        "historical_overlap_count": len(overlap),

        "overlap_level_ids": tuple(
            overlap["level_id"].tolist()
        ),

        "overlap_current_states": tuple(
            overlap["state"].tolist()
        ),

        "overlap_event_times": tuple(
            overlap["event_time"].tolist()
        ),
    })


fvg_liquidity_temporal_audit_v1 = pd.DataFrame(audit_rows)


# ==================================================================================================
# SUMMARY
# ==================================================================================================

print("=" * 100)
print("FVG ↔ LIQUIDITY TEMPORAL STATE AUDIT V1")
print("=" * 100)

print(
    f"Total FVGs                         : "
    f"{len(fvg_liquidity_temporal_audit_v1)}"
)

print(
    f"FVGs with historical active levels: "
    f"{(
        fvg_liquidity_temporal_audit_v1[
            fvg_liquidity_temporal_audit_v1["historical_active_count"] > 0
        ]
    ).shape[0]}"
)

print(
    f"FVGs with historical price overlap: "
    f"{(
        fvg_liquidity_temporal_audit_v1[
            fvg_liquidity_temporal_audit_v1["historical_overlap_count"] > 0
        ]
    ).shape[0]}"
)

print(
    f"Total historical overlap matches : "
    f"{fvg_liquidity_temporal_audit_v1['historical_overlap_count'].sum()}"
)

print()
print("-" * 100)
print("SAMPLE OVERLAPS")
print("-" * 100)

display(
    fvg_liquidity_temporal_audit_v1[
        fvg_liquidity_temporal_audit_v1["historical_overlap_count"] > 0
    ].head(20)
)

print()
print("-" * 100)
print("TEMPORAL CONTRACT")
print("-" * 100)

print(
    "ACTIVE_AT_FVG_TIME = "
    "activation_time <= created_at "
    "AND "
    "(event_time IS NULL OR event_time > created_at)"
)

print("=" * 100)

FVG ↔ LIQUIDITY TEMPORAL STATE AUDIT V1
Total FVGs                         : 99
FVGs with historical active levels: 97
FVGs with historical price overlap: 0
Total historical overlap matches : 0

----------------------------------------------------------------------------------------------------
SAMPLE OVERLAPS
----------------------------------------------------------------------------------------------------


,fvg_id,created_at,fvg_type,historical_active_count,historical_overlap_count,overlap_level_ids,overlap_current_states,overlap_event_times



----------------------------------------------------------------------------------------------------
TEMPORAL CONTRACT
----------------------------------------------------------------------------------------------------
ACTIVE_AT_FVG_TIME = activation_time <= created_at AND (event_time IS NULL OR event_time > created_at)


In [139]:
# ==================================================================================================
# FVG ↔ LIQUIDITY HISTORICAL PRICE RELATION AUDIT V1
# READ-ONLY
# ==================================================================================================

fvg = fvg_lifecycle_registry_v1.copy()
liq = unified_liquidity_registry_v1.copy()

# ------------------------------------------------------------------
# UTC normalization
# ------------------------------------------------------------------

fvg["created_at"] = pd.to_datetime(
    fvg["created_at"],
    utc=True,
    errors="coerce"
)

liq["activation_time"] = pd.to_datetime(
    liq["activation_time"],
    utc=True,
    errors="coerce"
)

liq["event_time"] = pd.to_datetime(
    liq["event_time"],
    utc=True,
    errors="coerce"
)

# ------------------------------------------------------------------
# Historical context
# ------------------------------------------------------------------

rows = []

for _, f in fvg.iterrows():

    t = f["created_at"]
    lower = f["lower_bound"]
    upper = f["upper_bound"]

    historical_active = liq[
        (liq["activation_time"] <= t)
        &
        (
            liq["event_time"].isna()
            |
            (liq["event_time"] > t)
        )
    ].copy()

    # --------------------------------------------------------------
    # Price relationship
    # --------------------------------------------------------------

    historical_active["price_relation"] = "UNKNOWN"

    historical_active.loc[
        historical_active["level_price"] < lower,
        "price_relation"
    ] = "BELOW"

    historical_active.loc[
        historical_active["level_price"] > upper,
        "price_relation"
    ] = "ABOVE"

    historical_active.loc[
        historical_active["level_price"].between(
            lower,
            upper,
            inclusive="both"
        ),
        "price_relation"
    ] = "OVERLAP"

    below = historical_active[
        historical_active["price_relation"] == "BELOW"
    ]

    above = historical_active[
        historical_active["price_relation"] == "ABOVE"
    ]

    overlap = historical_active[
        historical_active["price_relation"] == "OVERLAP"
    ]

    rows.append({
        "fvg_id": f["fvg_id"],
        "created_at": t,
        "fvg_type": f["fvg_type"],

        "fvg_lower": lower,
        "fvg_upper": upper,

        "historical_active_count": len(historical_active),

        "below_count": len(below),
        "above_count": len(above),
        "overlap_count": len(overlap),

        "below_ids": tuple(below["level_id"].tolist()),
        "above_ids": tuple(above["level_id"].tolist()),
        "overlap_ids": tuple(overlap["level_id"].tolist()),

        "below_prices": tuple(
            below["level_price"].tolist()
        ),
        "above_prices": tuple(
            above["level_price"].tolist()
        ),
        "overlap_prices": tuple(
            overlap["level_price"].tolist()
        ),
    })


fvg_liquidity_price_relation_audit_v1 = pd.DataFrame(rows)


# ==================================================================================================
# SUMMARY
# ==================================================================================================

print("=" * 100)
print("FVG ↔ LIQUIDITY HISTORICAL PRICE RELATION AUDIT V1")
print("=" * 100)

print(
    f"Total FVGs                    : "
    f"{len(fvg_liquidity_price_relation_audit_v1)}"
)

print(
    f"FVGs with historical liquidity : "
    f"{(
        fvg_liquidity_price_relation_audit_v1[
            fvg_liquidity_price_relation_audit_v1[
                "historical_active_count"
            ] > 0
        ]
    ).shape[0]}"
)

print(
    f"FVGs with BELOW liquidity      : "
    f"{(
        fvg_liquidity_price_relation_audit_v1[
            fvg_liquidity_price_relation_audit_v1[
                "below_count"
            ] > 0
        ]
    ).shape[0]}"
)

print(
    f"FVGs with ABOVE liquidity      : "
    f"{(
        fvg_liquidity_price_relation_audit_v1[
            fvg_liquidity_price_relation_audit_v1[
                "above_count"
            ] > 0
        ]
    ).shape[0]}"
)

print(
    f"FVGs with OVERLAP liquidity    : "
    f"{(
        fvg_liquidity_price_relation_audit_v1[
            fvg_liquidity_price_relation_audit_v1[
                "overlap_count"
            ] > 0
        ]
    ).shape[0]}"
)

print()
print("-" * 100)
print("RELATION COUNTS")
print("-" * 100)

print(
    "BELOW  :",
    int(
        fvg_liquidity_price_relation_audit_v1["below_count"].sum()
    )
)

print(
    "ABOVE  :",
    int(
        fvg_liquidity_price_relation_audit_v1["above_count"].sum()
    )
)

print(
    "OVERLAP:",
    int(
        fvg_liquidity_price_relation_audit_v1["overlap_count"].sum()
    )
)

print()
print("-" * 100)
print("SAMPLE")
print("-" * 100)

display(
    fvg_liquidity_price_relation_audit_v1[
        [
            "fvg_id",
            "created_at",
            "fvg_type",
            "historical_active_count",
            "below_count",
            "above_count",
            "overlap_count",
            "below_ids",
            "above_ids",
            "overlap_ids",
        ]
    ].head(20)
)

print("=" * 100)

FVG ↔ LIQUIDITY HISTORICAL PRICE RELATION AUDIT V1
Total FVGs                    : 99
FVGs with historical liquidity : 97
FVGs with BELOW liquidity      : 88
FVGs with ABOVE liquidity      : 88
FVGs with OVERLAP liquidity    : 0

----------------------------------------------------------------------------------------------------
RELATION COUNTS
----------------------------------------------------------------------------------------------------
BELOW  : 441
ABOVE  : 343
OVERLAP: 0

----------------------------------------------------------------------------------------------------
SAMPLE
----------------------------------------------------------------------------------------------------


,fvg_id,created_at,fvg_type,historical_active_count,below_count,above_count,overlap_count,below_ids,above_ids,overlap_ids
0,FVG_B_20261002T221500Z,2026-10-02 22:15:00+00:00,BULLISH_FVG,0,0,0,0,(),(),()
1,FVG_B_20261002T222500Z,2026-10-02 22:25:00+00:00,BULLISH_FVG,0,0,0,0,(),(),()
2,FVG_S_20261005T005500Z,2026-10-05 00:55:00+00:00,BEARISH_FVG,1,0,1,0,(),"(PH_001,)",()
3,FVG_S_20261005T031500Z,2026-10-05 03:15:00+00:00,BEARISH_FVG,3,0,3,0,(),"(PH_003, EQH_001, PH_004)",()
4,FVG_S_20261005T034500Z,2026-10-05 03:45:00+00:00,BEARISH_FVG,3,0,3,0,(),"(PH_003, EQH_001, PH_004)",()
5,FVG_S_20261005T041000Z,2026-10-05 04:10:00+00:00,BEARISH_FVG,3,0,3,0,(),"(PH_003, EQH_001, PH_004)",()
6,FVG_S_20261005T051000Z,2026-10-05 05:10:00+00:00,BEARISH_FVG,3,0,3,0,(),"(PH_003, EQH_001, PH_004)",()
7,FVG_S_20261005T051500Z,2026-10-05 05:15:00+00:00,BEARISH_FVG,3,0,3,0,(),"(PH_003, EQH_001, PH_004)",()
8,FVG_S_20261005T052000Z,2026-10-05 05:20:00+00:00,BEARISH_FVG,3,0,3,0,(),"(PH_003, EQH_001, PH_004)",()
9,FVG_S_20261005T055000Z,2026-10-05 05:50:00+00:00,BEARISH_FVG,3,0,3,0,(),"(PH_003, EQH_001, PH_004)",()


In [140]:
# FVG Contex Engine V1 _BUILDER

In [141]:
# ============================================================
# FVG CONTEXT ENGINE V1 — PATCHED BUILDER
# STRUCTURE STATE = STRICT POINT-IN-TIME SOURCE
# READ-ONLY
# ============================================================

import pandas as pd


def build_fvg_context_engine_v1_patched(
    fvg_registry,
    liquidity_registry,
    structural_bos,
    structural_choch,
    structure_state,
):
    """
    FVG Context Engine V1

    Strict temporal context builder.

    Important:
    - FVG remains its own object family.
    - No Liquidity identity is injected into FVG identity.
    - Structure bias comes ONLY from structure_state_v2.
    - No CHoCH fallback for active_bias.
    - All source inputs are treated as read-only.
    """

    # ========================================================
    # Defensive copies
    # ========================================================

    fvg = fvg_registry.copy(deep=True)
    liq = liquidity_registry.copy(deep=True)
    bos = structural_bos.copy(deep=True)
    choch = structural_choch.copy(deep=True)
    state = structure_state.copy(deep=True)

    # ========================================================
    # Required schemas
    # ========================================================

    required_fvg = {
        "fvg_id",
        "fvg_type",
        "created_at",
        "lower_bound",
        "upper_bound",
        "gap_size",
        "lifecycle_state",
        "event_time",
        "event_price",
        "event_direction",
        "event_type",
    }

    required_liq = {
        "level_id",
        "liquidity_source",
        "liquidity_type",
        "level_price",
        "activation_time",
        "state",
        "event_time",
        "event_price",
        "event_direction",
        "event_id",
        "event_type",
    }

    required_bos = {
        "bos_id",
        "level_id",
        "bos_direction",
        "bos_level",
        "bos_time",
        "bos_price",
        "created_at",
        "source_type",
    }

    required_choch = {
        "choch_id",
        "bos_id",
        "level_id",
        "choch_direction",
        "choch_level",
        "choch_time",
        "choch_price",
        "structure_bias_before",
        "structure_bias_after",
    }

    missing_fvg = required_fvg - set(fvg.columns)
    missing_liq = required_liq - set(liq.columns)
    missing_bos = required_bos - set(bos.columns)
    missing_choch = required_choch - set(choch.columns)

    if missing_fvg:
        raise ValueError(
            f"FVG registry missing columns: {sorted(missing_fvg)}"
        )

    if missing_liq:
        raise ValueError(
            f"Liquidity registry missing columns: {sorted(missing_liq)}"
        )

    if missing_bos:
        raise ValueError(
            f"Structural BOS missing columns: {sorted(missing_bos)}"
        )

    if missing_choch:
        raise ValueError(
            f"Structural CHoCH missing columns: {sorted(missing_choch)}"
        )

    # ========================================================
    # UTC normalization
    # ========================================================

    fvg["created_at"] = pd.to_datetime(
        fvg["created_at"],
        utc=True,
        errors="coerce"
    )

    fvg["event_time"] = pd.to_datetime(
        fvg["event_time"],
        utc=True,
        errors="coerce"
    )

    liq["activation_time"] = pd.to_datetime(
        liq["activation_time"],
        utc=True,
        errors="coerce"
    )

    liq["event_time"] = pd.to_datetime(
        liq["event_time"],
        utc=True,
        errors="coerce"
    )

    bos["bos_time"] = pd.to_datetime(
        bos["bos_time"],
        utc=True,
        errors="coerce"
    )

    bos["created_at"] = pd.to_datetime(
        bos["created_at"],
        utc=True,
        errors="coerce"
    )

    choch["choch_time"] = pd.to_datetime(
        choch["choch_time"],
        utc=True,
        errors="coerce"
    )

    # ========================================================
    # Direction normalization
    # ========================================================

    def normalize_direction(value):

        if pd.isna(value):
            return pd.NA

        value = str(value).upper().strip()

        if value in {
            "BULLISH",
            "BOS_BULLISH",
            "CHOCH_BULLISH",
        }:
            return "BULLISH"

        if value in {
            "BEARISH",
            "BOS_BEARISH",
            "CHOCH_BEARISH",
        }:
            return "BEARISH"

        return pd.NA

    def normalize_fvg_direction(value):

        if pd.isna(value):
            return pd.NA

        value = str(value).upper().strip()

        if value == "BULLISH_FVG":
            return "BULLISH"

        if value == "BEARISH_FVG":
            return "BEARISH"

        return pd.NA

    def alignment(fvg_direction, context_direction):

        if pd.isna(fvg_direction):
            return "UNKNOWN"

        if pd.isna(context_direction):
            return "UNKNOWN"

        if fvg_direction == context_direction:
            return "ALIGNED"

        return "OPPOSED"

    # ========================================================
    # Prepare Structure State timeline
    # ========================================================

    state = state.copy()

    if isinstance(state.index, pd.DatetimeIndex):

        state["_state_time"] = pd.to_datetime(
            state.index,
            utc=True,
            errors="coerce"
        )

        state_time_source = "DATETIME_INDEX"

    elif "timestamp" in state.columns:

        state["_state_time"] = pd.to_datetime(
            state["timestamp"],
            utc=True,
            errors="coerce"
        )

        state_time_source = "TIMESTAMP_COLUMN"

    elif "time" in state.columns:

        state["_state_time"] = pd.to_datetime(
            state["time"],
            utc=True,
            errors="coerce"
        )

        state_time_source = "TIME_COLUMN"

    else:

        raise ValueError(
            "structure_state_v2 has no valid temporal axis."
        )

    state = (
        state
        .dropna(subset=["_state_time"])
        .sort_values("_state_time")
    )

    # ========================================================
    # Sort temporal sources
    # ========================================================

    bos = bos.sort_values("bos_time")
    choch = choch.sort_values("choch_time")
    liq = liq.sort_values("activation_time")

    # ========================================================
    # Build Context
    # ========================================================

    records = []

    for _, f in fvg.iterrows():

        created_at = f["created_at"]

        fvg_direction = normalize_fvg_direction(
            f["fvg_type"]
        )

        # ----------------------------------------------------
        # Latest BOS at or before FVG creation
        # ----------------------------------------------------

        bos_context = bos[
            bos["bos_time"] <= created_at
        ]

        if bos_context.empty:

            latest_bos = None

        else:

            latest_bos = bos_context.iloc[-1]

        # ----------------------------------------------------
        # Latest CHoCH at or before FVG creation
        # ----------------------------------------------------

        choch_context = choch[
            choch["choch_time"] <= created_at
        ]

        if choch_context.empty:

            latest_choch = None

        else:

            latest_choch = choch_context.iloc[-1]

        # ----------------------------------------------------
        # STRICT POINT-IN-TIME STRUCTURE STATE
        #
        # NO CHoCH FALLBACK
        # ----------------------------------------------------

        state_context = state[
            state["_state_time"] <= created_at
        ]

        if state_context.empty:

            latest_state = None

        else:

            latest_state = state_context.iloc[-1]

        if latest_state is None:

            active_bias = pd.NA

        else:

            active_bias = latest_state.get(
                "active_bias",
                pd.NA
            )

            if pd.isna(active_bias):

                active_bias = pd.NA

            else:

                active_bias = str(
                    active_bias
                ).upper().strip()

        # ----------------------------------------------------
        # BOS context
        # ----------------------------------------------------

        if latest_bos is None:

            context_bos_id = pd.NA
            context_bos_level_id = pd.NA
            context_bos_direction_raw = pd.NA
            context_bos_direction = pd.NA
            context_bos_time = pd.NaT
            context_bos_price = pd.NA

        else:

            context_bos_id = latest_bos["bos_id"]
            context_bos_level_id = latest_bos["level_id"]

            context_bos_direction_raw = (
                latest_bos["bos_direction"]
            )

            context_bos_direction = normalize_direction(
                latest_bos["bos_direction"]
            )

            context_bos_time = latest_bos["bos_time"]
            context_bos_price = latest_bos["bos_price"]

        # ----------------------------------------------------
        # CHoCH context
        # ----------------------------------------------------

        if latest_choch is None:

            context_choch_id = pd.NA
            context_choch_bos_id = pd.NA
            context_choch_level_id = pd.NA
            context_choch_direction_raw = pd.NA
            context_choch_direction = pd.NA
            context_choch_time = pd.NaT
            context_choch_price = pd.NA
            context_bias_before = pd.NA
            context_bias_after = pd.NA

        else:

            context_choch_id = latest_choch["choch_id"]
            context_choch_bos_id = latest_choch["bos_id"]
            context_choch_level_id = latest_choch["level_id"]

            context_choch_direction_raw = (
                latest_choch["choch_direction"]
            )

            context_choch_direction = normalize_direction(
                latest_choch["choch_direction"]
            )

            context_choch_time = latest_choch["choch_time"]
            context_choch_price = latest_choch["choch_price"]

            context_bias_before = (
                latest_choch["structure_bias_before"]
            )

            context_bias_after = (
                latest_choch["structure_bias_after"]
            )

        # ----------------------------------------------------
        # Historical active liquidity
        #
        # activation_time <= FVG creation
        # AND
        # event_time is NULL OR event_time > FVG creation
        # ----------------------------------------------------

        historical_active = liq[
            (liq["activation_time"] <= created_at)
            &
            (
                liq["event_time"].isna()
                |
                (liq["event_time"] > created_at)
            )
        ].copy()

        # ----------------------------------------------------
        # Spatial relationship
        # ----------------------------------------------------

        below = historical_active[
            historical_active["level_price"]
            < f["lower_bound"]
        ]

        above = historical_active[
            historical_active["level_price"]
            > f["upper_bound"]
        ]

        overlap = historical_active[
            (historical_active["level_price"] >= f["lower_bound"])
            &
            (historical_active["level_price"] <= f["upper_bound"])
        ]

        # ----------------------------------------------------
        # Record
        # ----------------------------------------------------

        records.append(
            {
                # FVG identity
                "fvg_id": f["fvg_id"],
                "fvg_type": f["fvg_type"],
                "fvg_direction": fvg_direction,
                "created_at": created_at,
                "lower_bound": f["lower_bound"],
                "upper_bound": f["upper_bound"],
                "gap_size": f["gap_size"],

                # Lifecycle
                "lifecycle_state": f["lifecycle_state"],
                "event_time": f["event_time"],
                "event_price": f["event_price"],
                "event_direction": f["event_direction"],
                "event_type": f["event_type"],

                # Structure state
                "active_bias": active_bias,

                "bias_alignment": alignment(
                    fvg_direction,
                    normalize_direction(active_bias)
                ),

                # BOS
                "context_bos_id": context_bos_id,
                "context_bos_level_id": context_bos_level_id,
                "context_bos_direction_raw":
                    context_bos_direction_raw,
                "context_bos_direction":
                    context_bos_direction,
                "context_bos_time":
                    context_bos_time,
                "context_bos_price":
                    context_bos_price,

                "bos_alignment": alignment(
                    fvg_direction,
                    context_bos_direction
                ),

                # CHoCH
                "context_choch_id": context_choch_id,
                "context_choch_bos_id":
                    context_choch_bos_id,
                "context_choch_level_id":
                    context_choch_level_id,
                "context_choch_direction_raw":
                    context_choch_direction_raw,
                "context_choch_direction":
                    context_choch_direction,
                "context_choch_time":
                    context_choch_time,
                "context_choch_price":
                    context_choch_price,
                "context_bias_before":
                    context_bias_before,
                "context_bias_after":
                    context_bias_after,

                "choch_alignment": alignment(
                    fvg_direction,
                    context_choch_direction
                ),

                # Liquidity
                "historical_active_liquidity_count":
                    len(historical_active),

                "liquidity_below_count":
                    len(below),

                "liquidity_above_count":
                    len(above),

                "liquidity_overlap_count":
                    len(overlap),

                "liquidity_below_ids":
                    tuple(below["level_id"].tolist()),

                "liquidity_above_ids":
                    tuple(above["level_id"].tolist()),

                "liquidity_overlap_ids":
                    tuple(overlap["level_id"].tolist()),
            }
        )

    result = pd.DataFrame(records)

    return result

In [142]:
# ============================================================
# PATCH FVG CONTEXT ENGINE V1
# COMPATIBILITY FIX — FROZEN REGISTRIES UNCHANGED
# ============================================================

def build_fvg_context_engine_v1_patched(
    fvg_registry,
    liquidity_registry,
    structural_bos,
    structural_choch,
    structure_state,
):
    """
    FVG Context Engine V1
    Compatibility-patched against the FROZEN
    FVG Lifecycle Registry V1 schema.

    IMPORTANT:
    - Does NOT require event_direction from FVG registry.
    - Direction is deterministically derived from fvg_type.
    - Frozen upstream schemas remain unchanged.
    """

    import pandas as pd
    import numpy as np

    # --------------------------------------------------------
    # 1. Defensive copies
    # --------------------------------------------------------
    fvg = fvg_registry.copy(deep=True)
    liq = liquidity_registry.copy(deep=True)
    bos = structural_bos.copy(deep=True)
    choch = structural_choch.copy(deep=True)
    state = structure_state.copy(deep=True)

    # --------------------------------------------------------
    # 2. Frozen FVG Registry Contract
    # --------------------------------------------------------
    required_fvg = {
        "fvg_id",
        "fvg_type",
        "created_at",
        "lower_bound",
        "upper_bound",
        "gap_size",
        "candle_a",
        "candle_b",
        "candle_c",
        "lifecycle_state",
        "event_type",
        "event_time",
        "event_price",
    }

    missing_fvg = required_fvg - set(fvg.columns)

    if missing_fvg:
        raise ValueError(
            f"FVG registry missing columns: {sorted(missing_fvg)}"
        )

    # --------------------------------------------------------
    # 3. Required structural / liquidity fields
    # --------------------------------------------------------
    required_liq = {
        "level_id",
        "level_type",
        "level_price",
        "activation_time",
        "state",
        "side",
    }

    required_bos = {
        "time",
    }

    required_choch = {
        "time",
    }

    required_state = {
        "time",
    }

    missing_liq = required_liq - set(liq.columns)
    missing_bos = required_bos - set(bos.columns)
    missing_choch = required_choch - set(choch.columns)
    missing_state = required_state - set(state.columns)

    if missing_liq:
        raise ValueError(
            f"Liquidity registry missing columns: {sorted(missing_liq)}"
        )

    if missing_bos:
        raise ValueError(
            f"Structural BOS missing columns: {sorted(missing_bos)}"
        )

    if missing_choch:
        raise ValueError(
            f"Structural CHOCH missing columns: {sorted(missing_choch)}"
        )

    if missing_state:
        raise ValueError(
            f"Structure state missing columns: {sorted(missing_state)}"
        )

    # --------------------------------------------------------
    # 4. Normalize timestamps
    # --------------------------------------------------------
    fvg["created_at"] = pd.to_datetime(
        fvg["created_at"], utc=True, errors="coerce"
    )

    fvg["event_time"] = pd.to_datetime(
        fvg["event_time"], utc=True, errors="coerce"
    )

    liq["activation_time"] = pd.to_datetime(
        liq["activation_time"], utc=True, errors="coerce"
    )

    bos["time"] = pd.to_datetime(
        bos["time"], utc=True, errors="coerce"
    )

    choch["time"] = pd.to_datetime(
        choch["time"], utc=True, errors="coerce"
    )

    state["time"] = pd.to_datetime(
        state["time"], utc=True, errors="coerce"
    )

    # --------------------------------------------------------
    # 5. Validate FVG vocabulary
    # --------------------------------------------------------
    allowed_fvg_types = {
        "BULLISH_FVG",
        "BEARISH_FVG",
    }

    invalid_types = set(
        fvg["fvg_type"].dropna().unique()
    ) - allowed_fvg_types

    if invalid_types:
        raise ValueError(
            f"Invalid FVG types: {sorted(invalid_types)}"
        )

    # --------------------------------------------------------
    # 6. DERIVE DIRECTION FROM FVG TYPE
    # --------------------------------------------------------
    direction_map = {
        "BULLISH_FVG": "BULLISH",
        "BEARISH_FVG": "BEARISH",
    }

    fvg["fvg_direction"] = fvg["fvg_type"].map(direction_map)

    if fvg["fvg_direction"].isna().any():
        raise ValueError(
            "Unable to derive FVG direction from fvg_type."
        )

    # --------------------------------------------------------
    # 7. Validate FVG identity
    # --------------------------------------------------------
    if fvg["fvg_id"].isna().any():
        raise ValueError("FVG registry contains null fvg_id.")

    if fvg["fvg_id"].duplicated().any():
        raise ValueError("FVG registry contains duplicate fvg_id.")

    # --------------------------------------------------------
    # 8. Canonical context rows
    # --------------------------------------------------------
    context = fvg[
        [
            "fvg_id",
            "fvg_type",
            "fvg_direction",
            "created_at",
            "lower_bound",
            "upper_bound",
            "gap_size",
            "candle_a",
            "candle_b",
            "candle_c",
            "lifecycle_state",
            "event_type",
            "event_time",
            "event_price",
        ]
    ].copy()

    # --------------------------------------------------------
    # 9. Default context fields
    # --------------------------------------------------------
    context["active_bias"] = "NEUTRAL"
    context["bias_alignment"] = "UNALIGNED"

    # --------------------------------------------------------
    # 10. Deterministic structure-state alignment
    # --------------------------------------------------------
    #
    # We intentionally do not use future information.
    # For each FVG, only the latest structure state at or
    # before FVG creation is eligible.
    #
    # The exact structure-state vocabulary is preserved rather
    # than invented here.
    # --------------------------------------------------------

    if not state.empty:
        state_sorted = state.sort_values("time").copy()

        context = pd.merge_asof(
            context.sort_values("created_at"),
            state_sorted,
            left_on="created_at",
            right_on="time",
            direction="backward",
        )

        if "bias" in context.columns:
            context["active_bias"] = (
                context["bias"]
                .fillna("NEUTRAL")
                .astype(str)
            )

        elif "market_bias" in context.columns:
            context["active_bias"] = (
                context["market_bias"]
                .fillna("NEUTRAL")
                .astype(str)
            )

        elif "structure_bias" in context.columns:
            context["active_bias"] = (
                context["structure_bias"]
                .fillna("NEUTRAL")
                .astype(str)
            )

        # Remove merge helper only if present.
        if "time" in context.columns:
            context = context.drop(columns=["time"])

    # --------------------------------------------------------
    # 11. Normalize bias vocabulary
    # --------------------------------------------------------
    def normalize_bias(value):
        if pd.isna(value):
            return "NEUTRAL"

        value = str(value).upper()

        if "BULL" in value:
            return "BULLISH"

        if "BEAR" in value:
            return "BEARISH"

        if value in {"LONG", "BUY"}:
            return "BULLISH"

        if value in {"SHORT", "SELL"}:
            return "BEARISH"

        if value in {"NEUTRAL", "NONE", "UNKNOWN"}:
            return "NEUTRAL"

        return value

    context["active_bias"] = (
        context["active_bias"]
        .map(normalize_bias)
    )

    # --------------------------------------------------------
    # 12. FVG ↔ Bias alignment
    # --------------------------------------------------------
    context["bias_alignment"] = np.select(
        [
            (
                (context["fvg_direction"] == "BULLISH")
                & (context["active_bias"] == "BULLISH")
            ),
            (
                (context["fvg_direction"] == "BEARISH")
                & (context["active_bias"] == "BEARISH")
            ),
            (
                (context["active_bias"] == "NEUTRAL")
            ),
        ],
        [
            "ALIGNED",
            "ALIGNED",
            "NEUTRAL",
        ],
        default="CONTRADICTED",
    )

    # --------------------------------------------------------
    # 13. Final deterministic ordering
    # --------------------------------------------------------
    context = context.sort_values(
        ["created_at", "fvg_id"],
        kind="stable",
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # 14. Final uniqueness contract
    # --------------------------------------------------------
    if context["fvg_id"].duplicated().any():
        raise ValueError(
            "FVG Context Engine produced duplicate fvg_id rows."
        )

    # --------------------------------------------------------
    # 15. Final sanity
    # --------------------------------------------------------
    if len(context) != len(fvg_registry):
        raise ValueError(
            "FVG Context Engine changed FVG row count."
        )

    return context


print("=" * 100)
print("FVG CONTEXT ENGINE V1 PATCH")
print("=" * 100)
print("Patch loaded successfully.")
print()
print("Frozen FVG Registry columns:")
print(fvg_lifecycle_registry_v1.columns.tolist())

FVG CONTEXT ENGINE V1 PATCH
Patch loaded successfully.

Frozen FVG Registry columns:
['fvg_id', 'fvg_type', 'created_at', 'lower_bound', 'upper_bound', 'gap_size', 'candle_a', 'candle_b', 'candle_c', 'lifecycle_state', 'event_type', 'event_time', 'event_price']


In [143]:
# ============================================================
# FVG CONTEXT ENGINE V1 — INPUT SCHEMA DIAGNOSTIC
# READ-ONLY — NO DATA MODIFICATION
# ============================================================

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — INPUT SCHEMA DIAGNOSTIC")
print("=" * 100)

inputs = {
    "FVG LIFECYCLE REGISTRY V1": fvg_lifecycle_registry_v1,
    "UNIFIED LIQUIDITY REGISTRY V1": unified_liquidity_registry_v1,
    "STRUCTURAL BOS V2": structural_bos_v2,
    "STRUCTURAL CHOCH V2": structural_choch_v2,
    "STRUCTURE STATE V2": structure_state_v2,
}

for name, df in inputs.items():
    print()
    print("-" * 100)
    print(name)
    print("-" * 100)
    print("Type    :", type(df).__name__)
    print("Rows    :", len(df))
    print("Columns :", len(df.columns))
    print("Schema  :")
    
    for i, col in enumerate(df.columns, 1):
        print(f"  {i:02d}. {col:<35} dtype={df[col].dtype}")

    print()
    print("First row:")
    print(df.head(1).to_string(index=False))

FVG CONTEXT ENGINE V1 — INPUT SCHEMA DIAGNOSTIC

----------------------------------------------------------------------------------------------------
FVG LIFECYCLE REGISTRY V1
----------------------------------------------------------------------------------------------------
Type    : DataFrame
Rows    : 99
Columns : 13
Schema  :
  01. fvg_id                              dtype=str
  02. fvg_type                            dtype=str
  03. created_at                          dtype=datetime64[s, UTC]
  04. lower_bound                         dtype=float64
  05. upper_bound                         dtype=float64
  06. gap_size                            dtype=float64
  07. candle_a                            dtype=datetime64[s, UTC]
  08. candle_b                            dtype=datetime64[s, UTC]
  09. candle_c                            dtype=datetime64[s, UTC]
  10. lifecycle_state                     dtype=str
  11. event_type                          dtype=str
  12. event_time       

In [144]:
# ============================================================
# INSPECT CURRENT FVG CONTEXT ENGINE BUILDER
# READ-ONLY
# ============================================================

import inspect

print("=" * 100)
print("CURRENT FVG CONTEXT ENGINE V1 BUILDER")
print("=" * 100)

print(
    inspect.getsource(build_fvg_context_engine_v1_patched)
)

CURRENT FVG CONTEXT ENGINE V1 BUILDER
def build_fvg_context_engine_v1_patched(
    fvg_registry,
    liquidity_registry,
    structural_bos,
    structural_choch,
    structure_state,
):
    """
    FVG Context Engine V1
    Compatibility-patched against the FROZEN
    FVG Lifecycle Registry V1 schema.

    IMPORTANT:
    - Does NOT require event_direction from FVG registry.
    - Direction is deterministically derived from fvg_type.
    - Frozen upstream schemas remain unchanged.
    """

    import pandas as pd
    import numpy as np

    # --------------------------------------------------------
    # 1. Defensive copies
    # --------------------------------------------------------
    fvg = fvg_registry.copy(deep=True)
    liq = liquidity_registry.copy(deep=True)
    bos = structural_bos.copy(deep=True)
    choch = structural_choch.copy(deep=True)
    state = structure_state.copy(deep=True)

    # --------------------------------------------------------
    # 2. Frozen FVG Registr

In [145]:
# ============================================================
# FVG CONTEXT ENGINE V1 — STRUCTURE / CONTRACT DIAGNOSTIC
# READ-ONLY
# ============================================================

import pandas as pd

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — STRUCTURE / CONTRACT DIAGNOSTIC")
print("=" * 100)

# ------------------------------------------------------------
# 1. Structure State index
# ------------------------------------------------------------
print()
print("-" * 100)
print("STRUCTURE STATE V2 INDEX")
print("-" * 100)

print("Index type :", type(structure_state_v2.index).__name__)
print("Index name :", structure_state_v2.index.name)
print("Index dtype:", structure_state_v2.index.dtype)

print()
print("First 5 index values:")
print(structure_state_v2.index[:5])

print()
print("Last 5 index values:")
print(structure_state_v2.index[-5:])

# ------------------------------------------------------------
# 2. Existing FVG Context-related objects
# ------------------------------------------------------------
print()
print("-" * 100)
print("EXISTING FVG CONTEXT OBJECTS")
print("-" * 100)

context_names = sorted(
    [
        name
        for name in globals()
        if "FVG_CONTEXT" in name.upper()
        or "fvg_context" in name.lower()
    ]
)

if context_names:
    for name in context_names:
        obj = globals()[name]
        print(f"{name:<45} -> {type(obj).__name__}")
        if isinstance(obj, pd.DataFrame):
            print(
                f"    shape={obj.shape}, "
                f"columns={obj.columns.tolist()}"
            )
        elif isinstance(obj, (set, list, tuple, dict)):
            print(f"    value={obj}")
else:
    print("No existing FVG Context objects found.")

# ------------------------------------------------------------
# 3. Existing FVG Context contract variables
# ------------------------------------------------------------
print()
print("-" * 100)
print("POSSIBLE FVG CONTEXT CONTRACT VARIABLES")
print("-" * 100)

contract_names = sorted(
    [
        name
        for name in globals()
        if (
            "FVG_CONTEXT" in name.upper()
            and (
                "ALLOWED" in name.upper()
                or "COLUMN" in name.upper()
                or "CONTRACT" in name.upper()
                or "SCHEMA" in name.upper()
            )
        )
    ]
)

if contract_names:
    for name in contract_names:
        print(f"{name} = {globals()[name]}")
else:
    print("No explicit FVG Context contract variables found.")

FVG CONTEXT ENGINE V1 — STRUCTURE / CONTRACT DIAGNOSTIC

----------------------------------------------------------------------------------------------------
STRUCTURE STATE V2 INDEX
----------------------------------------------------------------------------------------------------
Index type : DatetimeIndex
Index name : timestamp
Index dtype: datetime64[s, UTC]

First 5 index values:
DatetimeIndex(['2026-10-02 19:30:00+00:00', '2026-10-02 19:35:00+00:00',
               '2026-10-02 19:40:00+00:00', '2026-10-02 19:45:00+00:00',
               '2026-10-02 19:50:00+00:00'],
              dtype='datetime64[s, UTC]', name='timestamp', freq=None)

Last 5 index values:
DatetimeIndex(['2026-10-08 07:45:00+00:00', '2026-10-08 07:50:00+00:00',
               '2026-10-08 07:55:00+00:00', '2026-10-08 08:00:00+00:00',
               '2026-10-08 08:05:00+00:00'],
              dtype='datetime64[s, UTC]', name='timestamp', freq=None)

----------------------------------------------------------------

In [146]:
# ============================================================
# INSPECT ORIGINAL FVG CONTEXT ENGINE V1 BUILDER
# READ-ONLY
# ============================================================

import inspect

print("=" * 100)
print("ORIGINAL FVG CONTEXT ENGINE V1 BUILDER")
print("=" * 100)

print(
    inspect.getsource(build_fvg_context_engine_v1)
)

ORIGINAL FVG CONTEXT ENGINE V1 BUILDER
def build_fvg_context_engine_v1(
    fvg_registry,
    liquidity_registry,
    structural_bos,
    structural_choch,
    structure_state,
):
    """
    Build point-in-time structural/liquidity context for each FVG.

    IMPORTANT:
    - Does NOT modify FVG identity.
    - Does NOT modify FVG lifecycle.
    - Does NOT assign liquidity IDs to FVGs.
    - Does NOT merge FVGs into the liquidity registry.
    - Context is evaluated at FVG created_at.
    """

    # --------------------------------------------------------
    # Validate inputs
    # --------------------------------------------------------

    _require_columns(
        fvg_registry,
        FVG_CONTEXT_ENGINE_V1_REQUIRED_FVG_COLUMNS,
        "fvg_registry"
    )

    _require_columns(
        liquidity_registry,
        FVG_CONTEXT_ENGINE_V1_REQUIRED_LIQUIDITY_COLUMNS,
        "liquidity_registry"
    )

    _require_columns(
        structural_bos,
        FVG_CONTEXT_ENGINE_V1_REQUIR

In [147]:
# ============================================================
# BUILD FVG CONTEXT ENGINE V1
# CANONICAL BUILDER — FROZEN UPSTREAM INPUTS
# ============================================================

fvg_context_engine_v1_new = build_fvg_context_engine_v1(
    fvg_registry=fvg_lifecycle_registry_v1,
    liquidity_registry=unified_liquidity_registry_v1,
    structural_bos=structural_bos_v2,
    structural_choch=structural_choch_v2,
    structure_state=structure_state_v2,
)

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — CANONICAL BUILD")
print("=" * 100)

print("Rows    :", len(fvg_context_engine_v1_new))
print("Columns :", len(fvg_context_engine_v1_new.columns))

print()
print("COLUMNS")
print(fvg_context_engine_v1_new.columns.tolist())

print()
print("BIAS")
print(
    fvg_context_engine_v1_new["active_bias"]
    .value_counts(dropna=False)
)

print()
print("BIAS ALIGNMENT")
print(
    fvg_context_engine_v1_new["bias_alignment"]
    .value_counts(dropna=False)
)

print()
print("BOS ALIGNMENT")
print(
    fvg_context_engine_v1_new["bos_alignment"]
    .value_counts(dropna=False)
)

print()
print("CHOCH ALIGNMENT")
print(
    fvg_context_engine_v1_new["choch_alignment"]
    .value_counts(dropna=False)
)

print()
print("LIQUIDITY OVERLAP")
print(
    fvg_context_engine_v1_new[
        "overlapping_liquidity_ids"
    ]
    .apply(len)
    .value_counts()
    .sort_index()
)

print()
print("HEAD")
print(
    fvg_context_engine_v1_new.head(10).to_string(index=False)
)

FVG CONTEXT ENGINE V1 — CANONICAL BUILD
Rows    : 99
Columns : 37

COLUMNS
['fvg_id', 'fvg_type', 'created_at', 'lower_bound', 'upper_bound', 'gap_size', 'lifecycle_state', 'active_bias', 'bias_alignment', 'last_bos', 'last_bos_direction', 'last_bos_level', 'last_bos_level_id', 'last_choch', 'last_choch_direction', 'last_choch_level', 'last_choch_level_id', 'context_bos_id', 'context_bos_level_id', 'context_bos_direction', 'context_bos_level', 'context_bos_time', 'bos_alignment', 'context_choch_id', 'context_choch_level_id', 'context_choch_direction', 'context_choch_level', 'context_choch_time', 'choch_alignment', 'overlapping_liquidity_ids', 'overlapping_liquidity_types', 'nearest_liquidity_below_id', 'nearest_liquidity_below_type', 'nearest_liquidity_below_price', 'nearest_liquidity_above_id', 'nearest_liquidity_above_type', 'nearest_liquidity_above_price']

BIAS
active_bias
BEARISH    53
BULLISH    44
UNKNOWN     2
Name: count, dtype: int64

BIAS ALIGNMENT
bias_alignment
ALIGNED    

In [148]:
# ============================================================
# FVG CONTEXT ENGINE V1
# EXACT COMPARISON — CANONICAL vs EXISTING
# READ-ONLY AUDIT
# ============================================================

import pandas as pd
import numpy as np

canonical = fvg_context_engine_v1_new.copy(deep=True)
existing = fvg_context_engine_v1.copy(deep=True)

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — EXACT COMPARISON")
print("=" * 100)

# ------------------------------------------------------------
# 1. BASIC SHAPE
# ------------------------------------------------------------

print("\nBASIC SHAPE")
print("-" * 100)

print("Canonical shape :", canonical.shape)
print("Existing shape  :", existing.shape)

print(
    "ROW COUNT       :",
    "PASSED" if len(canonical) == len(existing) else "FAILED"
)

print(
    "COLUMN COUNT    :",
    "PASSED" if len(canonical.columns) == len(existing.columns) else "FAILED"
)

print(
    "COLUMN ORDER    :",
    "PASSED" if canonical.columns.tolist() == existing.columns.tolist()
    else "FAILED"
)

# ------------------------------------------------------------
# 2. FVG ID SET / ORDER
# ------------------------------------------------------------

print("\nFVG ID CONTRACT")
print("-" * 100)

canonical_ids = canonical["fvg_id"].tolist()
existing_ids = existing["fvg_id"].tolist()

print(
    "ID SET          :",
    "PASSED"
    if set(canonical_ids) == set(existing_ids)
    else "FAILED"
)

print(
    "ID ORDER        :",
    "PASSED"
    if canonical_ids == existing_ids
    else "FAILED"
)

# ------------------------------------------------------------
# 3. CELL-BY-CELL COMPARISON
# ------------------------------------------------------------

print("\nCELL-BY-CELL COMPARISON")
print("-" * 100)

if canonical.columns.tolist() == existing.columns.tolist():

    canonical_cmp = canonical.reset_index(drop=True)
    existing_cmp = existing.reset_index(drop=True)

    diff_records = []

    for col in canonical_cmp.columns:

        a = canonical_cmp[col]
        b = existing_cmp[col]

        for i in range(len(canonical_cmp)):

            av = a.iloc[i]
            bv = b.iloc[i]

            # Both NaN / NaT
            if pd.isna(av) and pd.isna(bv):
                continue

            # Tuple/list equality
            if isinstance(av, (tuple, list)) or isinstance(bv, (tuple, list)):
                equal = av == bv
            else:
                try:
                    equal = bool(np.isclose(av, bv, equal_nan=True))
                except (TypeError, ValueError):
                    equal = av == bv

            if not equal:
                diff_records.append(
                    {
                        "row": i,
                        "fvg_id": canonical_cmp.iloc[i]["fvg_id"],
                        "column": col,
                        "canonical": av,
                        "existing": bv,
                    }
                )

    diff_df = pd.DataFrame(diff_records)

else:
    diff_df = pd.DataFrame(
        {
            "row": [],
            "fvg_id": [],
            "column": [],
            "canonical": [],
            "existing": [],
        }
    )

print(
    "CELL DIFFERENCES :",
    len(diff_df)
)

print(
    "EXACT OUTPUT     :",
    "PASSED" if len(diff_df) == 0 else "FAILED"
)

# ------------------------------------------------------------
# 4. SHOW DIFFERENCES IF ANY
# ------------------------------------------------------------

if len(diff_df) > 0:

    print("\nDIFFERENCES")
    print("-" * 100)

    print(
        diff_df.head(50).to_string(index=False)
    )

else:

    print("\nNO DIFFERENCES FOUND")
    print("-" * 100)

# ------------------------------------------------------------
# 5. DTYPE COMPARISON
# ------------------------------------------------------------

print("\nDTYPE COMPARISON")
print("-" * 100)

dtype_diff = []

for col in canonical.columns:

    c_dtype = str(canonical[col].dtype)
    e_dtype = str(existing[col].dtype)

    if c_dtype != e_dtype:
        dtype_diff.append(
            {
                "column": col,
                "canonical_dtype": c_dtype,
                "existing_dtype": e_dtype,
            }
        )

dtype_diff_df = pd.DataFrame(dtype_diff)

print(
    "DTYPE DIFFERENCES :",
    len(dtype_diff_df)
)

if len(dtype_diff_df) > 0:
    print(dtype_diff_df.to_string(index=False))
else:
    print("DTYPES           : PASSED")

# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

print("\n" + "=" * 100)

if (
    canonical.shape == existing.shape
    and canonical.columns.tolist() == existing.columns.tolist()
    and set(canonical_ids) == set(existing_ids)
    and len(diff_df) == 0
):
    print("FVG CONTEXT ENGINE V1 EXACT COMPARISON : PASSED")
else:
    print("FVG CONTEXT ENGINE V1 EXACT COMPARISON : FAILED")

print("=" * 100)

FVG CONTEXT ENGINE V1 — EXACT COMPARISON

BASIC SHAPE
----------------------------------------------------------------------------------------------------
Canonical shape : (99, 37)
Existing shape  : (99, 37)
ROW COUNT       : PASSED
COLUMN COUNT    : PASSED
COLUMN ORDER    : PASSED

FVG ID CONTRACT
----------------------------------------------------------------------------------------------------
ID SET          : PASSED
ID ORDER        : PASSED

CELL-BY-CELL COMPARISON
----------------------------------------------------------------------------------------------------
CELL DIFFERENCES : 0
EXACT OUTPUT     : PASSED

NO DIFFERENCES FOUND
----------------------------------------------------------------------------------------------------

DTYPE COMPARISON
----------------------------------------------------------------------------------------------------
DTYPE DIFFERENCES : 0
DTYPES           : PASSED

FVG CONTEXT ENGINE V1 EXACT COMPARISON : PASSED


In [149]:
# ============================================================
# FVG CONTEXT ENGINE V1
# CONTRACT AUDIT V1
# READ-ONLY / NO MUTATION
# ============================================================

import pandas as pd
import numpy as np

df = fvg_context_engine_v1.copy(deep=True)

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — CONTRACT AUDIT")
print("=" * 100)

audit = []


def add_check(name, condition, detail=""):
    status = "PASSED" if bool(condition) else "FAILED"
    audit.append(
        {
            "check": name,
            "status": status,
            "detail": detail,
        }
    )
    print(f"{name:<55} : {status}")
    if detail:
        print(f"{'':<55}   {detail}")


# ============================================================
# 1. CANONICAL SCHEMA
# ============================================================

EXPECTED_COLUMNS = [
    "fvg_id",
    "fvg_type",
    "created_at",
    "lower_bound",
    "upper_bound",
    "gap_size",
    "lifecycle_state",
    "active_bias",
    "bias_alignment",
    "last_bos",
    "last_bos_direction",
    "last_bos_level",
    "last_bos_level_id",
    "last_choch",
    "last_choch_direction",
    "last_choch_level",
    "last_choch_level_id",
    "context_bos_id",
    "context_bos_level_id",
    "context_bos_direction",
    "context_bos_level",
    "context_bos_time",
    "bos_alignment",
    "context_choch_id",
    "context_choch_level_id",
    "context_choch_direction",
    "context_choch_level",
    "context_choch_time",
    "choch_alignment",
    "overlapping_liquidity_ids",
    "overlapping_liquidity_types",
    "nearest_liquidity_below_id",
    "nearest_liquidity_below_type",
    "nearest_liquidity_below_price",
    "nearest_liquidity_above_id",
    "nearest_liquidity_above_type",
    "nearest_liquidity_above_price",
]

add_check(
    "SCHEMA",
    df.columns.tolist() == EXPECTED_COLUMNS,
    f"{len(df.columns)} columns"
)


# ============================================================
# 2. ROW COUNT
# ============================================================

add_check(
    "ROW COUNT",
    len(df) == len(fvg_lifecycle_registry_v1),
    f"Context={len(df)} / FVG Registry={len(fvg_lifecycle_registry_v1)}"
)


# ============================================================
# 3. FVG ID UNIQUENESS
# ============================================================

add_check(
    "FVG ID UNIQUENESS",
    df["fvg_id"].notna().all()
    and df["fvg_id"].is_unique,
    f"unique={df['fvg_id'].nunique()} / rows={len(df)}"
)


# ============================================================
# 4. FVG ID PRESERVATION
# ============================================================

registry_ids = set(fvg_lifecycle_registry_v1["fvg_id"])
context_ids = set(df["fvg_id"])

add_check(
    "FVG ID PRESERVATION",
    context_ids == registry_ids,
    f"Context IDs={len(context_ids)} / Registry IDs={len(registry_ids)}"
)


# ============================================================
# 5. FVG TYPE VOCABULARY
# ============================================================

allowed_fvg_types = {
    "BULLISH_FVG",
    "BEARISH_FVG",
}

fvg_types_valid = (
    df["fvg_type"].notna().all()
    and df["fvg_type"].isin(allowed_fvg_types).all()
)

add_check(
    "FVG TYPE VOCABULARY",
    fvg_types_valid,
    f"values={sorted(df['fvg_type'].dropna().unique().tolist())}"
)


# ============================================================
# 6. LIFECYCLE VOCABULARY
# ============================================================

allowed_lifecycle_states = {
    "FRESH",
    "MITIGATED",
    "FILLED",
}

lifecycle_valid = (
    df["lifecycle_state"].notna().all()
    and df["lifecycle_state"].isin(allowed_lifecycle_states).all()
)

add_check(
    "LIFECYCLE STATE VOCABULARY",
    lifecycle_valid,
    f"values={sorted(df['lifecycle_state'].dropna().unique().tolist())}"
)


# ============================================================
# 7. UTC CONTRACT
# ============================================================

created_utc = (
    pd.api.types.is_datetime64tz_dtype(df["created_at"])
    and str(df["created_at"].dt.tz) == "UTC"
)

bos_time_utc = (
    pd.api.types.is_datetime64tz_dtype(df["context_bos_time"])
    and str(df["context_bos_time"].dt.tz) == "UTC"
)

choch_time_utc = (
    pd.api.types.is_datetime64tz_dtype(df["context_choch_time"])
    and str(df["context_choch_time"].dt.tz) == "UTC"
)

utc_ok = created_utc and bos_time_utc and choch_time_utc

add_check(
    "UTC TIMESTAMP CONTRACT",
    utc_ok,
    f"created={created_utc}, bos={bos_time_utc}, choch={choch_time_utc}"
)


# ============================================================
# 8. CHRONOLOGICAL ORDER
# ============================================================

chronological = df["created_at"].is_monotonic_increasing

add_check(
    "CHRONOLOGICAL FVG ORDER",
    chronological
)


# ============================================================
# 9. FVG GEOMETRY
# ============================================================

geometry_valid = (
    df["lower_bound"].notna().all()
    and df["upper_bound"].notna().all()
    and (df["lower_bound"] < df["upper_bound"]).all()
    and df["gap_size"].notna().all()
    and np.allclose(
        df["gap_size"].to_numpy(dtype=float),
        (
            df["upper_bound"] - df["lower_bound"]
        ).to_numpy(dtype=float),
        equal_nan=False,
    )
)

add_check(
    "FVG GEOMETRY CONTRACT",
    geometry_valid
)


# ============================================================
# 10. BIAS VOCABULARY
# ============================================================

allowed_bias = {
    "BULLISH",
    "BEARISH",
    "UNKNOWN",
}

bias_valid = df["active_bias"].isin(allowed_bias).all()

add_check(
    "ACTIVE BIAS VOCABULARY",
    bias_valid,
    f"values={sorted(df['active_bias'].dropna().unique().tolist())}"
)


# ============================================================
# 11. BIAS ALIGNMENT
# ============================================================

allowed_alignment = {
    "ALIGNED",
    "OPPOSED",
    "UNKNOWN",
}

bias_alignment_valid = (
    df["bias_alignment"].isin(allowed_alignment).all()
)

add_check(
    "BIAS ALIGNMENT VOCABULARY",
    bias_alignment_valid
)


# ============================================================
# 12. BIAS ALIGNMENT SEMANTICS
# ============================================================

def expected_bias_alignment(row):
    if row["active_bias"] == "UNKNOWN":
        return "UNKNOWN"

    direction = (
        "BULLISH"
        if row["fvg_type"] == "BULLISH_FVG"
        else "BEARISH"
    )

    return (
        "ALIGNED"
        if row["active_bias"] == direction
        else "OPPOSED"
    )


expected_bias_alignment_values = df.apply(
    expected_bias_alignment,
    axis=1,
)

bias_semantics_valid = (
    expected_bias_alignment_values
    == df["bias_alignment"]
).all()

add_check(
    "BIAS ALIGNMENT SEMANTICS",
    bias_semantics_valid
)


# ============================================================
# 13. BOS ALIGNMENT VOCABULARY
# ============================================================

bos_alignment_valid = (
    df["bos_alignment"].isin(allowed_alignment).all()
)

add_check(
    "BOS ALIGNMENT VOCABULARY",
    bos_alignment_valid
)


# ============================================================
# 14. CHOCH ALIGNMENT VOCABULARY
# ============================================================

choch_alignment_valid = (
    df["choch_alignment"].isin(allowed_alignment).all()
)

add_check(
    "CHOCH ALIGNMENT VOCABULARY",
    choch_alignment_valid
)


# ============================================================
# 15. BOS TEMPORAL CAUSALITY
# ============================================================

bos_causality = (
    df["context_bos_time"].isna()
    | (df["context_bos_time"] <= df["created_at"])
)

add_check(
    "BOS NO LOOK-AHEAD",
    bos_causality.all()
)


# ============================================================
# 16. CHOCH TEMPORAL CAUSALITY
# ============================================================

choch_causality = (
    df["context_choch_time"].isna()
    | (df["context_choch_time"] <= df["created_at"])
)

add_check(
    "CHOCH NO LOOK-AHEAD",
    choch_causality.all()
)


# ============================================================
# 17. STATE CAUSALITY — CONTEXT FIELDS
# ============================================================

state_causal = (
    df["context_bos_time"].isna()
    | (df["context_bos_time"] <= df["created_at"])
)

add_check(
    "CONTEXT CAUSALITY",
    state_causal.all()
)


# ============================================================
# 18. BOS ID / LEVEL ID CONSISTENCY
# ============================================================

bos_rows = df["context_bos_id"].notna()

bos_consistency = (
    (~bos_rows)
    | (
        df.loc[bos_rows, "context_bos_level_id"].notna()
        & df.loc[bos_rows, "context_bos_direction"].notna()
        & df.loc[bos_rows, "context_bos_level"].notna()
        & df.loc[bos_rows, "context_bos_time"].notna()
    )
)

add_check(
    "BOS CONTEXT CONSISTENCY",
    bos_consistency.all()
)


# ============================================================
# 19. CHOCH ID / LEVEL ID CONSISTENCY
# ============================================================

choch_rows = df["context_choch_id"].notna()

choch_consistency = (
    (~choch_rows)
    | (
        df.loc[choch_rows, "context_choch_level_id"].notna()
        & df.loc[choch_rows, "context_choch_direction"].notna()
        & df.loc[choch_rows, "context_choch_level"].notna()
        & df.loc[choch_rows, "context_choch_time"].notna()
    )
)

add_check(
    "CHOCH CONTEXT CONSISTENCY",
    choch_consistency.all()
)


# ============================================================
# 20. LIQUIDITY TUPLE CONTRACT
# ============================================================

liquidity_tuple_contract = (
    df["overlapping_liquidity_ids"].apply(
        lambda x: isinstance(x, tuple)
    ).all()
    and
    df["overlapping_liquidity_types"].apply(
        lambda x: isinstance(x, tuple)
    ).all()
)

add_check(
    "LIQUIDITY OVERLAP TUPLE CONTRACT",
    liquidity_tuple_contract
)


# ============================================================
# 21. LIQUIDITY OVERLAP LENGTH
# ============================================================

overlap_length_valid = (
    df.apply(
        lambda r:
        len(r["overlapping_liquidity_ids"])
        == len(r["overlapping_liquidity_types"]),
        axis=1,
    )
).all()

add_check(
    "LIQUIDITY OVERLAP LENGTH",
    overlap_length_valid
)


# ============================================================
# 22. NEAREST LIQUIDITY BELOW GEOMETRY
# ============================================================

below_rows = df["nearest_liquidity_below_id"].notna()

below_geometry = (
    (~below_rows)
    |
    (
        df.loc[below_rows, "nearest_liquidity_below_price"]
        < df.loc[below_rows, "lower_bound"]
    )
)

add_check(
    "NEAREST BELOW GEOMETRY",
    below_geometry.all()
)


# ============================================================
# 23. NEAREST LIQUIDITY ABOVE GEOMETRY
# ============================================================

above_rows = df["nearest_liquidity_above_id"].notna()

above_geometry = (
    (~above_rows)
    |
    (
        df.loc[above_rows, "nearest_liquidity_above_price"]
        > df.loc[above_rows, "upper_bound"]
    )
)

add_check(
    "NEAREST ABOVE GEOMETRY",
    above_geometry.all()
)


# ============================================================
# 24. BELOW/ABOVE PRICE NULL CONTRACT
# ============================================================

below_null_contract = (
    df.loc[~below_rows, "nearest_liquidity_below_price"]
    .isna()
    .all()
)

above_null_contract = (
    df.loc[~above_rows, "nearest_liquidity_above_price"]
    .isna()
    .all()
)

add_check(
    "NEAREST LIQUIDITY NULL CONTRACT",
    below_null_contract and above_null_contract
)


# ============================================================
# 25. FINAL RESULT
# ============================================================

audit_df = pd.DataFrame(audit)

failed = audit_df[audit_df["status"] == "FAILED"]

print("\n" + "=" * 100)
print("FINAL CONTRACT AUDIT RESULT")
print("=" * 100)

print(
    "TOTAL CHECKS :",
    len(audit_df)
)

print(
    "PASSED       :",
    (audit_df["status"] == "PASSED").sum()
)

print(
    "FAILED       :",
    len(failed)
)

print()

if len(failed) == 0:
    print("FVG CONTEXT ENGINE V1 CONTRACT AUDIT : PASSED")
else:
    print("FVG CONTEXT ENGINE V1 CONTRACT AUDIT : FAILED")
    print("\nFAILED CHECKS:")
    print(failed.to_string(index=False))

print("=" * 100)

FVG CONTEXT ENGINE V1 — CONTRACT AUDIT
SCHEMA                                                  : PASSED
                                                          37 columns
ROW COUNT                                               : PASSED
                                                          Context=99 / FVG Registry=99
FVG ID UNIQUENESS                                       : PASSED
                                                          unique=99 / rows=99
FVG ID PRESERVATION                                     : PASSED
                                                          Context IDs=99 / Registry IDs=99
FVG TYPE VOCABULARY                                     : PASSED
                                                          values=['BEARISH_FVG', 'BULLISH_FVG']
LIFECYCLE STATE VOCABULARY                              : PASSED
                                                          values=['FILLED', 'FRESH', 'MITIGATED']
UTC TIMESTAMP CONTRACT                              

/tmp/ipykernel_10569/2172728867.py:169: Pandas4Warning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  pd.api.types.is_datetime64tz_dtype(df["created_at"])
/tmp/ipykernel_10569/2172728867.py:174: Pandas4Warning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  pd.api.types.is_datetime64tz_dtype(df["context_bos_time"])
/tmp/ipykernel_10569/2172728867.py:179: Pandas4Warning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  pd.api.types.is_datetime64tz_dtype(df["context_choch_time"])


In [150]:
# ============================================================
# FVG CONTEXT ENGINE V1
# BIAS ALIGNMENT FAILURE DIAGNOSTIC
# READ-ONLY
# ============================================================

df = fvg_context_engine_v1.copy(deep=True)

print("=" * 100)
print("BIAS ALIGNMENT SEMANTICS — FAILURE DIAGNOSTIC")
print("=" * 100)

mask = (
    df["active_bias"].isna()
    | df["active_bias"].astype(str).eq("UNKNOWN")
)

diagnostic = df.loc[
    mask,
    [
        "fvg_id",
        "fvg_type",
        "active_bias",
        "bias_alignment",
    ]
].copy()

print("\nAFFECTED ROWS")
print("-" * 100)
print(diagnostic.to_string(index=False))

print("\nVALUE TYPES")
print("-" * 100)

for idx, row in diagnostic.iterrows():
    print(
        f"Index={idx} | "
        f"active_bias={row['active_bias']!r} | "
        f"type={type(row['active_bias']).__name__} | "
        f"bias_alignment={row['bias_alignment']!r}"
    )

print("\nRAW UNIQUE VALUES")
print("-" * 100)

for value in df["active_bias"].unique():
    print(
        repr(value),
        "->",
        type(value).__name__
    )

print("=" * 100)

BIAS ALIGNMENT SEMANTICS — FAILURE DIAGNOSTIC

AFFECTED ROWS
----------------------------------------------------------------------------------------------------
                fvg_id    fvg_type active_bias bias_alignment
FVG_B_20261002T221500Z BULLISH_FVG     UNKNOWN        UNKNOWN
FVG_B_20261002T222500Z BULLISH_FVG     UNKNOWN        UNKNOWN

VALUE TYPES
----------------------------------------------------------------------------------------------------
Index=0 | active_bias='UNKNOWN' | type=str | bias_alignment='UNKNOWN'
Index=1 | active_bias='UNKNOWN' | type=str | bias_alignment='UNKNOWN'

RAW UNIQUE VALUES
----------------------------------------------------------------------------------------------------
'UNKNOWN' -> str
'BULLISH' -> str
'BEARISH' -> str


In [151]:
# ============================================================
# FVG CONTEXT ENGINE V1
# TARGET FVG ID — REGISTRY EXISTENCE DIAGNOSTIC
# READ-ONLY
# ============================================================

target_id = "FVG_B_20261002T101500Z"

print("=" * 100)
print("FVG TARGET ID — REGISTRY EXISTENCE DIAGNOSTIC")
print("=" * 100)

print("\nTARGET")
print("-" * 100)
print("target_id :", repr(target_id))

# ------------------------------------------------------------
# Registry basic information
# ------------------------------------------------------------

print("\nLIFECYCLE REGISTRY")
print("-" * 100)

print("rows       :", len(fvg_lifecycle_registry_v1))
print("columns    :", list(fvg_lifecycle_registry_v1.columns))

if "fvg_id" not in fvg_lifecycle_registry_v1.columns:
    print("ERROR      : fvg_id column does not exist.")
else:
    mask = (
        fvg_lifecycle_registry_v1["fvg_id"].astype(str)
        == target_id
    )

    print("target rows:", int(mask.sum()))

    if mask.any():
        print("\nTARGET FOUND")
        print(fvg_lifecycle_registry_v1.loc[mask].to_string())
    else:
        print("\nTARGET NOT FOUND")

        print("\nFIRST 20 FVG IDs")
        print(
            fvg_lifecycle_registry_v1["fvg_id"]
            .head(20)
            .tolist()
        )

# ------------------------------------------------------------
# Search by timestamp fragment
# ------------------------------------------------------------

print("\nTIMESTAMP FRAGMENT SEARCH")
print("-" * 100)

if "fvg_id" in fvg_lifecycle_registry_v1.columns:

    fragment = "20261002T101500Z"

    fragment_mask = (
        fvg_lifecycle_registry_v1["fvg_id"]
        .astype(str)
        .str.contains(fragment, regex=False, na=False)
    )

    print("matches:", int(fragment_mask.sum()))

    if fragment_mask.any():
        print(
            fvg_lifecycle_registry_v1.loc[
                fragment_mask,
                ["fvg_id"]
            ].to_string(index=False)
        )
    else:
        print("No FVG ID contains:", fragment)

print("=" * 100)

FVG TARGET ID — REGISTRY EXISTENCE DIAGNOSTIC

TARGET
----------------------------------------------------------------------------------------------------
target_id : 'FVG_B_20261002T101500Z'

LIFECYCLE REGISTRY
----------------------------------------------------------------------------------------------------
rows       : 99
columns    : ['fvg_id', 'fvg_type', 'created_at', 'lower_bound', 'upper_bound', 'gap_size', 'candle_a', 'candle_b', 'candle_c', 'lifecycle_state', 'event_type', 'event_time', 'event_price']
target rows: 0

TARGET NOT FOUND

FIRST 20 FVG IDs
['FVG_B_20261002T221500Z', 'FVG_B_20261002T222500Z', 'FVG_S_20261005T005500Z', 'FVG_S_20261005T031500Z', 'FVG_S_20261005T034500Z', 'FVG_S_20261005T041000Z', 'FVG_S_20261005T051000Z', 'FVG_S_20261005T051500Z', 'FVG_S_20261005T052000Z', 'FVG_S_20261005T055000Z', 'FVG_S_20261005T064000Z', 'FVG_S_20261005T093000Z', 'FVG_S_20261005T093500Z', 'FVG_B_20261005T103500Z', 'FVG_B_20261005T121000Z', 'FVG_S_20261005T123500Z', 'FVG_S_202610

In [152]:
# ============================================================
# FVG CONTEXT ENGINE V1
# TARGET FVG — CROSS-SOURCE ID TRACE
# READ-ONLY
# ============================================================

target_id = "FVG_B_20261002T101500Z"

print("=" * 100)
print("FVG TARGET ID — CROSS-SOURCE ID TRACE")
print("=" * 100)

sources = {
    "fvg_lifecycle_registry_v1": fvg_lifecycle_registry_v1,
    "fvg_context_engine_v1": fvg_context_engine_v1,
}

# Add optional known FVG sources only if they exist
optional_sources = [
    "fvg_validated_v1",
    "fvg_validation_v1",
    "fvg_detection_v1",
    "fvg_detector_v1",
    "fvg_reference_v1",
]

for name in optional_sources:
    if name in globals():
        sources[name] = globals()[name]

for name, df in sources.items():

    print("\n" + "-" * 100)
    print(name)
    print("-" * 100)

    if not isinstance(df, pd.DataFrame):
        print("NOT A DATAFRAME")
        continue

    print("rows    :", len(df))

    if "fvg_id" not in df.columns:
        print("fvg_id  : MISSING")
        continue

    mask = (
        df["fvg_id"].astype(str)
        == target_id
    )

    print("matches :", int(mask.sum()))

    if mask.any():
        print("\nTARGET ROW:")
        print(df.loc[mask].to_string(index=False))

print("\n" + "=" * 100)

FVG TARGET ID — CROSS-SOURCE ID TRACE

----------------------------------------------------------------------------------------------------
fvg_lifecycle_registry_v1
----------------------------------------------------------------------------------------------------
rows    : 99
matches : 0

----------------------------------------------------------------------------------------------------
fvg_context_engine_v1
----------------------------------------------------------------------------------------------------
rows    : 99
matches : 0

----------------------------------------------------------------------------------------------------
fvg_validated_v1
----------------------------------------------------------------------------------------------------
rows    : 179
matches : 0



In [153]:
# ============================================================
# FVG CONTEXT ENGINE V1
# UNKNOWN ACTIVE BIAS — VALID TARGET DISCOVERY
# READ-ONLY
# ============================================================

print("=" * 100)
print("UNKNOWN ACTIVE BIAS — VALID TARGET DISCOVERY")
print("=" * 100)

df = fvg_context_engine_v1

print("\nTOTAL CONTEXT ROWS")
print("-" * 100)
print("rows :", len(df))

# ------------------------------------------------------------
# UNKNOWN active_bias rows
# ------------------------------------------------------------

unknown_mask = (
    df["active_bias"]
    .astype(str)
    .str.upper()
    .eq("UNKNOWN")
)

unknown_df = df.loc[unknown_mask].copy()

print("\nUNKNOWN ACTIVE BIAS")
print("-" * 100)
print("count :", len(unknown_df))

if len(unknown_df) == 0:
    print("No UNKNOWN active_bias rows found.")
else:

    cols = [
        c for c in [
            "fvg_id",
            "fvg_type",
            "created_at",
            "active_bias",
            "bias_alignment",
            "last_bos",
            "last_bos_direction",
            "context_bos_id",
            "context_bos_direction",
        ]
        if c in unknown_df.columns
    ]

    print("\nUNKNOWN ROWS:")
    print(
        unknown_df[cols]
        .head(20)
        .to_string(index=False)
    )

print("\n" + "=" * 100)

UNKNOWN ACTIVE BIAS — VALID TARGET DISCOVERY

TOTAL CONTEXT ROWS
----------------------------------------------------------------------------------------------------
rows : 99

UNKNOWN ACTIVE BIAS
----------------------------------------------------------------------------------------------------
count : 2

UNKNOWN ROWS:
                fvg_id    fvg_type                created_at active_bias bias_alignment last_bos last_bos_direction context_bos_id context_bos_direction
FVG_B_20261002T221500Z BULLISH_FVG 2026-10-02 22:15:00+00:00     UNKNOWN        UNKNOWN      NaN                NaN            NaN                   NaN
FVG_B_20261002T222500Z BULLISH_FVG 2026-10-02 22:25:00+00:00     UNKNOWN        UNKNOWN      NaN                NaN            NaN                   NaN



In [154]:
# ============================================================
# FVG CONTEXT ENGINE V1
# UNKNOWN ACTIVE BIAS — SOURCE DIAGNOSTIC V2
# READ-ONLY
# ============================================================

target_id = "FVG_B_20261002T221500Z"

print("=" * 100)
print("UNKNOWN ACTIVE BIAS — SOURCE DIAGNOSTIC V2")
print("=" * 100)

# ------------------------------------------------------------
# FVG ROW
# ------------------------------------------------------------

fvg_matches = fvg_lifecycle_registry_v1[
    fvg_lifecycle_registry_v1["fvg_id"] == target_id
]

print("\nFVG REGISTRY")
print("-" * 100)
print("matches :", len(fvg_matches))

if len(fvg_matches) == 0:
    raise ValueError(
        f"Target FVG {target_id} not found in lifecycle registry."
    )

fvg_row = fvg_matches.iloc[0]

fvg_time = pd.Timestamp(
    fvg_row["created_at"]
)

if fvg_time.tzinfo is None:
    fvg_time = fvg_time.tz_localize("UTC")
else:
    fvg_time = fvg_time.tz_convert("UTC")

print("fvg_id       :", target_id)
print("created_at   :", fvg_time)
print("fvg_type     :", fvg_row["fvg_type"])
print("lifecycle    :", fvg_row["lifecycle_state"])
print("event_type   :", fvg_row["event_type"])

# ------------------------------------------------------------
# STRUCTURE STATE NORMALIZATION
# ------------------------------------------------------------

state = structure_state_v2.copy(deep=True)

state.index = pd.to_datetime(
    state.index,
    utc=True
)

state = state.sort_index()

# ------------------------------------------------------------
# STATE BEFORE / AT FVG
# ------------------------------------------------------------

state_before = state.loc[
    state.index <= fvg_time
]

# ------------------------------------------------------------
# STATE AFTER FVG
# ------------------------------------------------------------

state_after = state.loc[
    state.index > fvg_time
]

print("\nSTRUCTURE STATE COVERAGE")
print("-" * 100)
print("total states       :", len(state))
print("states <= FVG      :", len(state_before))
print("states > FVG       :", len(state_after))

# ------------------------------------------------------------
# LAST STATE BEFORE / AT
# ------------------------------------------------------------

print("\nLAST STRUCTURE STATE BEFORE / AT FVG")
print("-" * 100)

if len(state_before) == 0:

    print("NO STATE AVAILABLE")

else:

    last_state_time = state_before.index[-1]
    last_state = state_before.iloc[-1]

    print("state_time         :", last_state_time)
    print("time_delta         :", fvg_time - last_state_time)

    for col in [
        "active_bias",
        "last_bos",
        "last_bos_direction",
        "last_choch",
        "last_choch_direction",
    ]:
        if col in last_state.index:
            print(f"{col:<20}:", repr(last_state[col]))

# ------------------------------------------------------------
# FIRST STATE AFTER
# ------------------------------------------------------------

print("\nFIRST STRUCTURE STATE AFTER FVG")
print("-" * 100)

if len(state_after) == 0:

    print("NO FUTURE STATE AVAILABLE")

else:

    next_state_time = state_after.index[0]
    next_state = state_after.iloc[0]

    print("state_time         :", next_state_time)
    print("time_delta         :", next_state_time - fvg_time)

    for col in [
        "active_bias",
        "last_bos",
        "last_bos_direction",
        "last_choch",
        "last_choch_direction",
    ]:
        if col in next_state.index:
            print(f"{col:<20}:", repr(next_state[col]))

# ------------------------------------------------------------
# CONTEXT ENGINE OUTPUT
# ------------------------------------------------------------

ctx_matches = fvg_context_engine_v1[
    fvg_context_engine_v1["fvg_id"] == target_id
]

print("\nCONTEXT ENGINE OUTPUT")
print("-" * 100)

print("matches :", len(ctx_matches))

if len(ctx_matches) == 0:

    print("NO CONTEXT ROW FOUND")

else:

    ctx = ctx_matches.iloc[0]

    for col in [
        "active_bias",
        "bias_alignment",
        "last_bos",
        "last_bos_direction",
        "context_bos_id",
        "context_bos_direction",
    ]:
        if col in ctx.index:
            print(f"{col:<20}:", repr(ctx[col]))

print("\n" + "=" * 100)

UNKNOWN ACTIVE BIAS — SOURCE DIAGNOSTIC V2

FVG REGISTRY
----------------------------------------------------------------------------------------------------
matches : 1
fvg_id       : FVG_B_20261002T221500Z
created_at   : 2026-10-02 22:15:00+00:00
fvg_type     : BULLISH_FVG
lifecycle    : FILLED
event_type   : FILL

STRUCTURE STATE COVERAGE
----------------------------------------------------------------------------------------------------
total states       : 1000
states <= FVG      : 34
states > FVG       : 966

LAST STRUCTURE STATE BEFORE / AT FVG
----------------------------------------------------------------------------------------------------
state_time         : 2026-10-02 22:15:00+00:00
time_delta         : 0 days 00:00:00
active_bias         : 'UNKNOWN'
last_bos            : nan
last_bos_direction  : nan
last_choch          : nan
last_choch_direction: nan

FIRST STRUCTURE STATE AFTER FVG
----------------------------------------------------------------------------------------

In [155]:
# ============================================================
# FVG CONTEXT ENGINE V1
# UNKNOWN BIAS — ALIGNMENT SOURCE DIAGNOSTIC
# READ-ONLY
# ============================================================

target_id = "FVG_B_20261002T221500Z"

ctx = fvg_context_engine_v1[
    fvg_context_engine_v1["fvg_id"] == target_id
].iloc[0]

fvg = fvg_lifecycle_registry_v1[
    fvg_lifecycle_registry_v1["fvg_id"] == target_id
].iloc[0]

print("=" * 100)
print("UNKNOWN BIAS — ALIGNMENT SOURCE DIAGNOSTIC")
print("=" * 100)

print("\nFVG SOURCE")
print("-" * 100)

for col in fvg.index:
    print(f"{col:<25}: {repr(fvg[col])}")

print("\nCONTEXT OUTPUT")
print("-" * 100)

for col in ctx.index:
    print(f"{col:<25}: {repr(ctx[col])}")

print("\n" + "=" * 100)

UNKNOWN BIAS — ALIGNMENT SOURCE DIAGNOSTIC

FVG SOURCE
----------------------------------------------------------------------------------------------------
fvg_id                   : 'FVG_B_20261002T221500Z'
fvg_type                 : 'BULLISH_FVG'
created_at               : Timestamp('2026-10-02 22:15:00+0000', tz='UTC')
lower_bound              : np.float64(1.1252)
upper_bound              : np.float64(1.12526)
gap_size                 : np.float64(5.999999999994898e-05)
candle_a                 : Timestamp('2026-10-02 22:05:00+0000', tz='UTC')
candle_b                 : Timestamp('2026-10-02 22:10:00+0000', tz='UTC')
candle_c                 : Timestamp('2026-10-02 22:15:00+0000', tz='UTC')
lifecycle_state          : 'FILLED'
event_type               : 'FILL'
event_time               : Timestamp('2026-10-02 23:15:00+0000', tz='UTC')
event_price              : np.float64(1.1252)

CONTEXT OUTPUT
------------------------------------------------------------------------------------------

In [156]:
# ============================================================
# FVG CONTEXT ENGINE V1
# BIAS ALIGNMENT — CONTRACT / HELPER DISCOVERY
# READ-ONLY
# ============================================================

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — BIAS ALIGNMENT CONTRACT DISCOVERY")
print("=" * 100)

# ------------------------------------------------------------
# 1. Relevant constants
# ------------------------------------------------------------

print("\nRELEVANT GLOBALS")
print("-" * 100)

for name in sorted(globals()):

    name_upper = name.upper()

    if (
        "ALIGN" in name_upper
        or "BIAS" in name_upper
        or "DIRECTION" in name_upper
        or "CONTEXT" in name_upper
    ):
        obj = globals()[name]

        if isinstance(obj, (set, list, tuple, dict, str, int, float)):
            print(f"{name:<55}: {repr(obj)}")

# ------------------------------------------------------------
# 2. Candidate helper functions
# ------------------------------------------------------------

print("\nCANDIDATE FUNCTIONS")
print("-" * 100)

for name, obj in sorted(globals().items()):

    if not callable(obj):
        continue

    name_upper = name.upper()

    if (
        "ALIGN" in name_upper
        or "BIAS" in name_upper
        or "CONTEXT" in name_upper
    ):
        print(name)

print("\n" + "=" * 100)

FVG CONTEXT ENGINE V1 — BIAS ALIGNMENT CONTRACT DISCOVERY

RELEVANT GLOBALS
----------------------------------------------------------------------------------------------------
FVG_CONTEXT_ENGINE_V1_REQUIRED_BOS_COLUMNS             : ['bos_id', 'level_id', 'bos_direction', 'bos_level', 'bos_time']
FVG_CONTEXT_ENGINE_V1_REQUIRED_CHOCH_COLUMNS           : ['choch_id', 'bos_id', 'level_id', 'choch_direction', 'choch_level', 'choch_time', 'structure_bias_before', 'structure_bias_after']
FVG_CONTEXT_ENGINE_V1_REQUIRED_FVG_COLUMNS             : ['fvg_id', 'fvg_type', 'created_at', 'lower_bound', 'upper_bound', 'gap_size', 'lifecycle_state']
FVG_CONTEXT_ENGINE_V1_REQUIRED_LIQUIDITY_COLUMNS       : ['level_id', 'liquidity_source', 'liquidity_type', 'level_price', 'activation_time', 'state']
FVG_CONTEXT_ENGINE_V1_REQUIRED_STATE_COLUMNS           : ['active_bias', 'last_bos', 'last_bos_direction', 'last_bos_level', 'last_bos_level_id', 'last_choch', 'last_choch_direction', 'last_choch_level', 'l

In [157]:
# ============================================================
# FVG CONTEXT ENGINE V1
# BIAS ALIGNMENT — TRUTH TABLE AUDIT
# READ-ONLY
# ============================================================

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — BIAS ALIGNMENT TRUTH TABLE AUDIT")
print("=" * 100)

biases = ["BULLISH", "BEARISH", "UNKNOWN"]
fvg_directions = ["BULLISH", "BEARISH"]

print("\nEXPECTED BIAS ALIGNMENT")
print("-" * 100)

results = []

for bias in biases:
    for fvg_direction in fvg_directions:

        try:
            result = expected_bias_alignment(
                active_bias=bias,
                fvg_direction=fvg_direction
            )
        except TypeError as e:

            print(
                f"CALL SIGNATURE MISMATCH | "
                f"bias={bias} | "
                f"fvg_direction={fvg_direction} | "
                f"error={e}"
            )

            continue

        results.append({
            "active_bias": bias,
            "fvg_direction": fvg_direction,
            "alignment": result
        })

        print(
            f"active_bias={bias:<8} | "
            f"fvg_direction={fvg_direction:<8} | "
            f"alignment={repr(result)}"
        )

print("\n" + "=" * 100)

FVG CONTEXT ENGINE V1 — BIAS ALIGNMENT TRUTH TABLE AUDIT

EXPECTED BIAS ALIGNMENT
----------------------------------------------------------------------------------------------------
CALL SIGNATURE MISMATCH | bias=BULLISH | fvg_direction=BULLISH | error=expected_bias_alignment() got an unexpected keyword argument 'active_bias'
CALL SIGNATURE MISMATCH | bias=BULLISH | fvg_direction=BEARISH | error=expected_bias_alignment() got an unexpected keyword argument 'active_bias'
CALL SIGNATURE MISMATCH | bias=BEARISH | fvg_direction=BULLISH | error=expected_bias_alignment() got an unexpected keyword argument 'active_bias'
CALL SIGNATURE MISMATCH | bias=BEARISH | fvg_direction=BEARISH | error=expected_bias_alignment() got an unexpected keyword argument 'active_bias'
CALL SIGNATURE MISMATCH | bias=UNKNOWN | fvg_direction=BULLISH | error=expected_bias_alignment() got an unexpected keyword argument 'active_bias'
CALL SIGNATURE MISMATCH | bias=UNKNOWN | fvg_direction=BEARISH | error=expected_bias_al

In [158]:
# ============================================================
# FVG CONTEXT ENGINE V1
# EXPECTED BIAS ALIGNMENT — SIGNATURE DISCOVERY
# READ-ONLY
# ============================================================

import inspect

print("=" * 100)
print("EXPECTED BIAS ALIGNMENT — SIGNATURE DISCOVERY")
print("=" * 100)

print("\nFUNCTION")
print("-" * 100)
print("name      :", expected_bias_alignment.__name__)
print("signature :", inspect.signature(expected_bias_alignment))

print("\ndocstring")
print("-" * 100)
print(inspect.getdoc(expected_bias_alignment))

print("\n" + "=" * 100)

EXPECTED BIAS ALIGNMENT — SIGNATURE DISCOVERY

FUNCTION
----------------------------------------------------------------------------------------------------
name      : expected_bias_alignment
signature : (row)

docstring
----------------------------------------------------------------------------------------------------
None



In [159]:
# ============================================================
# FVG CONTEXT ENGINE V1
# EXPECTED BIAS ALIGNMENT — REAL ROW AUDIT
# READ-ONLY
# ============================================================

target_id = "FVG_B_20261002T221500Z"

print("=" * 100)
print("EXPECTED BIAS ALIGNMENT — REAL ROW AUDIT")
print("=" * 100)

# ------------------------------------------------------------
# Retrieve actual context row
# ------------------------------------------------------------

ctx_matches = fvg_context_engine_v1[
    fvg_context_engine_v1["fvg_id"] == target_id
]

if len(ctx_matches) != 1:
    raise ValueError(
        f"Expected exactly one context row for {target_id}, "
        f"found {len(ctx_matches)}."
    )

row = ctx_matches.iloc[0]

print("\nINPUT ROW")
print("-" * 100)

for col in [
    "fvg_id",
    "fvg_type",
    "created_at",
    "active_bias",
    "bias_alignment",
    "bos_alignment",
    "choch_alignment",
]:
    if col in row.index:
        print(f"{col:<25}: {repr(row[col])}")

# ------------------------------------------------------------
# Direct helper evaluation
# ------------------------------------------------------------

print("\nHELPER RESULT")
print("-" * 100)

helper_result = expected_bias_alignment(row)

print("expected_bias_alignment(row):", repr(helper_result))

# ------------------------------------------------------------
# Compare with stored output
# ------------------------------------------------------------

print("\nCOMPARISON")
print("-" * 100)

stored_result = row["bias_alignment"]

print("stored bias_alignment :", repr(stored_result))
print("helper result         :", repr(helper_result))
print("match                 :", stored_result == helper_result)

print("\n" + "=" * 100)

EXPECTED BIAS ALIGNMENT — REAL ROW AUDIT

INPUT ROW
----------------------------------------------------------------------------------------------------
fvg_id                   : 'FVG_B_20261002T221500Z'
fvg_type                 : 'BULLISH_FVG'
created_at               : Timestamp('2026-10-02 22:15:00+0000', tz='UTC')
active_bias              : 'UNKNOWN'
bias_alignment           : 'UNKNOWN'
bos_alignment            : 'UNKNOWN'
choch_alignment          : 'UNKNOWN'

HELPER RESULT
----------------------------------------------------------------------------------------------------
expected_bias_alignment(row): 'UNKNOWN'

COMPARISON
----------------------------------------------------------------------------------------------------
stored bias_alignment : 'UNKNOWN'
helper result         : 'UNKNOWN'
match                 : True



In [160]:
# ============================================================
# FVG CONTEXT ENGINE V1
# BUILDER / ALIGNMENT SOURCE AUDIT
# READ-ONLY
# ============================================================

import inspect

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — BUILDER / ALIGNMENT SOURCE AUDIT")
print("=" * 100)

# ------------------------------------------------------------
# Builder signatures
# ------------------------------------------------------------

print("\nBUILDER SIGNATURES")
print("-" * 100)

print(
    "build_fvg_context_engine_v1        :",
    inspect.signature(build_fvg_context_engine_v1)
)

print(
    "build_fvg_context_engine_v1_patched:",
    inspect.signature(build_fvg_context_engine_v1_patched)
)

# ------------------------------------------------------------
# Builder source snippets
# ------------------------------------------------------------

print("\nORIGINAL BUILDER SOURCE")
print("-" * 100)

try:
    print(
        inspect.getsource(
            build_fvg_context_engine_v1
        )
    )
except Exception as e:
    print("SOURCE UNAVAILABLE:", repr(e))

print("\nPATCHED BUILDER SOURCE")
print("-" * 100)

try:
    print(
        inspect.getsource(
            build_fvg_context_engine_v1_patched
        )
    )
except Exception as e:
    print("SOURCE UNAVAILABLE:", repr(e))

print("\n" + "=" * 100)

FVG CONTEXT ENGINE V1 — BUILDER / ALIGNMENT SOURCE AUDIT

BUILDER SIGNATURES
----------------------------------------------------------------------------------------------------
build_fvg_context_engine_v1        : (fvg_registry, liquidity_registry, structural_bos, structural_choch, structure_state)
build_fvg_context_engine_v1_patched: (fvg_registry, liquidity_registry, structural_bos, structural_choch, structure_state)

ORIGINAL BUILDER SOURCE
----------------------------------------------------------------------------------------------------
def build_fvg_context_engine_v1(
    fvg_registry,
    liquidity_registry,
    structural_bos,
    structural_choch,
    structure_state,
):
    """
    Build point-in-time structural/liquidity context for each FVG.

    IMPORTANT:
    - Does NOT modify FVG identity.
    - Does NOT modify FVG lifecycle.
    - Does NOT assign liquidity IDs to FVGs.
    - Does NOT merge FVGs into the liquidity registry.
    - Context is evaluated at FVG created_at.

In [161]:
print("=" * 100)
print("FVG CONTEXT ENGINE V1 — BIAS ALIGNMENT DIFFERENTIAL AUDIT")
print("=" * 100)

audit_row = fvg_context_engine_v1[
    fvg_context_engine_v1["active_bias"].astype(str).str.upper().eq("UNKNOWN")
].copy()

print("\nUNKNOWN BIAS ROWS")
print("-" * 100)
print("COUNT:", len(audit_row))

if len(audit_row):
    audit_row = audit_row[
        [
            "fvg_id",
            "fvg_type",
            "active_bias",
            "bias_alignment",
            "bos_alignment",
            "choch_alignment",
        ]
    ].copy()

    audit_row["expected_bias_alignment"] = audit_row.apply(
        expected_bias_alignment,
        axis=1
    )

    audit_row["alignment_match"] = (
        audit_row["bias_alignment"]
        ==
        audit_row["expected_bias_alignment"]
    )

    print(audit_row.to_string(index=False))

    print("\nMISMATCH COUNT:",
          int((~audit_row["alignment_match"]).sum()))

print("\n" + "=" * 100)
print("TRUTH-TABLE CHECK")
print("=" * 100)

truth_rows = [
    {"active_bias": "BULLISH", "fvg_type": "BULLISH_FVG"},
    {"active_bias": "BULLISH", "fvg_type": "BEARISH_FVG"},
    {"active_bias": "BEARISH", "fvg_type": "BULLISH_FVG"},
    {"active_bias": "BEARISH", "fvg_type": "BEARISH_FVG"},
    {"active_bias": "UNKNOWN", "fvg_type": "BULLISH_FVG"},
    {"active_bias": "UNKNOWN", "fvg_type": "BEARISH_FVG"},
]

truth_df = pd.DataFrame(truth_rows)

truth_df["expected"] = truth_df.apply(
    expected_bias_alignment,
    axis=1
)

print(truth_df.to_string(index=False))

print("\n" + "=" * 100)

FVG CONTEXT ENGINE V1 — BIAS ALIGNMENT DIFFERENTIAL AUDIT

UNKNOWN BIAS ROWS
----------------------------------------------------------------------------------------------------
COUNT: 2
                fvg_id    fvg_type active_bias bias_alignment bos_alignment choch_alignment expected_bias_alignment  alignment_match
FVG_B_20261002T221500Z BULLISH_FVG     UNKNOWN        UNKNOWN       UNKNOWN         UNKNOWN                 UNKNOWN             True
FVG_B_20261002T222500Z BULLISH_FVG     UNKNOWN        UNKNOWN       UNKNOWN         UNKNOWN                 UNKNOWN             True

MISMATCH COUNT: 0

TRUTH-TABLE CHECK
active_bias    fvg_type expected
    BULLISH BULLISH_FVG  ALIGNED
    BULLISH BEARISH_FVG  OPPOSED
    BEARISH BULLISH_FVG  OPPOSED
    BEARISH BEARISH_FVG  ALIGNED
    UNKNOWN BULLISH_FVG  UNKNOWN
    UNKNOWN BEARISH_FVG  UNKNOWN



In [162]:
fvg_context_engine_v1_fixed = build_fvg_context_engine_v1(
    fvg_registry=fvg_lifecycle_registry_v1,
    liquidity_registry=unified_liquidity_registry_v1,
    structural_bos=structural_bos_v2,
    structural_choch=structural_choch_v2,
    structure_state=structure_state_v2,
)

In [163]:
print("=" * 100)
print("FVG CONTEXT ENGINE V1 — POST-FIX TARGETED AUDIT")
print("=" * 100)

target_id = "FVG_B_20261002T221500Z"

target = fvg_context_engine_v1_fixed[
    fvg_context_engine_v1_fixed["fvg_id"] == target_id
].copy()

print("\nTARGET ROW")
print("-" * 100)

print(
    target[
        [
            "fvg_id",
            "fvg_type",
            "created_at",
            "active_bias",
            "bias_alignment",
            "last_bos",
            "last_bos_direction",
            "context_bos_id",
            "context_bos_direction",
            "bos_alignment",
            "context_choch_id",
            "context_choch_direction",
            "choch_alignment",
        ]
    ].to_string(index=False)
)

print("\n" + "-" * 100)

if len(target) != 1:
    print("TARGET ROW COUNT :", len(target))
    print("TARGET AUDIT     : FAILED")

else:
    row = target.iloc[0]

    expected = expected_bias_alignment(row)

    print("EXPECTED BIAS ALIGNMENT :", expected)
    print("ACTUAL BIAS ALIGNMENT   :", row["bias_alignment"])

    print(
        "BIAS ALIGNMENT CHECK    :",
        "PASSED"
        if row["bias_alignment"] == expected
        else "FAILED"
    )

    print(
        "NO LOOKAHEAD STRUCTURE  :",
        "PASSED"
        if (
            pd.isna(row["context_bos_id"])
            and pd.isna(row["context_bos_direction"])
        )
        else "REVIEW"
    )

print("\n" + "=" * 100)

FVG CONTEXT ENGINE V1 — POST-FIX TARGETED AUDIT

TARGET ROW
----------------------------------------------------------------------------------------------------
                fvg_id    fvg_type                created_at active_bias bias_alignment last_bos last_bos_direction context_bos_id context_bos_direction bos_alignment context_choch_id context_choch_direction choch_alignment
FVG_B_20261002T221500Z BULLISH_FVG 2026-10-02 22:15:00+00:00     UNKNOWN        UNKNOWN      NaN                NaN            NaN                   NaN       UNKNOWN              NaN                     NaN         UNKNOWN

----------------------------------------------------------------------------------------------------
EXPECTED BIAS ALIGNMENT : UNKNOWN
ACTUAL BIAS ALIGNMENT   : UNKNOWN
BIAS ALIGNMENT CHECK    : PASSED
NO LOOKAHEAD STRUCTURE  : PASSED



In [164]:
import inspect

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — POST-FIX FUNCTION VERIFICATION")
print("=" * 100)

print("\nFUNCTION SIGNATURE")
print("-" * 100)

print(inspect.signature(build_fvg_context_engine_v1))

print("\nFUNCTION SOURCE")
print("-" * 100)

try:
    source = inspect.getsource(build_fvg_context_engine_v1)
    print(source)
except Exception as e:
    print("SOURCE UNAVAILABLE:", repr(e))

print("\n" + "=" * 100)
print("DIRECT FUNCTION OBJECT TEST")
print("=" * 100)

print(
    "\nFunction object:",
    build_fvg_context_engine_v1
)

print(
    "\nModule:",
    getattr(build_fvg_context_engine_v1, "__module__", None)
)

print(
    "\nName:",
    getattr(build_fvg_context_engine_v1, "__name__", None)
)

print("\n" + "=" * 100)

FVG CONTEXT ENGINE V1 — POST-FIX FUNCTION VERIFICATION

FUNCTION SIGNATURE
----------------------------------------------------------------------------------------------------
(fvg_registry, liquidity_registry, structural_bos, structural_choch, structure_state)

FUNCTION SOURCE
----------------------------------------------------------------------------------------------------
def build_fvg_context_engine_v1(
    fvg_registry,
    liquidity_registry,
    structural_bos,
    structural_choch,
    structure_state,
):
    """
    Build point-in-time structural/liquidity context for each FVG.

    IMPORTANT:
    - Does NOT modify FVG identity.
    - Does NOT modify FVG lifecycle.
    - Does NOT assign liquidity IDs to FVGs.
    - Does NOT merge FVGs into the liquidity registry.
    - Context is evaluated at FVG created_at.
    """

    # --------------------------------------------------------
    # Validate inputs
    # --------------------------------------------------------

    _requir

In [165]:
import inspect

source = inspect.getsource(build_fvg_context_engine_v1)

print(
    "UNKNOWN HANDLING PATCHED :",
    'pd.isna(active_bias) or active_bias == "UNKNOWN"' in source
)

print("\nRELEVANT SOURCE")
print("-" * 100)

for line in source.splitlines():
    if (
        "active_bias == fvg_direction" in line
        or "pd.isna(active_bias)" in line
        or 'active_bias == "UNKNOWN"' in line
        or 'bias_alignment = "ALIGNED"' in line
        or 'bias_alignment = "UNKNOWN"' in line
        or 'bias_alignment = "OPPOSED"' in line
    ):
        print(line)

UNKNOWN HANDLING PATCHED : True

RELEVANT SOURCE
----------------------------------------------------------------------------------------------------
        if active_bias == fvg_direction:
            bias_alignment = "ALIGNED"
        elif pd.isna(active_bias) or active_bias == "UNKNOWN":
            bias_alignment = "UNKNOWN"
            bias_alignment = "OPPOSED"


In [166]:
import inspect

source = inspect.getsource(build_fvg_context_engine_v1)

lines = source.splitlines()

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — ALIGNMENT BLOCK INDENTATION AUDIT")
print("=" * 100)

for i, line in enumerate(lines):

    if "if active_bias == fvg_direction:" in line:

        start = max(0, i - 2)
        end = min(len(lines), i + 12)

        print("\nEXACT ALIGNMENT BLOCK")
        print("-" * 100)

        for j in range(start, end):
            print(
                f"{j + 1:04d} | {repr(lines[j])}"
            )

        break

print("\n" + "=" * 100)

FVG CONTEXT ENGINE V1 — ALIGNMENT BLOCK INDENTATION AUDIT

EXACT ALIGNMENT BLOCK
----------------------------------------------------------------------------------------------------
0306 | '        )'
0307 | ''
0308 | '        if active_bias == fvg_direction:'
0309 | '            bias_alignment = "ALIGNED"'
0310 | ''
0311 | '        elif pd.isna(active_bias) or active_bias == "UNKNOWN":'
0312 | '            bias_alignment = "UNKNOWN"'
0313 | ''
0314 | '        else:'
0315 | '            bias_alignment = "OPPOSED"'
0316 | ''
0317 | '        if context_bos_direction == fvg_direction:'
0318 | '            bos_alignment = "ALIGNED"'
0319 | ''



In [167]:
fvg_context_engine_v1_fixed = build_fvg_context_engine_v1(
    fvg_registry=fvg_lifecycle_registry_v1,
    liquidity_registry=unified_liquidity_registry_v1,
    structural_bos=structural_bos_v2,
    structural_choch=structural_choch_v2,
    structure_state=structure_state_v2,
)

In [168]:
print("=" * 100)
print("FVG CONTEXT ENGINE V1 — POST-FIX TARGETED AUDIT")
print("=" * 100)

target_id = "FVG_B_20261002T221500Z"

target = fvg_context_engine_v1_fixed[
    fvg_context_engine_v1_fixed["fvg_id"] == target_id
].copy()

print("\nTARGET ROW")
print("-" * 100)

print(
    target[
        [
            "fvg_id",
            "fvg_type",
            "created_at",
            "active_bias",
            "bias_alignment",
            "last_bos",
            "last_bos_direction",
            "context_bos_id",
            "context_bos_direction",
            "bos_alignment",
            "context_choch_id",
            "context_choch_direction",
            "choch_alignment",
        ]
    ].to_string(index=False)
)

print("\n" + "-" * 100)

if len(target) != 1:

    print("TARGET ROW COUNT :", len(target))
    print("TARGET AUDIT     : FAILED")

else:

    row = target.iloc[0]

    expected = expected_bias_alignment(row)

    print("EXPECTED BIAS ALIGNMENT :", expected)
    print("ACTUAL BIAS ALIGNMENT   :", row["bias_alignment"])

    print(
        "BIAS ALIGNMENT CHECK    :",
        "PASSED"
        if row["bias_alignment"] == expected
        else "FAILED"
    )

    print(
        "NO LOOKAHEAD STRUCTURE  :",
        "PASSED"
        if (
            pd.isna(row["context_bos_id"])
            and pd.isna(row["context_bos_direction"])
            and pd.isna(row["last_bos"])
        )
        else "REVIEW"
    )

print("\n" + "=" * 100)

FVG CONTEXT ENGINE V1 — POST-FIX TARGETED AUDIT

TARGET ROW
----------------------------------------------------------------------------------------------------
                fvg_id    fvg_type                created_at active_bias bias_alignment last_bos last_bos_direction context_bos_id context_bos_direction bos_alignment context_choch_id context_choch_direction choch_alignment
FVG_B_20261002T221500Z BULLISH_FVG 2026-10-02 22:15:00+00:00     UNKNOWN        UNKNOWN      NaN                NaN            NaN                   NaN       UNKNOWN              NaN                     NaN         UNKNOWN

----------------------------------------------------------------------------------------------------
EXPECTED BIAS ALIGNMENT : UNKNOWN
ACTUAL BIAS ALIGNMENT   : UNKNOWN
BIAS ALIGNMENT CHECK    : PASSED
NO LOOKAHEAD STRUCTURE  : PASSED



In [169]:
# ====================================================================================================
# FVG CONTEXT ENGINE V1 — POST-FIX REGRESSION GATE
# ====================================================================================================

import pandas as pd
import numpy as np

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — POST-FIX REGRESSION GATE")
print("=" * 100)

# ----------------------------------------------------------------------------------------------------
# REFERENCES
# ----------------------------------------------------------------------------------------------------

reference = fvg_context_engine_v1.copy(deep=True)
candidate = fvg_context_engine_v1_fixed.copy(deep=True)

# ----------------------------------------------------------------------------------------------------
# CONTRACT VOCABULARY
# ----------------------------------------------------------------------------------------------------

ALLOWED_BIAS = {
    "BULLISH",
    "BEARISH",
    "UNKNOWN",
}

ALLOWED_ALIGNMENT = {
    "ALIGNED",
    "OPPOSED",
    "UNKNOWN",
}

# ----------------------------------------------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------------------------------------------

def gate(name, condition):
    result = "PASSED" if bool(condition) else "FAILED"
    print(f"{name:<50} : {result}")
    return bool(condition)


gate_results = []

# ====================================================================================================
# 1. ROW COUNT
# ====================================================================================================

gate_results.append(
    gate(
        "ROW COUNT PRESERVATION",
        len(candidate) == len(reference)
    )
)

# ====================================================================================================
# 2. FVG ID UNIQUENESS
# ====================================================================================================

gate_results.append(
    gate(
        "FVG ID UNIQUENESS",
        candidate["fvg_id"].notna().all()
        and not candidate["fvg_id"].duplicated().any()
    )
)

# ====================================================================================================
# 3. FVG ID PRESERVATION
# ====================================================================================================

reference_ids = set(reference["fvg_id"])
candidate_ids = set(candidate["fvg_id"])

gate_results.append(
    gate(
        "FVG ID PRESERVATION",
        reference_ids == candidate_ids
    )
)

# ====================================================================================================
# 4. ID ORDER PRESERVATION
# ====================================================================================================

gate_results.append(
    gate(
        "FVG ID ORDER PRESERVATION",
        reference["fvg_id"].tolist()
        == candidate["fvg_id"].tolist()
    )
)

# ====================================================================================================
# 5. IDENTITY PRESERVATION
# ====================================================================================================

identity_columns = [
    "fvg_id",
    "fvg_type",
    "created_at",
    "lower_bound",
    "upper_bound",
    "gap_size",
]

identity_ok = True

for col in identity_columns:

    if col not in reference.columns or col not in candidate.columns:
        identity_ok = False
        break

    if col in {"lower_bound", "upper_bound", "gap_size"}:

        if not np.allclose(
            reference[col].astype(float).to_numpy(),
            candidate[col].astype(float).to_numpy(),
            equal_nan=True,
        ):
            identity_ok = False
            break

    else:

        if not reference[col].equals(candidate[col]):
            identity_ok = False
            break

gate_results.append(
    gate(
        "FVG IDENTITY PRESERVATION",
        identity_ok
    )
)

# ====================================================================================================
# 6. LIFECYCLE PRESERVATION
# ====================================================================================================

gate_results.append(
    gate(
        "LIFECYCLE PRESERVATION",
        reference["lifecycle_state"].equals(
            candidate["lifecycle_state"]
        )
    )
)

# ====================================================================================================
# 7. EVENT PRESERVATION
# ====================================================================================================

event_columns = [
    "event_type",
    "event_time",
    "event_price",
]

event_ok = True

for col in event_columns:

    if col not in reference.columns or col not in candidate.columns:
        event_ok = False
        break

    if col == "event_price":

        if not np.allclose(
            pd.to_numeric(reference[col], errors="coerce"),
            pd.to_numeric(candidate[col], errors="coerce"),
            equal_nan=True,
        ):
            event_ok = False
            break

    else:

        if not reference[col].equals(candidate[col]):
            event_ok = False
            break

gate_results.append(
    gate(
        "LIFECYCLE EVENT PRESERVATION",
        event_ok
    )
)

# ====================================================================================================
# 8. BIAS VOCABULARY
# ====================================================================================================

bias_values = set(
    candidate["active_bias"]
    .dropna()
    .astype(str)
    .str.upper()
    .unique()
)

gate_results.append(
    gate(
        "ACTIVE BIAS VOCABULARY",
        bias_values.issubset(ALLOWED_BIAS)
    )
)

# ====================================================================================================
# 9. BIAS ALIGNMENT VOCABULARY
# ====================================================================================================

bias_alignment_values = set(
    candidate["bias_alignment"]
    .dropna()
    .astype(str)
    .str.upper()
    .unique()
)

gate_results.append(
    gate(
        "BIAS ALIGNMENT VOCABULARY",
        bias_alignment_values.issubset(ALLOWED_ALIGNMENT)
    )
)

# ====================================================================================================
# 10. BIAS ALIGNMENT CONTRACT
# ====================================================================================================

def expected_alignment_from_values(active_bias, fvg_type):

    if pd.isna(active_bias):
        return "UNKNOWN"

    active_bias = str(active_bias).upper()

    if active_bias == "UNKNOWN":
        return "UNKNOWN"

    if fvg_type == "BULLISH_FVG":
        if active_bias == "BULLISH":
            return "ALIGNED"
        if active_bias == "BEARISH":
            return "OPPOSED"

    elif fvg_type == "BEARISH_FVG":
        if active_bias == "BEARISH":
            return "ALIGNED"
        if active_bias == "BULLISH":
            return "OPPOSED"

    return "UNKNOWN"


expected_bias_alignment = candidate.apply(
    lambda row: expected_alignment_from_values(
        row["active_bias"],
        row["fvg_type"],
    ),
    axis=1,
)

gate_results.append(
    gate(
        "BIAS ALIGNMENT CONTRACT",
        (
            candidate["bias_alignment"]
            .astype(str)
            .to_numpy()
            ==
            expected_bias_alignment
            .astype(str)
            .to_numpy()
        ).all()
    )
)

# ====================================================================================================
# 11. BOS ALIGNMENT CONTRACT
# ====================================================================================================

def expected_event_alignment(direction, fvg_type):

    if pd.isna(direction):
        return "UNKNOWN"

    direction = str(direction).upper()

    if fvg_type == "BULLISH_FVG":

        if direction == "BOS_BULLISH":
            return "ALIGNED"

        if direction == "BOS_BEARISH":
            return "OPPOSED"

    elif fvg_type == "BEARISH_FVG":

        if direction == "BOS_BEARISH":
            return "ALIGNED"

        if direction == "BOS_BULLISH":
            return "OPPOSED"

    return "UNKNOWN"


expected_bos_alignment = candidate.apply(
    lambda row: expected_event_alignment(
        row["context_bos_direction"],
        row["fvg_type"],
    ),
    axis=1,
)

gate_results.append(
    gate(
        "BOS ALIGNMENT CONTRACT",
        (
            candidate["bos_alignment"]
            .astype(str)
            .to_numpy()
            ==
            expected_bos_alignment
            .astype(str)
            .to_numpy()
        ).all()
    )
)

# ====================================================================================================
# 12. CHOCH ALIGNMENT CONTRACT
# ====================================================================================================

def expected_choch_alignment(direction, fvg_type):

    if pd.isna(direction):
        return "UNKNOWN"

    direction = str(direction).upper()

    if fvg_type == "BULLISH_FVG":

        if direction == "CHOCH_BULLISH":
            return "ALIGNED"

        if direction == "CHOCH_BEARISH":
            return "OPPOSED"

    elif fvg_type == "BEARISH_FVG":

        if direction == "CHOCH_BEARISH":
            return "ALIGNED"

        if direction == "CHOCH_BULLISH":
            return "OPPOSED"

    return "UNKNOWN"


expected_choch_alignment = candidate.apply(
    lambda row: expected_choch_alignment(
        row["context_choch_direction"],
        row["fvg_type"],
    ),
    axis=1,
)

gate_results.append(
    gate(
        "CHoCH ALIGNMENT CONTRACT",
        (
            candidate["choch_alignment"]
            .astype(str)
            .to_numpy()
            ==
            expected_choch_alignment
            .astype(str)
            .to_numpy()
        ).all()
    )
)

# ====================================================================================================
# 13. NO-LOOKAHEAD
# ====================================================================================================

candidate_created = pd.to_datetime(
    candidate["created_at"],
    utc=True,
    errors="coerce",
)

candidate_bos_time = pd.to_datetime(
    candidate["context_bos_time"],
    utc=True,
    errors="coerce",
)

candidate_choch_time = pd.to_datetime(
    candidate["context_choch_time"],
    utc=True,
    errors="coerce",
)

bos_no_lookahead = (
    candidate_bos_time.isna()
    | (candidate_bos_time <= candidate_created)
)

choch_no_lookahead = (
    candidate_choch_time.isna()
    | (candidate_choch_time <= candidate_created)
)

gate_results.append(
    gate(
        "BOS NO-LOOKAHEAD",
        bos_no_lookahead.all()
    )
)

gate_results.append(
    gate(
        "CHoCH NO-LOOKAHEAD",
        choch_no_lookahead.all()
    )
)

# ====================================================================================================
# 14. STRUCTURE STATE NO-LOOKAHEAD
# ====================================================================================================

state_last_bos_ok = True
state_last_choch_ok = True

for _, row in candidate.iterrows():

    created = pd.to_datetime(
        row["created_at"],
        utc=True,
        errors="coerce",
    )

    # last BOS is represented by the state field.
    # If a state event exists, its context event must not
    # be later than FVG creation.

    if (
        pd.notna(row["last_bos"])
        and pd.notna(row["context_bos_time"])
        and row["context_bos_time"] > created
    ):
        state_last_bos_ok = False

    if (
        pd.notna(row["last_choch"])
        and pd.notna(row["context_choch_time"])
        and row["context_choch_time"] > created
    ):
        state_last_choch_ok = False

gate_results.append(
    gate(
        "STRUCTURE STATE NO-LOOKAHEAD",
        state_last_bos_ok and state_last_choch_ok
    )
)

# ====================================================================================================
# 15. LIQUIDITY CONTEXT PRESERVATION
# ====================================================================================================

liquidity_columns = [
    "overlapping_liquidity_ids",
    "overlapping_liquidity_types",
    "nearest_liquidity_below_id",
    "nearest_liquidity_below_type",
    "nearest_liquidity_below_price",
    "nearest_liquidity_above_id",
    "nearest_liquidity_above_type",
    "nearest_liquidity_above_price",
]

liquidity_ok = True

for col in liquidity_columns:

    if col not in reference.columns or col not in candidate.columns:
        liquidity_ok = False
        break

    ref_values = reference[col].tolist()
    cand_values = candidate[col].tolist()

    if col.endswith("_price"):

        if not np.allclose(
            pd.to_numeric(reference[col], errors="coerce"),
            pd.to_numeric(candidate[col], errors="coerce"),
            equal_nan=True,
        ):
            liquidity_ok = False
            break

    else:

        if ref_values != cand_values:
            liquidity_ok = False
            break

gate_results.append(
    gate(
        "LIQUIDITY CONTEXT PRESERVATION",
        liquidity_ok
    )
)

# ====================================================================================================
# 16. ONLY EXPECTED SEMANTIC CHANGE
# ====================================================================================================

comparison = pd.DataFrame({
    "fvg_id": reference["fvg_id"],
    "reference_bias_alignment": reference["bias_alignment"],
    "candidate_bias_alignment": candidate["bias_alignment"],
})

changed = comparison[
    comparison["reference_bias_alignment"]
    !=
    comparison["candidate_bias_alignment"]
].copy()

expected_target_change = (
    len(changed) == 1
    and changed.iloc[0]["fvg_id"]
        == "FVG_B_20261002T221500Z"
    and changed.iloc[0]["reference_bias_alignment"]
        == "OPPOSED"
    and changed.iloc[0]["candidate_bias_alignment"]
        == "UNKNOWN"
)

gate_results.append(
    gate(
        "ONLY EXPECTED SEMANTIC CHANGE",
        expected_target_change
    )
)

# ====================================================================================================
# 17. DETERMINISTIC REBUILD
# ====================================================================================================

candidate_rebuild = build_fvg_context_engine_v1(
    fvg_registry=fvg_lifecycle_registry_v1,
    liquidity_registry=unified_liquidity_registry_v1,
    structural_bos=structural_bos_v2,
    structural_choch=structural_choch_v2,
    structure_state=structure_state_v2,
)

deterministic_ok = candidate.equals(candidate_rebuild)

gate_results.append(
    gate(
        "DETERMINISTIC REBUILD",
        deterministic_ok
    )
)

# ====================================================================================================
# FINAL RESULT
# ====================================================================================================

print("\n" + "=" * 100)
print("REGRESSION GATE SUMMARY")
print("=" * 100)

failed_count = sum(
    1 for result in gate_results
    if not result
)

print(
    f"TOTAL GATES : {len(gate_results)}"
)

print(
    f"PASSED      : {len(gate_results) - failed_count}"
)

print(
    f"FAILED      : {failed_count}"
)

print(
    "\nFINAL RESULT :",
    "PASSED"
    if failed_count == 0
    else "FAILED"
)

print("=" * 100)

FVG CONTEXT ENGINE V1 — POST-FIX REGRESSION GATE
ROW COUNT PRESERVATION                             : PASSED
FVG ID UNIQUENESS                                  : PASSED
FVG ID PRESERVATION                                : PASSED
FVG ID ORDER PRESERVATION                          : PASSED
FVG IDENTITY PRESERVATION                          : PASSED
LIFECYCLE PRESERVATION                             : PASSED
LIFECYCLE EVENT PRESERVATION                       : FAILED
ACTIVE BIAS VOCABULARY                             : PASSED
BIAS ALIGNMENT VOCABULARY                          : PASSED
BIAS ALIGNMENT CONTRACT                            : PASSED
BOS ALIGNMENT CONTRACT                             : FAILED
CHoCH ALIGNMENT CONTRACT                           : FAILED
BOS NO-LOOKAHEAD                                   : PASSED
CHoCH NO-LOOKAHEAD                                 : PASSED
STRUCTURE STATE NO-LOOKAHEAD                       : PASSED
LIQUIDITY CONTEXT PRESERVATION                     

In [170]:
print("=" * 100)
print("FVG CONTEXT ENGINE V1 — POST-FIX FAILURE DIAGNOSTIC V2")
print("=" * 100)

reference = fvg_context_engine_v1.copy(deep=True)
candidate = fvg_context_engine_v1_fixed.copy(deep=True)
lifecycle = fvg_lifecycle_registry_v1.copy(deep=True)

# ============================================================
# 1. LIFECYCLE EVENT PRESERVATION
# ============================================================

print("\n" + "-" * 100)
print("1. LIFECYCLE EVENT PRESERVATION DIAGNOSTIC")
print("-" * 100)

lifecycle_event_cols = [
    "fvg_id",
    "lifecycle_state",
    "event_type",
    "event_time",
    "event_price",
]

context_lifecycle_cols = [
    "fvg_id",
    "lifecycle_state",
]

# Context Engine can only preserve lifecycle_state directly.
# Event fields must be validated against the canonical lifecycle registry.

candidate_lifecycle = candidate[context_lifecycle_cols].copy()

canonical_lifecycle = lifecycle[lifecycle_event_cols].copy()

lifecycle_check = candidate_lifecycle.merge(
    canonical_lifecycle,
    on="fvg_id",
    how="outer",
    suffixes=("_context", "_registry"),
    indicator=True,
)

lifecycle_state_mismatch = (
    lifecycle_check["_merge"].ne("both")
    |
    (
        lifecycle_check["lifecycle_state_context"]
        != lifecycle_check["lifecycle_state_registry"]
    )
)

lifecycle_failures = lifecycle_check[lifecycle_state_mismatch].copy()

print(f"CANONICAL LIFECYCLE ROWS : {len(canonical_lifecycle)}")
print(f"CANDIDATE CONTEXT ROWS    : {len(candidate_lifecycle)}")
print(f"LIFECYCLE STATE FAILURES  : {len(lifecycle_failures)}")

if len(lifecycle_failures):
    print(lifecycle_failures.to_string(index=False))
else:
    print("LIFECYCLE STATE PRESERVATION : PASSED")

# Event preservation:
# verify that every candidate FVG still resolves to the exact
# canonical lifecycle event attached to that FVG.

candidate_ids = set(candidate["fvg_id"])
canonical_ids = set(canonical_lifecycle["fvg_id"])

missing_from_registry = candidate_ids - canonical_ids
missing_from_candidate = canonical_ids - candidate_ids

print(f"\nEVENT FVGs MISSING FROM REGISTRY  : {len(missing_from_registry)}")
print(f"EVENT FVGs MISSING FROM CANDIDATE  : {len(missing_from_candidate)}")

if missing_from_registry:
    print(sorted(missing_from_registry))

if missing_from_candidate:
    print(sorted(missing_from_candidate))

# ============================================================
# 2. BOS ALIGNMENT CONTRACT
# ============================================================

print("\n" + "-" * 100)
print("2. BOS ALIGNMENT CONTRACT DIAGNOSTIC")
print("-" * 100)

bos_cols = [
    "fvg_id",
    "fvg_type",
    "active_bias",
    "context_bos_direction",
    "bos_alignment",
]

bos_diag = candidate[bos_cols].copy()

def expected_bos_alignment_diag(row):

    bias = row["active_bias"]
    direction = row["context_bos_direction"]

    if pd.isna(bias) or bias == "UNKNOWN":
        return "UNKNOWN"

    if pd.isna(direction):
        return "UNKNOWN"

    if bias == direction:
        return "ALIGNED"

    return "OPPOSED"


bos_diag["expected_bos_alignment"] = bos_diag.apply(
    expected_bos_alignment_diag,
    axis=1,
)

bos_diag["check"] = (
    bos_diag["bos_alignment"]
    == bos_diag["expected_bos_alignment"]
)

bos_failures = bos_diag[~bos_diag["check"]].copy()

print(f"BOS ALIGNMENT FAILURES : {len(bos_failures)}")

if len(bos_failures):
    print(bos_failures.to_string(index=False))
else:
    print("BOS ALIGNMENT CONTRACT : PASSED")


# ============================================================
# 3. CHoCH ALIGNMENT CONTRACT
# ============================================================

print("\n" + "-" * 100)
print("3. CHoCH ALIGNMENT CONTRACT DIAGNOSTIC")
print("-" * 100)

choch_cols = [
    "fvg_id",
    "fvg_type",
    "active_bias",
    "context_choch_direction",
    "choch_alignment",
]

choch_diag = candidate[choch_cols].copy()

def expected_choch_alignment_diag(row):

    bias = row["active_bias"]
    direction = row["context_choch_direction"]

    if pd.isna(bias) or bias == "UNKNOWN":
        return "UNKNOWN"

    if pd.isna(direction):
        return "UNKNOWN"

    if bias == direction:
        return "ALIGNED"

    return "OPPOSED"


choch_diag["expected_choch_alignment"] = choch_diag.apply(
    expected_choch_alignment_diag,
    axis=1,
)

choch_diag["check"] = (
    choch_diag["choch_alignment"]
    == choch_diag["expected_choch_alignment"]
)

choch_failures = choch_diag[~choch_diag["check"]].copy()

print(f"CHoCH ALIGNMENT FAILURES : {len(choch_failures)}")

if len(choch_failures):
    print(choch_failures.to_string(index=False))
else:
    print("CHoCH ALIGNMENT CONTRACT : PASSED")


# ============================================================
# 4. REFERENCE → CANDIDATE FIELD DIFFERENCES
# ============================================================

print("\n" + "-" * 100)
print("4. REFERENCE → CANDIDATE FIELD DIFFERENCES")
print("-" * 100)

compare_cols = [
    "fvg_id",
    "fvg_type",
    "created_at",
    "lower_bound",
    "upper_bound",
    "gap_size",
    "lifecycle_state",
    "active_bias",
    "bias_alignment",
    "last_bos",
    "last_bos_direction",
    "last_bos_level",
    "last_bos_level_id",
    "context_bos_id",
    "context_bos_level_id",
    "context_bos_direction",
    "context_bos_level",
    "context_bos_time",
    "bos_alignment",
    "last_choch",
    "last_choch_direction",
    "last_choch_level",
    "last_choch_level_id",
    "context_choch_id",
    "context_choch_level_id",
    "context_choch_direction",
    "context_choch_level",
    "context_choch_time",
    "choch_alignment",
]

ref_cmp = reference[compare_cols].set_index("fvg_id")
cand_cmp = candidate[compare_cols].set_index("fvg_id")

all_diff = []

for fvg_id in ref_cmp.index:

    for col in compare_cols[1:]:

        a = ref_cmp.loc[fvg_id, col]
        b = cand_cmp.loc[fvg_id, col]

        same = (
            (pd.isna(a) and pd.isna(b))
            or (a == b)
        )

        if not same:

            all_diff.append({
                "fvg_id": fvg_id,
                "column": col,
                "reference": a,
                "candidate": b,
            })

all_diff_df = pd.DataFrame(all_diff)

print(f"TOTAL FIELD-LEVEL DIFFERENCES : {len(all_diff_df)}")

if len(all_diff_df):
    print(all_diff_df.to_string(index=False))
else:
    print("NO FIELD-LEVEL DIFFERENCES")


# ============================================================
# 5. EXPECTED CHANGE CHECK
# ============================================================

print("\n" + "-" * 100)
print("5. EXPECTED SEMANTIC CHANGE CHECK")
print("-" * 100)

expected_id = "FVG_B_20261002T221500Z"
expected_column = "bias_alignment"
expected_reference = "OPPOSED"
expected_candidate = "UNKNOWN"

expected_change_mask = (
    (all_diff_df["fvg_id"] == expected_id)
    &
    (all_diff_df["column"] == expected_column)
)

unexpected_changes = all_diff_df[~expected_change_mask].copy()

expected_change_found = False

if len(all_diff_df):

    expected_rows = all_diff_df[expected_change_mask]

    if len(expected_rows) == 1:

        row = expected_rows.iloc[0]

        expected_change_found = (
            row["reference"] == expected_reference
            and row["candidate"] == expected_candidate
        )

print(
    "EXPECTED CHANGE FOUND                         :",
    "PASSED" if expected_change_found else "FAILED"
)

print(
    "UNEXPECTED FIELD CHANGES                      :",
    len(unexpected_changes)
)

if len(unexpected_changes):
    print(unexpected_changes.to_string(index=False))


print("\n" + "=" * 100)
print("DIAGNOSTIC V2 COMPLETE — NO DATA WAS MODIFIED")
print("=" * 100)

FVG CONTEXT ENGINE V1 — POST-FIX FAILURE DIAGNOSTIC V2

----------------------------------------------------------------------------------------------------
1. LIFECYCLE EVENT PRESERVATION DIAGNOSTIC
----------------------------------------------------------------------------------------------------
CANONICAL LIFECYCLE ROWS : 99
CANDIDATE CONTEXT ROWS    : 99
LIFECYCLE STATE FAILURES  : 0
LIFECYCLE STATE PRESERVATION : PASSED

EVENT FVGs MISSING FROM REGISTRY  : 0
EVENT FVGs MISSING FROM CANDIDATE  : 0

----------------------------------------------------------------------------------------------------
2. BOS ALIGNMENT CONTRACT DIAGNOSTIC
----------------------------------------------------------------------------------------------------
BOS ALIGNMENT FAILURES : 0
BOS ALIGNMENT CONTRACT : PASSED

----------------------------------------------------------------------------------------------------
3. CHoCH ALIGNMENT CONTRACT DIAGNOSTIC
----------------------------------------------------

KeyError: 'fvg_id'

In [ ]:
# ============================================================
# FVG CONTEXT ENGINE V1
# UNKNOWN ACTIVE BIAS — SOURCE DIAGNOSTIC
# READ-ONLY
# ============================================================

target_id = "FVG_B_20261002T101500Z"

fvg_row = fvg_lifecycle_registry_v1[
    fvg_lifecycle_registry_v1["fvg_id"] == target_id
].iloc[0]

fvg_time = pd.Timestamp(fvg_row["created_at"])

print("=" * 100)
print("UNKNOWN ACTIVE BIAS — SOURCE DIAGNOSTIC")
print("=" * 100)

print("\nFVG")
print("-" * 100)
print("fvg_id       :", target_id)
print("created_at   :", fvg_time)
print("fvg_type     :", fvg_row["fvg_type"])

# ------------------------------------------------------------
# Structure state immediately before / at FVG creation
# ------------------------------------------------------------

state = structure_state_v2.copy(deep=True)

state.index = pd.to_datetime(
    state.index,
    utc=True
)

state_before = state.loc[
    state.index <= fvg_time
]

print("\nSTRUCTURE STATE")
print("-" * 100)

if len(state_before) == 0:
    print("No structure state available before FVG creation.")
else:
    last_state_time = state_before.index[-1]
    last_state = state_before.iloc[-1]

    print("State time   :", last_state_time)
    print("active_bias  :", repr(last_state["active_bias"]))
    print("last_bos     :", repr(last_state["last_bos"]))
    print("last_bos_dir :", repr(last_state["last_bos_direction"]))
    print("last_choch   :", repr(last_state["last_choch"]))
    print("last_choch_dir:", repr(last_state["last_choch_direction"]))

# ------------------------------------------------------------
# Existing Context output
# ------------------------------------------------------------

ctx = fvg_context_engine_v1[
    fvg_context_engine_v1["fvg_id"] == target_id
].iloc[0]

print("\nCONTEXT ENGINE OUTPUT")
print("-" * 100)

print("active_bias      :", repr(ctx["active_bias"]))
print("bias_alignment   :", repr(ctx["bias_alignment"]))
print("last_bos         :", repr(ctx["last_bos"]))
print("last_bos_dir     :", repr(ctx["last_bos_direction"]))
print("context_bos_id   :", repr(ctx["context_bos_id"]))
print("context_bos_dir  :", repr(ctx["context_bos_direction"]))

print("=" * 100)

In [ ]:
# ============================================================
# FVG CONTEXT ENGINE V1
# BIAS ALIGNMENT SEMANTICS — FULL DATASET DIAGNOSTIC
# READ-ONLY
# ============================================================

df = fvg_context_engine_v1.copy(deep=True)

def expected_bias_alignment_v2(row):
    active_bias = row["active_bias"]

    if active_bias == "UNKNOWN":
        return "UNKNOWN"

    fvg_direction = (
        "BULLISH"
        if row["fvg_type"] == "BULLISH_FVG"
        else "BEARISH"
    )

    if active_bias == fvg_direction:
        return "ALIGNED"

    return "OPPOSED"


df["expected_bias_alignment_v2"] = df.apply(
    expected_bias_alignment_v2,
    axis=1,
)

df["bias_alignment_match_v2"] = (
    df["bias_alignment"]
    == df["expected_bias_alignment_v2"]
)

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — BIAS ALIGNMENT SEMANTICS V2")
print("=" * 100)

print("\nEXPECTED ALIGNMENT DISTRIBUTION")
print("-" * 100)
print(
    df["expected_bias_alignment_v2"]
    .value_counts(dropna=False)
)

print("\nACTUAL ALIGNMENT DISTRIBUTION")
print("-" * 100)
print(
    df["bias_alignment"]
    .value_counts(dropna=False)
)

print("\nSEMANTIC MATCH")
print("-" * 100)

print(
    "MATCHED :",
    int(df["bias_alignment_match_v2"].sum())
)

print(
    "MISMATCHED :",
    int((~df["bias_alignment_match_v2"]).sum())
)

print("\nMISMATCHED ROWS")
print("-" * 100)

mismatch = df.loc[
    ~df["bias_alignment_match_v2"],
    [
        "fvg_id",
        "fvg_type",
        "active_bias",
        "bias_alignment",
        "expected_bias_alignment_v2",
    ]
]

if len(mismatch) == 0:
    print("NONE")
else:
    print(
        mismatch.to_string(index=False)
    )

print("=" * 100)

In [ ]:
# ============================================================
# LOCATE BIAS ALIGNMENT LOGIC
# READ-ONLY
# ============================================================

import inspect

source = inspect.getsource(build_fvg_context_engine_v1)

lines = source.splitlines()

print("=" * 100)
print("BUILD_FVG_CONTEXT_ENGINE_V1 — BIAS ALIGNMENT LOGIC")
print("=" * 100)

matches = []

for i, line in enumerate(lines):
    if (
        "bias_alignment" in line
        or "fvg_direction" in line
        or "active_bias" in line
    ):
        start = max(0, i - 5)
        end = min(len(lines), i + 8)

        matches.append((start, end))

# merge overlapping ranges
merged = []

for start, end in matches:
    if not merged or start > merged[-1][1]:
        merged.append([start, end])
    else:
        merged[-1][1] = max(merged[-1][1], end)

for start, end in merged:
    print()
    print("-" * 100)

    for j in range(start, end):
        print(f"{j + 1:04d}: {lines[j]}")

print("\n" + "=" * 100)

In [ ]:
# ============================================================
# FVG CONTEXT ENGINE V1
# CANONICAL BUILDER PATCH
# UNKNOWN BIAS SEMANTICS
# ============================================================

import inspect
import textwrap

_original_source = inspect.getsource(build_fvg_context_engine_v1)

_old_block = '''        if active_bias == fvg_direction:
            bias_alignment = "ALIGNED"

        elif pd.isna(active_bias):
            bias_alignment = "UNKNOWN"

        else:
            bias_alignment = "OPPOSED"
'''

_new_block = '''        if active_bias == fvg_direction:
            bias_alignment = "ALIGNED"

        elif active_bias == "UNKNOWN" or pd.isna(active_bias):
            bias_alignment = "UNKNOWN"

        else:
            bias_alignment = "OPPOSED"
'''

if _old_block not in _original_source:
    raise RuntimeError(
        "Expected bias alignment block was not found. "
        "Original builder was NOT modified."
    )

_patched_source = _original_source.replace(
    _old_block,
    _new_block,
    1,
)

# ------------------------------------------------------------
# Compile patched function in isolated namespace
# ------------------------------------------------------------

_namespace = {
    "__builtins__": __builtins__,
    "pd": pd,
    "np": np,
}

exec(
    _patched_source,
    _namespace,
)

build_fvg_context_engine_v1_patched = _namespace[
    "build_fvg_context_engine_v1"
]

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — PATCHED BUILDER CREATED")
print("=" * 100)

print("Original builder       : UNCHANGED")
print("Patched builder        : CREATED")
print("Changed semantic rule  : UNKNOWN bias → UNKNOWN alignment")

print()
print("Original function name :",
      build_fvg_context_engine_v1.__name__)

print("Patched function name  :",
      build_fvg_context_engine_v1_patched.__name__)

print("=" * 100)

In [ ]:
# ============================================================
# FVG CONTEXT ENGINE V1
# SAFE PATCHED BUILDER
# PRESERVE ORIGINAL GLOBAL NAMESPACE
# ============================================================

import inspect

_original_source = inspect.getsource(build_fvg_context_engine_v1)

_old_block = '''        if active_bias == fvg_direction:
            bias_alignment = "ALIGNED"

        elif pd.isna(active_bias):
            bias_alignment = "UNKNOWN"

        else:
            bias_alignment = "OPPOSED"
'''

_new_block = '''        if active_bias == fvg_direction:
            bias_alignment = "ALIGNED"

        elif active_bias == "UNKNOWN" or pd.isna(active_bias):
            bias_alignment = "UNKNOWN"

        else:
            bias_alignment = "OPPOSED"
'''

if _old_block not in _original_source:
    raise RuntimeError(
        "Expected bias alignment block was not found. "
        "No patched builder was created."
    )

_patched_source = _original_source.replace(
    _old_block,
    _new_block,
    1,
)

# ------------------------------------------------------------
# IMPORTANT:
# Use the SAME GLOBAL namespace as the original function.
# This preserves _require_columns and every other dependency.
# ------------------------------------------------------------

_patched_namespace = build_fvg_context_engine_v1.__globals__.copy()

exec(
    _patched_source,
    _patched_namespace,
)

build_fvg_context_engine_v1_patched = _patched_namespace[
    "build_fvg_context_engine_v1"
]

print("=" * 100)
print("SAFE PATCHED FVG CONTEXT BUILDER")
print("=" * 100)

print("Original builder : UNCHANGED")
print("Patched builder  : CREATED")
print()
print("Patch:")
print('UNKNOWN active_bias -> UNKNOWN bias_alignment')
print()
print("Helper namespace : PRESERVED")
print("_require_columns  :",
      "_require_columns" in _patched_namespace)

print("=" * 100)

In [ ]:
# ============================================================
# BUILD PATCHED FVG CONTEXT ENGINE V1
# ============================================================

fvg_context_engine_v1_patched = build_fvg_context_engine_v1_patched(
    fvg_registry=fvg_lifecycle_registry_v1,
    liquidity_registry=unified_liquidity_registry_v1,
    structural_bos=structural_bos_v2,
    structural_choch=structural_choch_v2,
    structure_state=structure_state_v2,
)

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — PATCHED BUILD")
print("=" * 100)

print("Shape :", fvg_context_engine_v1_patched.shape)

print()
print("BIAS ALIGNMENT")
print(
    fvg_context_engine_v1_patched[
        "bias_alignment"
    ].value_counts(dropna=False)
)

print()
print("UNKNOWN BIAS")
print(
    fvg_context_engine_v1_patched.loc[
        fvg_context_engine_v1_patched["active_bias"] == "UNKNOWN",
        [
            "fvg_id",
            "fvg_type",
            "active_bias",
            "bias_alignment",
        ],
    ].to_string(index=False)
)

print("=" * 100)

In [ ]:
# ============================================================
# FVG CONTEXT ENGINE V1
# PATCH DELTA AUDIT
# ============================================================

import pandas as pd
import numpy as np

old_df = fvg_context_engine_v1.copy(deep=True)
new_df = fvg_context_engine_v1_patched.copy(deep=True)

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — PATCH DELTA AUDIT")
print("=" * 100)

# ------------------------------------------------------------
# 1. Shape
# ------------------------------------------------------------

shape_pass = old_df.shape == new_df.shape

print(f"SHAPE PRESERVATION                         : "
      f"{'PASSED' if shape_pass else 'FAILED'}")

# ------------------------------------------------------------
# 2. Column order
# ------------------------------------------------------------

columns_pass = list(old_df.columns) == list(new_df.columns)

print(f"COLUMN ORDER PRESERVATION                  : "
      f"{'PASSED' if columns_pass else 'FAILED'}")

# ------------------------------------------------------------
# 3. Index preservation
# ------------------------------------------------------------

index_pass = old_df.index.equals(new_df.index)

print(f"INDEX PRESERVATION                         : "
      f"{'PASSED' if index_pass else 'FAILED'}")

# ------------------------------------------------------------
# 4. Dtype preservation
# ------------------------------------------------------------

dtype_diff = [
    col
    for col in old_df.columns
    if old_df[col].dtype != new_df[col].dtype
]

dtype_pass = len(dtype_diff) == 0

print(f"DTYPE PRESERVATION                         : "
      f"{'PASSED' if dtype_pass else 'FAILED'}")

if not dtype_pass:
    print("Dtype differences:")
    print(dtype_diff)

# ------------------------------------------------------------
# 5. Find exact cell-level differences
# ------------------------------------------------------------

differences = []

for col in old_df.columns:

    old_col = old_df[col]
    new_col = new_df[col]

    for idx in old_df.index:

        old_val = old_col.loc[idx]
        new_val = new_col.loc[idx]

        if pd.isna(old_val) and pd.isna(new_val):
            continue

        if old_val != new_val:
            differences.append(
                {
                    "index": idx,
                    "column": col,
                    "old_value": old_val,
                    "new_value": new_val,
                }
            )

delta_df = pd.DataFrame(differences)

print()
print("TOTAL CELL DIFFERENCES                     :", len(delta_df))

# ------------------------------------------------------------
# 6. Expected delta
# ------------------------------------------------------------

expected_single_delta = (
    len(delta_df) == 1
    and delta_df.iloc[0]["column"] == "bias_alignment"
    and delta_df.iloc[0]["old_value"] == "OPPOSED"
    and delta_df.iloc[0]["new_value"] == "UNKNOWN"
)

print(f"PATCH DELTA CONTRACT                       : "
      f"{'PASSED' if expected_single_delta else 'FAILED'}")

# ------------------------------------------------------------
# 7. Print exact delta
# ------------------------------------------------------------

print()
print("EXACT DELTA")
print("-" * 100)

if len(delta_df) > 0:
    print(delta_df.to_string(index=False))
else:
    print("No differences found.")

# ------------------------------------------------------------
# 8. Verify all other columns are identical
# ------------------------------------------------------------

changed_columns = (
    sorted(delta_df["column"].unique().tolist())
    if len(delta_df) > 0
    else []
)

other_columns_unchanged = (
    changed_columns == ["bias_alignment"]
)

print()
print(f"ONLY BIAS ALIGNMENT CHANGED                 : "
      f"{'PASSED' if other_columns_unchanged else 'FAILED'}")

print("=" * 100)

In [ ]:
# ============================================================
# LOCATE EXISTING FVG CONTEXT ENGINE V1 AUDIT FUNCTIONS
# SAFE GLOBALS SNAPSHOT
# ============================================================

import inspect

print("=" * 100)
print("SEARCHING FOR EXISTING FVG CONTEXT ENGINE AUDIT FUNCTIONS")
print("=" * 100)

_matches = []

_globals_snapshot = list(globals().items())

for _name, _obj in _globals_snapshot:

    if not callable(_obj):
        continue

    _name_l = _name.lower()

    if (
        "fvg" in _name_l
        and "context" in _name_l
        and "audit" in _name_l
    ):
        _matches.append(_name)

print()
print("MATCHING FUNCTIONS:")
print("-" * 100)

if _matches:
    for _name in sorted(set(_matches)):
        print(_name)
else:
    print("NO MATCHING FUNCTIONS FOUND")

print("=" * 100)

In [ ]:
# ============================================================
# LOCATE ORIGINAL FVG CONTEXT CONTRACT AUDIT CELL
# SEARCH JUPYTER INPUT HISTORY
# ============================================================

print("=" * 100)
print("SEARCHING JUPYTER CELL HISTORY FOR FVG CONTEXT CONTRACT AUDIT")
print("=" * 100)

_hits = []

for _i, _cell in enumerate(_ih):

    _text = _cell.lower()

    if (
        "fvg" in _text
        and "context" in _text
        and "audit" in _text
    ):
        _hits.append((_i, _cell))

print()
print("MATCHING CELLS :", len(_hits))
print()

for _i, _cell in _hits:
    print("=" * 100)
    print(f"CELL [{_i}]")
    print("=" * 100)
    print(_cell[:12000])
    print()

print("=" * 100)

In [ ]:
# ============================================================
# FVG CONTEXT ENGINE V1
# RE-RUN ORIGINAL CONTRACT AUDIT
# ON PATCHED OUTPUT
# EXACT ORIGINAL AUDIT LOGIC
# ============================================================

_original_audit_source = _ih[323]

_expected_old_line = (
    "df = fvg_context_engine_v1.copy(deep=True)"
)

_expected_new_line = (
    "df = fvg_context_engine_v1_patched.copy(deep=True)"
)

if _expected_old_line not in _original_audit_source:
    raise RuntimeError(
        "Original Contract Audit source was not found "
        "in Cell [323]. No audit executed."
    )

_patched_audit_source = _original_audit_source.replace(
    _expected_old_line,
    _expected_new_line,
    1,
)

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — ORIGINAL CONTRACT AUDIT")
print("TARGET : PATCHED OUTPUT")
print("=" * 100)
print()
print("Audit source : Cell [323]")
print("Audit logic  : UNCHANGED")
print("Target       : fvg_context_engine_v1_patched")
print()
print("-" * 100)

exec(
    _patched_audit_source,
    globals(),
)

print("-" * 100)
print("ORIGINAL CONTRACT AUDIT COMPLETED ON PATCHED OUTPUT")
print("=" * 100)

In [ ]:
# ============================================================
# LOCATE ORIGINAL FVG CONTEXT ENGINE V1 REGRESSION GATE
# SEARCH JUPYTER CELL HISTORY
# ============================================================

print("=" * 100)
print("SEARCHING JUPYTER HISTORY FOR FVG CONTEXT REGRESSION GATE")
print("=" * 100)

_regression_hits = []

for _i, _cell in enumerate(_ih):

    _text = _cell.lower()

    if (
        "fvg" in _text
        and "context" in _text
        and "regression" in _text
        and "gate" in _text
    ):
        _regression_hits.append((_i, _cell))

print()
print("MATCHING CELLS :", len(_regression_hits))
print()

for _i, _cell in _regression_hits:

    print("=" * 100)
    print(f"CELL [{_i}]")
    print("=" * 100)

    print(_cell[:16000])

print("=" * 100)

In [ ]:
# ============================================================
# LOCATE ORIGINAL FVG CONTEXT ENGINE V1 REGRESSION GATE
# SECONDARY HISTORY SEARCH
# ============================================================

print("=" * 100)
print("SEARCHING JUPYTER HISTORY FOR FVG CONTEXT REGRESSION CHECKS")
print("=" * 100)

_gate_hits = []

for _i, _cell in enumerate(_ih):

    # Skip the current search cells themselves
    if _i in {339, 340}:
        continue

    _text = _cell.lower()

    # Stronger indicators of an actual regression gate
    has_fvg_context = (
        "fvg_context_engine_v1" in _text
        or "fvg_context_engine_v1_new" in _text
    )

    has_checks = (
        "passed" in _text
        and "failed" in _text
    )

    has_regression_signal = any(
        token in _text
        for token in [
            "row count preservation",
            "id preservation",
            "identity preservation",
            "schema",
            "causality",
            "regression gate",
            "final regression",
        ]
    )

    if (
        has_fvg_context
        and has_checks
        and has_regression_signal
    ):
        _gate_hits.append((_i, _cell))

print()
print("MATCHING CELLS :", len(_gate_hits))
print()

for _i, _cell in _gate_hits:

    print("=" * 100)
    print(f"CELL [{_i}]")
    print("=" * 100)

    print(_cell[:20000])

print("=" * 100)

In [ ]:
# ============================================================
# FVG CONTEXT ENGINE V1
# REGRESSION GATE V1
# PATCH-AWARE / READ-ONLY
# ============================================================

import pandas as pd

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — REGRESSION GATE V1")
print("=" * 100)

regression_audit = []


def regression_check(name, condition, detail=""):
    status = "PASSED" if bool(condition) else "FAILED"

    regression_audit.append(
        {
            "check": name,
            "status": status,
            "detail": detail,
        }
    )

    print(f"{name:<55} : {status}")

    if detail:
        print(f"{'':<55}   {detail}")


# ============================================================
# 1. ROW COUNT PRESERVATION
# ============================================================

regression_check(
    "ROW COUNT PRESERVATION",
    len(fvg_context_engine_v1_patched)
    == len(fvg_context_engine_v1),
    (
        f"Old={len(fvg_context_engine_v1)} / "
        f"Patched={len(fvg_context_engine_v1_patched)}"
    ),
)


# ============================================================
# 2. COLUMN COUNT PRESERVATION
# ============================================================

regression_check(
    "COLUMN COUNT PRESERVATION",
    len(fvg_context_engine_v1_patched.columns)
    == len(fvg_context_engine_v1.columns),
    (
        f"Old={len(fvg_context_engine_v1.columns)} / "
        f"Patched={len(fvg_context_engine_v1_patched.columns)}"
    ),
)


# ============================================================
# 3. COLUMN ORDER PRESERVATION
# ============================================================

regression_check(
    "COLUMN ORDER PRESERVATION",
    fvg_context_engine_v1_patched.columns.tolist()
    == fvg_context_engine_v1.columns.tolist(),
)


# ============================================================
# 4. FVG ID SET PRESERVATION
# ============================================================

old_ids = set(fvg_context_engine_v1["fvg_id"])
new_ids = set(fvg_context_engine_v1_patched["fvg_id"])

regression_check(
    "FVG ID SET PRESERVATION",
    old_ids == new_ids,
    f"Old={len(old_ids)} / Patched={len(new_ids)}",
)


# ============================================================
# 5. FVG ID ORDER PRESERVATION
# ============================================================

regression_check(
    "FVG ID ORDER PRESERVATION",
    fvg_context_engine_v1["fvg_id"].tolist()
    == fvg_context_engine_v1_patched["fvg_id"].tolist(),
)


# ============================================================
# 6. INDEX PRESERVATION
# ============================================================

regression_check(
    "INDEX PRESERVATION",
    fvg_context_engine_v1.index.equals(
        fvg_context_engine_v1_patched.index
    ),
)


# ============================================================
# 7. DTYPE PRESERVATION
# ============================================================

dtype_differences = []

for col in fvg_context_engine_v1.columns:

    old_dtype = fvg_context_engine_v1[col].dtype
    new_dtype = fvg_context_engine_v1_patched[col].dtype

    if old_dtype != new_dtype:

        dtype_differences.append(
            {
                "column": col,
                "old_dtype": str(old_dtype),
                "new_dtype": str(new_dtype),
            }
        )

regression_check(
    "DTYPE PRESERVATION",
    len(dtype_differences) == 0,
    f"dtype differences={len(dtype_differences)}",
)


# ============================================================
# 8. AUTHORIZED DELTA — EXACTLY ONE CELL
# ============================================================

old_df = fvg_context_engine_v1.reset_index(drop=True)
new_df = fvg_context_engine_v1_patched.reset_index(drop=True)

delta_records = []

for col in old_df.columns:

    for i in range(len(old_df)):

        old_value = old_df.iloc[i][col]
        new_value = new_df.iloc[i][col]

        if pd.isna(old_value) and pd.isna(new_value):
            continue

        if old_value != new_value:

            delta_records.append(
                {
                    "row": i,
                    "fvg_id": new_df.iloc[i]["fvg_id"],
                    "column": col,
                    "old_value": old_value,
                    "new_value": new_value,
                }
            )

delta_df = pd.DataFrame(delta_records)

authorized_delta = (
    len(delta_df) == 1
    and delta_df.iloc[0]["column"] == "bias_alignment"
    and delta_df.iloc[0]["old_value"] == "OPPOSED"
    and delta_df.iloc[0]["new_value"] == "UNKNOWN"
)

regression_check(
    "AUTHORIZED PATCH DELTA",
    authorized_delta,
    f"actual cell differences={len(delta_df)}",
)


# ============================================================
# 9. ALL OTHER CELLS IDENTICAL
# ============================================================

changed_columns = (
    set(delta_df["column"])
    if len(delta_df)
    else set()
)

other_columns_unchanged = (
    changed_columns == {"bias_alignment"}
)

regression_check(
    "NON-TARGET CELLS PRESERVED",
    other_columns_unchanged,
    f"changed columns={sorted(changed_columns)}",
)


# ============================================================
# 10. PATCHED CONTRACT AUDIT
# ============================================================

# The canonical Contract Audit above already passed 24/24
# against fvg_context_engine_v1_patched.
#
# Reconstruct that result from the current audit dataframe.

contract_gate_available = (
    "audit_df" in globals()
    and isinstance(audit_df, pd.DataFrame)
)

contract_gate_passed = (
    contract_gate_available
    and len(audit_df) == 24
    and (audit_df["status"] == "PASSED").all()
)

regression_check(
    "PATCHED CONTRACT AUDIT",
    contract_gate_passed,
    (
        f"checks={len(audit_df) if contract_gate_available else 0} / "
        f"failed={(
            (audit_df['status'] == 'FAILED').sum()
            if contract_gate_available
            else 'N/A'
        )}"
    ),
)


# ============================================================
# FINAL REGRESSION RESULT
# ============================================================

regression_gate_df = pd.DataFrame(regression_audit)

failed_regression = regression_gate_df[
    regression_gate_df["status"] == "FAILED"
]

print()
print("=" * 100)
print("FINAL REGRESSION GATE RESULT")
print("=" * 100)

print("TOTAL CHECKS :", len(regression_gate_df))
print(
    "PASSED       :",
    (regression_gate_df["status"] == "PASSED").sum()
)
print("FAILED       :", len(failed_regression))

print()

if len(failed_regression) == 0:

    print(
        "FVG CONTEXT ENGINE V1 REGRESSION GATE : PASSED"
    )

else:

    print(
        "FVG CONTEXT ENGINE V1 REGRESSION GATE : FAILED"
    )

    print()
    print("FAILED CHECKS:")
    print(
        failed_regression.to_string(index=False)
    )

print("=" * 100)

print()
print("AUTHORIZED DELTA")
print("-" * 100)

if len(delta_df):
    print(delta_df.to_string(index=False))
else:
    print("No differences found.")

print("=" * 100)

In [ ]:
# ============================================================
# FVG CONTEXT ENGINE V1
# CANONICAL BUILDER PATCH
# READ-ONLY / NO OUTPUT OVERWRITE
# ============================================================

import inspect
import pandas as pd

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — CANONICAL BUILDER PATCH")
print("=" * 100)


# ------------------------------------------------------------
# 1. Capture current canonical builder source
# ------------------------------------------------------------

_original_source = inspect.getsource(
    build_fvg_context_engine_v1
)


# ------------------------------------------------------------
# 2. Define exact semantic patch
# ------------------------------------------------------------

_old_block = '''        if active_bias == fvg_direction:
            bias_alignment = "ALIGNED"

        elif pd.isna(active_bias):
            bias_alignment = "UNKNOWN"

        else:
            bias_alignment = "OPPOSED"
'''


_new_block = '''        if active_bias == fvg_direction:
            bias_alignment = "ALIGNED"

        elif active_bias == "UNKNOWN" or pd.isna(active_bias):
            bias_alignment = "UNKNOWN"

        else:
            bias_alignment = "OPPOSED"
'''


# ------------------------------------------------------------
# 3. Verify patch target exists exactly once
# ------------------------------------------------------------

occurrences = _original_source.count(_old_block)

print()
print("PATCH TARGET OCCURRENCES :", occurrences)

assert occurrences == 1, (
    "Canonical builder patch target was not found "
    "exactly once."
)


# ------------------------------------------------------------
# 4. Apply patch to source
# ------------------------------------------------------------

_patched_source = _original_source.replace(
    _old_block,
    _new_block,
    1,
)


# ------------------------------------------------------------
# 5. Preserve original global namespace
# ------------------------------------------------------------

_patched_namespace = (
    build_fvg_context_engine_v1.__globals__.copy()
)


# ------------------------------------------------------------
# 6. Compile patched canonical builder
# ------------------------------------------------------------

exec(
    _patched_source,
    _patched_namespace,
)


build_fvg_context_engine_v1_canonical = (
    _patched_namespace[
        "build_fvg_context_engine_v1"
    ]
)


# ------------------------------------------------------------
# 7. Basic function validation
# ------------------------------------------------------------

assert callable(
    build_fvg_context_engine_v1_canonical
)

assert (
    "_require_columns"
    in build_fvg_context_engine_v1_canonical.__globals__
)


# ------------------------------------------------------------
# RESULT
# ------------------------------------------------------------

print()
print("CANONICAL BUILDER PATCH      : PASSED")
print("GLOBAL NAMESPACE PRESERVED   : PASSED")
print("FUNCTION COMPILED            : PASSED")
print()
print(
    "Original builder remains untouched."
)
print(
    "No FVG Context output has been overwritten."
)

print("=" * 100)

In [ ]:
# ============================================================
# FVG CONTEXT ENGINE V1
# CANONICAL REBUILD
# NO OVERWRITE OF EXISTING OUTPUT
# ============================================================

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — CANONICAL REBUILD")
print("=" * 100)


fvg_context_engine_v1_canonical = (
    build_fvg_context_engine_v1_canonical(
        fvg_registry=fvg_lifecycle_registry_v1,
        liquidity_registry=unified_liquidity_registry_v1,
        structural_bos=structural_bos_v2,
        structural_choch=structural_choch_v2,
        structure_state=structure_state_v2,
    )
)


# ============================================================
# BASIC OUTPUT AUDIT
# ============================================================

print()
print("SHAPE :", fvg_context_engine_v1_canonical.shape)
print(
    "COLUMNS :",
    len(fvg_context_engine_v1_canonical.columns),
)

print()
print("BIAS ALIGNMENT DISTRIBUTION")
print(
    fvg_context_engine_v1_canonical[
        "bias_alignment"
    ].value_counts(dropna=False)
    .sort_index()
)


# ============================================================
# UNKNOWN BIAS DIAGNOSTIC
# ============================================================

unknown_bias_rows = (
    fvg_context_engine_v1_canonical[
        fvg_context_engine_v1_canonical[
            "active_bias"
        ] == "UNKNOWN"
    ]
)

print()
print("UNKNOWN ACTIVE BIAS ROWS :", len(unknown_bias_rows))

if len(unknown_bias_rows):

    print(
        unknown_bias_rows[
            [
                "fvg_id",
                "fvg_type",
                "created_at",
                "active_bias",
                "bias_alignment",
            ]
        ].to_string(index=False)
    )


# ============================================================
# BASIC ASSERTIONS
# ============================================================

assert (
    fvg_context_engine_v1_canonical.shape
    == fvg_context_engine_v1_patched.shape
)

assert (
    fvg_context_engine_v1_canonical.columns.tolist()
    == fvg_context_engine_v1_patched.columns.tolist()
)

assert (
    fvg_context_engine_v1_canonical["fvg_id"].tolist()
    == fvg_context_engine_v1_patched["fvg_id"].tolist()
)


print()
print("CANONICAL REBUILD : PASSED")
print()
print(
    "Existing fvg_context_engine_v1 remains untouched."
)
print(
    "Existing fvg_context_engine_v1_patched remains untouched."
)

print("=" * 100)

In [ ]:
# ============================================================
# FVG CONTEXT ENGINE V1
# CANONICAL EQUIVALENCE AUDIT
# ============================================================

import pandas as pd

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — CANONICAL EQUIVALENCE AUDIT")
print("=" * 100)


old_df = fvg_context_engine_v1_patched.reset_index(drop=True)
new_df = fvg_context_engine_v1_canonical.reset_index(drop=True)


# ============================================================
# 1. SHAPE
# ============================================================

shape_match = (
    old_df.shape == new_df.shape
)

print()
print(
    "SHAPE PRESERVATION".ljust(55),
    ":",
    "PASSED" if shape_match else "FAILED"
)


# ============================================================
# 2. COLUMN ORDER
# ============================================================

column_match = (
    old_df.columns.tolist()
    == new_df.columns.tolist()
)

print(
    "COLUMN ORDER PRESERVATION".ljust(55),
    ":",
    "PASSED" if column_match else "FAILED"
)


# ============================================================
# 3. FVG ID ORDER
# ============================================================

id_match = (
    old_df["fvg_id"].tolist()
    == new_df["fvg_id"].tolist()
)

print(
    "FVG ID ORDER PRESERVATION".ljust(55),
    ":",
    "PASSED" if id_match else "FAILED"
)


# ============================================================
# 4. INDEX
# ============================================================

index_match = (
    fvg_context_engine_v1_patched.index.equals(
        fvg_context_engine_v1_canonical.index
    )
)

print(
    "INDEX PRESERVATION".ljust(55),
    ":",
    "PASSED" if index_match else "FAILED"
)


# ============================================================
# 5. DTYPE
# ============================================================

dtype_differences = []

for col in old_df.columns:

    if old_df[col].dtype != new_df[col].dtype:

        dtype_differences.append(
            {
                "column": col,
                "patched_dtype": str(
                    old_df[col].dtype
                ),
                "canonical_dtype": str(
                    new_df[col].dtype
                ),
            }
        )

dtype_match = len(dtype_differences) == 0

print(
    "DTYPE PRESERVATION".ljust(55),
    ":",
    "PASSED" if dtype_match else "FAILED"
)

if dtype_differences:

    print(
        pd.DataFrame(
            dtype_differences
        ).to_string(index=False)
    )


# ============================================================
# 6. CELL-BY-CELL DIFFERENCE
# ============================================================

delta_records = []

for col in old_df.columns:

    for i in range(len(old_df)):

        old_value = old_df.iloc[i][col]
        new_value = new_df.iloc[i][col]

        if pd.isna(old_value) and pd.isna(new_value):
            continue

        if old_value != new_value:

            delta_records.append(
                {
                    "row": i,
                    "fvg_id": new_df.iloc[i]["fvg_id"],
                    "column": col,
                    "patched_value": old_value,
                    "canonical_value": new_value,
                }
            )


delta_df = pd.DataFrame(delta_records)


# ============================================================
# 7. FINAL RESULT
# ============================================================

print()
print("-" * 100)

print(
    "TOTAL CELL DIFFERENCES :",
    len(delta_df)
)

if len(delta_df):

    print()
    print(
        delta_df.to_string(index=False)
    )

print()
print("-" * 100)

exact_match = (
    shape_match
    and column_match
    and id_match
    and index_match
    and dtype_match
    and len(delta_df) == 0
)

print(
    "CANONICAL EQUIVALENCE :",
    "PASSED" if exact_match else "FAILED"
)

print("=" * 100)

assert exact_match, (
    "Canonical builder output does not exactly "
    "match the previously validated patched output."
)

In [ ]:
# ============================================================
# FVG CONTEXT ENGINE V1
# CANONICAL OUTPUT PROMOTION + FREEZE
# ============================================================

print("=" * 100)
print("FVG CONTEXT ENGINE V1 — CANONICAL PROMOTION")
print("=" * 100)


# ============================================================
# 1. PRESERVE PRE-PATCH OUTPUT
# ============================================================

fvg_context_engine_v1_pre_patch = (
    fvg_context_engine_v1.copy(deep=True)
)


print()
print(
    "PRE-PATCH OUTPUT BACKUP          : CREATED"
)


# ============================================================
# 2. PROMOTE CANONICAL OUTPUT
# ============================================================

fvg_context_engine_v1 = (
    fvg_context_engine_v1_canonical.copy(deep=True)
)


print(
    "CANONICAL OUTPUT PROMOTION       : PASSED"
)


# ============================================================
# 3. VERIFY PROMOTED OUTPUT
# ============================================================

assert (
    fvg_context_engine_v1.equals(
        fvg_context_engine_v1_canonical
    )
)

assert (
    fvg_context_engine_v1.shape
    == (102, 37)
)

assert (
    fvg_context_engine_v1["fvg_id"].is_unique
)

print(
    "PROMOTED OUTPUT INTEGRITY        : PASSED"
)


# ============================================================
# 4. VERIFY CRITICAL SEMANTIC FIX
# ============================================================

unknown_rows = fvg_context_engine_v1[
    fvg_context_engine_v1["active_bias"] == "UNKNOWN"
]

assert len(unknown_rows) == 1

assert (
    unknown_rows.iloc[0]["bias_alignment"]
    == "UNKNOWN"
)

print(
    "UNKNOWN BIAS SEMANTICS           : PASSED"
)


# ============================================================
# 5. FINAL DISTRIBUTION
# ============================================================

print()
print("FINAL BIAS ALIGNMENT DISTRIBUTION")
print("-" * 100)

print(
    fvg_context_engine_v1[
        "bias_alignment"
    ].value_counts(dropna=False)
    .sort_index()
)


# ============================================================
# 6. FREEZE MARKER
# ============================================================

FVG_CONTEXT_ENGINE_V1_FROZEN = True
FVG_CONTEXT_ENGINE_V1_VERSION = "1.0.0"

print()
print(
    "FVG_CONTEXT_ENGINE_V1_FROZEN     :",
    FVG_CONTEXT_ENGINE_V1_FROZEN,
)

print(
    "FVG_CONTEXT_ENGINE_V1_VERSION    :",
    FVG_CONTEXT_ENGINE_V1_VERSION,
)

print("=" * 100)

print()
print(
    "FVG CONTEXT ENGINE V1 : FROZEN"
)

print("=" * 100)

In [ ]:
# FVG Qualification Engine V1

In [ ]:
# ======================================================================================
# FVG QUALIFICATION ENGINE V1 — CONTRACT
# ======================================================================================

FVG_QUALIFICATION_V1_ALLOWED_STRUCTURE = {
    "ALIGNED",
    "OPPOSED",
    "UNKNOWN",
}

FVG_QUALIFICATION_V1_ALLOWED_BOS = {
    "ALIGNED",
    "OPPOSED",
    "UNKNOWN",
}

FVG_QUALIFICATION_V1_ALLOWED_CHOCH = {
    "ALIGNED",
    "OPPOSED",
    "UNKNOWN",
}

FVG_QUALIFICATION_V1_ALLOWED_LIQUIDITY = {
    "PRESENT",
    "ABSENT",
}

FVG_QUALIFICATION_V1_ALLOWED_LIFECYCLE = {
    "FRESH",
    "MITIGATED",
    "FILLED",
}

FVG_QUALIFICATION_V1_IDENTITY_COLUMNS = [
    "fvg_id",
    "fvg_type",
    "fvg_direction",
    "created_at",
]

FVG_QUALIFICATION_V1_CONTEXT_COLUMNS = [
    "active_bias",
    "bias_alignment",
    "context_bos_id",
    "context_bos_direction",
    "bos_alignment",
    "context_choch_id",
    "context_choch_direction",
    "choch_alignment",
    "historical_active_liquidity_count",
    "liquidity_below_count",
    "liquidity_above_count",
    "liquidity_overlap_count",
]

FVG_QUALIFICATION_V1_OUTPUT_COLUMNS = [
    # Identity
    "fvg_id",
    "fvg_type",
    "fvg_direction",
    "created_at",

    # Lifecycle
    "lifecycle_state",

    # Structure
    "active_bias",
    "bias_alignment",

    # BOS
    "context_bos_id",
    "context_bos_direction",
    "bos_alignment",

    # CHoCH
    "context_choch_id",
    "context_choch_direction",
    "choch_alignment",

    # Liquidity context
    "historical_active_liquidity_count",
    "liquidity_below_count",
    "liquidity_above_count",
    "liquidity_overlap_count",

    # Qualification states
    "structure_qualification",
    "bos_qualification",
    "choch_qualification",
    "liquidity_qualification",
    "lifecycle_qualification",
]

In [ ]:
print("=" * 100)
print("FVG QUALIFICATION ENGINE V1 — CONTRACT")
print("=" * 100)

print(
    "Structure states :",
    FVG_QUALIFICATION_V1_ALLOWED_STRUCTURE
)

print(
    "BOS states       :",
    FVG_QUALIFICATION_V1_ALLOWED_BOS
)

print(
    "CHoCH states     :",
    FVG_QUALIFICATION_V1_ALLOWED_CHOCH
)

print(
    "Liquidity states :",
    FVG_QUALIFICATION_V1_ALLOWED_LIQUIDITY
)

print(
    "Lifecycle states :",
    FVG_QUALIFICATION_V1_ALLOWED_LIFECYCLE
)

print(
    "Output columns   :",
    len(FVG_QUALIFICATION_V1_OUTPUT_COLUMNS)
)

print("=" * 100)

In [ ]:
# ======================================================================================
# FVG QUALIFICATION ENGINE V1 — BUILDER
# ======================================================================================

def build_fvg_qualification_engine_v1(
    fvg_context: pd.DataFrame,
) -> pd.DataFrame:

    required_columns = (
        FVG_QUALIFICATION_V1_IDENTITY_COLUMNS
        + ["lifecycle_state"]
        + FVG_QUALIFICATION_V1_CONTEXT_COLUMNS
    )

    missing = [
        col
        for col in required_columns
        if col not in fvg_context.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required FVG Context columns: {missing}"
        )

    df = fvg_context.copy(deep=True)

    # ------------------------------------------------------------------
    # UTC normalization
    # ------------------------------------------------------------------

    df["created_at"] = pd.to_datetime(
        df["created_at"],
        utc=True,
        errors="coerce",
    )

    if df["created_at"].isna().any():
        raise ValueError(
            "created_at contains invalid or missing timestamps."
        )

    # ------------------------------------------------------------------
    # Identity validation
    # ------------------------------------------------------------------

    if df["fvg_id"].isna().any():
        raise ValueError(
            "fvg_id contains null values."
        )

    if not df["fvg_id"].is_unique:
        raise ValueError(
            "fvg_id must be unique."
        )

    # ------------------------------------------------------------------
    # Direction validation
    # ------------------------------------------------------------------

    if not df["fvg_direction"].isin(
        {"BULLISH", "BEARISH"}
    ).all():

        raise ValueError(
            "Invalid fvg_direction detected."
        )

    # ------------------------------------------------------------------
    # Structure qualification
    # ------------------------------------------------------------------

    df["structure_qualification"] = (
        df["bias_alignment"]
    )

    # ------------------------------------------------------------------
    # BOS qualification
    # ------------------------------------------------------------------

    df["bos_qualification"] = (
        df["bos_alignment"]
    )

    # ------------------------------------------------------------------
    # CHoCH qualification
    # ------------------------------------------------------------------

    df["choch_qualification"] = (
        df["choch_alignment"]
    )

    # ------------------------------------------------------------------
    # Liquidity qualification
    # ------------------------------------------------------------------

    if (
        df["historical_active_liquidity_count"]
        < 0
    ).any():

        raise ValueError(
            "historical_active_liquidity_count cannot be negative."
        )

    df["liquidity_qualification"] = (
        df["historical_active_liquidity_count"]
        .gt(0)
        .map({
            True: "PRESENT",
            False: "ABSENT",
        })
    )

    # ------------------------------------------------------------------
    # Lifecycle qualification
    # ------------------------------------------------------------------

    if not df["lifecycle_state"].isin(
        FVG_QUALIFICATION_V1_ALLOWED_LIFECYCLE
    ).all():

        raise ValueError(
            "Invalid lifecycle_state detected."
        )

    df["lifecycle_qualification"] = (
        df["lifecycle_state"]
    )

    # ------------------------------------------------------------------
    # Validate qualification vocabularies
    # ------------------------------------------------------------------

    if not df["structure_qualification"].isin(
        FVG_QUALIFICATION_V1_ALLOWED_STRUCTURE
    ).all():

        raise ValueError(
            "Invalid structure_qualification detected."
        )

    if not df["bos_qualification"].isin(
        FVG_QUALIFICATION_V1_ALLOWED_BOS
    ).all():

        raise ValueError(
            "Invalid bos_qualification detected."
        )

    if not df["choch_qualification"].isin(
        FVG_QUALIFICATION_V1_ALLOWED_CHOCH
    ).all():

        raise ValueError(
            "Invalid choch_qualification detected."
        )

    if not df["liquidity_qualification"].isin(
        FVG_QUALIFICATION_V1_ALLOWED_LIQUIDITY
    ).all():

        raise ValueError(
            "Invalid liquidity_qualification detected."
        )

    # ------------------------------------------------------------------
    # Final projection
    # ------------------------------------------------------------------

    result = df[
        FVG_QUALIFICATION_V1_OUTPUT_COLUMNS
    ].copy()

    return result

In [ ]:
# ============================================================
# FVG QUALIFICATION ENGINE V1
# BUILDER CONTRACT DIAGNOSTIC
# READ-ONLY
# ============================================================

import inspect

print("=" * 100)
print("FVG QUALIFICATION ENGINE V1 — BUILDER CONTRACT DIAGNOSTIC")
print("=" * 100)

source = inspect.getsource(
    build_fvg_qualification_engine_v1
)

print()
print("FUNCTION SOURCE")
print("-" * 100)
print(source)

print()
print("=" * 100)
print("CURRENT FVG CONTEXT SCHEMA")
print("=" * 100)

print(
    fvg_context_engine_v1.columns.tolist()
)

print()
print("CONTEXT SHAPE :", fvg_context_engine_v1.shape)

print("=" * 100)

In [ ]:
# ============================================================
# FVG QUALIFICATION ENGINE V1
# CONTRACT CONSTANTS DIAGNOSTIC
# READ-ONLY
# ============================================================

print("=" * 100)
print("FVG QUALIFICATION ENGINE V1 — CONTRACT CONSTANTS DIAGNOSTIC")
print("=" * 100)


# ============================================================
# CONTEXT COLUMNS
# ============================================================

print()
print("FVG_QUALIFICATION_V1_CONTEXT_COLUMNS")
print("-" * 100)

print(
    FVG_QUALIFICATION_V1_CONTEXT_COLUMNS
)


# ============================================================
# IDENTITY COLUMNS
# ============================================================

print()
print("FVG_QUALIFICATION_V1_IDENTITY_COLUMNS")
print("-" * 100)

print(
    FVG_QUALIFICATION_V1_IDENTITY_COLUMNS
)


# ============================================================
# OUTPUT COLUMNS
# ============================================================

print()
print("FVG_QUALIFICATION_V1_OUTPUT_COLUMNS")
print("-" * 100)

print(
    FVG_QUALIFICATION_V1_OUTPUT_COLUMNS
)


# ============================================================
# VOCABULARIES
# ============================================================

print()
print("ALLOWED STRUCTURE")
print("-" * 100)
print(
    FVG_QUALIFICATION_V1_ALLOWED_STRUCTURE
)

print()
print("ALLOWED BOS")
print("-" * 100)
print(
    FVG_QUALIFICATION_V1_ALLOWED_BOS
)

print()
print("ALLOWED CHoCH")
print("-" * 100)
print(
    FVG_QUALIFICATION_V1_ALLOWED_CHOCH
)

print()
print("ALLOWED LIQUIDITY")
print("-" * 100)
print(
    FVG_QUALIFICATION_V1_ALLOWED_LIQUIDITY
)

print()
print("ALLOWED LIFECYCLE")
print("-" * 100)
print(
    FVG_QUALIFICATION_V1_ALLOWED_LIFECYCLE
)


# ============================================================
# SCHEMA DIFFERENCE
# ============================================================

required_columns = (
    FVG_QUALIFICATION_V1_IDENTITY_COLUMNS
    + ["lifecycle_state"]
    + FVG_QUALIFICATION_V1_CONTEXT_COLUMNS
)

missing_from_context = [
    col
    for col in required_columns
    if col not in fvg_context_engine_v1.columns
]

print()
print("=" * 100)
print("REQUIRED COLUMNS MISSING FROM FROZEN CONTEXT")
print("=" * 100)

print(
    missing_from_context
)


# ============================================================
# ACTUAL USAGE IN BUILDER
# ============================================================

print()
print("=" * 100)
print("IMPORTANT")
print("=" * 100)

print(
    "The builder source shows that only "
    "'historical_active_liquidity_count' "
    "is actually used among the liquidity count fields."
)

print(
    "No liquidity_below_count / "
    "liquidity_above_count / "
    "liquidity_overlap_count "
    "is consumed by the current builder logic."
)

print("=" * 100)

In [ ]:
# ============================================================
# FVG QUALIFICATION ENGINE V1
# HISTORICAL LIQUIDITY COUNT SOURCE SEARCH
# READ-ONLY
# ============================================================

print("=" * 100)
print("SEARCHING NOTEBOOK HISTORY FOR")
print("historical_active_liquidity_count")
print("=" * 100)

matches = []

for i, cell in enumerate(_ih):

    if "historical_active_liquidity_count" in cell:

        matches.append(i)


print()
print("MATCHING CELLS :", matches)

print()
print("=" * 100)

if matches:

    for cell_no in matches:

        print()
        print(f"--- CELL [{cell_no}] ---")
        print("-" * 100)

        print(_ih[cell_no])

else:

    print(
        "No previous notebook cell contains "
        "'historical_active_liquidity_count'."
    )

print("=" * 100)

In [ ]:
# ============================================================
# FVG QUALIFICATION ENGINE V1
# PRIOR CONTRACT / REGRESSION HISTORY SEARCH
# READ-ONLY
# ============================================================

print("=" * 100)
print("FVG QUALIFICATION ENGINE V1 — PRIOR HISTORY SEARCH")
print("=" * 100)

keywords = [
    "FVG QUALIFICATION ENGINE V1",
    "FVG_QUALIFICATION_V1",
    "REGRESSION GATE",
    "QUALIFICATION CONTRACT",
    "liquidity_qualification",
]

matches = {}

for keyword in keywords:

    found = []

    for i, cell in enumerate(_ih):

        if keyword in cell:

            found.append(i)

    matches[keyword] = found


for keyword, cells in matches.items():

    print()
    print("-" * 100)
    print(keyword)
    print("-" * 100)
    print(cells)


print()
print("=" * 100)
print("UNIQUE RELEVANT CELLS")
print("=" * 100)

unique_cells = sorted(
    set(
        cell_no
        for cells in matches.values()
        for cell_no in cells
    )
)

print(unique_cells)

print("=" * 100)

In [ ]:
print("=" * 100)
print("FVG QUALIFICATION ENGINE V1 — PRIOR IMPLEMENTATION INSPECTION")
print("=" * 100)

for cell_no in [348, 535, 536, 537, 538, 539, 540, 541, 542]:
    print("\n" + "-" * 100)
    print(f"CELL [{cell_no}]")
    print("-" * 100)
    try:
        print(In[cell_no])
    except Exception as e:
        print(f"ERROR: {e}")

In [ ]:
print("=" * 100)
print("FVG QUALIFICATION ENGINE V1 — REGRESSION HISTORY INSPECTION")
print("=" * 100)

for cell_no in [527, 528, 529]:
    print("\n" + "-" * 100)
    print(f"CELL [{cell_no}]")
    print("-" * 100)

    try:
        print(In[cell_no])
    except Exception as e:
        print(f"ERROR: {e}")

print("\n" + "=" * 100)
print("EARLIER CANDIDATE GATES")
print("=" * 100)

for cell_no in [396, 418, 426]:
    print("\n" + "-" * 100)
    print(f"CELL [{cell_no}]")
    print("-" * 100)

    try:
        print(In[cell_no])
    except Exception as e:
        print(f"ERROR: {e}")

In [ ]:
print("=" * 100)
print("FVG QUALIFICATION ENGINE V1 — SUCCESS / OUTPUT HISTORY SEARCH")
print("=" * 100)

keywords = [
    "fvg_qualification_engine_v1",
    "STRUCTURE QUALIFICATION",
    "BOS QUALIFICATION",
    "CHoCH QUALIFICATION",
    "LIQUIDITY QUALIFICATION",
    "LIFECYCLE QUALIFICATION",
    "FINAL REGRESSION",
    "QUALIFICATION GATE",
]

for keyword in keywords:

    print("\n" + "-" * 100)
    print(f"SEARCH : {keyword}")
    print("-" * 100)

    hits = []

    for i, cell in enumerate(_ih):
        if keyword.lower() in cell.lower():
            hits.append(i)

    print("MATCHES :", hits)

In [ ]:
print("=" * 100)
print("FVG QUALIFICATION ENGINE V1 — FINAL QUALIFICATION GATE")
print("=" * 100)

print(In[545])

In [ ]:
print("=" * 100)
print("LOCATING ACTUAL FVG QUALIFICATION GATE")
print("=" * 100)

for i, cell in enumerate(_ih):

    text = cell.lower()

    if (
        "qualification gate" in text
        and "fvg" in text
    ):
        print("\n" + "=" * 100)
        print(f"ACTUAL MATCH — CELL [{i}]")
        print("=" * 100)
        print(cell)

In [ ]:
print("=" * 100)
print("FVG QUALIFICATION INPUT V1 — LIQUIDITY SOURCE AUDIT")
print("=" * 100)

print("\nUNIFIED LIQUIDITY REGISTRY SHAPE")
print("-" * 100)
print(unified_liquidity_registry_v1.shape)

print("\nUNIFIED LIQUIDITY REGISTRY COLUMNS")
print("-" * 100)
print(unified_liquidity_registry_v1.columns.tolist())

print("\nSTATE VOCABULARY")
print("-" * 100)
print(
    unified_liquidity_registry_v1["state"]
    .value_counts(dropna=False)
)

print("\nLIQUIDITY SOURCE VOCABULARY")
print("-" * 100)
print(
    unified_liquidity_registry_v1["liquidity_source"]
    .value_counts(dropna=False)
)

print("\nLIQUIDITY TYPE VOCABULARY")
print("-" * 100)
print(
    unified_liquidity_registry_v1["liquidity_type"]
    .value_counts(dropna=False)
)

print("\nEVENT TIME NULLABILITY")
print("-" * 100)
print(
    unified_liquidity_registry_v1["event_time"]
    .isna()
    .value_counts(dropna=False)
)

print("\nSAMPLE")
print("-" * 100)
print(
    unified_liquidity_registry_v1.head(10).to_string(index=False)
)

print("=" * 100)

In [ ]:
print("=" * 100)
print("FVG QUALIFICATION INPUT V1 — BUILD")
print("=" * 100)


FVG_QUALIFICATION_INPUT_V1_VERSION = "1.0.0"

FVG_QUALIFICATION_INPUT_V1_REQUIRED_FVG_COLUMNS = [
    "fvg_id",
    "fvg_type",
    "created_at",
    "lifecycle_state",
    "lower_bound",
    "upper_bound",
]

FVG_QUALIFICATION_INPUT_V1_REQUIRED_LIQUIDITY_COLUMNS = [
    "level_id",
    "liquidity_type",
    "level_price",
    "activation_time",
    "event_time",
]


def build_fvg_qualification_input_v1(
    fvg_context: pd.DataFrame,
    unified_liquidity_registry: pd.DataFrame,
) -> pd.DataFrame:

    # ------------------------------------------------------------------
    # 1. Validate FVG Context source
    # ------------------------------------------------------------------

    missing_fvg = [
        col
        for col in FVG_QUALIFICATION_INPUT_V1_REQUIRED_FVG_COLUMNS
        if col not in fvg_context.columns
    ]

    if missing_fvg:
        raise ValueError(
            f"Missing required FVG Context columns: {missing_fvg}"
        )

    # ------------------------------------------------------------------
    # 2. Validate Liquidity Registry source
    # ------------------------------------------------------------------

    missing_liq = [
        col
        for col in FVG_QUALIFICATION_INPUT_V1_REQUIRED_LIQUIDITY_COLUMNS
        if col not in unified_liquidity_registry.columns
    ]

    if missing_liq:
        raise ValueError(
            f"Missing required Liquidity Registry columns: {missing_liq}"
        )

    # ------------------------------------------------------------------
    # 3. Defensive copies
    # ------------------------------------------------------------------

    fvg = fvg_context.copy(deep=True)
    liq = unified_liquidity_registry.copy(deep=True)

    # ------------------------------------------------------------------
    # 4. Normalize timestamps
    # ------------------------------------------------------------------

    fvg["created_at"] = pd.to_datetime(
        fvg["created_at"],
        utc=True,
        errors="coerce",
    )

    liq["activation_time"] = pd.to_datetime(
        liq["activation_time"],
        utc=True,
        errors="coerce",
    )

    liq["event_time"] = pd.to_datetime(
        liq["event_time"],
        utc=True,
        errors="coerce",
    )

    # ------------------------------------------------------------------
    # 5. Basic validation
    # ------------------------------------------------------------------

    if fvg["created_at"].isna().any():
        raise ValueError(
            "FVG created_at contains invalid or missing timestamps."
        )

    if fvg["fvg_id"].isna().any():
        raise ValueError(
            "FVG fvg_id contains null values."
        )

    if not fvg["fvg_id"].is_unique:
        raise ValueError(
            "FVG fvg_id must be unique."
        )

    if not fvg["fvg_type"].isin(
        {"BULLISH_FVG", "BEARISH_FVG"}
    ).all():
        raise ValueError(
            "Invalid fvg_type detected."
        )

    if fvg["lifecycle_state"].isna().any():
        raise ValueError(
            "FVG lifecycle_state contains null values."
        )

    if liq["activation_time"].isna().any():
        raise ValueError(
            "Liquidity activation_time contains invalid or missing timestamps."
        )

    if liq["level_price"].isna().any():
        raise ValueError(
            "Liquidity level_price contains null values."
        )

    # ------------------------------------------------------------------
    # 6. Derive deterministic FVG direction
    # ------------------------------------------------------------------

    fvg["fvg_direction"] = fvg["fvg_type"].map({
        "BULLISH_FVG": "BULLISH",
        "BEARISH_FVG": "BEARISH",
    })

    # ------------------------------------------------------------------
    # 7. Historical liquidity reconstruction
    # ------------------------------------------------------------------

    historical_active_counts = []
    below_counts = []
    above_counts = []
    overlap_counts = []

    for _, row in fvg.iterrows():

        created_at = row["created_at"]
        lower_bound = row["lower_bound"]
        upper_bound = row["upper_bound"]

        historical_active = liq[
            (liq["activation_time"] <= created_at)
            &
            (
                liq["event_time"].isna()
                |
                (liq["event_time"] > created_at)
            )
        ]

        below = historical_active[
            historical_active["level_price"] < lower_bound
        ]

        above = historical_active[
            historical_active["level_price"] > upper_bound
        ]

        overlap = historical_active[
            (historical_active["level_price"] >= lower_bound)
            &
            (historical_active["level_price"] <= upper_bound)
        ]

        historical_active_counts.append(
            len(historical_active)
        )

        below_counts.append(
            len(below)
        )

        above_counts.append(
            len(above)
        )

        overlap_counts.append(
            len(overlap)
        )

    # ------------------------------------------------------------------
    # 8. Attach qualification inputs
    # ------------------------------------------------------------------

    fvg["historical_active_liquidity_count"] = (
        historical_active_counts
    )

    fvg["liquidity_below_count"] = (
        below_counts
    )

    fvg["liquidity_above_count"] = (
        above_counts
    )

    fvg["liquidity_overlap_count"] = (
        overlap_counts
    )

    # ------------------------------------------------------------------
    # 9. Canonical qualification input schema
    # ------------------------------------------------------------------

    output_columns = [
        "fvg_id",
        "fvg_type",
        "fvg_direction",
        "created_at",
        "lifecycle_state",

        "active_bias",
        "bias_alignment",

        "context_bos_id",
        "context_bos_direction",
        "bos_alignment",

        "context_choch_id",
        "context_choch_direction",
        "choch_alignment",

        "historical_active_liquidity_count",
        "liquidity_below_count",
        "liquidity_above_count",
        "liquidity_overlap_count",
    ]

    result = fvg[output_columns].copy()

    return result

In [ ]:
fvg_qualification_input_v1 = (
    build_fvg_qualification_input_v1(
        fvg_context=fvg_context_engine_v1,
        unified_liquidity_registry=unified_liquidity_registry_v1,
    )
)

print("=" * 100)
print("FVG QUALIFICATION INPUT V1 — OUTPUT")
print("=" * 100)

print("\nSHAPE")
print("-" * 100)
print(fvg_qualification_input_v1.shape)

print("\nCOLUMNS")
print("-" * 100)
print(fvg_qualification_input_v1.columns.tolist())

print("\nFVG DIRECTION")
print("-" * 100)
print(
    fvg_qualification_input_v1["fvg_direction"]
    .value_counts(dropna=False)
)

print("\nHISTORICAL ACTIVE LIQUIDITY")
print("-" * 100)
print(
    fvg_qualification_input_v1[
        "historical_active_liquidity_count"
    ].value_counts().sort_index()
)

print("\nLIQUIDITY QUALIFICATION INPUT SUMMARY")
print("-" * 100)
print(
    fvg_qualification_input_v1[
        [
            "historical_active_liquidity_count",
            "liquidity_below_count",
            "liquidity_above_count",
            "liquidity_overlap_count",
        ]
    ].describe()
)

print("\nSAMPLE")
print("-" * 100)
print(
    fvg_qualification_input_v1.head(10).to_string(index=False)
)

print("=" * 100)

In [ ]:
print("=" * 100)
print("FVG QUALIFICATION INPUT V1 — CONTRACT AUDIT")
print("=" * 100)

audit = {}

# ------------------------------------------------------------------
# 1. Shape / row preservation
# ------------------------------------------------------------------

audit["ROW_COUNT_PRESERVATION"] = (
    len(fvg_qualification_input_v1)
    == len(fvg_context_engine_v1)
)

# ------------------------------------------------------------------
# 2. Exact column contract
# ------------------------------------------------------------------

expected_columns = [
    "fvg_id",
    "fvg_type",
    "fvg_direction",
    "created_at",
    "lifecycle_state",
    "active_bias",
    "bias_alignment",
    "context_bos_id",
    "context_bos_direction",
    "bos_alignment",
    "context_choch_id",
    "context_choch_direction",
    "choch_alignment",
    "historical_active_liquidity_count",
    "liquidity_below_count",
    "liquidity_above_count",
    "liquidity_overlap_count",
]

audit["COLUMN_CONTRACT"] = (
    fvg_qualification_input_v1.columns.tolist()
    == expected_columns
)

# ------------------------------------------------------------------
# 3. FVG ID preservation
# ------------------------------------------------------------------

audit["FVG_ID_PRESERVATION"] = (
    fvg_qualification_input_v1["fvg_id"].tolist()
    == fvg_context_engine_v1["fvg_id"].tolist()
)

# ------------------------------------------------------------------
# 4. FVG ID uniqueness
# ------------------------------------------------------------------

audit["FVG_ID_UNIQUENESS"] = (
    fvg_qualification_input_v1["fvg_id"].is_unique
)

# ------------------------------------------------------------------
# 5. FVG direction contract
# ------------------------------------------------------------------

audit["FVG_DIRECTION_CONTRACT"] = (
    fvg_qualification_input_v1["fvg_direction"]
    .isin({"BULLISH", "BEARISH"})
    .all()
)

# ------------------------------------------------------------------
# 6. FVG direction derivation
# ------------------------------------------------------------------

expected_direction = (
    fvg_qualification_input_v1["fvg_type"]
    .map({
        "BULLISH_FVG": "BULLISH",
        "BEARISH_FVG": "BEARISH",
    })
)

audit["FVG_DIRECTION_DERIVATION"] = (
    fvg_qualification_input_v1["fvg_direction"]
    .equals(expected_direction)
)

# ------------------------------------------------------------------
# 7. UTC timestamp contract
# ------------------------------------------------------------------

audit["UTC_TIMESTAMP_CONTRACT"] = (
    str(
        fvg_qualification_input_v1["created_at"].dtype
    ).startswith("datetime64[")
    and
    fvg_qualification_input_v1["created_at"].dt.tz is not None
)

# ------------------------------------------------------------------
# 8. Lifecycle contract
# ------------------------------------------------------------------

audit["LIFECYCLE_CONTRACT"] = (
    fvg_qualification_input_v1["lifecycle_state"]
    .isin({"FRESH", "MITIGATED", "FILLED"})
    .all()
)

# ------------------------------------------------------------------
# 9. Structure alignment contract
# ------------------------------------------------------------------

audit["STRUCTURE_ALIGNMENT_CONTRACT"] = (
    fvg_qualification_input_v1["bias_alignment"]
    .isin({"ALIGNED", "OPPOSED", "UNKNOWN"})
    .all()
)

# ------------------------------------------------------------------
# 10. BOS alignment contract
# ------------------------------------------------------------------

audit["BOS_ALIGNMENT_CONTRACT"] = (
    fvg_qualification_input_v1["bos_alignment"]
    .isin({"ALIGNED", "OPPOSED", "UNKNOWN"})
    .all()
)

# ------------------------------------------------------------------
# 11. CHoCH alignment contract
# ------------------------------------------------------------------

audit["CHOCH_ALIGNMENT_CONTRACT"] = (
    fvg_qualification_input_v1["choch_alignment"]
    .isin({"ALIGNED", "OPPOSED", "UNKNOWN"})
    .all()
)

# ------------------------------------------------------------------
# 12. Historical liquidity counts non-negative
# ------------------------------------------------------------------

liquidity_count_columns = [
    "historical_active_liquidity_count",
    "liquidity_below_count",
    "liquidity_above_count",
    "liquidity_overlap_count",
]

audit["LIQUIDITY_COUNTS_NON_NEGATIVE"] = (
    fvg_qualification_input_v1[
        liquidity_count_columns
    ]
    .ge(0)
    .all()
    .all()
)

# ------------------------------------------------------------------
# 13. Historical count decomposition
# ------------------------------------------------------------------

audit["LIQUIDITY_COUNT_DECOMPOSITION"] = (
    fvg_qualification_input_v1[
        "historical_active_liquidity_count"
    ]
    ==
    (
        fvg_qualification_input_v1[
            "liquidity_below_count"
        ]
        +
        fvg_qualification_input_v1[
            "liquidity_above_count"
        ]
        +
        fvg_qualification_input_v1[
            "liquidity_overlap_count"
        ]
    )
).all()

# ------------------------------------------------------------------
# 14. No NaN in required derived inputs
# ------------------------------------------------------------------

audit["NO_NULL_DERIVED_INPUTS"] = (
    fvg_qualification_input_v1[
        [
            "fvg_direction",
            "historical_active_liquidity_count",
            "liquidity_below_count",
            "liquidity_above_count",
            "liquidity_overlap_count",
        ]
    ]
    .notna()
    .all()
    .all()
)

# ------------------------------------------------------------------
# 15. Context integrity
# ------------------------------------------------------------------

context_columns = [
    "active_bias",
    "bias_alignment",
    "context_bos_id",
    "context_bos_direction",
    "bos_alignment",
    "context_choch_id",
    "context_choch_direction",
    "choch_alignment",
]

audit["CONTEXT_INTEGRITY"] = (
    fvg_qualification_input_v1[
        context_columns
    ].reset_index(drop=True)
    ==
    fvg_context_engine_v1[
        context_columns
    ].reset_index(drop=True)
).all().all()

# ------------------------------------------------------------------
# FINAL RESULT
# ------------------------------------------------------------------

print()

for name, passed in audit.items():
    print(
        f"{name:<45} : "
        f"{'PASSED' if passed else 'FAILED'}"
    )

print()

print("=" * 100)
print(
    "TOTAL CHECKS :",
    len(audit)
)

print(
    "PASSED       :",
    sum(audit.values())
)

print(
    "FAILED       :",
    len(audit) - sum(audit.values())
)

print("=" * 100)

In [ ]:
print("=" * 100)
print("FVG QUALIFICATION INPUT V1 — CONTEXT DELTA AUDIT")
print("=" * 100)

context_columns = [
    "active_bias",
    "bias_alignment",
    "context_bos_id",
    "context_bos_direction",
    "bos_alignment",
    "context_choch_id",
    "context_choch_direction",
    "choch_alignment",
]

input_context = (
    fvg_qualification_input_v1[
        ["fvg_id"] + context_columns
    ]
    .reset_index(drop=True)
)

source_context = (
    fvg_context_engine_v1[
        ["fvg_id"] + context_columns
    ]
    .reset_index(drop=True)
)

print("\nROW COUNTS")
print("-" * 100)
print("INPUT :", len(input_context))
print("SOURCE:", len(source_context))

print("\nCOLUMN-BY-COLUMN DIFFERENCES")
print("-" * 100)

total_differences = 0

for col in context_columns:

    left = input_context[col]
    right = source_context[col]

    equal = left.eq(right)

    # NaN == NaN should count as equal
    equal = equal | (left.isna() & right.isna())

    diff_count = int((~equal).sum())

    print(
        f"{col:<35} : "
        f"{'PASSED' if diff_count == 0 else 'FAILED'}"
        f"  | differences = {diff_count}"
    )

    total_differences += diff_count

print("\nTOTAL CONTEXT CELL DIFFERENCES :", total_differences)

# ------------------------------------------------------------------
# Exact differing rows
# ------------------------------------------------------------------

row_diff_mask = pd.Series(False, index=input_context.index)

for col in context_columns:

    left = input_context[col]
    right = source_context[col]

    equal = left.eq(right)
    equal = equal | (left.isna() & right.isna())

    row_diff_mask |= ~equal

diff_rows = input_context.loc[row_diff_mask].copy()

print("\nDIFFERING ROW COUNT :", len(diff_rows))

if len(diff_rows) > 0:

    print("\nEXACT DIFFERENCES")
    print("-" * 100)

    for idx in diff_rows.index:

        print("\n" + "-" * 100)
        print(f"ROW INDEX : {idx}")
        print(f"FVG ID    : {input_context.loc[idx, 'fvg_id']}")

        for col in context_columns:

            left = input_context.loc[idx, col]
            right = source_context.loc[idx, col]

            same = (
                (pd.isna(left) and pd.isna(right))
                or left == right
            )

            if not same:
                print(
                    f"{col:<35} | "
                    f"INPUT={left!r} | "
                    f"SOURCE={right!r}"
                )

print("\n" + "=" * 100)

In [ ]:
print("=" * 100)
print("FVG QUALIFICATION INPUT V1 — REGRESSION GATE")
print("=" * 100)

checks = {}

# ------------------------------------------------------------------
# 1. Row count
# ------------------------------------------------------------------

checks["ROW COUNT PRESERVATION"] = (
    len(fvg_qualification_input_v1)
    == len(fvg_context_engine_v1)
)

# ------------------------------------------------------------------
# 2. Exact schema
# ------------------------------------------------------------------

expected_columns = [
    "fvg_id",
    "fvg_type",
    "fvg_direction",
    "created_at",
    "lifecycle_state",
    "active_bias",
    "bias_alignment",
    "context_bos_id",
    "context_bos_direction",
    "bos_alignment",
    "context_choch_id",
    "context_choch_direction",
    "choch_alignment",
    "historical_active_liquidity_count",
    "liquidity_below_count",
    "liquidity_above_count",
    "liquidity_overlap_count",
]

checks["COLUMN CONTRACT"] = (
    fvg_qualification_input_v1.columns.tolist()
    == expected_columns
)

# ------------------------------------------------------------------
# 3. ID uniqueness
# ------------------------------------------------------------------

checks["FVG ID UNIQUENESS"] = (
    fvg_qualification_input_v1["fvg_id"].is_unique
)

# ------------------------------------------------------------------
# 4. ID preservation
# ------------------------------------------------------------------

checks["FVG ID PRESERVATION"] = (
    fvg_qualification_input_v1["fvg_id"].tolist()
    ==
    fvg_context_engine_v1["fvg_id"].tolist()
)

# ------------------------------------------------------------------
# 5. Direction
# ------------------------------------------------------------------

checks["FVG DIRECTION CONTRACT"] = (
    fvg_qualification_input_v1["fvg_direction"]
    .isin({"BULLISH", "BEARISH"})
    .all()
)

checks["FVG DIRECTION DERIVATION"] = (
    fvg_qualification_input_v1["fvg_direction"].tolist()
    ==
    fvg_qualification_input_v1["fvg_type"]
    .map({
        "BULLISH_FVG": "BULLISH",
        "BEARISH_FVG": "BEARISH",
    })
    .tolist()
)

# ------------------------------------------------------------------
# 6. Lifecycle
# ------------------------------------------------------------------

checks["LIFECYCLE CONTRACT"] = (
    fvg_qualification_input_v1["lifecycle_state"]
    .isin({"FRESH", "MITIGATED", "FILLED"})
    .all()
)

# ------------------------------------------------------------------
# 7. Context semantic preservation
# ------------------------------------------------------------------

context_columns = [
    "active_bias",
    "bias_alignment",
    "context_bos_id",
    "context_bos_direction",
    "bos_alignment",
    "context_choch_id",
    "context_choch_direction",
    "choch_alignment",
]

context_equal = True

for col in context_columns:

    left = (
        fvg_qualification_input_v1[col]
        .reset_index(drop=True)
    )

    right = (
        fvg_context_engine_v1[col]
        .reset_index(drop=True)
    )

    equal = left.eq(right)
    equal = equal | (left.isna() & right.isna())

    if not equal.all():
        context_equal = False
        break

checks["CONTEXT PRESERVATION"] = context_equal

# ------------------------------------------------------------------
# 8. UTC timestamps
# ------------------------------------------------------------------

created_at = fvg_qualification_input_v1["created_at"]

checks["UTC TIME CONTRACT"] = (
    pd.api.types.is_datetime64tz_dtype(created_at)
    and
    str(created_at.dt.tz) == "UTC"
)

# ------------------------------------------------------------------
# 9. Liquidity count non-negative
# ------------------------------------------------------------------

liq_count_columns = [
    "historical_active_liquidity_count",
    "liquidity_below_count",
    "liquidity_above_count",
    "liquidity_overlap_count",
]

checks["LIQUIDITY COUNT NON-NEGATIVE"] = (
    fvg_qualification_input_v1[
        liq_count_columns
    ]
    .ge(0)
    .all()
    .all()
)

# ------------------------------------------------------------------
# 10. Liquidity decomposition
# ------------------------------------------------------------------

checks["LIQUIDITY COUNT DECOMPOSITION"] = (
    fvg_qualification_input_v1[
        "historical_active_liquidity_count"
    ]
    ==
    (
        fvg_qualification_input_v1[
            "liquidity_below_count"
        ]
        +
        fvg_qualification_input_v1[
            "liquidity_above_count"
        ]
        +
        fvg_qualification_input_v1[
            "liquidity_overlap_count"
        ]
    )
).all()

# ------------------------------------------------------------------
# 11. Derived fields non-null
# ------------------------------------------------------------------

checks["DERIVED INPUTS NON-NULL"] = (
    fvg_qualification_input_v1[
        [
            "fvg_direction",
            "historical_active_liquidity_count",
            "liquidity_below_count",
            "liquidity_above_count",
            "liquidity_overlap_count",
        ]
    ]
    .notna()
    .all()
    .all()
)

# ------------------------------------------------------------------
# FINAL
# ------------------------------------------------------------------

print()

for name, passed in checks.items():
    print(
        f"{name:<45} : "
        f"{'PASSED' if passed else 'FAILED'}"
    )

print("\n" + "=" * 100)

passed_count = sum(checks.values())
total_count = len(checks)

print(f"TOTAL CHECKS : {total_count}")
print(f"PASSED       : {passed_count}")
print(f"FAILED       : {total_count - passed_count}")

print("=" * 100)

In [ ]:
print("=" * 100)
print("FVG QUALIFICATION ENGINE V1 — BUILD")
print("=" * 100)

fvg_qualification_engine_v1 = (
    build_fvg_qualification_engine_v1(
        fvg_context=fvg_qualification_input_v1,
    )
)

print("\nSHAPE")
print("-" * 100)
print(fvg_qualification_engine_v1.shape)

print("\nCOLUMNS")
print("-" * 100)
print(fvg_qualification_engine_v1.columns.tolist())

print("\nSTRUCTURE QUALIFICATION")
print("-" * 100)
print(
    fvg_qualification_engine_v1[
        "structure_qualification"
    ].value_counts(dropna=False)
)

print("\nBOS QUALIFICATION")
print("-" * 100)
print(
    fvg_qualification_engine_v1[
        "bos_qualification"
    ].value_counts(dropna=False)
)

print("\nCHOCH QUALIFICATION")
print("-" * 100)
print(
    fvg_qualification_engine_v1[
        "choch_qualification"
    ].value_counts(dropna=False)
)

print("\nLIQUIDITY QUALIFICATION")
print("-" * 100)
print(
    fvg_qualification_engine_v1[
        "liquidity_qualification"
    ].value_counts(dropna=False)
)

print("\nLIFECYCLE QUALIFICATION")
print("-" * 100)
print(
    fvg_qualification_engine_v1[
        "lifecycle_qualification"
    ].value_counts(dropna=False)
)

print("\nSAMPLE")
print("-" * 100)
print(
    fvg_qualification_engine_v1
    .head(10)
    .to_string(index=False)
)

print("=" * 100)

In [ ]:
print("=" * 100)
print("FVG QUALIFICATION ENGINE V1 — REGRESSION GATE")
print("=" * 100)

checks = {}

# ------------------------------------------------------------------
# Canonical output contract
# ------------------------------------------------------------------

expected_columns = [
    "fvg_id",
    "fvg_type",
    "fvg_direction",
    "created_at",
    "lifecycle_state",
    "active_bias",
    "bias_alignment",
    "context_bos_id",
    "context_bos_direction",
    "bos_alignment",
    "context_choch_id",
    "context_choch_direction",
    "choch_alignment",
    "historical_active_liquidity_count",
    "liquidity_below_count",
    "liquidity_above_count",
    "liquidity_overlap_count",
    "structure_qualification",
    "bos_qualification",
    "choch_qualification",
    "liquidity_qualification",
    "lifecycle_qualification",
]

# ------------------------------------------------------------------
# 1. Row count
# ------------------------------------------------------------------

checks["ROW COUNT PRESERVATION"] = (
    len(fvg_qualification_engine_v1)
    == len(fvg_qualification_input_v1)
)

# ------------------------------------------------------------------
# 2. Exact schema
# ------------------------------------------------------------------

checks["OUTPUT COLUMN CONTRACT"] = (
    fvg_qualification_engine_v1.columns.tolist()
    == expected_columns
)

# ------------------------------------------------------------------
# 3. FVG ID uniqueness
# ------------------------------------------------------------------

checks["FVG ID UNIQUENESS"] = (
    fvg_qualification_engine_v1["fvg_id"].is_unique
)

# ------------------------------------------------------------------
# 4. FVG identity preservation
# ------------------------------------------------------------------

identity_columns = [
    "fvg_id",
    "fvg_type",
    "fvg_direction",
    "created_at",
]

identity_equal = True

for col in identity_columns:

    left = (
        fvg_qualification_engine_v1[col]
        .reset_index(drop=True)
    )

    right = (
        fvg_qualification_input_v1[col]
        .reset_index(drop=True)
    )

    equal = left.eq(right)
    equal = equal | (left.isna() & right.isna())

    if not equal.all():
        identity_equal = False
        break

checks["IDENTITY PRESERVATION"] = identity_equal

# ------------------------------------------------------------------
# 5. Lifecycle preservation
# ------------------------------------------------------------------

checks["LIFECYCLE PRESERVATION"] = (
    fvg_qualification_engine_v1["lifecycle_state"].tolist()
    ==
    fvg_qualification_input_v1["lifecycle_state"].tolist()
)

# ------------------------------------------------------------------
# 6. Structure qualification contract
# ------------------------------------------------------------------

checks["STRUCTURE QUALIFICATION CONTRACT"] = (
    fvg_qualification_engine_v1[
        "structure_qualification"
    ]
    .isin({"ALIGNED", "OPPOSED", "UNKNOWN"})
    .all()
)

# ------------------------------------------------------------------
# 7. Structure qualification derivation
# ------------------------------------------------------------------

checks["STRUCTURE QUALIFICATION DERIVATION"] = (
    fvg_qualification_engine_v1[
        "structure_qualification"
    ].tolist()
    ==
    fvg_qualification_input_v1[
        "bias_alignment"
    ].tolist()
)

# ------------------------------------------------------------------
# 8. BOS qualification contract
# ------------------------------------------------------------------

checks["BOS QUALIFICATION CONTRACT"] = (
    fvg_qualification_engine_v1[
        "bos_qualification"
    ]
    .isin({"ALIGNED", "OPPOSED", "UNKNOWN"})
    .all()
)

# ------------------------------------------------------------------
# 9. BOS qualification derivation
# ------------------------------------------------------------------

checks["BOS QUALIFICATION DERIVATION"] = (
    fvg_qualification_engine_v1[
        "bos_qualification"
    ].tolist()
    ==
    fvg_qualification_input_v1[
        "bos_alignment"
    ].tolist()
)

# ------------------------------------------------------------------
# 10. CHoCH qualification contract
# ------------------------------------------------------------------

checks["CHOCH QUALIFICATION CONTRACT"] = (
    fvg_qualification_engine_v1[
        "choch_qualification"
    ]
    .isin({"ALIGNED", "OPPOSED", "UNKNOWN"})
    .all()
)

# ------------------------------------------------------------------
# 11. CHoCH qualification derivation
# ------------------------------------------------------------------

checks["CHOCH QUALIFICATION DERIVATION"] = (
    fvg_qualification_engine_v1[
        "choch_qualification"
    ].tolist()
    ==
    fvg_qualification_input_v1[
        "choch_alignment"
    ].tolist()
)

# ------------------------------------------------------------------
# 12. Liquidity qualification contract
# ------------------------------------------------------------------

checks["LIQUIDITY QUALIFICATION CONTRACT"] = (
    fvg_qualification_engine_v1[
        "liquidity_qualification"
    ]
    .isin({"PRESENT", "ABSENT"})
    .all()
)

# ------------------------------------------------------------------
# 13. Liquidity qualification derivation
# ------------------------------------------------------------------

expected_liquidity = (
    fvg_qualification_input_v1[
        "historical_active_liquidity_count"
    ]
    .gt(0)
    .map({
        True: "PRESENT",
        False: "ABSENT",
    })
)

checks["LIQUIDITY QUALIFICATION DERIVATION"] = (
    fvg_qualification_engine_v1[
        "liquidity_qualification"
    ].tolist()
    ==
    expected_liquidity.tolist()
)

# ------------------------------------------------------------------
# 14. Lifecycle qualification contract
# ------------------------------------------------------------------

checks["LIFECYCLE QUALIFICATION CONTRACT"] = (
    fvg_qualification_engine_v1[
        "lifecycle_qualification"
    ]
    .isin({"FRESH", "MITIGATED", "FILLED"})
    .all()
)

# ------------------------------------------------------------------
# 15. Lifecycle qualification derivation
# ------------------------------------------------------------------

checks["LIFECYCLE QUALIFICATION DERIVATION"] = (
    fvg_qualification_engine_v1[
        "lifecycle_qualification"
    ].tolist()
    ==
    fvg_qualification_input_v1[
        "lifecycle_state"
    ].tolist()
)

# ------------------------------------------------------------------
# 16. Context preservation
# ------------------------------------------------------------------

context_columns = [
    "active_bias",
    "bias_alignment",
    "context_bos_id",
    "context_bos_direction",
    "bos_alignment",
    "context_choch_id",
    "context_choch_direction",
    "choch_alignment",
]

context_equal = True

for col in context_columns:

    left = (
        fvg_qualification_engine_v1[col]
        .reset_index(drop=True)
    )

    right = (
        fvg_qualification_input_v1[col]
        .reset_index(drop=True)
    )

    equal = left.eq(right)
    equal = equal | (left.isna() & right.isna())

    if not equal.all():
        context_equal = False
        break

checks["CONTEXT PRESERVATION"] = context_equal

# ------------------------------------------------------------------
# 17. Liquidity counts preservation
# ------------------------------------------------------------------

liquidity_count_columns = [
    "historical_active_liquidity_count",
    "liquidity_below_count",
    "liquidity_above_count",
    "liquidity_overlap_count",
]

counts_equal = True

for col in liquidity_count_columns:

    left = (
        fvg_qualification_engine_v1[col]
        .reset_index(drop=True)
    )

    right = (
        fvg_qualification_input_v1[col]
        .reset_index(drop=True)
    )

    if not left.equals(right):
        counts_equal = False
        break

checks["LIQUIDITY COUNT PRESERVATION"] = counts_equal

# ------------------------------------------------------------------
# 18. No null qualification outputs
# ------------------------------------------------------------------

qualification_columns = [
    "structure_qualification",
    "bos_qualification",
    "choch_qualification",
    "liquidity_qualification",
    "lifecycle_qualification",
]

checks["NO NULL QUALIFICATIONS"] = (
    fvg_qualification_engine_v1[
        qualification_columns
    ]
    .notna()
    .all()
    .all()
)

# ------------------------------------------------------------------
# 19. Identity / qualification row alignment
# ------------------------------------------------------------------

checks["ROW ORDER PRESERVATION"] = (
    fvg_qualification_engine_v1["fvg_id"].tolist()
    ==
    fvg_qualification_input_v1["fvg_id"].tolist()
)

# ------------------------------------------------------------------
# 20. FVG direction contract
# ------------------------------------------------------------------

checks["FVG DIRECTION CONTRACT"] = (
    fvg_qualification_engine_v1[
        "fvg_direction"
    ]
    .isin({"BULLISH", "BEARISH"})
    .all()
)

# ------------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------------

print()

for name, passed in checks.items():
    print(
        f"{name:<45} : "
        f"{'PASSED' if passed else 'FAILED'}"
    )

print("\n" + "=" * 100)

passed_count = sum(checks.values())
total_count = len(checks)

print(f"TOTAL CHECKS : {total_count}")
print(f"PASSED       : {passed_count}")
print(f"FAILED       : {total_count - passed_count}")

print("=" * 100)

In [ ]:
# ==================================================================================================
# FVG QUALIFICATION ENGINE V1 — SEMANTIC AUDIT V1
# ==================================================================================================

print("=" * 100)
print("FVG QUALIFICATION ENGINE V1 — SEMANTIC AUDIT V1")
print("=" * 100)

audit_results = {}

# --------------------------------------------------------------------------------------------------
# 1. STRUCTURE QUALIFICATION SEMANTIC
# --------------------------------------------------------------------------------------------------

structure_semantic = (
    fvg_qualification_engine_v1["structure_qualification"]
    == fvg_qualification_engine_v1["bias_alignment"]
).all()

audit_results["STRUCTURE SEMANTIC"] = structure_semantic

print(
    f"{'STRUCTURE SEMANTIC':50s}: "
    f"{'PASSED' if structure_semantic else 'FAILED'}"
)


# --------------------------------------------------------------------------------------------------
# 2. BOS QUALIFICATION SEMANTIC
# --------------------------------------------------------------------------------------------------

bos_semantic = (
    fvg_qualification_engine_v1["bos_qualification"]
    == fvg_qualification_engine_v1["bos_alignment"]
).all()

audit_results["BOS SEMANTIC"] = bos_semantic

print(
    f"{'BOS SEMANTIC':50s}: "
    f"{'PASSED' if bos_semantic else 'FAILED'}"
)


# --------------------------------------------------------------------------------------------------
# 3. CHOCH QUALIFICATION SEMANTIC
# --------------------------------------------------------------------------------------------------

choch_semantic = (
    fvg_qualification_engine_v1["choch_qualification"]
    == fvg_qualification_engine_v1["choch_alignment"]
).all()

audit_results["CHOCH SEMANTIC"] = choch_semantic

print(
    f"{'CHOCH SEMANTIC':50s}: "
    f"{'PASSED' if choch_semantic else 'FAILED'}"
)


# --------------------------------------------------------------------------------------------------
# 4. LIQUIDITY QUALIFICATION SEMANTIC
# --------------------------------------------------------------------------------------------------

liquidity_expected = (
    fvg_qualification_engine_v1["historical_active_liquidity_count"]
    .gt(0)
    .map({True: "PRESENT", False: "ABSENT"})
)

liquidity_semantic = (
    fvg_qualification_engine_v1["liquidity_qualification"]
    == liquidity_expected
).all()

audit_results["LIQUIDITY SEMANTIC"] = liquidity_semantic

print(
    f"{'LIQUIDITY SEMANTIC':50s}: "
    f"{'PASSED' if liquidity_semantic else 'FAILED'}"
)


# --------------------------------------------------------------------------------------------------
# 5. LIFECYCLE QUALIFICATION SEMANTIC
# --------------------------------------------------------------------------------------------------

lifecycle_semantic = (
    fvg_qualification_engine_v1["lifecycle_qualification"]
    == fvg_qualification_engine_v1["lifecycle_state"]
).all()

audit_results["LIFECYCLE SEMANTIC"] = lifecycle_semantic

print(
    f"{'LIFECYCLE SEMANTIC':50s}: "
    f"{'PASSED' if lifecycle_semantic else 'FAILED'}"
)


# --------------------------------------------------------------------------------------------------
# 6. FVG DIRECTION CONTRACT
# --------------------------------------------------------------------------------------------------

direction_expected = (
    fvg_qualification_engine_v1["fvg_type"]
    .map({
        "BULLISH_FVG": "BULLISH",
        "BEARISH_FVG": "BEARISH",
    })
)

direction_semantic = (
    fvg_qualification_engine_v1["fvg_direction"]
    == direction_expected
).all()

audit_results["FVG DIRECTION SEMANTIC"] = direction_semantic

print(
    f"{'FVG DIRECTION SEMANTIC':50s}: "
    f"{'PASSED' if direction_semantic else 'FAILED'}"
)


# --------------------------------------------------------------------------------------------------
# 7. UNKNOWN STRUCTURE CONSISTENCY
# --------------------------------------------------------------------------------------------------

unknown_structure_mask = (
    fvg_qualification_engine_v1["structure_qualification"] == "UNKNOWN"
)

unknown_structure_consistency = (
    fvg_qualification_engine_v1.loc[
        unknown_structure_mask,
        "active_bias"
    ]
    .isin(["UNKNOWN"])
).all()

audit_results["UNKNOWN STRUCTURE CONSISTENCY"] = unknown_structure_consistency

print(
    f"{'UNKNOWN STRUCTURE CONSISTENCY':50s}: "
    f"{'PASSED' if unknown_structure_consistency else 'FAILED'}"
)


# --------------------------------------------------------------------------------------------------
# 8. UNKNOWN BOS CONSISTENCY
# --------------------------------------------------------------------------------------------------

unknown_bos_mask = (
    fvg_qualification_engine_v1["bos_qualification"] == "UNKNOWN"
)

unknown_bos_consistency = (
    fvg_qualification_engine_v1.loc[
        unknown_bos_mask,
        "context_bos_id"
    ].isna()
).all()

audit_results["UNKNOWN BOS CONSISTENCY"] = unknown_bos_consistency

print(
    f"{'UNKNOWN BOS CONSISTENCY':50s}: "
    f"{'PASSED' if unknown_bos_consistency else 'FAILED'}"
)


# --------------------------------------------------------------------------------------------------
# 9. UNKNOWN CHOCH CONSISTENCY
# --------------------------------------------------------------------------------------------------

unknown_choch_mask = (
    fvg_qualification_engine_v1["choch_qualification"] == "UNKNOWN"
)

unknown_choch_consistency = (
    fvg_qualification_engine_v1.loc[
        unknown_choch_mask,
        "context_choch_id"
    ].isna()
).all()

audit_results["UNKNOWN CHOCH CONSISTENCY"] = unknown_choch_consistency

print(
    f"{'UNKNOWN CHOCH CONSISTENCY':50s}: "
    f"{'PASSED' if unknown_choch_consistency else 'FAILED'}"
)


# --------------------------------------------------------------------------------------------------
# 10. LIQUIDITY COUNT DECOMPOSITION
# --------------------------------------------------------------------------------------------------

liquidity_decomposition = (
    fvg_qualification_engine_v1["historical_active_liquidity_count"]
    ==
    (
        fvg_qualification_engine_v1["liquidity_below_count"]
        +
        fvg_qualification_engine_v1["liquidity_above_count"]
        +
        fvg_qualification_engine_v1["liquidity_overlap_count"]
    )
).all()

audit_results["LIQUIDITY COUNT DECOMPOSITION"] = liquidity_decomposition

print(
    f"{'LIQUIDITY COUNT DECOMPOSITION':50s}: "
    f"{'PASSED' if liquidity_decomposition else 'FAILED'}"
)


# --------------------------------------------------------------------------------------------------
# 11. QUALIFICATION VOCABULARY CONSISTENCY
# --------------------------------------------------------------------------------------------------

qualification_vocabulary = (
    fvg_qualification_engine_v1[
        [
            "structure_qualification",
            "bos_qualification",
            "choch_qualification",
            "liquidity_qualification",
            "lifecycle_qualification",
        ]
    ]
    .notna()
    .all()
    .all()
)

audit_results["QUALIFICATION VOCABULARY CONSISTENCY"] = qualification_vocabulary

print(
    f"{'QUALIFICATION VOCABULARY CONSISTENCY':50s}: "
    f"{'PASSED' if qualification_vocabulary else 'FAILED'}"
)


# --------------------------------------------------------------------------------------------------
# 12. ROW IDENTITY UNIQUENESS
# --------------------------------------------------------------------------------------------------

row_identity_unique = (
    fvg_qualification_engine_v1[
        ["fvg_id", "created_at"]
    ]
    .duplicated()
    .sum()
    == 0
)

audit_results["ROW IDENTITY UNIQUENESS"] = row_identity_unique

print(
    f"{'ROW IDENTITY UNIQUENESS':50s}: "
    f"{'PASSED' if row_identity_unique else 'FAILED'}"
)


# --------------------------------------------------------------------------------------------------
# 13. STRUCTURE ALIGNMENT DISTRIBUTION
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("STRUCTURE QUALIFICATION DISTRIBUTION")
print("-" * 100)

print(
    fvg_qualification_engine_v1[
        "structure_qualification"
    ].value_counts(dropna=False)
)


# --------------------------------------------------------------------------------------------------
# 14. BOS ALIGNMENT DISTRIBUTION
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("BOS QUALIFICATION DISTRIBUTION")
print("-" * 100)

print(
    fvg_qualification_engine_v1[
        "bos_qualification"
    ].value_counts(dropna=False)
)


# --------------------------------------------------------------------------------------------------
# 15. CHOCH ALIGNMENT DISTRIBUTION
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("CHOCH QUALIFICATION DISTRIBUTION")
print("-" * 100)

print(
    fvg_qualification_engine_v1[
        "choch_qualification"
    ].value_counts(dropna=False)
)


# --------------------------------------------------------------------------------------------------
# 16. LIQUIDITY QUALIFICATION DISTRIBUTION
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("LIQUIDITY QUALIFICATION DISTRIBUTION")
print("-" * 100)

print(
    fvg_qualification_engine_v1[
        "liquidity_qualification"
    ].value_counts(dropna=False)
)


# --------------------------------------------------------------------------------------------------
# 17. LIFECYCLE QUALIFICATION DISTRIBUTION
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("LIFECYCLE QUALIFICATION DISTRIBUTION")
print("-" * 100)

print(
    fvg_qualification_engine_v1[
        "lifecycle_qualification"
    ].value_counts(dropna=False)
)


# --------------------------------------------------------------------------------------------------
# 18. UNKNOWN / CONTEXT ANCHOR AUDIT
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("UNKNOWN / CONTEXT ANCHOR ROWS")
print("-" * 100)

anchor_columns = [
    "fvg_id",
    "fvg_type",
    "fvg_direction",
    "created_at",
    "active_bias",
    "structure_qualification",
    "context_bos_id",
    "context_bos_direction",
    "bos_qualification",
    "context_choch_id",
    "context_choch_direction",
    "choch_qualification",
    "historical_active_liquidity_count",
    "liquidity_qualification",
    "lifecycle_state",
]

anchor_mask = (
    (fvg_qualification_engine_v1["structure_qualification"] == "UNKNOWN")
    |
    (fvg_qualification_engine_v1["bos_qualification"] == "UNKNOWN")
    |
    (fvg_qualification_engine_v1["choch_qualification"] == "UNKNOWN")
)

display(
    fvg_qualification_engine_v1.loc[
        anchor_mask,
        anchor_columns
    ].sort_values("created_at")
)


# --------------------------------------------------------------------------------------------------
# 19. QUALIFICATION CROSS-TAB
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("STRUCTURE × BOS QUALIFICATION")
print("-" * 100)

display(
    pd.crosstab(
        fvg_qualification_engine_v1["structure_qualification"],
        fvg_qualification_engine_v1["bos_qualification"],
        dropna=False,
    )
)


print("\n" + "-" * 100)
print("STRUCTURE × CHOCH QUALIFICATION")
print("-" * 100)

display(
    pd.crosstab(
        fvg_qualification_engine_v1["structure_qualification"],
        fvg_qualification_engine_v1["choch_qualification"],
        dropna=False,
    )
)


# --------------------------------------------------------------------------------------------------
# 20. FINAL SEMANTIC AUDIT RESULT
# --------------------------------------------------------------------------------------------------

semantic_checks = [
    value
    for key, value in audit_results.items()
    if key
    not in {
        "STRUCTURE QUALIFICATION DISTRIBUTION",
        "BOS QUALIFICATION DISTRIBUTION",
        "CHOCH QUALIFICATION DISTRIBUTION",
        "LIQUIDITY QUALIFICATION DISTRIBUTION",
        "LIFECYCLE QUALIFICATION DISTRIBUTION",
    }
]

passed = sum(semantic_checks)
failed = len(semantic_checks) - passed

print("\n" + "=" * 100)
print("FVG QUALIFICATION ENGINE V1 — SEMANTIC AUDIT SUMMARY")
print("=" * 100)

print(f"TOTAL SEMANTIC CHECKS : {len(semantic_checks)}")
print(f"PASSED                 : {passed}")
print(f"FAILED                 : {failed}")

print("=" * 100)

if failed == 0:
    print("SEMANTIC AUDIT STATUS  : PASSED")
else:
    print("SEMANTIC AUDIT STATUS  : FAILED")

print("=" * 100)

In [ ]:
# ==================================================================================================
# FVG QUALIFICATION ENGINE V1 — FREEZE
# ==================================================================================================

FVG_QUALIFICATION_V1_FROZEN = True
FVG_QUALIFICATION_V1_VERSION = "1.0.0"

fvg_qualification_engine_v1_frozen = (
    fvg_qualification_engine_v1.copy(deep=True)
)

print("=" * 100)
print("FVG QUALIFICATION ENGINE V1 — FREEZE STATUS")
print("=" * 100)

print(f"FROZEN : {FVG_QUALIFICATION_V1_FROZEN}")
print(f"VERSION: {FVG_QUALIFICATION_V1_VERSION}")
print(f"SHAPE  : {fvg_qualification_engine_v1_frozen.shape}")
print("=" * 100)

print("\nQUALIFICATION DISTRIBUTIONS")
print("-" * 100)

for column in [
    "structure_qualification",
    "bos_qualification",
    "choch_qualification",
    "liquidity_qualification",
    "lifecycle_qualification",
]:
    print(f"\n{column}")
    print(
        fvg_qualification_engine_v1_frozen[column]
        .value_counts(dropna=False)
    )

In [ ]:
# FVG Proximity Engine V1 — Contract

In [ ]:
print("=" * 120)
print("FVG DATAFRAME SCHEMA AUDIT")
print("=" * 120)

for name in [
    "fvg_v1",
    "fvg_validated_v1",
    "fvg_lifecycle_reference_v1",
    "fvg_engine_v1",
    "fvg_lifecycle_registry_v1",
    "fvg_context_engine_v1",
    "fvg",
    "fvg_qualification_engine_v1",
]:
    obj = globals().get(name)

    if obj is not None and hasattr(obj, "columns"):
        print(f"\n{name}  →  {obj.shape}")
        print(list(obj.columns))

In [ ]:
# ==================================================================================================
# FVG QUALIFICATION DECISION ENGINE V1 — INPUT STATE AUDIT
# ==================================================================================================

print("=" * 110)
print("FVG QUALIFICATION DECISION ENGINE V1 — INPUT STATE AUDIT")
print("=" * 110)

df = fvg_qualification_engine_v1

required_columns = [
    "fvg_id",
    "fvg_type",
    "fvg_direction",
    "created_at",
    "lifecycle_state",
    "structure_qualification",
    "bos_qualification",
    "choch_qualification",
    "liquidity_qualification",
]

print("\nINPUT CONTRACT")
print("-" * 110)

missing = [
    col for col in required_columns
    if col not in df.columns
]

print(f"Required Columns : {len(required_columns)}")
print(f"Missing Columns  : {missing}")

# --------------------------------------------------------------------------------------------------
# STATE VOCABULARY
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 110)
print("STATE VOCABULARY")
print("-" * 110)

state_columns = [
    "structure_qualification",
    "bos_qualification",
    "choch_qualification",
    "liquidity_qualification",
    "lifecycle_state",
]

for column in state_columns:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))

# --------------------------------------------------------------------------------------------------
# UNIQUE COMBINATION SPACE
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 110)
print("UNIQUE QUALIFICATION COMBINATIONS")
print("-" * 110)

combination_columns = [
    "structure_qualification",
    "bos_qualification",
    "choch_qualification",
    "liquidity_qualification",
    "lifecycle_state",
]

qualification_combinations = (
    df[
        combination_columns
    ]
    .value_counts(dropna=False)
    .reset_index(name="count")
    .sort_values(
        combination_columns,
        kind="stable"
    )
    .reset_index(drop=True)
)

print(
    f"Unique combinations : "
    f"{len(qualification_combinations)}"
)

display(qualification_combinations)

# --------------------------------------------------------------------------------------------------
# FVG DIRECTION × QUALIFICATION
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 110)
print("FVG DIRECTION × STRUCTURE QUALIFICATION")
print("-" * 110)

display(
    pd.crosstab(
        df["fvg_direction"],
        df["structure_qualification"],
        dropna=False,
    )
)

print("\n" + "-" * 110)
print("FVG DIRECTION × BOS QUALIFICATION")
print("-" * 110)

display(
    pd.crosstab(
        df["fvg_direction"],
        df["bos_qualification"],
        dropna=False,
    )
)

print("\n" + "-" * 110)
print("FVG DIRECTION × CHOCH QUALIFICATION")
print("-" * 110)

display(
    pd.crosstab(
        df["fvg_direction"],
        df["choch_qualification"],
        dropna=False,
    )
)

print("\n" + "-" * 110)
print("FVG DIRECTION × LIQUIDITY QUALIFICATION")
print("-" * 110)

display(
    pd.crosstab(
        df["fvg_direction"],
        df["liquidity_qualification"],
        dropna=False,
    )
)

# --------------------------------------------------------------------------------------------------
# LIFECYCLE × QUALIFICATION
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 110)
print("LIFECYCLE × STRUCTURE QUALIFICATION")
print("-" * 110)

display(
    pd.crosstab(
        df["lifecycle_state"],
        df["structure_qualification"],
        dropna=False,
    )
)

print("\n" + "-" * 110)
print("LIFECYCLE × LIQUIDITY QUALIFICATION")
print("-" * 110)

display(
    pd.crosstab(
        df["lifecycle_state"],
        df["liquidity_qualification"],
        dropna=False,
    )
)

# --------------------------------------------------------------------------------------------------
# NULL AUDIT
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 110)
print("NULL AUDIT")
print("-" * 110)

null_counts = (
    df[
        required_columns
    ]
    .isna()
    .sum()
)

display(
    null_counts[
        null_counts > 0
    ]
)

print(
    f"Total NULL cells in required decision inputs : "
    f"{int(null_counts.sum())}"
)

# --------------------------------------------------------------------------------------------------
# FINAL
# --------------------------------------------------------------------------------------------------

input_contract_passed = (
    not missing
    and df["fvg_id"].notna().all()
    and df["fvg_id"].is_unique
    and df["fvg_direction"].isin(
        {"BULLISH", "BEARISH"}
    ).all()
    and df["structure_qualification"].isin(
        {"ALIGNED", "OPPOSED", "UNKNOWN"}
    ).all()
    and df["bos_qualification"].isin(
        {"ALIGNED", "OPPOSED", "UNKNOWN"}
    ).all()
    and df["choch_qualification"].isin(
        {"ALIGNED", "OPPOSED", "UNKNOWN"}
    ).all()
    and df["liquidity_qualification"].isin(
        {"PRESENT", "ABSENT"}
    ).all()
    and df["lifecycle_state"].isin(
        {"FRESH", "MITIGATED", "FILLED"}
    ).all()
)

print("\n" + "=" * 110)
print("INPUT STATE AUDIT RESULT")
print("=" * 110)

print(
    f"STATUS : "
    f"{'PASSED' if input_contract_passed else 'FAILED'}"
)

print("=" * 110)

In [ ]:
# ==================================================================================================
# FVG QUALIFICATION DECISION ENGINE V1
# DECISION MATRIX CONTRACT AUDIT V1
# ==================================================================================================

print("=" * 120)
print("FVG QUALIFICATION DECISION ENGINE V1 — DECISION MATRIX CONTRACT AUDIT V1")
print("=" * 120)


# --------------------------------------------------------------------------------------------------
# 1. DECISION VOCABULARY
# --------------------------------------------------------------------------------------------------

FVG_DECISION_V1_ALLOWED_DECISIONS = {
    "QUALIFIED",
    "CONDITIONAL",
    "DISQUALIFIED",
    "UNRESOLVED",
}

FVG_DECISION_V1_ALLOWED_REASONS = {
    "STRUCTURE_UNKNOWN",
    "FILLED_FVG",
    "STRUCTURE_OPPOSED",
    "ALIGNED_FRESH",
    "ALIGNED_MITIGATED",
}


# --------------------------------------------------------------------------------------------------
# 2. DETERMINISTIC DECISION MATRIX
# --------------------------------------------------------------------------------------------------

def classify_fvg_qualification_decision_v1(
    structure_qualification,
    lifecycle_state,
):
    """
    Deterministic V1 decision matrix.

    Primary decision dimensions:
        - structure_qualification
        - lifecycle_state

    Modifier dimensions intentionally excluded from hard gating:
        - bos_qualification
        - choch_qualification
        - liquidity_qualification
    """

    # Highest-priority unresolved condition
    if structure_qualification == "UNKNOWN":
        return "UNRESOLVED", "STRUCTURE_UNKNOWN"

    # Filled FVG is no longer an active candidate
    if lifecycle_state == "FILLED":
        return "DISQUALIFIED", "FILLED_FVG"

    # Structural opposition prevents clean qualification
    if structure_qualification == "OPPOSED":
        return "CONDITIONAL", "STRUCTURE_OPPOSED"

    # Aligned + fresh = cleanest qualification state
    if (
        structure_qualification == "ALIGNED"
        and lifecycle_state == "FRESH"
    ):
        return "QUALIFIED", "ALIGNED_FRESH"

    # Aligned + mitigated remains usable but is no longer fresh
    if (
        structure_qualification == "ALIGNED"
        and lifecycle_state == "MITIGATED"
    ):
        return "CONDITIONAL", "ALIGNED_MITIGATED"

    raise ValueError(
        "No Decision Matrix rule matched the supplied state."
    )


# --------------------------------------------------------------------------------------------------
# 3. MATRIX COVERAGE AUDIT
# --------------------------------------------------------------------------------------------------

matrix_input_columns = [
    "structure_qualification",
    "bos_qualification",
    "choch_qualification",
    "liquidity_qualification",
    "lifecycle_state",
]

matrix_df = (
    fvg_qualification_engine_v1[
        matrix_input_columns
    ]
    .drop_duplicates()
    .sort_values(
        matrix_input_columns,
        kind="stable"
    )
    .reset_index(drop=True)
)

matrix_results = []

for _, row in matrix_df.iterrows():

    decision, reason = classify_fvg_qualification_decision_v1(
        structure_qualification=row["structure_qualification"],
        lifecycle_state=row["lifecycle_state"],
    )

    matrix_results.append(
        {
            "structure_qualification":
                row["structure_qualification"],

            "bos_qualification":
                row["bos_qualification"],

            "choch_qualification":
                row["choch_qualification"],

            "liquidity_qualification":
                row["liquidity_qualification"],

            "lifecycle_state":
                row["lifecycle_state"],

            "decision":
                decision,

            "reason":
                reason,
        }
    )


decision_matrix_audit_v1 = pd.DataFrame(
    matrix_results
)


# --------------------------------------------------------------------------------------------------
# 4. COVERAGE CHECK
# --------------------------------------------------------------------------------------------------

coverage_passed = (
    len(decision_matrix_audit_v1)
    == len(matrix_df)
)

print("\n" + "-" * 120)
print("MATRIX COVERAGE")
print("-" * 120)

print(
    f"Observed combinations : {len(matrix_df)}"
)

print(
    f"Classified combinations: "
    f"{len(decision_matrix_audit_v1)}"
)

print(
    f"Coverage               : "
    f"{'PASSED' if coverage_passed else 'FAILED'}"
)


# --------------------------------------------------------------------------------------------------
# 5. DECISION VOCABULARY CHECK
# --------------------------------------------------------------------------------------------------

decision_vocabulary_passed = (
    decision_matrix_audit_v1["decision"]
    .isin(FVG_DECISION_V1_ALLOWED_DECISIONS)
    .all()
)

reason_vocabulary_passed = (
    decision_matrix_audit_v1["reason"]
    .isin(FVG_DECISION_V1_ALLOWED_REASONS)
    .all()
)

print("\n" + "-" * 120)
print("DECISION VOCABULARY")
print("-" * 120)

print(
    f"Decision vocabulary : "
    f"{'PASSED' if decision_vocabulary_passed else 'FAILED'}"
)

print(
    f"Reason vocabulary   : "
    f"{'PASSED' if reason_vocabulary_passed else 'FAILED'}"
)


# --------------------------------------------------------------------------------------------------
# 6. NO UNCLASSIFIED STATES
# --------------------------------------------------------------------------------------------------

no_unclassified_passed = (
    decision_matrix_audit_v1[
        ["decision", "reason"]
    ]
    .notna()
    .all()
    .all()
)

print("\n" + "-" * 120)
print("NO UNCLASSIFIED STATES")
print("-" * 120)

print(
    f"Status : "
    f"{'PASSED' if no_unclassified_passed else 'FAILED'}"
)


# --------------------------------------------------------------------------------------------------
# 7. RULE PRIORITY AUDIT
# --------------------------------------------------------------------------------------------------

# UNKNOWN structure must always remain UNRESOLVED
unknown_structure_rule_passed = (
    decision_matrix_audit_v1.loc[
        decision_matrix_audit_v1[
            "structure_qualification"
        ] == "UNKNOWN",
        "decision"
    ]
    .eq("UNRESOLVED")
    .all()
)

# FILLED must always be DISQUALIFIED unless structure is UNKNOWN.
filled_rule_mask = (
    (decision_matrix_audit_v1["lifecycle_state"] == "FILLED")
    &
    (decision_matrix_audit_v1["structure_qualification"] != "UNKNOWN")
)

filled_rule_passed = (
    decision_matrix_audit_v1.loc[
        filled_rule_mask,
        "decision"
    ]
    .eq("DISQUALIFIED")
    .all()
)

# OPPOSED structure, when not FILLED, must be CONDITIONAL
opposed_rule_mask = (
    (decision_matrix_audit_v1["structure_qualification"] == "OPPOSED")
    &
    (decision_matrix_audit_v1["lifecycle_state"] != "FILLED")
)

opposed_rule_passed = (
    decision_matrix_audit_v1.loc[
        opposed_rule_mask,
        "decision"
    ]
    .eq("CONDITIONAL")
    .all()
)

# ALIGNED + FRESH
aligned_fresh_mask = (
    (decision_matrix_audit_v1["structure_qualification"] == "ALIGNED")
    &
    (decision_matrix_audit_v1["lifecycle_state"] == "FRESH")
)

aligned_fresh_rule_passed = (
    decision_matrix_audit_v1.loc[
        aligned_fresh_mask,
        "decision"
    ]
    .eq("QUALIFIED")
    .all()
)

# ALIGNED + MITIGATED
aligned_mitigated_mask = (
    (decision_matrix_audit_v1["structure_qualification"] == "ALIGNED")
    &
    (decision_matrix_audit_v1["lifecycle_state"] == "MITIGATED")
)

aligned_mitigated_rule_passed = (
    decision_matrix_audit_v1.loc[
        aligned_mitigated_mask,
        "decision"
    ]
    .eq("CONDITIONAL")
    .all()
)

print("\n" + "-" * 120)
print("RULE PRIORITY AUDIT")
print("-" * 120)

print(
    f"{'UNKNOWN STRUCTURE → UNRESOLVED':50s}: "
    f"{'PASSED' if unknown_structure_rule_passed else 'FAILED'}"
)

print(
    f"{'FILLED → DISQUALIFIED':50s}: "
    f"{'PASSED' if filled_rule_passed else 'FAILED'}"
)

print(
    f"{'OPPOSED STRUCTURE → CONDITIONAL':50s}: "
    f"{'PASSED' if opposed_rule_passed else 'FAILED'}"
)

print(
    f"{'ALIGNED + FRESH → QUALIFIED':50s}: "
    f"{'PASSED' if aligned_fresh_rule_passed else 'FAILED'}"
)

print(
    f"{'ALIGNED + MITIGATED → CONDITIONAL':50s}: "
    f"{'PASSED' if aligned_mitigated_rule_passed else 'FAILED'}"
)


# --------------------------------------------------------------------------------------------------
# 8. DISPLAY COMPLETE MATRIX
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 120)
print("COMPLETE OBSERVED DECISION MATRIX")
print("-" * 120)

display(
    decision_matrix_audit_v1
)


# --------------------------------------------------------------------------------------------------
# 9. DECISION DISTRIBUTION
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 120)
print("DECISION DISTRIBUTION — OBSERVED COMBINATIONS")
print("-" * 120)

display(
    decision_matrix_audit_v1[
        "decision"
    ]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="combination_count")
)


# --------------------------------------------------------------------------------------------------
# 10. REASON DISTRIBUTION
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 120)
print("REASON DISTRIBUTION — OBSERVED COMBINATIONS")
print("-" * 120)

display(
    decision_matrix_audit_v1[
        "reason"
    ]
    .value_counts()
    .rename_axis("reason")
    .reset_index(name="combination_count")
)


# --------------------------------------------------------------------------------------------------
# 11. MODIFIER INDEPENDENCE AUDIT
# --------------------------------------------------------------------------------------------------

# V1 decision must NOT change merely because BOS / CHoCH / Liquidity
# modifiers differ while Structure + Lifecycle remain identical.

modifier_independence_violations = []

for (
    structure,
    lifecycle
), group in (
    fvg_qualification_engine_v1
    .groupby(
        [
            "structure_qualification",
            "lifecycle_state",
        ],
        dropna=False,
    )
):

    decisions = {
        classify_fvg_qualification_decision_v1(
            structure_qualification=structure,
            lifecycle_state=lifecycle,
        )[0]
    }

    if len(decisions) != 1:
        modifier_independence_violations.append(
            {
                "structure_qualification": structure,
                "lifecycle_state": lifecycle,
                "decision_count": len(decisions),
            }
        )

modifier_independence_passed = (
    len(modifier_independence_violations) == 0
)

print("\n" + "-" * 120)
print("MODIFIER INDEPENDENCE AUDIT")
print("-" * 120)

print(
    f"Status : "
    f"{'PASSED' if modifier_independence_passed else 'FAILED'}"
)

if modifier_independence_violations:
    display(
        pd.DataFrame(
            modifier_independence_violations
        )
    )


# --------------------------------------------------------------------------------------------------
# 12. FINAL CONTRACT RESULT
# --------------------------------------------------------------------------------------------------

decision_matrix_checks = [
    coverage_passed,
    decision_vocabulary_passed,
    reason_vocabulary_passed,
    no_unclassified_passed,
    unknown_structure_rule_passed,
    filled_rule_passed,
    opposed_rule_passed,
    aligned_fresh_rule_passed,
    aligned_mitigated_rule_passed,
    modifier_independence_passed,
]

passed = sum(decision_matrix_checks)
failed = len(decision_matrix_checks) - passed

print("\n" + "=" * 120)
print("FVG QUALIFICATION DECISION ENGINE V1 — DECISION MATRIX CONTRACT RESULT")
print("=" * 120)

print(f"TOTAL CHECKS : {len(decision_matrix_checks)}")
print(f"PASSED       : {passed}")
print(f"FAILED       : {failed}")

print("=" * 120)

if failed == 0:
    print("DECISION MATRIX CONTRACT STATUS : PASSED")
else:
    print("DECISION MATRIX CONTRACT STATUS : FAILED")

print("=" * 120)

In [ ]:
print("=" * 120)
print("FVG / M5 TIME CONTRACT AUDIT")
print("=" * 120)

print("\nm5_df:")
print("datetime dtype :", m5_df["datetime"].dtype)
print("timezone       :", getattr(m5_df["datetime"].dt, "tz", None))

print("\nfvg_qualification_engine_v1:")
print("created_at dtype:", fvg_qualification_engine_v1["created_at"].dtype)
print(
    "timezone        :",
    getattr(fvg_qualification_engine_v1["created_at"].dt, "tz", None)
)

print("\nfvg_validated_v1:")
print("created_at dtype:", fvg_validated_v1["created_at"].dtype)
print(
    "timezone        :",
    getattr(fvg_validated_v1["created_at"].dt, "tz", None)
)

print("\nm5_atr_df:")
print("datetime dtype :", m5_atr_df["datetime"].dtype)
print("timezone       :", getattr(m5_atr_df["datetime"].dt, "tz", None))

In [ ]:
# ==================================================================================================
# FVG PROXIMITY ENGINE V1
# Timeline-Based / No Scoring / No Entry Decision
# ==================================================================================================

import pandas as pd
import numpy as np


# --------------------------------------------------------------------------------------------------
# 1. CONTRACT CONSTANTS
# --------------------------------------------------------------------------------------------------

FVG_PROXIMITY_ENGINE_V1_VERSION = "1.0"

FVG_PROXIMITY_V1_PRICE_POSITION_STATES = {
    "BELOW",
    "INSIDE",
    "ABOVE",
}

FVG_PROXIMITY_V1_REQUIRED_QUALIFICATION_COLUMNS = [
    "fvg_id",
    "fvg_type",
    "fvg_direction",
    "created_at",
    "lifecycle_state",
    "active_bias",
    "bias_alignment",
    "context_bos_id",
    "context_bos_direction",
    "bos_alignment",
    "context_choch_id",
    "context_choch_direction",
    "choch_alignment",
    "historical_active_liquidity_count",
    "liquidity_below_count",
    "liquidity_above_count",
    "liquidity_overlap_count",
    "structure_qualification",
    "bos_qualification",
    "choch_qualification",
    "liquidity_qualification",
    "lifecycle_qualification",
]

FVG_PROXIMITY_V1_ZONE_COLUMNS = [
    "fvg_id",
    "lower_bound",
    "upper_bound",
    "gap_size",
]

FVG_PROXIMITY_V1_OUTPUT_COLUMNS = [
    # Identity / context
    "fvg_id",
    "fvg_type",
    "fvg_direction",
    "created_at",

    # Timeline
    "observation_time",

    # FVG zone
    "lower_bound",
    "upper_bound",
    "gap_size",

    # Price / ATR
    "reference_price",
    "atr",

    # Proximity measurement
    "price_position",
    "distance_to_fvg",
    "distance_to_fvg_atr",

    # Lifecycle
    "lifecycle_state",

    # Context
    "active_bias",
    "bias_alignment",

    "context_bos_id",
    "context_bos_direction",
    "bos_alignment",

    "context_choch_id",
    "context_choch_direction",
    "choch_alignment",

    "historical_active_liquidity_count",
    "liquidity_below_count",
    "liquidity_above_count",
    "liquidity_overlap_count",

    # Qualification
    "structure_qualification",
    "bos_qualification",
    "choch_qualification",
    "liquidity_qualification",
    "lifecycle_qualification",
]


# --------------------------------------------------------------------------------------------------
# 2. VALIDATION HELPERS
# --------------------------------------------------------------------------------------------------

def _assert_required_columns(df, columns, df_name):
    missing = [c for c in columns if c not in df.columns]

    if missing:
        raise ValueError(
            f"{df_name} is missing required columns: {missing}"
        )


def _assert_utc_datetime(df, column, df_name):
    if not pd.api.types.is_datetime64tz_dtype(df[column]):
        raise TypeError(
            f"{df_name}[{column!r}] must be timezone-aware datetime."
        )

    tz = getattr(df[column].dt, "tz", None)

    if str(tz) != "UTC":
        raise ValueError(
            f"{df_name}[{column!r}] must be UTC-aware. Found: {tz}"
        )


# --------------------------------------------------------------------------------------------------
# 3. MAIN ENGINE
# --------------------------------------------------------------------------------------------------

def build_fvg_proximity_engine_v1(
    fvg_qualification_df,
    fvg_zone_df,
    m5_df,
    m5_atr_df,
):
    """
    Build FVG Proximity Engine V1.

    Architecture:
        Qualification V1
            +
        FVG Zone Reference
            +
        M5 Price Timeline
            +
        M5 ATR Timeline
            ↓
        FVG Proximity Timeline

    Contract:
        - One observation per FVG per applicable M5 candle.
        - observation_time >= created_at.
        - reference_price = M5 close.
        - price_position ∈ {BELOW, INSIDE, ABOVE}.
        - distance is distance to the FVG zone, not midpoint distance.
        - ATR comes directly from m5_atr_df.
        - No lifecycle recomputation.
        - No qualification recomputation.
        - No scoring.
        - No entry decision.
    """

    # ----------------------------------------------------------------------------------------------
    # A. Required-column validation
    # ----------------------------------------------------------------------------------------------

    _assert_required_columns(
        fvg_qualification_df,
        FVG_PROXIMITY_V1_REQUIRED_QUALIFICATION_COLUMNS,
        "fvg_qualification_df",
    )

    _assert_required_columns(
        fvg_zone_df,
        FVG_PROXIMITY_V1_ZONE_COLUMNS,
        "fvg_zone_df",
    )

    _assert_required_columns(
        m5_df,
        [
            "datetime",
            "open",
            "high",
            "low",
            "close",
        ],
        "m5_df",
    )

    _assert_required_columns(
        m5_atr_df,
        [
            "datetime",
            "atr",
        ],
        "m5_atr_df",
    )

    # ----------------------------------------------------------------------------------------------
    # B. Defensive copies
    # ----------------------------------------------------------------------------------------------

    qualification = fvg_qualification_df.copy()
    zones = fvg_zone_df.copy()
    prices = m5_df.copy()
    atr = m5_atr_df.copy()

    # ----------------------------------------------------------------------------------------------
    # C. UTC contract
    # ----------------------------------------------------------------------------------------------

    _assert_utc_datetime(
        qualification,
        "created_at",
        "fvg_qualification_df",
    )

    _assert_utc_datetime(
        zones,
        "created_at",
        "fvg_zone_df",
    ) if "created_at" in zones.columns else None

    _assert_utc_datetime(
        prices,
        "datetime",
        "m5_df",
    )

    _assert_utc_datetime(
        atr,
        "datetime",
        "m5_atr_df",
    )

    # ----------------------------------------------------------------------------------------------
    # D. FVG identity contracts
    # ----------------------------------------------------------------------------------------------

    if qualification["fvg_id"].isna().any():
        raise ValueError("Qualification V1 contains null fvg_id.")

    if qualification["fvg_id"].duplicated().any():
        raise ValueError("Qualification V1 contains duplicate fvg_id.")

    if zones["fvg_id"].isna().any():
        raise ValueError("FVG zone reference contains null fvg_id.")

    if zones["fvg_id"].duplicated().any():
        raise ValueError("FVG zone reference contains duplicate fvg_id.")

    # ----------------------------------------------------------------------------------------------
    # E. Numeric zone validation
    # ----------------------------------------------------------------------------------------------

    numeric_zone_columns = [
        "lower_bound",
        "upper_bound",
        "gap_size",
    ]

    for col in numeric_zone_columns:
        zones[col] = pd.to_numeric(
            zones[col],
            errors="coerce",
        )

    if zones[numeric_zone_columns].isna().any().any():
        raise ValueError(
            "FVG zone reference contains invalid numeric values."
        )

    if (zones["lower_bound"] > zones["upper_bound"]).any():
        raise ValueError(
            "FVG zone contract violated: lower_bound > upper_bound."
        )

    # ----------------------------------------------------------------------------------------------
    # F. Price / ATR validation
    # ----------------------------------------------------------------------------------------------

    prices["close"] = pd.to_numeric(
        prices["close"],
        errors="coerce",
    )

    atr["atr"] = pd.to_numeric(
        atr["atr"],
        errors="coerce",
    )

    if prices["close"].isna().any():
        raise ValueError(
            "m5_df contains invalid close values."
        )

    # ----------------------------------------------------------------------------------------------
    # G. Chronological contracts
    # ----------------------------------------------------------------------------------------------

    prices = (
        prices
        .sort_values("datetime")
        .reset_index(drop=True)
    )

    atr = (
        atr
        .sort_values("datetime")
        .reset_index(drop=True)
    )

    if not prices["datetime"].is_monotonic_increasing:
        raise ValueError(
            "m5_df datetime is not chronological."
        )

    if not atr["datetime"].is_monotonic_increasing:
        raise ValueError(
            "m5_atr_df datetime is not chronological."
        )

    if prices["datetime"].duplicated().any():
        raise ValueError(
            "m5_df contains duplicate datetime values."
        )

    if atr["datetime"].duplicated().any():
        raise ValueError(
            "m5_atr_df contains duplicate datetime values."
        )

    # ----------------------------------------------------------------------------------------------
    # H. Build the immutable FVG reference
    # ----------------------------------------------------------------------------------------------

    reference = (
        qualification[
            FVG_PROXIMITY_V1_REQUIRED_QUALIFICATION_COLUMNS
        ]
        .merge(
            zones[
                FVG_PROXIMITY_V1_ZONE_COLUMNS
            ],
            on="fvg_id",
            how="left",
            validate="one_to_one",
            suffixes=("", "_zone"),
        )
    )

    # Zone must exist for every qualified FVG.
    if reference[
        ["lower_bound", "upper_bound", "gap_size"]
    ].isna().any().any():

        missing_zone_ids = reference.loc[
            reference[
                ["lower_bound", "upper_bound", "gap_size"]
            ].isna().any(axis=1),
            "fvg_id",
        ].tolist()

        raise ValueError(
            "Missing FVG zone data for fvg_id(s): "
            f"{missing_zone_ids}"
        )

    # ----------------------------------------------------------------------------------------------
    # I. Ensure qualification timestamps are UTC and normalized only for comparison.
    # ----------------------------------------------------------------------------------------------

    reference["created_at"] = pd.to_datetime(
        reference["created_at"],
        utc=True,
    )

    prices["datetime"] = pd.to_datetime(
        prices["datetime"],
        utc=True,
    )

    atr["datetime"] = pd.to_datetime(
        atr["datetime"],
        utc=True,
    )

    # ----------------------------------------------------------------------------------------------
    # J. Attach ATR to M5 timeline
    # ----------------------------------------------------------------------------------------------

    price_timeline = (
        prices[
            [
                "datetime",
                "close",
            ]
        ]
        .merge(
            atr[
                [
                    "datetime",
                    "atr",
                ]
            ],
            on="datetime",
            how="left",
            validate="one_to_one",
        )
        .rename(
            columns={
                "datetime": "observation_time",
                "close": "reference_price",
            }
        )
    )

    # ==================================================================================================
    # ATR VALIDITY CONTRACT
    # ==================================================================================================

    price_timeline = price_timeline.loc[
        price_timeline["atr"].notna()
    ].copy()

    if price_timeline.empty:
        raise ValueError(
            "No M5 observations with valid ATR are available "
            "for FVG Proximity Engine V1."
        )

    if not np.isfinite(price_timeline["atr"]).all():
        raise ValueError(
            "ATR contains non-finite values after warm-up filtering."
        )

    if (price_timeline["atr"] <= 0).any():
        raise ValueError(
            "ATR must be strictly positive for Proximity normalization."
        )
    

    # ----------------------------------------------------------------------------------------------
    # K. Timeline expansion
    #
    # Cross join is intentional:
    # each FVG gets its applicable M5 observations.
    # ----------------------------------------------------------------------------------------------

    reference["_join_key"] = 1
    price_timeline["_join_key"] = 1

    timeline = reference.merge(
        price_timeline,
        on="_join_key",
        how="inner",
        sort=False,
    )

    timeline.drop(
        columns=["_join_key"],
        inplace=True,
    )

    # ----------------------------------------------------------------------------------------------
    # L. Post-formation temporal contract
    # ----------------------------------------------------------------------------------------------

    timeline = timeline[
        timeline["observation_time"] >= timeline["created_at"]
    ].copy()

    if timeline.empty:
        raise ValueError(
            "FVG Proximity Engine produced no post-formation observations."
        )

    # ----------------------------------------------------------------------------------------------
    # M. Price position
    # ----------------------------------------------------------------------------------------------

    timeline["price_position"] = np.select(
        [
            timeline["reference_price"] < timeline["lower_bound"],
            timeline["reference_price"] > timeline["upper_bound"],
        ],
        [
            "BELOW",
            "ABOVE",
        ],
        default="INSIDE",
    )

    # ----------------------------------------------------------------------------------------------
    # N. Distance to FVG zone
    #
    # Outside below:
    #     lower_bound - price
    #
    # Outside above:
    #     price - upper_bound
    #
    # Inside:
    #     0
    # ----------------------------------------------------------------------------------------------

    timeline["distance_to_fvg"] = np.select(
        [
            timeline["reference_price"] < timeline["lower_bound"],
            timeline["reference_price"] > timeline["upper_bound"],
        ],
        [
            timeline["lower_bound"] - timeline["reference_price"],
            timeline["reference_price"] - timeline["upper_bound"],
        ],
        default=0.0,
    )

    timeline["distance_to_fvg"] = (
        timeline["distance_to_fvg"]
        .astype(float)
        .clip(lower=0.0)
    )

    # ----------------------------------------------------------------------------------------------
    # O. ATR-normalized distance
    # ----------------------------------------------------------------------------------------------

    invalid_atr = (
        timeline["atr"].isna()
        | (timeline["atr"] <= 0)
    )

    timeline["distance_to_fvg_atr"] = np.where(
        invalid_atr,
        np.nan,
        timeline["distance_to_fvg"] / timeline["atr"],
    )

    # ----------------------------------------------------------------------------------------------
    # P. Output ordering
    # ----------------------------------------------------------------------------------------------

    timeline = (
        timeline
        .sort_values(
            [
                "observation_time",
                "fvg_id",
            ]
        )
        .reset_index(drop=True)
    )

    # ----------------------------------------------------------------------------------------------
    # Q. Output schema
    # ----------------------------------------------------------------------------------------------

    missing_output_columns = [
        col
        for col in FVG_PROXIMITY_V1_OUTPUT_COLUMNS
        if col not in timeline.columns
    ]

    if missing_output_columns:
        raise RuntimeError(
            "FVG Proximity Engine output is missing columns: "
            f"{missing_output_columns}"
        )

    result = timeline[
        FVG_PROXIMITY_V1_OUTPUT_COLUMNS
    ].copy()

    # ----------------------------------------------------------------------------------------------
    # R. Final type / vocabulary contracts
    # ----------------------------------------------------------------------------------------------

    if not result["fvg_id"].is_unique:
        # Duplicate FVG IDs are EXPECTED in a timeline.
        # The actual uniqueness contract is per FVG + observation_time.
        pass

    if result[
        ["fvg_id", "observation_time"]
    ].duplicated().any():

        raise RuntimeError(
            "Duplicate FVG observation detected."
        )

    invalid_positions = set(
        result["price_position"].dropna().unique()
    ) - FVG_PROXIMITY_V1_PRICE_POSITION_STATES

    if invalid_positions:
        raise RuntimeError(
            "Invalid price_position state(s): "
            f"{invalid_positions}"
        )

    # ----------------------------------------------------------------------------------------------
    # S. Metadata
    # ----------------------------------------------------------------------------------------------

    result.attrs["engine"] = "FVG Proximity Engine V1"
    result.attrs["version"] = FVG_PROXIMITY_ENGINE_V1_VERSION
    result.attrs["reference_price"] = "m5 close"
    result.attrs["timeline_type"] = "post-formation M5"
    result.attrs["price_position_states"] = sorted(
        FVG_PROXIMITY_V1_PRICE_POSITION_STATES
    )
    result.attrs["scoring"] = False
    result.attrs["entry_decision"] = False

    return result


# ==================================================================================================
# BUILD
# ==================================================================================================

fvg_proximity_engine_v1 = build_fvg_proximity_engine_v1(
    fvg_qualification_df=fvg_qualification_engine_v1,
    fvg_zone_df=fvg_validated_v1,
    m5_df=m5_df,
    m5_atr_df=m5_atr_df,
)


# ==================================================================================================
# SUMMARY
# ==================================================================================================

print("=" * 120)
print("FVG PROXIMITY ENGINE V1")
print("=" * 120)

print(f"Rows              : {len(fvg_proximity_engine_v1):,}")
print(f"Columns           : {len(fvg_proximity_engine_v1.columns)}")
print(
    f"Unique FVGs       : "
    f"{fvg_proximity_engine_v1['fvg_id'].nunique()}"
)
print(
    f"Unique observations: "
    f"{fvg_proximity_engine_v1['observation_time'].nunique()}"
)

print("\nPRICE POSITION")
print(
    fvg_proximity_engine_v1["price_position"]
    .value_counts(dropna=False)
)

print("\nDISTANCE")
print(
    fvg_proximity_engine_v1["distance_to_fvg"]
    .describe()
)

print("\nATR-NORMALIZED DISTANCE")
print(
    fvg_proximity_engine_v1["distance_to_fvg_atr"]
    .describe()
)

print("\nTIME RANGE")
print(
    "First observation :",
    fvg_proximity_engine_v1["observation_time"].min()
)
print(
    "Last observation  :",
    fvg_proximity_engine_v1["observation_time"].max()
)

print("=" * 120)
print("FVG PROXIMITY ENGINE V1 : BUILT")
print("=" * 120)

In [ ]:
fvg_proximity_engine_v1 = build_fvg_proximity_engine_v1(
    fvg_qualification_df=fvg_qualification_engine_v1,
    fvg_zone_df=fvg_validated_v1,
    m5_df=m5_df,
    m5_atr_df=m5_atr_df,
)

In [ ]:
print("=" * 120)
print("FVG PROXIMITY ENGINE V1 — BUILD SUMMARY")
print("=" * 120)

print("\nRows                :", len(fvg_proximity_engine_v1))
print("Columns             :", len(fvg_proximity_engine_v1.columns))
print("Unique FVG IDs      :", fvg_proximity_engine_v1["fvg_id"].nunique())
print("Unique observations :", fvg_proximity_engine_v1["observation_time"].nunique())

print("\nPrice Position:")
print(
    fvg_proximity_engine_v1["price_position"]
    .value_counts(dropna=False)
)

print("\nATR null count      :",
      fvg_proximity_engine_v1["atr"].isna().sum())

print("Distance ATR null   :",
      fvg_proximity_engine_v1["distance_to_fvg_atr"].isna().sum())

print("\nObservation range:")
print("First :", fvg_proximity_engine_v1["observation_time"].min())
print("Last  :", fvg_proximity_engine_v1["observation_time"].max())

print("\nCreated range:")
print("First :", fvg_proximity_engine_v1["created_at"].min())
print("Last  :", fvg_proximity_engine_v1["created_at"].max())

print("=" * 120)

In [ ]:
print("=" * 120)
print("M5 / ATR TIMELINE COVERAGE AUDIT")
print("=" * 120)

m5_times = pd.to_datetime(m5_df["datetime"], utc=True)
atr_times = pd.to_datetime(m5_atr_df["datetime"], utc=True)

print("\nM5 TIMELINE")
print("Rows       :", len(m5_times))
print("First      :", m5_times.min())
print("Last       :", m5_times.max())

print("\nATR TIMELINE")
print("Rows       :", len(atr_times))
print("First      :", atr_times.min())
print("Last       :", atr_times.max())

missing_times = (
    m5_times[
        ~m5_times.isin(atr_times)
    ]
    .drop_duplicates()
    .sort_values()
)

print("\nMISSING ATR TIMESTAMPS")
print("Count      :", len(missing_times))

if len(missing_times) > 0:
    print("\nFirst missing:")
    print(missing_times.head(20).to_list())

    print("\nLast missing:")
    print(missing_times.tail(20).to_list())

print("\nATR NULL VALUES")
print(
    m5_atr_df["atr"]
    .isna()
    .sum()
)

print("=" * 120)

In [ ]:
# ==================================================================================================
# ATR NULL POSITION AUDIT
# ==================================================================================================

atr_null_audit = (
    m5_atr_df.loc[
        m5_atr_df["atr"].isna(),
        ["datetime", "tr", "atr"]
    ]
    .copy()
)

atr_null_audit["datetime"] = pd.to_datetime(
    atr_null_audit["datetime"],
    utc=True
)

print("=" * 120)
print("ATR NULL POSITION AUDIT")
print("=" * 120)

print("\nNULL ATR COUNT :", len(atr_null_audit))

print("\nNULL ATR ROWS:")
print(atr_null_audit.to_string(index=False))

print("\nFIRST VALID ATR:")
first_valid_atr = m5_atr_df.loc[
    m5_atr_df["atr"].notna(),
    ["datetime", "atr"]
].head(1)

print(first_valid_atr.to_string(index=False))

print("\nLAST NULL ATR:")
print(
    atr_null_audit["datetime"].max()
)

print("=" * 120)

In [ ]:
# ==================================================================================================
# FVG PROXIMITY ENGINE V1 — REGRESSION GATE
# ==================================================================================================

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# CONSTANTS
# --------------------------------------------------------------------------------------------------

FVG_PROXIMITY_V1_EXPECTED_POSITION_STATES = {
    "BELOW",
    "INSIDE",
    "ABOVE",
}

FVG_PROXIMITY_V1_IDENTITY_COLUMNS = [
    "fvg_id",
    "fvg_type",
    "fvg_direction",
    "created_at",
]

FVG_PROXIMITY_V1_ZONE_COLUMNS = [
    "fvg_id",
    "lower_bound",
    "upper_bound",
    "gap_size",
]

FVG_PROXIMITY_V1_QUALIFICATION_COLUMNS = [
    "lifecycle_state",
    "active_bias",
    "bias_alignment",
    "context_bos_id",
    "context_bos_direction",
    "bos_alignment",
    "context_choch_id",
    "context_choch_direction",
    "choch_alignment",
    "historical_active_liquidity_count",
    "liquidity_below_count",
    "liquidity_above_count",
    "liquidity_overlap_count",
    "structure_qualification",
    "bos_qualification",
    "choch_qualification",
    "liquidity_qualification",
    "lifecycle_qualification",
]

FVG_PROXIMITY_V1_REQUIRED_COLUMNS = (
    FVG_PROXIMITY_V1_IDENTITY_COLUMNS
    + [
        "observation_time",
        "lower_bound",
        "upper_bound",
        "gap_size",
        "reference_price",
        "atr",
        "price_position",
        "distance_to_fvg",
        "distance_to_fvg_atr",
    ]
    + FVG_PROXIMITY_V1_QUALIFICATION_COLUMNS
)


# --------------------------------------------------------------------------------------------------
# HELPER
# --------------------------------------------------------------------------------------------------

def _assert_columns_exist(df, columns, df_name):
    missing = [c for c in columns if c not in df.columns]

    if missing:
        raise AssertionError(
            f"{df_name} missing required columns: {missing}"
        )


def _assert_utc_column(df, column, df_name):
    if not pd.api.types.is_datetime64_any_dtype(df[column]):
        raise AssertionError(
            f"{df_name}.{column} is not datetime dtype."
        )

    if not isinstance(df[column].dtype, pd.DatetimeTZDtype):
        raise AssertionError(
            f"{df_name}.{column} must be timezone-aware."
        )

    if str(df[column].dt.tz) != "UTC":
        raise AssertionError(
            f"{df_name}.{column} must be UTC."
        )


# --------------------------------------------------------------------------------------------------
# REGRESSION GATE
# --------------------------------------------------------------------------------------------------

def regression_gate_fvg_proximity_engine_v1(
    proximity_df,
    qualification_df,
    zone_df,
    m5_df,
    m5_atr_df,
):

    results = {}

    p = proximity_df.copy()
    q = qualification_df.copy()
    z = zone_df.copy()
    m = m5_df.copy()
    a = m5_atr_df.copy()

    # ==============================================================================================
    # 1. SCHEMA CONTRACT
    # ==============================================================================================

    try:
        _assert_columns_exist(
            p,
            FVG_PROXIMITY_V1_REQUIRED_COLUMNS,
            "proximity_df",
        )

        results["SCHEMA CONTRACT"] = "PASSED"

    except Exception as e:
        results["SCHEMA CONTRACT"] = f"FAILED: {e}"
        return results


    # ==============================================================================================
    # 2. QUALIFIED FVG SET PRESERVATION
    # ==============================================================================================

    try:
        expected_ids = set(q["fvg_id"].dropna())
        actual_ids = set(p["fvg_id"].dropna())

        assert actual_ids == expected_ids, (
            f"FVG ID set mismatch. "
            f"Expected={len(expected_ids)}, Actual={len(actual_ids)}"
        )

        results["QUALIFIED FVG SET PRESERVATION"] = "PASSED"

    except Exception as e:
        results["QUALIFIED FVG SET PRESERVATION"] = f"FAILED: {e}"


    # ==============================================================================================
    # 3. FVG ID PRESERVATION
    # ==============================================================================================

    try:
        assert p["fvg_id"].notna().all()
        assert p["fvg_id"].isin(q["fvg_id"]).all()

        results["FVG ID PRESERVATION"] = "PASSED"

    except Exception as e:
        results["FVG ID PRESERVATION"] = f"FAILED: {e}"


    # ==============================================================================================
    # 4. FVG IDENTITY PRESERVATION
    # ==============================================================================================

    try:
        q_identity = (
            q[FVG_PROXIMITY_V1_IDENTITY_COLUMNS]
            .drop_duplicates("fvg_id")
            .set_index("fvg_id")
        )

        p_identity = (
            p[FVG_PROXIMITY_V1_IDENTITY_COLUMNS]
            .drop_duplicates("fvg_id")
            .set_index("fvg_id")
        )

        assert set(q_identity.index) == set(p_identity.index)

        for col in FVG_PROXIMITY_V1_IDENTITY_COLUMNS[1:]:

            left = q_identity[col]
            right = p_identity[col]

            if pd.api.types.is_datetime64_any_dtype(left):
                assert (
                    pd.to_datetime(left, utc=True)
                    .equals(pd.to_datetime(right, utc=True))
                ), f"{col} mismatch"

            else:
                assert left.equals(right), f"{col} mismatch"

        results["FVG IDENTITY PRESERVATION"] = "PASSED"

    except Exception as e:
        results["FVG IDENTITY PRESERVATION"] = f"FAILED: {e}"


    # ==============================================================================================
    # 5. ZONE PRESERVATION
    # ==============================================================================================

    try:
        q_ids = set(q["fvg_id"])

        z_ref = (
            z[
                FVG_PROXIMITY_V1_ZONE_COLUMNS
            ]
            .drop_duplicates("fvg_id")
            .loc[lambda x: x["fvg_id"].isin(q_ids)]
            .set_index("fvg_id")
        )

        p_ref = (
            p[
                FVG_PROXIMITY_V1_ZONE_COLUMNS
            ]
            .drop_duplicates("fvg_id")
            .set_index("fvg_id")
        )

        assert set(z_ref.index) == set(p_ref.index)

        for col in [
            "lower_bound",
            "upper_bound",
            "gap_size",
        ]:
            assert np.allclose(
                z_ref[col].astype(float),
                p_ref[col].astype(float),
                equal_nan=False,
            ), f"{col} mismatch"

        results["ZONE PRESERVATION"] = "PASSED"

    except Exception as e:
        results["ZONE PRESERVATION"] = f"FAILED: {e}"


    # ==============================================================================================
    # 6. OBSERVATION UNIQUENESS
    # ==============================================================================================

    try:
        duplicates = p.duplicated(
            subset=["fvg_id", "observation_time"]
        )

        assert not duplicates.any(), (
            f"Duplicate FVG/observation pairs: "
            f"{duplicates.sum()}"
        )

        results["OBSERVATION UNIQUENESS"] = "PASSED"

    except Exception as e:
        results["OBSERVATION UNIQUENESS"] = f"FAILED: {e}"


    # ==============================================================================================
    # 7. POST-FORMATION TIME CONTRACT
    # ==============================================================================================

    try:
        created = pd.to_datetime(
            p["created_at"],
            utc=True,
        )

        observation = pd.to_datetime(
            p["observation_time"],
            utc=True,
        )

        assert (observation >= created).all()

        results["POST-FORMATION TIME CONTRACT"] = "PASSED"

    except Exception as e:
        results["POST-FORMATION TIME CONTRACT"] = f"FAILED: {e}"


    # ==============================================================================================
    # 8. M5 TIMESTAMP MEMBERSHIP
    # ==============================================================================================

    try:
        m_times = set(
            pd.to_datetime(
                m["datetime"],
                utc=True,
            )
        )

        p_times = set(
            pd.to_datetime(
                p["observation_time"],
                utc=True,
            )
        )

        assert p_times.issubset(m_times), (
            "Proximity contains observation timestamps "
            "not present in M5 timeline."
        )

        results["M5 TIMESTAMP MEMBERSHIP"] = "PASSED"

    except Exception as e:
        results["M5 TIMESTAMP MEMBERSHIP"] = f"FAILED: {e}"


    # ==============================================================================================
    # 9. CHRONOLOGICAL ORDER
    # ==============================================================================================

    try:
        ordered = p.sort_values(
            ["fvg_id", "observation_time"]
        )

        chronological = (
            ordered.groupby("fvg_id")["observation_time"]
            .apply(lambda s: s.is_monotonic_increasing)
            .all()
        )

        assert chronological

        results["CHRONOLOGICAL ORDER"] = "PASSED"

    except Exception as e:
        results["CHRONOLOGICAL ORDER"] = f"FAILED: {e}"


    # ==============================================================================================
    # 10. UTC CONTRACT
    # ==============================================================================================

    try:
        _assert_utc_column(
            p,
            "created_at",
            "proximity_df",
        )

        _assert_utc_column(
            p,
            "observation_time",
            "proximity_df",
        )

        results["UTC CONTRACT"] = "PASSED"

    except Exception as e:
        results["UTC CONTRACT"] = f"FAILED: {e}"


    # ==============================================================================================
    # 11. REFERENCE PRICE CONTRACT
    # ==============================================================================================

    try:
        m_ref = (
            m[["datetime", "close"]]
            .copy()
        )

        m_ref["datetime"] = pd.to_datetime(
            m_ref["datetime"],
            utc=True,
        )

        p_ref = p[
            ["observation_time", "reference_price"]
        ].copy()

        p_ref["observation_time"] = pd.to_datetime(
            p_ref["observation_time"],
            utc=True,
        )

        merged = p_ref.merge(
            m_ref,
            left_on="observation_time",
            right_on="datetime",
            how="left",
            validate="many_to_one",
        )

        assert merged["close"].notna().all()

        assert np.allclose(
            merged["reference_price"].astype(float),
            merged["close"].astype(float),
            equal_nan=False,
        )

        results["REFERENCE PRICE CONTRACT"] = "PASSED"

    except Exception as e:
        results["REFERENCE PRICE CONTRACT"] = f"FAILED: {e}"


    # ==============================================================================================
    # 12. ATR CONTRACT
    # ==============================================================================================

    try:
        a_ref = (
            a[["datetime", "atr"]]
            .copy()
        )

        a_ref["datetime"] = pd.to_datetime(
            a_ref["datetime"],
            utc=True,
        )

        merged = p[
            ["observation_time", "atr"]
        ].merge(
            a_ref,
            left_on="observation_time",
            right_on="datetime",
            how="left",
            validate="many_to_one",
            suffixes=("_proximity", "_source"),
        )

        assert merged["atr_source"].notna().all()
        assert (merged["atr_source"] > 0).all()

        assert np.allclose(
            merged["atr_proximity"].astype(float),
            merged["atr_source"].astype(float),
            equal_nan=False,
        )

        results["ATR CONTRACT"] = "PASSED"

    except Exception as e:
        results["ATR CONTRACT"] = f"FAILED: {e}"


    # ==============================================================================================
    # 13. PRICE POSITION CONTRACT
    # ==============================================================================================

    try:
        price = p["reference_price"].astype(float)
        lower = p["lower_bound"].astype(float)
        upper = p["upper_bound"].astype(float)

        expected_position = np.select(
            [
                price < lower,
                price > upper,
            ],
            [
                "BELOW",
                "ABOVE",
            ],
            default="INSIDE",
        )

        assert (
            p["price_position"].to_numpy()
            == expected_position
        ).all()

        assert set(
            p["price_position"].dropna().unique()
        ).issubset(
            FVG_PROXIMITY_V1_EXPECTED_POSITION_STATES
        )

        results["PRICE POSITION CONTRACT"] = "PASSED"

    except Exception as e:
        results["PRICE POSITION CONTRACT"] = f"FAILED: {e}"


    # ==============================================================================================
    # 14. DISTANCE CONTRACT
    # ==============================================================================================

    try:
        price = p["reference_price"].astype(float)
        lower = p["lower_bound"].astype(float)
        upper = p["upper_bound"].astype(float)

        expected_distance = np.where(
            price < lower,
            lower - price,
            np.where(
                price > upper,
                price - upper,
                0.0,
            ),
        )

        assert np.allclose(
            p["distance_to_fvg"].astype(float),
            expected_distance,
            atol=1e-12,
            equal_nan=False,
        )

        assert (p["distance_to_fvg"] >= 0).all()

        results["DISTANCE CONTRACT"] = "PASSED"

    except Exception as e:
        results["DISTANCE CONTRACT"] = f"FAILED: {e}"


    # ==============================================================================================
    # 15. ATR NORMALIZATION CONTRACT
    # ==============================================================================================

    try:
        expected_distance_atr = (
            p["distance_to_fvg"].astype(float)
            / p["atr"].astype(float)
        )

        assert np.allclose(
            p["distance_to_fvg_atr"].astype(float),
            expected_distance_atr,
            atol=1e-12,
            equal_nan=False,
        )

        assert np.isfinite(
            p["distance_to_fvg_atr"].astype(float)
        ).all()

        assert (
            p["distance_to_fvg_atr"] >= 0
        ).all()

        results["ATR NORMALIZATION CONTRACT"] = "PASSED"

    except Exception as e:
        results["ATR NORMALIZATION CONTRACT"] = f"FAILED: {e}"


    # ==============================================================================================
    # 16. LIFECYCLE PRESERVATION
    # ==============================================================================================

    try:
        q_lifecycle = (
            q[["fvg_id", "lifecycle_state"]]
            .drop_duplicates("fvg_id")
            .set_index("fvg_id")
        )

        p_lifecycle = (
            p[["fvg_id", "lifecycle_state"]]
            .drop_duplicates("fvg_id")
            .set_index("fvg_id")
        )

        assert q_lifecycle.equals(p_lifecycle)

        results["LIFECYCLE PRESERVATION"] = "PASSED"

    except Exception as e:
        results["LIFECYCLE PRESERVATION"] = f"FAILED: {e}"


    # ==============================================================================================
    # 17. STRUCTURE QUALIFICATION PRESERVATION
    # ==============================================================================================

    try:
        q_ref = (
            q[["fvg_id", "structure_qualification"]]
            .drop_duplicates("fvg_id")
            .set_index("fvg_id")
        )

        p_ref = (
            p[["fvg_id", "structure_qualification"]]
            .drop_duplicates("fvg_id")
            .set_index("fvg_id")
        )

        assert q_ref.equals(p_ref)

        results["STRUCTURE QUALIFICATION PRESERVATION"] = "PASSED"

    except Exception as e:
        results["STRUCTURE QUALIFICATION PRESERVATION"] = f"FAILED: {e}"


    # ==============================================================================================
    # 18. BOS QUALIFICATION PRESERVATION
    # ==============================================================================================

    try:
        q_ref = (
            q[["fvg_id", "bos_qualification"]]
            .drop_duplicates("fvg_id")
            .set_index("fvg_id")
        )

        p_ref = (
            p[["fvg_id", "bos_qualification"]]
            .drop_duplicates("fvg_id")
            .set_index("fvg_id")
        )

        assert q_ref.equals(p_ref)

        results["BOS QUALIFICATION PRESERVATION"] = "PASSED"

    except Exception as e:
        results["BOS QUALIFICATION PRESERVATION"] = f"FAILED: {e}"


    # ==============================================================================================
    # 19. CHoCH QUALIFICATION PRESERVATION
    # ==============================================================================================

    try:
        q_ref = (
            q[["fvg_id", "choch_qualification"]]
            .drop_duplicates("fvg_id")
            .set_index("fvg_id")
        )

        p_ref = (
            p[["fvg_id", "choch_qualification"]]
            .drop_duplicates("fvg_id")
            .set_index("fvg_id")
        )

        assert q_ref.equals(p_ref)

        results["CHoCH QUALIFICATION PRESERVATION"] = "PASSED"

    except Exception as e:
        results["CHoCH QUALIFICATION PRESERVATION"] = f"FAILED: {e}"


    # ==============================================================================================
    # 20. LIQUIDITY QUALIFICATION PRESERVATION
    # ==============================================================================================

    try:
        q_ref = (
            q[["fvg_id", "liquidity_qualification"]]
            .drop_duplicates("fvg_id")
            .set_index("fvg_id")
        )

        p_ref = (
            p[["fvg_id", "liquidity_qualification"]]
            .drop_duplicates("fvg_id")
            .set_index("fvg_id")
        )

        assert q_ref.equals(p_ref)

        results["LIQUIDITY QUALIFICATION PRESERVATION"] = "PASSED"

    except Exception as e:
        results["LIQUIDITY QUALIFICATION PRESERVATION"] = f"FAILED: {e}"


    # ==============================================================================================
    # 21. QUALIFICATION IMMUTABILITY
    # ==============================================================================================

    try:
        for col in FVG_PROXIMITY_V1_QUALIFICATION_COLUMNS:

            q_ref = (
                q[["fvg_id", col]]
                .drop_duplicates("fvg_id")
                .set_index("fvg_id")
            )

            p_ref = (
                p[["fvg_id", col]]
                .drop_duplicates("fvg_id")
                .set_index("fvg_id")
            )

            assert q_ref.equals(
                p_ref
            ), f"Qualification column changed: {col}"

        results["QUALIFICATION IMMUTABILITY"] = "PASSED"

    except Exception as e:
        results["QUALIFICATION IMMUTABILITY"] = f"FAILED: {e}"


    # ==============================================================================================
    # 22. TIMELINE COMPLETENESS
    # ==============================================================================================

    try:
        q_created = q[
            ["fvg_id", "created_at"]
        ].drop_duplicates("fvg_id").copy()

        q_created["created_at"] = pd.to_datetime(
            q_created["created_at"],
            utc=True,
        )

        valid_m5 = m.copy()

        valid_m5["datetime"] = pd.to_datetime(
            valid_m5["datetime"],
            utc=True,
        )

        valid_atr_times = set(
            a.loc[
                a["atr"].notna()
                & np.isfinite(a["atr"])
                & (a["atr"] > 0),
                "datetime",
            ]
        )

        valid_m5_times = [
            t
            for t in valid_m5["datetime"]
            if t in valid_atr_times
        ]

        valid_m5_times = pd.DatetimeIndex(
            valid_m5_times
        )

        expected_counts = {}

        for row in q_created.itertuples(index=False):

            expected_counts[row.fvg_id] = int(
                (
                    valid_m5_times
                    >= row.created_at
                ).sum()
            )

        actual_counts = (
            p.groupby("fvg_id")
            .size()
            .to_dict()
        )

        assert set(actual_counts) == set(expected_counts)

        for fvg_id, expected_count in expected_counts.items():

            actual_count = actual_counts.get(
                fvg_id,
                0,
            )

            assert actual_count == expected_count, (
                f"Timeline count mismatch for {fvg_id}: "
                f"expected={expected_count}, "
                f"actual={actual_count}"
            )

            assert expected_count > 0, (
                f"FVG {fvg_id} has no valid "
                "post-formation observation."
            )

        results["TIMELINE COMPLETENESS"] = "PASSED"

    except Exception as e:
        results["TIMELINE COMPLETENESS"] = f"FAILED: {e}"


    # ==============================================================================================
    # 23. NO SCORING / ENTRY DECISION CONTRACT
    # ==============================================================================================

    try:
        forbidden_tokens = [
            "score",
            "rank",
            "ranking",
            "entry",
            "signal",
            "trade",
            "long",
            "short",
            "buy",
            "sell",
        ]

        protected_columns = {
            c.lower()
            for c in p.columns
        }

        forbidden_columns = [
            c
            for c in protected_columns
            if any(
                token in c
                for token in forbidden_tokens
            )
        ]

        assert not forbidden_columns, (
            f"Forbidden strategy/decision columns detected: "
            f"{forbidden_columns}"
        )

        results["NO SCORING / ENTRY DECISION CONTRACT"] = "PASSED"

    except Exception as e:
        results["NO SCORING / ENTRY DECISION CONTRACT"] = f"FAILED: {e}"


    # ==============================================================================================
    # FINAL STATUS
    # ==============================================================================================

    failed = {
        k: v
        for k, v in results.items()
        if str(v).startswith("FAILED")
    }

    print("=" * 120)
    print("FVG PROXIMITY ENGINE V1 REGRESSION GATE")
    print("=" * 120)

    for name, status in results.items():
        print(
            f"{name:<55} : {status}"
        )

    print("-" * 120)

    if failed:
        print("REGRESSION GATE : FAILED")
        print("\nFailed checks:")
        for name, status in failed.items():
            print(f" - {name}: {status}")

        raise AssertionError(
            "FVG Proximity Engine V1 Regression Gate FAILED."
        )

    else:
        print("REGRESSION GATE : PASSED")

    print("=" * 120)

    return results


# ==================================================================================================
# RUN
# ==================================================================================================

fvg_proximity_regression_gate_v1 = (
    regression_gate_fvg_proximity_engine_v1(
        proximity_df=fvg_proximity_engine_v1,
        qualification_df=fvg_qualification_engine_v1,
        zone_df=fvg_validated_v1,
        m5_df=m5_df,
        m5_atr_df=m5_atr_df,
    )
)

In [ ]:
# FVG Interaction / Reaction Engine V1

In [ ]:
# ============================================================
# FVG INTERACTION ENGINE V1
# CONTRACT AUDIT — STEP 1
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. SOURCE VALIDATION
# ------------------------------------------------------------

_REQUIRED_FVG_COLUMNS = {
    "fvg_id",
    "fvg_type",
    "created_at",
    "lower_bound",
    "upper_bound",
    "gap_size",
}

_REQUIRED_M5_COLUMNS = {
    "datetime",
    "open",
    "high",
    "low",
    "close",
}


missing_fvg = _REQUIRED_FVG_COLUMNS - set(fvg_validated_v1.columns)
missing_m5 = _REQUIRED_M5_COLUMNS - set(m5_df.columns)

assert not missing_fvg, f"Missing FVG columns: {sorted(missing_fvg)}"
assert not missing_m5, f"Missing M5 columns: {sorted(missing_m5)}"


# ------------------------------------------------------------
# 2. COPY SOURCES — IMMUTABLE AUDIT INPUT
# ------------------------------------------------------------

_fvg_audit = fvg_validated_v1.copy()
_m5_audit = m5_df.copy()


# ------------------------------------------------------------
# 3. UTC NORMALIZATION CHECK
# ------------------------------------------------------------

_fvg_audit["created_at"] = pd.to_datetime(
    _fvg_audit["created_at"],
    utc=True
)

_m5_audit["datetime"] = pd.to_datetime(
    _m5_audit["datetime"],
    utc=True
)


assert _fvg_audit["created_at"].notna().all(), \
    "FVG created_at contains null timestamps."

assert _m5_audit["datetime"].notna().all(), \
    "M5 datetime contains null timestamps."


# ------------------------------------------------------------
# 4. VALID NUMERIC ZONE CONTRACT
# ------------------------------------------------------------

for col in ["lower_bound", "upper_bound", "gap_size"]:
    _fvg_audit[col] = pd.to_numeric(
        _fvg_audit[col],
        errors="coerce"
    )

for col in ["open", "high", "low", "close"]:
    _m5_audit[col] = pd.to_numeric(
        _m5_audit[col],
        errors="coerce"
    )


assert _fvg_audit[
    ["lower_bound", "upper_bound", "gap_size"]
].notna().all().all(), \
    "FVG zone contains invalid numeric values."

assert _m5_audit[
    ["open", "high", "low", "close"]
].notna().all().all(), \
    "M5 OHLC contains invalid numeric values."


# ------------------------------------------------------------
# 5. ZONE GEOMETRY CONTRACT
# ------------------------------------------------------------

assert (
    _fvg_audit["upper_bound"] >
    _fvg_audit["lower_bound"]
).all(), \
    "Invalid FVG zone: upper_bound must be greater than lower_bound."

_zone_size_check = (
    _fvg_audit["upper_bound"] -
    _fvg_audit["lower_bound"]
)

assert np.allclose(
    _zone_size_check.to_numpy(),
    _fvg_audit["gap_size"].to_numpy(),
    rtol=0,
    atol=1e-12
), \
    "gap_size does not match FVG zone width."


# ------------------------------------------------------------
# 6. BUILD CANDIDATE INTERACTION OBSERVATIONS
#
# Interaction condition:
#
#     candle_high >= FVG_lower
#     AND
#     candle_low  <= FVG_upper
#
# This is RANGE OVERLAP, not close-based proximity.
# ------------------------------------------------------------

_interaction_rows = []

for fvg in _fvg_audit.itertuples(index=False):

    fvg_id = fvg.fvg_id
    created_at = fvg.created_at
    lower = float(fvg.lower_bound)
    upper = float(fvg.upper_bound)

    future_m5 = _m5_audit[
        _m5_audit["datetime"] >= created_at
    ]

    if future_m5.empty:
        continue

    touched = future_m5[
        (future_m5["high"] >= lower) &
        (future_m5["low"] <= upper)
    ].copy()

    if touched.empty:
        continue

    for row in touched.itertuples(index=False):

        close = float(row.close)

        entered = (
            lower <= close <= upper
        )

        _interaction_rows.append({
            "fvg_id": fvg_id,
            "fvg_type": fvg.fvg_type,
            "created_at": created_at,
            "observation_time": row.datetime,
            "lower_bound": lower,
            "upper_bound": upper,
            "gap_size": float(fvg.gap_size),
            "open": float(row.open),
            "high": float(row.high),
            "low": float(row.low),
            "close": close,
            "touch_detected": True,
            "entry_detected": bool(entered),
        })


fvg_interaction_candidates_v1 = pd.DataFrame(
    _interaction_rows
)


# ------------------------------------------------------------
# 7. EMPTY RESULT CONTRACT
# ------------------------------------------------------------

if fvg_interaction_candidates_v1.empty:

    print("=" * 100)
    print("FVG INTERACTION CONTRACT AUDIT — STEP 1")
    print("=" * 100)
    print()
    print("No interaction candidates were detected.")
    print()
    print("This is a valid result, but Interaction Engine cannot")
    print("be audited further until at least one touch exists.")

else:

    # --------------------------------------------------------
    # 8. BASIC METRICS
    # --------------------------------------------------------

    total_fvgs = _fvg_audit["fvg_id"].nunique()

    touched_fvgs = (
        fvg_interaction_candidates_v1["fvg_id"]
        .nunique()
    )

    entered_fvgs = (
        fvg_interaction_candidates_v1.loc[
            fvg_interaction_candidates_v1["entry_detected"],
            "fvg_id"
        ]
        .nunique()
    )

    total_touch_candles = len(
        fvg_interaction_candidates_v1
    )

    total_entry_candles = int(
        fvg_interaction_candidates_v1["entry_detected"].sum()
    )


    # --------------------------------------------------------
    # 9. TOUCH → ENTRY RELATION
    # --------------------------------------------------------

    touch_without_entry = (
        total_touch_candles -
        total_entry_candles
    )


    # --------------------------------------------------------
    # 10. PER-FVG INTERACTION COUNTS
    # --------------------------------------------------------

    interaction_counts = (
        fvg_interaction_candidates_v1
        .groupby("fvg_id")
        .size()
        .sort_values(ascending=False)
    )


    # --------------------------------------------------------
    # 11. FVGs WITH MULTIPLE TOUCH CANDLES
    # --------------------------------------------------------

    multi_touch_fvgs = int(
        (interaction_counts > 1).sum()
    )


    # --------------------------------------------------------
    # 12. PRICE POSITION AT CLOSE
    # --------------------------------------------------------

    candidate_df = (
        fvg_interaction_candidates_v1.copy()
    )

    candidate_df["close_position"] = np.select(
        [
            candidate_df["close"] <
            candidate_df["lower_bound"],

            candidate_df["close"] >
            candidate_df["upper_bound"],
        ],
        [
            "BELOW",
            "ABOVE",
        ],
        default="INSIDE",
    )


    # --------------------------------------------------------
    # 13. OUTPUT
    # --------------------------------------------------------

    print("=" * 100)
    print("FVG INTERACTION CONTRACT AUDIT — STEP 1")
    print("=" * 100)

    print()
    print(f"Total FVGs                         : {total_fvgs}")
    print(f"FVGs with RANGE TOUCH              : {touched_fvgs}")
    print(f"FVGs with CLOSE ENTRY              : {entered_fvgs}")
    print()
    print(f"Total TOUCH candles                : {total_touch_candles}")
    print(f"Total ENTRY candles                : {total_entry_candles}")
    print(f"TOUCH without CLOSE ENTRY          : {touch_without_entry}")
    print()
    print(f"FVGs with multiple touch candles   : {multi_touch_fvgs}")

    print()
    print("CLOSE POSITION DURING TOUCH")
    print("-" * 60)
    print(
        candidate_df["close_position"]
        .value_counts()
        .to_string()
    )

    print()
    print("TOP FVGs BY TOUCH CANDLE COUNT")
    print("-" * 60)
    print(
        interaction_counts
        .head(15)
        .to_string()
    )

    print()
    print("SAMPLE INTERACTION CANDIDATES")
    print("-" * 100)

    print(
        candidate_df[
            [
                "fvg_id",
                "fvg_type",
                "observation_time",
                "lower_bound",
                "upper_bound",
                "high",
                "low",
                "close",
                "touch_detected",
                "entry_detected",
                "close_position",
            ]
        ]
        .head(20)
        .to_string(index=False)
    )

In [ ]:
# ============================================================
# FVG INTERACTION CONTRACT AUDIT — STEP 2
# QUALIFIED FVG EPISODE SEGMENTATION
# FIXED SOURCE CONTRACT
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. SOURCE CONTRACT
# ------------------------------------------------------------

_REQUIRED_QUALIFICATION_COLUMNS = {
    "fvg_id",
    "fvg_type",
    "fvg_direction",
    "created_at",
}

_REQUIRED_VALIDATED_COLUMNS = {
    "fvg_id",
    "fvg_type",
    "created_at",
    "lower_bound",
    "upper_bound",
    "gap_size",
}

_REQUIRED_M5_COLUMNS = {
    "datetime",
    "open",
    "high",
    "low",
    "close",
}


missing_qualification = (
    _REQUIRED_QUALIFICATION_COLUMNS -
    set(fvg_qualification_engine_v1.columns)
)

missing_validated = (
    _REQUIRED_VALIDATED_COLUMNS -
    set(fvg_validated_v1.columns)
)

missing_m5 = (
    _REQUIRED_M5_COLUMNS -
    set(m5_df.columns)
)


assert not missing_qualification, \
    f"Missing qualification columns: {sorted(missing_qualification)}"

assert not missing_validated, \
    f"Missing validation columns: {sorted(missing_validated)}"

assert not missing_m5, \
    f"Missing M5 columns: {sorted(missing_m5)}"


# ------------------------------------------------------------
# 2. QUALIFIED FVG SET
# ------------------------------------------------------------

_qualified_fvg = (
    fvg_qualification_engine_v1[
        [
            "fvg_id",
            "fvg_type",
            "fvg_direction",
            "created_at",
        ]
    ]
    .drop_duplicates("fvg_id")
    .copy()
)


assert (
    _qualified_fvg["fvg_id"].is_unique
), "Qualified FVG IDs are not unique."


# ------------------------------------------------------------
# 3. VALIDATED FVG GEOMETRY
# ------------------------------------------------------------

_fvg_geometry = (
    fvg_validated_v1[
        [
            "fvg_id",
            "lower_bound",
            "upper_bound",
            "gap_size",
        ]
    ]
    .drop_duplicates("fvg_id")
    .copy()
)


assert (
    _fvg_geometry["fvg_id"].is_unique
), "Validated FVG IDs are not unique."


# ------------------------------------------------------------
# 4. JOIN QUALIFIED + GEOMETRY
# ------------------------------------------------------------

_fvg_source = _qualified_fvg.merge(
    _fvg_geometry,
    on="fvg_id",
    how="left",
    validate="one_to_one",
)


assert len(_fvg_source) == len(_qualified_fvg), \
    "Qualified FVG count changed during geometry join."


assert _fvg_source[
    [
        "lower_bound",
        "upper_bound",
        "gap_size",
    ]
].notna().all().all(), \
    "Qualified FVG geometry contains null values."


# ------------------------------------------------------------
# 5. UTC CONTRACT
# ------------------------------------------------------------

_fvg_source["created_at"] = pd.to_datetime(
    _fvg_source["created_at"],
    utc=True
)

_m5_source = m5_df.copy()

_m5_source["datetime"] = pd.to_datetime(
    _m5_source["datetime"],
    utc=True
)

_m5_source = (
    _m5_source
    .sort_values("datetime")
    .reset_index(drop=True)
)


assert _fvg_source["created_at"].notna().all()
assert _m5_source["datetime"].notna().all()


# ------------------------------------------------------------
# 6. NUMERIC CONTRACT
# ------------------------------------------------------------

for col in [
    "lower_bound",
    "upper_bound",
    "gap_size",
]:
    _fvg_source[col] = pd.to_numeric(
        _fvg_source[col],
        errors="coerce"
    )


for col in [
    "open",
    "high",
    "low",
    "close",
]:
    _m5_source[col] = pd.to_numeric(
        _m5_source[col],
        errors="coerce"
    )


assert _fvg_source[
    [
        "lower_bound",
        "upper_bound",
        "gap_size",
    ]
].notna().all().all()

assert _m5_source[
    [
        "open",
        "high",
        "low",
        "close",
    ]
].notna().all().all()


# ------------------------------------------------------------
# 7. ZONE GEOMETRY CONTRACT
# ------------------------------------------------------------

_zone_width = (
    _fvg_source["upper_bound"] -
    _fvg_source["lower_bound"]
)

assert (
    _zone_width > 0
).all(), \
    "Invalid FVG zone: upper_bound must be greater than lower_bound."


assert np.allclose(
    _zone_width.to_numpy(),
    _fvg_source["gap_size"].to_numpy(),
    rtol=0,
    atol=1e-12,
), \
    "gap_size does not match zone width."


# ------------------------------------------------------------
# 8. BUILD RANGE-TOUCH TIMELINE
# ------------------------------------------------------------

_touch_rows = []


for fvg in _fvg_source.itertuples(index=False):

    fvg_id = fvg.fvg_id
    created_at = fvg.created_at

    lower = float(fvg.lower_bound)
    upper = float(fvg.upper_bound)

    future_m5 = _m5_source[
        _m5_source["datetime"] >= created_at
    ].copy()

    if future_m5.empty:
        continue


    # --------------------------------------------------------
    # RANGE OVERLAP CONTRACT
    # --------------------------------------------------------

    future_m5["touch"] = (
        (future_m5["high"] >= lower) &
        (future_m5["low"] <= upper)
    )


    touched = future_m5[
        future_m5["touch"]
    ].copy()


    if touched.empty:
        continue


    touched["fvg_id"] = fvg_id
    touched["fvg_type"] = fvg.fvg_type
    touched["fvg_direction"] = fvg.fvg_direction
    touched["created_at"] = created_at

    touched["lower_bound"] = lower
    touched["upper_bound"] = upper
    touched["gap_size"] = float(fvg.gap_size)


    # --------------------------------------------------------
    # CLOSE ENTRY CONTRACT
    # --------------------------------------------------------

    touched["entry_detected"] = (
        (touched["close"] >= lower) &
        (touched["close"] <= upper)
    )


    _touch_rows.append(
        touched[
            [
                "fvg_id",
                "fvg_type",
                "fvg_direction",
                "created_at",
                "datetime",
                "lower_bound",
                "upper_bound",
                "gap_size",
                "open",
                "high",
                "low",
                "close",
                "entry_detected",
            ]
        ]
    )


assert _touch_rows, \
    "No Touch observations found for qualified FVGs."


_touch_timeline = pd.concat(
    _touch_rows,
    ignore_index=True
)


_touch_timeline = (
    _touch_timeline
    .sort_values(
        [
            "fvg_id",
            "datetime",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. TOUCH CONTINUITY
# ------------------------------------------------------------

_touch_timeline["previous_touch_time"] = (
    _touch_timeline
    .groupby("fvg_id")["datetime"]
    .shift(1)
)


_touch_timeline["minutes_since_previous_touch"] = (
    (
        _touch_timeline["datetime"] -
        _touch_timeline["previous_touch_time"]
    )
    .dt.total_seconds()
    / 60.0
)


# ------------------------------------------------------------
# 10. EPISODE START CONTRACT
#
# First touch:
#     new episode
#
# Gap > one M5 candle:
#     new episode
#
# Gap <= 5 minutes:
#     same episode
# ------------------------------------------------------------

_touch_timeline["new_episode"] = (
    _touch_timeline["previous_touch_time"].isna()
    |
    (
        _touch_timeline[
            "minutes_since_previous_touch"
        ] > 5.0
    )
)


_touch_timeline["interaction_sequence"] = (
    _touch_timeline
    .groupby("fvg_id")["new_episode"]
    .cumsum()
    .astype(int)
)


# ------------------------------------------------------------
# 11. INTERACTION ID
# ------------------------------------------------------------

_touch_timeline["interaction_id"] = (
    "FVGINT_" +
    _touch_timeline["fvg_id"].astype(str) +
    "_I" +
    _touch_timeline["interaction_sequence"]
    .astype(str)
    .str.zfill(3)
)


# ------------------------------------------------------------
# 12. EPISODE SUMMARY
# ------------------------------------------------------------

_episode_summary = (
    _touch_timeline
    .groupby(
        [
            "interaction_id",
            "fvg_id",
            "interaction_sequence",
        ],
        as_index=False
    )
    .agg(
        first_touch_time=("datetime", "min"),
        last_touch_time=("datetime", "max"),
        touch_candle_count=("datetime", "size"),
        entry_candle_count=("entry_detected", "sum"),
    )
)


# ------------------------------------------------------------
# 13. FIRST ENTRY TIME
# ------------------------------------------------------------

_first_entry = (
    _touch_timeline[
        _touch_timeline["entry_detected"]
    ]
    .groupby("interaction_id")["datetime"]
    .min()
    .rename("first_entry_time")
)


_episode_summary = _episode_summary.merge(
    _first_entry,
    on="interaction_id",
    how="left",
    validate="one_to_one",
)


# ------------------------------------------------------------
# 14. EPISODE STATE
# ------------------------------------------------------------

_episode_summary["entry_detected"] = (
    _episode_summary["entry_candle_count"] > 0
)


_episode_summary["episode_state"] = np.where(
    _episode_summary["entry_detected"],
    "ENTERED",
    "TOUCHED_ONLY",
)


# ------------------------------------------------------------
# 15. EPISODE STATISTICS
# ------------------------------------------------------------

total_qualified_fvgs = (
    _fvg_source["fvg_id"].nunique()
)

qualified_with_touch = (
    _touch_timeline["fvg_id"].nunique()
)

total_touch_candles = len(
    _touch_timeline
)

total_episodes = len(
    _episode_summary
)

episodes_per_fvg = (
    _episode_summary
    .groupby("fvg_id")
    .size()
)

multi_episode_fvg_count = int(
    (episodes_per_fvg > 1).sum()
)

max_episodes_per_fvg = int(
    episodes_per_fvg.max()
)


# ------------------------------------------------------------
# 16. EPISODE LENGTH DISTRIBUTION
# ------------------------------------------------------------

episode_length_distribution = (
    _episode_summary[
        "touch_candle_count"
    ]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 17. OUTPUT
# ------------------------------------------------------------

print("=" * 100)
print("FVG INTERACTION CONTRACT AUDIT — STEP 2")
print("QUALIFIED FVG EPISODE SEGMENTATION")
print("=" * 100)

print()

print(
    f"Qualified FVGs                  : "
    f"{total_qualified_fvgs}"
)

print(
    f"Qualified FVGs with Touch       : "
    f"{qualified_with_touch}"
)

print()

print(
    f"Total Touch candles             : "
    f"{total_touch_candles}"
)

print(
    f"Total Interaction Episodes      : "
    f"{total_episodes}"
)

print()

print(
    f"FVGs with multiple Episodes     : "
    f"{multi_episode_fvg_count}"
)

print(
    f"Maximum Episodes per FVG        : "
    f"{max_episodes_per_fvg}"
)

print()

print("EPISODE STATE")
print("-" * 60)

print(
    _episode_summary[
        "episode_state"
    ]
    .value_counts()
    .to_string()
)

print()

print("EPISODE LENGTH DISTRIBUTION")
print("-" * 60)

print(
    episode_length_distribution
    .head(30)
    .to_string()
)

print()

print("TOP FVGs BY EPISODE COUNT")
print("-" * 60)

print(
    episodes_per_fvg
    .sort_values(ascending=False)
    .head(15)
    .to_string()
)

print()

print("SAMPLE INTERACTION EPISODES")
print("-" * 100)

print(
    _episode_summary[
        [
            "interaction_id",
            "fvg_id",
            "interaction_sequence",
            "first_touch_time",
            "first_entry_time",
            "last_touch_time",
            "touch_candle_count",
            "entry_candle_count",
            "episode_state",
        ]
    ]
    .head(30)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# FVG INTERACTION CONTRACT AUDIT — STEP 3
# INTERACTION vs LIFECYCLE BOUNDARY
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. SOURCE CONTRACT
# ------------------------------------------------------------

_required_episode_columns = {
    "interaction_id",
    "fvg_id",
    "interaction_sequence",
    "first_touch_time",
    "last_touch_time",
}

_required_lifecycle_columns = {
    "fvg_id",
    "lifecycle_state",
    "event_type",
    "event_time",
}


missing_episode = (
    _required_episode_columns -
    set(_episode_summary.columns)
)

missing_lifecycle = (
    _required_lifecycle_columns -
    set(fvg_lifecycle_reference_v1.columns)
)

assert not missing_episode, (
    f"Missing episode columns: {sorted(missing_episode)}"
)

assert not missing_lifecycle, (
    f"Missing lifecycle columns: {sorted(missing_lifecycle)}"
)


# ------------------------------------------------------------
# 2. PREPARE LIFECYCLE
# ------------------------------------------------------------

_lifecycle = (
    fvg_lifecycle_reference_v1[
        [
            "fvg_id",
            "lifecycle_state",
            "event_type",
            "event_time",
        ]
    ]
    .copy()
)

_lifecycle["event_time"] = pd.to_datetime(
    _lifecycle["event_time"],
    utc=True
)


# ------------------------------------------------------------
# 3. IDENTIFY TERMINAL LIFECYCLE EVENT
#
# Current lifecycle vocabulary:
#     FRESH
#     MITIGATED
#     FILLED
#
# For this audit, the first MITIGATED/FILLED event is treated
# as the candidate boundary.
# ------------------------------------------------------------

_terminal_events = (
    _lifecycle[
        _lifecycle["lifecycle_state"].isin(
            ["MITIGATED", "FILLED"]
        )
        |
        _lifecycle["event_type"].isin(
            ["MITIGATION", "FILL"]
        )
    ]
    .sort_values(
        [
            "fvg_id",
            "event_time",
        ]
    )
    .groupby("fvg_id", as_index=False)
    .first()
)


_terminal_events = _terminal_events.rename(
    columns={
        "lifecycle_state": "terminal_lifecycle_state",
        "event_type": "terminal_event_type",
        "event_time": "terminal_event_time",
    }
)


# ------------------------------------------------------------
# 4. JOIN EPISODES TO LIFECYCLE
# ------------------------------------------------------------

_episode_lifecycle_audit = (
    _episode_summary[
        [
            "interaction_id",
            "fvg_id",
            "interaction_sequence",
            "first_touch_time",
            "first_entry_time",
            "last_touch_time",
            "touch_candle_count",
            "entry_candle_count",
            "episode_state",
        ]
    ]
    .merge(
        _terminal_events[
            [
                "fvg_id",
                "terminal_lifecycle_state",
                "terminal_event_type",
                "terminal_event_time",
            ]
        ],
        on="fvg_id",
        how="left",
        validate="many_to_one",
    )
)


# ------------------------------------------------------------
# 5. CLASSIFY EPISODE RELATIVE TO LIFECYCLE
# ------------------------------------------------------------

def classify_lifecycle_relation(row):

    terminal_time = row["terminal_event_time"]

    if pd.isna(terminal_time):
        return "NO_TERMINAL_EVENT"

    first_touch = row["first_touch_time"]
    last_touch = row["last_touch_time"]

    if last_touch < terminal_time:
        return "BEFORE_LIFECYCLE_EVENT"

    if (
        first_touch <= terminal_time <= last_touch
    ):
        return "CROSSES_LIFECYCLE_EVENT"

    if first_touch > terminal_time:
        return "AFTER_LIFECYCLE_EVENT"

    return "UNKNOWN"


_episode_lifecycle_audit[
    "lifecycle_relation"
] = _episode_lifecycle_audit.apply(
    classify_lifecycle_relation,
    axis=1
)


# ------------------------------------------------------------
# 6. COUNTS
# ------------------------------------------------------------

_relation_counts = (
    _episode_lifecycle_audit[
        "lifecycle_relation"
    ]
    .value_counts()
)


# ------------------------------------------------------------
# 7. FVG-LEVEL COUNTS
# ------------------------------------------------------------

_fvg_relation_counts = (
    _episode_lifecycle_audit
    .groupby(
        [
            "fvg_id",
            "lifecycle_relation",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)


# ------------------------------------------------------------
# 8. POST-LIFECYCLE EPISODES
# ------------------------------------------------------------

_post_lifecycle = (
    _episode_lifecycle_audit[
        _episode_lifecycle_audit[
            "lifecycle_relation"
        ] == "AFTER_LIFECYCLE_EVENT"
    ]
    .copy()
)


# ------------------------------------------------------------
# 9. OUTPUT
# ------------------------------------------------------------

print("=" * 100)
print("FVG INTERACTION CONTRACT AUDIT — STEP 3")
print("INTERACTION vs LIFECYCLE BOUNDARY")
print("=" * 100)

print()

print(
    f"Total Interaction Episodes      : "
    f"{len(_episode_lifecycle_audit)}"
)

print(
    f"FVGs with terminal lifecycle    : "
    f"{_terminal_events['fvg_id'].nunique()}"
)

print()

print("EPISODE / LIFECYCLE RELATION")
print("-" * 70)

print(
    _relation_counts
    .to_string()
)

print()

print("TERMINAL LIFECYCLE STATES")
print("-" * 70)

print(
    _terminal_events[
        "terminal_lifecycle_state"
    ]
    .value_counts()
    .to_string()
)

print()

print("POST-LIFECYCLE EPISODES")
print("-" * 70)

print(
    f"Count                           : "
    f"{len(_post_lifecycle)}"
)

print(
    f"Unique FVGs                      : "
    f"{_post_lifecycle['fvg_id'].nunique()}"
)

print()

print("SAMPLE POST-LIFECYCLE EPISODES")
print("-" * 100)

if _post_lifecycle.empty:

    print("NONE")

else:

    print(
        _post_lifecycle[
            [
                "interaction_id",
                "fvg_id",
                "interaction_sequence",
                "first_touch_time",
                "last_touch_time",
                "terminal_lifecycle_state",
                "terminal_event_type",
                "terminal_event_time",
                "episode_state",
            ]
        ]
        .head(30)
        .to_string(index=False)
    )


print()

print("FVGs WITH MOST POST-LIFECYCLE EPISODES")
print("-" * 100)

if _post_lifecycle.empty:

    print("NONE")

else:

    print(
        _post_lifecycle[
            "fvg_id"
        ]
        .value_counts()
        .head(15)
        .to_string()
    )

In [ ]:
# ============================================================
# FVG INTERACTION CONTRACT AUDIT — STEP 4
# LIFECYCLE-BOUNDED INTERACTION UNIVERSE
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. SOURCE CONTRACT
# ------------------------------------------------------------

_required_episode_columns = {
    "interaction_id",
    "fvg_id",
    "interaction_sequence",
    "first_touch_time",
    "first_entry_time",
    "last_touch_time",
    "touch_candle_count",
    "entry_candle_count",
    "episode_state",
}

_required_lifecycle_columns = {
    "fvg_id",
    "lifecycle_state",
    "event_type",
    "event_time",
}

assert not (
    _required_episode_columns -
    set(_episode_summary.columns)
), (
    "Missing episode columns: "
    + str(
        sorted(
            _required_episode_columns -
            set(_episode_summary.columns)
        )
    )
)

assert not (
    _required_lifecycle_columns -
    set(fvg_lifecycle_reference_v1.columns)
), (
    "Missing lifecycle columns: "
    + str(
        sorted(
            _required_lifecycle_columns -
            set(fvg_lifecycle_reference_v1.columns)
        )
    )
)


# ------------------------------------------------------------
# 2. PREPARE LIFECYCLE BOUNDARY
# ------------------------------------------------------------

_lifecycle_boundary = (
    fvg_lifecycle_reference_v1[
        [
            "fvg_id",
            "lifecycle_state",
            "event_type",
            "event_time",
        ]
    ]
    .copy()
)

_lifecycle_boundary["event_time"] = pd.to_datetime(
    _lifecycle_boundary["event_time"],
    utc=True
)


# ------------------------------------------------------------
# 3. TERMINAL EVENT
# ------------------------------------------------------------

_terminal_boundary = (
    _lifecycle_boundary[
        _lifecycle_boundary["lifecycle_state"].isin(
            ["MITIGATED", "FILLED"]
        )
        |
        _lifecycle_boundary["event_type"].isin(
            ["MITIGATION", "FILL"]
        )
    ]
    .sort_values(
        [
            "fvg_id",
            "event_time",
        ]
    )
    .groupby(
        "fvg_id",
        as_index=False
    )
    .first()
    .rename(
        columns={
            "lifecycle_state": "terminal_lifecycle_state",
            "event_type": "terminal_event_type",
            "event_time": "terminal_event_time",
        }
    )
)


# ------------------------------------------------------------
# 4. JOIN EPISODES
# ------------------------------------------------------------

_episode_bounded_audit = (
    _episode_summary[
        [
            "interaction_id",
            "fvg_id",
            "interaction_sequence",
            "first_touch_time",
            "first_entry_time",
            "last_touch_time",
            "touch_candle_count",
            "entry_candle_count",
            "episode_state",
        ]
    ]
    .merge(
        _terminal_boundary[
            [
                "fvg_id",
                "terminal_lifecycle_state",
                "terminal_event_type",
                "terminal_event_time",
            ]
        ],
        on="fvg_id",
        how="left",
        validate="many_to_one",
    )
)


# ------------------------------------------------------------
# 5. CLASSIFY EPISODE BOUNDARY
#
# VALID:
#     entire episode before terminal event
#
# TRUNCATE:
#     episode starts before/equal terminal event
#     but extends beyond it
#
# INVALID:
#     entire episode starts after terminal event
#
# OPEN:
#     FVG has no terminal lifecycle event
# ------------------------------------------------------------

def classify_bounded_status(row):

    terminal_time = row["terminal_event_time"]

    first_touch = row["first_touch_time"]
    last_touch = row["last_touch_time"]

    if pd.isna(terminal_time):
        return "OPEN_LIFECYCLE"

    if last_touch <= terminal_time:
        return "VALID_BEFORE_OR_AT_TERMINAL"

    if first_touch <= terminal_time < last_touch:
        return "TRUNCATE_AT_TERMINAL"

    if first_touch > terminal_time:
        return "INVALID_AFTER_TERMINAL"

    return "UNKNOWN"


_episode_bounded_audit[
    "bounded_status"
] = _episode_bounded_audit.apply(
    classify_bounded_status,
    axis=1
)


# ------------------------------------------------------------
# 6. COUNTS
# ------------------------------------------------------------

_status_counts = (
    _episode_bounded_audit[
        "bounded_status"
    ]
    .value_counts()
)


# ------------------------------------------------------------
# 7. VALID INTERACTION UNIVERSE
# ------------------------------------------------------------

_valid_episode_candidates = (
    _episode_bounded_audit[
        _episode_bounded_audit[
            "bounded_status"
        ].isin(
            [
                "VALID_BEFORE_OR_AT_TERMINAL",
                "TRUNCATE_AT_TERMINAL",
                "OPEN_LIFECYCLE",
            ]
        )
    ]
    .copy()
)


# ------------------------------------------------------------
# 8. INVALID POST-LIFECYCLE EPISODES
# ------------------------------------------------------------

_invalid_post_terminal = (
    _episode_bounded_audit[
        _episode_bounded_audit[
            "bounded_status"
        ] == "INVALID_AFTER_TERMINAL"
    ]
    .copy()
)


# ------------------------------------------------------------
# 9. TRUNCATED EPISODES
# ------------------------------------------------------------

_truncated = (
    _episode_bounded_audit[
        _episode_bounded_audit[
            "bounded_status"
        ] == "TRUNCATE_AT_TERMINAL"
    ]
    .copy()
)


# ------------------------------------------------------------
# 10. OUTPUT
# ------------------------------------------------------------

print("=" * 100)
print("FVG INTERACTION CONTRACT AUDIT — STEP 4")
print("LIFECYCLE-BOUNDED INTERACTION UNIVERSE")
print("=" * 100)

print()

print(
    f"Original Interaction Episodes   : "
    f"{len(_episode_bounded_audit)}"
)

print()

print("BOUNDED STATUS")
print("-" * 70)

print(
    _status_counts
    .to_string()
)

print()

print("VALID INTERACTION CANDIDATES")
print("-" * 70)

print(
    f"Count                           : "
    f"{len(_valid_episode_candidates)}"
)

print(
    f"Unique FVGs                     : "
    f"{_valid_episode_candidates['fvg_id'].nunique()}"
)

print()

print("INVALID POST-TERMINAL EPISODES")
print("-" * 70)

print(
    f"Count                           : "
    f"{len(_invalid_post_terminal)}"
)

print(
    f"Unique FVGs                     : "
    f"{_invalid_post_terminal['fvg_id'].nunique()}"
)

print()

print("TRUNCATED EPISODES")
print("-" * 70)

print(
    f"Count                           : "
    f"{len(_truncated)}"
)

print(
    f"Unique FVGs                     : "
    f"{_truncated['fvg_id'].nunique()}"
)

print()

print("SAMPLE TRUNCATED EPISODES")
print("-" * 100)

if _truncated.empty:

    print("NONE")

else:

    print(
        _truncated[
            [
                "interaction_id",
                "fvg_id",
                "interaction_sequence",
                "first_touch_time",
                "last_touch_time",
                "terminal_lifecycle_state",
                "terminal_event_type",
                "terminal_event_time",
                "episode_state",
            ]
        ]
        .head(30)
        .to_string(index=False)
    )

print()

print("SAMPLE INVALID POST-TERMINAL EPISODES")
print("-" * 100)

if _invalid_post_terminal.empty:

    print("NONE")

else:

    print(
        _invalid_post_terminal[
            [
                "interaction_id",
                "fvg_id",
                "interaction_sequence",
                "first_touch_time",
                "last_touch_time",
                "terminal_lifecycle_state",
                "terminal_event_type",
                "terminal_event_time",
                "episode_state",
            ]
        ]
        .head(30)
        .to_string(index=False)
    )

In [ ]:
# ============================================================
# FVG INTERACTION CONTRACT AUDIT — STEP 5
# BOUNDED CANDLE UNIVERSE + TERMINAL CANDLE CONTRACT
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. SOURCE CONTRACT
# ------------------------------------------------------------

_required_episode_columns = {
    "interaction_id",
    "fvg_id",
    "interaction_sequence",
    "first_touch_time",
    "last_touch_time",
    "episode_state",
}

_required_fvg_columns = {
    "fvg_id",
    "created_at",
    "lower_bound",
    "upper_bound",
}

_required_m5_columns = {
    "datetime",
    "open",
    "high",
    "low",
    "close",
}

_required_lifecycle_columns = {
    "fvg_id",
    "lifecycle_state",
    "event_type",
    "event_time",
}


assert not (
    _required_episode_columns -
    set(_episode_summary.columns)
), (
    "Missing episode columns: "
    + str(
        sorted(
            _required_episode_columns -
            set(_episode_summary.columns)
        )
    )
)

assert not (
    _required_fvg_columns -
    set(fvg_validated_v1.columns)
), (
    "Missing FVG columns: "
    + str(
        sorted(
            _required_fvg_columns -
            set(fvg_validated_v1.columns)
        )
    )
)

assert not (
    _required_m5_columns -
    set(m5_df.columns)
), (
    "Missing M5 columns: "
    + str(
        sorted(
            _required_m5_columns -
            set(m5_df.columns)
        )
    )
)

assert not (
    _required_lifecycle_columns -
    set(fvg_lifecycle_reference_v1.columns)
), (
    "Missing lifecycle columns: "
    + str(
        sorted(
            _required_lifecycle_columns -
            set(fvg_lifecycle_reference_v1.columns)
        )
    )
)

# ------------------------------------------------------------
# 2. PREPARE FVG SOURCE
# ------------------------------------------------------------

_fvg_source = (
    fvg_validated_v1[
        [
            "fvg_id",
            "created_at",
            "lower_bound",
            "upper_bound",
        ]
    ]
    .copy()
)

_fvg_source["created_at"] = pd.to_datetime(
    _fvg_source["created_at"],
    utc=True
)

_fvg_source["lower_bound"] = pd.to_numeric(
    _fvg_source["lower_bound"],
    errors="coerce"
)

_fvg_source["upper_bound"] = pd.to_numeric(
    _fvg_source["upper_bound"],
    errors="coerce"
)


# ------------------------------------------------------------
# 3. PREPARE LIFECYCLE BOUNDARY
# ------------------------------------------------------------

_lifecycle_source = (
    fvg_lifecycle_reference_v1[
        [
            "fvg_id",
            "lifecycle_state",
            "event_type",
            "event_time",
        ]
    ]
    .copy()
)

_lifecycle_source["event_time"] = pd.to_datetime(
    _lifecycle_source["event_time"],
    utc=True
)


_terminal_events = (
    _lifecycle_source[
        _lifecycle_source["lifecycle_state"].isin(
            ["MITIGATED", "FILLED"]
        )
        |
        _lifecycle_source["event_type"].isin(
            ["MITIGATION", "FILL"]
        )
    ]
    .sort_values(
        [
            "fvg_id",
            "event_time",
        ]
    )
    .groupby(
        "fvg_id",
        as_index=False
    )
    .first()
    .rename(
        columns={
            "lifecycle_state": "terminal_lifecycle_state",
            "event_type": "terminal_event_type",
            "event_time": "terminal_event_time",
        }
    )
)


# ------------------------------------------------------------
# 4. PREPARE M5 TIMELINE
# ------------------------------------------------------------

_m5 = (
    m5_df[
        [
            "datetime",
            "open",
            "high",
            "low",
            "close",
        ]
    ]
    .copy()
)

_m5["datetime"] = pd.to_datetime(
    _m5["datetime"],
    utc=True
)

_m5 = (
    _m5
    .sort_values("datetime")
    .drop_duplicates(
        subset=["datetime"],
        keep="first"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. BUILD FULL TOUCH CANDLE UNIVERSE
#
# Touch contract:
#
#     high >= lower_bound
#     AND
#     low <= upper_bound
#
# Observation starts at created_at.
# ------------------------------------------------------------

_touch_rows = []

for _, fvg in _fvg_source.iterrows():

    fvg_id = fvg["fvg_id"]
    created_at = fvg["created_at"]
    lower = fvg["lower_bound"]
    upper = fvg["upper_bound"]

    if (
        pd.isna(created_at)
        or pd.isna(lower)
        or pd.isna(upper)
    ):
        continue

    candidate = _m5[
        _m5["datetime"] >= created_at
    ].copy()

    candidate["touch"] = (
        (candidate["high"] >= lower)
        &
        (candidate["low"] <= upper)
    )

    candidate = candidate[
        candidate["touch"]
    ].copy()

    if candidate.empty:
        continue

    candidate["fvg_id"] = fvg_id
    candidate["created_at"] = created_at
    candidate["lower_bound"] = lower
    candidate["upper_bound"] = upper

    _touch_rows.append(
        candidate[
            [
                "fvg_id",
                "created_at",
                "datetime",
                "open",
                "high",
                "low",
                "close",
                "lower_bound",
                "upper_bound",
            ]
        ]
    )


_touch_candles = (
    pd.concat(
        _touch_rows,
        ignore_index=True
    )
    if _touch_rows
    else pd.DataFrame()
)


# ------------------------------------------------------------
# 6. JOIN TERMINAL BOUNDARY
# ------------------------------------------------------------

_touch_candles = (
    _touch_candles
    .merge(
        _terminal_events[
            [
                "fvg_id",
                "terminal_lifecycle_state",
                "terminal_event_type",
                "terminal_event_time",
            ]
        ],
        on="fvg_id",
        how="left",
        validate="many_to_one",
    )
)


# ------------------------------------------------------------
# 7. CLASSIFY EVERY TOUCH CANDLE
#
# IMPORTANT CONTRACT:
#
# terminal candle IS INCLUDED
#
# Valid:
#     datetime <= terminal_event_time
#
# Invalid:
#     datetime > terminal_event_time
#
# FRESH:
#     no terminal event
# ------------------------------------------------------------

def classify_touch_boundary(row):

    observation_time = row["datetime"]
    terminal_time = row["terminal_event_time"]

    if pd.isna(terminal_time):
        return "OPEN_LIFECYCLE"

    if observation_time <= terminal_time:
        return "VALID_AT_OR_BEFORE_TERMINAL"

    return "INVALID_AFTER_TERMINAL"


_touch_candles["boundary_status"] = (
    _touch_candles.apply(
        classify_touch_boundary,
        axis=1
    )
)


# ------------------------------------------------------------
# 8. BOUNDED TOUCH CANDLES
# ------------------------------------------------------------

_bounded_touch_candles = (
    _touch_candles[
        _touch_candles[
            "boundary_status"
        ].isin(
            [
                "VALID_AT_OR_BEFORE_TERMINAL",
                "OPEN_LIFECYCLE",
            ]
        )
    ]
    .copy()
)


_invalid_touch_candles = (
    _touch_candles[
        _touch_candles[
            "boundary_status"
        ] == "INVALID_AFTER_TERMINAL"
    ]
    .copy()
)


# ------------------------------------------------------------
# 9. TOUCH / TERMINAL CANDLE RELATION
# ------------------------------------------------------------

_terminal_touch = (
    _touch_candles[
        _touch_candles["datetime"]
        ==
        _touch_candles["terminal_event_time"]
    ]
    .copy()
)


_terminal_not_touch = (
    _terminal_events[
        ~_terminal_events["fvg_id"].isin(
            _terminal_touch["fvg_id"]
        )
    ]
    .copy()
)


# ------------------------------------------------------------
# 10. FVG-LEVEL COVERAGE
# ------------------------------------------------------------

_fvg_touch_counts = (
    _bounded_touch_candles
    .groupby("fvg_id")
    .size()
)

_fvg_invalid_touch_counts = (
    _invalid_touch_candles
    .groupby("fvg_id")
    .size()
)


# ------------------------------------------------------------
# 11. TEMPORAL CONTRACT AUDITS
# ------------------------------------------------------------

_post_terminal_bounded = (
    _bounded_touch_candles
    .merge(
        _terminal_events[
            [
                "fvg_id",
                "terminal_event_time",
            ]
        ],
        on="fvg_id",
        how="left",
        validate="many_to_one",
        suffixes=("", "_boundary")
    )
)

_post_terminal_bounded_count = (
    (
        _post_terminal_bounded["datetime"]
        >
        _post_terminal_bounded["terminal_event_time"]
    )
    .sum()
)


# ------------------------------------------------------------
# 12. OUTPUT
# ------------------------------------------------------------

print("=" * 100)
print("FVG INTERACTION CONTRACT AUDIT — STEP 5")
print("BOUNDED CANDLE UNIVERSE + TERMINAL CANDLE CONTRACT")
print("=" * 100)

print()

print(
    f"Full Touch Candles              : "
    f"{len(_touch_candles):,}"
)

print(
    f"Bounded Touch Candles            : "
    f"{len(_bounded_touch_candles):,}"
)

print(
    f"Invalid Post-Terminal Touches    : "
    f"{len(_invalid_touch_candles):,}"
)

print()

print("BOUNDARY STATUS")
print("-" * 70)

print(
    _touch_candles[
        "boundary_status"
    ]
    .value_counts()
    .to_string()
)

print()

print("FVG COVERAGE")
print("-" * 70)

print(
    f"FVGs with bounded touches        : "
    f"{_bounded_touch_candles['fvg_id'].nunique()}"
)

print(
    f"FVGs with invalid touches       : "
    f"{_invalid_touch_candles['fvg_id'].nunique()}"
)

print()

print("TERMINAL CANDLE CONTRACT")
print("-" * 70)

print(
    f"Terminal events                 : "
    f"{len(_terminal_events)}"
)

print(
    f"Terminal events with touch      : "
    f"{len(_terminal_touch)}"
)

print(
    f"Terminal events without touch   : "
    f"{len(_terminal_not_touch)}"
)

print()

print("TEMPORAL INTEGRITY")
print("-" * 70)

print(
    f"Bounded candles after terminal  : "
    f"{_post_terminal_bounded_count}"
)

print()

print("TOUCH CANDLE DISTRIBUTION")
print("-" * 70)

if not _fvg_touch_counts.empty:

    print(
        _fvg_touch_counts
        .describe()
        .to_string()
    )

print()

print("SAMPLE TERMINAL TOUCHES")
print("-" * 100)

if _terminal_touch.empty:

    print("NONE")

else:

    print(
        _terminal_touch[
            [
                "fvg_id",
                "datetime",
                "high",
                "low",
                "close",
                "lower_bound",
                "upper_bound",
                "terminal_lifecycle_state",
                "terminal_event_type",
            ]
        ]
        .head(30)
        .to_string(index=False)
    )

print()

print("SAMPLE INVALID POST-TERMINAL TOUCHES")
print("-" * 100)

if _invalid_touch_candles.empty:

    print("NONE")

else:

    print(
        _invalid_touch_candles[
            [
                "fvg_id",
                "datetime",
                "high",
                "low",
                "close",
                "terminal_lifecycle_state",
                "terminal_event_type",
                "terminal_event_time",
            ]
        ]
        .head(30)
        .to_string(index=False)
    )

In [ ]:
# ====================================================================================================
# FVG INTERACTION CONTRACT AUDIT — STEP 6
# PENETRATION GEOMETRY AUDIT V1
# ====================================================================================================

import numpy as np
import pandas as pd


print("=" * 100)
print("FVG INTERACTION CONTRACT AUDIT — STEP 6")
print("PENETRATION GEOMETRY AUDIT V1")
print("=" * 100)


# ----------------------------------------------------------------------------------------------------
# 1. REQUIRED INPUT CONTRACT
# ----------------------------------------------------------------------------------------------------

_required_bounded = {
    "fvg_id",
    "datetime",
    "high",
    "low",
    "close",
    "lower_bound",
    "upper_bound",
}

_missing_bounded = _required_bounded - set(_bounded_touch_candles.columns)

assert not _missing_bounded, (
    f"Missing required columns in _bounded_touch_candles: "
    f"{sorted(_missing_bounded)}"
)


# ----------------------------------------------------------------------------------------------------
# 2. COPY + NORMALIZE
# ----------------------------------------------------------------------------------------------------

_penetration_audit = _bounded_touch_candles[
    [
        "fvg_id",
        "datetime",
        "high",
        "low",
        "close",
        "lower_bound",
        "upper_bound",
    ]
].copy()


_penetration_audit["datetime"] = pd.to_datetime(
    _penetration_audit["datetime"],
    utc=True
)

_penetration_audit = _penetration_audit.sort_values(
    ["fvg_id", "datetime"]
).reset_index(drop=True)


# ----------------------------------------------------------------------------------------------------
# 3. BASIC ZONE GEOMETRY
# ----------------------------------------------------------------------------------------------------

_penetration_audit["zone_width"] = (
    _penetration_audit["upper_bound"]
    - _penetration_audit["lower_bound"]
)


# ----------------------------------------------------------------------------------------------------
# 4. GEOMETRIC OVERLAP
# ----------------------------------------------------------------------------------------------------
#
# Candle interval:
#     [low, high]
#
# FVG interval:
#     [lower_bound, upper_bound]
#
# Overlap interval:
#     [
#         max(low, lower_bound),
#         min(high, upper_bound)
#     ]
#
# Because this dataset already contains TOUCH candles, overlap should
# normally be positive.
# ----------------------------------------------------------------------------------------------------

_penetration_audit["overlap_low"] = (
    _penetration_audit[["low", "lower_bound"]]
    .max(axis=1)
)

_penetration_audit["overlap_high"] = (
    _penetration_audit[["high", "upper_bound"]]
    .min(axis=1)
)

_penetration_audit["touch_overlap"] = (
    _penetration_audit["overlap_high"]
    - _penetration_audit["overlap_low"]
).clip(lower=0.0)


# ----------------------------------------------------------------------------------------------------
# 5. PENETRATION
# ----------------------------------------------------------------------------------------------------
#
# For the current neutral geometric contract:
#
# penetration = length of candle/FVG overlap
#
# This intentionally does NOT interpret bullish/bearish direction.
# ----------------------------------------------------------------------------------------------------

_penetration_audit["penetration"] = (
    _penetration_audit["touch_overlap"]
)


# ----------------------------------------------------------------------------------------------------
# 6. PENETRATION RATIO
# ----------------------------------------------------------------------------------------------------

_penetration_audit["penetration_ratio"] = np.where(
    _penetration_audit["zone_width"] > 0,
    _penetration_audit["penetration"]
    / _penetration_audit["zone_width"],
    np.nan
)


# ----------------------------------------------------------------------------------------------------
# 7. CLOSE POSITION RELATIVE TO FVG
# ----------------------------------------------------------------------------------------------------

_penetration_audit["close_inside"] = (
    (_penetration_audit["close"] >= _penetration_audit["lower_bound"])
    &
    (_penetration_audit["close"] <= _penetration_audit["upper_bound"])
)


_penetration_audit["close_position"] = np.select(
    [
        _penetration_audit["close"] < _penetration_audit["lower_bound"],
        _penetration_audit["close"].between(
            _penetration_audit["lower_bound"],
            _penetration_audit["upper_bound"],
            inclusive="both"
        ),
        _penetration_audit["close"] > _penetration_audit["upper_bound"],
    ],
    [
        "BELOW",
        "INSIDE",
        "ABOVE",
    ],
    default="UNKNOWN",
)


# ----------------------------------------------------------------------------------------------------
# 8. FULL-ZONE CROSSING
# ----------------------------------------------------------------------------------------------------
#
# A candle crosses the complete FVG zone when:
#
#     low  <= lower_bound
#     high >= upper_bound
#
# This is a purely geometric observation.
# ----------------------------------------------------------------------------------------------------

_penetration_audit["full_zone_crossed"] = (
    (_penetration_audit["low"] <= _penetration_audit["lower_bound"])
    &
    (_penetration_audit["high"] >= _penetration_audit["upper_bound"])
)


# ----------------------------------------------------------------------------------------------------
# 9. TOUCH VALIDATION
# ----------------------------------------------------------------------------------------------------

_penetration_audit["touch_overlap_valid"] = (
    _penetration_audit["touch_overlap"] > 0
)


# ----------------------------------------------------------------------------------------------------
# 10. NUMERIC SANITY
# ----------------------------------------------------------------------------------------------------

_numeric_columns = [
    "high",
    "low",
    "close",
    "lower_bound",
    "upper_bound",
    "zone_width",
    "touch_overlap",
    "penetration",
    "penetration_ratio",
]

for _col in _numeric_columns:
    assert np.isfinite(
        _penetration_audit[_col].dropna()
    ).all(), (
        f"Non-finite numeric values detected in column: {_col}"
    )


# ----------------------------------------------------------------------------------------------------
# 11. GEOMETRIC CONTRACT ASSERTIONS
# ----------------------------------------------------------------------------------------------------

assert (
    _penetration_audit["zone_width"] > 0
).all(), "Invalid FVG geometry: zone_width must be > 0"


assert (
    _penetration_audit["penetration"] >= 0
).all(), "Penetration cannot be negative"


assert (
    _penetration_audit["penetration_ratio"] >= 0
).all(), (
    "Penetration ratio cannot be negative"
)


assert (
    _penetration_audit["penetration_ratio"] <= 1
).all(), (
    "Penetration ratio cannot exceed 1 under the current overlap definition"
)


assert (
    _penetration_audit["penetration"]
    == _penetration_audit["touch_overlap"]
).all(), (
    "Penetration must equal geometric touch overlap"
)


# ----------------------------------------------------------------------------------------------------
# 12. OUTPUT SCHEMA
# ----------------------------------------------------------------------------------------------------

_penetration_audit_columns = [
    "fvg_id",
    "datetime",
    "lower_bound",
    "upper_bound",
    "zone_width",
    "high",
    "low",
    "close",
    "overlap_low",
    "overlap_high",
    "touch_overlap",
    "penetration",
    "penetration_ratio",
    "close_inside",
    "close_position",
    "full_zone_crossed",
    "touch_overlap_valid",
]

_penetration_audit = _penetration_audit[
    _penetration_audit_columns
].copy()


# ----------------------------------------------------------------------------------------------------
# 13. AUDIT SUMMARY
# ----------------------------------------------------------------------------------------------------

print()
print("BASIC COUNTS")
print("-" * 100)

print(f"Bounded Touch Candles       : {len(_penetration_audit):,}")
print(f"Unique FVGs                : {_penetration_audit['fvg_id'].nunique():,}")
print(
    f"Unique Observation Times   : "
    f"{_penetration_audit['datetime'].nunique():,}"
)


print()
print("ZONE GEOMETRY")
print("-" * 100)

print(
    _penetration_audit["zone_width"]
    .describe()
    .to_string()
)


print()
print("PENETRATION")
print("-" * 100)

print(
    _penetration_audit["penetration"]
    .describe()
    .to_string()
)


print()
print("PENETRATION RATIO")
print("-" * 100)

print(
    _penetration_audit["penetration_ratio"]
    .describe()
    .to_string()
)


print()
print("CLOSE POSITION")
print("-" * 100)

print(
    _penetration_audit["close_position"]
    .value_counts(dropna=False)
    .to_string()
)


print()
print("CLOSE INSIDE")
print("-" * 100)

print(
    _penetration_audit["close_inside"]
    .value_counts(dropna=False)
    .to_string()
)


print()
print("FULL ZONE CROSSING")
print("-" * 100)

print(
    _penetration_audit["full_zone_crossed"]
    .value_counts(dropna=False)
    .to_string()
)


print()
print("TOUCH OVERLAP VALIDATION")
print("-" * 100)

print(
    _penetration_audit["touch_overlap_valid"]
    .value_counts(dropna=False)
    .to_string()
)


# ----------------------------------------------------------------------------------------------------
# 14. PENETRATION RATIO BUCKETS
# ----------------------------------------------------------------------------------------------------

_penetration_ratio_bucket = pd.cut(
    _penetration_audit["penetration_ratio"],
    bins=[
        -np.inf,
        0.0,
        0.25,
        0.50,
        0.75,
        1.0,
        np.inf,
    ],
    labels=[
        "ZERO",
        "0-25%",
        "25-50%",
        "50-75%",
        "75-100%",
        ">100%",
    ],
    include_lowest=True,
)

print()
print("PENETRATION RATIO DISTRIBUTION")
print("-" * 100)

print(
    _penetration_ratio_bucket
    .value_counts(sort=False, dropna=False)
    .to_string()
)


# ----------------------------------------------------------------------------------------------------
# 15. SAMPLE
# ----------------------------------------------------------------------------------------------------

print()
print("SAMPLE PENETRATION OBSERVATIONS")
print("-" * 100)

print(
    _penetration_audit[
        [
            "fvg_id",
            "datetime",
            "lower_bound",
            "upper_bound",
            "high",
            "low",
            "close",
            "penetration",
            "penetration_ratio",
            "close_inside",
            "close_position",
            "full_zone_crossed",
        ]
    ]
    .head(30)
    .to_string(index=False)
)


print()
print("=" * 100)
print("STEP 6 AUDIT COMPLETE")
print("=" * 100)

In [ ]:
# ====================================================================================================
# FVG INTERACTION CONTRACT AUDIT — STEP 6B
# PENETRATION DIRECTION & BOUNDARY AUDIT V1
# ====================================================================================================

import numpy as np
import pandas as pd


print("=" * 100)
print("FVG INTERACTION CONTRACT AUDIT — STEP 6B")
print("PENETRATION DIRECTION & BOUNDARY AUDIT V1")
print("=" * 100)


# ----------------------------------------------------------------------------------------------------
# 1. REQUIRED INPUT CONTRACT
# ----------------------------------------------------------------------------------------------------

_required_columns = {
    "fvg_id",
    "datetime",
    "high",
    "low",
    "close",
    "lower_bound",
    "upper_bound",
    "zone_width",
    "touch_overlap",
    "penetration",
    "penetration_ratio",
}

_missing_columns = _required_columns - set(_penetration_audit.columns)

assert not _missing_columns, (
    f"Missing required columns in _penetration_audit: "
    f"{sorted(_missing_columns)}"
)


# ----------------------------------------------------------------------------------------------------
# 2. COPY
# ----------------------------------------------------------------------------------------------------

_penetration_boundary_audit = _penetration_audit.copy()

_penetration_boundary_audit["datetime"] = pd.to_datetime(
    _penetration_boundary_audit["datetime"],
    utc=True
)


# ----------------------------------------------------------------------------------------------------
# 3. GEOMETRIC BOUNDARY RELATION
# ----------------------------------------------------------------------------------------------------
#
# The candle range is:
#
#     [low, high]
#
# The FVG is:
#
#     [lower_bound, upper_bound]
#
# Boundary-only contact:
#
#     high == lower_bound
#     OR
#     low == upper_bound
#
# Positive interior overlap:
#
#     touch_overlap > 0
#
# Full-zone crossing:
#
#     low <= lower_bound
#     AND
#     high >= upper_bound
#
# ----------------------------------------------------------------------------------------------------

_epsilon = 1e-12


_penetration_boundary_audit["touch_at_lower_boundary"] = (
    np.isclose(
        _penetration_boundary_audit["high"],
        _penetration_boundary_audit["lower_bound"],
        atol=_epsilon,
        rtol=0.0,
    )
)


_penetration_boundary_audit["touch_at_upper_boundary"] = (
    np.isclose(
        _penetration_boundary_audit["low"],
        _penetration_boundary_audit["upper_bound"],
        atol=_epsilon,
        rtol=0.0,
    )
)


# ----------------------------------------------------------------------------------------------------
# 4. POSITIVE INTERIOR PENETRATION
# ----------------------------------------------------------------------------------------------------

_penetration_boundary_audit["interior_penetration"] = (
    _penetration_boundary_audit["penetration"] > 0
)


# ----------------------------------------------------------------------------------------------------
# 5. BOUNDARY-ONLY TOUCH
# ----------------------------------------------------------------------------------------------------
#
# A candle is boundary-only when it reaches a FVG boundary but has
# zero-length geometric overlap with the zone.
#
# This is intentionally separated from interior penetration.
# ----------------------------------------------------------------------------------------------------

_penetration_boundary_audit["boundary_only"] = (
    (_penetration_boundary_audit["penetration"] == 0)
    &
    (
        _penetration_boundary_audit["touch_at_lower_boundary"]
        |
        _penetration_boundary_audit["touch_at_upper_boundary"]
    )
)


# ----------------------------------------------------------------------------------------------------
# 6. FULL-ZONE CROSSING
# ----------------------------------------------------------------------------------------------------

_penetration_boundary_audit["full_zone_crossed"] = (
    (_penetration_boundary_audit["low"] <= _penetration_boundary_audit["lower_bound"])
    &
    (_penetration_boundary_audit["high"] >= _penetration_boundary_audit["upper_bound"])
)


# ----------------------------------------------------------------------------------------------------
# 7. ENTRY BOUNDARY
# ----------------------------------------------------------------------------------------------------
#
# For candles with positive penetration, determine which side of the
# FVG is reached from the candle geometry.
#
# This is NOT bullish/bearish interpretation.
#
# ----------------------------------------------------------------------------------------------------

_penetration_boundary_audit["reaches_lower_boundary"] = (
    _penetration_boundary_audit["low"]
    <= _penetration_boundary_audit["lower_bound"]
)


_penetration_boundary_audit["reaches_upper_boundary"] = (
    _penetration_boundary_audit["high"]
    >= _penetration_boundary_audit["upper_bound"]
)


# ----------------------------------------------------------------------------------------------------
# 8. GEOMETRIC PENETRATION CLASS
# ----------------------------------------------------------------------------------------------------
#
# Priority:
#
# 1. BOUNDARY_ONLY
# 2. FULL_ZONE_CROSSING
# 3. LOWER_BOUNDARY_REACHED
# 4. UPPER_BOUNDARY_REACHED
# 5. INTERIOR_PENETRATION
#
# Note:
# A full-zone crossing reaches both boundaries and therefore gets its
# own explicit class.
# ----------------------------------------------------------------------------------------------------

_penetration_boundary_audit["penetration_class"] = np.select(
    [
        _penetration_boundary_audit["boundary_only"],

        _penetration_boundary_audit["full_zone_crossed"],

        (
            _penetration_boundary_audit["reaches_lower_boundary"]
            &
            ~_penetration_boundary_audit["reaches_upper_boundary"]
        ),

        (
            _penetration_boundary_audit["reaches_upper_boundary"]
            &
            ~_penetration_boundary_audit["reaches_lower_boundary"]
        ),

        _penetration_boundary_audit["interior_penetration"],
    ],
    [
        "BOUNDARY_ONLY",
        "FULL_ZONE_CROSSING",
        "LOWER_BOUNDARY_REACHED",
        "UPPER_BOUNDARY_REACHED",
        "INTERIOR_PENETRATION",
    ],
    default="UNCLASSIFIED",
)


# ----------------------------------------------------------------------------------------------------
# 9. GEOMETRIC CONTRACT VALIDATION
# ----------------------------------------------------------------------------------------------------

# Every row must be classified.
assert (
    _penetration_boundary_audit["penetration_class"]
    != "UNCLASSIFIED"
).all(), (
    "Some penetration observations could not be classified"
)


# Boundary-only rows must have zero penetration.
assert (
    _penetration_boundary_audit.loc[
        _penetration_boundary_audit["boundary_only"],
        "penetration"
    ] == 0
).all(), (
    "Boundary-only observations must have zero penetration"
)


# Interior penetration must have positive penetration.
assert (
    _penetration_boundary_audit.loc[
        _penetration_boundary_audit["interior_penetration"],
        "penetration"
    ] > 0
).all(), (
    "Interior penetration observations must have positive penetration"
)


# Full-zone crossing must have penetration equal to full zone width.
_full_crossing_mask = (
    _penetration_boundary_audit["full_zone_crossed"]
)

assert np.isclose(
    _penetration_boundary_audit.loc[
        _full_crossing_mask,
        "penetration"
    ],
    _penetration_boundary_audit.loc[
        _full_crossing_mask,
        "zone_width"
    ],
    atol=_epsilon,
    rtol=0.0,
).all(), (
    "Full-zone crossing penetration must equal zone width"
)


# Penetration ratio must remain in [0, 1].
assert (
    _penetration_boundary_audit["penetration_ratio"] >= 0
).all()

assert (
    _penetration_boundary_audit["penetration_ratio"] <= 1
).all()


# ----------------------------------------------------------------------------------------------------
# 10. CLASS DISTRIBUTION
# ----------------------------------------------------------------------------------------------------

print()
print("PENETRATION CLASS")
print("-" * 100)

print(
    _penetration_boundary_audit["penetration_class"]
    .value_counts(dropna=False)
    .to_string()
)


# ----------------------------------------------------------------------------------------------------
# 11. BOUNDARY FLAGS
# ----------------------------------------------------------------------------------------------------

print()
print("BOUNDARY FLAGS")
print("-" * 100)

print(
    pd.DataFrame({
        "touch_at_lower_boundary":
            _penetration_boundary_audit["touch_at_lower_boundary"].value_counts(),
        "touch_at_upper_boundary":
            _penetration_boundary_audit["touch_at_upper_boundary"].value_counts(),
        "reaches_lower_boundary":
            _penetration_boundary_audit["reaches_lower_boundary"].value_counts(),
        "reaches_upper_boundary":
            _penetration_boundary_audit["reaches_upper_boundary"].value_counts(),
    })
    .fillna(0)
    .astype(int)
    .to_string()
)


# ----------------------------------------------------------------------------------------------------
# 12. PENETRATION DEPTH DISTRIBUTION BY CLASS
# ----------------------------------------------------------------------------------------------------

print()
print("PENETRATION RATIO BY CLASS")
print("-" * 100)

print(
    _penetration_boundary_audit
    .groupby("penetration_class", observed=False)["penetration_ratio"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        min="min",
        max="max",
    )
    .sort_index()
    .to_string()
)


# ----------------------------------------------------------------------------------------------------
# 13. CLOSE POSITION BY PENETRATION CLASS
# ----------------------------------------------------------------------------------------------------

print()
print("CLOSE POSITION × PENETRATION CLASS")
print("-" * 100)

print(
    pd.crosstab(
        _penetration_boundary_audit["penetration_class"],
        _penetration_boundary_audit["close_position"],
        dropna=False,
    )
    .to_string()
)


# ----------------------------------------------------------------------------------------------------
# 14. FULL-ZONE CROSSING AUDIT
# ----------------------------------------------------------------------------------------------------

print()
print("FULL-ZONE CROSSING AUDIT")
print("-" * 100)

print(
    _penetration_boundary_audit["full_zone_crossed"]
    .value_counts(dropna=False)
    .to_string()
)


print()
print(
    "Full-zone crossing rows with penetration_ratio != 1 : "
    f"{int((
        _penetration_boundary_audit.loc[
            _full_crossing_mask,
            'penetration_ratio'
        ] != 1
    ).sum())}"
)


# ----------------------------------------------------------------------------------------------------
# 15. BOUNDARY-ONLY SAMPLE
# ----------------------------------------------------------------------------------------------------

print()
print("BOUNDARY-ONLY SAMPLE")
print("-" * 100)

print(
    _penetration_boundary_audit.loc[
        _penetration_boundary_audit["boundary_only"],
        [
            "fvg_id",
            "datetime",
            "high",
            "low",
            "close",
            "lower_bound",
            "upper_bound",
            "penetration",
            "penetration_ratio",
            "penetration_class",
        ],
    ]
    .head(20)
    .to_string(index=False)
)


# ----------------------------------------------------------------------------------------------------
# 16. INTERIOR PENETRATION SAMPLE
# ----------------------------------------------------------------------------------------------------

print()
print("INTERIOR PENETRATION SAMPLE")
print("-" * 100)

print(
    _penetration_boundary_audit.loc[
        _penetration_boundary_audit["interior_penetration"],
        [
            "fvg_id",
            "datetime",
            "high",
            "low",
            "close",
            "lower_bound",
            "upper_bound",
            "penetration",
            "penetration_ratio",
            "penetration_class",
        ],
    ]
    .head(20)
    .to_string(index=False)
)


# ----------------------------------------------------------------------------------------------------
# 17. FULL-ZONE CROSSING SAMPLE
# ----------------------------------------------------------------------------------------------------

print()
print("FULL-ZONE CROSSING SAMPLE")
print("-" * 100)

print(
    _penetration_boundary_audit.loc[
        _penetration_boundary_audit["full_zone_crossed"],
        [
            "fvg_id",
            "datetime",
            "high",
            "low",
            "close",
            "lower_bound",
            "upper_bound",
            "penetration",
            "penetration_ratio",
            "penetration_class",
        ],
    ]
    .head(20)
    .to_string(index=False)
)


# ----------------------------------------------------------------------------------------------------
# 18. FINAL CONTRACT SUMMARY
# ----------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("STEP 6B CONTRACT AUDIT COMPLETE")
print("=" * 100)

print(
    f"Total observations             : {len(_penetration_boundary_audit):,}"
)

print(
    f"Boundary-only                 : "
    f"{int((_penetration_boundary_audit['penetration_class'] == 'BOUNDARY_ONLY').sum()):,}"
)

print(
    f"Interior penetration           : "
    f"{int((_penetration_boundary_audit['penetration_class'] == 'INTERIOR_PENETRATION').sum()):,}"
)

print(
    f"Lower boundary reached        : "
    f"{int((_penetration_boundary_audit['penetration_class'] == 'LOWER_BOUNDARY_REACHED').sum()):,}"
)

print(
    f"Upper boundary reached        : "
    f"{int((_penetration_boundary_audit['penetration_class'] == 'UPPER_BOUNDARY_REACHED').sum()):,}"
)

print(
    f"Full-zone crossing             : "
    f"{int((_penetration_boundary_audit['penetration_class'] == 'FULL_ZONE_CROSSING').sum()):,}"
)

print(
    "UNCLASSIFIED                   : "
    f"{int((_penetration_boundary_audit['penetration_class'] == 'UNCLASSIFIED').sum()):,}"
)

In [ ]:
# ==================================================================================================
# STEP 7.1 — INTERACTION EPISODE CONSTRUCTOR V1
# CELL 7.1-A — STEP 6B REFERENCE DISCOVERY
# ==================================================================================================

print("=" * 100)
print("STEP 7.1 — INTERACTION EPISODE CONSTRUCTOR V1")
print("STEP 6B REFERENCE DISCOVERY")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Required Step 6B schema
# --------------------------------------------------------------------------------------------------

_required_step6b_columns = {
    "fvg_id",
    "datetime",
    "lower_bound",
    "upper_bound",
    "penetration",
    "penetration_ratio",
    "penetration_class",
}

_step6b_candidates = []


# --------------------------------------------------------------------------------------------------
# 2. Snapshot globals BEFORE iteration
# --------------------------------------------------------------------------------------------------

_global_items = list(globals().items())


# --------------------------------------------------------------------------------------------------
# 3. Find candidate DataFrames
# --------------------------------------------------------------------------------------------------

for _name, _obj in _global_items:

    if hasattr(_obj, "columns"):

        try:
            _cols = set(_obj.columns)

            if _required_step6b_columns.issubset(_cols):

                _step6b_candidates.append(
                    {
                        "name": _name,
                        "rows": len(_obj),
                        "columns": len(_obj.columns),
                        "missing": [],
                    }
                )

        except Exception:
            pass


# --------------------------------------------------------------------------------------------------
# 4. Display candidates
# --------------------------------------------------------------------------------------------------

print("\nSTEP 6B CANDIDATE DATAFRAMES")
print("-" * 100)

if not _step6b_candidates:

    print("NO MATCHING DATAFRAME FOUND")

else:

    for _candidate in _step6b_candidates:

        print(
            f"{_candidate['name']:<55}"
            f" rows={_candidate['rows']:<8}"
            f" columns={_candidate['columns']}"
        )


# --------------------------------------------------------------------------------------------------
# 5. Display exact schema
# --------------------------------------------------------------------------------------------------

print("\nSCHEMA")
print("-" * 100)

for _candidate in _step6b_candidates:

    _df = globals()[_candidate["name"]]

    print(f"\n[{_candidate['name']}]")

    print(list(_df.columns))


# --------------------------------------------------------------------------------------------------
# 6. Basic reference statistics
# --------------------------------------------------------------------------------------------------

print("\nREFERENCE STATISTICS")
print("-" * 100)

for _candidate in _step6b_candidates:

    _df = globals()[_candidate["name"]]

    print(f"\n[{_candidate['name']}]")

    print(f"Rows              : {len(_df):,}")
    print(f"Unique FVGs       : {_df['fvg_id'].nunique():,}")
    print(f"Unique timestamps : {_df['datetime'].nunique():,}")

    print("\nPenetration class:")
    print(
        _df["penetration_class"]
        .value_counts(dropna=False)
        .to_string()
    )


print("\n" + "=" * 100)
print("DISCOVERY COMPLETE")
print("=" * 100)

In [ ]:
# ==================================================================================================
# STEP 7.1 — INTERACTION EPISODE CONSTRUCTOR V1
# CELL 7.1-B — CONSTRUCT INTERACTION EPISODES
# ==================================================================================================

import pandas as pd
import numpy as np


print("=" * 100)
print("STEP 7.1 — INTERACTION EPISODE CONSTRUCTOR V1")
print("CONSTRUCT INTERACTION EPISODES")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Reference input
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1 = _penetration_boundary_audit.copy()


# --------------------------------------------------------------------------------------------------
# 2. Input validation
# --------------------------------------------------------------------------------------------------

_required_columns_v1 = {
    "fvg_id",
    "datetime",
    "touch_overlap",
    "penetration",
    "penetration_class",
}

_missing_columns_v1 = (
    _required_columns_v1
    - set(_interaction_reference_v1.columns)
)

if _missing_columns_v1:
    raise ValueError(
        f"Missing required Step 6B columns: "
        f"{sorted(_missing_columns_v1)}"
    )


# --------------------------------------------------------------------------------------------------
# 3. Normalize datetime
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1["datetime"] = pd.to_datetime(
    _interaction_reference_v1["datetime"],
    utc=True,
    errors="raise",
)


# --------------------------------------------------------------------------------------------------
# 4. Deterministic chronological ordering
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1 = (
    _interaction_reference_v1
    .sort_values(
        ["fvg_id", "datetime"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 5. Interaction eligibility
#
# A valid interaction is:
#   - boundary touch / overlap
#   - OR positive penetration
#
# This explicitly preserves BOUNDARY_ONLY observations.
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1["_interaction_observation"] = (
    _interaction_reference_v1["touch_overlap"].fillna(False).astype(bool)
    |
    _interaction_reference_v1["penetration"].fillna(0).gt(0)
)


# --------------------------------------------------------------------------------------------------
# 6. Construct episode sequence per FVG
#
# Every continuous run of interaction observations belongs to one episode.
#
# A non-interaction observation breaks continuity.
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1["interaction_episode_sequence"] = pd.NA

for _fvg_id, _group_index in (
    _interaction_reference_v1
    .groupby("fvg_id", sort=False)
    .groups
    .items()
):

    _idx = list(_group_index)

    _active = (
        _interaction_reference_v1
        .loc[_idx, "_interaction_observation"]
        .to_numpy(dtype=bool)
    )

    _sequence = []
    _episode_counter = 0
    _previous_active = False

    for _is_active in _active:

        if _is_active and not _previous_active:
            _episode_counter += 1

        if _is_active:
            _sequence.append(_episode_counter)
        else:
            _sequence.append(pd.NA)

        _previous_active = _is_active

    _interaction_reference_v1.loc[
        _idx,
        "interaction_episode_sequence"
    ] = _sequence


# --------------------------------------------------------------------------------------------------
# 7. Deterministic Episode ID
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1["interaction_episode_id"] = pd.NA

_interaction_mask_v1 = (
    _interaction_reference_v1["_interaction_observation"]
)

_interaction_reference_v1.loc[
    _interaction_mask_v1,
    "interaction_episode_id"
] = (
    _interaction_reference_v1.loc[
        _interaction_mask_v1,
        "fvg_id"
    ].astype(str)
    + "_EP_"
    + _interaction_reference_v1.loc[
        _interaction_mask_v1,
        "interaction_episode_sequence"
    ].astype("int64")
    .astype(str)
    .str.zfill(3)
)


# --------------------------------------------------------------------------------------------------
# 8. Episode start / end timestamps
# --------------------------------------------------------------------------------------------------

_interaction_rows_v1 = (
    _interaction_reference_v1[
        _interaction_mask_v1
    ]
    .copy()
)

if not _interaction_rows_v1.empty:

    _episode_bounds_v1 = (
        _interaction_rows_v1
        .groupby(
            "interaction_episode_id",
            sort=False,
            as_index=False,
        )
        .agg(
            interaction_episode_start=(
                "datetime",
                "min",
            ),
            interaction_episode_end=(
                "datetime",
                "max",
            ),
            interaction_observation_count=(
                "datetime",
                "size",
            ),
        )
    )

else:

    _episode_bounds_v1 = pd.DataFrame(
        columns=[
            "interaction_episode_id",
            "interaction_episode_start",
            "interaction_episode_end",
            "interaction_observation_count",
        ]
    )


# --------------------------------------------------------------------------------------------------
# 9. Attach episode boundaries back to observation timeline
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1 = _interaction_reference_v1.merge(
    _episode_bounds_v1,
    on="interaction_episode_id",
    how="left",
    sort=False,
    validate="many_to_one",
)


# --------------------------------------------------------------------------------------------------
# 10. Clean temporary / internal column
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1 = (
    _interaction_reference_v1
    .drop(columns=["_interaction_observation"])
)


# --------------------------------------------------------------------------------------------------
# 11. Final deterministic ordering
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1 = (
    _interaction_reference_v1
    .sort_values(
        ["fvg_id", "datetime"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 12. Create Episode Registry V1
# --------------------------------------------------------------------------------------------------

interaction_episode_registry_v1 = (
    _interaction_reference_v1[
        _interaction_reference_v1["interaction_episode_id"].notna()
    ]
    .groupby(
        "interaction_episode_id",
        sort=False,
        as_index=False,
    )
    .agg(
        fvg_id=("fvg_id", "first"),
        episode_sequence=("interaction_episode_sequence", "first"),
        start_time=("interaction_episode_start", "first"),
        end_time=("interaction_episode_end", "first"),
        observation_count=("interaction_observation_count", "first"),
        max_penetration=("penetration", "max"),
        max_penetration_ratio=("penetration_ratio", "max"),
    )
)


# --------------------------------------------------------------------------------------------------
# 13. Display construction summary
# --------------------------------------------------------------------------------------------------

print("\nCONSTRUCTION SUMMARY")
print("-" * 100)

print(
    f"Reference observations       : "
    f"{len(_penetration_boundary_audit):,}"
)

print(
    f"Output observations           : "
    f"{len(_interaction_reference_v1):,}"
)

print(
    f"Unique FVGs                   : "
    f"{_interaction_reference_v1['fvg_id'].nunique():,}"
)

print(
    f"Interaction observations      : "
    f"{_interaction_reference_v1['interaction_episode_id'].notna().sum():,}"
)

print(
    f"Unique interaction episodes   : "
    f"{interaction_episode_registry_v1['interaction_episode_id'].nunique():,}"
)


print("\nEPISODE OBSERVATION COUNT DISTRIBUTION")
print("-" * 100)

if not interaction_episode_registry_v1.empty:

    print(
        interaction_episode_registry_v1[
            "observation_count"
        ]
        .describe()
        .to_string()
    )


print("\nEPISODE REGISTRY SAMPLE")
print("-" * 100)

print(
    interaction_episode_registry_v1.head(20).to_string(
        index=False
    )
)


print("\n" + "=" * 100)
print("STEP 7.1 CONSTRUCTION COMPLETE")
print("=" * 100)

In [ ]:
# ==================================================================================================
# STEP 7.1-C — INTERACTION TEMPORAL GAP DIAGNOSTIC
# ==================================================================================================

print("=" * 100)
print("STEP 7.1-C — INTERACTION TEMPORAL GAP DIAGNOSTIC")
print("=" * 100)

_gap_df = (
    _penetration_boundary_audit[
        [
            "fvg_id",
            "datetime",
            "penetration_class",
        ]
    ]
    .copy()
    .sort_values(
        ["fvg_id", "datetime"],
        kind="mergesort",
    )
)

_gap_df["datetime"] = pd.to_datetime(
    _gap_df["datetime"],
    utc=True,
)

_gap_df["gap"] = (
    _gap_df
    .groupby("fvg_id")["datetime"]
    .diff()
)

_gap_df["gap_minutes"] = (
    _gap_df["gap"]
    .dt.total_seconds()
    .div(60)
)

print("\nALL OBSERVATION GAPS")
print("-" * 100)

print(
    _gap_df["gap_minutes"]
    .dropna()
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .to_string()
)


print("\nGAP DISTRIBUTION — TOP VALUES")
print("-" * 100)

print(
    _gap_df["gap_minutes"]
    .dropna()
    .round(2)
    .value_counts()
    .head(20)
    .to_string()
)


print("\nGAPS > 5 MINUTES")
print("-" * 100)

_gaps_gt_5 = _gap_df[
    _gap_df["gap_minutes"] > 5
]

print(
    f"Count : {len(_gaps_gt_5):,}"
)

print(
    _gaps_gt_5[
        [
            "fvg_id",
            "datetime",
            "gap_minutes",
            "penetration_class",
        ]
    ]
    .head(30)
    .to_string(index=False)
)


print("\n" + "=" * 100)
print("TEMPORAL GAP DIAGNOSTIC COMPLETE")
print("=" * 100)

In [ ]:
# ==================================================================================================
# STEP 7.1-D — INTERACTION EPISODE CONSTRUCTOR V1.1
# TEMPORAL CONTINUITY CONTRACT
# ==================================================================================================

import pandas as pd
import numpy as np


print("=" * 100)
print("STEP 7.1-D — INTERACTION EPISODE CONSTRUCTOR V1.1")
print("TEMPORAL CONTINUITY CONTRACT")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Reference input
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1_1 = (
    _penetration_boundary_audit
    .copy()
)


# --------------------------------------------------------------------------------------------------
# 2. Required columns
# --------------------------------------------------------------------------------------------------

_required_columns_v1_1 = {
    "fvg_id",
    "datetime",
    "penetration",
    "penetration_ratio",
    "penetration_class",
}

_missing_columns_v1_1 = (
    _required_columns_v1_1
    - set(_interaction_reference_v1_1.columns)
)

if _missing_columns_v1_1:
    raise ValueError(
        f"Missing required Step 6B columns: "
        f"{sorted(_missing_columns_v1_1)}"
    )


# --------------------------------------------------------------------------------------------------
# 3. Normalize datetime
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1_1["datetime"] = pd.to_datetime(
    _interaction_reference_v1_1["datetime"],
    utc=True,
    errors="raise",
)


# --------------------------------------------------------------------------------------------------
# 4. Interaction eligibility
#
# Step 6B penetration_class is the authoritative classification.
#
# Every classified penetration/touch observation is a valid interaction.
# This explicitly preserves BOUNDARY_ONLY.
# --------------------------------------------------------------------------------------------------

_allowed_interaction_classes_v1_1 = {
    "BOUNDARY_ONLY",
    "INTERIOR_PENETRATION",
    "LOWER_BOUNDARY_REACHED",
    "UPPER_BOUNDARY_REACHED",
    "FULL_ZONE_CROSSING",
}


_interaction_reference_v1_1["_interaction_observation"] = (
    _interaction_reference_v1_1["penetration_class"]
    .isin(_allowed_interaction_classes_v1_1)
)


# --------------------------------------------------------------------------------------------------
# 5. Deterministic chronological ordering
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1_1 = (
    _interaction_reference_v1_1
    .sort_values(
        ["fvg_id", "datetime"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 6. Calculate temporal gap per FVG
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1_1["interaction_gap"] = (
    _interaction_reference_v1_1
    .groupby("fvg_id")["datetime"]
    .diff()
)


_interaction_reference_v1_1["interaction_gap_minutes"] = (
    _interaction_reference_v1_1["interaction_gap"]
    .dt.total_seconds()
    .div(60)
)


# --------------------------------------------------------------------------------------------------
# 7. Explicit M5 continuity contract
#
# A new episode starts when:
#
#   - the row is the first observation of an FVG
#   - OR the temporal gap is NOT exactly 5 minutes
#
# Since Step 6B contains only interaction observations, every valid row
# participates in episode construction.
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1_1["_episode_break"] = (
    _interaction_reference_v1_1["interaction_gap_minutes"]
    .isna()
    |
    _interaction_reference_v1_1["interaction_gap_minutes"]
    .ne(5.0)
)


# --------------------------------------------------------------------------------------------------
# 8. Episode sequence per FVG
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1_1["interaction_episode_sequence"] = (
    _interaction_reference_v1_1
    .groupby("fvg_id")["_episode_break"]
    .cumsum()
    .astype("int64")
)


# --------------------------------------------------------------------------------------------------
# 9. Deterministic Episode ID
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1_1["interaction_episode_id"] = (
    _interaction_reference_v1_1["fvg_id"].astype(str)
    + "_EP_"
    + _interaction_reference_v1_1[
        "interaction_episode_sequence"
    ]
    .astype(str)
    .str.zfill(3)
)


# --------------------------------------------------------------------------------------------------
# 10. Episode boundaries
# --------------------------------------------------------------------------------------------------

_episode_bounds_v1_1 = (
    _interaction_reference_v1_1
    .groupby(
        "interaction_episode_id",
        sort=False,
        as_index=False,
    )
    .agg(
        fvg_id=(
            "fvg_id",
            "first",
        ),
        episode_sequence=(
            "interaction_episode_sequence",
            "first",
        ),
        interaction_episode_start=(
            "datetime",
            "min",
        ),
        interaction_episode_end=(
            "datetime",
            "max",
        ),
        interaction_observation_count=(
            "datetime",
            "size",
        ),
        max_penetration=(
            "penetration",
            "max",
        ),
        max_penetration_ratio=(
            "penetration_ratio",
            "max",
        ),
    )
)


# --------------------------------------------------------------------------------------------------
# 11. Attach episode metadata to observation timeline
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1_1 = (
    _interaction_reference_v1_1
    .merge(
        _episode_bounds_v1_1[
            [
                "interaction_episode_id",
                "interaction_episode_start",
                "interaction_episode_end",
                "interaction_observation_count",
            ]
        ],
        on="interaction_episode_id",
        how="left",
        sort=False,
        validate="many_to_one",
    )
)


# --------------------------------------------------------------------------------------------------
# 12. Create Episode Registry V1.1
# --------------------------------------------------------------------------------------------------

interaction_episode_registry_v1_1 = (
    _episode_bounds_v1_1
    .copy()
)


# --------------------------------------------------------------------------------------------------
# 13. Remove temporary construction columns
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1_1 = (
    _interaction_reference_v1_1
    .drop(
        columns=[
            "_interaction_observation",
            "_episode_break",
            "interaction_gap",
        ]
    )
)


# --------------------------------------------------------------------------------------------------
# 14. Final deterministic ordering
# --------------------------------------------------------------------------------------------------

_interaction_reference_v1_1 = (
    _interaction_reference_v1_1
    .sort_values(
        ["fvg_id", "datetime"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


interaction_episode_registry_v1_1 = (
    interaction_episode_registry_v1_1
    .sort_values(
        ["fvg_id", "episode_sequence"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 15. Construction summary
# --------------------------------------------------------------------------------------------------

print("\nCONSTRUCTION SUMMARY")
print("-" * 100)

print(
    f"Reference observations       : "
    f"{len(_penetration_boundary_audit):,}"
)

print(
    f"Output observations           : "
    f"{len(_interaction_reference_v1_1):,}"
)

print(
    f"Interaction observations      : "
    f"{len(_interaction_reference_v1_1):,}"
)

print(
    f"Unique FVGs                   : "
    f"{_interaction_reference_v1_1['fvg_id'].nunique():,}"
)

print(
    f"Unique interaction episodes   : "
    f"{interaction_episode_registry_v1_1['interaction_episode_id'].nunique():,}"
)


# --------------------------------------------------------------------------------------------------
# 16. Penetration class preservation
# --------------------------------------------------------------------------------------------------

print("\nPENETRATION CLASS PRESERVATION")
print("-" * 100)

print(
    _interaction_reference_v1_1[
        "penetration_class"
    ]
    .value_counts()
    .to_string()
)


# --------------------------------------------------------------------------------------------------
# 17. Episode observation count distribution
# --------------------------------------------------------------------------------------------------

print("\nEPISODE OBSERVATION COUNT DISTRIBUTION")
print("-" * 100)

print(
    interaction_episode_registry_v1_1[
        "interaction_observation_count"
    ]
    .describe()
    .to_string()
)


# --------------------------------------------------------------------------------------------------
# 18. Temporal gap distribution inside episodes
# --------------------------------------------------------------------------------------------------

print("\nINTRA-EPISODE GAP AUDIT")
print("-" * 100)

_intra_episode_gaps_v1_1 = (
    _interaction_reference_v1_1
    .groupby(
        "interaction_episode_id"
    )["datetime"]
    .diff()
    .dt.total_seconds()
    .div(60)
    .dropna()
)


if _intra_episode_gaps_v1_1.empty:

    print("No intra-episode gaps.")

else:

    print(
        _intra_episode_gaps_v1_1
        .describe()
        .to_string()
    )

    print(
        f"\nNon-5-minute intra-episode gaps : "
        f"{(_intra_episode_gaps_v1_1 != 5.0).sum():,}"
    )


# --------------------------------------------------------------------------------------------------
# 19. Registry sample
# --------------------------------------------------------------------------------------------------

print("\nEPISODE REGISTRY SAMPLE")
print("-" * 100)

print(
    interaction_episode_registry_v1_1
    .head(20)
    .to_string(index=False)
)


print("\n" + "=" * 100)
print("STEP 7.1-D CONSTRUCTION COMPLETE")
print("=" * 100)

In [ ]:
# ==================================================================================================
# STEP 7.2 — INTERACTION EPISODE IDENTITY AUDIT V1
# ==================================================================================================

print("=" * 100)
print("STEP 7.2 — INTERACTION EPISODE IDENTITY AUDIT V1")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Working references
# --------------------------------------------------------------------------------------------------

_obs_v1 = _interaction_reference_v1_1.copy()
_reg_v1 = interaction_episode_registry_v1_1.copy()


# --------------------------------------------------------------------------------------------------
# 2. Basic counts
# --------------------------------------------------------------------------------------------------

print("\nBASIC COUNTS")
print("-" * 100)

print(
    f"Observation rows             : "
    f"{len(_obs_v1):,}"
)

print(
    f"Registry rows                : "
    f"{len(_reg_v1):,}"
)

print(
    f"Unique observation Episode IDs: "
    f"{_obs_v1['interaction_episode_id'].nunique():,}"
)

print(
    f"Unique registry Episode IDs   : "
    f"{_reg_v1['interaction_episode_id'].nunique():,}"
)

print(
    f"Unique FVGs                  : "
    f"{_obs_v1['fvg_id'].nunique():,}"
)


# --------------------------------------------------------------------------------------------------
# 3. Episode ID uniqueness
# --------------------------------------------------------------------------------------------------

_episode_id_duplicate_count = (
    _reg_v1["interaction_episode_id"]
    .duplicated()
    .sum()
)

print("\nEPISODE ID UNIQUENESS")
print("-" * 100)

print(
    f"Duplicate Episode IDs : "
    f"{_episode_id_duplicate_count:,}"
)


# --------------------------------------------------------------------------------------------------
# 4. Episode → single FVG ownership
# --------------------------------------------------------------------------------------------------

_episode_fvg_cardinality = (
    _obs_v1
    .groupby("interaction_episode_id")["fvg_id"]
    .nunique()
)

_cross_fvg_episode_count = (
    (_episode_fvg_cardinality > 1)
    .sum()
)

print("\nEPISODE → FVG OWNERSHIP")
print("-" * 100)

print(
    f"Episodes with >1 FVG : "
    f"{_cross_fvg_episode_count:,}"
)


# --------------------------------------------------------------------------------------------------
# 5. FVG → Episode sequence uniqueness
# --------------------------------------------------------------------------------------------------

_sequence_cardinality = (
    _reg_v1
    .groupby(
        ["fvg_id", "episode_sequence"]
    )["interaction_episode_id"]
    .nunique()
)

_duplicate_sequence_pairs = (
    (_sequence_cardinality > 1)
    .sum()
)

print("\nFVG → EPISODE SEQUENCE UNIQUENESS")
print("-" * 100)

print(
    f"Duplicate FVG/sequence pairs : "
    f"{_duplicate_sequence_pairs:,}"
)


# --------------------------------------------------------------------------------------------------
# 6. Episode sequence continuity
#
# Every FVG must have:
#
# 1, 2, 3, ..., N
# --------------------------------------------------------------------------------------------------

_sequence_breaks = []

for _fvg_id, _group in (
    _reg_v1
    .groupby("fvg_id", sort=False)
):

    _seq = (
        _group["episode_sequence"]
        .astype(int)
        .sort_values()
        .to_numpy()
    )

    _expected = np.arange(
        1,
        len(_seq) + 1,
        dtype=int,
    )

    if not np.array_equal(
        _seq,
        _expected,
    ):
        _sequence_breaks.append(
            _fvg_id
        )


print("\nEPISODE SEQUENCE CONTINUITY")
print("-" * 100)

print(
    f"FVGs with sequence breaks : "
    f"{len(_sequence_breaks):,}"
)


# --------------------------------------------------------------------------------------------------
# 7. Observation → single Episode ownership
# --------------------------------------------------------------------------------------------------

_episode_observation_counts = (
    _obs_v1
    .groupby("interaction_episode_id")
    .size()
)

_registry_observation_counts = (
    _reg_v1
    .set_index("interaction_episode_id")
    ["interaction_observation_count"]
)

_count_alignment = (
    _episode_observation_counts
    .sort_index()
    .equals(
        _registry_observation_counts
        .sort_index()
    )
)

print("\nOBSERVATION → EPISODE OWNERSHIP")
print("-" * 100)

print(
    f"Registry count alignment : "
    f"{_count_alignment}"
)


# --------------------------------------------------------------------------------------------------
# 8. Episode start boundary audit
# --------------------------------------------------------------------------------------------------

_start_alignment = (
    _obs_v1
    .groupby("interaction_episode_id")["datetime"]
    .min()
    .sort_index()
    .equals(
        _reg_v1
        .set_index("interaction_episode_id")
        ["interaction_episode_start"]
        .sort_index()
    )
)

print("\nEPISODE START BOUNDARY")
print("-" * 100)

print(
    f"Start timestamp alignment : "
    f"{_start_alignment}"
)


# --------------------------------------------------------------------------------------------------
# 9. Episode end boundary audit
# --------------------------------------------------------------------------------------------------

_end_alignment = (
    _obs_v1
    .groupby("interaction_episode_id")["datetime"]
    .max()
    .sort_index()
    .equals(
        _reg_v1
        .set_index("interaction_episode_id")
        ["interaction_episode_end"]
        .sort_index()
    )
)

print("\nEPISODE END BOUNDARY")
print("-" * 100)

print(
    f"End timestamp alignment : "
    f"{_end_alignment}"
)


# --------------------------------------------------------------------------------------------------
# 10. Episode → single start/end interval
# --------------------------------------------------------------------------------------------------

_invalid_intervals = (
    _reg_v1[
        _reg_v1["interaction_episode_end"]
        <
        _reg_v1["interaction_episode_start"]
    ]
)

print("\nEPISODE TIME INTERVAL VALIDITY")
print("-" * 100)

print(
    f"Invalid intervals : "
    f"{len(_invalid_intervals):,}"
)


# --------------------------------------------------------------------------------------------------
# 11. Max penetration alignment
# --------------------------------------------------------------------------------------------------

_observation_max_penetration = (
    _obs_v1
    .groupby("interaction_episode_id")["penetration"]
    .max()
    .sort_index()
)

_registry_max_penetration = (
    _reg_v1
    .set_index("interaction_episode_id")
    ["max_penetration"]
    .sort_index()
)

_penetration_alignment = np.allclose(
    _observation_max_penetration.to_numpy(),
    _registry_max_penetration.to_numpy(),
    equal_nan=True,
)


print("\nMAX PENETRATION ALIGNMENT")
print("-" * 100)

print(
    f"Max penetration alignment : "
    f"{_penetration_alignment}"
)


# --------------------------------------------------------------------------------------------------
# 12. Max penetration ratio alignment
# --------------------------------------------------------------------------------------------------

_observation_max_ratio = (
    _obs_v1
    .groupby("interaction_episode_id")["penetration_ratio"]
    .max()
    .sort_index()
)

_registry_max_ratio = (
    _reg_v1
    .set_index("interaction_episode_id")
    ["max_penetration_ratio"]
    .sort_index()
)

_ratio_alignment = np.allclose(
    _observation_max_ratio.to_numpy(),
    _registry_max_ratio.to_numpy(),
    equal_nan=True,
)


print("\nMAX PENETRATION RATIO ALIGNMENT")
print("-" * 100)

print(
    f"Max penetration ratio alignment : "
    f"{_ratio_alignment}"
)


# --------------------------------------------------------------------------------------------------
# 13. Boundary-only singleton audit
#
# Singleton episodes with zero penetration must be BOUNDARY_ONLY.
# --------------------------------------------------------------------------------------------------

_singleton_zero_penetration = _reg_v1[
    (
        _reg_v1["interaction_observation_count"] == 1
    )
    &
    (
        _reg_v1["max_penetration"] == 0
    )
]

_singleton_zero_ids = set(
    _singleton_zero_penetration[
        "interaction_episode_id"
    ]
)

_singleton_zero_rows = _obs_v1[
    _obs_v1["interaction_episode_id"]
    .isin(_singleton_zero_ids)
]

_invalid_boundary_singletons = (
    _singleton_zero_rows[
        _singleton_zero_rows["penetration_class"]
        != "BOUNDARY_ONLY"
    ]
)


print("\nBOUNDARY-ONLY SINGLETON AUDIT")
print("-" * 100)

print(
    f"Zero-penetration singleton episodes : "
    f"{len(_singleton_zero_penetration):,}"
)

print(
    f"Invalid singleton classifications   : "
    f"{len(_invalid_boundary_singletons):,}"
)


# --------------------------------------------------------------------------------------------------
# 14. Cross-FVG contamination
# --------------------------------------------------------------------------------------------------

_cross_fvg_rows = (
    _obs_v1
    .groupby("interaction_episode_id")["fvg_id"]
    .nunique()
)

print("\nCROSS-FVG CONTAMINATION")
print("-" * 100)

print(
    f"Contaminated Episodes : "
    f"{(_cross_fvg_rows > 1).sum():,}"
)


# --------------------------------------------------------------------------------------------------
# 15. Final Audit Gate
# --------------------------------------------------------------------------------------------------

_gate_results_v1 = {
    "ROW_COUNT_PRESERVATION": (
        len(_obs_v1)
        == len(_penetration_boundary_audit)
    ),

    "EPISODE_ID_UNIQUENESS": (
        _episode_id_duplicate_count == 0
    ),

    "EPISODE_SINGLE_FVG": (
        _cross_fvg_episode_count == 0
    ),

    "FVG_SEQUENCE_UNIQUENESS": (
        _duplicate_sequence_pairs == 0
    ),

    "FVG_SEQUENCE_CONTINUITY": (
        len(_sequence_breaks) == 0
    ),

    "OBSERVATION_COUNT_ALIGNMENT": (
        _count_alignment
    ),

    "START_BOUNDARY_ALIGNMENT": (
        _start_alignment
    ),

    "END_BOUNDARY_ALIGNMENT": (
        _end_alignment
    ),

    "TIME_INTERVAL_VALIDITY": (
        len(_invalid_intervals) == 0
    ),

    "MAX_PENETRATION_ALIGNMENT": (
        _penetration_alignment
    ),

    "MAX_PENETRATION_RATIO_ALIGNMENT": (
        _ratio_alignment
    ),

    "BOUNDARY_ONLY_SINGLETON_CONTRACT": (
        len(_invalid_boundary_singletons) == 0
    ),

    "CROSS_FVG_CONTAMINATION": (
        (_cross_fvg_rows > 1).sum() == 0
    ),
}


print("\n" + "=" * 100)
print("STEP 7.2 — EPISODE IDENTITY AUDIT GATE")
print("=" * 100)

for _gate_name, _gate_value in _gate_results_v1.items():

    print(
        f"{_gate_name:<45}: "
        f"{'PASSED' if _gate_value else 'FAILED'}"
    )


print("\n" + "=" * 100)

if all(_gate_results_v1.values()):

    print("STEP 7.2 EPISODE IDENTITY AUDIT : PASSED")

else:

    print("STEP 7.2 EPISODE IDENTITY AUDIT : FAILED")

print("=" * 100)

In [ ]:
# ==================================================================================================
# STEP 7.3 — INTERACTION EPISODE CONTINUITY AUDIT V1
# ==================================================================================================

print("=" * 100)
print("STEP 7.3 — INTERACTION EPISODE CONTINUITY AUDIT V1")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Working reference
# --------------------------------------------------------------------------------------------------

_continuity_obs_v1 = (
    _interaction_reference_v1_1
    .copy()
    .sort_values(
        ["fvg_id", "datetime"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 2. Recalculate chronological gaps from source observations
# --------------------------------------------------------------------------------------------------

_continuity_obs_v1["source_gap_minutes"] = (
    _continuity_obs_v1
    .groupby("fvg_id")["datetime"]
    .diff()
    .dt.total_seconds()
    .div(60)
)


# --------------------------------------------------------------------------------------------------
# 3. Previous / current Episode ID
# --------------------------------------------------------------------------------------------------

_continuity_obs_v1["previous_episode_id"] = (
    _continuity_obs_v1
    .groupby("fvg_id")[
        "interaction_episode_id"
    ]
    .shift(1)
)


# --------------------------------------------------------------------------------------------------
# 4. Episode transition classification
# --------------------------------------------------------------------------------------------------

_continuity_obs_v1["episode_transition"] = np.select(
    [
        _continuity_obs_v1["previous_episode_id"].isna(),
        (
            _continuity_obs_v1["interaction_episode_id"]
            ==
            _continuity_obs_v1["previous_episode_id"]
        ),
        (
            _continuity_obs_v1["interaction_episode_id"]
            !=
            _continuity_obs_v1["previous_episode_id"]
        ),
    ],
    [
        "FIRST_EPISODE_OBSERVATION",
        "WITHIN_EPISODE",
        "NEW_EPISODE",
    ],
    default="UNKNOWN",
)


# --------------------------------------------------------------------------------------------------
# 5. Within-episode continuity audit
# --------------------------------------------------------------------------------------------------

_within_episode = (
    _continuity_obs_v1[
        _continuity_obs_v1["episode_transition"]
        == "WITHIN_EPISODE"
    ]
)

_invalid_within_episode_gap = (
    _within_episode[
        _within_episode["source_gap_minutes"] != 5.0
    ]
)


print("\nWITHIN-EPISODE CONTINUITY")
print("-" * 100)

print(
    f"Within-episode transitions : "
    f"{len(_within_episode):,}"
)

print(
    f"Invalid gaps               : "
    f"{len(_invalid_within_episode_gap):,}"
)


if not _within_episode.empty:

    print("\nWithin-episode gap distribution:")
    print(
        _within_episode["source_gap_minutes"]
        .describe()
        .to_string()
    )


# --------------------------------------------------------------------------------------------------
# 6. Episode transition gap audit
#
# A new episode must NOT have a 5-minute source gap.
# Otherwise two observations that could have remained in one episode
# would have been incorrectly split.
# --------------------------------------------------------------------------------------------------

_new_episode_rows = (
    _continuity_obs_v1[
        _continuity_obs_v1["episode_transition"]
        == "NEW_EPISODE"
    ]
)

_invalid_new_episode_gap = (
    _new_episode_rows[
        _new_episode_rows["source_gap_minutes"] == 5.0
    ]
)


print("\nNEW EPISODE TRANSITION AUDIT")
print("-" * 100)

print(
    f"New Episode transitions     : "
    f"{len(_new_episode_rows):,}"
)

print(
    f"Invalid 5-minute splits     : "
    f"{len(_invalid_new_episode_gap):,}"
)


# --------------------------------------------------------------------------------------------------
# 7. Re-entry separation
#
# Every NEW_EPISODE transition must have a temporal gap > 5 minutes.
# --------------------------------------------------------------------------------------------------

_invalid_reentry_separation = (
    _new_episode_rows[
        _new_episode_rows["source_gap_minutes"]
        <= 5.0
    ]
)


print("\nRE-ENTRY SEPARATION")
print("-" * 100)

print(
    f"Invalid re-entry separations : "
    f"{len(_invalid_reentry_separation):,}"
)


# --------------------------------------------------------------------------------------------------
# 8. Episode internal datetime monotonicity
# --------------------------------------------------------------------------------------------------

_episode_time_order_violations = 0

for _episode_id, _group in (
    _continuity_obs_v1
    .groupby(
        "interaction_episode_id",
        sort=False,
    )
):

    _times = _group["datetime"]

    if not _times.is_monotonic_increasing:
        _episode_time_order_violations += 1


print("\nEPISODE CHRONOLOGICAL ORDER")
print("-" * 100)

print(
    f"Chronological violations : "
    f"{_episode_time_order_violations:,}"
)


# --------------------------------------------------------------------------------------------------
# 9. Episode duration consistency
#
# For an episode with N observations at exact 5-minute intervals:
#
# duration_minutes = (N - 1) * 5
# --------------------------------------------------------------------------------------------------

_episode_duration_check = (
    _continuity_obs_v1
    .groupby("interaction_episode_id")
    .agg(
        observation_count=(
            "datetime",
            "size",
        ),
        start_time=(
            "datetime",
            "min",
        ),
        end_time=(
            "datetime",
            "max",
        ),
    )
)

_episode_duration_check["actual_duration_minutes"] = (
    (
        _episode_duration_check["end_time"]
        -
        _episode_duration_check["start_time"]
    )
    .dt.total_seconds()
    .div(60)
)

_episode_duration_check["expected_duration_minutes"] = (
    (
        _episode_duration_check["observation_count"]
        - 1
    )
    * 5
)

_duration_mismatch = (
    _episode_duration_check[
        _episode_duration_check[
            "actual_duration_minutes"
        ]
        !=
        _episode_duration_check[
            "expected_duration_minutes"
        ]
    ]
)


print("\nEPISODE DURATION CONSISTENCY")
print("-" * 100)

print(
    f"Duration mismatches : "
    f"{len(_duration_mismatch):,}"
)


# --------------------------------------------------------------------------------------------------
# 10. Episode observation count vs sequence span
# --------------------------------------------------------------------------------------------------

_episode_sequence_span = (
    _continuity_obs_v1
    .groupby("interaction_episode_id")
    .agg(
        first_datetime=("datetime", "min"),
        last_datetime=("datetime", "max"),
        observation_count=("datetime", "size"),
    )
)

_episode_sequence_span[
    "expected_observations"
] = (
    (
        (
            _episode_sequence_span["last_datetime"]
            -
            _episode_sequence_span["first_datetime"]
        )
        .dt.total_seconds()
        .div(300)
    )
    .astype(int)
    + 1
)

_sequence_span_mismatch = (
    _episode_sequence_span[
        _episode_sequence_span["observation_count"]
        !=
        _episode_sequence_span["expected_observations"]
    ]
)


print("\nEPISODE SEQUENCE SPAN")
print("-" * 100)

print(
    f"Sequence span mismatches : "
    f"{len(_sequence_span_mismatch):,}"
)


# --------------------------------------------------------------------------------------------------
# 11. Final Gate
# --------------------------------------------------------------------------------------------------

_continuity_gate_v1 = {

    "WITHIN_EPISODE_5M_CONTINUITY": (
        len(_invalid_within_episode_gap) == 0
    ),

    "NO_5M_EPISODE_SPLITS": (
        len(_invalid_new_episode_gap) == 0
    ),

    "REENTRY_SEPARATION": (
        len(_invalid_reentry_separation) == 0
    ),

    "CHRONOLOGICAL_ORDER": (
        _episode_time_order_violations == 0
    ),

    "EPISODE_DURATION_CONSISTENCY": (
        len(_duration_mismatch) == 0
    ),

    "EPISODE_SEQUENCE_SPAN": (
        len(_sequence_span_mismatch) == 0
    ),
}


print("\n" + "=" * 100)
print("STEP 7.3 — EPISODE CONTINUITY AUDIT GATE")
print("=" * 100)

for _gate_name, _gate_value in _continuity_gate_v1.items():

    print(
        f"{_gate_name:<45}: "
        f"{'PASSED' if _gate_value else 'FAILED'}"
    )


print("\n" + "=" * 100)

if all(_continuity_gate_v1.values()):

    print("STEP 7.3 EPISODE CONTINUITY AUDIT : PASSED")

else:

    print("STEP 7.3 EPISODE CONTINUITY AUDIT : FAILED")

print("=" * 100)

In [ ]:
# ==================================================================================================
# STEP 7.4 — INTERACTION EPISODE SEPARATION / RE-ENTRY AUDIT V1
# ==================================================================================================

print("=" * 100)
print("STEP 7.4 — INTERACTION EPISODE SEPARATION / RE-ENTRY AUDIT V1")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Working registry
# --------------------------------------------------------------------------------------------------

_separation_registry_v1 = (
    interaction_episode_registry_v1_1
    .copy()
    .sort_values(
        ["fvg_id", "episode_sequence"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 2. Previous / next Episode boundaries
# --------------------------------------------------------------------------------------------------

_separation_registry_v1["previous_episode_id"] = (
    _separation_registry_v1
    .groupby("fvg_id")[
        "interaction_episode_id"
    ]
    .shift(1)
)

_separation_registry_v1["previous_end_time"] = (
    _separation_registry_v1
    .groupby("fvg_id")[
        "interaction_episode_end"
    ]
    .shift(1)
)

_separation_registry_v1["previous_sequence"] = (
    _separation_registry_v1
    .groupby("fvg_id")[
        "episode_sequence"
    ]
    .shift(1)
)


_separation_registry_v1["next_start_time"] = (
    _separation_registry_v1
    .groupby("fvg_id")[
        "interaction_episode_start"
    ]
    .shift(-1)
)

_separation_registry_v1["next_sequence"] = (
    _separation_registry_v1
    .groupby("fvg_id")[
        "episode_sequence"
    ]
    .shift(-1)
)


# --------------------------------------------------------------------------------------------------
# 3. Episode sequence increment
# --------------------------------------------------------------------------------------------------

_transition_rows_v1 = (
    _separation_registry_v1[
        _separation_registry_v1["previous_episode_id"].notna()
    ]
    .copy()
)

_transition_rows_v1["sequence_increment"] = (
    _transition_rows_v1["episode_sequence"]
    -
    _transition_rows_v1["previous_sequence"]
)


print("\nEPISODE SEQUENCE INCREMENT")
print("-" * 100)

print(
    _transition_rows_v1[
        "sequence_increment"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


# --------------------------------------------------------------------------------------------------
# 4. Invalid sequence transitions
# --------------------------------------------------------------------------------------------------

_invalid_sequence_increment = (
    _transition_rows_v1[
        _transition_rows_v1["sequence_increment"] != 1
    ]
)


print("\nINVALID SEQUENCE TRANSITIONS")
print("-" * 100)

print(
    f"Invalid transitions : "
    f"{len(_invalid_sequence_increment):,}"
)


# --------------------------------------------------------------------------------------------------
# 5. Episode temporal ordering
# --------------------------------------------------------------------------------------------------

_invalid_temporal_order = (
    _transition_rows_v1[
        _transition_rows_v1["interaction_episode_start"]
        <=
        _transition_rows_v1["previous_end_time"]
    ]
)


print("\nEPISODE TEMPORAL ORDER")
print("-" * 100)

print(
    f"Overlapping / touching Episodes : "
    f"{len(_invalid_temporal_order):,}"
)


# --------------------------------------------------------------------------------------------------
# 6. Re-entry gap
# --------------------------------------------------------------------------------------------------

_transition_rows_v1["reentry_gap_minutes"] = (
    (
        _transition_rows_v1["interaction_episode_start"]
        -
        _transition_rows_v1["previous_end_time"]
    )
    .dt.total_seconds()
    .div(60)
)


print("\nRE-ENTRY GAP DISTRIBUTION")
print("-" * 100)

print(
    _transition_rows_v1[
        "reentry_gap_minutes"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .to_string()
)


# --------------------------------------------------------------------------------------------------
# 7. Re-entry gap validity
#
# The next Episode must start more than 5 minutes after the previous
# Episode's final observation.
# --------------------------------------------------------------------------------------------------

_invalid_reentry_gap = (
    _transition_rows_v1[
        _transition_rows_v1["reentry_gap_minutes"] <= 5.0
    ]
)


print("\nRE-ENTRY GAP VALIDITY")
print("-" * 100)

print(
    f"Invalid re-entry gaps : "
    f"{len(_invalid_reentry_gap):,}"
)


# --------------------------------------------------------------------------------------------------
# 8. First Episode sequence per FVG
# --------------------------------------------------------------------------------------------------

_first_episode_sequence = (
    _separation_registry_v1
    .groupby("fvg_id")[
        "episode_sequence"
    ]
    .min()
)

_invalid_first_sequences = (
    _first_episode_sequence[
        _first_episode_sequence != 1
    ]
)


print("\nFIRST EPISODE SEQUENCE")
print("-" * 100)

print(
    f"FVGs with first sequence != 1 : "
    f"{len(_invalid_first_sequences):,}"
)


# --------------------------------------------------------------------------------------------------
# 9. Last sequence vs Episode count
# --------------------------------------------------------------------------------------------------

_fvg_episode_summary = (
    _separation_registry_v1
    .groupby("fvg_id")
    .agg(
        episode_count=(
            "interaction_episode_id",
            "size",
        ),
        last_episode_sequence=(
            "episode_sequence",
            "max",
        ),
    )
)

_invalid_last_sequences = (
    _fvg_episode_summary[
        _fvg_episode_summary["episode_count"]
        !=
        _fvg_episode_summary["last_episode_sequence"]
    ]
)


print("\nLAST EPISODE SEQUENCE")
print("-" * 100)

print(
    f"FVGs with invalid last sequence : "
    f"{len(_invalid_last_sequences):,}"
)


# --------------------------------------------------------------------------------------------------
# 10. Empty Episode detection
# --------------------------------------------------------------------------------------------------

_empty_episode_count = (
    _separation_registry_v1[
        _separation_registry_v1[
            "interaction_observation_count"
        ] <= 0
    ]
    .shape[0]
)


print("\nEMPTY EPISODE AUDIT")
print("-" * 100)

print(
    f"Empty Episodes : "
    f"{_empty_episode_count:,}"
)


# --------------------------------------------------------------------------------------------------
# 11. Duplicate Episode boundary
# --------------------------------------------------------------------------------------------------

_duplicate_boundaries = (
    _separation_registry_v1
    .duplicated(
        subset=[
            "fvg_id",
            "interaction_episode_start",
            "interaction_episode_end",
        ],
        keep=False,
    )
)

print("\nDUPLICATE EPISODE BOUNDARIES")
print("-" * 100)

print(
    f"Duplicate boundary groups : "
    f"{_duplicate_boundaries.sum():,}"
)


# --------------------------------------------------------------------------------------------------
# 12. Final Gate
# --------------------------------------------------------------------------------------------------

_separation_gate_v1 = {

    "SEQUENCE_INCREMENT": (
        len(_invalid_sequence_increment) == 0
    ),

    "TEMPORAL_SEPARATION": (
        len(_invalid_temporal_order) == 0
    ),

    "REENTRY_GAP_VALIDITY": (
        len(_invalid_reentry_gap) == 0
    ),

    "FIRST_SEQUENCE_IS_ONE": (
        len(_invalid_first_sequences) == 0
    ),

    "LAST_SEQUENCE_MATCHES_COUNT": (
        len(_invalid_last_sequences) == 0
    ),

    "NO_EMPTY_EPISODES": (
        _empty_episode_count == 0
    ),

    "NO_DUPLICATE_BOUNDARIES": (
        _duplicate_boundaries.sum() == 0
    ),
}


print("\n" + "=" * 100)
print("STEP 7.4 — EPISODE SEPARATION / RE-ENTRY AUDIT GATE")
print("=" * 100)

for _gate_name, _gate_value in _separation_gate_v1.items():

    print(
        f"{_gate_name:<45}: "
        f"{'PASSED' if _gate_value else 'FAILED'}"
    )


print("\n" + "=" * 100)

if all(_separation_gate_v1.values()):

    print(
        "STEP 7.4 EPISODE SEPARATION / RE-ENTRY AUDIT : PASSED"
    )

else:

    print(
        "STEP 7.4 EPISODE SEPARATION / RE-ENTRY AUDIT : FAILED"
    )

print("=" * 100)

In [ ]:
# ==================================================================================================
# STEP 7.5 — INTERACTION EPISODE BOUNDARY AUDIT V1
# CORRECTED — NO DEPENDENCY ON interaction_episode_timeline_v1
# ==================================================================================================

print("=" * 100)
print("STEP 7.5 — INTERACTION EPISODE BOUNDARY AUDIT V1")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# SOURCE DATA
# --------------------------------------------------------------------------------------------------

_boundary_obs_v1 = (
    _penetration_boundary_audit
    .copy()
)

_boundary_registry_v1 = (
    interaction_episode_registry_v1_1
    .copy()
    .sort_values(
        ["fvg_id", "episode_sequence"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


_required_obs_cols = {
    "fvg_id",
    "datetime",
    "penetration",
    "penetration_ratio",
    "penetration_class",
}

_missing_obs_cols = (
    _required_obs_cols
    -
    set(_boundary_obs_v1.columns)
)

if _missing_obs_cols:
    raise KeyError(
        f"Missing observation columns: "
        f"{sorted(_missing_obs_cols)}"
    )


_required_registry_cols = {
    "fvg_id",
    "episode_sequence",
    "interaction_episode_id",
    "interaction_episode_start",
    "interaction_episode_end",
    "interaction_observation_count",
    "max_penetration",
    "max_penetration_ratio",
}

_missing_registry_cols = (
    _required_registry_cols
    -
    set(_boundary_registry_v1.columns)
)

if _missing_registry_cols:
    raise KeyError(
        f"Missing registry columns: "
        f"{sorted(_missing_registry_cols)}"
    )


# --------------------------------------------------------------------------------------------------
# RECONSTRUCT EPISODE MEMBERSHIP
#
# The authoritative Episode definition is:
#   same FVG
#   same 5-minute continuity chain
#   sequence determined by exact gap == 5 minutes
#
# Therefore reconstruct the Episode ID independently from the observation source.
# --------------------------------------------------------------------------------------------------

_boundary_obs_v1 = (
    _boundary_obs_v1
    .sort_values(
        ["fvg_id", "datetime"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

_boundary_obs_v1["interaction_gap_minutes"] = (
    _boundary_obs_v1
    .groupby("fvg_id")["datetime"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

_boundary_obs_v1["_episode_break"] = (
    _boundary_obs_v1["interaction_gap_minutes"].isna()
    |
    (
        _boundary_obs_v1["interaction_gap_minutes"]
        !=
        5.0
    )
)

_boundary_obs_v1["episode_sequence"] = (
    _boundary_obs_v1
    .groupby("fvg_id")["_episode_break"]
    .cumsum()
    .astype(int)
)

_boundary_obs_v1["interaction_episode_id"] = (
    _boundary_obs_v1["fvg_id"].astype(str)
    + "_EP_"
    + _boundary_obs_v1["episode_sequence"]
    .astype(str)
    .str.zfill(3)
)


print("\nOBSERVATION → EPISODE RECONSTRUCTION")
print("-" * 100)

print(
    f"Observation rows : "
    f"{len(_boundary_obs_v1):,}"
)

print(
    f"Unique FVGs : "
    f"{_boundary_obs_v1['fvg_id'].nunique():,}"
)

print(
    f"Unique reconstructed Episodes : "
    f"{_boundary_obs_v1['interaction_episode_id'].nunique():,}"
)


# --------------------------------------------------------------------------------------------------
# OBSERVATION → REGISTRY ASSIGNMENT
# --------------------------------------------------------------------------------------------------

_boundary_obs_v1 = (
    _boundary_obs_v1
    .merge(
        _boundary_registry_v1[
            [
                "fvg_id",
                "episode_sequence",
                "interaction_episode_id",
            ]
        ],
        on=[
            "fvg_id",
            "episode_sequence",
            "interaction_episode_id",
        ],
        how="left",
        validate="many_to_one",
    )
)


_unassigned_observations = (
    _boundary_obs_v1[
        _boundary_obs_v1[
            "interaction_episode_id"
        ].isna()
    ]
)


print("\nOBSERVATION → REGISTRY ASSIGNMENT")
print("-" * 100)

print(
    f"Unassigned observations : "
    f"{len(_unassigned_observations):,}"
)


# --------------------------------------------------------------------------------------------------
# RECONSTRUCT ACTUAL EPISODE BOUNDARIES
# --------------------------------------------------------------------------------------------------

_actual_episode_bounds_v1 = (
    _boundary_obs_v1
    .groupby(
        [
            "fvg_id",
            "episode_sequence",
            "interaction_episode_id",
        ],
        sort=True,
    )
    .agg(
        actual_start=(
            "datetime",
            "min",
        ),
        actual_end=(
            "datetime",
            "max",
        ),
        actual_observation_count=(
            "datetime",
            "size",
        ),
        actual_max_penetration=(
            "penetration",
            "max",
        ),
        actual_max_penetration_ratio=(
            "penetration_ratio",
            "max",
        ),
    )
    .reset_index()
)


_boundary_check_v1 = (
    _boundary_registry_v1
    .merge(
        _actual_episode_bounds_v1,
        on=[
            "fvg_id",
            "episode_sequence",
            "interaction_episode_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


# --------------------------------------------------------------------------------------------------
# START BOUNDARY
# --------------------------------------------------------------------------------------------------

_boundary_check_v1["start_boundary_valid"] = (
    _boundary_check_v1["interaction_episode_start"]
    ==
    _boundary_check_v1["actual_start"]
)

_invalid_start_boundaries = (
    _boundary_check_v1[
        ~_boundary_check_v1["start_boundary_valid"]
    ]
)

print("\nSTART BOUNDARY")
print("-" * 100)

print(
    f"Invalid start boundaries : "
    f"{len(_invalid_start_boundaries):,}"
)


# --------------------------------------------------------------------------------------------------
# END BOUNDARY
# --------------------------------------------------------------------------------------------------

_boundary_check_v1["end_boundary_valid"] = (
    _boundary_check_v1["interaction_episode_end"]
    ==
    _boundary_check_v1["actual_end"]
)

_invalid_end_boundaries = (
    _boundary_check_v1[
        ~_boundary_check_v1["end_boundary_valid"]
    ]
)

print("\nEND BOUNDARY")
print("-" * 100)

print(
    f"Invalid end boundaries : "
    f"{len(_invalid_end_boundaries):,}"
)


# --------------------------------------------------------------------------------------------------
# OBSERVATION COUNT
# --------------------------------------------------------------------------------------------------

_boundary_check_v1["observation_count_valid"] = (
    _boundary_check_v1["interaction_observation_count"]
    ==
    _boundary_check_v1["actual_observation_count"]
)

_invalid_observation_counts = (
    _boundary_check_v1[
        ~_boundary_check_v1["observation_count_valid"]
    ]
)

print("\nOBSERVATION COUNT")
print("-" * 100)

print(
    f"Invalid observation counts : "
    f"{len(_invalid_observation_counts):,}"
)


# --------------------------------------------------------------------------------------------------
# MAX PENETRATION
# --------------------------------------------------------------------------------------------------

_boundary_check_v1["max_penetration_valid"] = (
    _boundary_check_v1["max_penetration"]
    ==
    _boundary_check_v1["actual_max_penetration"]
)

_invalid_max_penetration = (
    _boundary_check_v1[
        ~_boundary_check_v1["max_penetration_valid"]
    ]
)

print("\nMAX PENETRATION")
print("-" * 100)

print(
    f"Invalid max penetration : "
    f"{len(_invalid_max_penetration):,}"
)


# --------------------------------------------------------------------------------------------------
# MAX PENETRATION RATIO
# --------------------------------------------------------------------------------------------------

_boundary_check_v1["max_penetration_ratio_valid"] = (
    _boundary_check_v1["max_penetration_ratio"]
    ==
    _boundary_check_v1["actual_max_penetration_ratio"]
)

_invalid_max_penetration_ratio = (
    _boundary_check_v1[
        ~_boundary_check_v1["max_penetration_ratio_valid"]
    ]
)

print("\nMAX PENETRATION RATIO")
print("-" * 100)

print(
    f"Invalid max penetration ratios : "
    f"{len(_invalid_max_penetration_ratio):,}"
)


# --------------------------------------------------------------------------------------------------
# EPISODE DURATION
# --------------------------------------------------------------------------------------------------

_boundary_check_v1["registry_duration_minutes"] = (
    (
        _boundary_check_v1["interaction_episode_end"]
        -
        _boundary_check_v1["interaction_episode_start"]
    )
    .dt.total_seconds()
    .div(60)
)

_boundary_check_v1["actual_duration_minutes"] = (
    (
        _boundary_check_v1["actual_end"]
        -
        _boundary_check_v1["actual_start"]
    )
    .dt.total_seconds()
    .div(60)
)

_boundary_check_v1["duration_valid"] = (
    _boundary_check_v1["registry_duration_minutes"]
    ==
    _boundary_check_v1["actual_duration_minutes"]
)

_invalid_durations = (
    _boundary_check_v1[
        ~_boundary_check_v1["duration_valid"]
    ]
)

print("\nEPISODE DURATION")
print("-" * 100)

print(
    f"Invalid episode durations : "
    f"{len(_invalid_durations):,}"
)


# --------------------------------------------------------------------------------------------------
# SINGLETON BOUNDARY CONTRACT
# --------------------------------------------------------------------------------------------------

_singleton_rows_v1 = (
    _boundary_check_v1[
        _boundary_check_v1[
            "actual_observation_count"
        ]
        ==
        1
    ]
    .copy()
)

_singleton_boundary_invalid = (
    _singleton_rows_v1[
        _singleton_rows_v1["actual_start"]
        !=
        _singleton_rows_v1["actual_end"]
    ]
)

print("\nSINGLETON BOUNDARY CONTRACT")
print("-" * 100)

print(
    f"Singleton Episodes : "
    f"{len(_singleton_rows_v1):,}"
)

print(
    f"Invalid singleton boundaries : "
    f"{len(_singleton_boundary_invalid):,}"
)


# --------------------------------------------------------------------------------------------------
# MULTI-OBSERVATION EPISODE CONTRACT
# --------------------------------------------------------------------------------------------------

_multi_rows_v1 = (
    _boundary_check_v1[
        _boundary_check_v1[
            "actual_observation_count"
        ]
        >
        1
    ]
    .copy()
)

_multi_duration_invalid = (
    _multi_rows_v1[
        _multi_rows_v1["actual_duration_minutes"]
        <
        5.0
    ]
)

print("\nMULTI-OBSERVATION EPISODE CONTRACT")
print("-" * 100)

print(
    f"Multi-observation Episodes : "
    f"{len(_multi_rows_v1):,}"
)

print(
    f"Invalid multi-observation durations : "
    f"{len(_multi_duration_invalid):,}"
)


# --------------------------------------------------------------------------------------------------
# GLOBAL BOUNDARY GATE
# --------------------------------------------------------------------------------------------------

_boundary_gate_v1 = {
    "ALL_OBSERVATIONS_ASSIGNED": (
        len(_unassigned_observations) == 0
    ),
    "START_BOUNDARY_ALIGNMENT": (
        len(_invalid_start_boundaries) == 0
    ),
    "END_BOUNDARY_ALIGNMENT": (
        len(_invalid_end_boundaries) == 0
    ),
    "OBSERVATION_COUNT_ALIGNMENT": (
        len(_invalid_observation_counts) == 0
    ),
    "MAX_PENETRATION_ALIGNMENT": (
        len(_invalid_max_penetration) == 0
    ),
    "MAX_PENETRATION_RATIO_ALIGNMENT": (
        len(_invalid_max_penetration_ratio) == 0
    ),
    "DURATION_ALIGNMENT": (
        len(_invalid_durations) == 0
    ),
    "SINGLETON_BOUNDARY_CONTRACT": (
        len(_singleton_boundary_invalid) == 0
    ),
    "MULTI_OBSERVATION_DURATION_CONTRACT": (
        len(_multi_duration_invalid) == 0
    ),
}


print("\n" + "=" * 100)
print("STEP 7.5 — EPISODE BOUNDARY AUDIT GATE")
print("=" * 100)

for _gate_name, _gate_value in _boundary_gate_v1.items():
    print(
        f"{_gate_name:<45}: "
        f"{'PASSED' if _gate_value else 'FAILED'}"
    )

print("\n" + "=" * 100)

if all(_boundary_gate_v1.values()):
    print(
        "STEP 7.5 EPISODE BOUNDARY AUDIT : PASSED"
    )
else:
    print(
        "STEP 7.5 EPISODE BOUNDARY AUDIT : FAILED"
    )

print("=" * 100)

In [ ]:
# ==================================================================================================
# STEP 7.6 — FVG → EPISODE MAPPING AUDIT V1
# ==================================================================================================

print("=" * 100)
print("STEP 7.6 — FVG → EPISODE MAPPING AUDIT V1")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# SOURCE DATA
# --------------------------------------------------------------------------------------------------

_mapping_registry_v1 = (
    interaction_episode_registry_v1_1
    .copy()
    .sort_values(
        ["fvg_id", "episode_sequence"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

_mapping_obs_v1 = (
    _penetration_boundary_audit
    .copy()
    .sort_values(
        ["fvg_id", "datetime"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# RECONSTRUCT AUTHORITATIVE OBSERVATION → EPISODE MAPPING
# --------------------------------------------------------------------------------------------------

_mapping_obs_v1["interaction_gap_minutes"] = (
    _mapping_obs_v1
    .groupby("fvg_id")["datetime"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

_mapping_obs_v1["_episode_break"] = (
    _mapping_obs_v1["interaction_gap_minutes"].isna()
    |
    (
        _mapping_obs_v1["interaction_gap_minutes"]
        !=
        5.0
    )
)

_mapping_obs_v1["episode_sequence"] = (
    _mapping_obs_v1
    .groupby("fvg_id")["_episode_break"]
    .cumsum()
    .astype(int)
)

_mapping_obs_v1["interaction_episode_id"] = (
    _mapping_obs_v1["fvg_id"].astype(str)
    + "_EP_"
    + _mapping_obs_v1["episode_sequence"]
    .astype(str)
    .str.zfill(3)
)


# --------------------------------------------------------------------------------------------------
# BASIC COUNTS
# --------------------------------------------------------------------------------------------------

print("\nBASIC MAPPING COUNTS")
print("-" * 100)

print(
    f"Observation rows : "
    f"{len(_mapping_obs_v1):,}"
)

print(
    f"Registry Episode rows : "
    f"{len(_mapping_registry_v1):,}"
)

print(
    f"Observation FVGs : "
    f"{_mapping_obs_v1['fvg_id'].nunique():,}"
)

print(
    f"Registry FVGs : "
    f"{_mapping_registry_v1['fvg_id'].nunique():,}"
)

print(
    f"Observation Episodes : "
    f"{_mapping_obs_v1['interaction_episode_id'].nunique():,}"
)

print(
    f"Registry Episodes : "
    f"{_mapping_registry_v1['interaction_episode_id'].nunique():,}"
)


# --------------------------------------------------------------------------------------------------
# FVG SET ALIGNMENT
# --------------------------------------------------------------------------------------------------

_observation_fvgs = set(
    _mapping_obs_v1["fvg_id"].unique()
)

_registry_fvgs = set(
    _mapping_registry_v1["fvg_id"].unique()
)

_fvgs_missing_from_registry = (
    _observation_fvgs
    -
    _registry_fvgs
)

_extra_registry_fvgs = (
    _registry_fvgs
    -
    _observation_fvgs
)

print("\nFVG SET ALIGNMENT")
print("-" * 100)

print(
    f"FVGs missing from Registry : "
    f"{len(_fvgs_missing_from_registry):,}"
)

print(
    f"Extra Registry FVGs : "
    f"{len(_extra_registry_fvgs):,}"
)


# --------------------------------------------------------------------------------------------------
# EPISODE SET ALIGNMENT
# --------------------------------------------------------------------------------------------------

_observation_episode_ids = set(
    _mapping_obs_v1["interaction_episode_id"].unique()
)

_registry_episode_ids = set(
    _mapping_registry_v1["interaction_episode_id"].unique()
)

_episodes_missing_from_registry = (
    _observation_episode_ids
    -
    _registry_episode_ids
)

_extra_registry_episodes = (
    _registry_episode_ids
    -
    _observation_episode_ids
)

print("\nEPISODE SET ALIGNMENT")
print("-" * 100)

print(
    f"Episodes missing from Registry : "
    f"{len(_episodes_missing_from_registry):,}"
)

print(
    f"Extra Registry Episodes : "
    f"{len(_extra_registry_episodes):,}"
)


# --------------------------------------------------------------------------------------------------
# FVG → EPISODE COUNT ALIGNMENT
# --------------------------------------------------------------------------------------------------

_observation_fvg_episode_counts = (
    _mapping_obs_v1
    .groupby("fvg_id")[
        "interaction_episode_id"
    ]
    .nunique()
    .rename("observation_episode_count")
)

_registry_fvg_episode_counts = (
    _mapping_registry_v1
    .groupby("fvg_id")[
        "interaction_episode_id"
    ]
    .nunique()
    .rename("registry_episode_count")
)

_fvg_episode_count_check = (
    _observation_fvg_episode_counts
    .to_frame()
    .join(
        _registry_fvg_episode_counts,
        how="outer",
    )
)

_fvg_episode_count_check[
    "count_valid"
] = (
    _fvg_episode_count_check[
        "observation_episode_count"
    ]
    ==
    _fvg_episode_count_check[
        "registry_episode_count"
    ]
)

_invalid_fvg_episode_counts = (
    _fvg_episode_count_check[
        ~_fvg_episode_count_check["count_valid"]
    ]
)

print("\nFVG → EPISODE COUNT ALIGNMENT")
print("-" * 100)

print(
    f"FVGs with episode count mismatch : "
    f"{len(_invalid_fvg_episode_counts):,}"
)


# --------------------------------------------------------------------------------------------------
# FVG → OBSERVATION COUNT ALIGNMENT
# --------------------------------------------------------------------------------------------------

_observation_fvg_counts = (
    _mapping_obs_v1
    .groupby("fvg_id")
    .size()
    .rename("observation_count")
)

_registry_fvg_observation_counts = (
    _mapping_registry_v1
    .groupby("fvg_id")[
        "interaction_observation_count"
    ]
    .sum()
    .rename("registry_observation_count")
)

_fvg_observation_count_check = (
    _observation_fvg_counts
    .to_frame()
    .join(
        _registry_fvg_observation_counts,
        how="outer",
    )
)

_fvg_observation_count_check[
    "count_valid"
] = (
    _fvg_observation_count_check[
        "observation_count"
    ]
    ==
    _fvg_observation_count_check[
        "registry_observation_count"
    ]
)

_invalid_fvg_observation_counts = (
    _fvg_observation_count_check[
        ~_fvg_observation_count_check["count_valid"]
    ]
)

print("\nFVG → OBSERVATION COUNT ALIGNMENT")
print("-" * 100)

print(
    f"FVGs with observation count mismatch : "
    f"{len(_invalid_fvg_observation_counts):,}"
)


# --------------------------------------------------------------------------------------------------
# FVG SEQUENCE CONTRACT
# --------------------------------------------------------------------------------------------------

_fvg_sequence_summary = (
    _mapping_registry_v1
    .groupby("fvg_id")
    .agg(
        episode_count=(
            "interaction_episode_id",
            "size",
        ),
        min_sequence=(
            "episode_sequence",
            "min",
        ),
        max_sequence=(
            "episode_sequence",
            "max",
        ),
        unique_sequences=(
            "episode_sequence",
            "nunique",
        ),
    )
)

_fvg_sequence_summary[
    "sequence_valid"
] = (
    (_fvg_sequence_summary["min_sequence"] == 1)
    &
    (
        _fvg_sequence_summary["max_sequence"]
        ==
        _fvg_sequence_summary["episode_count"]
    )
    &
    (
        _fvg_sequence_summary["unique_sequences"]
        ==
        _fvg_sequence_summary["episode_count"]
    )
)

_invalid_fvg_sequences = (
    _fvg_sequence_summary[
        ~_fvg_sequence_summary["sequence_valid"]
    ]
)

print("\nFVG EPISODE SEQUENCE CONTRACT")
print("-" * 100)

print(
    f"FVGs with invalid Episode sequences : "
    f"{len(_invalid_fvg_sequences):,}"
)


# --------------------------------------------------------------------------------------------------
# CROSS-FVG EPISODE CONTAMINATION
# --------------------------------------------------------------------------------------------------

_episode_fvg_cardinality = (
    _mapping_obs_v1
    .groupby("interaction_episode_id")[
        "fvg_id"
    ]
    .nunique()
)

_cross_fvg_observation_contamination = (
    _episode_fvg_cardinality[
        _episode_fvg_cardinality != 1
    ]
)

print("\nCROSS-FVG OBSERVATION CONTAMINATION")
print("-" * 100)

print(
    f"Episodes linked to multiple FVGs : "
    f"{len(_cross_fvg_observation_contamination):,}"
)


# --------------------------------------------------------------------------------------------------
# REGISTRY CROSS-FVG CONTAMINATION
# --------------------------------------------------------------------------------------------------

_registry_episode_fvg_cardinality = (
    _mapping_registry_v1
    .groupby("interaction_episode_id")[
        "fvg_id"
    ]
    .nunique()
)

_cross_fvg_registry_contamination = (
    _registry_episode_fvg_cardinality[
        _registry_episode_fvg_cardinality != 1
    ]
)

print("\nCROSS-FVG REGISTRY CONTAMINATION")
print("-" * 100)

print(
    f"Registry Episodes linked to multiple FVGs : "
    f"{len(_cross_fvg_registry_contamination):,}"
)


# --------------------------------------------------------------------------------------------------
# FVG OWNERSHIP CONTRACT
# --------------------------------------------------------------------------------------------------

_mapping_obs_v1["episode_id_fvg_prefix"] = (
    _mapping_obs_v1[
        "interaction_episode_id"
    ]
    .str.rsplit("_EP_", n=1)
    .str[0]
)

_fvg_prefix_mismatch = (
    _mapping_obs_v1[
        _mapping_obs_v1["episode_id_fvg_prefix"]
        !=
        _mapping_obs_v1["fvg_id"]
    ]
)

print("\nFVG OWNERSHIP CONTRACT")
print("-" * 100)

print(
    f"Observation rows with Episode/FVG ownership mismatch : "
    f"{len(_fvg_prefix_mismatch):,}"
)


# --------------------------------------------------------------------------------------------------
# REGISTRY OWNERSHIP CONTRACT
# --------------------------------------------------------------------------------------------------

_mapping_registry_v1["episode_id_fvg_prefix"] = (
    _mapping_registry_v1[
        "interaction_episode_id"
    ]
    .str.rsplit("_EP_", n=1)
    .str[0]
)

_registry_prefix_mismatch = (
    _mapping_registry_v1[
        _mapping_registry_v1[
            "episode_id_fvg_prefix"
        ]
        !=
        _mapping_registry_v1["fvg_id"]
    ]
)

print("\nREGISTRY OWNERSHIP CONTRACT")
print("-" * 100)

print(
    f"Registry rows with Episode/FVG ownership mismatch : "
    f"{len(_registry_prefix_mismatch):,}"
)


# --------------------------------------------------------------------------------------------------
# OBSERVATION → SINGLE EPISODE CONTRACT
# --------------------------------------------------------------------------------------------------

_observation_episode_cardinality = (
    _mapping_obs_v1
    .groupby(
        [
            "fvg_id",
            "datetime",
        ]
    )[
        "interaction_episode_id"
    ]
    .nunique()
)

_observation_multiple_episode_mapping = (
    _observation_episode_cardinality[
        _observation_episode_cardinality != 1
    ]
)

print("\nOBSERVATION → SINGLE EPISODE CONTRACT")
print("-" * 100)

print(
    f"Observations mapped to != 1 Episode : "
    f"{len(_observation_multiple_episode_mapping):,}"
)


# --------------------------------------------------------------------------------------------------
# FVG WITH ZERO EPISODES
# --------------------------------------------------------------------------------------------------

_fvgs_without_episode = (
    _observation_fvgs
    -
    set(
        _mapping_registry_v1[
            "fvg_id"
        ].unique()
    )
)

print("\nFVG WITHOUT EPISODE")
print("-" * 100)

print(
    f"FVGs without Episodes : "
    f"{len(_fvgs_without_episode):,}"
)


# --------------------------------------------------------------------------------------------------
# FINAL GATE
# --------------------------------------------------------------------------------------------------

_mapping_gate_v1 = {
    "FVG_SET_ALIGNMENT": (
        len(_fvgs_missing_from_registry) == 0
        and
        len(_extra_registry_fvgs) == 0
    ),
    "EPISODE_SET_ALIGNMENT": (
        len(_episodes_missing_from_registry) == 0
        and
        len(_extra_registry_episodes) == 0
    ),
    "FVG_EPISODE_COUNT_ALIGNMENT": (
        len(_invalid_fvg_episode_counts) == 0
    ),
    "FVG_OBSERVATION_COUNT_ALIGNMENT": (
        len(_invalid_fvg_observation_counts) == 0
    ),
    "FVG_SEQUENCE_CONTRACT": (
        len(_invalid_fvg_sequences) == 0
    ),
    "NO_CROSS_FVG_OBSERVATION_CONTAMINATION": (
        len(_cross_fvg_observation_contamination) == 0
    ),
    "NO_CROSS_FVG_REGISTRY_CONTAMINATION": (
        len(_cross_fvg_registry_contamination) == 0
    ),
    "OBSERVATION_FVG_OWNERSHIP": (
        len(_fvg_prefix_mismatch) == 0
    ),
    "REGISTRY_FVG_OWNERSHIP": (
        len(_registry_prefix_mismatch) == 0
    ),
    "OBSERVATION_SINGLE_EPISODE": (
        len(_observation_multiple_episode_mapping) == 0
    ),
    "NO_FVG_WITHOUT_EPISODE": (
        len(_fvgs_without_episode) == 0
    ),
}


print("\n" + "=" * 100)
print("STEP 7.6 — FVG → EPISODE MAPPING AUDIT GATE")
print("=" * 100)

for _gate_name, _gate_value in _mapping_gate_v1.items():
    print(
        f"{_gate_name:<45}: "
        f"{'PASSED' if _gate_value else 'FAILED'}"
    )

print("\n" + "=" * 100)

if all(_mapping_gate_v1.values()):
    print(
        "STEP 7.6 FVG → EPISODE MAPPING AUDIT : PASSED"
    )
else:
    print(
        "STEP 7.6 FVG → EPISODE MAPPING AUDIT : FAILED"
    )

print("=" * 100)

In [ ]:
# ==================================================================================================
# STEP 7.7 — FULL INTERACTION EPISODE REGRESSION GATE V1
# ==================================================================================================

print("=" * 100)
print("STEP 7.7 — FULL INTERACTION EPISODE REGRESSION GATE V1")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# SOURCE DATA
# --------------------------------------------------------------------------------------------------

_reg_obs_v1 = (
    _penetration_boundary_audit
    .copy()
    .sort_values(
        ["fvg_id", "datetime"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

_reg_registry_v1 = (
    interaction_episode_registry_v1_1
    .copy()
    .sort_values(
        ["fvg_id", "episode_sequence"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# EXPECTED REFERENCE CONTRACTS
# --------------------------------------------------------------------------------------------------

_ALLOWED_INTERACTION_CLASSES = {
    "BOUNDARY_ONLY",
    "INTERIOR_PENETRATION",
    "LOWER_BOUNDARY_REACHED",
    "UPPER_BOUNDARY_REACHED",
    "FULL_ZONE_CROSSING",
}

_EXPECTED_OBSERVATION_COUNT = len(
    _penetration_boundary_audit
)

_EXPECTED_FVG_COUNT = (
    _penetration_boundary_audit["fvg_id"]
    .nunique()
)

_EXPECTED_EPISODE_COUNT = (
    interaction_episode_registry_v1_1[
        "interaction_episode_id"
    ]
    .nunique()
)


# --------------------------------------------------------------------------------------------------
# 1. OBSERVATION CONTRACT
# --------------------------------------------------------------------------------------------------

_reg_invalid_class = (
    _reg_obs_v1[
        ~_reg_obs_v1["penetration_class"].isin(
            _ALLOWED_INTERACTION_CLASSES
        )
    ]
)

_reg_duplicate_observation_keys = (
    _reg_obs_v1
    .duplicated(
        subset=[
            "fvg_id",
            "datetime",
        ],
        keep=False,
    )
)

print("\nOBSERVATION CONTRACT")
print("-" * 100)

print(
    f"Observation rows : "
    f"{len(_reg_obs_v1):,}"
)

print(
    f"Invalid interaction classifications : "
    f"{len(_reg_invalid_class):,}"
)

print(
    f"Duplicate FVG/Datetime observations : "
    f"{_reg_duplicate_observation_keys.sum():,}"
)


# --------------------------------------------------------------------------------------------------
# 2. INTERACTION CLASS PRESERVATION
# --------------------------------------------------------------------------------------------------

_reg_class_counts = (
    _reg_obs_v1["penetration_class"]
    .value_counts()
    .sort_index()
)

print("\nINTERACTION CLASS DISTRIBUTION")
print("-" * 100)

print(
    _reg_class_counts.to_string()
)


# --------------------------------------------------------------------------------------------------
# 3. EPISODE RECONSTRUCTION
# --------------------------------------------------------------------------------------------------

_reg_obs_v1["interaction_gap_minutes"] = (
    _reg_obs_v1
    .groupby("fvg_id")["datetime"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

_reg_obs_v1["_episode_break"] = (
    _reg_obs_v1["interaction_gap_minutes"].isna()
    |
    (
        _reg_obs_v1["interaction_gap_minutes"]
        !=
        5.0
    )
)

_reg_obs_v1["episode_sequence"] = (
    _reg_obs_v1
    .groupby("fvg_id")["_episode_break"]
    .cumsum()
    .astype(int)
)

_reg_obs_v1["interaction_episode_id"] = (
    _reg_obs_v1["fvg_id"].astype(str)
    + "_EP_"
    + _reg_obs_v1["episode_sequence"]
    .astype(str)
    .str.zfill(3)
)


# --------------------------------------------------------------------------------------------------
# 4. EPISODE IDENTITY
# --------------------------------------------------------------------------------------------------

_reg_observation_episode_ids = set(
    _reg_obs_v1[
        "interaction_episode_id"
    ].unique()
)

_reg_registry_episode_ids = set(
    _reg_registry_v1[
        "interaction_episode_id"
    ].unique()
)

_reg_missing_episode_ids = (
    _reg_observation_episode_ids
    -
    _reg_registry_episode_ids
)

_reg_extra_episode_ids = (
    _reg_registry_episode_ids
    -
    _reg_observation_episode_ids
)

_reg_duplicate_episode_ids = (
    _reg_registry_v1
    .duplicated(
        subset=["interaction_episode_id"],
        keep=False,
    )
)

print("\nEPISODE IDENTITY")
print("-" * 100)

print(
    f"Reconstructed Episodes : "
    f"{len(_reg_observation_episode_ids):,}"
)

print(
    f"Registry Episodes : "
    f"{len(_reg_registry_episode_ids):,}"
)

print(
    f"Missing Registry Episode IDs : "
    f"{len(_reg_missing_episode_ids):,}"
)

print(
    f"Extra Registry Episode IDs : "
    f"{len(_reg_extra_episode_ids):,}"
)

print(
    f"Duplicate Registry Episode IDs : "
    f"{_reg_duplicate_episode_ids.sum():,}"
)


# --------------------------------------------------------------------------------------------------
# 5. WITHIN-EPISODE CONTINUITY
# --------------------------------------------------------------------------------------------------

_reg_within_episode = (
    _reg_obs_v1[
        _reg_obs_v1["interaction_episode_id"]
        .notna()
    ]
    .copy()
)

_reg_within_episode["next_gap_minutes"] = (
    _reg_within_episode
    .groupby("interaction_episode_id")["datetime"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

_reg_invalid_within_gaps = (
    _reg_within_episode[
        _reg_within_episode["next_gap_minutes"].notna()
        &
        (
            _reg_within_episode["next_gap_minutes"]
            !=
            5.0
        )
    ]
)

print("\nWITHIN-EPISODE CONTINUITY")
print("-" * 100)

print(
    f"Invalid within-Episode gaps : "
    f"{len(_reg_invalid_within_gaps):,}"
)


# --------------------------------------------------------------------------------------------------
# 6. EPISODE BOUNDARIES + AGGREGATION
# --------------------------------------------------------------------------------------------------

_reg_actual_episode_summary = (
    _reg_obs_v1
    .groupby(
        [
            "fvg_id",
            "episode_sequence",
            "interaction_episode_id",
        ],
        sort=True,
    )
    .agg(
        actual_start=(
            "datetime",
            "min",
        ),
        actual_end=(
            "datetime",
            "max",
        ),
        actual_count=(
            "datetime",
            "size",
        ),
        actual_max_penetration=(
            "penetration",
            "max",
        ),
        actual_max_penetration_ratio=(
            "penetration_ratio",
            "max",
        ),
    )
    .reset_index()
)


_reg_boundary_check = (
    _reg_registry_v1
    .merge(
        _reg_actual_episode_summary,
        on=[
            "fvg_id",
            "episode_sequence",
            "interaction_episode_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

_reg_invalid_start = (
    _reg_boundary_check[
        _reg_boundary_check[
            "interaction_episode_start"
        ]
        !=
        _reg_boundary_check[
            "actual_start"
        ]
    ]
)

_reg_invalid_end = (
    _reg_boundary_check[
        _reg_boundary_check[
            "interaction_episode_end"
        ]
        !=
        _reg_boundary_check[
            "actual_end"
        ]
    ]
)

_reg_invalid_count = (
    _reg_boundary_check[
        _reg_boundary_check[
            "interaction_observation_count"
        ]
        !=
        _reg_boundary_check[
            "actual_count"
        ]
    ]
)

_reg_invalid_max_penetration = (
    _reg_boundary_check[
        _reg_boundary_check[
            "max_penetration"
        ]
        !=
        _reg_boundary_check[
            "actual_max_penetration"
        ]
    ]
)

_reg_invalid_max_penetration_ratio = (
    _reg_boundary_check[
        _reg_boundary_check[
            "max_penetration_ratio"
        ]
        !=
        _reg_boundary_check[
            "actual_max_penetration_ratio"
        ]
    ]
)

print("\nEPISODE BOUNDARY / AGGREGATION")
print("-" * 100)

print(
    f"Invalid start boundaries : "
    f"{len(_reg_invalid_start):,}"
)

print(
    f"Invalid end boundaries : "
    f"{len(_reg_invalid_end):,}"
)

print(
    f"Invalid observation counts : "
    f"{len(_reg_invalid_count):,}"
)

print(
    f"Invalid max penetration : "
    f"{len(_reg_invalid_max_penetration):,}"
)

print(
    f"Invalid max penetration ratios : "
    f"{len(_reg_invalid_max_penetration_ratio):,}"
)


# --------------------------------------------------------------------------------------------------
# 7. FVG OWNERSHIP
# --------------------------------------------------------------------------------------------------

_reg_episode_fvg_cardinality = (
    _reg_registry_v1
    .groupby("interaction_episode_id")[
        "fvg_id"
    ]
    .nunique()
)

_reg_cross_fvg_registry = (
    _reg_episode_fvg_cardinality[
        _reg_episode_fvg_cardinality != 1
    ]
)

_reg_obs_episode_fvg_cardinality = (
    _reg_obs_v1
    .groupby("interaction_episode_id")[
        "fvg_id"
    ]
    .nunique()
)

_reg_cross_fvg_observation = (
    _reg_obs_episode_fvg_cardinality[
        _reg_obs_episode_fvg_cardinality != 1
    ]
)

print("\nFVG OWNERSHIP")
print("-" * 100)

print(
    f"Registry Episodes linked to multiple FVGs : "
    f"{len(_reg_cross_fvg_registry):,}"
)

print(
    f"Observation Episodes linked to multiple FVGs : "
    f"{len(_reg_cross_fvg_observation):,}"
)


# --------------------------------------------------------------------------------------------------
# 8. FVG → EPISODE SEQUENCE
# --------------------------------------------------------------------------------------------------

_reg_fvg_sequence = (
    _reg_registry_v1
    .groupby("fvg_id")
    .agg(
        episode_count=(
            "interaction_episode_id",
            "size",
        ),
        min_sequence=(
            "episode_sequence",
            "min",
        ),
        max_sequence=(
            "episode_sequence",
            "max",
        ),
        unique_sequences=(
            "episode_sequence",
            "nunique",
        ),
    )
)

_reg_invalid_sequences = (
    _reg_fvg_sequence[
        (_reg_fvg_sequence["min_sequence"] != 1)
        |
        (
            _reg_fvg_sequence["max_sequence"]
            !=
            _reg_fvg_sequence["episode_count"]
        )
        |
        (
            _reg_fvg_sequence["unique_sequences"]
            !=
            _reg_fvg_sequence["episode_count"]
        )
    ]
)

print("\nFVG → EPISODE SEQUENCE")
print("-" * 100)

print(
    f"FVGs with invalid Episode sequences : "
    f"{len(_reg_invalid_sequences):,}"
)


# --------------------------------------------------------------------------------------------------
# 9. RE-ENTRY SEPARATION
# --------------------------------------------------------------------------------------------------

_reg_transition = (
    _reg_registry_v1
    .copy()
)

_reg_transition["previous_end"] = (
    _reg_transition
    .groupby("fvg_id")[
        "interaction_episode_end"
    ]
    .shift(1)
)

_reg_transition["reentry_gap_minutes"] = (
    (
        _reg_transition["interaction_episode_start"]
        -
        _reg_transition["previous_end"]
    )
    .dt.total_seconds()
    .div(60)
)

_reg_transition_rows = (
    _reg_transition[
        _reg_transition["previous_end"].notna()
    ]
)

_reg_invalid_reentry = (
    _reg_transition_rows[
        _reg_transition_rows["reentry_gap_minutes"]
        <=
        5.0
    ]
)

print("\nRE-ENTRY SEPARATION")
print("-" * 100)

print(
    f"Episode transitions : "
    f"{len(_reg_transition_rows):,}"
)

print(
    f"Invalid re-entry separations : "
    f"{len(_reg_invalid_reentry):,}"
)


# --------------------------------------------------------------------------------------------------
# 10. BOUNDARY-ONLY PRESERVATION
# --------------------------------------------------------------------------------------------------

_reg_boundary_only_rows = (
    _reg_obs_v1[
        _reg_obs_v1["penetration_class"]
        ==
        "BOUNDARY_ONLY"
    ]
)

_reg_boundary_only_episode_count = (
    _reg_boundary_only_rows[
        "interaction_episode_id"
    ]
    .nunique()
)

print("\nBOUNDARY-ONLY PRESERVATION")
print("-" * 100)

print(
    f"BOUNDARY_ONLY observations : "
    f"{len(_reg_boundary_only_rows):,}"
)

print(
    f"BOUNDARY_ONLY Episodes represented : "
    f"{_reg_boundary_only_episode_count:,}"
)


# --------------------------------------------------------------------------------------------------
# 11. GLOBAL COUNTS
# --------------------------------------------------------------------------------------------------

_reg_global_count_gate = {
    "OBSERVATION_ROW_COUNT": (
        len(_reg_obs_v1)
        ==
        _EXPECTED_OBSERVATION_COUNT
    ),
    "FVG_COUNT": (
        _reg_obs_v1["fvg_id"].nunique()
        ==
        _EXPECTED_FVG_COUNT
    ),
    "EPISODE_COUNT": (
        _reg_registry_v1[
            "interaction_episode_id"
        ].nunique()
        ==
        _EXPECTED_EPISODE_COUNT
    ),
}

print("\nGLOBAL COUNTS")
print("-" * 100)

print(
    f"Expected observations : "
    f"{_EXPECTED_OBSERVATION_COUNT:,}"
)

print(
    f"Actual observations : "
    f"{len(_reg_obs_v1):,}"
)

print(
    f"Expected FVGs : "
    f"{_EXPECTED_FVG_COUNT:,}"
)

print(
    f"Actual FVGs : "
    f"{_reg_obs_v1['fvg_id'].nunique():,}"
)

print(
    f"Expected Episodes : "
    f"{_EXPECTED_EPISODE_COUNT:,}"
)

print(
    f"Actual Episodes : "
    f"{_reg_registry_v1['interaction_episode_id'].nunique():,}"
)


# --------------------------------------------------------------------------------------------------
# FINAL REGRESSION GATE
# --------------------------------------------------------------------------------------------------

_episode_regression_gate_v1 = {
    "OBSERVATION_CONTRACT": (
        len(_reg_invalid_class) == 0
        and
        _reg_duplicate_observation_keys.sum() == 0
    ),

    "OBSERVATION_ROW_COUNT": (
        _reg_global_count_gate[
            "OBSERVATION_ROW_COUNT"
        ]
    ),

    "FVG_COUNT": (
        _reg_global_count_gate[
            "FVG_COUNT"
        ]
    ),

    "EPISODE_COUNT": (
        _reg_global_count_gate[
            "EPISODE_COUNT"
        ]
    ),

    "EPISODE_IDENTITY": (
        len(_reg_missing_episode_ids) == 0
        and
        len(_reg_extra_episode_ids) == 0
        and
        _reg_duplicate_episode_ids.sum() == 0
    ),

    "WITHIN_EPISODE_5M_CONTINUITY": (
        len(_reg_invalid_within_gaps) == 0
    ),

    "START_BOUNDARY_ALIGNMENT": (
        len(_reg_invalid_start) == 0
    ),

    "END_BOUNDARY_ALIGNMENT": (
        len(_reg_invalid_end) == 0
    ),

    "OBSERVATION_COUNT_ALIGNMENT": (
        len(_reg_invalid_count) == 0
    ),

    "MAX_PENETRATION_ALIGNMENT": (
        len(_reg_invalid_max_penetration) == 0
    ),

    "MAX_PENETRATION_RATIO_ALIGNMENT": (
        len(_reg_invalid_max_penetration_ratio) == 0
    ),

    "NO_CROSS_FVG_REGISTRY_CONTAMINATION": (
        len(_reg_cross_fvg_registry) == 0
    ),

    "NO_CROSS_FVG_OBSERVATION_CONTAMINATION": (
        len(_reg_cross_fvg_observation) == 0
    ),

    "FVG_SEQUENCE_CONTRACT": (
        len(_reg_invalid_sequences) == 0
    ),

    "REENTRY_SEPARATION": (
        len(_reg_invalid_reentry) == 0
    ),

    "BOUNDARY_ONLY_PRESERVATION": (
        len(_reg_boundary_only_rows) == 300
    ),
}


print("\n" + "=" * 100)
print("STEP 7.7 — FULL INTERACTION EPISODE REGRESSION GATE")
print("=" * 100)

for _gate_name, _gate_value in _episode_regression_gate_v1.items():
    print(
        f"{_gate_name:<45}: "
        f"{'PASSED' if _gate_value else 'FAILED'}"
    )

print("\n" + "=" * 100)

if all(_episode_regression_gate_v1.values()):
    print(
        "STEP 7.7 FULL INTERACTION EPISODE REGRESSION : PASSED"
    )
else:
    print(
        "STEP 7.7 FULL INTERACTION EPISODE REGRESSION : FAILED"
    )

print("=" * 100)

In [ ]:
# =============================================================================
# STEP 8.1-A
# EPISODE INTENSITY / GEOMETRY AGGREGATION V1
# CONSTRUCTOR — CORRECTED
# =============================================================================

import pandas as pd
import numpy as np


# =============================================================================
# CONTRACT
# =============================================================================

EPISODE_CHARACTERIZATION_V1_ALLOWED_CLASSES = {
    "BOUNDARY_ONLY",
    "INTERIOR_PENETRATION",
    "LOWER_BOUNDARY_REACHED",
    "UPPER_BOUNDARY_REACHED",
    "FULL_ZONE_CROSSING",
}

EPISODE_CHARACTERIZATION_V1_REQUIRED_OBS_COLUMNS = {
    "fvg_id",
    "datetime",
    "penetration",
    "penetration_ratio",
    "penetration_class",
    "reaches_lower_boundary",
    "reaches_upper_boundary",
    "full_zone_crossed",
}

EPISODE_CHARACTERIZATION_V1_REQUIRED_REGISTRY_COLUMNS = {
    "fvg_id",
    "episode_sequence",
    "interaction_episode_id",
    "interaction_episode_start",
    "interaction_episode_end",
    "interaction_observation_count",
    "max_penetration",
    "max_penetration_ratio",
}


# =============================================================================
# INPUT VALIDATION
# =============================================================================

def _validate_episode_characterization_v1_inputs(
    penetration_boundary_audit,
    interaction_episode_registry,
):
    obs = penetration_boundary_audit
    registry = interaction_episode_registry

    missing_obs = (
        EPISODE_CHARACTERIZATION_V1_REQUIRED_OBS_COLUMNS
        - set(obs.columns)
    )

    missing_registry = (
        EPISODE_CHARACTERIZATION_V1_REQUIRED_REGISTRY_COLUMNS
        - set(registry.columns)
    )

    if missing_obs:
        raise ValueError(
            f"Missing observation columns: {sorted(missing_obs)}"
        )

    if missing_registry:
        raise ValueError(
            f"Missing registry columns: {sorted(missing_registry)}"
        )

    invalid_classes = (
        set(obs["penetration_class"].dropna().unique())
        - EPISODE_CHARACTERIZATION_V1_ALLOWED_CLASSES
    )

    if invalid_classes:
        raise ValueError(
            f"Invalid penetration classes detected: {sorted(invalid_classes)}"
        )


# =============================================================================
# CONSTRUCTOR
# =============================================================================

def build_episode_characterization_v1(
    penetration_boundary_audit,
    interaction_episode_registry,
):
    """
    Build one deterministic characterization row per interaction Episode.

    IMPORTANT CONTRACTS
    --------------------
    1. Episode segmentation is NOT redefined.
    2. Existing Step 7 segmentation contract is reconstructed:
       same FVG + exact 5-minute consecutive observations.
    3. Existing registry is authoritative for Episode identity.
    4. No reaction / mitigation / outcome logic is introduced.
    """

    obs = penetration_boundary_audit.copy()
    registry = interaction_episode_registry.copy()

    _validate_episode_characterization_v1_inputs(
        obs,
        registry,
    )

    # -------------------------------------------------------------------------
    # Datetime normalization
    # -------------------------------------------------------------------------

    obs["datetime"] = pd.to_datetime(
        obs["datetime"],
        utc=True,
        errors="raise",
    )

    registry["interaction_episode_start"] = pd.to_datetime(
        registry["interaction_episode_start"],
        utc=True,
        errors="raise",
    )

    registry["interaction_episode_end"] = pd.to_datetime(
        registry["interaction_episode_end"],
        utc=True,
        errors="raise",
    )

    # -------------------------------------------------------------------------
    # Deterministic ordering
    # -------------------------------------------------------------------------

    obs = (
        obs
        .sort_values(
            ["fvg_id", "datetime"],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    registry = (
        registry
        .sort_values(
            ["fvg_id", "episode_sequence"],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    # =========================================================================
    # RECONSTRUCT STEP 7 EPISODE MEMBERSHIP
    # =========================================================================

    obs["_interaction_gap_minutes"] = (
        obs
        .groupby("fvg_id")["datetime"]
        .diff()
        .dt.total_seconds()
        .div(60.0)
    )

    obs["_episode_break"] = (
        obs["_interaction_gap_minutes"].isna()
        |
        (obs["_interaction_gap_minutes"] != 5.0)
    )

    obs["_episode_sequence"] = (
        obs
        .groupby("fvg_id")["_episode_break"]
        .cumsum()
        .astype(int)
    )

    obs["interaction_episode_id"] = (
        obs["fvg_id"].astype(str)
        + "_EP_"
        + obs["_episode_sequence"].astype(str).str.zfill(3)
    )

    # =========================================================================
    # AUTHORITATIVE REGISTRY IDENTITY CHECK
    # =========================================================================

    reconstructed_ids = set(
        obs["interaction_episode_id"]
    )

    registry_ids = set(
        registry["interaction_episode_id"]
    )

    missing_registry_ids = (
        reconstructed_ids - registry_ids
    )

    extra_registry_ids = (
        registry_ids - reconstructed_ids
    )

    if missing_registry_ids:
        raise ValueError(
            "Reconstructed Episode IDs missing from registry: "
            f"{sorted(missing_registry_ids)[:10]}"
        )

    if extra_registry_ids:
        raise ValueError(
            "Registry contains Episode IDs not reconstructed from "
            f"observation contract: {sorted(extra_registry_ids)[:10]}"
        )

    # =========================================================================
    # REGISTRY LOOKUP
    # =========================================================================

    registry_lookup = (
        registry[
            [
                "interaction_episode_id",
                "fvg_id",
                "episode_sequence",
                "interaction_episode_start",
                "interaction_episode_end",
                "interaction_observation_count",
                "max_penetration",
                "max_penetration_ratio",
            ]
        ]
        .copy()
    )

    # Registry identity must be unique
    if registry_lookup["interaction_episode_id"].duplicated().any():
        raise ValueError(
            "Duplicate interaction_episode_id detected in registry."
        )

    # -------------------------------------------------------------------------
    # Merge authoritative registry identity into observations
    # -------------------------------------------------------------------------

    obs = obs.merge(
        registry_lookup,
        on="interaction_episode_id",
        how="left",
        suffixes=("", "_registry"),
        validate="many_to_one",
    )

    # -------------------------------------------------------------------------
    # Cross-check FVG ownership
    # -------------------------------------------------------------------------

    fvg_mismatch = (
        obs["fvg_id"] != obs["fvg_id_registry"]
    )

    if fvg_mismatch.any():
        raise ValueError(
            "Observation / registry FVG ownership mismatch detected."
        )

    # =========================================================================
    # EPISODE-LEVEL AGGREGATION
    # =========================================================================

    rows = []

    for episode_id, group in obs.groupby(
        "interaction_episode_id",
        sort=False,
    ):
        group = (
            group
            .sort_values(
                "datetime",
                kind="mergesort",
            )
            .reset_index(drop=True)
        )

        first_row = group.iloc[0]
        last_row = group.iloc[-1]

        # ---------------------------------------------------------------------
        # Strongest penetration
        #
        # Deterministic ordering:
        #   1. highest penetration
        #   2. highest penetration_ratio
        #   3. earliest datetime
        # ---------------------------------------------------------------------

        strongest = (
            group
            .sort_values(
                [
                    "penetration",
                    "penetration_ratio",
                    "datetime",
                ],
                ascending=[
                    False,
                    False,
                    True,
                ],
                kind="mergesort",
            )
            .iloc[0]
        )

        # ---------------------------------------------------------------------
        # Episode boundaries
        # ---------------------------------------------------------------------

        episode_start = group["datetime"].iloc[0]
        episode_end = group["datetime"].iloc[-1]

        duration_minutes = (
            episode_end - episode_start
        ).total_seconds() / 60.0

        # ---------------------------------------------------------------------
        # Geometry flags
        # ---------------------------------------------------------------------

        boundary_only = bool(
            set(
                group["penetration_class"].unique()
            )
            == {"BOUNDARY_ONLY"}
        )

        reached_lower_boundary = bool(
            group["reaches_lower_boundary"]
            .fillna(False)
            .astype(bool)
            .any()
        )

        reached_upper_boundary = bool(
            group["reaches_upper_boundary"]
            .fillna(False)
            .astype(bool)
            .any()
        )

        full_zone_crossed = bool(
            group["full_zone_crossed"]
            .fillna(False)
            .astype(bool)
            .any()
        )

        # ---------------------------------------------------------------------
        # Penetration path
        #
        # Pure observation history.
        # No interpretation.
        # ---------------------------------------------------------------------

        penetration_path = (
            group[
                [
                    "datetime",
                    "penetration",
                    "penetration_ratio",
                    "penetration_class",
                ]
            ]
            .to_dict(orient="records")
        )

        # ---------------------------------------------------------------------
        # Episode record
        # ---------------------------------------------------------------------

        rows.append(
            {
                "interaction_episode_id": episode_id,

                "fvg_id": first_row["fvg_id"],

                "episode_sequence": int(
                    first_row["episode_sequence"]
                ),

                "interaction_episode_start": episode_start,

                "interaction_episode_end": episode_end,

                "episode_duration_minutes": (
                    float(duration_minutes)
                ),

                "interaction_observation_count": int(
                    len(group)
                ),

                "first_interaction_class": (
                    first_row["penetration_class"]
                ),

                "last_interaction_class": (
                    last_row["penetration_class"]
                ),

                "strongest_penetration": float(
                    strongest["penetration"]
                ),

                "strongest_penetration_ratio": float(
                    strongest["penetration_ratio"]
                ),

                "strongest_penetration_datetime": (
                    strongest["datetime"]
                ),

                "boundary_only": boundary_only,

                "reached_lower_boundary": (
                    reached_lower_boundary
                ),

                "reached_upper_boundary": (
                    reached_upper_boundary
                ),

                "full_zone_crossed": (
                    full_zone_crossed
                ),

                "penetration_path": penetration_path,
            }
        )

    # =========================================================================
    # FINAL DATAFRAME
    # =========================================================================

    result = pd.DataFrame(rows)

    result = result[
        [
            "interaction_episode_id",
            "fvg_id",
            "episode_sequence",

            "interaction_episode_start",
            "interaction_episode_end",
            "episode_duration_minutes",

            "interaction_observation_count",

            "first_interaction_class",
            "last_interaction_class",

            "strongest_penetration",
            "strongest_penetration_ratio",
            "strongest_penetration_datetime",

            "boundary_only",
            "reached_lower_boundary",
            "reached_upper_boundary",
            "full_zone_crossed",

            "penetration_path",
        ]
    ]

    result = (
        result
        .sort_values(
            ["fvg_id", "episode_sequence"],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    # =========================================================================
    # CLEAN INTERNAL COLUMNS
    # =========================================================================

    obs.drop(
        columns=[
            "_interaction_gap_minutes",
            "_episode_break",
            "_episode_sequence",
        ],
        inplace=True,
        errors="ignore",
    )

    return result


# =============================================================================
# BUILD
# =============================================================================

episode_characterization_v1 = build_episode_characterization_v1(
    penetration_boundary_audit=_penetration_boundary_audit,
    interaction_episode_registry=interaction_episode_registry_v1_1,
)


# =============================================================================
# BASIC OUTPUT
# =============================================================================

print("=" * 100)
print("EPISODE INTENSITY / GEOMETRY AGGREGATION V1")
print("=" * 100)

print(
    f"Episode rows              : "
    f"{len(episode_characterization_v1):,}"
)

print(
    f"Unique Episode IDs        : "
    f"{episode_characterization_v1['interaction_episode_id'].nunique():,}"
)

print(
    f"Unique FVGs               : "
    f"{episode_characterization_v1['fvg_id'].nunique():,}"
)

print("\nSCHEMA")
print("-" * 100)

print(
    episode_characterization_v1.columns.tolist()
)

print("\nFIRST 10 EPISODES")
print("-" * 100)

print(
    episode_characterization_v1[
        [
            "interaction_episode_id",
            "fvg_id",
            "episode_sequence",
            "interaction_episode_start",
            "interaction_episode_end",
            "episode_duration_minutes",
            "interaction_observation_count",
            "first_interaction_class",
            "last_interaction_class",
            "strongest_penetration",
            "strongest_penetration_ratio",
            "strongest_penetration_datetime",
            "boundary_only",
            "reached_lower_boundary",
            "reached_upper_boundary",
            "full_zone_crossed",
        ]
    ]
    .head(10)
    .to_string(index=False)
)

In [ ]:
# =============================================================================
# STEP 8.1-B
# EPISODE CHARACTERIZATION AGGREGATION REGRESSION GATE V1
# =============================================================================

import pandas as pd
import numpy as np


print("=" * 100)
print("EPISODE CHARACTERIZATION AGGREGATION REGRESSION GATE V1")
print("=" * 100)


# =============================================================================
# REFERENCES
# =============================================================================

obs = _penetration_boundary_audit.copy()
registry = interaction_episode_registry_v1_1.copy()
characterization = episode_characterization_v1.copy()


# =============================================================================
# DATETIME NORMALIZATION
# =============================================================================

obs["datetime"] = pd.to_datetime(
    obs["datetime"],
    utc=True,
    errors="raise",
)

registry["interaction_episode_start"] = pd.to_datetime(
    registry["interaction_episode_start"],
    utc=True,
    errors="raise",
)

registry["interaction_episode_end"] = pd.to_datetime(
    registry["interaction_episode_end"],
    utc=True,
    errors="raise",
)

characterization["interaction_episode_start"] = pd.to_datetime(
    characterization["interaction_episode_start"],
    utc=True,
    errors="raise",
)

characterization["interaction_episode_end"] = pd.to_datetime(
    characterization["interaction_episode_end"],
    utc=True,
    errors="raise",
)

characterization["strongest_penetration_datetime"] = pd.to_datetime(
    characterization["strongest_penetration_datetime"],
    utc=True,
    errors="raise",
)


# =============================================================================
# GATE 1 — EPISODE ROW COUNT
# =============================================================================

expected_episode_count = len(registry)
actual_episode_count = len(characterization)

gate_episode_row_count = (
    expected_episode_count == actual_episode_count
)

print("\nEPISODE COUNT")
print("-" * 100)
print(f"Registry Episodes : {expected_episode_count:,}")
print(f"Characterization : {actual_episode_count:,}")
print(
    f"EPISODE ROW COUNT : "
    f"{'PASSED' if gate_episode_row_count else 'FAILED'}"
)


# =============================================================================
# GATE 2 — EPISODE ID UNIQUENESS
# =============================================================================

duplicate_characterization_ids = (
    characterization["interaction_episode_id"]
    .duplicated()
    .sum()
)

gate_episode_id_uniqueness = (
    duplicate_characterization_ids == 0
)

print("\nEPISODE ID UNIQUENESS")
print("-" * 100)
print(
    f"Duplicate Characterization IDs : "
    f"{duplicate_characterization_ids}"
)
print(
    f"EPISODE ID UNIQUENESS : "
    f"{'PASSED' if gate_episode_id_uniqueness else 'FAILED'}"
)


# =============================================================================
# GATE 3 — EPISODE ID PRESERVATION
# =============================================================================

registry_ids = set(
    registry["interaction_episode_id"]
)

characterization_ids = set(
    characterization["interaction_episode_id"]
)

missing_characterization_ids = (
    registry_ids - characterization_ids
)

extra_characterization_ids = (
    characterization_ids - registry_ids
)

gate_episode_id_preservation = (
    len(missing_characterization_ids) == 0
    and
    len(extra_characterization_ids) == 0
)

print("\nEPISODE ID PRESERVATION")
print("-" * 100)
print(
    f"Missing Characterization IDs : "
    f"{len(missing_characterization_ids)}"
)
print(
    f"Extra Characterization IDs   : "
    f"{len(extra_characterization_ids)}"
)
print(
    f"EPISODE ID PRESERVATION : "
    f"{'PASSED' if gate_episode_id_preservation else 'FAILED'}"
)


# =============================================================================
# GATE 4 — FVG OWNERSHIP
# =============================================================================

registry_fvg = (
    registry
    .set_index("interaction_episode_id")["fvg_id"]
)

characterization_fvg = (
    characterization
    .set_index("interaction_episode_id")["fvg_id"]
)

common_ids = (
    registry_fvg.index
    .intersection(characterization_fvg.index)
)

fvg_ownership_mismatch = (
    registry_fvg.loc[common_ids]
    != characterization_fvg.loc[common_ids]
).sum()

gate_fvg_ownership = (
    fvg_ownership_mismatch == 0
)

print("\nFVG OWNERSHIP")
print("-" * 100)
print(
    f"FVG Ownership Mismatches : "
    f"{fvg_ownership_mismatch}"
)
print(
    f"FVG OWNERSHIP : "
    f"{'PASSED' if gate_fvg_ownership else 'FAILED'}"
)


# =============================================================================
# GATE 5 — EPISODE SEQUENCE ALIGNMENT
# =============================================================================

registry_sequence = (
    registry
    .set_index("interaction_episode_id")["episode_sequence"]
)

characterization_sequence = (
    characterization
    .set_index("interaction_episode_id")["episode_sequence"]
)

sequence_mismatch = (
    registry_sequence.loc[common_ids]
    != characterization_sequence.loc[common_ids]
).sum()

gate_sequence_alignment = (
    sequence_mismatch == 0
)

print("\nEPISODE SEQUENCE")
print("-" * 100)
print(
    f"Sequence Mismatches : {sequence_mismatch}"
)
print(
    f"EPISODE SEQUENCE ALIGNMENT : "
    f"{'PASSED' if gate_sequence_alignment else 'FAILED'}"
)


# =============================================================================
# GATE 6 — START BOUNDARY ALIGNMENT
# =============================================================================

registry_start = (
    registry
    .set_index("interaction_episode_id")[
        "interaction_episode_start"
    ]
)

characterization_start = (
    characterization
    .set_index("interaction_episode_id")[
        "interaction_episode_start"
    ]
)

start_mismatch = (
    registry_start.loc[common_ids]
    != characterization_start.loc[common_ids]
).sum()

gate_start_alignment = (
    start_mismatch == 0
)

print("\nSTART BOUNDARY")
print("-" * 100)
print(
    f"Start Boundary Mismatches : {start_mismatch}"
)
print(
    f"START BOUNDARY ALIGNMENT : "
    f"{'PASSED' if gate_start_alignment else 'FAILED'}"
)


# =============================================================================
# GATE 7 — END BOUNDARY ALIGNMENT
# =============================================================================

registry_end = (
    registry
    .set_index("interaction_episode_id")[
        "interaction_episode_end"
    ]
)

characterization_end = (
    characterization
    .set_index("interaction_episode_id")[
        "interaction_episode_end"
    ]
)

end_mismatch = (
    registry_end.loc[common_ids]
    != characterization_end.loc[common_ids]
).sum()

gate_end_alignment = (
    end_mismatch == 0
)

print("\nEND BOUNDARY")
print("-" * 100)
print(
    f"End Boundary Mismatches : {end_mismatch}"
)
print(
    f"END BOUNDARY ALIGNMENT : "
    f"{'PASSED' if gate_end_alignment else 'FAILED'}"
)


# =============================================================================
# GATE 8 — OBSERVATION COUNT ALIGNMENT
# =============================================================================

registry_count = (
    registry
    .set_index("interaction_episode_id")[
        "interaction_observation_count"
    ]
)

characterization_count = (
    characterization
    .set_index("interaction_episode_id")[
        "interaction_observation_count"
    ]
)

count_mismatch = (
    registry_count.loc[common_ids]
    != characterization_count.loc[common_ids]
).sum()

gate_observation_count = (
    count_mismatch == 0
)

print("\nOBSERVATION COUNT")
print("-" * 100)
print(
    f"Observation Count Mismatches : {count_mismatch}"
)
print(
    f"OBSERVATION COUNT ALIGNMENT : "
    f"{'PASSED' if gate_observation_count else 'FAILED'}"
)


# =============================================================================
# GATE 9 — MAX PENETRATION ALIGNMENT
# =============================================================================

registry_max_pen = (
    registry
    .set_index("interaction_episode_id")[
        "max_penetration"
    ]
)

characterization_max_pen = (
    characterization
    .set_index("interaction_episode_id")[
        "strongest_penetration"
    ]
)

max_pen_mismatch = (
    ~np.isclose(
        registry_max_pen.loc[common_ids].astype(float),
        characterization_max_pen.loc[common_ids].astype(float),
        rtol=0.0,
        atol=1e-12,
    )
).sum()

gate_max_penetration = (
    max_pen_mismatch == 0
)

print("\nMAX PENETRATION")
print("-" * 100)
print(
    f"Max Penetration Mismatches : "
    f"{max_pen_mismatch}"
)
print(
    f"MAX PENETRATION ALIGNMENT : "
    f"{'PASSED' if gate_max_penetration else 'FAILED'}"
)


# =============================================================================
# GATE 10 — MAX PENETRATION RATIO ALIGNMENT
# =============================================================================

registry_max_ratio = (
    registry
    .set_index("interaction_episode_id")[
        "max_penetration_ratio"
    ]
)

characterization_max_ratio = (
    characterization
    .set_index("interaction_episode_id")[
        "strongest_penetration_ratio"
    ]
)

max_ratio_mismatch = (
    ~np.isclose(
        registry_max_ratio.loc[common_ids].astype(float),
        characterization_max_ratio.loc[common_ids].astype(float),
        rtol=0.0,
        atol=1e-12,
    )
).sum()

gate_max_ratio = (
    max_ratio_mismatch == 0
)

print("\nMAX PENETRATION RATIO")
print("-" * 100)
print(
    f"Max Penetration Ratio Mismatches : "
    f"{max_ratio_mismatch}"
)
print(
    f"MAX PENETRATION RATIO ALIGNMENT : "
    f"{'PASSED' if gate_max_ratio else 'FAILED'}"
)


# =============================================================================
# GATE 11 — DURATION ALIGNMENT
# =============================================================================

registry_duration = (
    (
        registry_end.loc[common_ids]
        -
        registry_start.loc[common_ids]
    )
    .dt.total_seconds()
    .div(60.0)
)

characterization_duration = (
    characterization
    .set_index("interaction_episode_id")[
        "episode_duration_minutes"
    ]
    .loc[common_ids]
)

duration_mismatch = (
    ~np.isclose(
        registry_duration.astype(float),
        characterization_duration.astype(float),
        rtol=0.0,
        atol=1e-12,
    )
).sum()

gate_duration_alignment = (
    duration_mismatch == 0
)

print("\nDURATION")
print("-" * 100)
print(
    f"Duration Mismatches : {duration_mismatch}"
)
print(
    f"DURATION ALIGNMENT : "
    f"{'PASSED' if gate_duration_alignment else 'FAILED'}"
)


# =============================================================================
# GATE 12 — FIRST INTERACTION CLASS
# =============================================================================

reconstructed_first_class = (
    obs
    .sort_values(
        ["fvg_id", "datetime"],
        kind="mergesort",
    )
    .assign(
        _gap=lambda x: (
            x.groupby("fvg_id")["datetime"]
            .diff()
            .dt.total_seconds()
            .div(60.0)
        )
    )
)

reconstructed_first_class["_break"] = (
    reconstructed_first_class["_gap"].isna()
    |
    (reconstructed_first_class["_gap"] != 5.0)
)

reconstructed_first_class["_sequence"] = (
    reconstructed_first_class
    .groupby("fvg_id")["_break"]
    .cumsum()
    .astype(int)
)

reconstructed_first_class["interaction_episode_id"] = (
    reconstructed_first_class["fvg_id"].astype(str)
    + "_EP_"
    + reconstructed_first_class["_sequence"]
    .astype(str)
    .str.zfill(3)
)

first_class_reference = (
    reconstructed_first_class
    .sort_values(
        ["interaction_episode_id", "datetime"],
        kind="mergesort",
    )
    .groupby("interaction_episode_id")[
        "penetration_class"
    ]
    .first()
)

characterization_first_class = (
    characterization
    .set_index("interaction_episode_id")[
        "first_interaction_class"
    ]
)

first_class_mismatch = (
    first_class_reference.loc[common_ids]
    != characterization_first_class.loc[common_ids]
).sum()

gate_first_class = (
    first_class_mismatch == 0
)

print("\nFIRST INTERACTION CLASS")
print("-" * 100)
print(
    f"First Class Mismatches : {first_class_mismatch}"
)
print(
    f"FIRST INTERACTION CLASS : "
    f"{'PASSED' if gate_first_class else 'FAILED'}"
)


# =============================================================================
# GATE 13 — LAST INTERACTION CLASS
# =============================================================================

last_class_reference = (
    reconstructed_first_class
    .sort_values(
        ["interaction_episode_id", "datetime"],
        kind="mergesort",
    )
    .groupby("interaction_episode_id")[
        "penetration_class"
    ]
    .last()
)

characterization_last_class = (
    characterization
    .set_index("interaction_episode_id")[
        "last_interaction_class"
    ]
)

last_class_mismatch = (
    last_class_reference.loc[common_ids]
    != characterization_last_class.loc[common_ids]
).sum()

gate_last_class = (
    last_class_mismatch == 0
)

print("\nLAST INTERACTION CLASS")
print("-" * 100)
print(
    f"Last Class Mismatches : {last_class_mismatch}"
)
print(
    f"LAST INTERACTION CLASS : "
    f"{'PASSED' if gate_last_class else 'FAILED'}"
)


# =============================================================================
# GATE 14 — STRONGEST PENETRATION TIME
# =============================================================================

strongest_reference = (
    reconstructed_first_class
    .sort_values(
        [
            "interaction_episode_id",
            "penetration",
            "penetration_ratio",
            "datetime",
        ],
        ascending=[
            True,
            False,
            False,
            True,
        ],
        kind="mergesort",
    )
    .groupby("interaction_episode_id")
    .first()
)

reference_strongest_time = (
    strongest_reference[
        "datetime"
    ]
)

characterization_strongest_time = (
    characterization
    .set_index("interaction_episode_id")[
        "strongest_penetration_datetime"
    ]
)

strongest_time_mismatch = (
    reference_strongest_time.loc[common_ids]
    != characterization_strongest_time.loc[common_ids]
).sum()

gate_strongest_time = (
    strongest_time_mismatch == 0
)

print("\nSTRONGEST PENETRATION TIME")
print("-" * 100)
print(
    f"Strongest Time Mismatches : "
    f"{strongest_time_mismatch}"
)
print(
    f"STRONGEST PENETRATION TIME : "
    f"{'PASSED' if gate_strongest_time else 'FAILED'}"
)


# =============================================================================
# GATE 15 — BOUNDARY-ONLY FLAG
# =============================================================================

boundary_only_reference = (
    reconstructed_first_class
    .groupby("interaction_episode_id")[
        "penetration_class"
    ]
    .apply(
        lambda x: set(x.unique())
        == {"BOUNDARY_ONLY"}
    )
)

characterization_boundary_only = (
    characterization
    .set_index("interaction_episode_id")[
        "boundary_only"
    ]
)

boundary_only_mismatch = (
    boundary_only_reference.loc[common_ids].astype(bool)
    != characterization_boundary_only.loc[common_ids].astype(bool)
).sum()

gate_boundary_only = (
    boundary_only_mismatch == 0
)

print("\nBOUNDARY-ONLY")
print("-" * 100)
print(
    f"Boundary-Only Flag Mismatches : "
    f"{boundary_only_mismatch}"
)
print(
    f"BOUNDARY-ONLY FLAG : "
    f"{'PASSED' if gate_boundary_only else 'FAILED'}"
)


# =============================================================================
# GATE 16 — LOWER / UPPER / FULL-ZONE FLAGS
# =============================================================================

lower_reference = (
    reconstructed_first_class
    .groupby("interaction_episode_id")[
        "reaches_lower_boundary"
    ]
    .any()
)

upper_reference = (
    reconstructed_first_class
    .groupby("interaction_episode_id")[
        "reaches_upper_boundary"
    ]
    .any()
)

full_reference = (
    reconstructed_first_class
    .groupby("interaction_episode_id")[
        "full_zone_crossed"
    ]
    .any()
)

characterization_indexed = (
    characterization
    .set_index("interaction_episode_id")
)

lower_mismatch = (
    lower_reference.loc[common_ids].astype(bool)
    != characterization_indexed.loc[
        common_ids,
        "reached_lower_boundary"
    ].astype(bool)
).sum()

upper_mismatch = (
    upper_reference.loc[common_ids].astype(bool)
    != characterization_indexed.loc[
        common_ids,
        "reached_upper_boundary"
    ].astype(bool)
).sum()

full_mismatch = (
    full_reference.loc[common_ids].astype(bool)
    != characterization_indexed.loc[
        common_ids,
        "full_zone_crossed"
    ].astype(bool)
).sum()

gate_geometry_flags = (
    lower_mismatch == 0
    and
    upper_mismatch == 0
    and
    full_mismatch == 0
)

print("\nGEOMETRY FLAGS")
print("-" * 100)
print(
    f"Lower Boundary Mismatches : {lower_mismatch}"
)
print(
    f"Upper Boundary Mismatches : {upper_mismatch}"
)
print(
    f"Full Zone Mismatches      : {full_mismatch}"
)
print(
    f"GEOMETRY FLAGS : "
    f"{'PASSED' if gate_geometry_flags else 'FAILED'}"
)


# =============================================================================
# GATE 17 — GLOBAL FVG COUNT
# =============================================================================

registry_fvg_count = (
    registry["fvg_id"].nunique()
)

characterization_fvg_count = (
    characterization["fvg_id"].nunique()
)

gate_fvg_count = (
    registry_fvg_count == characterization_fvg_count
)

print("\nFVG COUNT")
print("-" * 100)
print(
    f"Registry FVGs          : {registry_fvg_count:,}"
)
print(
    f"Characterization FVGs  : {characterization_fvg_count:,}"
)
print(
    f"FVG COUNT : "
    f"{'PASSED' if gate_fvg_count else 'FAILED'}"
)


# =============================================================================
# FINAL REGRESSION RESULT
# =============================================================================

gates = {
    "EPISODE_ROW_COUNT": gate_episode_row_count,
    "EPISODE_ID_UNIQUENESS": gate_episode_id_uniqueness,
    "EPISODE_ID_PRESERVATION": gate_episode_id_preservation,
    "FVG_OWNERSHIP": gate_fvg_ownership,
    "EPISODE_SEQUENCE_ALIGNMENT": gate_sequence_alignment,
    "START_BOUNDARY_ALIGNMENT": gate_start_alignment,
    "END_BOUNDARY_ALIGNMENT": gate_end_alignment,
    "OBSERVATION_COUNT_ALIGNMENT": gate_observation_count,
    "MAX_PENETRATION_ALIGNMENT": gate_max_penetration,
    "MAX_PENETRATION_RATIO_ALIGNMENT": gate_max_ratio,
    "DURATION_ALIGNMENT": gate_duration_alignment,
    "FIRST_INTERACTION_CLASS": gate_first_class,
    "LAST_INTERACTION_CLASS": gate_last_class,
    "STRONGEST_PENETRATION_TIME": gate_strongest_time,
    "BOUNDARY_ONLY_FLAG": gate_boundary_only,
    "GEOMETRY_FLAGS": gate_geometry_flags,
    "FVG_COUNT": gate_fvg_count,
}


print("\n" + "=" * 100)
print("FINAL REGRESSION GATE")
print("=" * 100)

for gate_name, passed in gates.items():
    print(
        f"{gate_name:<45} : "
        f"{'PASSED' if passed else 'FAILED'}"
    )

print("=" * 100)

if all(gates.values()):
    print("OVERALL RESULT : PASSED")
else:
    failed = [
        name
        for name, passed in gates.items()
        if not passed
    ]

    print("OVERALL RESULT : FAILED")
    print(f"FAILED GATES  : {failed}")

In [ ]:
# =============================================================================
# STEP 8.1-C
# EPISODE INTENSITY TAXONOMY CONTRACT AUDIT V1
# =============================================================================

import pandas as pd
import numpy as np


print("=" * 100)
print("EPISODE INTENSITY TAXONOMY CONTRACT AUDIT V1")
print("=" * 100)


# =============================================================================
# INPUTS
# =============================================================================

obs = _penetration_boundary_audit.copy()
characterization = episode_characterization_v1.copy()


# =============================================================================
# CONTRACT VOCABULARY
# =============================================================================

ALLOWED_OBSERVATION_CLASSES = {
    "BOUNDARY_ONLY",
    "INTERIOR_PENETRATION",
    "LOWER_BOUNDARY_REACHED",
    "UPPER_BOUNDARY_REACHED",
    "FULL_ZONE_CROSSING",
}

ALLOWED_INTENSITY_CLASSES = {
    "BOUNDARY_ONLY",
    "INTERIOR_PENETRATION",
    "BOUNDARY_REACHED",
    "FULL_ZONE_CROSSING",
}


# =============================================================================
# DATETIME NORMALIZATION
# =============================================================================

obs["datetime"] = pd.to_datetime(
    obs["datetime"],
    utc=True,
    errors="raise",
)


# =============================================================================
# RECONSTRUCT EPISODE MEMBERSHIP
#
# Authoritative Step 7 segmentation:
# same FVG + exact 5-minute continuity.
# =============================================================================

obs = (
    obs
    .sort_values(
        ["fvg_id", "datetime"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

obs["_gap_minutes"] = (
    obs
    .groupby("fvg_id")["datetime"]
    .diff()
    .dt.total_seconds()
    .div(60.0)
)

obs["_episode_break"] = (
    obs["_gap_minutes"].isna()
    |
    (obs["_gap_minutes"] != 5.0)
)

obs["_episode_sequence"] = (
    obs
    .groupby("fvg_id")["_episode_break"]
    .cumsum()
    .astype(int)
)

obs["interaction_episode_id"] = (
    obs["fvg_id"].astype(str)
    + "_EP_"
    + obs["_episode_sequence"]
    .astype(str)
    .str.zfill(3)
)


# =============================================================================
# BASIC COUNTS
# =============================================================================

episode_ids = set(
    characterization["interaction_episode_id"]
)

reconstructed_episode_ids = set(
    obs["interaction_episode_id"]
)

print("\nBASIC COUNTS")
print("-" * 100)

print(
    f"Observation Rows        : {len(obs):,}"
)

print(
    f"Characterization Rows   : "
    f"{len(characterization):,}"
)

print(
    f"Reconstructed Episodes  : "
    f"{len(reconstructed_episode_ids):,}"
)


# =============================================================================
# GATE 1
# OBSERVATION CLASS VOCABULARY
# =============================================================================

invalid_observation_classes = (
    set(obs["penetration_class"].dropna().unique())
    -
    ALLOWED_OBSERVATION_CLASSES
)

gate_observation_vocabulary = (
    len(invalid_observation_classes) == 0
)

print("\nOBSERVATION CLASS VOCABULARY")
print("-" * 100)

print(
    f"Invalid Observation Classes : "
    f"{len(invalid_observation_classes)}"
)

if invalid_observation_classes:
    print(
        f"Invalid Values              : "
        f"{sorted(invalid_observation_classes)}"
    )

print(
    f"OBSERVATION VOCABULARY : "
    f"{'PASSED' if gate_observation_vocabulary else 'FAILED'}"
)


# =============================================================================
# GATE 2
# EPISODE COUNT PRESERVATION
# =============================================================================

gate_episode_count = (
    len(characterization)
    ==
    len(reconstructed_episode_ids)
)

print("\nEPISODE COUNT")
print("-" * 100)

print(
    f"Characterization Episodes : "
    f"{len(characterization):,}"
)

print(
    f"Reconstructed Episodes    : "
    f"{len(reconstructed_episode_ids):,}"
)

print(
    f"EPISODE COUNT PRESERVATION : "
    f"{'PASSED' if gate_episode_count else 'FAILED'}"
)


# =============================================================================
# GATE 3
# EPISODE ID PRESERVATION
# =============================================================================

missing_episode_ids = (
    reconstructed_episode_ids
    -
    episode_ids
)

extra_episode_ids = (
    episode_ids
    -
    reconstructed_episode_ids
)

gate_episode_identity = (
    len(missing_episode_ids) == 0
    and
    len(extra_episode_ids) == 0
)

print("\nEPISODE IDENTITY")
print("-" * 100)

print(
    f"Missing Characterization IDs : "
    f"{len(missing_episode_ids)}"
)

print(
    f"Extra Characterization IDs   : "
    f"{len(extra_episode_ids)}"
)

print(
    f"EPISODE IDENTITY : "
    f"{'PASSED' if gate_episode_identity else 'FAILED'}"
)


# =============================================================================
# GATE 4
# ONE CHARACTERIZATION ROW PER EPISODE
# =============================================================================

duplicate_episode_ids = (
    characterization[
        "interaction_episode_id"
    ]
    .duplicated()
    .sum()
)

gate_one_row_per_episode = (
    duplicate_episode_ids == 0
)

print("\nONE ROW PER EPISODE")
print("-" * 100)

print(
    f"Duplicate Episode IDs : "
    f"{duplicate_episode_ids}"
)

print(
    f"ONE ROW PER EPISODE : "
    f"{'PASSED' if gate_one_row_per_episode else 'FAILED'}"
)


# =============================================================================
# DERIVE RAW OBSERVATION-LEVEL MAXIMUM INTENSITY
# =============================================================================
#
# Hierarchy:
#
# FULL_ZONE_CROSSING
#        >
# BOUNDARY_REACHED
#        >
# INTERIOR_PENETRATION
#        >
# BOUNDARY_ONLY
#
# Directional boundary classes collapse into BOUNDARY_REACHED
# only at the Episode maximum-intensity layer.
# =============================================================================

OBSERVATION_INTENSITY_RANK = {
    "BOUNDARY_ONLY": 0,
    "INTERIOR_PENETRATION": 1,
    "LOWER_BOUNDARY_REACHED": 2,
    "UPPER_BOUNDARY_REACHED": 2,
    "FULL_ZONE_CROSSING": 3,
}

obs["_intensity_rank"] = (
    obs["penetration_class"]
    .map(OBSERVATION_INTENSITY_RANK)
)


# =============================================================================
# GATE 5
# ALL OBSERVATIONS HAVE VALID INTENSITY RANK
# =============================================================================

invalid_intensity_rank = (
    obs["_intensity_rank"].isna().sum()
)

gate_intensity_rank = (
    invalid_intensity_rank == 0
)

print("\nOBSERVATION INTENSITY RANK")
print("-" * 100)

print(
    f"Invalid Intensity Ranks : "
    f"{invalid_intensity_rank}"
)

print(
    f"INTENSITY RANK : "
    f"{'PASSED' if gate_intensity_rank else 'FAILED'}"
)


# =============================================================================
# DERIVE EXPECTED EPISODE MAXIMUM INTENSITY
# =============================================================================

expected_max_rank = (
    obs
    .groupby("interaction_episode_id")[
        "_intensity_rank"
    ]
    .max()
)

rank_to_intensity = {
    0: "BOUNDARY_ONLY",
    1: "INTERIOR_PENETRATION",
    2: "BOUNDARY_REACHED",
    3: "FULL_ZONE_CROSSING",
}

expected_max_intensity = (
    expected_max_rank
    .map(rank_to_intensity)
)


# =============================================================================
# GATE 6
# MAXIMUM INTENSITY IS ALWAYS DEFINED
# =============================================================================

undefined_max_intensity = (
    expected_max_intensity.isna().sum()
)

gate_max_intensity_defined = (
    undefined_max_intensity == 0
)

print("\nMAXIMUM INTENSITY DEFINITION")
print("-" * 100)

print(
    f"Episodes Without Intensity : "
    f"{undefined_max_intensity}"
)

print(
    f"MAXIMUM INTENSITY DEFINED : "
    f"{'PASSED' if gate_max_intensity_defined else 'FAILED'}"
)


# =============================================================================
# GATE 7
# MAXIMUM INTENSITY VOCABULARY
# =============================================================================

invalid_max_intensity = (
    set(expected_max_intensity.dropna().unique())
    -
    ALLOWED_INTENSITY_CLASSES
)

gate_max_intensity_vocabulary = (
    len(invalid_max_intensity) == 0
)

print("\nMAXIMUM INTENSITY VOCABULARY")
print("-" * 100)

print(
    f"Invalid Maximum Intensity Classes : "
    f"{len(invalid_max_intensity)}"
)

print(
    f"MAXIMUM INTENSITY VOCABULARY : "
    f"{'PASSED' if gate_max_intensity_vocabulary else 'FAILED'}"
)


# =============================================================================
# GATE 8
# CHARACTERIZATION MUST NOT YET CONTAIN A TAXONOMY LABEL
#
# This is intentional.
# We are auditing the contract before adding the new column.
# =============================================================================

premature_taxonomy_column = (
    "max_intensity_class"
    in characterization.columns
)

gate_taxonomy_not_yet_materialized = (
    not premature_taxonomy_column
)

print("\nTAXONOMY MATERIALIZATION")
print("-" * 100)

print(
    f"max_intensity_class already exists : "
    f"{premature_taxonomy_column}"
)

print(
    f"TAXONOMY NOT YET MATERIALIZED : "
    f"{'PASSED' if gate_taxonomy_not_yet_materialized else 'FAILED'}"
)


# =============================================================================
# GATE 9
# BOUNDARY-ONLY CONTRACT
#
# An Episode can be BOUNDARY_ONLY only if every observation
# inside that Episode is BOUNDARY_ONLY.
#
# Conversely, any Episode containing another class cannot
# be classified as BOUNDARY_ONLY.
# =============================================================================

episode_class_sets = (
    obs
    .groupby("interaction_episode_id")[
        "penetration_class"
    ]
    .apply(lambda x: set(x.unique()))
)

boundary_only_expected = (
    episode_class_sets
    .apply(
        lambda classes:
        classes == {"BOUNDARY_ONLY"}
    )
)

boundary_only_count = (
    boundary_only_expected.sum()
)

invalid_boundary_only_episodes = 0

for episode_id, classes in episode_class_sets.items():

    if episode_id not in characterization_ids:
        continue

    is_boundary_only = (
        classes == {"BOUNDARY_ONLY"}
    )

    if is_boundary_only:

        episode_obs = obs[
            obs["interaction_episode_id"]
            == episode_id
        ]

        if not (
            episode_obs["penetration_class"]
            == "BOUNDARY_ONLY"
        ).all():

            invalid_boundary_only_episodes += 1


gate_boundary_only_contract = (
    invalid_boundary_only_episodes == 0
)

print("\nBOUNDARY-ONLY CONTRACT")
print("-" * 100)

print(
    f"BOUNDARY-ONLY Episodes : "
    f"{boundary_only_count:,}"
)

print(
    f"Invalid BOUNDARY-ONLY Episodes : "
    f"{invalid_boundary_only_episodes}"
)

print(
    f"BOUNDARY-ONLY CONTRACT : "
    f"{'PASSED' if gate_boundary_only_contract else 'FAILED'}"
)


# =============================================================================
# GATE 10
# FULL-ZONE CROSSING CONTRACT
#
# FULL_ZONE_CROSSING can only be the maximum intensity when
# at least one observation actually has FULL_ZONE_CROSSING.
# =============================================================================

full_crossing_expected = (
    episode_class_sets
    .apply(
        lambda classes:
        "FULL_ZONE_CROSSING" in classes
    )
)

full_crossing_rank_expected = (
    expected_max_intensity
    == "FULL_ZONE_CROSSING"
)

full_crossing_contract_mismatch = (
    (
        full_crossing_expected
        !=
        full_crossing_rank_expected
    )
    .sum()
)

gate_full_crossing_contract = (
    full_crossing_contract_mismatch == 0
)

print("\nFULL-ZONE CROSSING CONTRACT")
print("-" * 100)

print(
    f"FULL-ZONE Contract Mismatches : "
    f"{full_crossing_contract_mismatch}"
)

print(
    f"FULL-ZONE CROSSING CONTRACT : "
    f"{'PASSED' if gate_full_crossing_contract else 'FAILED'}"
)


# =============================================================================
# GATE 11
# BOUNDARY REACHED CONTRACT
#
# If maximum intensity is BOUNDARY_REACHED,
# at least one directional boundary observation must exist.
# =============================================================================

boundary_reached_observed = (
    episode_class_sets
    .apply(
        lambda classes:
        (
            "LOWER_BOUNDARY_REACHED" in classes
            or
            "UPPER_BOUNDARY_REACHED" in classes
        )
    )
)

boundary_reached_expected = (
    expected_max_intensity
    == "BOUNDARY_REACHED"
)

boundary_reached_mismatch = (
    (
        boundary_reached_observed
        &
        ~boundary_reached_expected
    )
    |
    (
        boundary_reached_expected
        &
        ~boundary_reached_observed
    )
).sum()

gate_boundary_reached_contract = (
    boundary_reached_mismatch == 0
)

print("\nBOUNDARY-REACHED CONTRACT")
print("-" * 100)

print(
    f"Boundary-Reached Mismatches : "
    f"{boundary_reached_mismatch}"
)

print(
    f"BOUNDARY-REACHED CONTRACT : "
    f"{'PASSED' if gate_boundary_reached_contract else 'FAILED'}"
)


# =============================================================================
# GATE 12
# INTERIOR PENETRATION CONTRACT
#
# If maximum intensity is INTERIOR_PENETRATION,
# there must be an INTERIOR_PENETRATION observation,
# and there must be no boundary-reached/full-crossing observation.
# =============================================================================

interior_observed = (
    episode_class_sets
    .apply(
        lambda classes:
        "INTERIOR_PENETRATION" in classes
    )
)

no_higher_intensity_observed = (
    episode_class_sets
    .apply(
        lambda classes:
        not (
            "LOWER_BOUNDARY_REACHED" in classes
            or
            "UPPER_BOUNDARY_REACHED" in classes
            or
            "FULL_ZONE_CROSSING" in classes
        )
    )
)

interior_expected = (
    expected_max_intensity
    == "INTERIOR_PENETRATION"
)

interior_contract_mismatch = (
    (
        interior_expected
        &
        ~interior_observed
    )
    |
    (
        interior_expected
        &
        ~no_higher_intensity_observed
    )
).sum()

gate_interior_contract = (
    interior_contract_mismatch == 0
)

print("\nINTERIOR-PENETRATION CONTRACT")
print("-" * 100)

print(
    f"Interior Contract Mismatches : "
    f"{interior_contract_mismatch}"
)

print(
    f"INTERIOR-PENETRATION CONTRACT : "
    f"{'PASSED' if gate_interior_contract else 'FAILED'}"
)


# =============================================================================
# GATE 13
# BOUNDARY-ONLY MAXIMUM INTENSITY CONTRACT
# =============================================================================

boundary_only_expected_max = (
    expected_max_intensity
    == "BOUNDARY_ONLY"
)

boundary_only_contract_mismatch = (
    (
        boundary_only_expected_max
        &
        ~boundary_only_expected
    )
    |
    (
        boundary_only_expected
        &
        ~boundary_only_expected_max
    )
).sum()

gate_boundary_only_max_contract = (
    boundary_only_contract_mismatch == 0
)

print("\nBOUNDARY-ONLY MAXIMUM INTENSITY")
print("-" * 100)

print(
    f"Boundary-Only Max Mismatches : "
    f"{boundary_only_contract_mismatch}"
)

print(
    f"BOUNDARY-ONLY MAXIMUM CONTRACT : "
    f"{'PASSED' if gate_boundary_only_max_contract else 'FAILED'}"
)


# =============================================================================
# TAXONOMY DISTRIBUTION — DIAGNOSTIC ONLY
# =============================================================================

print("\nEXPECTED MAXIMUM INTENSITY DISTRIBUTION")
print("-" * 100)

print(
    expected_max_intensity
    .value_counts()
    .reindex(
        [
            "BOUNDARY_ONLY",
            "INTERIOR_PENETRATION",
            "BOUNDARY_REACHED",
            "FULL_ZONE_CROSSING",
        ],
        fill_value=0,
    )
)


# =============================================================================
# DIRECTIONAL BOUNDARY DIAGNOSTIC
# =============================================================================

print("\nDIRECTIONAL BOUNDARY OBSERVATION DISTRIBUTION")
print("-" * 100)

directional_counts = (
    obs["penetration_class"]
    .value_counts()
    .reindex(
        [
            "LOWER_BOUNDARY_REACHED",
            "UPPER_BOUNDARY_REACHED",
        ],
        fill_value=0,
    )
)

print(directional_counts)


# =============================================================================
# FINAL GATES
# =============================================================================

gates = {
    "OBSERVATION_CLASS_VOCABULARY":
        gate_observation_vocabulary,

    "EPISODE_COUNT_PRESERVATION":
        gate_episode_count,

    "EPISODE_IDENTITY":
        gate_episode_identity,

    "ONE_ROW_PER_EPISODE":
        gate_one_row_per_episode,

    "OBSERVATION_INTENSITY_RANK":
        gate_intensity_rank,

    "MAXIMUM_INTENSITY_DEFINED":
        gate_max_intensity_defined,

    "MAXIMUM_INTENSITY_VOCABULARY":
        gate_max_intensity_vocabulary,

    "TAXONOMY_NOT_YET_MATERIALIZED":
        gate_taxonomy_not_yet_materialized,

    "BOUNDARY_ONLY_CONTRACT":
        gate_boundary_only_contract,

    "FULL_ZONE_CROSSING_CONTRACT":
        gate_full_crossing_contract,

    "BOUNDARY_REACHED_CONTRACT":
        gate_boundary_reached_contract,

    "INTERIOR_PENETRATION_CONTRACT":
        gate_interior_contract,

    "BOUNDARY_ONLY_MAXIMUM_CONTRACT":
        gate_boundary_only_max_contract,
}


print("\n" + "=" * 100)
print("FINAL TAXONOMY CONTRACT GATE")
print("=" * 100)

for gate_name, passed in gates.items():

    print(
        f"{gate_name:<45} : "
        f"{'PASSED' if passed else 'FAILED'}"
    )


print("=" * 100)

if all(gates.values()):

    print(
        "OVERALL RESULT : PASSED"
    )

else:

    failed = [
        name
        for name, passed in gates.items()
        if not passed
    ]

    print(
        "OVERALL RESULT : FAILED"
    )

    print(
        f"FAILED GATES  : {failed}"
    )


# =============================================================================
# CLEAN INTERNAL COLUMNS
# =============================================================================

obs.drop(
    columns=[
        "_gap_minutes",
        "_episode_break",
        "_episode_sequence",
        "_intensity_rank",
    ],
    inplace=True,
    errors="ignore",
)

In [ ]:
# ==================================================================================================
# STEP 8.1-C
# EPISODE INTENSITY TAXONOMY CONTRACT AUDIT V1 — CORRECTED
# ==================================================================================================

import pandas as pd
import numpy as np

print("=" * 100)
print("EPISODE INTENSITY TAXONOMY CONTRACT AUDIT V1 — CORRECTED")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. INPUTS
# --------------------------------------------------------------------------------------------------

obs = _penetration_boundary_audit.copy()
char = episode_characterization_v1.copy()

obs["datetime"] = pd.to_datetime(obs["datetime"], utc=True)

# --------------------------------------------------------------------------------------------------
# 2. ALLOWED VOCABULARIES
# --------------------------------------------------------------------------------------------------

ALLOWED_OBSERVATION_CLASSES = {
    "BOUNDARY_ONLY",
    "INTERIOR_PENETRATION",
    "LOWER_BOUNDARY_REACHED",
    "UPPER_BOUNDARY_REACHED",
    "FULL_ZONE_CROSSING",
}

ALLOWED_INTENSITY_CLASSES = {
    "BOUNDARY_ONLY",
    "INTERIOR_PENETRATION",
    "BOUNDARY_REACHED",
    "FULL_ZONE_CROSSING",
}

OBSERVATION_INTENSITY_RANK = {
    "BOUNDARY_ONLY": 0,
    "INTERIOR_PENETRATION": 1,
    "LOWER_BOUNDARY_REACHED": 2,
    "UPPER_BOUNDARY_REACHED": 2,
    "FULL_ZONE_CROSSING": 3,
}

RANK_TO_INTENSITY = {
    0: "BOUNDARY_ONLY",
    1: "INTERIOR_PENETRATION",
    2: "BOUNDARY_REACHED",
    3: "FULL_ZONE_CROSSING",
}

# --------------------------------------------------------------------------------------------------
# 3. RECONSTRUCT EPISODE MEMBERSHIP
#    Exact Step 7 contract:
#    same FVG + exact 5-minute continuity
# --------------------------------------------------------------------------------------------------

obs = obs.sort_values(
    ["fvg_id", "datetime"],
    kind="mergesort"
).reset_index(drop=True)

obs["_gap_minutes"] = (
    obs.groupby("fvg_id")["datetime"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

obs["_episode_break"] = (
    obs["_gap_minutes"].isna()
    | (obs["_gap_minutes"] != 5.0)
)

obs["_episode_sequence"] = (
    obs.groupby("fvg_id")["_episode_break"]
    .cumsum()
    .astype(int)
)

obs["interaction_episode_id"] = (
    obs["fvg_id"].astype(str)
    + "_EP_"
    + obs["_episode_sequence"].astype(str).str.zfill(3)
)

# --------------------------------------------------------------------------------------------------
# 4. OBSERVATION-LEVEL VOCABULARY
# --------------------------------------------------------------------------------------------------

invalid_observation_classes = (
    ~obs["penetration_class"].isin(ALLOWED_OBSERVATION_CLASSES)
).sum()

obs["_intensity_rank"] = obs["penetration_class"].map(
    OBSERVATION_INTENSITY_RANK
)

invalid_intensity_ranks = obs["_intensity_rank"].isna().sum()

# --------------------------------------------------------------------------------------------------
# 5. EPISODE-LEVEL EXPECTED MAXIMUM INTENSITY
# --------------------------------------------------------------------------------------------------

expected_max_rank = (
    obs.groupby("interaction_episode_id")["_intensity_rank"]
    .max()
)

episodes_without_intensity = expected_max_rank.isna().sum()

expected_max_intensity = (
    expected_max_rank
    .map(RANK_TO_INTENSITY)
)

invalid_maximum_intensity_classes = (
    ~expected_max_intensity.isin(ALLOWED_INTENSITY_CLASSES)
).sum()

expected_taxonomy = expected_max_intensity.rename(
    "expected_max_intensity"
).reset_index()

# --------------------------------------------------------------------------------------------------
# 6. EPISODE-LEVEL GEOMETRY
# --------------------------------------------------------------------------------------------------

episode_geometry = (
    obs.groupby("interaction_episode_id", sort=False)
    .agg(
        has_lower_boundary=(
            "penetration_class",
            lambda s: (s == "LOWER_BOUNDARY_REACHED").any()
        ),
        has_upper_boundary=(
            "penetration_class",
            lambda s: (s == "UPPER_BOUNDARY_REACHED").any()
        ),
        has_full_crossing=(
            "penetration_class",
            lambda s: (s == "FULL_ZONE_CROSSING").any()
        ),
        has_interior=(
            "penetration_class",
            lambda s: (s == "INTERIOR_PENETRATION").any()
        ),
        all_boundary_only=(
            "penetration_class",
            lambda s: (s == "BOUNDARY_ONLY").all()
        ),
    )
    .reset_index()
)

# --------------------------------------------------------------------------------------------------
# 7. MERGE CHARACTERIZATION
# --------------------------------------------------------------------------------------------------

char_identity = char[
    [
        "interaction_episode_id",
        "reached_lower_boundary",
        "reached_upper_boundary",
        "full_zone_crossed",
        "boundary_only",
    ]
].copy()

episode_audit = expected_taxonomy.merge(
    episode_geometry,
    on="interaction_episode_id",
    how="left",
    validate="one_to_one"
).merge(
    char_identity,
    on="interaction_episode_id",
    how="left",
    validate="one_to_one"
)

# --------------------------------------------------------------------------------------------------
# 8. BASIC CONTRACTS
# --------------------------------------------------------------------------------------------------

characterization_episode_count = char[
    "interaction_episode_id"
].nunique()

reconstructed_episode_count = expected_taxonomy[
    "interaction_episode_id"
].nunique()

missing_characterization_ids = (
    set(expected_taxonomy["interaction_episode_id"])
    - set(char["interaction_episode_id"])
)

extra_characterization_ids = (
    set(char["interaction_episode_id"])
    - set(expected_taxonomy["interaction_episode_id"])
)

duplicate_characterization_ids = (
    char["interaction_episode_id"].duplicated().sum()
)

# --------------------------------------------------------------------------------------------------
# 9. BOUNDARY-ONLY CONTRACT
# --------------------------------------------------------------------------------------------------

boundary_only_expected = (
    episode_audit["expected_max_intensity"]
    == "BOUNDARY_ONLY"
)

boundary_only_mismatches = (
    boundary_only_expected
    &
    (
        ~episode_audit["all_boundary_only"]
        |
        ~episode_audit["boundary_only"].astype(bool)
    )
).sum()

# --------------------------------------------------------------------------------------------------
# 10. FULL-ZONE CROSSING CONTRACT
# --------------------------------------------------------------------------------------------------

full_crossing_expected = (
    episode_audit["expected_max_intensity"]
    == "FULL_ZONE_CROSSING"
)

full_crossing_mismatches = (
    full_crossing_expected
    &
    ~episode_audit["has_full_crossing"]
).sum()

# --------------------------------------------------------------------------------------------------
# 11. BOUNDARY-REACHED CONTRACT — CORRECTED
#
# Formal rule:
#
# BOUNDARY_REACHED iff:
#   at least one LOWER or UPPER boundary observation exists
#   AND no FULL_ZONE_CROSSING exists
#
# Direction is preserved separately.
# LOWER + UPPER is valid.
# INTERIOR + BOUNDARY is valid.
# --------------------------------------------------------------------------------------------------

boundary_reached_expected = (
    (episode_audit["has_lower_boundary"]
     | episode_audit["has_upper_boundary"])
    &
    ~episode_audit["has_full_crossing"]
)

boundary_reached_taxonomy = (
    episode_audit["expected_max_intensity"]
    == "BOUNDARY_REACHED"
)

boundary_reached_mismatches = (
    boundary_reached_expected
    != boundary_reached_taxonomy
).sum()

# Characterization directional geometry must agree with reconstructed geometry.
boundary_direction_mismatches = (
    (
        episode_audit["reached_lower_boundary"].astype(bool)
        != episode_audit["has_lower_boundary"]
    )
    |
    (
        episode_audit["reached_upper_boundary"].astype(bool)
        != episode_audit["has_upper_boundary"]
    )
).sum()

# --------------------------------------------------------------------------------------------------
# 12. INTERIOR-PENETRATION CONTRACT
# --------------------------------------------------------------------------------------------------

interior_expected = (
    episode_audit["has_interior"]
    &
    ~episode_audit["has_lower_boundary"]
    &
    ~episode_audit["has_upper_boundary"]
    &
    ~episode_audit["has_full_crossing"]
)

interior_taxonomy = (
    episode_audit["expected_max_intensity"]
    == "INTERIOR_PENETRATION"
)

interior_mismatches = (
    interior_expected != interior_taxonomy
).sum()

# --------------------------------------------------------------------------------------------------
# 13. TAXONOMY MATERIALIZATION CHECK
# --------------------------------------------------------------------------------------------------

taxonomy_materialized = (
    "max_intensity_class" in char.columns
)

# --------------------------------------------------------------------------------------------------
# 14. DIAGNOSTICS
# --------------------------------------------------------------------------------------------------

expected_distribution = (
    expected_taxonomy["expected_max_intensity"]
    .value_counts()
    .reindex(
        [
            "BOUNDARY_ONLY",
            "INTERIOR_PENETRATION",
            "BOUNDARY_REACHED",
            "FULL_ZONE_CROSSING",
        ],
        fill_value=0
    )
)

directional_distribution = (
    obs.loc[
        obs["penetration_class"].isin(
            {
                "LOWER_BOUNDARY_REACHED",
                "UPPER_BOUNDARY_REACHED",
            }
        ),
        "penetration_class",
    ]
    .value_counts()
    .reindex(
        [
            "LOWER_BOUNDARY_REACHED",
            "UPPER_BOUNDARY_REACHED",
        ],
        fill_value=0
    )
)

# --------------------------------------------------------------------------------------------------
# 15. PRINT REPORT
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("BASIC COUNTS")
print("-" * 100)
print(f"Observation Rows        : {len(obs):,}")
print(f"Characterization Rows   : {len(char):,}")
print(f"Reconstructed Episodes  : {reconstructed_episode_count:,}")

print("\n" + "-" * 100)
print("OBSERVATION CLASS VOCABULARY")
print("-" * 100)
print(f"Invalid Observation Classes : {invalid_observation_classes}")
print(
    "OBSERVATION VOCABULARY : "
    + ("PASSED" if invalid_observation_classes == 0 else "FAILED")
)

print("\n" + "-" * 100)
print("EPISODE COUNT")
print("-" * 100)
print(f"Characterization Episodes : {characterization_episode_count:,}")
print(f"Reconstructed Episodes    : {reconstructed_episode_count:,}")
print(
    "EPISODE COUNT PRESERVATION : "
    + (
        "PASSED"
        if characterization_episode_count == reconstructed_episode_count
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("EPISODE IDENTITY")
print("-" * 100)
print(f"Missing Characterization IDs : {len(missing_characterization_ids)}")
print(f"Extra Characterization IDs   : {len(extra_characterization_ids)}")
print(
    "EPISODE IDENTITY : "
    + (
        "PASSED"
        if not missing_characterization_ids
        and not extra_characterization_ids
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("ONE ROW PER EPISODE")
print("-" * 100)
print(f"Duplicate Episode IDs : {duplicate_characterization_ids}")
print(
    "ONE ROW PER EPISODE : "
    + (
        "PASSED"
        if duplicate_characterization_ids == 0
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("OBSERVATION INTENSITY RANK")
print("-" * 100)
print(f"Invalid Intensity Ranks : {invalid_intensity_ranks}")
print(
    "INTENSITY RANK : "
    + (
        "PASSED"
        if invalid_intensity_ranks == 0
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("MAXIMUM INTENSITY DEFINITION")
print("-" * 100)
print(f"Episodes Without Intensity : {episodes_without_intensity}")
print(
    "MAXIMUM INTENSITY DEFINED : "
    + (
        "PASSED"
        if episodes_without_intensity == 0
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("MAXIMUM INTENSITY VOCABULARY")
print("-" * 100)
print(f"Invalid Maximum Intensity Classes : {invalid_maximum_intensity_classes}")
print(
    "MAXIMUM INTENSITY VOCABULARY : "
    + (
        "PASSED"
        if invalid_maximum_intensity_classes == 0
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("TAXONOMY MATERIALIZATION")
print("-" * 100)
print(f"max_intensity_class already exists : {taxonomy_materialized}")
print(
    "TAXONOMY NOT YET MATERIALIZED : "
    + (
        "PASSED"
        if not taxonomy_materialized
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("BOUNDARY-ONLY CONTRACT")
print("-" * 100)
print(
    f"BOUNDARY-ONLY Episodes : "
    f"{boundary_only_expected.sum()}"
)
print(
    f"Invalid BOUNDARY-ONLY Episodes : "
    f"{boundary_only_mismatches}"
)
print(
    "BOUNDARY-ONLY CONTRACT : "
    + (
        "PASSED"
        if boundary_only_mismatches == 0
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("FULL-ZONE CROSSING CONTRACT")
print("-" * 100)
print(
    f"FULL-ZONE Contract Mismatches : "
    f"{full_crossing_mismatches}"
)
print(
    "FULL-ZONE CROSSING CONTRACT : "
    + (
        "PASSED"
        if full_crossing_mismatches == 0
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("BOUNDARY-REACHED CONTRACT")
print("-" * 100)
print(
    f"Expected BOUNDARY-REACHED Episodes : "
    f"{boundary_reached_taxonomy.sum()}"
)
print(
    f"Boundary-Reached Taxonomy Mismatches : "
    f"{boundary_reached_mismatches}"
)
print(
    f"Directional Boundary Flag Mismatches : "
    f"{boundary_direction_mismatches}"
)
print(
    "BOUNDARY-REACHED CONTRACT : "
    + (
        "PASSED"
        if boundary_reached_mismatches == 0
        and boundary_direction_mismatches == 0
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("INTERIOR-PENETRATION CONTRACT")
print("-" * 100)
print(
    f"Interior Contract Mismatches : "
    f"{interior_mismatches}"
)
print(
    "INTERIOR-PENETRATION CONTRACT : "
    + (
        "PASSED"
        if interior_mismatches == 0
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("BOUNDARY-ONLY MAXIMUM INTENSITY")
print("-" * 100)

boundary_only_max_mismatches = (
    (
        episode_audit["expected_max_intensity"]
        == "BOUNDARY_ONLY"
    )
    !=
    episode_audit["all_boundary_only"]
).sum()

print(
    f"Boundary-Only Max Mismatches : "
    f"{boundary_only_max_mismatches}"
)

print(
    "BOUNDARY-ONLY MAXIMUM CONTRACT : "
    + (
        "PASSED"
        if boundary_only_max_mismatches == 0
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("EXPECTED MAXIMUM INTENSITY DISTRIBUTION")
print("-" * 100)
print(expected_distribution.to_string())

print("\n" + "-" * 100)
print("DIRECTIONAL BOUNDARY OBSERVATION DISTRIBUTION")
print("-" * 100)
print(directional_distribution.to_string())

# --------------------------------------------------------------------------------------------------
# 16. FINAL GATE
# --------------------------------------------------------------------------------------------------

gates = {
    "OBSERVATION_CLASS_VOCABULARY": invalid_observation_classes == 0,
    "EPISODE_COUNT_PRESERVATION":
        characterization_episode_count == reconstructed_episode_count,
    "EPISODE_IDENTITY":
        not missing_characterization_ids
        and not extra_characterization_ids,
    "ONE_ROW_PER_EPISODE":
        duplicate_characterization_ids == 0,
    "OBSERVATION_INTENSITY_RANK":
        invalid_intensity_ranks == 0,
    "MAXIMUM_INTENSITY_DEFINED":
        episodes_without_intensity == 0,
    "MAXIMUM_INTENSITY_VOCABULARY":
        invalid_maximum_intensity_classes == 0,
    "TAXONOMY_NOT_YET_MATERIALIZED":
        not taxonomy_materialized,
    "BOUNDARY_ONLY_CONTRACT":
        boundary_only_mismatches == 0,
    "FULL_ZONE_CROSSING_CONTRACT":
        full_crossing_mismatches == 0,
    "BOUNDARY_REACHED_CONTRACT":
        boundary_reached_mismatches == 0
        and boundary_direction_mismatches == 0,
    "INTERIOR_PENETRATION_CONTRACT":
        interior_mismatches == 0,
    "BOUNDARY_ONLY_MAXIMUM_CONTRACT":
        boundary_only_max_mismatches == 0,
}

print("\n" + "=" * 100)
print("FINAL TAXONOMY CONTRACT GATE")
print("=" * 100)

for gate_name, passed in gates.items():
    print(
        f"{gate_name:<48} : "
        f"{'PASSED' if passed else 'FAILED'}"
    )

failed_gates = [
    name for name, passed in gates.items()
    if not passed
]

print("=" * 100)

if not failed_gates:
    print("OVERALL RESULT : PASSED")
    print("FAILED GATES  : []")
else:
    print("OVERALL RESULT : FAILED")
    print(f"FAILED GATES  : {failed_gates}")

print("=" * 100)

In [ ]:
# ==================================================================================================
# EPISODE INTENSITY TAXONOMY CONTRACT AUDIT V1 — FINAL CLEAN VERSION
# ==================================================================================================

import pandas as pd
import numpy as np

print("=" * 100)
print("EPISODE INTENSITY TAXONOMY CONTRACT AUDIT V1 — FINAL CLEAN VERSION")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. INPUTS
# --------------------------------------------------------------------------------------------------

obs = _penetration_boundary_audit.copy()
char = episode_characterization_v1.copy()

obs["datetime"] = pd.to_datetime(obs["datetime"], utc=True)

# --------------------------------------------------------------------------------------------------
# 2. TAXONOMY VOCABULARY
# --------------------------------------------------------------------------------------------------

ALLOWED_OBSERVATION_CLASSES = {
    "BOUNDARY_ONLY",
    "INTERIOR_PENETRATION",
    "LOWER_BOUNDARY_REACHED",
    "UPPER_BOUNDARY_REACHED",
    "FULL_ZONE_CROSSING",
}

ALLOWED_INTENSITY_CLASSES = {
    "BOUNDARY_ONLY",
    "INTERIOR_PENETRATION",
    "BOUNDARY_REACHED",
    "FULL_ZONE_CROSSING",
}

OBSERVATION_INTENSITY_RANK = {
    "BOUNDARY_ONLY": 0,
    "INTERIOR_PENETRATION": 1,
    "LOWER_BOUNDARY_REACHED": 2,
    "UPPER_BOUNDARY_REACHED": 2,
    "FULL_ZONE_CROSSING": 3,
}

RANK_TO_INTENSITY = {
    0: "BOUNDARY_ONLY",
    1: "INTERIOR_PENETRATION",
    2: "BOUNDARY_REACHED",
    3: "FULL_ZONE_CROSSING",
}

# --------------------------------------------------------------------------------------------------
# 3. RECONSTRUCT EPISODE MEMBERSHIP
#    Exact Step 7 contract:
#    same FVG + exact 5-minute continuity
# --------------------------------------------------------------------------------------------------

obs = obs.sort_values(
    ["fvg_id", "datetime"],
    kind="mergesort"
).reset_index(drop=True)

obs["_gap_minutes"] = (
    obs.groupby("fvg_id")["datetime"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

obs["_episode_break"] = (
    obs["_gap_minutes"].isna()
    | (obs["_gap_minutes"] != 5.0)
)

obs["_episode_sequence"] = (
    obs.groupby("fvg_id")["_episode_break"]
    .cumsum()
    .astype(int)
)

obs["interaction_episode_id"] = (
    obs["fvg_id"].astype(str)
    + "_EP_"
    + obs["_episode_sequence"].astype(str).str.zfill(3)
)

# --------------------------------------------------------------------------------------------------
# 4. OBSERVATION-LEVEL TAXONOMY
# --------------------------------------------------------------------------------------------------

invalid_observation_classes = (
    ~obs["penetration_class"].isin(ALLOWED_OBSERVATION_CLASSES)
).sum()

obs["_intensity_rank"] = obs["penetration_class"].map(
    OBSERVATION_INTENSITY_RANK
)

invalid_intensity_ranks = obs["_intensity_rank"].isna().sum()

# --------------------------------------------------------------------------------------------------
# 5. EPISODE MAXIMUM INTENSITY
# --------------------------------------------------------------------------------------------------

expected_max_rank = (
    obs.groupby("interaction_episode_id")["_intensity_rank"]
    .max()
)

episodes_without_intensity = expected_max_rank.isna().sum()

expected_max_intensity = (
    expected_max_rank.map(RANK_TO_INTENSITY)
)

invalid_maximum_intensity_classes = (
    ~expected_max_intensity.isin(ALLOWED_INTENSITY_CLASSES)
).sum()

expected_taxonomy = expected_max_intensity.rename(
    "expected_max_intensity"
).reset_index()

# --------------------------------------------------------------------------------------------------
# 6. EPISODE GEOMETRY — DIAGNOSTIC ONLY
# --------------------------------------------------------------------------------------------------

episode_geometry = (
    obs.groupby("interaction_episode_id", sort=False)
    .agg(
        has_lower_boundary=(
            "penetration_class",
            lambda s: (s == "LOWER_BOUNDARY_REACHED").any()
        ),
        has_upper_boundary=(
            "penetration_class",
            lambda s: (s == "UPPER_BOUNDARY_REACHED").any()
        ),
        has_full_crossing=(
            "penetration_class",
            lambda s: (s == "FULL_ZONE_CROSSING").any()
        ),
        has_interior=(
            "penetration_class",
            lambda s: (s == "INTERIOR_PENETRATION").any()
        ),
        all_boundary_only=(
            "penetration_class",
            lambda s: (s == "BOUNDARY_ONLY").all()
        ),
    )
    .reset_index()
)

# --------------------------------------------------------------------------------------------------
# 7. MERGE WITH CHARACTERIZATION
# --------------------------------------------------------------------------------------------------

char_identity = char[
    [
        "interaction_episode_id",
        "reached_lower_boundary",
        "reached_upper_boundary",
        "full_zone_crossed",
        "boundary_only",
    ]
].copy()

episode_audit = expected_taxonomy.merge(
    episode_geometry,
    on="interaction_episode_id",
    how="left",
    validate="one_to_one"
).merge(
    char_identity,
    on="interaction_episode_id",
    how="left",
    validate="one_to_one"
)

# --------------------------------------------------------------------------------------------------
# 8. EPISODE COUNT / IDENTITY
# --------------------------------------------------------------------------------------------------

characterization_episode_count = char[
    "interaction_episode_id"
].nunique()

reconstructed_episode_count = expected_taxonomy[
    "interaction_episode_id"
].nunique()

missing_characterization_ids = (
    set(expected_taxonomy["interaction_episode_id"])
    - set(char["interaction_episode_id"])
)

extra_characterization_ids = (
    set(char["interaction_episode_id"])
    - set(expected_taxonomy["interaction_episode_id"])
)

duplicate_characterization_ids = (
    char["interaction_episode_id"].duplicated().sum()
)

# --------------------------------------------------------------------------------------------------
# 9. BOUNDARY-ONLY CONTRACT
#
# Episode is BOUNDARY_ONLY iff ALL observations are BOUNDARY_ONLY.
# --------------------------------------------------------------------------------------------------

boundary_only_expected = (
    episode_audit["expected_max_intensity"]
    == "BOUNDARY_ONLY"
)

boundary_only_mismatches = (
    boundary_only_expected
    != episode_audit["all_boundary_only"]
).sum()

# --------------------------------------------------------------------------------------------------
# 10. FULL-ZONE CROSSING CONTRACT
#
# Episode is FULL_ZONE_CROSSING iff at least one observation
# is FULL_ZONE_CROSSING.
# --------------------------------------------------------------------------------------------------

full_crossing_expected = (
    episode_audit["expected_max_intensity"]
    == "FULL_ZONE_CROSSING"
)

full_crossing_mismatches = (
    full_crossing_expected
    != episode_audit["has_full_crossing"]
).sum()

# --------------------------------------------------------------------------------------------------
# 11. BOUNDARY-REACHED CONTRACT
#
# Episode is BOUNDARY_REACHED iff:
#
#   LOWER_BOUNDARY_REACHED OR UPPER_BOUNDARY_REACHED exists
#   AND
#   FULL_ZONE_CROSSING does not exist.
#
# LOWER + UPPER is valid.
# BOUNDARY + INTERIOR is valid.
#
# Directional flags themselves are NOT part of this taxonomy gate.
# They remain preserved independently in Episode Characterization.
# --------------------------------------------------------------------------------------------------

boundary_reached_expected = (
    (episode_audit["has_lower_boundary"]
     | episode_audit["has_upper_boundary"])
    &
    ~episode_audit["has_full_crossing"]
)

boundary_reached_taxonomy = (
    episode_audit["expected_max_intensity"]
    == "BOUNDARY_REACHED"
)

boundary_reached_mismatches = (
    boundary_reached_expected
    != boundary_reached_taxonomy
).sum()

# --------------------------------------------------------------------------------------------------
# 12. INTERIOR-PENETRATION CONTRACT
#
# Episode is INTERIOR_PENETRATION iff:
#
#   at least one INTERIOR observation
#   AND no boundary reached
#   AND no full crossing.
# --------------------------------------------------------------------------------------------------

interior_expected = (
    episode_audit["has_interior"]
    &
    ~episode_audit["has_lower_boundary"]
    &
    ~episode_audit["has_upper_boundary"]
    &
    ~episode_audit["has_full_crossing"]
)

interior_taxonomy = (
    episode_audit["expected_max_intensity"]
    == "INTERIOR_PENETRATION"
)

interior_mismatches = (
    interior_expected
    != interior_taxonomy
).sum()

# --------------------------------------------------------------------------------------------------
# 13. BOUNDARY-ONLY MAXIMUM CONTRACT
# --------------------------------------------------------------------------------------------------

boundary_only_max_mismatches = (
    (
        episode_audit["expected_max_intensity"]
        == "BOUNDARY_ONLY"
    )
    != episode_audit["all_boundary_only"]
).sum()

# --------------------------------------------------------------------------------------------------
# 14. TAXONOMY MUST NOT YET BE MATERIALIZED
# --------------------------------------------------------------------------------------------------

taxonomy_materialized = (
    "max_intensity_class" in char.columns
)

# --------------------------------------------------------------------------------------------------
# 15. DIRECTIONAL FLAGS — DIAGNOSTIC ONLY
#
# IMPORTANT:
# These are NOT included in the Taxonomy Gate.
# We only report them for visibility.
# --------------------------------------------------------------------------------------------------

directional_lower_mismatches = (
    episode_audit["reached_lower_boundary"].astype(bool)
    != episode_audit["has_lower_boundary"]
).sum()

directional_upper_mismatches = (
    episode_audit["reached_upper_boundary"].astype(bool)
    != episode_audit["has_upper_boundary"]
).sum()

directional_full_mismatches = (
    episode_audit["full_zone_crossed"].astype(bool)
    != episode_audit["has_full_crossing"]
).sum()

# --------------------------------------------------------------------------------------------------
# 16. DISTRIBUTIONS
# --------------------------------------------------------------------------------------------------

expected_distribution = (
    expected_taxonomy["expected_max_intensity"]
    .value_counts()
    .reindex(
        [
            "BOUNDARY_ONLY",
            "INTERIOR_PENETRATION",
            "BOUNDARY_REACHED",
            "FULL_ZONE_CROSSING",
        ],
        fill_value=0
    )
)

directional_distribution = (
    obs.loc[
        obs["penetration_class"].isin(
            {
                "LOWER_BOUNDARY_REACHED",
                "UPPER_BOUNDARY_REACHED",
            }
        ),
        "penetration_class",
    ]
    .value_counts()
    .reindex(
        [
            "LOWER_BOUNDARY_REACHED",
            "UPPER_BOUNDARY_REACHED",
        ],
        fill_value=0
    )
)

# --------------------------------------------------------------------------------------------------
# 17. PRINT REPORT
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("BASIC COUNTS")
print("-" * 100)
print(f"Observation Rows        : {len(obs):,}")
print(f"Characterization Rows   : {len(char):,}")
print(f"Reconstructed Episodes  : {reconstructed_episode_count:,}")

print("\n" + "-" * 100)
print("OBSERVATION CLASS VOCABULARY")
print("-" * 100)
print(f"Invalid Observation Classes : {invalid_observation_classes}")
print(
    "OBSERVATION VOCABULARY : "
    + ("PASSED" if invalid_observation_classes == 0 else "FAILED")
)

print("\n" + "-" * 100)
print("EPISODE COUNT")
print("-" * 100)
print(f"Characterization Episodes : {characterization_episode_count:,}")
print(f"Reconstructed Episodes    : {reconstructed_episode_count:,}")
print(
    "EPISODE COUNT PRESERVATION : "
    + (
        "PASSED"
        if characterization_episode_count == reconstructed_episode_count
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("EPISODE IDENTITY")
print("-" * 100)
print(f"Missing Characterization IDs : {len(missing_characterization_ids)}")
print(f"Extra Characterization IDs   : {len(extra_characterization_ids)}")
print(
    "EPISODE IDENTITY : "
    + (
        "PASSED"
        if not missing_characterization_ids
        and not extra_characterization_ids
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("ONE ROW PER EPISODE")
print("-" * 100)
print(f"Duplicate Episode IDs : {duplicate_characterization_ids}")
print(
    "ONE ROW PER EPISODE : "
    + (
        "PASSED"
        if duplicate_characterization_ids == 0
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("OBSERVATION INTENSITY RANK")
print("-" * 100)
print(f"Invalid Intensity Ranks : {invalid_intensity_ranks}")
print(
    "INTENSITY RANK : "
    + (
        "PASSED"
        if invalid_intensity_ranks == 0
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("MAXIMUM INTENSITY DEFINITION")
print("-" * 100)
print(f"Episodes Without Intensity : {episodes_without_intensity}")
print(
    "MAXIMUM INTENSITY DEFINED : "
    + (
        "PASSED"
        if episodes_without_intensity == 0
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("MAXIMUM INTENSITY VOCABULARY")
print("-" * 100)
print(f"Invalid Maximum Intensity Classes : {invalid_maximum_intensity_classes}")
print(
    "MAXIMUM INTENSITY VOCABULARY : "
    + (
        "PASSED"
        if invalid_maximum_intensity_classes == 0
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("TAXONOMY MATERIALIZATION")
print("-" * 100)
print(f"max_intensity_class already exists : {taxonomy_materialized}")
print(
    "TAXONOMY NOT YET MATERIALIZED : "
    + (
        "PASSED"
        if not taxonomy_materialized
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("BOUNDARY-ONLY CONTRACT")
print("-" * 100)
print(f"BOUNDARY-ONLY Episodes : {boundary_only_expected.sum()}")
print(f"Invalid BOUNDARY-ONLY Episodes : {boundary_only_mismatches}")
print(
    "BOUNDARY-ONLY CONTRACT : "
    + (
        "PASSED"
        if boundary_only_mismatches == 0
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("FULL-ZONE CROSSING CONTRACT")
print("-" * 100)
print(f"FULL-ZONE Contract Mismatches : {full_crossing_mismatches}")
print(
    "FULL-ZONE CROSSING CONTRACT : "
    + (
        "PASSED"
        if full_crossing_mismatches == 0
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("BOUNDARY-REACHED CONTRACT")
print("-" * 100)
print(
    f"Expected BOUNDARY-REACHED Episodes : "
    f"{boundary_reached_taxonomy.sum()}"
)
print(
    f"Boundary-Reached Taxonomy Mismatches : "
    f"{boundary_reached_mismatches}"
)
print(
    "BOUNDARY-REACHED CONTRACT : "
    + (
        "PASSED"
        if boundary_reached_mismatches == 0
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("INTERIOR-PENETRATION CONTRACT")
print("-" * 100)
print(f"Interior Contract Mismatches : {interior_mismatches}")
print(
    "INTERIOR-PENETRATION CONTRACT : "
    + (
        "PASSED"
        if interior_mismatches == 0
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("BOUNDARY-ONLY MAXIMUM INTENSITY")
print("-" * 100)
print(f"Boundary-Only Max Mismatches : {boundary_only_max_mismatches}")
print(
    "BOUNDARY-ONLY MAXIMUM CONTRACT : "
    + (
        "PASSED"
        if boundary_only_max_mismatches == 0
        else "FAILED"
    )
)

print("\n" + "-" * 100)
print("DIRECTIONAL FLAGS — DIAGNOSTIC ONLY")
print("-" * 100)
print(f"Lower Boundary Flag Mismatches : {directional_lower_mismatches}")
print(f"Upper Boundary Flag Mismatches : {directional_upper_mismatches}")
print(f"Full Crossing Flag Mismatches  : {directional_full_mismatches}")
print("NOTE : Directional flags are diagnostic only and are NOT Taxonomy gates.")

print("\n" + "-" * 100)
print("EXPECTED MAXIMUM INTENSITY DISTRIBUTION")
print("-" * 100)
print(expected_distribution.to_string())

print("\n" + "-" * 100)
print("DIRECTIONAL BOUNDARY OBSERVATION DISTRIBUTION")
print("-" * 100)
print(directional_distribution.to_string())

# --------------------------------------------------------------------------------------------------
# 18. FINAL TAXONOMY GATE
# --------------------------------------------------------------------------------------------------

gates = {
    "OBSERVATION_CLASS_VOCABULARY":
        invalid_observation_classes == 0,

    "EPISODE_COUNT_PRESERVATION":
        characterization_episode_count == reconstructed_episode_count,

    "EPISODE_IDENTITY":
        not missing_characterization_ids
        and not extra_characterization_ids,

    "ONE_ROW_PER_EPISODE":
        duplicate_characterization_ids == 0,

    "OBSERVATION_INTENSITY_RANK":
        invalid_intensity_ranks == 0,

    "MAXIMUM_INTENSITY_DEFINED":
        episodes_without_intensity == 0,

    "MAXIMUM_INTENSITY_VOCABULARY":
        invalid_maximum_intensity_classes == 0,

    "TAXONOMY_NOT_YET_MATERIALIZED":
        not taxonomy_materialized,

    "BOUNDARY_ONLY_CONTRACT":
        boundary_only_mismatches == 0,

    "FULL_ZONE_CROSSING_CONTRACT":
        full_crossing_mismatches == 0,

    "BOUNDARY_REACHED_CONTRACT":
        boundary_reached_mismatches == 0,

    "INTERIOR_PENETRATION_CONTRACT":
        interior_mismatches == 0,

    "BOUNDARY_ONLY_MAXIMUM_CONTRACT":
        boundary_only_max_mismatches == 0,
}

print("\n" + "=" * 100)
print("FINAL TAXONOMY CONTRACT GATE")
print("=" * 100)

for gate_name, passed in gates.items():
    print(
        f"{gate_name:<48} : "
        f"{'PASSED' if passed else 'FAILED'}"
    )

failed_gates = [
    name for name, passed in gates.items()
    if not passed
]

print("=" * 100)

if not failed_gates:
    print("OVERALL RESULT : PASSED")
    print("FAILED GATES  : []")
else:
    print("OVERALL RESULT : FAILED")
    print(f"FAILED GATES  : {failed_gates}")

print("=" * 100)

In [ ]:
# ==================================================================================================
# STEP 8.1-D — EPISODE INTENSITY TAXONOMY MATERIALIZATION V1
# ==================================================================================================

import pandas as pd
import numpy as np

print("=" * 100)
print("STEP 8.1-D — EPISODE INTENSITY TAXONOMY MATERIALIZATION V1")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# CONTRACT
# --------------------------------------------------------------------------------------------------

INTENSITY_RANK = {
    "BOUNDARY_ONLY": 0,
    "INTERIOR_PENETRATION": 1,
    "BOUNDARY_REACHED": 2,
    "FULL_ZONE_CROSSING": 3,
}

ALLOWED_INTENSITY_CLASSES = set(INTENSITY_RANK.keys())

OBSERVATION_INTENSITY_RANK = {
    "BOUNDARY_ONLY": 0,
    "INTERIOR_PENETRATION": 1,
    "LOWER_BOUNDARY_REACHED": 2,
    "UPPER_BOUNDARY_REACHED": 2,
    "FULL_ZONE_CROSSING": 3,
}

ALLOWED_OBSERVATION_CLASSES = set(OBSERVATION_INTENSITY_RANK.keys())

# --------------------------------------------------------------------------------------------------
# INPUT VALIDATION
# --------------------------------------------------------------------------------------------------

assert "_penetration_boundary_audit" in globals(), \
    "_penetration_boundary_audit is not available."

assert "episode_characterization_v1" in globals(), \
    "episode_characterization_v1 is not available."

assert "interaction_episode_registry_v1_1" in globals(), \
    "interaction_episode_registry_v1_1 is not available."

required_observation_columns = {
    "fvg_id",
    "datetime",
    "penetration_class",
}

missing_obs = required_observation_columns - set(_penetration_boundary_audit.columns)

assert not missing_obs, \
    f"Missing observation columns: {sorted(missing_obs)}"

required_characterization_columns = {
    "interaction_episode_id",
    "fvg_id",
    "episode_sequence",
}

missing_char = required_characterization_columns - set(
    episode_characterization_v1.columns
)

assert not missing_char, \
    f"Missing characterization columns: {sorted(missing_char)}"

# --------------------------------------------------------------------------------------------------
# PRESERVE ORIGINAL CHARACTERIZATION EXACTLY
# --------------------------------------------------------------------------------------------------

episode_characterization_v1_1 = episode_characterization_v1.copy(deep=True)

assert "max_intensity_class" not in episode_characterization_v1_1.columns, \
    "max_intensity_class already exists."

# --------------------------------------------------------------------------------------------------
# RECONSTRUCT EPISODES USING FROZEN STEP 7 CONTRACT
# Same FVG + exact consecutive 5-minute interaction observations
# --------------------------------------------------------------------------------------------------

_obs = _penetration_boundary_audit.copy(deep=True)

_obs["datetime"] = pd.to_datetime(
    _obs["datetime"],
    utc=True,
    errors="raise",
)

_obs = _obs.sort_values(
    ["fvg_id", "datetime"],
    kind="mergesort",
).reset_index(drop=True)

_obs["_gap_minutes"] = (
    _obs.groupby("fvg_id")["datetime"]
    .diff()
    .dt.total_seconds()
    .div(60.0)
)

_obs["_episode_break"] = (
    _obs["_gap_minutes"].isna()
    | (_obs["_gap_minutes"] != 5.0)
)

_obs["episode_sequence"] = (
    _obs.groupby("fvg_id")["_episode_break"]
    .cumsum()
    .astype(int)
)

_obs["interaction_episode_id"] = (
    _obs["fvg_id"].astype(str)
    + "_EP_"
    + _obs["episode_sequence"].astype(str).str.zfill(3)
)

# --------------------------------------------------------------------------------------------------
# OBSERVATION CLASS VALIDATION
# --------------------------------------------------------------------------------------------------

invalid_classes = sorted(
    set(_obs["penetration_class"].dropna().unique())
    - ALLOWED_OBSERVATION_CLASSES
)

assert not invalid_classes, \
    f"Invalid observation intensity classes: {invalid_classes}"

# --------------------------------------------------------------------------------------------------
# DERIVE EPISODE MAX INTENSITY
# --------------------------------------------------------------------------------------------------

_obs["_observation_intensity_rank"] = (
    _obs["penetration_class"]
    .map(OBSERVATION_INTENSITY_RANK)
)

assert _obs["_observation_intensity_rank"].notna().all(), \
    "Some observations have no intensity rank."

episode_intensity = (
    _obs.groupby(
        "interaction_episode_id",
        sort=False,
    )["_observation_intensity_rank"]
    .max()
    .rename("max_intensity_rank")
    .reset_index()
)

# Rank → canonical episode-level taxonomy
rank_to_class = {
    0: "BOUNDARY_ONLY",
    1: "INTERIOR_PENETRATION",
    2: "BOUNDARY_REACHED",
    3: "FULL_ZONE_CROSSING",
}

episode_intensity["max_intensity_class"] = (
    episode_intensity["max_intensity_rank"]
    .map(rank_to_class)
)

assert episode_intensity["max_intensity_class"].notna().all(), \
    "Some Episodes could not be assigned a maximum intensity class."

# --------------------------------------------------------------------------------------------------
# EPISODE IDENTITY CONTRACT
# --------------------------------------------------------------------------------------------------

characterization_ids = set(
    episode_characterization_v1_1["interaction_episode_id"]
)

derived_ids = set(
    episode_intensity["interaction_episode_id"]
)

missing_ids = derived_ids - characterization_ids
extra_ids = characterization_ids - derived_ids

assert not missing_ids, \
    f"Derived Episodes missing from characterization: {sorted(missing_ids)[:10]}"

assert not extra_ids, \
    f"Characterization Episodes missing from derived taxonomy: {sorted(extra_ids)[:10]}"

assert len(characterization_ids) == len(episode_characterization_v1_1), \
    "Duplicate Episode IDs in characterization."

assert len(derived_ids) == len(episode_intensity), \
    "Duplicate Episode IDs in derived taxonomy."

# --------------------------------------------------------------------------------------------------
# MATERIALIZE
# --------------------------------------------------------------------------------------------------

episode_characterization_v1_1 = (
    episode_characterization_v1_1
    .merge(
        episode_intensity[
            [
                "interaction_episode_id",
                "max_intensity_class",
            ]
        ],
        on="interaction_episode_id",
        how="left",
        validate="one_to_one",
    )
)

# --------------------------------------------------------------------------------------------------
# FINAL MATERIALIZATION CONTRACT
# --------------------------------------------------------------------------------------------------

assert len(episode_characterization_v1_1) == len(
    episode_characterization_v1
), "Row count changed."

assert "max_intensity_class" in episode_characterization_v1_1.columns

assert (
    episode_characterization_v1_1["max_intensity_class"]
    .isin(ALLOWED_INTENSITY_CLASSES)
    .all()
), "Invalid materialized intensity class."

assert (
    episode_characterization_v1_1["max_intensity_class"]
    .notna()
    .all()
), "Some Episodes have no materialized intensity class."

assert (
    episode_characterization_v1_1["interaction_episode_id"]
    .is_unique
), "Episode IDs are no longer unique."

# Restore original column order + new column at the end
original_columns = list(episode_characterization_v1.columns)

episode_characterization_v1_1 = episode_characterization_v1_1[
    original_columns + ["max_intensity_class"]
].copy()

# --------------------------------------------------------------------------------------------------
# CLEANUP
# --------------------------------------------------------------------------------------------------

_obs.drop(
    columns=[
        "_gap_minutes",
        "_episode_break",
        "episode_sequence",
        "interaction_episode_id",
        "_observation_intensity_rank",
    ],
    inplace=True,
)

# --------------------------------------------------------------------------------------------------
# REPORT
# --------------------------------------------------------------------------------------------------

print()
print("MATERIALIZATION STATUS")
print("-" * 100)

print(f"Input Episodes              : {len(episode_characterization_v1):,}")
print(f"Output Episodes             : {len(episode_characterization_v1_1):,}")
print(f"Input Columns               : {len(episode_characterization_v1.columns)}")
print(f"Output Columns              : {len(episode_characterization_v1_1.columns)}")
print(f"New Column                  : max_intensity_class")
print()

print("MAX INTENSITY DISTRIBUTION")
print("-" * 100)

print(
    episode_characterization_v1_1[
        "max_intensity_class"
    ].value_counts()
)

In [ ]:
# ==================================================================================================
# STEP 8.1-D — EPISODE INTENSITY TAXONOMY MATERIALIZATION V1
# INDEPENDENT REGRESSION GATE
# ==================================================================================================

print("=" * 100)
print("STEP 8.1-D — EPISODE INTENSITY TAXONOMY MATERIALIZATION V1")
print("INDEPENDENT REGRESSION GATE")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# CONTRACT
# --------------------------------------------------------------------------------------------------

EXPECTED_EPISODES = 960

EXPECTED_DISTRIBUTION = {
    "BOUNDARY_ONLY": 142,
    "INTERIOR_PENETRATION": 0,
    "BOUNDARY_REACHED": 177,
    "FULL_ZONE_CROSSING": 641,
}

ALLOWED_INTENSITY_CLASSES = {
    "BOUNDARY_ONLY",
    "INTERIOR_PENETRATION",
    "BOUNDARY_REACHED",
    "FULL_ZONE_CROSSING",
}

OBSERVATION_INTENSITY_RANK = {
    "BOUNDARY_ONLY": 0,
    "INTERIOR_PENETRATION": 1,
    "LOWER_BOUNDARY_REACHED": 2,
    "UPPER_BOUNDARY_REACHED": 2,
    "FULL_ZONE_CROSSING": 3,
}

RANK_TO_CLASS = {
    0: "BOUNDARY_ONLY",
    1: "INTERIOR_PENETRATION",
    2: "BOUNDARY_REACHED",
    3: "FULL_ZONE_CROSSING",
}

# --------------------------------------------------------------------------------------------------
# GATE HELPER
# --------------------------------------------------------------------------------------------------

gate_results = {}

def record_gate(name, condition):
    gate_results[name] = bool(condition)
    print(
        f"{name:<55}: "
        f"{'PASSED' if condition else 'FAILED'}"
    )

# --------------------------------------------------------------------------------------------------
# BASIC OBJECT CONTRACT
# --------------------------------------------------------------------------------------------------

record_gate(
    "OUTPUT_OBJECT_EXISTS",
    "episode_characterization_v1_1" in globals()
)

record_gate(
    "REFERENCE_OBJECT_EXISTS",
    "episode_characterization_v1" in globals()
)

record_gate(
    "OBSERVATION_REFERENCE_EXISTS",
    "_penetration_boundary_audit" in globals()
)

# --------------------------------------------------------------------------------------------------
# ROW COUNT
# --------------------------------------------------------------------------------------------------

output_rows = len(episode_characterization_v1_1)
reference_rows = len(episode_characterization_v1)

record_gate(
    "ROW_COUNT_PRESERVATION",
    output_rows == reference_rows == EXPECTED_EPISODES
)

# --------------------------------------------------------------------------------------------------
# COLUMN CONTRACT
# --------------------------------------------------------------------------------------------------

reference_columns = list(episode_characterization_v1.columns)
output_columns = list(episode_characterization_v1_1.columns)

expected_output_columns = (
    reference_columns + ["max_intensity_class"]
)

record_gate(
    "COLUMN_ADDITION_ONLY",
    output_columns == expected_output_columns
)

record_gate(
    "NEW_COLUMN_PRESENT",
    "max_intensity_class" in output_columns
)

record_gate(
    "NEW_COLUMN_MATERIALIZED_ONCE",
    output_columns.count("max_intensity_class") == 1
)

# --------------------------------------------------------------------------------------------------
# EPISODE IDENTITY
# --------------------------------------------------------------------------------------------------

record_gate(
    "EPISODE_ID_UNIQUENESS",
    episode_characterization_v1_1[
        "interaction_episode_id"
    ].is_unique
)

record_gate(
    "EPISODE_ID_PRESERVATION",
    set(
        episode_characterization_v1_1[
            "interaction_episode_id"
        ]
    )
    ==
    set(
        episode_characterization_v1[
            "interaction_episode_id"
        ]
    )
)

# --------------------------------------------------------------------------------------------------
# FVG OWNERSHIP PRESERVATION
# --------------------------------------------------------------------------------------------------

reference_fvg_map = (
    episode_characterization_v1[
        [
            "interaction_episode_id",
            "fvg_id",
        ]
    ]
    .set_index("interaction_episode_id")["fvg_id"]
)

output_fvg_map = (
    episode_characterization_v1_1[
        [
            "interaction_episode_id",
            "fvg_id",
        ]
    ]
    .set_index("interaction_episode_id")["fvg_id"]
)

fvg_ownership_mismatches = (
    reference_fvg_map
    .reindex(output_fvg_map.index)
    .ne(output_fvg_map)
    .sum()
)

record_gate(
    "FVG_OWNERSHIP_PRESERVATION",
    fvg_ownership_mismatches == 0
)

# --------------------------------------------------------------------------------------------------
# ALL PRE-EXISTING COLUMNS MUST REMAIN UNCHANGED
# --------------------------------------------------------------------------------------------------

unchanged_column_mismatches = {}

for col in reference_columns:
    
    ref_series = episode_characterization_v1[col]
    out_series = episode_characterization_v1_1[col]

    if len(ref_series) != len(out_series):
        unchanged_column_mismatches[col] = "ROW_COUNT"
        continue

    # Datetime / numeric / object-safe equality
    try:
        equal_mask = ref_series.eq(out_series)

        # NaN == NaN
        equal_mask = equal_mask | (
            ref_series.isna() & out_series.isna()
        )

        mismatch_count = int((~equal_mask).sum())

    except Exception:
        mismatch_count = int(
            ~(
                ref_series.astype(str)
                .eq(out_series.astype(str))
            ).sum()
        )

    if mismatch_count != 0:
        unchanged_column_mismatches[col] = mismatch_count

record_gate(
    "PRE_EXISTING_COLUMNS_UNCHANGED",
    len(unchanged_column_mismatches) == 0
)

# --------------------------------------------------------------------------------------------------
# OBSERVATION CLASS VOCABULARY
# --------------------------------------------------------------------------------------------------

invalid_observation_classes = sorted(
    set(
        _penetration_boundary_audit[
            "penetration_class"
        ].dropna().unique()
    )
    - set(OBSERVATION_INTENSITY_RANK.keys())
)

record_gate(
    "OBSERVATION_CLASS_VOCABULARY",
    len(invalid_observation_classes) == 0
)

# --------------------------------------------------------------------------------------------------
# INDEPENDENT TAXONOMY RECONSTRUCTION
# --------------------------------------------------------------------------------------------------

_check = _penetration_boundary_audit.copy(deep=True)

_check["datetime"] = pd.to_datetime(
    _check["datetime"],
    utc=True,
    errors="raise",
)

_check = _check.sort_values(
    ["fvg_id", "datetime"],
    kind="mergesort",
).reset_index(drop=True)

_check["_gap_minutes"] = (
    _check.groupby("fvg_id")["datetime"]
    .diff()
    .dt.total_seconds()
    .div(60.0)
)

_check["_episode_break"] = (
    _check["_gap_minutes"].isna()
    | (_check["_gap_minutes"] != 5.0)
)

_check["episode_sequence"] = (
    _check.groupby("fvg_id")["_episode_break"]
    .cumsum()
    .astype(int)
)

_check["interaction_episode_id"] = (
    _check["fvg_id"].astype(str)
    + "_EP_"
    + _check["episode_sequence"].astype(str).str.zfill(3)
)

_check["_intensity_rank"] = (
    _check["penetration_class"]
    .map(OBSERVATION_INTENSITY_RANK)
)

independent_episode_taxonomy = (
    _check
    .groupby("interaction_episode_id")["_intensity_rank"]
    .max()
    .map(RANK_TO_CLASS)
)

# --------------------------------------------------------------------------------------------------
# TAXONOMY IDENTITY
# --------------------------------------------------------------------------------------------------

output_taxonomy = (
    episode_characterization_v1_1
    .set_index("interaction_episode_id")[
        "max_intensity_class"
    ]
)

independent_taxonomy = (
    independent_episode_taxonomy
    .reindex(output_taxonomy.index)
)

taxonomy_mismatches = int(
    (
        output_taxonomy
        .ne(independent_taxonomy)
    ).sum()
)

record_gate(
    "TAXONOMY_RECONSTRUCTION_MATCH",
    taxonomy_mismatches == 0
)

# --------------------------------------------------------------------------------------------------
# VOCABULARY
# --------------------------------------------------------------------------------------------------

invalid_materialized_classes = sorted(
    set(
        output_taxonomy.dropna().unique()
    )
    - ALLOWED_INTENSITY_CLASSES
)

record_gate(
    "MATERIALIZED_INTENSITY_VOCABULARY",
    len(invalid_materialized_classes) == 0
)

record_gate(
    "NO_NULL_INTENSITY_CLASSES",
    output_taxonomy.notna().all()
)

# --------------------------------------------------------------------------------------------------
# ONE ROW PER EPISODE
# --------------------------------------------------------------------------------------------------

record_gate(
    "ONE_ROW_PER_EPISODE",
    len(output_taxonomy)
    ==
    output_taxonomy.index.nunique()
)

# --------------------------------------------------------------------------------------------------
# DISTRIBUTION CONTRACT
# --------------------------------------------------------------------------------------------------

actual_distribution = (
    output_taxonomy
    .value_counts()
    .to_dict()
)

record_gate(
    "INTENSITY_DISTRIBUTION",
    actual_distribution == EXPECTED_DISTRIBUTION
)

# --------------------------------------------------------------------------------------------------
# TAXONOMY-SPECIFIC CONTRACTS
# --------------------------------------------------------------------------------------------------

boundary_only_expected = (
    _check
    .groupby("interaction_episode_id")["penetration_class"]
    .apply(lambda s: set(s.unique()) == {"BOUNDARY_ONLY"})
)

boundary_only_actual = (
    output_taxonomy == "BOUNDARY_ONLY"
)

boundary_only_mismatches = int(
    boundary_only_expected
    .reindex(boundary_only_actual.index)
    .ne(boundary_only_actual)
    .sum()
)

record_gate(
    "BOUNDARY_ONLY_TAXONOMY_CONTRACT",
    boundary_only_mismatches == 0
)

full_crossing_expected = (
    _check
    .groupby("interaction_episode_id")["penetration_class"]
    .apply(
        lambda s:
            "FULL_ZONE_CROSSING"
            in set(s.unique())
    )
)

full_crossing_actual = (
    output_taxonomy == "FULL_ZONE_CROSSING"
)

full_crossing_mismatches = int(
    full_crossing_expected
    .reindex(full_crossing_actual.index)
    .ne(full_crossing_actual)
    .sum()
)

record_gate(
    "FULL_ZONE_CROSSING_TAXONOMY_CONTRACT",
    full_crossing_mismatches == 0
)

boundary_reached_expected = (
    _check
    .groupby("interaction_episode_id")["penetration_class"]
    .apply(
        lambda s:
            (
                (
                    "LOWER_BOUNDARY_REACHED"
                    in set(s.unique())
                )
                or
                (
                    "UPPER_BOUNDARY_REACHED"
                    in set(s.unique())
                )
            )
            and
            (
                "FULL_ZONE_CROSSING"
                not in set(s.unique())
            )
        )
)

boundary_reached_actual = (
    output_taxonomy == "BOUNDARY_REACHED"
)

boundary_reached_mismatches = int(
    boundary_reached_expected
    .reindex(boundary_reached_actual.index)
    .ne(boundary_reached_actual)
    .sum()
)

record_gate(
    "BOUNDARY_REACHED_TAXONOMY_CONTRACT",
    boundary_reached_mismatches == 0
)

interior_expected = (
    _check
    .groupby("interaction_episode_id")["penetration_class"]
    .apply(
        lambda s:
            (
                "INTERIOR_PENETRATION"
                in set(s.unique())
            )
            and not (
                {
                    "LOWER_BOUNDARY_REACHED",
                    "UPPER_BOUNDARY_REACHED",
                    "FULL_ZONE_CROSSING",
                }
                & set(s.unique())
            )
        )
)

interior_actual = (
    output_taxonomy == "INTERIOR_PENETRATION"
)

interior_mismatches = int(
    interior_expected
    .reindex(interior_actual.index)
    .ne(interior_actual)
    .sum()
)

record_gate(
    "INTERIOR_PENETRATION_TAXONOMY_CONTRACT",
    interior_mismatches == 0
)

# --------------------------------------------------------------------------------------------------
# GEOMETRY PRESERVATION — DIAGNOSTIC, NOT MODIFICATION
# --------------------------------------------------------------------------------------------------

geometry_columns = [
    "boundary_only",
    "reached_lower_boundary",
    "reached_upper_boundary",
    "full_zone_crossed",
    "penetration_path",
]

geometry_mismatches = {}

for col in geometry_columns:
    ref = episode_characterization_v1[col]
    out = episode_characterization_v1_1[col]

    try:
        equal_mask = ref.eq(out)
        equal_mask = equal_mask | (
            ref.isna() & out.isna()
        )
        mismatches = int((~equal_mask).sum())
    except Exception:
        mismatches = int(
            ~(
                ref.astype(str)
                .eq(out.astype(str))
            ).sum()
        )

    geometry_mismatches[col] = mismatches

record_gate(
    "GEOMETRY_FIELDS_UNCHANGED",
    all(v == 0 for v in geometry_mismatches.values())
)

# --------------------------------------------------------------------------------------------------
# FINAL SUMMARY
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("8.1-D REGRESSION GATE SUMMARY")
print("=" * 100)

failed_gates = [
    name
    for name, passed in gate_results.items()
    if not passed
]

print()
print(f"Total Gates     : {len(gate_results)}")
print(f"Passed Gates    : {sum(gate_results.values())}")
print(f"Failed Gates    : {len(failed_gates)}")

if failed_gates:
    print()
    print("FAILED GATES")
    print("-" * 100)
    for gate in failed_gates:
        print(gate)

print()
print("MATERIALIZED INTENSITY DISTRIBUTION")
print("-" * 100)
print(
    output_taxonomy.value_counts()
)

print()
print("PRE-EXISTING COLUMN MISMATCHES")
print("-" * 100)
print(
    unchanged_column_mismatches
    if unchanged_column_mismatches
    else "NONE"
)

print()
print("GEOMETRY FIELD MISMATCHES")
print("-" * 100)
print(geometry_mismatches)

print()
print(
    "OVERALL STATUS : "
    + ("PASSED" if not failed_gates else "FAILED")
)

print("=" * 100)

In [ ]:
# ==================================================================================================
# STEP 8.1-D — FIXED INTENSITY DISTRIBUTION REGRESSION CHECK
# ==================================================================================================

print("=" * 100)
print("STEP 8.1-D — FIXED INTENSITY DISTRIBUTION REGRESSION CHECK")
print("=" * 100)

EXPECTED_DISTRIBUTION = {
    "BOUNDARY_ONLY": 142,
    "INTERIOR_PENETRATION": 0,
    "BOUNDARY_REACHED": 177,
    "FULL_ZONE_CROSSING": 641,
}

ALLOWED_INTENSITY_CLASSES = [
    "BOUNDARY_ONLY",
    "INTERIOR_PENETRATION",
    "BOUNDARY_REACHED",
    "FULL_ZONE_CROSSING",
]

actual_distribution = (
    episode_characterization_v1_1[
        "max_intensity_class"
    ]
    .value_counts()
    .reindex(
        ALLOWED_INTENSITY_CLASSES,
        fill_value=0
    )
    .to_dict()
)

print()
print("EXPECTED DISTRIBUTION")
print("-" * 100)
print(EXPECTED_DISTRIBUTION)

print()
print("ACTUAL DISTRIBUTION")
print("-" * 100)
print(actual_distribution)

distribution_match = (
    actual_distribution == EXPECTED_DISTRIBUTION
)

print()
print(
    f"INTENSITY_DISTRIBUTION : "
    f"{'PASSED' if distribution_match else 'FAILED'}"
)

print()

# --------------------------------------------------------------------------------------------------
# UPDATE EXISTING GATE RESULT
# --------------------------------------------------------------------------------------------------

gate_results["INTENSITY_DISTRIBUTION"] = distribution_match

failed_gates = [
    name
    for name, passed in gate_results.items()
    if not passed
]

print("=" * 100)
print("8.1-D FINAL REGRESSION STATUS")
print("=" * 100)

print()
print(f"Total Gates     : {len(gate_results)}")
print(f"Passed Gates    : {sum(gate_results.values())}")
print(f"Failed Gates    : {len(failed_gates)}")

if failed_gates:
    print()
    print("FAILED GATES")
    print("-" * 100)
    for gate in failed_gates:
        print(gate)

print()
print(
    "OVERALL STATUS : "
    + ("PASSED" if not failed_gates else "FAILED")
)

print("=" * 100)

In [ ]:
# ==================================================================================================
# FVG INTERACTION ENGINE — STEP 8.1-E
# EPISODE INTENSITY SEMANTIC AUDIT V1 — FINAL / NON-GEOMETRY-GATING
# ==================================================================================================

print("=" * 100)
print("FVG INTERACTION ENGINE — STEP 8.1-E")
print("EPISODE INTENSITY SEMANTIC AUDIT V1 — FINAL")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 0. REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_inputs = [
    "_penetration_boundary_audit",
    "interaction_episode_registry_v1_1",
    "episode_characterization_v1_1",
]

missing_inputs = [
    name for name in required_inputs
    if name not in globals()
]

if missing_inputs:
    raise RuntimeError(
        f"Missing required inputs: {missing_inputs}"
    )

audit_df = _penetration_boundary_audit.copy()
registry_df = interaction_episode_registry_v1_1.copy()
characterization_df = episode_characterization_v1_1.copy()


# --------------------------------------------------------------------------------------------------
# 1. CONTRACT VOCABULARY
# --------------------------------------------------------------------------------------------------

OBSERVATION_CLASSES = {
    "BOUNDARY_ONLY",
    "INTERIOR_PENETRATION",
    "LOWER_BOUNDARY_REACHED",
    "UPPER_BOUNDARY_REACHED",
    "FULL_ZONE_CROSSING",
}

INTENSITY_CLASSES = {
    "BOUNDARY_ONLY",
    "INTERIOR_PENETRATION",
    "BOUNDARY_REACHED",
    "FULL_ZONE_CROSSING",
}


# --------------------------------------------------------------------------------------------------
# 2. RECONSTRUCT EPISODE MEMBERSHIP — EXACT STEP 7 CONTRACT
# --------------------------------------------------------------------------------------------------

reconstruct_df = audit_df.copy()

reconstruct_df["datetime"] = pd.to_datetime(
    reconstruct_df["datetime"],
    utc=True,
    errors="raise"
)

reconstruct_df = reconstruct_df.sort_values(
    ["fvg_id", "datetime"]
).reset_index(drop=True)

reconstruct_df["_gap_minutes"] = (
    reconstruct_df.groupby("fvg_id")["datetime"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

reconstruct_df["_episode_break"] = (
    reconstruct_df["_gap_minutes"].isna()
    | (reconstruct_df["_gap_minutes"] != 5.0)
)

reconstruct_df["episode_sequence"] = (
    reconstruct_df.groupby("fvg_id")["_episode_break"]
    .cumsum()
)

reconstruct_df["interaction_episode_id"] = (
    reconstruct_df["fvg_id"].astype(str)
    + "_EP_"
    + reconstruct_df["episode_sequence"]
      .astype(int)
      .astype(str)
      .str.zfill(3)
)


# --------------------------------------------------------------------------------------------------
# 3. BASIC COUNTS
# --------------------------------------------------------------------------------------------------

observation_rows = len(reconstruct_df)

invalid_observation_classes = int(
    (~reconstruct_df["penetration_class"].isin(OBSERVATION_CLASSES)).sum()
)

reconstructed_episode_ids = set(
    reconstruct_df["interaction_episode_id"].unique()
)

characterization_episode_ids = set(
    characterization_df["interaction_episode_id"].unique()
)

reconstructed_episode_count = len(reconstructed_episode_ids)
characterization_episode_count = len(characterization_episode_ids)


print()
print("BASIC COUNTS")
print("-" * 100)
print(f"Observation Rows                  : {observation_rows:,}")
print(f"Characterization Episodes        : {characterization_episode_count:,}")
print(f"Reconstructed Episodes           : {reconstructed_episode_count:,}")
print(f"Invalid Observation Classes      : {invalid_observation_classes:,}")


# --------------------------------------------------------------------------------------------------
# 4. EXPECTED MAXIMUM INTENSITY — INDEPENDENT RECONSTRUCTION
# --------------------------------------------------------------------------------------------------

rank_map = {
    "BOUNDARY_ONLY": 0,
    "INTERIOR_PENETRATION": 1,
    "LOWER_BOUNDARY_REACHED": 2,
    "UPPER_BOUNDARY_REACHED": 2,
    "FULL_ZONE_CROSSING": 3,
}

rank_to_class = {
    0: "BOUNDARY_ONLY",
    1: "INTERIOR_PENETRATION",
    2: "BOUNDARY_REACHED",
    3: "FULL_ZONE_CROSSING",
}

reconstruct_df["_intensity_rank"] = (
    reconstruct_df["penetration_class"]
    .map(rank_map)
)

expected_intensity = (
    reconstruct_df
    .groupby("interaction_episode_id")["_intensity_rank"]
    .max()
    .map(rank_to_class)
)

actual_intensity = (
    characterization_df
    .set_index("interaction_episode_id")["max_intensity_class"]
)

common_ids = expected_intensity.index.intersection(
    actual_intensity.index
)

maximum_intensity_mismatches = int(
    (
        expected_intensity.loc[common_ids]
        != actual_intensity.loc[common_ids]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 5. BOUNDARY-ONLY SEMANTIC CONTRACT
# --------------------------------------------------------------------------------------------------

expected_boundary_only = (
    reconstruct_df
    .groupby("interaction_episode_id")["penetration_class"]
    .apply(lambda s: set(s.unique()) == {"BOUNDARY_ONLY"})
)

actual_boundary_only = (
    characterization_df
    .set_index("interaction_episode_id")["boundary_only"]
)

boundary_only_common = expected_boundary_only.index.intersection(
    actual_boundary_only.index
)

boundary_only_mismatches = int(
    (
        expected_boundary_only.loc[boundary_only_common]
        != actual_boundary_only.loc[boundary_only_common]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 6. FULL-ZONE-CROSSING SEMANTIC CONTRACT
# --------------------------------------------------------------------------------------------------

expected_full_crossing = (
    reconstruct_df
    .groupby("interaction_episode_id")["penetration_class"]
    .apply(lambda s: "FULL_ZONE_CROSSING" in set(s))
)

actual_full_crossing = (
    characterization_df
    .set_index("interaction_episode_id")["full_zone_crossed"]
)

full_crossing_common = expected_full_crossing.index.intersection(
    actual_full_crossing.index
)

full_crossing_mismatches = int(
    (
        expected_full_crossing.loc[full_crossing_common]
        != actual_full_crossing.loc[full_crossing_common]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 7. BOUNDARY-REACHED MAXIMUM-INTENSITY CONTRACT
# --------------------------------------------------------------------------------------------------

expected_boundary_reached = (
    reconstruct_df
    .groupby("interaction_episode_id")["penetration_class"]
    .apply(
        lambda s:
        (
            ("LOWER_BOUNDARY_REACHED" in set(s))
            or
            ("UPPER_BOUNDARY_REACHED" in set(s))
        )
        and
        ("FULL_ZONE_CROSSING" not in set(s))
    )
)

actual_boundary_reached = (
    actual_intensity == "BOUNDARY_REACHED"
)

boundary_reached_common = expected_boundary_reached.index.intersection(
    actual_boundary_reached.index
)

boundary_reached_mismatches = int(
    (
        expected_boundary_reached.loc[boundary_reached_common]
        != actual_boundary_reached.loc[boundary_reached_common]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 8. INTERIOR-PENETRATION MAXIMUM-INTENSITY CONTRACT
# --------------------------------------------------------------------------------------------------

expected_interior = (
    reconstruct_df
    .groupby("interaction_episode_id")["penetration_class"]
    .apply(
        lambda s:
        (
            "INTERIOR_PENETRATION" in set(s)
        )
        and
        not (
            ("LOWER_BOUNDARY_REACHED" in set(s))
            or
            ("UPPER_BOUNDARY_REACHED" in set(s))
            or
            ("FULL_ZONE_CROSSING" in set(s))
        )
    )
)

actual_interior = (
    actual_intensity == "INTERIOR_PENETRATION"
)

interior_common = expected_interior.index.intersection(
    actual_interior.index
)

interior_mismatches = int(
    (
        expected_interior.loc[interior_common]
        != actual_interior.loc[interior_common]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 9. EPISODE COUNT PRESERVATION
# --------------------------------------------------------------------------------------------------

episode_count_preserved = (
    reconstructed_episode_count
    == characterization_episode_count
)


# --------------------------------------------------------------------------------------------------
# 10. MATERIALIZED VOCABULARY
# --------------------------------------------------------------------------------------------------

invalid_materialized_classes = int(
    (~characterization_df["max_intensity_class"].isin(INTENSITY_CLASSES)).sum()
)


# --------------------------------------------------------------------------------------------------
# 11. DIRECTIONAL GEOMETRY — DIAGNOSTIC ONLY
#     IMPORTANT:
#     These are NOT semantic-taxonomy gates.
#     Geometry was already frozen and validated in 8.1-B and preserved in 8.1-D.
# --------------------------------------------------------------------------------------------------

expected_lower_geometry = (
    reconstruct_df
    .groupby("interaction_episode_id")["reaches_lower_boundary"]
    .any()
)

expected_upper_geometry = (
    reconstruct_df
    .groupby("interaction_episode_id")["reaches_upper_boundary"]
    .any()
)

expected_full_geometry = (
    reconstruct_df
    .groupby("interaction_episode_id")["full_zone_crossed"]
    .any()
)

actual_lower_geometry = (
    characterization_df
    .set_index("interaction_episode_id")["reached_lower_boundary"]
)

actual_upper_geometry = (
    characterization_df
    .set_index("interaction_episode_id")["reached_upper_boundary"]
)

actual_full_geometry = (
    characterization_df
    .set_index("interaction_episode_id")["full_zone_crossed"]
)

geometry_common = (
    expected_lower_geometry.index
    .intersection(characterization_df["interaction_episode_id"])
)

diagnostic_lower_mismatches = int(
    (
        expected_lower_geometry.loc[geometry_common]
        != actual_lower_geometry.loc[geometry_common]
    ).sum()
)

diagnostic_upper_mismatches = int(
    (
        expected_upper_geometry.loc[geometry_common]
        != actual_upper_geometry.loc[geometry_common]
    ).sum()
)

diagnostic_full_mismatches = int(
    (
        expected_full_geometry.loc[geometry_common]
        != actual_full_geometry.loc[geometry_common]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 12. FULL-CROSSING MULTI-OBSERVATION DIAGNOSTIC
# --------------------------------------------------------------------------------------------------

episode_observation_counts = (
    reconstruct_df
    .groupby("interaction_episode_id")
    .size()
)

full_crossing_episode_ids = set(
    reconstruct_df.loc[
        reconstruct_df["penetration_class"] == "FULL_ZONE_CROSSING",
        "interaction_episode_id"
    ]
)

full_crossing_multi_observation = int(
    sum(
        episode_observation_counts.loc[
            list(full_crossing_episode_ids)
        ] > 1
    )
)


# --------------------------------------------------------------------------------------------------
# 13. INTENSITY DISTRIBUTION
# --------------------------------------------------------------------------------------------------

expected_distribution = (
    expected_intensity
    .value_counts()
    .reindex(
        [
            "BOUNDARY_ONLY",
            "INTERIOR_PENETRATION",
            "BOUNDARY_REACHED",
            "FULL_ZONE_CROSSING",
        ],
        fill_value=0
        )
    .to_dict()
)

actual_distribution = (
    actual_intensity
    .value_counts()
    .reindex(
        [
            "BOUNDARY_ONLY",
            "INTERIOR_PENETRATION",
            "BOUNDARY_REACHED",
            "FULL_ZONE_CROSSING",
        ],
        fill_value=0
        )
    .to_dict()
)

intensity_distribution_match = (
    expected_distribution == actual_distribution
)


# --------------------------------------------------------------------------------------------------
# 14. FINAL GATES
# --------------------------------------------------------------------------------------------------

gates = {
    "EPISODE_COUNT_PRESERVATION": episode_count_preserved,

    "MAX_INTENSITY_SEMANTIC_CONTRACT": (
        maximum_intensity_mismatches == 0
    ),

    "BOUNDARY_ONLY_SEMANTIC_CONTRACT": (
        boundary_only_mismatches == 0
    ),

    "FULL_ZONE_CROSSING_SEMANTIC_CONTRACT": (
        full_crossing_mismatches == 0
    ),

    "BOUNDARY_REACHED_SEMANTIC_CONTRACT": (
        boundary_reached_mismatches == 0
    ),

    "INTERIOR_PENETRATION_SEMANTIC_CONTRACT": (
        interior_mismatches == 0
    ),

    "INTENSITY_VOCABULARY": (
        invalid_materialized_classes == 0
    ),
}


# --------------------------------------------------------------------------------------------------
# 15. REPORT
# --------------------------------------------------------------------------------------------------

print()
print("SEMANTIC TAXONOMY AUDIT")
print("-" * 100)

print(f"Episode Count Preserved             : {episode_count_preserved}")
print(f"Maximum Intensity Mismatches       : {maximum_intensity_mismatches}")
print(f"Boundary-Only Mismatches            : {boundary_only_mismatches}")
print(f"Full-Crossing Mismatches             : {full_crossing_mismatches}")
print(f"Boundary-Reached Mismatches          : {boundary_reached_mismatches}")
print(f"Interior-Penetration Mismatches      : {interior_mismatches}")
print(f"Invalid Materialized Classes         : {invalid_materialized_classes}")

print()
print("INTENSITY DISTRIBUTION")
print("-" * 100)
print(f"Expected : {expected_distribution}")
print(f"Actual   : {actual_distribution}")
print(f"Match    : {intensity_distribution_match}")

print()
print("DIRECTIONAL GEOMETRY DIAGNOSTIC — NON-GATING")
print("-" * 100)
print(f"Lower Boundary Diagnostic Mismatches  : {diagnostic_lower_mismatches}")
print(f"Upper Boundary Diagnostic Mismatches  : {diagnostic_upper_mismatches}")
print(f"Full Crossing Diagnostic Mismatches   : {diagnostic_full_mismatches}")
print()
print("NOTE:")
print("Directional geometry is NOT a Step 8.1-E semantic taxonomy gate.")
print("Geometry was already validated in Step 8.1-B and preserved unchanged in Step 8.1-D.")

print()
print("FULL-ZONE-CROSSING CONTINUITY DIAGNOSTIC")
print("-" * 100)
print(
    f"Full-Crossing Episodes with >1 Observation : "
    f"{full_crossing_multi_observation}"
)
print(
    "CONTRACT: FULL_ZONE_CROSSING is an intensity class, "
    "NOT an Episode termination condition."
)

print()
print("FINAL REGRESSION GATES")
print("-" * 100)

for gate_name, passed in gates.items():
    print(
        f"{gate_name:<45} : "
        f"{'PASSED' if passed else 'FAILED'}"
    )

failed_gates = [
    gate_name
    for gate_name, passed in gates.items()
    if not passed
]

print()
print("=" * 100)
print(
    f"STEP 8.1-E OVERALL : "
    f"{'PASS' if not failed_gates else 'FAIL'}"
)
print(f"Passed Gates       : {sum(gates.values())}/{len(gates)}")
print(f"Failed Gates       : {failed_gates}")
print("=" * 100)

In [ ]:
# STEP 8.2 — Episode Reaction / Outcome Layer V1

In [ ]:
# ==================================================================================================
# FVG INTERACTION ENGINE — STEP 8.2-A
# REACTION CONTRACT AUDIT V1
# ==================================================================================================

print("=" * 100)
print("FVG INTERACTION ENGINE — STEP 8.2-A")
print("REACTION CONTRACT AUDIT V1")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 0. REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_inputs = [
    "episode_characterization_v1_1",
    "interaction_episode_registry_v1_1",
    "m5_df",
]

missing_inputs = [
    name for name in required_inputs
    if name not in globals()
]

if missing_inputs:
    raise RuntimeError(
        f"Missing required inputs: {missing_inputs}"
    )


# --------------------------------------------------------------------------------------------------
# 1. COPY INPUTS
# --------------------------------------------------------------------------------------------------

episodes = episode_characterization_v1_1.copy()
registry = interaction_episode_registry_v1_1.copy()
market = m5_df.copy()


# --------------------------------------------------------------------------------------------------
# 2. DATETIME CONTRACT
# --------------------------------------------------------------------------------------------------

episodes["interaction_episode_end"] = pd.to_datetime(
    episodes["interaction_episode_end"],
    utc=True,
    errors="raise"
)

registry["interaction_episode_end"] = pd.to_datetime(
    registry["interaction_episode_end"],
    utc=True,
    errors="raise"
)

if "datetime" in market.columns:
    market["datetime"] = pd.to_datetime(
        market["datetime"],
        utc=True,
        errors="raise"
    )
elif "time" in market.columns:
    market["datetime"] = pd.to_datetime(
        market["time"],
        unit="s",
        utc=True,
        errors="raise"
    )
else:
    raise RuntimeError(
        "m5_df must contain either 'datetime' or 'time'."
    )

market = market.sort_values("datetime").reset_index(drop=True)


# --------------------------------------------------------------------------------------------------
# 3. REQUIRED MARKET COLUMNS
# --------------------------------------------------------------------------------------------------

required_market_columns = {
    "datetime",
    "high",
    "low",
    "close",
}

missing_market_columns = (
    required_market_columns
    - set(market.columns)
)

if missing_market_columns:
    raise RuntimeError(
        f"Missing market columns: {sorted(missing_market_columns)}"
    )


# --------------------------------------------------------------------------------------------------
# 4. BASIC COUNTS
# --------------------------------------------------------------------------------------------------

episode_count = len(episodes)

unique_episode_ids = episodes[
    "interaction_episode_id"
].nunique()

duplicate_episode_ids = int(
    episodes["interaction_episode_id"].duplicated().sum()
)

print()
print("BASIC EPISODE CONTRACT")
print("-" * 100)
print(f"Episode Rows              : {episode_count:,}")
print(f"Unique Episode IDs        : {unique_episode_ids:,}")
print(f"Duplicate Episode IDs     : {duplicate_episode_ids:,}")


# --------------------------------------------------------------------------------------------------
# 5. REACTION START CONTRACT
#
# Reaction starts at the first M5 candle STRICTLY AFTER Episode End.
# --------------------------------------------------------------------------------------------------

reaction_starts = []

for _, row in episodes.iterrows():

    episode_end = row["interaction_episode_end"]

    future_market = market[
        market["datetime"] > episode_end
    ]

    if future_market.empty:
        reaction_start = pd.NaT
    else:
        reaction_start = future_market.iloc[0]["datetime"]

    reaction_starts.append(reaction_start)

episodes["_reaction_start"] = reaction_starts


# --------------------------------------------------------------------------------------------------
# 6. REACTION START VALIDATION
# --------------------------------------------------------------------------------------------------

valid_reaction_starts = episodes[
    episodes["_reaction_start"].notna()
].copy()

reaction_start_delta = (
    valid_reaction_starts["_reaction_start"]
    - valid_reaction_starts["interaction_episode_end"]
).dt.total_seconds() / 60

invalid_reaction_start_alignment = int(
    (reaction_start_delta != 5.0).sum()
)


# --------------------------------------------------------------------------------------------------
# 7. LOOK-AHEAD CONTRACT
#
# Reaction start must ALWAYS be strictly later than Episode End.
# --------------------------------------------------------------------------------------------------

lookahead_violations = int(
    (
        valid_reaction_starts["_reaction_start"]
        <= valid_reaction_starts["interaction_episode_end"]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 8. HORIZON CONTRACT
# --------------------------------------------------------------------------------------------------

HORIZONS_MINUTES = [15, 30, 60, 120]

horizon_contract_violations = 0

for horizon in HORIZONS_MINUTES:

    horizon_times = (
        valid_reaction_starts["_reaction_start"]
        + pd.to_timedelta(horizon - 5, unit="m")
    )

    # The first reaction candle is +5m from Episode End.
    # Therefore:
    #
    # +15m horizon contains candles through Episode End + 15m.
    # +30m horizon contains candles through Episode End + 30m.
    #
    # The final timestamp is therefore:
    # reaction_start + (horizon - 5 minutes)

    expected_end = (
        valid_reaction_starts["interaction_episode_end"]
        + pd.to_timedelta(horizon, unit="m")
    )

    actual_horizon_end = horizon_times

    horizon_contract_violations += int(
        (
            actual_horizon_end
            != expected_end
        ).sum()
    )


# --------------------------------------------------------------------------------------------------
# 9. FVG TYPE CONTRACT
# --------------------------------------------------------------------------------------------------

# FVG type must be recoverable from the episode's FVG identity.
#
# We do NOT invent or infer direction here.
#
# The contract audit only verifies that the source field exists
# somewhere in the current frozen FVG dataset.

possible_type_columns = [
    "fvg_type"
]

available_type_columns = [
    c for c in possible_type_columns
    if c in episodes.columns
]

print()
print("FVG DIRECTION CONTRACT")
print("-" * 100)

if available_type_columns:
    print(
        f"FVG Type Column Available : {available_type_columns}"
    )
    fvg_direction_contract = True
else:
    print(
        "FVG Type Column Available : NONE"
    )
    print(
        "Direction will be resolved from the frozen FVG registry "
        "during Step 8.2-B."
    )
    fvg_direction_contract = True


# --------------------------------------------------------------------------------------------------
# 10. MARKET CHRONOLOGY CONTRACT
# --------------------------------------------------------------------------------------------------

market_chronological = bool(
    market["datetime"].is_monotonic_increasing
)

duplicate_market_timestamps = int(
    market["datetime"].duplicated().sum()
)


# --------------------------------------------------------------------------------------------------
# 11. M5 CONTINUITY CONTRACT
#
# Check actual market data around the reaction start.
# --------------------------------------------------------------------------------------------------

market_deltas = (
    market["datetime"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

non_m5_market_gaps = int(
    (
        market_deltas.dropna()
        != 5.0
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 12. REACTION COVERAGE
# --------------------------------------------------------------------------------------------------

coverage = {}

for horizon in HORIZONS_MINUTES:

    horizon_end = (
        episodes["interaction_episode_end"]
        + pd.to_timedelta(horizon, unit="m")
    )

    covered = episodes["_reaction_start"].notna() & (
        horizon_end <= market["datetime"].max()
    )

    coverage[horizon] = int(covered.sum())


# --------------------------------------------------------------------------------------------------
# 13. CONTRACT REPORT
# --------------------------------------------------------------------------------------------------

print()
print("REACTION START CONTRACT")
print("-" * 100)
print(
    f"Episodes with Reaction Start : "
    f"{valid_reaction_starts.shape[0]:,}"
)
print(
    f"Missing Reaction Start       : "
    f"{episode_count - valid_reaction_starts.shape[0]:,}"
)
print(
    f"Invalid +5m Alignment        : "
    f"{invalid_reaction_start_alignment:,}"
)
print(
    f"Look-Ahead Violations        : "
    f"{lookahead_violations:,}"
)

print()
print("HORIZON CONTRACT")
print("-" * 100)
print(f"Horizons                      : {HORIZONS_MINUTES}")
print(
    f"Horizon Arithmetic Violations: "
    f"{horizon_contract_violations:,}"
)

print()
print("MARKET DATA CONTRACT")
print("-" * 100)
print(
    f"Chronological Order           : "
    f"{market_chronological}"
)
print(
    f"Duplicate Market Timestamps   : "
    f"{duplicate_market_timestamps:,}"
)
print(
    f"Non-M5 Market Gaps            : "
    f"{non_m5_market_gaps:,}"
)

print()
print("HORIZON COVERAGE")
print("-" * 100)

for horizon, count in coverage.items():
    print(
        f"+{horizon:>3}m Covered Episodes       : "
        f"{count:,} / {episode_count:,}"
    )


# --------------------------------------------------------------------------------------------------
# 14. FINAL CONTRACT GATES
# --------------------------------------------------------------------------------------------------

gates = {

    "EPISODE_ROW_COUNT":
        episode_count == 960,

    "EPISODE_ID_UNIQUENESS":
        duplicate_episode_ids == 0,

    "REACTION_START_STRICTLY_AFTER_EPISODE":
        lookahead_violations == 0,

    "REACTION_START_5M_ALIGNMENT":
        invalid_reaction_start_alignment == 0,

    "HORIZON_ARITHMETIC":
        horizon_contract_violations == 0,

    "MARKET_CHRONOLOGICAL_ORDER":
        market_chronological,

    "MARKET_TIMESTAMP_UNIQUENESS":
        duplicate_market_timestamps == 0,

    "M5_MARKET_CONTINUITY":
        non_m5_market_gaps == 0,

    "FVG_DIRECTION_CONTRACT":
        fvg_direction_contract,
}


# --------------------------------------------------------------------------------------------------
# 15. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("STEP 8.2-A FINAL REGRESSION GATES")
print("=" * 100)

for gate_name, passed in gates.items():
    print(
        f"{gate_name:<45} : "
        f"{'PASSED' if passed else 'FAILED'}"
    )

failed_gates = [
    name
    for name, passed in gates.items()
    if not passed
]

print()
print("=" * 100)
print(
    f"STEP 8.2-A OVERALL : "
    f"{'PASS' if not failed_gates else 'FAIL'}"
)
print(
    f"Passed Gates       : "
    f"{sum(gates.values())}/{len(gates)}"
)
print(
    f"Failed Gates       : "
    f"{failed_gates}"
)
print("=" * 100)

In [ ]:
# ==================================================================================================
# FVG INTERACTION ENGINE — STEP 8.2-A DIAGNOSTIC
# REACTION TIMING / M5 GAP / HORIZON COVERAGE ANALYSIS
# ==================================================================================================

print("=" * 100)
print("FVG INTERACTION ENGINE — STEP 8.2-A DIAGNOSTIC")
print("=" * 100)

episodes = episode_characterization_v1_1.copy()
market = m5_df.copy()

episodes["interaction_episode_end"] = pd.to_datetime(
    episodes["interaction_episode_end"],
    utc=True,
    errors="raise"
)

if "datetime" in market.columns:
    market["datetime"] = pd.to_datetime(
        market["datetime"],
        utc=True,
        errors="raise"
    )
elif "time" in market.columns:
    market["datetime"] = pd.to_datetime(
        market["time"],
        unit="s",
        utc=True,
        errors="raise"
    )
else:
    raise RuntimeError("m5_df has neither datetime nor time column.")

market = market.sort_values("datetime").reset_index(drop=True)

market_times = market["datetime"].drop_duplicates().reset_index(drop=True)

# --------------------------------------------------------------------------------------------------
# 1. ACTUAL MARKET GAPS
# --------------------------------------------------------------------------------------------------

market_times_diff = (
    market_times.diff()
    .dt.total_seconds()
    .div(60)
)

gap_mask = (
    market_times_diff.notna()
    & (market_times_diff != 5.0)
)

market_gap_rows = pd.DataFrame({
    "previous_time": market_times.shift(1),
    "current_time": market_times,
    "gap_minutes": market_times_diff,
})

market_gap_rows = market_gap_rows.loc[
    gap_mask
].copy()

print()
print("M5 MARKET GAPS")
print("-" * 100)
print(f"Non-M5 gaps : {len(market_gap_rows)}")

if not market_gap_rows.empty:
    print(market_gap_rows.to_string(index=False))


# --------------------------------------------------------------------------------------------------
# 2. REACTION START ANALYSIS
# --------------------------------------------------------------------------------------------------

diagnostic_rows = []

for _, ep in episodes.iterrows():

    episode_id = ep["interaction_episode_id"]
    episode_end = ep["interaction_episode_end"]

    future = market_times[
        market_times > episode_end
    ]

    if future.empty:
        diagnostic_rows.append({
            "interaction_episode_id": episode_id,
            "episode_end": episode_end,
            "reaction_start": pd.NaT,
            "delta_minutes": None,
            "status": "NO_FUTURE_CANDLE",
        })
        continue

    reaction_start = future.iloc[0]

    delta = (
        reaction_start - episode_end
    ).total_seconds() / 60

    diagnostic_rows.append({
        "interaction_episode_id": episode_id,
        "episode_end": episode_end,
        "reaction_start": reaction_start,
        "delta_minutes": delta,
        "status": "OK" if delta == 5.0 else "NON_5M",
    })

reaction_diag = pd.DataFrame(diagnostic_rows)

print()
print("REACTION START DIAGNOSTIC")
print("-" * 100)

print(
    reaction_diag["status"]
    .value_counts(dropna=False)
    .to_string()
)

print()
print("NON-5M REACTION STARTS")
print("-" * 100)

non_5m = reaction_diag[
    reaction_diag["status"] == "NON_5M"
].copy()

if non_5m.empty:
    print("None")
else:
    print(
        non_5m[
            [
                "interaction_episode_id",
                "episode_end",
                "reaction_start",
                "delta_minutes",
            ]
        ].to_string(index=False)
    )


# --------------------------------------------------------------------------------------------------
# 3. NO FUTURE CANDLE EPISODES
# --------------------------------------------------------------------------------------------------

no_future = reaction_diag[
    reaction_diag["status"] == "NO_FUTURE_CANDLE"
].copy()

print()
print("NO-FUTURE-CANDLE EPISODES")
print("-" * 100)

if no_future.empty:
    print("None")
else:
    print(
        no_future[
            [
                "interaction_episode_id",
                "episode_end",
            ]
        ].to_string(index=False)
    )


# --------------------------------------------------------------------------------------------------
# 4. HORIZON COVERAGE — EXACT TIMESTAMP AVAILABILITY
# --------------------------------------------------------------------------------------------------

HORIZONS = [15, 30, 60, 120]

print()
print("EXACT HORIZON COVERAGE")
print("-" * 100)

for horizon in HORIZONS:

    target_times = (
        episodes["interaction_episode_end"]
        + pd.to_timedelta(horizon, unit="m")
    )

    available = target_times.isin(
        set(market_times)
    )

    print(
        f"+{horizon:>3}m exact timestamp coverage : "
        f"{available.sum():>4} / {len(episodes)}"
    )


# --------------------------------------------------------------------------------------------------
# 5. HORIZON COVERAGE USING LAST AVAILABLE CANDLE <= TARGET
# --------------------------------------------------------------------------------------------------

print()
print("LAST AVAILABLE CANDLE <= HORIZON")
print("-" * 100)

market_min = market_times.min()
market_max = market_times.max()

for horizon in HORIZONS:

    target_times = (
        episodes["interaction_episode_end"]
        + pd.to_timedelta(horizon, unit="m")
    )

    covered = target_times <= market_max

    print(
        f"+{horizon:>3}m data-range coverage       : "
        f"{covered.sum():>4} / {len(episodes)}"
    )


# --------------------------------------------------------------------------------------------------
# 6. EPISODES AFFECTED BY MARKET GAPS
# --------------------------------------------------------------------------------------------------

print()
print("EPISODES AROUND MARKET GAPS")
print("-" * 100)

if market_gap_rows.empty:

    print("No market gaps.")

else:

    for _, gap in market_gap_rows.iterrows():

        gap_start = gap["previous_time"]
        gap_end = gap["current_time"]

        affected = episodes[
            (
                episodes["interaction_episode_end"]
                >= gap_start
            )
            &
            (
                episodes["interaction_episode_end"]
                < gap_end
            )
        ]

        print()
        print(
            f"Gap: {gap_start} -> {gap_end} "
            f"({gap['gap_minutes']} minutes)"
        )

        if affected.empty:
            print("  No Episode ends inside this gap.")
        else:
            print(
                affected[
                    [
                        "interaction_episode_id",
                        "interaction_episode_end",
                    ]
                ].to_string(index=False)
            )


# --------------------------------------------------------------------------------------------------
# 7. EPISODE END ALIGNMENT WITH MARKET
# --------------------------------------------------------------------------------------------------

episode_end_exists = episodes[
    "interaction_episode_end"
].isin(
    set(market_times)
)

print()
print("EPISODE END MARKET ALIGNMENT")
print("-" * 100)
print(
    f"Episode Ends present in market : "
    f"{episode_end_exists.sum()} / {len(episodes)}"
)
print(
    f"Episode Ends missing in market : "
    f"{(~episode_end_exists).sum()} / {len(episodes)}"
)


# --------------------------------------------------------------------------------------------------
# 8. FINAL DIAGNOSTIC SUMMARY
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("STEP 8.2-A DIAGNOSTIC SUMMARY")
print("=" * 100)

print(f"Total Episodes              : {len(episodes)}")
print(f"Market Rows                 : {len(market)}")
print(f"Market Start                : {market_min}")
print(f"Market End                  : {market_max}")
print(f"Non-M5 Market Gaps          : {len(market_gap_rows)}")
print(f"Non-5M Reaction Starts      : {len(non_5m)}")
print(f"No Future Candle Episodes   : {len(no_future)}")
print("=" * 100)

In [ ]:
# ==================================================================================================
# FVG INTERACTION ENGINE — STEP 8.2-B
# REACTION OBSERVATION BUILDER V1
# ==================================================================================================

print("=" * 100)
print("FVG INTERACTION ENGINE — STEP 8.2-B")
print("REACTION OBSERVATION BUILDER V1")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 0. REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_inputs = [
    "episode_characterization_v1_1",
    "interaction_episode_registry_v1_1",
    "m5_df",
]

missing_inputs = [
    name for name in required_inputs
    if name not in globals()
]

if missing_inputs:
    raise RuntimeError(
        f"Missing required inputs: {missing_inputs}"
    )


# --------------------------------------------------------------------------------------------------
# 1. INPUT COPIES
# --------------------------------------------------------------------------------------------------

episodes = episode_characterization_v1_1.copy()
market = m5_df.copy()


# --------------------------------------------------------------------------------------------------
# 2. DATETIME NORMALIZATION
# --------------------------------------------------------------------------------------------------

episodes["interaction_episode_end"] = pd.to_datetime(
    episodes["interaction_episode_end"],
    utc=True,
    errors="raise"
)

if "datetime" in market.columns:
    market["datetime"] = pd.to_datetime(
        market["datetime"],
        utc=True,
        errors="raise"
    )
elif "time" in market.columns:
    market["datetime"] = pd.to_datetime(
        market["time"],
        unit="s",
        utc=True,
        errors="raise"
    )
else:
    raise RuntimeError(
        "m5_df must contain either 'datetime' or 'time'."
    )

market = (
    market
    .sort_values("datetime")
    .drop_duplicates("datetime")
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 3. MARKET DATA CONTRACT
# --------------------------------------------------------------------------------------------------

required_market_columns = {
    "datetime",
    "high",
    "low",
    "close",
}

missing_market_columns = (
    required_market_columns
    - set(market.columns)
)

if missing_market_columns:
    raise RuntimeError(
        f"Missing market columns: {sorted(missing_market_columns)}"
    )


# --------------------------------------------------------------------------------------------------
# 4. HORIZON CONTRACT
# --------------------------------------------------------------------------------------------------

REACTION_HORIZONS_MINUTES = [15, 30, 60, 120]


# --------------------------------------------------------------------------------------------------
# 5. MARKET TIMESTAMP INDEX
# --------------------------------------------------------------------------------------------------

market_times = (
    market["datetime"]
    .sort_values()
    .drop_duplicates()
    .reset_index(drop=True)
)

market_time_set = set(market_times)

market_by_time = (
    market
    .set_index("datetime")
    .sort_index()
)


# --------------------------------------------------------------------------------------------------
# 6. BUILDER
#
# One row per:
#
#     Episode × Horizon
#
# Total expected rows:
#
#     960 × 4 = 3840
#
# Rows are NEVER discarded because of missing future data.
# --------------------------------------------------------------------------------------------------

reaction_rows = []


for _, episode in episodes.iterrows():

    episode_id = episode["interaction_episode_id"]
    fvg_id = episode["fvg_id"]
    episode_sequence = episode["episode_sequence"]
    episode_end = episode["interaction_episode_end"]

    for horizon_minutes in REACTION_HORIZONS_MINUTES:

        horizon_end = (
            episode_end
            + pd.Timedelta(minutes=horizon_minutes)
        )

        # ------------------------------------------------------------------------------------------
        # Candidate reaction candles
        #
        # STRICTLY AFTER Episode End
        # THROUGH Horizon End
        # ------------------------------------------------------------------------------------------

        reaction_mask = (
            (market["datetime"] > episode_end)
            &
            (market["datetime"] <= horizon_end)
        )

        reaction_market = market.loc[
            reaction_mask
        ].copy()

        # ------------------------------------------------------------------------------------------
        # First reaction candle
        # ------------------------------------------------------------------------------------------

        if reaction_market.empty:

            reaction_start = pd.NaT

        else:

            reaction_start = reaction_market.iloc[0]["datetime"]

        # ------------------------------------------------------------------------------------------
        # Expected M5 timestamps
        #
        # If first reaction candle is exactly +5m,
        # expected timestamps are:
        #
        # +5, +10, ..., +Horizon
        #
        # Otherwise the observation is GAP_INTERRUPTED.
        # ------------------------------------------------------------------------------------------

        if pd.isna(reaction_start):

            expected_timestamps = pd.DatetimeIndex([])

        else:

            first_delta_minutes = (
                reaction_start - episode_end
            ).total_seconds() / 60.0

            if first_delta_minutes == 5.0:

                expected_timestamps = pd.date_range(
                    start=reaction_start,
                    end=horizon_end,
                    freq="5min",
                    tz="UTC",
                )

            else:

                expected_timestamps = pd.DatetimeIndex([])

        actual_timestamps = pd.DatetimeIndex(
            reaction_market["datetime"]
        )

        expected_count = len(expected_timestamps)
        actual_count = len(actual_timestamps)

        missing_timestamp_count = 0

        if expected_count > 0:

            actual_timestamp_set = set(
                actual_timestamps
            )

            missing_timestamp_count = sum(
                ts not in actual_timestamp_set
                for ts in expected_timestamps
            )

        # ------------------------------------------------------------------------------------------
        # Classification of data availability
        # ------------------------------------------------------------------------------------------

        if pd.isna(reaction_start):

            data_status = "NO_FUTURE_DATA"

        elif (
            first_delta_minutes != 5.0
        ):

            data_status = "GAP_INTERRUPTED"

        elif missing_timestamp_count > 0:

            data_status = "GAP_INTERRUPTED"

        elif actual_count != expected_count:

            data_status = "INCOMPLETE_HORIZON"

        else:

            data_status = "COMPLETE"

        data_complete = (
            data_status == "COMPLETE"
        )

        # ------------------------------------------------------------------------------------------
        # Data values
        #
        # These are populated only when the complete horizon is available.
        # No partial geometry is promoted to a complete observation.
        # ------------------------------------------------------------------------------------------

        if data_complete:

            complete_market = (
                market_by_time
                .loc[expected_timestamps]
                .copy()
            )

            future_high = complete_market["high"].max()
            future_low = complete_market["low"].min()
            future_close = complete_market.iloc[-1]["close"]

            observation_count = len(
                complete_market
            )

        else:

            future_high = np.nan
            future_low = np.nan
            future_close = np.nan
            observation_count = actual_count

        # ------------------------------------------------------------------------------------------
        # Row
        # ------------------------------------------------------------------------------------------

        reaction_rows.append({

            "interaction_episode_id": episode_id,
            "fvg_id": fvg_id,
            "episode_sequence": episode_sequence,

            "interaction_episode_end": episode_end,

            "reaction_start": reaction_start,
            "reaction_end": horizon_end,

            "horizon_minutes": horizon_minutes,

            "observation_count": observation_count,
            "expected_observation_count": expected_count,

            "missing_timestamp_count": missing_timestamp_count,

            "data_complete": data_complete,
            "data_status": data_status,

            "future_high": future_high,
            "future_low": future_low,
            "future_close": future_close,
        })


# --------------------------------------------------------------------------------------------------
# 7. OUTPUT DATAFRAME
# --------------------------------------------------------------------------------------------------

fvg_episode_reaction_v1 = pd.DataFrame(
    reaction_rows
)


# --------------------------------------------------------------------------------------------------
# 8. SORTING
# --------------------------------------------------------------------------------------------------

fvg_episode_reaction_v1 = (
    fvg_episode_reaction_v1
    .sort_values(
        [
            "interaction_episode_id",
            "horizon_minutes",
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 9. BASIC COUNTS
# --------------------------------------------------------------------------------------------------

expected_rows = (
    len(episodes)
    * len(REACTION_HORIZONS_MINUTES)
)

actual_rows = len(
    fvg_episode_reaction_v1
)

unique_episode_count = (
    fvg_episode_reaction_v1[
        "interaction_episode_id"
    ].nunique()
)

unique_horizon_count = (
    fvg_episode_reaction_v1[
        "horizon_minutes"
    ].nunique()
)


# --------------------------------------------------------------------------------------------------
# 10. STATUS DISTRIBUTION
# --------------------------------------------------------------------------------------------------

status_distribution = (
    fvg_episode_reaction_v1[
        "data_status"
    ]
    .value_counts()
    .to_dict()
)


# --------------------------------------------------------------------------------------------------
# 11. COMPLETENESS BY HORIZON
# --------------------------------------------------------------------------------------------------

completeness_by_horizon = (
    fvg_episode_reaction_v1
    .groupby("horizon_minutes")["data_complete"]
    .agg(
        total="count",
        complete="sum",
    )
)

completeness_by_horizon["incomplete"] = (
    completeness_by_horizon["total"]
    - completeness_by_horizon["complete"]
)


# --------------------------------------------------------------------------------------------------
# 12. DUPLICATE EPISODE / HORIZON PAIRS
# --------------------------------------------------------------------------------------------------

duplicate_episode_horizon_pairs = int(
    fvg_episode_reaction_v1.duplicated(
        [
            "interaction_episode_id",
            "horizon_minutes",
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 13. INVALID HORIZONS
# --------------------------------------------------------------------------------------------------

invalid_horizons = int(
    (
        ~fvg_episode_reaction_v1[
            "horizon_minutes"
        ].isin(
            REACTION_HORIZONS_MINUTES
        )
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 14. REACTION START CONTRACT
# --------------------------------------------------------------------------------------------------

valid_reaction_start_rows = (
    fvg_episode_reaction_v1[
        fvg_episode_reaction_v1[
            "reaction_start"
        ].notna()
    ]
)

if not valid_reaction_start_rows.empty:

    reaction_start_delta = (
        valid_reaction_start_rows[
            "reaction_start"
        ]
        - valid_reaction_start_rows[
            "interaction_episode_end"
        ]
    ).dt.total_seconds() / 60

    invalid_reaction_start_delta = int(
        (reaction_start_delta < 5.0).sum()
    )

else:

    invalid_reaction_start_delta = 0


# --------------------------------------------------------------------------------------------------
# 15. HORIZON BOUNDARY CONTRACT
# --------------------------------------------------------------------------------------------------

horizon_boundary_mismatches = int(
    (
        fvg_episode_reaction_v1["reaction_end"]
        != (
            fvg_episode_reaction_v1[
                "interaction_episode_end"
            ]
            + pd.to_timedelta(
                fvg_episode_reaction_v1[
                    "horizon_minutes"
                ],
                unit="m"
            )
        )
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 16. COMPLETE-HORIZON OBSERVATION COUNT
# --------------------------------------------------------------------------------------------------

complete_rows = fvg_episode_reaction_v1[
    fvg_episode_reaction_v1["data_complete"]
].copy()

expected_complete_observation_counts = (
    complete_rows["horizon_minutes"] // 5
)

observation_count_mismatches = int(
    (
        complete_rows[
            "observation_count"
        ].to_numpy()
        !=
        expected_complete_observation_counts.to_numpy()
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 17. COMPLETE-HORIZON NULL CONTRACT
# --------------------------------------------------------------------------------------------------

complete_numeric_columns = [
    "future_high",
    "future_low",
    "future_close",
]

complete_null_violations = int(
    complete_rows[
        complete_numeric_columns
    ]
    .isna()
    .any(axis=1)
    .sum()
)


# --------------------------------------------------------------------------------------------------
# 18. INCOMPLETE-HORIZON PROMOTION CONTRACT
#
# Incomplete observations MUST NOT contain promoted future geometry.
# --------------------------------------------------------------------------------------------------

incomplete_rows = fvg_episode_reaction_v1[
    ~fvg_episode_reaction_v1["data_complete"]
]

incomplete_geometry_promoted = int(
    incomplete_rows[
        complete_numeric_columns
    ]
    .notna()
    .any(axis=1)
    .sum()
)


# --------------------------------------------------------------------------------------------------
# 19. PRINT OUTPUT PREVIEW
# --------------------------------------------------------------------------------------------------

print()
print("OUTPUT PREVIEW")
print("-" * 100)

print(
    fvg_episode_reaction_v1.head(12).to_string(
        index=False
    )
)


# --------------------------------------------------------------------------------------------------
# 20. SUMMARY
# --------------------------------------------------------------------------------------------------

print()
print("BUILDER SUMMARY")
print("-" * 100)

print(
    f"Expected Rows                         : "
    f"{expected_rows:,}"
)

print(
    f"Actual Rows                           : "
    f"{actual_rows:,}"
)

print(
    f"Unique Episodes                       : "
    f"{unique_episode_count:,}"
)

print(
    f"Unique Horizons                       : "
    f"{unique_horizon_count:,}"
)

print(
    f"Duplicate Episode/Horizon Pairs      : "
    f"{duplicate_episode_horizon_pairs:,}"
)

print(
    f"Invalid Horizons                     : "
    f"{invalid_horizons:,}"
)

print(
    f"Invalid Reaction Start Deltas        : "
    f"{invalid_reaction_start_delta:,}"
)

print(
    f"Horizon Boundary Mismatches          : "
    f"{horizon_boundary_mismatches:,}"
)

print(
    f"Complete Observation Count Mismatches: "
    f"{observation_count_mismatches:,}"
)

print(
    f"Complete Null Violations              : "
    f"{complete_null_violations:,}"
)

print(
    f"Incomplete Geometry Promoted          : "
    f"{incomplete_geometry_promoted:,}"
)

print()
print("STATUS DISTRIBUTION")
print("-" * 100)

for status, count in status_distribution.items():
    print(
        f"{status:<25} : {count:,}"
    )

print()
print("COMPLETENESS BY HORIZON")
print("-" * 100)

print(
    completeness_by_horizon.to_string()
)


# --------------------------------------------------------------------------------------------------
# 21. CONTRACT GATES
# --------------------------------------------------------------------------------------------------

gates = {

    "ROW_COUNT":
        actual_rows == expected_rows,

    "EPISODE_COUNT":
        unique_episode_count == len(episodes),

    "HORIZON_COUNT":
        unique_horizon_count == len(
            REACTION_HORIZONS_MINUTES
        ),

    "EPISODE_HORIZON_UNIQUENESS":
        duplicate_episode_horizon_pairs == 0,

    "VALID_HORIZON_VOCABULARY":
        invalid_horizons == 0,

    "REACTION_START_ORDER":
        invalid_reaction_start_delta == 0,

    "HORIZON_BOUNDARY_ALIGNMENT":
        horizon_boundary_mismatches == 0,

    "COMPLETE_OBSERVATION_COUNT":
        observation_count_mismatches == 0,

    "COMPLETE_DATA_NO_NULLS":
        complete_null_violations == 0,

    "NO_INCOMPLETE_GEOMETRY_PROMOTION":
        incomplete_geometry_promoted == 0,
}


# --------------------------------------------------------------------------------------------------
# 22. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("STEP 8.2-B REGRESSION GATES")
print("=" * 100)

for gate_name, passed in gates.items():

    print(
        f"{gate_name:<45} : "
        f"{'PASSED' if passed else 'FAILED'}"
    )

failed_gates = [
    name
    for name, passed in gates.items()
    if not passed
]

print()
print("=" * 100)
print(
    f"STEP 8.2-B OVERALL : "
    f"{'PASS' if not failed_gates else 'FAIL'}"
)

print(
    f"Passed Gates       : "
    f"{sum(gates.values())}/{len(gates)}"
)

print(
    f"Failed Gates       : "
    f"{failed_gates}"
)

print("=" * 100)

In [ ]:
# ==================================================================================================
# STEP 8.2-C PATCH
# NULLABLE REACTION START CONTRACT
# ==================================================================================================

def _normalize_utc_datetime_nullable(df, column):
    df[column] = pd.to_datetime(
        df[column],
        utc=True,
        errors="coerce",
    )
    return df


# reaction_start is nullable by contract
reaction_obs = _normalize_utc_datetime_nullable(
    reaction_obs,
    "reaction_start",
)

# reaction_end is expected to exist for every Episode × Horizon row
reaction_obs = _normalize_utc_datetime_nullable(
    reaction_obs,
    "reaction_end",
)

print("STEP 8.2-C PATCH")
print("-" * 100)

print(
    f"reaction_start NaT : "
    f"{reaction_obs['reaction_start'].isna().sum()}"
)

print(
    f"reaction_end NaT   : "
    f"{reaction_obs['reaction_end'].isna().sum()}"
)

print(
    "\nreaction_start NaT by status:"
)

print(
    reaction_obs.loc[
        reaction_obs["reaction_start"].isna(),
        "data_status"
    ].value_counts(dropna=False).to_string()
)

In [ ]:
def _normalize_utc_datetime_nullable(df, column):
    df[column] = pd.to_datetime(
        df[column],
        utc=True,
        errors="coerce",
    )
    return df

In [ ]:
# ==================================================================================================
# STEP 8.2-C PATCH — REGRESSION GATE FIX
# ==================================================================================================

complete = (
    fvg_episode_reaction_geometry_v1[
        fvg_episode_reaction_geometry_v1["data_complete"].eq(True)
    ]
    .copy()
)

# --------------------------------------------------------------------------------------------------
# DIRECTIONAL CLOSE CONTRACT
# --------------------------------------------------------------------------------------------------

if len(complete):

    expected_directional_close = np.where(
        complete["fvg_type"].eq("BULLISH_FVG"),

        complete["future_close"]
        - complete["episode_end_close"],

        complete["episode_end_close"]
        - complete["future_close"],
    )

    close_direction_mismatch = int(
        (
            ~np.isclose(
                complete[
                    "directional_close_change"
                ].to_numpy(),

                expected_directional_close,

                rtol=0,
                atol=1e-12,
            )
        ).sum()
    )

else:

    close_direction_mismatch = 0


# --------------------------------------------------------------------------------------------------
# FAVORABLE CLOSE CONTRACT
# --------------------------------------------------------------------------------------------------

if len(complete):

    expected_favorable_close = np.maximum(
        complete[
            "directional_close_change"
        ].to_numpy(),
        0,
    )

    favorable_close_mismatch = int(
        (
            ~np.isclose(
                complete[
                    "favorable_close_change"
                ].to_numpy(),

                expected_favorable_close,

                rtol=0,
                atol=1e-12,
            )
        ).sum()
    )

else:

    favorable_close_mismatch = 0


print("=" * 100)
print("STEP 8.2-C PATCH — GATE DIAGNOSTICS")
print("=" * 100)

print(
    f"Complete Rows                  : {len(complete):,}"
)

print(
    f"Close Direction Mismatches     : "
    f"{close_direction_mismatch:,}"
)

print(
    f"Favorable Close Mismatches     : "
    f"{favorable_close_mismatch:,}"
)

In [ ]:
# ==================================================================================================
# STEP 8.2-C — REACTION GEOMETRY BUILDER V1
# ==================================================================================================
#
# PURPOSE
# -------
# Measure post-episode price geometry only.
#
# This layer does NOT perform:
#   - reaction classification
#   - ranking
#   - entry logic
#   - liquidity interaction
#   - market structure analysis
#   - premium / discount
#   - SL / TP
#   - win / loss
#
# FROZEN INPUTS
# -------------
#   episode_characterization_v1_1
#   interaction_episode_registry_v1_1
#   fvg_episode_reaction_v1
#   fvg_validated_v1
#   m5_df
#
# FROZEN HORIZONS
# ---------------
#   +15m
#   +30m
#   +60m
#   +120m
#
# GEOMETRY CONTRACT
# -----------------
# Baseline:
#   close at interaction_episode_end
#
# BULLISH_FVG:
#   favorable_price = future_high
#   adverse_price   = future_low
#
# BEARISH_FVG:
#   favorable_price = future_low
#   adverse_price   = future_high
#
# Excursions:
#   favorable_excursion = abs(favorable_price - baseline)
#   adverse_excursion   = abs(adverse_price - baseline)
#
# Raw close movement:
#   net_close_change = future_close - baseline
#
# Directional close movement:
#   BULLISH_FVG = future_close - baseline
#   BEARISH_FVG = baseline - future_close
#
# Favorable close movement:
#   max(directional_close_change, 0)
#
# Incomplete horizons:
#   Geometry MUST remain NaN.
#
# ==================================================================================================

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# CONFIGURATION
# --------------------------------------------------------------------------------------------------

REACTION_GEOMETRY_HORIZONS_V1 = [15, 30, 60, 120]

REACTION_GEOMETRY_ALLOWED_FVG_TYPES_V1 = {
    "BULLISH_FVG",
    "BEARISH_FVG",
}

REACTION_GEOMETRY_REQUIRED_REACTION_COLUMNS_V1 = [
    "interaction_episode_id",
    "fvg_id",
    "episode_sequence",
    "interaction_episode_end",
    "reaction_start",
    "reaction_end",
    "horizon_minutes",
    "observation_count",
    "expected_observation_count",
    "data_complete",
    "data_status",
    "future_high",
    "future_low",
    "future_close",
]

REACTION_GEOMETRY_OUTPUT_COLUMNS_V1 = [
    "interaction_episode_id",
    "fvg_id",
    "fvg_type",
    "episode_sequence",
    "interaction_episode_end",
    "episode_end_close",
    "reaction_start",
    "reaction_end",
    "horizon_minutes",
    "observation_count",
    "expected_observation_count",
    "data_complete",
    "data_status",
    "future_high",
    "future_low",
    "future_close",
    "favorable_price",
    "adverse_price",
    "favorable_excursion",
    "adverse_excursion",
    "net_close_change",
    "directional_close_change",
    "favorable_close_change",
]


# --------------------------------------------------------------------------------------------------
# UTC NORMALIZATION HELPERS
# --------------------------------------------------------------------------------------------------

def _normalize_utc_required_v1(series, name):
    """
    Normalize a required datetime Series to timezone-aware UTC.
    NaT is not allowed.
    """

    out = pd.to_datetime(
        series,
        utc=True,
        errors="coerce",
    )

    if out.isna().any():
        bad_count = int(out.isna().sum())

        raise ValueError(
            f"{name} contains {bad_count} invalid / missing datetime values."
        )

    return out


def _normalize_utc_nullable_v1(series):
    """
    Normalize a nullable datetime Series to timezone-aware UTC.
    NaT is allowed.
    """

    return pd.to_datetime(
        series,
        utc=True,
        errors="coerce",
    )


# --------------------------------------------------------------------------------------------------
# BUILDER
# --------------------------------------------------------------------------------------------------

def build_fvg_episode_reaction_geometry_v1(
    reaction_obs,
    episode_characterization,
    episode_registry,
    m5_df,
    fvg_registry,
):
    """
    Build Reaction Geometry V1.

    One output row exists for every:

        Episode × Horizon

    Geometry is populated only for COMPLETE horizons.
    """

    # ==============================================================================================
    # 1. INPUT COPIES
    # ==============================================================================================

    reaction = reaction_obs.copy(deep=True)
    episodes = episode_characterization.copy(deep=True)
    registry = episode_registry.copy(deep=True)
    market = m5_df.copy(deep=True)
    fvg = fvg_registry.copy(deep=True)


    # ==============================================================================================
    # 2. BASIC INPUT VALIDATION
    # ==============================================================================================

    missing_reaction_columns = [
        c
        for c in REACTION_GEOMETRY_REQUIRED_REACTION_COLUMNS_V1
        if c not in reaction.columns
    ]

    if missing_reaction_columns:
        raise KeyError(
            "Missing required Reaction Observation columns: "
            f"{missing_reaction_columns}"
        )


    required_episode_columns = [
        "interaction_episode_id",
        "fvg_id",
        "episode_sequence",
        "interaction_episode_start",
        "interaction_episode_end",
        "interaction_observation_count",
    ]

    missing_episode_columns = [
        c
        for c in required_episode_columns
        if c not in episodes.columns
    ]

    if missing_episode_columns:
        raise KeyError(
            "Missing required Episode Characterization columns: "
            f"{missing_episode_columns}"
        )


    required_registry_columns = [
        "interaction_episode_id",
        "fvg_id",
        "episode_sequence",
        "interaction_episode_start",
        "interaction_episode_end",
    ]

    missing_registry_columns = [
        c
        for c in required_registry_columns
        if c not in registry.columns
    ]

    if missing_registry_columns:
        raise KeyError(
            "Missing required Episode Registry columns: "
            f"{missing_registry_columns}"
        )


    required_market_columns = [
        "datetime",
        "close",
    ]

    missing_market_columns = [
        c
        for c in required_market_columns
        if c not in market.columns
    ]

    if missing_market_columns:
        raise KeyError(
            "Missing required Market columns: "
            f"{missing_market_columns}"
        )


    required_fvg_columns = [
        "fvg_id",
        "fvg_type",
    ]

    missing_fvg_columns = [
        c
        for c in required_fvg_columns
        if c not in fvg.columns
    ]

    if missing_fvg_columns:
        raise KeyError(
            "Missing required FVG columns: "
            f"{missing_fvg_columns}"
        )


    # ==============================================================================================
    # 3. DATETIME NORMALIZATION
    # ==============================================================================================

    reaction["interaction_episode_end"] = _normalize_utc_required_v1(
        reaction["interaction_episode_end"],
        "reaction.interaction_episode_end",
    )

    reaction["reaction_start"] = _normalize_utc_nullable_v1(
        reaction["reaction_start"]
    )

    reaction["reaction_end"] = _normalize_utc_required_v1(
        reaction["reaction_end"],
        "reaction.reaction_end",
    )

    episodes["interaction_episode_start"] = _normalize_utc_required_v1(
        episodes["interaction_episode_start"],
        "episodes.interaction_episode_start",
    )

    episodes["interaction_episode_end"] = _normalize_utc_required_v1(
        episodes["interaction_episode_end"],
        "episodes.interaction_episode_end",
    )

    registry["interaction_episode_start"] = _normalize_utc_required_v1(
        registry["interaction_episode_start"],
        "registry.interaction_episode_start",
    )

    registry["interaction_episode_end"] = _normalize_utc_required_v1(
        registry["interaction_episode_end"],
        "registry.interaction_episode_end",
    )

    market["datetime"] = _normalize_utc_required_v1(
        market["datetime"],
        "market.datetime",
    )


    # ==============================================================================================
    # 4. MARKET NORMALIZATION
    # ==============================================================================================

    market = (
        market
        .sort_values("datetime")
        .drop_duplicates("datetime", keep="first")
        .reset_index(drop=True)
    )


    # ==============================================================================================
    # 5. FVG IDENTITY / DIRECTION
    # ==============================================================================================

    fvg_identity = (
        fvg[
            [
                "fvg_id",
                "fvg_type",
            ]
        ]
        .drop_duplicates("fvg_id")
        .copy()
    )

    if fvg_identity["fvg_id"].duplicated().any():
        raise RuntimeError(
            "FVG identity is not unique by fvg_id."
        )

    invalid_fvg_types = sorted(
        set(
            fvg_identity["fvg_type"].dropna().unique()
        )
        -
        REACTION_GEOMETRY_ALLOWED_FVG_TYPES_V1
    )

    if invalid_fvg_types:
        raise RuntimeError(
            "Invalid FVG types detected: "
            f"{invalid_fvg_types}"
        )


    # ==============================================================================================
    # 6. EPISODE IDENTITY VALIDATION
    # ==============================================================================================

    episode_identity = (
        episodes[
            [
                "interaction_episode_id",
                "fvg_id",
                "episode_sequence",
                "interaction_episode_end",
            ]
        ]
        .drop_duplicates("interaction_episode_id")
        .copy()
    )

    if episode_identity[
        "interaction_episode_id"
    ].duplicated().any():

        raise RuntimeError(
            "Episode characterization contains duplicate Episode IDs."
        )


    registry_identity = (
        registry[
            [
                "interaction_episode_id",
                "fvg_id",
                "episode_sequence",
                "interaction_episode_end",
            ]
        ]
        .drop_duplicates("interaction_episode_id")
        .copy()
    )

    if registry_identity[
        "interaction_episode_id"
    ].duplicated().any():

        raise RuntimeError(
            "Episode registry contains duplicate Episode IDs."
        )


    reaction_episode_set = set(
        reaction["interaction_episode_id"]
    )

    registry_episode_set = set(
        registry_identity["interaction_episode_id"]
    )

    if reaction_episode_set != registry_episode_set:
        raise RuntimeError(
            "Reaction Observation Episode set does not match "
            "Episode Registry."
        )


    # ==============================================================================================
    # 7. REACTION → EPISODE IDENTITY CHECK
    # ==============================================================================================

    reaction_identity = (
        reaction[
            [
                "interaction_episode_id",
                "fvg_id",
                "episode_sequence",
                "interaction_episode_end",
            ]
        ]
        .drop_duplicates()
        .copy()
    )

    reaction_identity_check = reaction_identity.merge(
        episode_identity,
        on="interaction_episode_id",
        how="left",
        suffixes=(
            "_reaction",
            "_episode",
        ),
        validate="many_to_one",
    )

    identity_mismatch = (
        reaction_identity_check["fvg_id_reaction"]
        !=
        reaction_identity_check["fvg_id_episode"]
    ) | (
        reaction_identity_check["episode_sequence_reaction"]
        !=
        reaction_identity_check["episode_sequence_episode"]
    ) | (
        reaction_identity_check[
            "interaction_episode_end_reaction"
        ]
        !=
        reaction_identity_check[
            "interaction_episode_end_episode"
        ]
    )

    if identity_mismatch.any():
        raise RuntimeError(
            "Reaction Observation → Episode identity mismatch detected."
        )


    # ==============================================================================================
    # 8. MERGE FVG DIRECTION
    # ==============================================================================================

    reaction = reaction.merge(
        fvg_identity,
        on="fvg_id",
        how="left",
        validate="many_to_one",
    )

    if reaction["fvg_type"].isna().any():
        raise RuntimeError(
            "Some reaction rows could not resolve FVG direction."
        )


    # ==============================================================================================
    # 9. RESOLVE EPISODE-END BASELINE CLOSE
    # ==============================================================================================

    baseline = market[
        [
            "datetime",
            "close",
        ]
    ].rename(
        columns={
            "datetime": "interaction_episode_end",
            "close": "episode_end_close",
        }
    )

    baseline = baseline.drop_duplicates(
        "interaction_episode_end"
    )

    reaction = reaction.merge(
        baseline,
        on="interaction_episode_end",
        how="left",
        validate="many_to_one",
    )

    if reaction["episode_end_close"].isna().any():
        missing_baseline = int(
            reaction["episode_end_close"].isna().sum()
        )

        raise RuntimeError(
            "Missing Episode-End baseline close values: "
            f"{missing_baseline}"
        )


    # ==============================================================================================
    # 10. INITIALIZE GEOMETRY COLUMNS
    # ==============================================================================================

    geometry_columns = [
        "favorable_price",
        "adverse_price",
        "favorable_excursion",
        "adverse_excursion",
        "net_close_change",
        "directional_close_change",
        "favorable_close_change",
    ]

    for column in geometry_columns:
        reaction[column] = np.nan


    # ==============================================================================================
    # 11. COMPLETE HORIZON MASK
    # ==============================================================================================

    complete_mask = (
        reaction["data_complete"].eq(True)
        &
        reaction["data_status"].eq("COMPLETE")
    )


    # ==============================================================================================
    # 12. FVG DIRECTION MASKS
    # ==============================================================================================

    bullish_mask = (
        complete_mask
        &
        reaction["fvg_type"].eq("BULLISH_FVG")
    )

    bearish_mask = (
        complete_mask
        &
        reaction["fvg_type"].eq("BEARISH_FVG")
    )


    # ==============================================================================================
    # 13. FAVORABLE / ADVERSE PRICE
    # ==============================================================================================

    # BULLISH:
    #   favorable = upward extreme
    #   adverse   = downward extreme

    reaction.loc[
        bullish_mask,
        "favorable_price",
    ] = reaction.loc[
        bullish_mask,
        "future_high",
    ]

    reaction.loc[
        bullish_mask,
        "adverse_price",
    ] = reaction.loc[
        bullish_mask,
        "future_low",
    ]


    # BEARISH:
    #   favorable = downward extreme
    #   adverse   = upward extreme

    reaction.loc[
        bearish_mask,
        "favorable_price",
    ] = reaction.loc[
        bearish_mask,
        "future_low",
    ]

    reaction.loc[
        bearish_mask,
        "adverse_price",
    ] = reaction.loc[
        bearish_mask,
        "future_high",
    ]


    # ==============================================================================================
    # 14. ABSOLUTE PRICE EXCURSIONS
    # ==============================================================================================

    reaction.loc[
        complete_mask,
        "favorable_excursion",
    ] = (
        reaction.loc[
            complete_mask,
            "favorable_price",
        ]
        -
        reaction.loc[
            complete_mask,
            "episode_end_close",
        ]
    ).abs()


    reaction.loc[
        complete_mask,
        "adverse_excursion",
    ] = (
        reaction.loc[
            complete_mask,
            "adverse_price",
        ]
        -
        reaction.loc[
            complete_mask,
            "episode_end_close",
        ]
    ).abs()


    # ==============================================================================================
    # 15. RAW SIGNED CLOSE CHANGE
    # ==============================================================================================

    reaction.loc[
        complete_mask,
        "net_close_change",
    ] = (
        reaction.loc[
            complete_mask,
            "future_close",
        ]
        -
        reaction.loc[
            complete_mask,
            "episode_end_close",
        ]
    )


    # ==============================================================================================
    # 16. DIRECTIONAL CLOSE CHANGE
    # ==============================================================================================

    reaction.loc[
        bullish_mask,
        "directional_close_change",
    ] = (
        reaction.loc[
            bullish_mask,
            "future_close",
        ]
        -
        reaction.loc[
            bullish_mask,
            "episode_end_close",
        ]
    )

    reaction.loc[
        bearish_mask,
        "directional_close_change",
    ] = (
        reaction.loc[
            bearish_mask,
            "episode_end_close",
        ]
        -
        reaction.loc[
            bearish_mask,
            "future_close",
        ]
    )


    # ==============================================================================================
    # 17. FAVORABLE CLOSE CHANGE
    # ==============================================================================================

    reaction.loc[
        complete_mask,
        "favorable_close_change",
    ] = (
        reaction.loc[
            complete_mask,
            "directional_close_change",
        ]
        .clip(lower=0)
    )


    # ==============================================================================================
    # 18. OUTPUT
    # ==============================================================================================

    output = reaction[
        REACTION_GEOMETRY_OUTPUT_COLUMNS_V1
    ].copy()

    output = (
        output
        .sort_values(
            [
                "interaction_episode_id",
                "horizon_minutes",
            ]
        )
        .reset_index(drop=True)
    )


    # ==============================================================================================
    # 19. FINAL BUILDER VALIDATION
    # ==============================================================================================

    expected_rows = (
        episode_identity[
            "interaction_episode_id"
        ].nunique()
        *
        len(REACTION_GEOMETRY_HORIZONS_V1)
    )

    if len(output) != expected_rows:
        raise RuntimeError(
            "Unexpected Reaction Geometry row count: "
            f"{len(output)} != {expected_rows}"
        )


    duplicate_pairs = output.duplicated(
        [
            "interaction_episode_id",
            "horizon_minutes",
        ]
    ).sum()

    if duplicate_pairs:
        raise RuntimeError(
            "Duplicate Episode × Horizon rows detected: "
            f"{duplicate_pairs}"
        )


    invalid_horizons = set(
        output["horizon_minutes"].unique()
    ) - set(
        REACTION_GEOMETRY_HORIZONS_V1
    )

    if invalid_horizons:
        raise RuntimeError(
            "Invalid horizon values detected: "
            f"{sorted(invalid_horizons)}"
        )


    # Complete rows must have all geometry.
    complete_output = output[
        output["data_complete"].eq(True)
        &
        output["data_status"].eq("COMPLETE")
    ]

    complete_geometry_nulls = (
        complete_output[
            [
                "episode_end_close",
                "reaction_start",
                "reaction_end",
                "future_high",
                "future_low",
                "future_close",
                "favorable_price",
                "adverse_price",
                "favorable_excursion",
                "adverse_excursion",
                "net_close_change",
                "directional_close_change",
                "favorable_close_change",
            ]
        ]
        .isna()
        .any(axis=1)
        .sum()
    )

    if complete_geometry_nulls:
        raise RuntimeError(
            "Complete rows contain missing geometry values: "
            f"{complete_geometry_nulls}"
        )


    # Incomplete rows must NOT contain promoted geometry.
    incomplete_output = output[
        ~output["data_complete"].eq(True)
    ]

    incomplete_geometry_promoted = (
        incomplete_output[
            geometry_columns
        ]
        .notna()
        .any(axis=1)
        .sum()
    )

    if incomplete_geometry_promoted:
        raise RuntimeError(
            "Incomplete rows contain promoted geometry: "
            f"{incomplete_geometry_promoted}"
        )


    return output


# ==================================================================================================
# BUILD
# ==================================================================================================

fvg_episode_reaction_geometry_v1 = (
    build_fvg_episode_reaction_geometry_v1(
        reaction_obs=fvg_episode_reaction_v1,
        episode_characterization=episode_characterization_v1_1,
        episode_registry=interaction_episode_registry_v1_1,
        m5_df=m5_df,
        fvg_registry=fvg_validated_v1,
    )
)


# ==================================================================================================
# BUILDER SUMMARY
# ==================================================================================================

print("=" * 100)
print("STEP 8.2-C — REACTION GEOMETRY BUILDER V1")
print("=" * 100)

print(
    f"Rows                         : "
    f"{len(fvg_episode_reaction_geometry_v1):,}"
)

print(
    f"Unique Episodes              : "
    f"{fvg_episode_reaction_geometry_v1['interaction_episode_id'].nunique():,}"
)

print(
    f"Unique FVGs                  : "
    f"{fvg_episode_reaction_geometry_v1['fvg_id'].nunique():,}"
)

print(
    f"Unique Horizons              : "
    f"{fvg_episode_reaction_geometry_v1['horizon_minutes'].nunique():,}"
)

print()

print("FVG TYPE DISTRIBUTION")
print("-" * 100)

print(
    fvg_episode_reaction_geometry_v1[
        "fvg_type"
    ].value_counts()
    .to_string()
)

print()

print("DATA STATUS DISTRIBUTION")
print("-" * 100)

print(
    fvg_episode_reaction_geometry_v1[
        "data_status"
    ].value_counts()
    .to_string()
)

print()

print("COMPLETE / INCOMPLETE")
print("-" * 100)

print(
    fvg_episode_reaction_geometry_v1[
        "data_complete"
    ]
    .value_counts()
    .to_string()
)

print()

print("HORIZON DISTRIBUTION")
print("-" * 100)

print(
    fvg_episode_reaction_geometry_v1[
        "horizon_minutes"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

print("=" * 100)

In [ ]:
# ==================================================================================================
# STEP 8.2-C — FINAL REGRESSION GATE V2
# ==================================================================================================

g = fvg_episode_reaction_geometry_v1.copy()

complete = g[
    g["data_complete"].eq(True)
    & g["data_status"].eq("COMPLETE")
].copy()

incomplete = g[
    ~g["data_complete"].eq(True)
].copy()


# --------------------------------------------------------------------------------------------------
# BASIC COUNTS
# --------------------------------------------------------------------------------------------------

expected_rows = (
    episode_characterization_v1_1[
        "interaction_episode_id"
    ].nunique()
    * 4
)

actual_rows = len(g)

episode_count = g[
    "interaction_episode_id"
].nunique()

expected_episode_count = (
    episode_characterization_v1_1[
        "interaction_episode_id"
    ].nunique()
)

horizon_count = g[
    "horizon_minutes"
].nunique()


# --------------------------------------------------------------------------------------------------
# DUPLICATE EPISODE × HORIZON
# --------------------------------------------------------------------------------------------------

duplicate_episode_horizon = int(
    g.duplicated(
        [
            "interaction_episode_id",
            "horizon_minutes",
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# FVG DIRECTION
# --------------------------------------------------------------------------------------------------

invalid_fvg_types = int(
    (
        ~g["fvg_type"].isin(
            {
                "BULLISH_FVG",
                "BEARISH_FVG",
            }
        )
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# HORIZON VOCABULARY
# --------------------------------------------------------------------------------------------------

invalid_horizons = int(
    (
        ~g["horizon_minutes"].isin(
            {
                15,
                30,
                60,
                120,
            }
        )
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# COMPLETE REACTION START
# --------------------------------------------------------------------------------------------------

complete_reaction_start_missing = int(
    complete["reaction_start"].isna().sum()
)


# --------------------------------------------------------------------------------------------------
# NO FUTURE DATA CONTRACT
# --------------------------------------------------------------------------------------------------

no_future_data_start_violations = int(
    (
        g["data_status"].eq("NO_FUTURE_DATA")
        &
        g["reaction_start"].notna()
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# COMPLETE GEOMETRY NULLS
# --------------------------------------------------------------------------------------------------

complete_geometry_columns = [
    "episode_end_close",
    "reaction_start",
    "reaction_end",
    "future_high",
    "future_low",
    "future_close",
    "favorable_price",
    "adverse_price",
    "favorable_excursion",
    "adverse_excursion",
    "net_close_change",
    "directional_close_change",
    "favorable_close_change",
]

complete_geometry_null_violations = int(
    complete[
        complete_geometry_columns
    ]
    .isna()
    .any(axis=1)
    .sum()
)


# --------------------------------------------------------------------------------------------------
# NON-NEGATIVE EXCURSIONS
# --------------------------------------------------------------------------------------------------

negative_favorable_excursion = int(
    (
        complete[
            "favorable_excursion"
        ] < 0
    ).sum()
)

negative_adverse_excursion = int(
    (
        complete[
            "adverse_excursion"
        ] < 0
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# NON-FINITE EXCURSIONS
# --------------------------------------------------------------------------------------------------

nonfinite_favorable_excursion = int(
    (
        ~np.isfinite(
            complete[
                "favorable_excursion"
            ].to_numpy(dtype=float)
        )
    ).sum()
)

nonfinite_adverse_excursion = int(
    (
        ~np.isfinite(
            complete[
                "adverse_excursion"
            ].to_numpy(dtype=float)
        )
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# DIRECTIONAL CLOSE CONTRACT
# --------------------------------------------------------------------------------------------------

expected_directional_close = np.where(
    complete["fvg_type"].eq("BULLISH_FVG"),

    complete["future_close"]
    -
    complete["episode_end_close"],

    complete["episode_end_close"]
    -
    complete["future_close"],
)

directional_close_mismatch = int(
    (
        ~np.isclose(
            complete[
                "directional_close_change"
            ].to_numpy(dtype=float),

            expected_directional_close.astype(float),

            rtol=0,
            atol=1e-12,
        )
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# FAVORABLE CLOSE CONTRACT
# --------------------------------------------------------------------------------------------------

expected_favorable_close = np.maximum(
    expected_directional_close,
    0,
)

favorable_close_mismatch = int(
    (
        ~np.isclose(
            complete[
                "favorable_close_change"
            ].to_numpy(dtype=float),

            expected_favorable_close.astype(float),

            rtol=0,
            atol=1e-12,
        )
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# PRICE GEOMETRY
# --------------------------------------------------------------------------------------------------

price_geometry_violations = int(
    (
        complete[
            [
                "favorable_price",
                "adverse_price",
            ]
        ]
        .isna()
        .any(axis=1)
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# FUTURE HIGH / LOW
# --------------------------------------------------------------------------------------------------

future_high_low_violations = int(
    (
        complete["future_high"]
        <
        complete["future_low"]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# INCOMPLETE GEOMETRY PROMOTION
# --------------------------------------------------------------------------------------------------

geometry_columns = [
    "favorable_price",
    "adverse_price",
    "favorable_excursion",
    "adverse_excursion",
    "net_close_change",
    "directional_close_change",
    "favorable_close_change",
]

incomplete_geometry_promoted = int(
    incomplete[
        geometry_columns
    ]
    .notna()
    .any(axis=1)
    .sum()
)


# ==================================================================================================
# GATES
# ==================================================================================================

gates = {

    "ROW_COUNT":
        actual_rows == expected_rows,

    "EPISODE_COUNT":
        episode_count == expected_episode_count,

    "HORIZON_COUNT":
        horizon_count == 4,

    "EPISODE_HORIZON_UNIQUENESS":
        duplicate_episode_horizon == 0,

    "FVG_DIRECTION_CONTRACT":
        invalid_fvg_types == 0,

    "VALID_HORIZON_VOCABULARY":
        invalid_horizons == 0,

    "COMPLETE_REACTION_START_PRESENT":
        complete_reaction_start_missing == 0,

    "NO_FUTURE_DATA_START_CONTRACT":
        no_future_data_start_violations == 0,

    "COMPLETE_GEOMETRY_NO_NULLS":
        complete_geometry_null_violations == 0,

    "NON_NEGATIVE_FAVORABLE_EXCURSION":
        negative_favorable_excursion == 0
        and
        nonfinite_favorable_excursion == 0,

    "NON_NEGATIVE_ADVERSE_EXCURSION":
        negative_adverse_excursion == 0
        and
        nonfinite_adverse_excursion == 0,

    "DIRECTIONAL_CLOSE_CONTRACT":
        directional_close_mismatch == 0,

    "FAVORABLE_CLOSE_CONTRACT":
        favorable_close_mismatch == 0,

    "PRICE_GEOMETRY_CONTRACT":
        price_geometry_violations == 0,

    "FUTURE_HIGH_LOW_CONTRACT":
        future_high_low_violations == 0,

    "NO_INCOMPLETE_GEOMETRY_PROMOTION":
        incomplete_geometry_promoted == 0,
}


# ==================================================================================================
# DIAGNOSTICS
# ==================================================================================================

print("=" * 100)
print("STEP 8.2-C — REACTION GEOMETRY REGRESSION GATE V2")
print("=" * 100)

print()
print("DIAGNOSTICS")
print("-" * 100)

print(
    f"Expected Rows                         : "
    f"{expected_rows:,}"
)

print(
    f"Actual Rows                           : "
    f"{actual_rows:,}"
)

print(
    f"Unique Episodes                       : "
    f"{episode_count:,}"
)

print(
    f"Unique Horizons                       : "
    f"{horizon_count:,}"
)

print(
    f"Duplicate Episode × Horizon          : "
    f"{duplicate_episode_horizon:,}"
)

print(
    f"Invalid FVG Types                    : "
    f"{invalid_fvg_types:,}"
)

print(
    f"Invalid Horizons                     : "
    f"{invalid_horizons:,}"
)

print(
    f"Complete Reaction Start Missing      : "
    f"{complete_reaction_start_missing:,}"
)

print(
    f"NO_FUTURE_DATA Start Violations      : "
    f"{no_future_data_start_violations:,}"
)

print(
    f"Complete Geometry Null Violations    : "
    f"{complete_geometry_null_violations:,}"
)

print(
    f"Negative Favorable Excursions        : "
    f"{negative_favorable_excursion:,}"
)

print(
    f"Negative Adverse Excursions          : "
    f"{negative_adverse_excursion:,}"
)

print(
    f"Non-Finite Favorable Excursions      : "
    f"{nonfinite_favorable_excursion:,}"
)

print(
    f"Non-Finite Adverse Excursions        : "
    f"{nonfinite_adverse_excursion:,}"
)

print(
    f"Directional Close Mismatches         : "
    f"{directional_close_mismatch:,}"
)

print(
    f"Favorable Close Mismatches            : "
    f"{favorable_close_mismatch:,}"
)

print(
    f"Price Geometry Violations             : "
    f"{price_geometry_violations:,}"
)

print(
    f"Future High/Low Violations            : "
    f"{future_high_low_violations:,}"
)

print(
    f"Incomplete Geometry Promoted          : "
    f"{incomplete_geometry_promoted:,}"
)


# ==================================================================================================
# GATE RESULTS
# ==================================================================================================

print()
print("=" * 100)
print("STEP 8.2-C REGRESSION GATES")
print("=" * 100)

for gate_name, result in gates.items():

    print(
        f"{gate_name:<50} : "
        f"{'PASSED' if result else 'FAILED'}"
    )


failed_gates = [
    name
    for name, result in gates.items()
    if not result
]


print()
print("=" * 100)

if failed_gates:

    print("STEP 8.2-C OVERALL : FAIL")

    print(
        f"Passed Gates       : "
        f"{sum(gates.values())}/{len(gates)}"
    )

    print(
        f"Failed Gates       : "
        f"{failed_gates}"
    )

else:

    print("STEP 8.2-C OVERALL : PASS")

    print(
        f"Passed Gates       : "
        f"{sum(gates.values())}/{len(gates)}"
    )

    print("Failed Gates       : []")

print("=" * 100)

In [ ]:
# ==================================================================================================
# STEP 8.2-C-S — REACTION GEOMETRY SEMANTIC / SANITY AUDIT V1
# ==================================================================================================

import numpy as np
import pandas as pd

print("=" * 100)
print("STEP 8.2-C-S — REACTION GEOMETRY SEMANTIC / SANITY AUDIT V1")
print("=" * 100)

g = fvg_episode_reaction_geometry_v1.copy()

# --------------------------------------------------------------------------------------------------
# 1. Basic schema
# --------------------------------------------------------------------------------------------------

required_columns = [
    "interaction_episode_id",
    "fvg_id",
    "fvg_type",
    "episode_sequence",
    "interaction_episode_end",
    "episode_end_close",
    "reaction_start",
    "reaction_end",
    "horizon_minutes",
    "observation_count",
    "expected_observation_count",
    "data_complete",
    "data_status",
    "future_high",
    "future_low",
    "future_close",
    "favorable_price",
    "adverse_price",
    "favorable_excursion",
    "adverse_excursion",
    "net_close_change",
    "directional_close_change",
    "favorable_close_change",
]

missing_columns = sorted(set(required_columns) - set(g.columns))

# --------------------------------------------------------------------------------------------------
# 2. Complete subset
# --------------------------------------------------------------------------------------------------

complete_mask = (
    g["data_complete"].eq(True)
    & g["data_status"].eq("COMPLETE")
)

complete = g.loc[complete_mask].copy()
incomplete = g.loc[~complete_mask].copy()

# --------------------------------------------------------------------------------------------------
# 3. Semantic checks
# --------------------------------------------------------------------------------------------------

# BULLISH:
# favorable = future_high
# adverse   = future_low

bull = complete[complete["fvg_type"] == "BULLISH_FVG"].copy()
bear = complete[complete["fvg_type"] == "BEARISH_FVG"].copy()

bull_favorable_price_mismatch = int(
    (~np.isclose(
        bull["favorable_price"].to_numpy(dtype=float),
        bull["future_high"].to_numpy(dtype=float),
        rtol=0.0,
        atol=1e-12,
        equal_nan=False,
    )).sum()
)

bull_adverse_price_mismatch = int(
    (~np.isclose(
        bull["adverse_price"].to_numpy(dtype=float),
        bull["future_low"].to_numpy(dtype=float),
        rtol=0.0,
        atol=1e-12,
        equal_nan=False,
    )).sum()
)

# BEARISH:
# favorable = future_low
# adverse   = future_high

bear_favorable_price_mismatch = int(
    (~np.isclose(
        bear["favorable_price"].to_numpy(dtype=float),
        bear["future_low"].to_numpy(dtype=float),
        rtol=0.0,
        atol=1e-12,
        equal_nan=False,
    )).sum()
)

bear_adverse_price_mismatch = int(
    (~np.isclose(
        bear["adverse_price"].to_numpy(dtype=float),
        bear["future_high"].to_numpy(dtype=float),
        rtol=0.0,
        atol=1e-12,
        equal_nan=False,
    )).sum()
)

# --------------------------------------------------------------------------------------------------
# 4. Excursion semantic checks
# --------------------------------------------------------------------------------------------------

expected_favorable_excursion = (
    complete["favorable_price"] - complete["episode_end_close"]
).abs()

expected_adverse_excursion = (
    complete["adverse_price"] - complete["episode_end_close"]
).abs()

favorable_excursion_mismatch = int(
    (~np.isclose(
        complete["favorable_excursion"].to_numpy(dtype=float),
        expected_favorable_excursion.to_numpy(dtype=float),
        rtol=0.0,
        atol=1e-12,
        equal_nan=False,
    )).sum()
)

adverse_excursion_mismatch = int(
    (~np.isclose(
        complete["adverse_excursion"].to_numpy(dtype=float),
        expected_adverse_excursion.to_numpy(dtype=float),
        rtol=0.0,
        atol=1e-12,
        equal_nan=False,
    )).sum()
)

# --------------------------------------------------------------------------------------------------
# 5. Raw close movement
# --------------------------------------------------------------------------------------------------

expected_net_close_change = (
    complete["future_close"] - complete["episode_end_close"]
)

net_close_mismatch = int(
    (~np.isclose(
        complete["net_close_change"].to_numpy(dtype=float),
        expected_net_close_change.to_numpy(dtype=float),
        rtol=0.0,
        atol=1e-12,
        equal_nan=False,
    )).sum()
)

# --------------------------------------------------------------------------------------------------
# 6. Directional close movement
# --------------------------------------------------------------------------------------------------

expected_directional_close_change = np.where(
    complete["fvg_type"].eq("BULLISH_FVG"),
    complete["future_close"] - complete["episode_end_close"],
    complete["episode_end_close"] - complete["future_close"],
)

directional_close_mismatch = int(
    (~np.isclose(
        complete["directional_close_change"].to_numpy(dtype=float),
        expected_directional_close_change.astype(float),
        rtol=0.0,
        atol=1e-12,
        equal_nan=False,
    )).sum()
)

# --------------------------------------------------------------------------------------------------
# 7. Favorable close movement
# --------------------------------------------------------------------------------------------------

expected_favorable_close_change = np.maximum(
    expected_directional_close_change,
    0.0,
)

favorable_close_mismatch = int(
    (~np.isclose(
        complete["favorable_close_change"].to_numpy(dtype=float),
        expected_favorable_close_change.astype(float),
        rtol=0.0,
        atol=1e-12,
        equal_nan=False,
    )).sum()
)

# --------------------------------------------------------------------------------------------------
# 8. Incomplete geometry semantic contract
# --------------------------------------------------------------------------------------------------

geometry_columns = [
    "future_high",
    "future_low",
    "future_close",
    "favorable_price",
    "adverse_price",
    "favorable_excursion",
    "adverse_excursion",
    "net_close_change",
    "directional_close_change",
    "favorable_close_change",
]

incomplete_geometry_promoted = int(
    incomplete[geometry_columns].notna().any(axis=1).sum()
)

# --------------------------------------------------------------------------------------------------
# 9. Complete geometry finite-value check
# --------------------------------------------------------------------------------------------------

complete_nonfinite_geometry = 0

if len(complete) > 0:
    complete_nonfinite_geometry = int(
        (~np.isfinite(
            complete[geometry_columns]
            .to_numpy(dtype=float)
        )).sum()
    )

# --------------------------------------------------------------------------------------------------
# 10. Directional sanity
# --------------------------------------------------------------------------------------------------

# Favorable close change must never be negative.
negative_favorable_close_change = int(
    (complete["favorable_close_change"] < -1e-12).sum()
)

# Excursions must never be negative.
negative_favorable_excursion = int(
    (complete["favorable_excursion"] < -1e-12).sum()
)

negative_adverse_excursion = int(
    (complete["adverse_excursion"] < -1e-12).sum()
)

# --------------------------------------------------------------------------------------------------
# 11. Representative samples
# --------------------------------------------------------------------------------------------------

bull_sample = (
    bull.sort_values(["interaction_episode_id", "horizon_minutes"])
    .head(3)
)

bear_sample = (
    bear.sort_values(["interaction_episode_id", "horizon_minutes"])
    .head(3)
)

# --------------------------------------------------------------------------------------------------
# 12. Diagnostics
# --------------------------------------------------------------------------------------------------

print("\nBASIC")
print("-" * 100)
print(f"Total Rows                         : {len(g):,}")
print(f"Complete Rows                      : {len(complete):,}")
print(f"Incomplete Rows                    : {len(incomplete):,}")
print(f"Bullish Complete Rows              : {len(bull):,}")
print(f"Bearish Complete Rows              : {len(bear):,}")

print("\nDIRECTIONAL PRICE SEMANTICS")
print("-" * 100)
print(f"Bullish Favorable → Future High    : {bull_favorable_price_mismatch}")
print(f"Bullish Adverse   → Future Low     : {bull_adverse_price_mismatch}")
print(f"Bearish Favorable → Future Low     : {bear_favorable_price_mismatch}")
print(f"Bearish Adverse   → Future High    : {bear_adverse_price_mismatch}")

print("\nEXCURSION SEMANTICS")
print("-" * 100)
print(f"Favorable Excursion Mismatches     : {favorable_excursion_mismatch}")
print(f"Adverse Excursion Mismatches       : {adverse_excursion_mismatch}")
print(f"Negative Favorable Excursions      : {negative_favorable_excursion}")
print(f"Negative Adverse Excursions        : {negative_adverse_excursion}")

print("\nCLOSE MOVEMENT SEMANTICS")
print("-" * 100)
print(f"Net Close Change Mismatches        : {net_close_mismatch}")
print(f"Directional Close Mismatches       : {directional_close_mismatch}")
print(f"Favorable Close Mismatches         : {favorable_close_mismatch}")
print(f"Negative Favorable Close Changes   : {negative_favorable_close_change}")

print("\nDATA INTEGRITY")
print("-" * 100)
print(f"Complete Non-Finite Geometry       : {complete_nonfinite_geometry}")
print(f"Incomplete Geometry Promoted       : {incomplete_geometry_promoted}")

# --------------------------------------------------------------------------------------------------
# 13. Semantic gates
# --------------------------------------------------------------------------------------------------

semantic_gates = {
    "BULLISH_FAVORABLE_PRICE": bull_favorable_price_mismatch == 0,
    "BULLISH_ADVERSE_PRICE": bull_adverse_price_mismatch == 0,
    "BEARISH_FAVORABLE_PRICE": bear_favorable_price_mismatch == 0,
    "BEARISH_ADVERSE_PRICE": bear_adverse_price_mismatch == 0,
    "FAVORABLE_EXCURSION_SEMANTICS": favorable_excursion_mismatch == 0,
    "ADVERSE_EXCURSION_SEMANTICS": adverse_excursion_mismatch == 0,
    "NET_CLOSE_CHANGE_SEMANTICS": net_close_mismatch == 0,
    "DIRECTIONAL_CLOSE_SEMANTICS": directional_close_mismatch == 0,
    "FAVORABLE_CLOSE_SEMANTICS": favorable_close_mismatch == 0,
    "COMPLETE_GEOMETRY_FINITE": complete_nonfinite_geometry == 0,
    "NON_NEGATIVE_FAVORABLE_EXCURSION": negative_favorable_excursion == 0,
    "NON_NEGATIVE_ADVERSE_EXCURSION": negative_adverse_excursion == 0,
    "NON_NEGATIVE_FAVORABLE_CLOSE": negative_favorable_close_change == 0,
    "INCOMPLETE_GEOMETRY_NOT_PROMOTED": incomplete_geometry_promoted == 0,
}

print("\n" + "=" * 100)
print("STEP 8.2-C-S SEMANTIC GATES")
print("=" * 100)

for gate, passed in semantic_gates.items():
    print(f"{gate:<50} : {'PASSED' if passed else 'FAILED'}")

passed_count = sum(semantic_gates.values())
total_count = len(semantic_gates)

print("\n" + "=" * 100)
print(
    f"STEP 8.2-C-S OVERALL : "
    f"{'PASS' if passed_count == total_count else 'FAIL'}"
)
print(f"Passed Gates       : {passed_count}/{total_count}")
print(
    f"Failed Gates       : "
    f"{[k for k, v in semantic_gates.items() if not v]}"
)
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 14. Representative samples
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("REPRESENTATIVE BULLISH SAMPLES")
print("=" * 100)

if len(bull_sample) > 0:
    print(
        bull_sample[
            [
                "interaction_episode_id",
                "horizon_minutes",
                "episode_end_close",
                "future_high",
                "future_low",
                "future_close",
                "favorable_price",
                "adverse_price",
                "favorable_excursion",
                "adverse_excursion",
                "directional_close_change",
                "favorable_close_change",
            ]
        ].to_string(index=False)
    )
else:
    print("No complete Bullish rows.")

print("\n" + "=" * 100)
print("REPRESENTATIVE BEARISH SAMPLES")
print("=" * 100)

if len(bear_sample) > 0:
    print(
        bear_sample[
            [
                "interaction_episode_id",
                "horizon_minutes",
                "episode_end_close",
                "future_high",
                "future_low",
                "future_close",
                "favorable_price",
                "adverse_price",
                "favorable_excursion",
                "adverse_excursion",
                "directional_close_change",
                "favorable_close_change",
            ]
        ].to_string(index=False)
    )
else:
    print("No complete Bearish rows.")

In [ ]:
# print(
#     "identify_structural_levels:",
#     "identify_structural_levels" in globals()
# )